# MFT China Futures — Research Notebook

One notebook for the whole study, ordered the way a quantitative researcher works:

**data → univariate facts → relationships → factors and ranks → regimes → dynamics and monitoring → strategies, portfolio and risk → today's opportunity board**

Attach the raw dataset (77 × `<PRODUCT>9999.<EXCH>.parquet`), switch **Internet on** (so `arch` can be installed), and use **Run All**. All parameters live in **Step 0** (0b for Parts B–I).

| Part | Step | Question it answers |
|---|---|---|
| **A — Data & EDA** | 0 | Config, registry, contract metadata; research settings (0b) |
| | 1 | Ingest 1-min files → clean daily panel (with the **roll gap**, a carry proxy) + **5-min bars for every product** |
| | 2–5 | Data quality (**2f: a treatment for every flag**); returns (log vs simple); price / vol / volume / OI; correlation and clustering |
| | 6–8 | Exploratory features (incl. **sector-neutral IC**), first hypotheses, EDA summary |
| **B — Univariate properties** | 9 | Splits (embargo ≥ longest forward horizon); ADF / KPSS; **fractional differentiation d\***; point-in-time checks |
| | 10 | Mean models: AR, **ARMA(1,1) ≡ ARIMA(1,1,1)**, Huber, LAD, Ridge, Lasso |
| | 11 | Volatility: EWMA, GARCH family (**GJR**, EGARCH, TARCH, APARCH, FIGARCH), HAR-RV |
| | 11b / 11c | **Robustness of the volatility model** over blocks, regimes and time frames (walk-forward selection); **volatility expansion / contraction monitor** |
| | 12 | Residual distributions (10 families) |
| | 13 | **EVT** tails + **tail-event opportunities** (continuation / reversal, time-of-day buckets, clustering, spillover) |
| | 13b | **Multivariate structure**: cross-section (PCA, random-matrix test), covariance estimators, Fama–MacBeth, sectors, between sectors; 13.4: links **beyond days** |
| | 13c / 13d | **Copulas** (Gaussian, t, Clayton, Gumbel; tail dependence; joint extremes); **overnight return puzzle and the T+1 rule** (EquityIndex vs the rest) |
| **C — Relationships** | 14 | **Granger at 5m / 15m / 30m / 60m / 1D / 1W**, blind-tested; **survival** out to 60 days; one table of **every** link |
| | 15 | Cointegration (within / across sectors, spreads), correlation stability, tail dependence, sector → product lead |
| | 16 / 16b | **Fundamental-linked relationships**: economic chains (processing margins, product spreads, input costs); 16b: rules by length of history, rolling hedge ratio, no cointegration gate, skip log, pooled test |
| | 17 | **Relationship library**: every relationship scored and ranked + current bull / bear opportunities |
| **D — Factors & ranks** | 18 | **Factor library** (trend, momentum, volatility, mean reversion, volume / OI, **carry** incl. term-structure slope, breadth) × (TS, XS, across-sector, within-sector), López de Prado evaluation |
| | 18b | **Factor families in pairs, triplets and quadruplets**: redundancy, incremental information, conditional effects, synergy |
| | 19 | **Multi-time-frame bull / bear ranks (UpR / DnR)**; MTF rule validation; latest scorecard |
| **E — Regimes** | 20 | Vol / Markov / breadth regimes, ICSS breaks, **rule-change event studies**, factor and rank behaviour by regime |
| **F — Dynamics & monitoring** | 21 | **Momentum, mean reversion and rotation** of factors, sectors and factor × sector cells; factor × sector map; crowding and dispersion |
| | 22 | **Drift and decay monitor**: CUSUM status of every edge, rank drift, and a test of whether switching decayed edges off helps |
| **G — Strategies, portfolio, risk** | 23 | Strategy library (rank, factor, relationship, EVT, regime, dynamics, monitor-gated, meta-labelled); costs, delay, walk-forward, DSR, **PBO** |
| | 24 | **Portfolio construction**: volatility targeting, risk parity, HRP, minimum variance; capacity and cost stress |
| | 24b–24f | Efficient frontier and portfolio copula; **selection audit and keep / drop ledger**; **investor profiles (risk appetite × lock-in)**; **cost-aware library and profitable-recipe search**; netting, test-phase behaviour and levers |
| | 25 | VaR / ES / Rachev ratio with backtests |
| **H — Board** | 26 | **Opportunity board** (long = bull / short = bear, with vehicle, health-adjusted) + HTML report |
| **I — Future** | 27+ | Conventions and template |

**Outputs.** Part A writes to `/kaggle/working/eda_cache`; Parts B–H write to `/kaggle/working/research_out` (CSVs, figures, sealed blind-test maps, `report.html`, `opportunity_board_latest.csv`).

**Runtime on Kaggle (77 files).** Step 1 takes ≈ 10–25 min once. Parts B–H take ≈ 45–90 min, with Steps 14, 18, 19 and 21 the heaviest. To save time:
* set `RUN_SLOW_VOL = False`;
* drop `"5m"` from `GRANGER_TFS` / `TFS`;
* move `INTRADAY_START` later.

---
# Part A — Data foundation and exploratory analysis (Steps 0–8)

**Dataset:** 77 continuous main-contract series (`<PRODUCT>9999.<EXCH>.parquet`), 1-minute OHLCV + turnover + open interest, roughly 2005 → 2021-10.

**How to use this notebook.** Each step is self-contained and reads from the cache written by Step 1, so you can re-run any step on its own after Step 1 has run once.

| Step | What it does | Output (in `OUT_DIR`) |
|---|---|---|
| 0 | Config, file registry, contract metadata | — |
| 1 | Ingest every 1-min file once → clean returns, trading-day/session labels, roll flags → daily panel, intraday profiles, 5-min returns | `daily_panel.parquet`, `quality_raw.csv`, `intraday_profiles.parquet`, `ret5m.parquet` |
| 2 | Data quality checks | `quality_report.csv` |
| 3 | Basic statistics & return distributions | `return_stats.csv` |
| 4 | Price, return, volatility, volume & open-interest analysis | `liquidity_rank.csv` |
| 5 | Correlation, clustering, lead–lag | `corr_daily.csv`, `leadlag.csv` |
| 6 | Feature exploration (features → forward returns, rank IC) | `features.parquet`, `ic_summary.csv` |
| 7 | Trading-signal hypotheses, each tested with one number | `hypotheses.csv` |
| 8 | Summary & next steps | `eda_summary.json` |

### Data dictionary
| Column | Meaning |
|---|---|
| `date` | bar timestamp (string in the files; bar *end* time, e.g. `09:01:00` = first minute) |
| `open, high, low, close` | prices of the contract named in `symbol` |
| `volume` | lots traded in the minute |
| `money` | turnover in CNY (≈ volume × price × multiplier; CZCE historically double-counted) |
| `open_interest` | open interest of `symbol` at bar close |
| `symbol` | the actual contract (e.g. `A0505`) — the `9999` series **switches contract on rolls** |

### Why the base notebook's numbers can't be trusted yet
1. **Prefix file matching.** `f.startswith('I')` also loads IC, IF, IH; `'M'` loads MA, ME; `'P'` loads PB, PF, PG, PK, PM, PP. That is why it reported 15,654 duplicate daily records, a mean daily return of 0.62 (62%) and a max of 2,509%, and why the correlation pivot failed. Step 0 matches the product code exactly.
2. **Roll jumps.** The `9999` series splices contracts. The close-to-close return on the bar where `symbol` changes is a calendar spread, not a market move. Step 1 drops those returns.
3. **Night session.** Bars from 21:00–02:30 belong to the *next* trading day. Grouping by calendar date splits one trading day in two. Step 1 assigns a proper `trading_day`.
4. **`groupby(['symbol', day])`** creates two rows on roll days. Step 1 aggregates per trading day only.

## Step 0 — Setup, configuration and file registry
Edit the config block only. `UNIVERSE = None` processes all 77 files (≈10–25 min on Kaggle, one file in memory at a time).

**Used later.** Every later step reads these settings and the file registry. Sector and contract metadata drive the sector-level work of Steps 13b, 18, 21 and 24.

In [ ]:
# step 0
import os, re, gc, glob, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 120)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)

# ----------------------------- CONFIG ---------------------------------------
def find_data_dir():
    # Folder holding the *9999.*.parquet files. Override with env var MFT_DATA_DIR.
    env = os.environ.get("MFT_DATA_DIR")
    if env:
        return Path(env)
    for root in ["/kaggle/input", "/kaggle/working", "."]:
        hits = glob.glob(os.path.join(root, "**", "*9999.*.parquet"), recursive=True) or glob.glob(os.path.join(root, "**", "*9999.*.csv"), recursive=True)
        hits = [h for h in hits if "/eda_cache/" not in h.replace(os.sep, "/")]
        if hits:
            return Path(pd.Series([os.path.dirname(h) for h in hits]).value_counts().index[0])
    raise FileNotFoundError("No *9999.*.parquet (or .csv) files found — set MFT_DATA_DIR")

DATA_DIR = find_data_dir()
OUT_DIR = Path(os.environ.get("MFT_OUT_DIR",
               "/kaggle/working/eda_cache" if Path("/kaggle/working").exists() else "./eda_cache"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
# A cache saved earlier and attached as an input dataset is copied in, so Step 1 loads it instead of rebuilding.
# Tried in order: env MFT_CACHE_IN, the saved dataset path below, any folder under /kaggle/input holding daily_panel.parquet, any eda_cache.zip under /kaggle/input (unzipped).
CACHE_IN = [os.environ.get("MFT_CACHE_IN", ""), "/kaggle/input/datasets/sumalya32/mft-china/eda_cache", "/kaggle/input/mft-china/eda_cache"]
if not (OUT_DIR / "daily_panel.parquet").exists() and Path("/kaggle/input").exists():
    import shutil, zipfile
    _src = next((Path(c) for c in CACHE_IN if c and (Path(c) / "daily_panel.parquet").exists()), None)
    if _src is None:
        _hits = sorted(glob.glob("/kaggle/input/**/daily_panel.parquet", recursive=True)); _src = Path(_hits[0]).parent if _hits else None
    if _src is not None:
        for _f in _src.iterdir():
            if _f.is_file() and not (OUT_DIR / _f.name).exists(): shutil.copy2(_f, OUT_DIR / _f.name)
        print(f"cache reused from input dataset: {_src} ({len(list(OUT_DIR.iterdir()))} files copied)")
    else:
        _zs = sorted(glob.glob("/kaggle/input/**/eda_cache.zip", recursive=True))
        if _zs:
            with zipfile.ZipFile(_zs[0]) as _z: _z.extractall(OUT_DIR)
            print(f"cache unzipped from {_zs[0]} ({len(list(OUT_DIR.iterdir()))} files)")
        else:
            print("no cache found under /kaggle/input (looked for", CACHE_IN[1], "and any daily_panel.parquet / eda_cache.zip): Step 1 rebuilds it")
else:
    print("cache already in", OUT_DIR, f"({len(list(OUT_DIR.iterdir()))} files)")

UNIVERSE = None                    # None = all files, or e.g. ["RB", "HC", "I", "J", "JM"]
FOCUS = ["RB", "HC", "I", "J", "JM", "M", "Y", "P", "CU", "AU", "MA", "TA", "IF", "T"]  # deep-dive set
LEADLAG_SET = ["RB", "HC", "I", "J", "JM", "M", "Y", "P", "IF", "IH", "IC"]   # Step 5 lead-lag exploration set
OPEN_K = (1, 5, 15, 30, 60)       # minutes of the day-session opening move kept for the overnight study (Steps 13d, 23c)
PRE_M = (5, 15, 30)               # minutes before the day-session close kept as possible entry points
INTRADAY_SET = None               # products whose 5-min bars are kept for multi-time-frame work (None = all)
INTRADAY_START = "auto"           # 5-min bars kept from this date. "auto" = INTRADAY_FRAC of the way into the history, so the one calendar split (60 / 20 / 20) leaves every
INTRADAY_FRAC = 0.40              # frequency a training period (more history = more memory). A date string such as "2017-01-01" overrides.
DEEP_DIVE = "RB"                   # single product for the detailed price/volume/OI chart
RECENT_START = "2019-01-01"        # window for intraday profiles, 5-min returns and "recent" stats
FORCE_REBUILD = False              # True = re-run Step 1 even if cache exists
MIN_XS = 8                         # min products per date for cross-sectional statistics
ANN = 252

print("DATA_DIR:", DATA_DIR)
print("OUT_DIR :", OUT_DIR)

In [ ]:
# step 0
# ----------------------------- FILE REGISTRY --------------------------------
# Exact match on the product code (fixes the startswith() bug).
FILE_RE = re.compile(r"^(?:[0-9a-f]{8}-)?([A-Z]+)9999\.([A-Z]+)\.(parquet|csv)$")      # also accepts CSV, and a download prefix such as 'd9ec6912-'
FILES, EXCH = {}, {}
for p in sorted(list(DATA_DIR.glob("*.csv")) + list(DATA_DIR.glob("*.parquet"))):             # parquet wins if both exist (listed last)
    m = FILE_RE.match(p.name)
    if m:
        FILES[m.group(1)] = p
        EXCH[m.group(1)] = m.group(2)

EXCH_NAME = {"XSGE": "SHFE", "XDCE": "DCE", "XZCE": "CZCE", "CCFX": "CFFEX", "XINE": "INE", "GFEX": "GFEX"}

# ----------------------------- CONTRACT METADATA ----------------------------
SECTOR = {
    **dict.fromkeys(["RB", "HC", "I", "J", "JM", "WR", "SS", "SF", "SM"], "Ferrous"),
    **dict.fromkeys(["CU", "AL", "ZN", "PB", "NI", "SN", "BC"], "BaseMetal"),
    **dict.fromkeys(["AU", "AG"], "Precious"),
    **dict.fromkeys(["SC", "FU", "BU", "LU", "PG", "ZC", "TC"], "Energy"),
    **dict.fromkeys(["MA", "ME", "TA", "EG", "EB", "L", "PP", "V", "UR", "SA", "PF", "RU", "NR"], "Chemical"),
    **dict.fromkeys(["FG", "SP", "FB", "BB"], "Building/Paper"),
    **dict.fromkeys(["A", "B", "M", "Y", "P", "RM", "OI", "RO", "RS"], "Oilseeds"),
    **dict.fromkeys(["C", "CS", "WH", "WS", "WT", "PM", "RI", "ER", "JR", "LR", "RR"], "Grains"),
    **dict.fromkeys(["CF", "CY", "SR", "AP", "CJ", "PK", "JD", "LH", "GN"], "Softs/Livestock"),
    **dict.fromkeys(["IF", "IH", "IC"], "EquityIndex"),
    **dict.fromkeys(["T", "TF", "TS"], "Bond"),
}
# Contract multiplier (units per lot) from the June-2026 reference sheet; used only as a sanity
# check against turnover/(volume*price). A mismatch can mean the multiplier changed over time.
MULT = {"CU": 5, "AL": 5, "ZN": 5, "PB": 25, "NI": 1, "SN": 1, "AU": 1000, "AG": 15, "RB": 10, "HC": 10,
        "WR": 10, "SS": 5, "RU": 10, "FU": 10, "BU": 10, "SP": 10, "SC": 1000, "LU": 10,
        "A": 10, "B": 10, "M": 10, "Y": 10, "P": 10, "C": 10, "CS": 10, "JD": 10, "LH": 16, "RR": 10,
        "L": 5, "V": 5, "PP": 5, "EG": 10, "EB": 5, "PG": 20, "J": 100, "JM": 60, "I": 100,
        "WH": 20, "PM": 50, "CF": 5, "SR": 10, "OI": 10, "RS": 10, "RM": 10, "PK": 5, "CJ": 5, "AP": 10,
        "RI": 20, "MA": 10, "TA": 5, "PF": 5, "UR": 20, "FG": 20, "SA": 20, "SF": 5, "SM": 5, "ZC": 200,
        "IF": 300, "IH": 300, "IC": 200, "TS": 20000, "TF": 10000, "T": 10000}
# Current daily price limit (%) from the reference sheet; historical limits differed.
LIMIT = {"RB": 5, "HC": 5, "AU": 16, "AG": 15, "NI": 9, "SN": 9, "RU": 9, "FU": 9, "BU": 9, "SP": 7,
         "LU": 8, "SC": 8, "MA": 9, "TA": 9, "PF": 9, "FG": 9, "SA": 9, "IF": 10, "IH": 10, "IC": 10,
         "TS": 0.4, "TF": 1.0, "T": 1.5,
         **{k: 6 for k in ["A", "B", "M", "Y", "C", "CS", "JD", "LH", "RR", "L", "V", "PP",
                           "WH", "PM", "SR", "OI", "RS", "RM", "RI"]}}
# Old CZCE codes renamed to new ones -> candidates to stitch into one longer history.
LEGACY = {"ME": "MA", "RO": "OI", "TC": "ZC", "WS": "WH", "ER": "RI"}

COLS = ["date", "open", "high", "low", "close", "volume", "money", "open_interest", "symbol"]
PRICE_COLS = ["open", "high", "low", "close"]

def read_any(path, columns=COLS):
    # one reader for parquet and csv; the schema is the contract of the whole notebook
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, usecols=lambda c: c in columns, dtype={"symbol": "string"})
    return pd.read_parquet(path, columns=columns)

def file_columns(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return list(pd.read_csv(path, nrows=0).columns)
    import pyarrow.parquet as pq
    return list(pq.ParquetFile(path).schema.names)

# schema check of every file BEFORE the long ingestion: missing columns would otherwise fail 20 minutes in
SCHEMA = pd.DataFrame([dict(product=c, file=p.name, n_columns=len(cols), missing=[x for x in COLS if x not in cols], extra=[x for x in cols if x not in COLS])
                       for c, p in FILES.items() for cols in [file_columns(p)]])
if len(SCHEMA) and SCHEMA.missing.map(len).gt(0).any():
    bad = SCHEMA[SCHEMA.missing.map(len) > 0]
    raise ValueError("files with missing columns (expected %s):\n%s" % (COLS, bad[["product", "file", "missing"]].to_string(index=False)))
print(f"schema check: {len(SCHEMA)} files, all contain {len(COLS)} required columns" + (f" (extra columns ignored: {sorted(set(sum(SCHEMA.extra.tolist(), [])))})" if SCHEMA.extra.map(len).gt(0).any() else ""))
if str(INTRADAY_START) == "auto":
    _lo, _hi = [], []
    for _c in sorted(FILES, key=lambda k: -FILES[k].stat().st_size)[:15]:
        _d = pd.to_datetime(read_any(FILES[_c], ["date"])["date"], errors="coerce").dropna()
        if len(_d):
            _lo.append(_d.min()); _hi.append(_d.max())
    INTRADAY_START = (min(_lo) + (max(_hi) - min(_lo)) * INTRADAY_FRAC).to_period("M").to_timestamp() if _lo else pd.Timestamp("2019-01-01")
else:
    INTRADAY_START = pd.Timestamp(INTRADAY_START)
print(f"5-minute bars are kept from {INTRADAY_START.date()}")

registry = pd.DataFrame({
    "product": list(FILES),
    "exchange": [EXCH_NAME.get(EXCH[k], EXCH[k]) for k in FILES],
    "sector": [SECTOR.get(k, "Other") for k in FILES],
    "size_MB": [round(FILES[k].stat().st_size / 1e6, 1) for k in FILES],
    "legacy_of": [LEGACY.get(k, "") for k in FILES],
})
print(f"{len(FILES)} product files found")
display(registry.groupby(["exchange", "sector"]).size().unstack(fill_value=0))
display(registry.sort_values("size_MB", ascending=False).head(10))

### 0b — Research settings (Parts B–F)
Every parameter for Steps 9–26 is set here, so the whole study is configured in one place. The defaults are reasonable for daily and 5-min Chinese futures data. Anything marked *(check)* is reference data to verify against current exchange rules.

In [ ]:
# step 0b
import os, re, gc, glob, json, hashlib, warnings, base64, io, time, itertools
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats, optimize
from scipy.special import xlogy
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.linear_model import Ridge, Lasso, HuberRegressor, QuantileRegressor
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, log_loss, silhouette_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, kpss, coint
from statsmodels.stats.diagnostic import het_arch
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
for _c in (RuntimeWarning, FutureWarning, UserWarning, matplotlib.MatplotlibDeprecationWarning):
    warnings.filterwarnings("ignore", category=_c)
warnings.filterwarnings("ignore", module="statsmodels")
try:
    import networkx as nx
except ImportError:
    nx = None
try:
    from arch import arch_model
except ImportError:
    # No internet on Kaggle: install from a wheel in any attached input dataset (arch-*.whl), else from the internet if it is on.
    import subprocess, sys, glob as _g
    arch_model = None
    _wh = sorted(_g.glob("/kaggle/input/**/arch-*.whl", recursive=True) + _g.glob("./arch-*.whl") + _g.glob("./**/arch-*.whl", recursive=True))
    _tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    _wh = [w for w in _wh if _tag in os.path.basename(w)] or _wh
    for _cmd in ([[sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", _wh[0]]] if _wh else []) + [[sys.executable, "-m", "pip", "install", "-q", "arch"]]:
        try:
            subprocess.run(_cmd, check=True, timeout=300, capture_output=True)
            import importlib; importlib.invalidate_caches()
            from arch import arch_model
            print("arch installed from", _cmd[-1]); break
        except Exception as _e:
            arch_model = None
    if arch_model is None:
        print("arch not available -> Step 11 uses EWMA and HAR-RV only. Offline fix: attach a dataset holding arch-*.whl (see pip_wheels.zip), or switch Internet on.")
try:
    import talib
except ImportError:                                   # optional: Step 18u uses TA-Lib for the RSI when present and an identical numpy Wilder RSI otherwise
    import subprocess, sys, glob as _g
    _tw = sorted(_g.glob("/kaggle/input/**/[tT][aA]_[lL]ib-*.whl", recursive=True) + _g.glob("./**/[tT][aA]_[lL]ib-*.whl", recursive=True)); _tag = f"cp{sys.version_info.major}{sys.version_info.minor}"; _tw = [w for w in _tw if _tag in os.path.basename(w)]
    for _cmd in ([[sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", _tw[0]]] if _tw else []) + [[sys.executable, "-m", "pip", "install", "-q", "TA-Lib"]]:
        try:
            subprocess.run(_cmd, check=True, timeout=300, capture_output=True); import importlib; importlib.invalidate_caches(); import talib; print("TA-Lib installed from", _cmd[-1]); break
        except Exception:
            pass

# ---------------- outputs ----------------
OUT2 = Path(os.environ.get("MFT_OUT2_DIR", "/kaggle/working/research_out" if Path("/kaggle/working").exists() else "./research_out"))
FIG_DIR, SEALED = OUT2 / "figs", OUT2 / "sealed"
for _d in (OUT2, FIG_DIR, SEALED):
    _d.mkdir(parents=True, exist_ok=True)

# ---------------- run recorder: figures, failed cells, warnings, flagged output lines (feeds the report deck, Step 26b) ----------------
AUTOFIGS, RUNLOG, WARNLOG, FLAGS = [], [], {}, []
CELLOUT, CELLSRC = {}, {}                                  # per cell: every printed / displayed text output, and the cell source (feeds report.md, Step 27)
CELL_CAP = 60_000                                           # characters kept per cell      # AUTOFIGS: (step, title, path, cell_no) for plots that are not saved by a step itself
_CUR = dict(step="0", cell=0, t0=time.time())
_FLAG_RE = re.compile(r"WARNING|failed|Failed|Traceback|\bError\b|skipp|excluded|could not|not enough|no data|empty|insufficient|singular|did not converge|not available", re.I)
def _cap(text, stream=False):
    k = _CUR["cell"]; cur = CELLOUT.setdefault(k, [])
    if sum(len(x[1]) for x in cur) >= CELL_CAP: return
    if stream and cur and cur[-1][0] == "s": cur[-1][1] += str(text)
    else: cur.append(["s" if stream else "d", str(text)[:CELL_CAP]])
_ip = get_ipython() if "get_ipython" in globals() else None
if _ip is not None:
    for _ev, _fn in list(globals().get("_HOOKS", [])):                 # re-running this cell must not stack hooks
        try: _ip.events.unregister(_ev, _fn)
        except Exception: pass
    def _pre(info=None):
        _CUR["t0"] = time.time()
        try:
            _CUR["cell"] = _ip.execution_count
            m = re.search(r"(?m)^#\s*(?:step\s+)?(\d+[a-z]?)[\.\s]", (info.raw_cell if info is not None else "") or "")
            if m: _CUR["step"] = m.group(1)
        except Exception: pass
    def _post(result=None):
        try:
            err = (result.error_in_exec or result.error_before_exec) if result is not None else None
            raw = (result.info.raw_cell or "") if result is not None else ""
            tb = "".join(traceback.format_exception(type(err), err, err.__traceback__))[-1500:] if err is not None else ""
            CELLSRC[_CUR["cell"]] = (_CUR["step"], raw)
            if result is not None and getattr(result, "result", None) is not None:
                try: _cap("[value] " + (result.result.to_string(max_rows=200, max_cols=40) if hasattr(result.result, "to_string") else repr(result.result)))
                except Exception: pass
            if err is not None: _cap("[error] " + f"{type(err).__name__}: {err}")
            RUNLOG.append(dict(cell=_CUR["cell"], step=_CUR["step"], first=raw.strip().split("\n")[0][:100], ok=err is None,
                               error=(f"{type(err).__name__}: {err}"[:500] if err is not None else ""), traceback=tb, secs=round(time.time() - _CUR["t0"], 1)))
        except Exception: pass
    if not getattr(_ip.display_pub, "_mft_wrapped", False):
        _orig_pub = _ip.display_pub.publish
        def _pub(*a, **k):
            try:
                d = k.get("data", a[0] if a else None)
                if isinstance(d, dict):
                    if "text/plain" in d and "image/png" not in d: _cap("[display] " + str(d["text/plain"]))
                    elif "image/png" in d: _cap("[figure shown]")
            except Exception: pass
            return _orig_pub(*a, **k)
        _ip.display_pub.publish = _pub; _ip.display_pub._mft_wrapped = True
    _ip.events.register("pre_run_cell", _pre); _ip.events.register("post_run_cell", _post)
    _HOOKS = [("pre_run_cell", _pre), ("post_run_cell", _post)]

import sys, traceback
class _Tee:
    # passes everything through and keeps the lines that look like warnings or failures
    def __init__(self, base): self._b = base; self._buf = ""
    def write(self, s):
        n = self._b.write(s)
        try:
            self._buf += s
            _cap(s, stream=True)
            while "\n" in self._buf:
                line, self._buf = self._buf.split("\n", 1)
                if _FLAG_RE.search(line) and len(FLAGS) < 400:
                    FLAGS.append((_CUR["cell"], _CUR["step"], line.strip()[:230]))
        except Exception: pass
        return n
    def __getattr__(self, k): return getattr(self._b, k)
if not isinstance(sys.stdout, _Tee):
    sys.stdout = _Tee(sys.stdout)

_old_showwarning = globals().get("_old_showwarning", warnings.showwarning)
def _showwarning(message, category, filename, lineno, file=None, line=None):
    k = (category.__name__, str(message)[:140]); WARNLOG[k] = WARNLOG.get(k, 0) + 1
    if WARNLOG[k] == 1:
        return _old_showwarning(message, category, filename, lineno, file, line)
warnings.showwarning = _showwarning

_show_orig = globals().get("_show_orig", plt.show)
def _show_and_keep(*a, **k):
    # every figure that reaches plt.show() is saved once, labelled with the step found in the cell's first comment ("# 2b. ...")
    for num in plt.get_fignums():
        fig = plt.figure(num)
        if getattr(fig, "_kept", False): continue
        fig._kept = True
        try:
            ttl = (fig._suptitle.get_text() if getattr(fig, "_suptitle", None) is not None else "") or (fig.axes[0].get_title() if fig.axes else "")
            path = FIG_DIR / f"auto_{len(AUTOFIGS):03d}_{_CUR['step']}.png"
            fig.savefig(path, dpi=110, bbox_inches="tight")
            AUTOFIGS.append((_CUR["step"], ttl or f"cell {_CUR['cell']}", path, _CUR["cell"]))
        except Exception: pass
    return _show_orig(*a, **k)
plt.show = _show_and_keep

# ---------------- Step 9: splits / scaling ----------------
VOL_HL = 20                     # EWMA half-life for ex-ante vol scaling (days)
SPLIT = (0.6, 0.2, 0.2)         # train / validation / test, by time
SENS_LAGS = [1, 2, 3, 5]; EMB = 20   # embargo between periods (bars) >= the longest forward horizon used anywhere (FWD_H = 20)
EXCLUDE = ["ME", "RO", "TC", "WS", "ER"]         # legacy CZCE codes (duplicates of MA, OI, ZC, WH, RI)
MIN_OBS_SPLIT = 150; MIN_OBS_SPLIT_INTRA = 300   # min obs per period (daily / intraday)
CLIP_Z = 8; SEED = 7
FD_GRID = np.round(np.arange(0, 1.01, 0.1), 2); FD_THRESH = 1e-3   # fractional differentiation search

# ---------------- Step 10: mean models ----------------
P_ALL = 2; ROBUST_MAX_N = 50_000
ARMA_GRID = [(1, 0), (0, 1), (1, 1), (2, 1), (1, 2), (2, 2)]

# ---------------- Step 11: volatility models ----------------
VOL_SPECS = {
    "GARCH-n":   dict(vol="GARCH", p=1, o=0, q=1, dist="normal"),
    "GARCH-t":   dict(vol="GARCH", p=1, o=0, q=1, dist="t"),
    "GARCH-skt": dict(vol="GARCH", p=1, o=0, q=1, dist="skewt"),
    "GJR-t":     dict(vol="GARCH", p=1, o=1, q=1, dist="t"),
    "GJR-ged":   dict(vol="GARCH", p=1, o=1, q=1, dist="ged"),
    "EGARCH-t":  dict(vol="EGARCH", p=1, o=1, q=1, dist="t"),
    "TARCH-t":   dict(vol="GARCH", p=1, o=1, q=1, power=1.0, dist="t"),
    "APARCH-t":  dict(vol="APARCH", p=1, o=1, q=1, dist="t"),
    "FIGARCH-t": dict(vol="FIGARCH", p=1, q=1, dist="t"),
}
SLOW_VOL = {"APARCH-t", "FIGARCH-t"}; RUN_SLOW_VOL = True; EWMA_LAMBDA = 0.94

# ---------------- Step 12: distributions ----------------
FAMILIES = ["norm", "t", "laplace", "logistic", "hypsecant", "gennorm", "skewnorm", "johnsonsu", "nct", "gmm2"]
BOOT_B = 49; MAX_FIT_N = 5000

# ---------------- Step 13: EVT ----------------
EVT_Q = 0.90; EVT_QS = [0.85, 0.875, 0.90, 0.925, 0.95, 0.97]; EVT_B = 49
BLOCK = 21; EI_Q, EI_RUN = 0.95, 5
EVT_EVENT_Q = 0.95                              # |z| quantile that defines a tail event (opportunity study)
EVT_H = [1, 2, 5, 10, 20, 40, 60]              # days after a daily tail event (beyond a few days: weeks and months)
EVT_H_BARS = [1, 3, 6, 12]                      # bars after an intraday tail event
TOD_BUCKETS = [("night", 21 * 60, 26 * 60 + 30), ("open", 9 * 60, 9 * 60 + 30), ("morning", 9 * 60 + 30, 11 * 60 + 30),
               ("afternoon", 13 * 60, 14 * 60 + 30), ("close", 14 * 60 + 30, 15 * 60 + 15)]   # minutes; night wraps midnight

# ---------------- Step 14: multi-time-frame Granger ----------------
GRANGER_TFS = ["5m", "15m", "30m", "60m", "1D", "1W"]
TF_MIN = {"5m": 5, "15m": 15, "30m": 30, "60m": 60, "1D": 1440, "1W": 7200}
P_LAG_TF = {"5m": 3, "15m": 2, "30m": 2, "60m": 2, "1D": 2, "1W": 2}
MIN_PR2_TF = {"5m": 0.0005, "15m": 0.001, "30m": 0.001, "60m": 0.002, "1D": 0.002, "1W": 0.005}
FDR_Q = 0.05; VAL_Q = 0.10; CONDITION_ON_MARKET = True; SHOW_NAMES_TRAIN = True
VOL_HL_DAYS = 5                                # intraday EWMA half-life in trading days
SURV_LAGS = [1, 2, 3, 4, 6, 8, 12, 24]         # lag-decay: how many bars ahead does X still predict Y (intraday)
SURV_LAGS_D = [1, 2, 3, 5, 10, 20, 40, 60]     # daily and weekly frames: out to 60 days / 60 weeks
MIN_OBS_SPLIT_TF = {"1W": 60}                  # min obs per period for frames with few observations
SURV_WIN_DAYS = {"intra": (10, 5), "1D": (126, 21), "1W": (104, 13)}   # rolling survival window and step (trading days)

# ---------------- Step 15: pairs, correlation, tail dependence ----------------
N_CROSS_PAIRS = 30; TAILDEP_Q = 0.95; COINT_P = 0.05
SPREADS = {"crush B~M+Y": ("B", ["M", "Y"]), "coke margin J~JM": ("J", ["JM"]), "steel margin RB~I+J": ("RB", ["I", "J"]),
           "crush A~M+Y": ("A", ["M", "Y"]), "IF~IH": ("IF", ["IH"]), "RB~HC": ("RB", ["HC"]), "Y~P": ("Y", ["P"])}

# ---------------- Step 17: relationship scorecard weights ----------------
REL_W = {"strength": 0.35, "oos": 0.30, "stability": 0.15, "survival": 0.10, "tradability": 0.10}

# ---------------- Step 18: factor library (López de Prado style) ----------------
FWD_H = [1, 5, 20]; TS_Z_WIN = 250; MIN_SEC_WS = 3
CUSUM_K = 1.0                                 # CUSUM event filter threshold, in daily sigmas
TB_H, TB_K = 10, 1.5                          # triple barrier: vertical barrier (days) and width (sigma*sqrt(H))
N_FOLDS = 5; EMBARGO_PCT = 0.01; RF_TREES = 200; MAX_EVENTS = 30_000

# ---------------- Step 19: multi-time-frame bull / bear ranks ----------------
TFS = ["5m", "15m", "30m", "60m", "1D", "1W"]
RANK_WIN = {"5m": (12, 48), "15m": (8, 32), "30m": (8, 32), "60m": (6, 24), "1D": (10, 50), "1W": (4, 13)}   # (fast, slow) bars
RANK_H = {"5m": 12, "15m": 8, "30m": 8, "60m": 6, "1D": 5, "1W": 4}      # forward bars used to validate each TF
TF_WEIGHTS = {"5m": 0.5, "15m": 0.75, "30m": 1.0, "60m": 1.5, "1D": 2.0, "1W": 2.0}   # emphasis on 60m and above
FAMILY_W = {"trend": 1.0, "momentum": 1.0, "volume": 0.75, "breadth": 0.5, "volatility": 0.5, "mean_reversion": 0.5}
RANK_THRESH = {tf: 50 for tf in TFS}          # UpR >= threshold counts as bullish on that TF (DnR <= -threshold bearish)
MTF_RULE_TFS = ["60m", "1D", "1W"]            # bull = UpR >= threshold on all of these (the "60m and above" rule)
RANK_VARIANT = "learned"                      # "rule" (fixed indicator directions) or "learned" (train-IC directions & weights)

# ---------------- Step 20: regimes ----------------
VOL_REGIME_Q = (1 / 3, 2 / 3); BREADTH_BULL, BREADTH_BEAR = 0.6, 0.4
ICSS_CRIT, ICSS_MIN_SEG = 1.358, 60; EVENT_WIN, EVENT_GAP = 120, 5
RULE_EVENTS = [   # (date, products, description) — (check) dates against exchange notices
    ("2015-09-07", ["IF", "IH", "IC"], "CFFEX index-futures curbs: higher margin, 10-lot intraday open limit, close-today fee raised"),
    ("2017-02-17", ["IF", "IH", "IC"], "CFFEX first relaxation of 2015 curbs"),
    ("2017-09-18", ["IF", "IH", "IC"], "CFFEX second relaxation"),
    ("2018-12-03", ["IF", "IH", "IC"], "CFFEX third relaxation"),
    ("2019-04-22", ["IF", "IH", "IC"], "CFFEX fourth relaxation"),
    ("2016-04-22", ["RB", "HC", "I", "J", "JM"], "Ferrous speculation curbs (fees and margins raised)"),
    ("2018-05-04", ["I"], "Iron ore opened to foreign investors"),
    ("2021-05-12", ["I", "RB", "HC", "J", "JM"], "Commodity price-stabilisation measures"),
    ("2021-10-19", ["ZC", "J", "JM"], "Coal market intervention"),
]

# ---------------- Steps 23-25: strategies, backtest, risk ----------------
DELAYS = [0, 1]; MAIN_DELAY = 1               # bars between signal (close t) and execution
FEE_BPS = 1.0; SLIP_TICKS = 1.0; DEFAULT_COST_BPS = 3.0
TARGET_VOL = 0.10; LEV_CAP = 3.0; PAIR_LOOKBACK = 500; PAIR_Z_IN, PAIR_Z_OUT, PAIR_Z_WIN = 2.0, 0.5, 60
EVT_HOLD = 5; PBO_S = 12
VAR_LEVELS = [0.95, 0.99]; ES_LEVEL = 0.975; RACHEV_A = 0.05; HS_WIN = 250
# minimum price tick (CNY per quoted unit) — (check)
TICK = {"CU": 10, "AL": 5, "ZN": 5, "PB": 5, "NI": 10, "SN": 10, "AU": 0.02, "AG": 1, "RB": 1, "HC": 1, "WR": 1, "SS": 5,
        "RU": 5, "NR": 5, "FU": 1, "BU": 1, "SP": 2, "SC": 0.1, "LU": 1, "BC": 10,
        "A": 1, "B": 1, "M": 1, "Y": 2, "P": 2, "C": 1, "CS": 1, "JD": 1, "LH": 5, "RR": 1, "L": 1, "V": 1, "PP": 1,
        "EG": 1, "EB": 1, "PG": 1, "J": 0.5, "JM": 0.5, "I": 0.5, "FB": 0.5, "BB": 0.05,
        "WH": 1, "PM": 1, "CF": 5, "CY": 5, "SR": 1, "OI": 1, "RS": 1, "RM": 1, "PK": 2, "CJ": 5, "AP": 1, "RI": 1,
        "JR": 1, "LR": 1, "MA": 1, "TA": 2, "PF": 2, "UR": 1, "FG": 1, "SA": 1, "SF": 2, "SM": 2, "ZC": 0.2,
        "IF": 0.2, "IH": 0.2, "IC": 0.2, "T": 0.005, "TF": 0.005, "TS": 0.002}
# exchange margin (fraction) from the June-2026 reference sheet — caps gross leverage per product at 1/margin
MARGIN = {"CU": .10, "AL": .10, "ZN": .10, "PB": .10, "NI": .12, "SN": .12, "AU": .17, "AG": .17, "RB": .07, "HC": .07, "WR": .10,
          "SS": .10, "RU": .11, "BR": .11, "FU": .11, "BU": .11, "SP": .08, "PO": .08, "SC": .15, "LU": .12, "EC": .15,
          "A": .08, "B": .08, "M": .08, "Y": .08, "P": .10, "C": .08, "CS": .08, "JD": .08, "LH": .08, "RR": .08, "L": .08,
          "V": .08, "PP": .08, "EG": .10, "EB": .10, "PG": .10, "J": .10, "JM": .10, "I": .10,
          "WH": .08, "PM": .08, "CF": .10, "SR": .08, "OI": .08, "RS": .08, "RM": .08, "PK": .10, "CJ": .10, "AP": .10, "RI": .08,
          "MA": .10, "TA": .10, "PX": .10, "PF": .10, "UR": .10, "FG": .10, "SA": .10, "SF": .10, "SM": .10, "ZC": .12,
          "IF": .12, "IH": .12, "IC": .12, "IM": .12, "TS": .005, "TF": .012, "T": .02, "TL": .035,
          "SI": .14, "LC": .12, "PS": .12, "PT": .12, "PD": .12}
EXCH_SUMMARY = pd.DataFrame({"products": [21, 19, 20, 8, 5], "min_margin_%": [7, 8, 8, 0.5, 12], "max_leverage_x": [14, 13, 13, 200, 8],
                             "coverage": ["Base/Precious Metals, Ferrous, Rubber, Energy, Paper", "Agriculture, Chemicals, Ferrous/Coal",
                                          "Agriculture, Chemicals, Glass, Energy (Coal)", "Stock Index, Treasury Bond",
                                          "New Energy Materials, Precious Metals"]}, index=["SHFE", "DCE", "CZCE", "CFFEX", "GFEX"])
SECTOR.update({**dict.fromkeys(["SI", "LC", "PS"], "NewEnergy"), **dict.fromkeys(["PT", "PD"], "Precious"),
               "BR": "Chemical", "PX": "Chemical", "PO": "Building/Paper", "EC": "Shipping", "IM": "EquityIndex", "TL": "Bond"})

# ---------------- Step 16: fundamental-linked relationships (dependent, [drivers]) ----------------
# economic priors: each chain says "the dependent product is priced off its drivers". Chains whose products are missing are skipped.
FUND_CHAINS = [
    ("Steel margin: rebar vs iron ore + coke", "RB", ["I", "J"]), ("Coke vs coking coal", "J", ["JM"]), ("HRC vs rebar", "HC", ["RB"]),
    ("Ferrosilicon vs rebar", "SF", ["RB"]), ("Silico-manganese vs rebar", "SM", ["RB"]), ("Stainless vs nickel", "SS", ["NI"]),
    ("Glass vs soda ash + thermal coal", "FG", ["SA", "ZC"]), ("Soda ash vs thermal coal", "SA", ["ZC"]),
    ("Bitumen vs crude", "BU", ["SC"]), ("Fuel oil vs crude", "FU", ["SC"]), ("LPG vs crude", "PG", ["SC"]), ("PTA vs crude", "TA", ["SC"]),
    ("MEG vs crude", "EG", ["SC"]), ("Styrene vs crude", "EB", ["SC"]), ("Polyester fibre vs PTA + MEG", "PF", ["TA", "EG"]),
    ("Methanol vs thermal coal", "MA", ["ZC"]), ("Urea vs thermal coal", "UR", ["ZC"]), ("PP vs methanol", "PP", ["MA"]),
    ("PP vs polyethylene", "PP", ["L"]), ("Polyethylene vs crude", "L", ["SC"]),
    ("Crush: soybean vs meal + oil", "A", ["M", "Y"]), ("Crush: soybean no.2 vs meal + oil", "B", ["M", "Y"]),
    ("Soy oil vs palm oil", "Y", ["P"]), ("Rapeseed oil vs soy oil", "OI", ["Y"]), ("Rapeseed meal vs soymeal", "RM", ["M"]),
    ("Starch vs corn", "CS", ["C"]), ("Hog vs corn + meal", "LH", ["C", "M"]), ("Eggs vs corn + meal", "JD", ["C", "M"]),
    ("Cotton yarn vs cotton", "CY", ["CF"]), ("NR vs RU", "NR", ["RU"]), ("BR vs RU", "BR", ["RU"]),
    ("Aluminium vs alumina", "AL", ["AO"]), ("Copper vs aluminium", "CU", ["AL"]), ("Silver vs gold", "AG", ["AU"]),
    ("Small-cap vs large-cap index", "IC", ["IH"]), ("Mid-cap vs large-cap index", "IM", ["IF"]), ("10y vs 5y bond", "T", ["TF"]),
]
FUND_Z_WIN = 120; FUND_MIN_N = 250; FUND_H = [1, 5, 20]

# ---------------- Step 16b: fundamental chains with a rolling hedge ratio (rules by length of history) ----------------
FUND_BETA_WIN = 250; FUND_KALMAN_DELTA = 1e-4; FUND_KALMAN_VE = 1e-4      # rolling hedge-ratio window (days, lagged one day); Kalman state noise and observation noise (log-price scale)
FUND_HL_MAX = 60; FUND_T_VAL = 1.5; FUND_T_TEST = 1.0; FUND_FDR_Q = 0.10   # half-life limit (days), t on validation / test, false-discovery rate across chains
FUND_CASE_A_TRAIN = 500; FUND_CASE_B_VAL = 250; FUND_MIN_SIGNAL = 120       # signal days needed: case A on train, case B on validation, any case in total
FUND_FULL_SIZE_YEARS = 8                                                    # a chain with this many years of signal gets full risk; less gets years / this
FUND_STANDIN = {"SC": "FU", "PG": "BU"}                                       # a long-history product that stands in for a short-history leg (used only if the two co-move)
FUND_STANDIN_MIN_CORR = 0.5

# ---------------- Step 21: factor / sector / factor-sector dynamics ----------------
DYN_LOOKBACKS = [10, 20, 60, 120]; DYN_HOLDS = [5, 20]; VR_Q = [5, 20, 60]
DYN_TOPK = {"factor": 2, "sector": 2, "factor_sector": 5}     # cells held by a rotation strategy (top-k, or top/bottom-k for sectors)
DYN_Z_WIN = 250; STATE_WIN = 60

# ---------------- Step 22: drift & decay monitor ----------------
MON_WIN = 120; MON_ARL0 = 500; MON_MIN_TRAIN_SHARPE = 0.3   # rolling window (days); target in-control run length of the CUSUM (days between false alarms); min train Sharpe to count as an edge
RANK_DRIFT_LAG = 20

# ---------------- Step 24: portfolio construction ----------------
PORT_TARGET_VOL = 0.10; PORT_COV_WIN = 250; PORT_MIN_OBS = 120; PORT_REBAL = 21; PORT_LEV_CAP = 3.0
PORT_VOL_HL = 30; COST_STRESS = [0.5, 1.0, 2.0, 3.0]; PARTICIPATION = 0.05

# ---------------- Step 11b / 11c: volatility robustness and expansion / contraction monitor ----------------
VOL_SLICE_DAYS = 30; VOL_WF_MIN_BLOCKS = 3             # trading days per evaluation block; blocks needed before walk-forward selection starts
VOL_TFS = ["5m", "15m", "30m", "60m", "1D", "1W"]; VOL_TF_N = 12; VOL_TF_ARCH = True; VOL_TF_MAX_FIT = 15_000     # time frames, products, GARCH on/off, max train bars per fit
VOLMON_S, VOLMON_L, VOLMON_UP, VOLMON_DN = 10, 60, 1.25, 0.80      # short / long window (days), expansion and contraction thresholds on the ratio

# ---------------- Step 13.4 / 13b: beyond days, multivariate structure ----------------
TAIL_SPILL_W = [1, 5, 20]; LR_H = [5, 20, 60]          # spillover windows and long lead-lag horizons (days)
COV_WIN, COV_REBAL = 250, 21                           # covariance estimators: estimation window, re-estimation interval (days)

# ---------------- Step 18b: factor families in groups ----------------
COMBO_LEVELS = ["XS", "WS"]; COMBO_MAX = 4; COMBO_REDUNDANT = 0.6    # levels with a cross-section, largest group size, correlation above which two families count as the same bet

# ---------------- Step 2f data treatment | 13c copulas | 13d overnight | 23c overnight strategy | 24b frontier | 26b findings ----------------
COPULA_MAX_PAIRS = 60; COPULA_Q = 0.10; COPULA_K = 12; COPULA_SIM = 20_000      # pairs fitted, tail quantile, products in the universe copula, Monte Carlo draws
FRONT_SHRINK = 0.5; FRONT_MAXW = 0.5; FRONT_N = 25; PORT_COPULA_SIM = 100_000   # shrinkage of mean returns toward their average, max weight, frontier points, simulated days
OVN_T_MIN = 2.0; OVN_M = (0, 5, 15, 30); OVN_K = (1, 5, 15, 30, 60); OVN_VOL_WIN = 60; OVN_COST_MULT = 1.0   # per-product bias |t|, minutes before close / after open, vol window, cost multiple
FOLLOW_MIN_VAL, FOLLOW_MIN_TEST, FOLLOW_MIN_EXEC = 0.5, 0.5, 0.3     # verdict thresholds (annualised Sharpe) of the findings report

# ---------------- Step 23a: execution model | Step 23b: CPCV ----------------
EXEC_Y = 0.7                                   # square-root impact constant: cost per unit traded = EXEC_Y * daily sigma * sqrt(traded value / ADV)
EXEC_AUM = [1e7, 1e8, 5e8, 2e9]; EXEC_AUM_MAIN = 1e8       # CNY: book sizes at which the library is re-costed; the main one drives selection checks
EXEC_ADV_WIN = 20                              # days, median traded value (lagged one day)
EXEC_HS_CAP_TICKS = 3.0                        # cap on the estimated half-spread, in ticks (bar-based estimators are noisy)
REG_WIN, REG_MIN = 500, 250                      # volatility regime (Steps 11d, 13e, 23d): trailing days for the percentile rank of 20-day volatility, minimum history
HARON_MIN = 250                                  # Step 11d: minimum training days with a night session for a product's HAR-RV-ON fit
PRE_BLK = 24; HV_WIN = 60; HV_X = (1.25, 1.5, 2.0); HV_CONFIRM = (1, 2); HV_DUR_MIN = 60; HV_K = (5, 15, 30); HV_FIXED = (0, 1, 3, 6, 12, 24)   # Step 23e: 5-minute pre-close blocks kept, baseline days, high-vol multiples, confirmation blocks, minimum expected duration (min), exits, fixed-entry controls (blocks)
OR_K = (5, 15, 30, 60); OR_THETA = (0.0, 0.5, 1.0, 1.5)   # Step 23d opening reversal: exit minute after the open, |gap| threshold in vol units
LOCK_FRAC, LOCK_RANGE = 0.9, 0.25              # limit-locked day: |return| >= LOCK_FRAC * limit and (high - low) / close <= LOCK_RANGE * limit
CPCV_N, CPCV_K, CPCV_PURGE, CPCV_TOPM = 10, 2, 10, 5       # groups, test groups per split, purge/embargo days each side of a test group, strategies per selected book

rng = np.random.default_rng(SEED)
print("research outputs →", OUT2)
display(EXCH_SUMMARY)

### 0c — Causality guardrails (no look-ahead, anywhere)

**Rule.** Every value used at time $t$ (feature, signal, state, weight, scaling statistic) must be a pure function of data available at $t$ and earlier. Only targets (forward returns, forward volatility) may look ahead, and they are never inputs.

| Forbidden | Why it leaks | Guardrail used in this notebook |
|---|---|---|
| `shift(-k)` in a feature | uses $t+k$ | only `shift(k)` with $k>0$; `shift(-k)` is allowed **only** to build a target (`FWD`, `fwd_*`), never an input |
| centred windows (`center=True`) | uses $t+1, t+2, \dots$ | trailing windows only: `rolling(w, min_periods=...)` |
| day-level / full-sample normalisation | the day's or sample's mean / std is not known at $t$ | trailing or expanding statistics, or **fixed statistics from the training period** |
| `bfill` / `interpolate` | fills a gap with the next value | `ffill` only; a warm-up gap is filled with a fixed training-period value |
| target scaling with the whole sample | the max / min / quantile of the future | fixed clip (`CLIP_Z`) or percentiles estimated on train only |
| state that depends on where the data start | a recursive feature changes when history is truncated | features are recomputed on the full history and the truncation test (below) proves they do not depend on later data |
| trading-day labels from the next bar | backfilling the date of the next day session uses the future | `session_day()` from the exchange calendar: evening bars → next business day, after-midnight bars → same date rolled to a business day |

**Truncation test.** `causal_test(name, fn, *inputs)` runs `fn` on the full data and on the data cut at several dates $T$, and asserts with `pandas.testing.assert_frame_equal` that every output row with $t \le T$ is **identical** (to 1e-8). It is run on every feature builder in the notebook (Steps 1, 11c, 18, 18b, 19), and every result is logged in `CAUSAL_LOG`. With `CAUSAL_STRICT = True` a failure stops the run.

**Self-test (this cell).** Before anything else: (1) a dummy minute-bar day with a night session is aggregated to 5-minute and daily bars and passes the test; (2) **negative controls** — `shift(-1)`, a centred window, `bfill`, day-level normalisation and full-sample z-scoring — must each be **caught**. If a control passes, the test itself is broken and the notebook stops.

**Static scan.** Step 26a scans the source of every executed cell for the forbidden patterns and lists each hit as `LABEL` (target construction, allowed), `ALLOWED` (descriptive EDA or same-date cross-section, marked `# causal-ok:` with the reason) or `REVIEW`.

**Used later.** `session_day` labels the bars of Step 1 and of every intraday frame. `causal_test` guards Steps 1, 11c, 18, 18b and 19. `CAUSAL_LOG` and the scan feed the causality audit (Step 26a) and the report deck (Step 26c).

In [ ]:
# step 0c
# causality-guardrail-definitions (this cell is excluded from the static scan: it contains the forbidden patterns on purpose)
import warnings as _w
from pandas.testing import assert_frame_equal
CAUSAL_STRICT = True          # a failed truncation test stops the run
CAUSAL_ATOL = 1e-8
CAUSAL_LOG = []

def session_day(ts):
    # trading day of every bar from the CALENDAR only: 20:00-23:59 → next business day; 00:00-03:59 → same date rolled forward to a business day
    # (Friday-night session → Monday). No backward fill: a bar's label never depends on bars that come after it.
    ts = pd.DatetimeIndex(pd.to_datetime(ts)); h = np.asarray(ts.hour); d = ts.normalize()
    u = d.unique()
    with _w.catch_warnings():
        _w.simplefilter("ignore")
        nxt = pd.Series(u + pd.offsets.BDay(1), index=u); rf = pd.Series(u + pd.offsets.BDay(0), index=u)
    out = d.to_numpy().copy(); eve, morn = h >= 20, h < 4
    if eve.any(): out[eve] = nxt.reindex(d[eve]).to_numpy()
    if morn.any(): out[morn] = rf.reindex(d[morn]).to_numpy()
    return pd.DatetimeIndex(out)

def _causal_frames(out, key="out"):
    # flatten any output (frame, series, dict, tuple) into {name: DataFrame with a DatetimeIndex}
    if isinstance(out, pd.DataFrame): return {key: out} if isinstance(out.index, pd.DatetimeIndex) else {}
    if isinstance(out, pd.Series): return {key: out.to_frame()} if isinstance(out.index, pd.DatetimeIndex) else {}
    r = {}
    if isinstance(out, dict):
        for k, v in out.items(): r.update(_causal_frames(v, f"{key}.{k}"))
    elif isinstance(out, (tuple, list)):
        for i, v in enumerate(out): r.update(_causal_frames(v, f"{key}[{i}]"))
    return r

def _causal_cut(x, T):
    if isinstance(x, (pd.DataFrame, pd.Series)) and isinstance(x.index, pd.DatetimeIndex): return x.loc[:T]
    if isinstance(x, dict): return {k: _causal_cut(v, T) for k, v in x.items()}
    if isinstance(x, tuple): return tuple(_causal_cut(v, T) for v in x)
    if isinstance(x, list): return [_causal_cut(v, T) for v in x]
    return x

def causal_test(name, fn, *args, cuts=None, step="", strict=None, atol=None, quiet=False, **kw):
    # truncation test: outputs of fn(data cut at T) must equal outputs of fn(full data) for every t <= T
    strict = CAUSAL_STRICT if strict is None else strict; atol = CAUSAL_ATOL if atol is None else atol
    full = _causal_frames(fn(*args, **kw))
    if not full:
        CAUSAL_LOG.append(dict(step=step, test=name, cut=None, output="-", passed=None, max_abs_diff=np.nan, first_bad=None, note="no time-indexed output"))
        return None
    idx = max((v.index for v in full.values()), key=len)
    if cuts is None:
        cuts = [idx[int(len(idx) * q)] for q in (0.35, 0.6, 0.85) if len(idx) > 10]
    ok_all = True
    for T in cuts:
        part = _causal_frames(fn(*_causal_cut(args, T), **_causal_cut(kw, T)))
        for k, a in part.items():
            if k not in full: continue
            a = a.loc[:T]; b = full[k].loc[:T]
            if a.shape != b.shape or not a.index.equals(b.index):
                b = b.reindex(index=a.index.union(b.index), columns=a.columns.union(b.columns)); a = a.reindex(index=b.index, columns=b.columns)
            num = a.select_dtypes("number").columns.intersection(b.select_dtypes("number").columns)
            dif = (a[num].astype(float) - b[num].astype(float)).abs() if len(num) else pd.DataFrame(index=a.index)
            mx = float(np.nanmax(dif.to_numpy())) if dif.size and np.isfinite(dif.to_numpy()).any() else 0.0
            try:
                assert_frame_equal(a, b, check_dtype=False, check_exact=False, atol=atol, rtol=0, check_like=True, check_freq=False, check_names=False)
                passed, first = True, None
            except AssertionError:
                passed = False
                bad = ((dif > atol) | (a[num].isna() != b[num].isna())).any(axis=1) if len(num) else (a.astype(str) != b.astype(str)).any(axis=1)
                obj = a.columns.difference(num)
                if len(obj): bad = bad | (a[obj].astype(str) != b[obj].astype(str)).any(axis=1)
                first = bad[bad].index.min() if bad.any() else None
            ok_all &= passed
            CAUSAL_LOG.append(dict(step=step, test=name, cut=pd.Timestamp(T), output=k, passed=passed, max_abs_diff=mx, first_bad=first, note=""))
    if not quiet:
        print(f"causality (truncation) test {'PASSED' if ok_all else 'FAILED'}: {name} | {len(cuts)} cut dates × {len(full)} outputs")
    if not ok_all and strict:
        bad = [r for r in CAUSAL_LOG if r["test"] == name and r["passed"] is False]
        raise AssertionError(f"LOOK-AHEAD: '{name}' changes when later data are removed: " + "; ".join(f"{r['output']} first differs at {r['first_bad']} (cut {r['cut'].date()})" for r in bad[:5]))
    return ok_all

CAUSAL_PATTERNS = [
    (r"\.shift\(\s*-", "shift(-k) uses t+k", "only shift(k>0); shift(-k) only to build a target"),
    (r"center\s*=\s*True", "centred window uses t+1, t+2, ...", "trailing rolling window"),
    (r"\bbfill\(|backfill|method\s*=\s*['\"]b", "backfill takes the next (future) value", "ffill, or a fixed training-period value"),
    (r"\.interpolate\(", "interpolation uses the next value", "ffill"),
    (r"groupby\([^)]*(trading_day|\bday\b|\bdate\b)[^)]*\)[^#\n]*transform\(\s*['\"](mean|std|sum|max|min)", "day-level normalisation", "trailing / expanding statistics or fixed training statistics"),
    (r"fit_transform\(|StandardScaler\(\)|MinMaxScaler\(\)|zscore\(", "global scaling fitted on all data", "fit on the training period only"),
    (r"\(\s*(\w+)\s*-\s*\1\.mean\(\)\s*\)\s*/\s*\1\.std\(\)", "full-sample z-score", "trailing / expanding z-score or training statistics"),
]
_LABEL_RE = re.compile(r"\b(fwd\w*|FWD\w*|fut\w*|future\w*|target\w*|label\w*|exp_next|Yw|y_next|next_\w*)\b")

def static_causality_scan(sources):
    # sources: {name: code}; returns one row per suspicious line with a verdict
    rows = []
    for nm, src in sources.items():
        if "causality-guardrail-definitions" in str(src):
            continue
        for ln, line in enumerate(str(src).split("\n"), 1):
            code_ = line.split("#", 1)[0] if "causal-ok" not in line else line
            for pat, why, fix in CAUSAL_PATTERNS:
                if re.search(pat, code_):
                    v = "ALLOWED" if "causal-ok" in line else ("LABEL" if (pat.startswith(r"\.shift") and _LABEL_RE.search(line)) else "REVIEW")
                    rows.append(dict(cell=nm, line=ln, verdict=v, issue=why, guardrail=fix, code=line.strip()[:160]))
    return pd.DataFrame(rows, columns=["cell", "line", "verdict", "issue", "guardrail", "code"])

# ---------------- self-test: a dummy minute-bar day, then negative controls ----------------
def _dummy_minutes(days=4, seed=0):
    r_ = np.random.default_rng(seed); out = []
    for d in pd.bdate_range("2021-03-01", periods=days):
        sess = list(pd.date_range(d - pd.Timedelta(hours=3), d - pd.Timedelta(hours=1), freq="1min")) if d.weekday() else []   # previous evening 21:00-23:00
        sess += list(pd.date_range(d + pd.Timedelta(hours=9), d + pd.Timedelta(hours=11, minutes=30), freq="1min")) + list(pd.date_range(d + pd.Timedelta(hours=13, minutes=30), d + pd.Timedelta(hours=15), freq="1min"))
        out += sess
    ts = pd.DatetimeIndex(sorted(set(out))); n = len(ts)
    px = 4000 * np.exp(np.cumsum(r_.normal(0, 4e-4, n)))
    return pd.DataFrame({"close": px, "volume": r_.integers(1, 50, n)}, index=ts)

def _bars(dfm):
    # the same rules as Step 1: calendar trading day, 5-minute bars stamped at their END (label/closed = right), daily bars per trading day
    x = dfm.assign(trading_day=session_day(dfm.index), ret=np.log(dfm["close"]).diff())
    b5 = x.resample("5min", label="right", closed="right").agg({"close": "last", "volume": "sum", "ret": "sum"}).dropna(subset=["close"])
    g = x.groupby("trading_day"); dl = pd.DataFrame({"close": g["close"].last(), "volume": g["volume"].sum(), "ret": g["ret"].sum()})
    dl.index = dl.index + pd.Timedelta(hours=15)      # a daily bar is stamped at the session close, the moment it is known
    return {"bars_5m": b5, "daily": dl.iloc[:-1]}     # the last trading day may still be in progress: never use an unfinished day

_dm = _dummy_minutes()
_cuts_m = [_dm.index[int(len(_dm) * q)] for q in (0.3, 0.55, 0.8)]
def _bars_upto(dfm):
    o = _bars(dfm); T = dfm.index[-1]
    return {"bars_5m": o["bars_5m"].loc[:T.floor("5min")], "daily": o["daily"]}
causal_test("Step 1 bar building on a dummy minute-bar day (5-min + daily)", _bars_upto, _dm, cuts=_cuts_m, step="0c")

_ctrl = {
    "shift(-1)":                     lambda s: s.shift(-1),
    "centred rolling mean":          lambda s: s.rolling(5, center=True, min_periods=1).mean(),
    "bfill":                         lambda s: s.where(s.index.minute % 7 != 0).bfill(),
    "day-level normalisation":       lambda s: s / s.groupby(s.index.normalize()).transform("mean"),
    "full-sample z-score":           lambda s: (s - s.mean()) / s.std(),
}
_good = {
    "trailing rolling mean":         lambda s: s.rolling(5, min_periods=1).mean(),
    "ffill":                         lambda s: s.where(s.index.minute % 7 != 0).ffill(),
    "expanding z-score (shifted)":   lambda s: (s - s.expanding(20).mean().shift(1)) / s.expanding(20).std().shift(1),
    "EWMA":                          lambda s: s.ewm(halflife=10).mean(),
}
_px = _dm["close"]; ctl_rows = []
_cm = _px.index[_px.index.minute % 7 == 0]; _cuts_c = [_cm[int(len(_cm) * q)] for q in (0.3, 0.55, 0.8)]   # cuts on masked minutes, so a backfill has something to leak
for nm, f in _ctrl.items():
    caught = causal_test(f"negative control: {nm}", f, _px, cuts=_cuts_c, step="0c", strict=False, quiet=True) is False
    ctl_rows.append(dict(operation=nm, expected="caught", result="caught" if caught else "MISSED"))
for nm, f in _good.items():
    ok = causal_test(f"positive control: {nm}", f, _px, cuts=_cuts_c, step="0c", strict=False, quiet=True)
    ctl_rows.append(dict(operation=nm, expected="passes", result="passes" if ok else "FALSE ALARM"))
CTL = pd.DataFrame(ctl_rows); print(CTL.to_string(index=False))
assert (CTL.result.isin(["caught", "passes"])).all(), "the truncation test itself is broken: " + str(CTL[~CTL.result.isin(['caught', 'passes'])].to_dict('records'))
assert (session_day(pd.DatetimeIndex(["2021-03-05 21:30", "2021-03-06 01:00", "2021-03-08 09:01", "2021-03-08 22:00"])) ==
        pd.DatetimeIndex(["2021-03-08", "2021-03-08", "2021-03-08", "2021-03-09"])).all(), "session_day calendar rule"
print("guardrails ready: session_day, causal_test, static_causality_scan | negative controls all caught, positive controls all pass")

## Step 1 — Ingestion pass (heavy; runs once, then cached)
For each product, one file at a time:
1. parse timestamps, sort, drop duplicate timestamps;
2. label **session** (night = 20:00–04:00) and **trading_day** (a night bar belongs to the next day-session date present in the data — this handles weekends and holidays without a calendar);
3. compute 1-min log returns and **null the return on bars where `symbol` changes** (roll);
4. aggregate to a daily bar per trading day: OHLC, volume, turnover, last OI, realized vol, night/day-session returns;
5. record raw quality statistics, an intraday volume/volatility profile (recent window) and 5-min returns for the lead–lag set.

> Daily return = sum of clean 1-min returns, i.e. close-to-close across trading days with roll jumps removed.

**Used later.** The daily panel feeds Steps 2–26. The 5-minute bars (return, realised variance, volume, OI, high / low / close) feed Steps 11b, 13, 14, 19 and 23a. The roll gap and the contract codes feed the carry factors of Step 18.

In [ ]:
# step 1
def process_product(code):
    raw = read_any(FILES[code])
    q = {"product": code, "exchange": EXCH_NAME.get(EXCH[code], EXCH[code]),
         "sector": SECTOR.get(code, "Other"), "rows_raw": len(raw)}

    # -- timestamps ----------------------------------------------------------
    raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
    q["bad_timestamps"] = int(raw["date"].isna().sum())
    raw = raw.dropna(subset=["date"])
    q["unsorted"] = bool(not raw["date"].is_monotonic_increasing)
    q["dup_timestamps"] = int(raw["date"].duplicated().sum())
    df = (raw.sort_values("date", kind="mergesort")
             .drop_duplicates("date", keep="last").reset_index(drop=True))
    del raw

    # -- field-level checks ---------------------------------------------------
    for c in PRICE_COLS + ["volume", "money", "open_interest", "symbol"]:
        q[f"nan_{c}"] = int(df[c].isna().sum())
    q["nonpos_price"] = int((df[PRICE_COLS] <= 0).any(axis=1).sum())
    hi_bad = df["high"] < df[["open", "close", "low"]].max(axis=1)
    lo_bad = df["low"] > df[["open", "close", "high"]].min(axis=1)
    q["ohlc_violations"] = int((hi_bad | lo_bad).sum())
    q["neg_volume"] = int((df["volume"] < 0).sum())
    q["zero_vol_pct"] = round(float((df["volume"] == 0).mean() * 100), 3)
    flat = (df["open"] == df["high"]) & (df["high"] == df["low"]) & (df["low"] == df["close"])
    q["stale_bar_pct"] = round(float((flat & (df["volume"] == 0)).mean() * 100), 3)
    q["zero_oi_bars"] = int((df["open_interest"] <= 0).sum())
    v = df["volume"] > 0
    q["implied_mult"] = float((df.loc[v, "money"] / (df.loc[v, "volume"] * df.loc[v, "close"])).median()) if v.any() else np.nan
    q["ref_mult"] = MULT.get(code, np.nan)

    # -- sessions & trading day ----------------------------------------------
    hour = df["date"].dt.hour
    night = (hour >= 20) | (hour < 4)
    td = pd.Series(session_day(df["date"]), index=df.index)        # calendar rule (Step 0c): no backfill from the next bar
    day_days = set(td[~night])
    keep = td.isin(day_days)                                         # drop night bars whose session has no day bars (holiday mismatch or the unfinished last session)
    q["orphan_night_bars"] = int((~keep).sum())
    df["trading_day"] = td
    df["night"] = night.to_numpy()
    df = df[keep.to_numpy()].reset_index(drop=True)
    q["night_bar_pct"] = round(float(df["night"].mean() * 100), 2)
    q["night_since"] = df.loc[df["night"], "trading_day"].min() if df["night"].any() else pd.NaT

    # -- rolls & clean returns -----------------------------------------------
    lr = np.log(df["close"].where(df["close"] > 0)).diff()
    roll = df["symbol"].ne(df["symbol"].shift())
    roll.iloc[0] = False
    df["roll"] = roll.to_numpy()
    q["n_symbols"] = int(df["symbol"].nunique())
    q["n_rolls"] = int(roll.sum())
    q["symbol_flipflops"] = int(roll.sum() - (df["symbol"].nunique() - 1))
    q["roll_jump_med_abs_pct"] = round(float(lr[roll].abs().median() * 100), 3) if roll.any() else np.nan
    df["ret"] = lr.where(~roll)
    df["roll_gap"] = lr.where(roll)          # calendar-spread jump at the roll (new minus old contract): + = contango, - = backwardation (carry proxy, Step 18)
    r = df["ret"].dropna()
    lim = LIMIT.get(code, 10) / 100
    q["min_ret_gt_limit"] = int((r.abs() > lim).sum())
    q["min_ret_gt_2pct"] = int((r.abs() > 0.02).sum())
    q["min_ret_std_bp"] = round(float(r.std() * 1e4), 2)
    q["min_ret_skew"] = round(float(r.skew()), 2)
    q["min_ret_kurt"] = round(float(r.kurt()), 1)
    q["min_ret_ac1"] = round(float(r.autocorr(1)), 4)

    # -- daily aggregation ---------------------------------------------------
    df["ret2"] = df["ret"] ** 2
    g = df.groupby("trading_day", sort=True)
    daily = pd.DataFrame({
        "open": g["open"].first(), "high": g["high"].max(), "low": g["low"].min(), "close": g["close"].last(),
        "volume": g["volume"].sum(), "money": g["money"].sum(), "oi": g["open_interest"].last(),
        "symbol": g["symbol"].last(), "n_bars": g.size(), "ret": g["ret"].sum(min_count=1),
        "rv": np.sqrt(g["ret2"].sum(min_count=1)), "roll": g["roll"].any(), "roll_gap": g["roll_gap"].sum(min_count=1), "n_roll": g["roll"].sum(),
    })
    day = df[~df["night"]]
    daily["ret_night"] = df[df["night"]].groupby("trading_day")["ret"].sum(min_count=1)
    daily["ret_day"] = day.groupby("trading_day")["ret"].sum(min_count=1)
    daily["ret_day_open"] = day.drop_duplicates("trading_day").set_index("trading_day")["ret"]
    daily["ret_day_ex_open"] = daily["ret_day"] - daily["ret_day_open"].fillna(0)
    # overnight / opening-move paths (Steps 13d and 23c). All returns are sums of clean 1-min returns, so a roll never enters them.
    #   ret_open{k}: cumulative return of the first k day-session bars, counted from the previous bar (the first bar carries the overnight gap)
    #   ret_pre{m}:  return of the last m day-session bars before the close (an entry "before the end of the day")
    #   vol_open15 / vol_pre30: volume in the first 15 / last 30 day-session bars
    dr = day.assign(r0=day["ret"].fillna(0.0)); gd = dr.groupby("trading_day", sort=True)
    kf = gd.cumcount(); kb = gd.cumcount(ascending=False)
    for k in OPEN_K:
        daily[f"ret_open{k}"] = dr["r0"].where(kf < k).groupby(dr["trading_day"]).sum()
    for m in PRE_M:
        daily[f"ret_pre{m}"] = dr["r0"].where(kb < m).groupby(dr["trading_day"]).sum()
    daily["vol_open15"] = dr["volume"].where(kf < 15).groupby(dr["trading_day"]).sum()
    daily["vol_pre30"] = dr["volume"].where(kb < 30).groupby(dr["trading_day"]).sum()
    # night / day / opening-window realised volatility (Steps 11d and 13e): sums of squared clean 1-min returns, so a roll never enters
    daily["rv_night"] = np.sqrt(df[df["night"]].groupby("trading_day")["ret2"].sum(min_count=1))
    daily["rv_day"] = np.sqrt(day.groupby("trading_day")["ret2"].sum(min_count=1))
    daily["rv_open30"] = np.sqrt(dr["ret2"].where((kf >= 1) & (kf < 30)).groupby(dr["trading_day"]).sum(min_count=1))        # bars 2..30; the first bar carries the gap
    daily["n_day_bars"] = day.groupby("trading_day").size()
    # last PRE_BLK five-minute blocks before the day-session close (Step 23e): variance and return of each block; block 1 = the last 5 trading minutes
    sub_ = dr[kb < 5 * PRE_BLK].assign(blk=(kb[kb < 5 * PRE_BLK] // 5) + 1)
    g2 = sub_.groupby(["trading_day", "blk"]).agg(rv2=("ret2", "sum"), r=("r0", "sum"), n=("r0", "size"))
    g2.loc[g2["n"] < 5, ["rv2", "r"]] = np.nan
    u_rv, u_r = g2["rv2"].unstack(), g2["r"].unstack()
    for j_ in range(1, PRE_BLK + 1):
        daily[f"pre_rv5_{j_}"] = u_rv[j_].reindex(daily.index) if j_ in u_rv.columns else np.nan
        daily[f"pre_ret5_{j_}"] = u_r[j_].reindex(daily.index) if j_ in u_r.columns else np.nan
    daily["product"] = code

    bpd = daily["n_bars"]
    q["start"], q["end"] = daily.index.min(), daily.index.max()
    q["n_days"] = len(daily)
    q["bars_per_day_med"] = float(bpd.median())
    q["short_day_pct"] = round(float((bpd < 0.5 * bpd.median()).mean() * 100), 2)
    gaps = daily.index.to_series().diff().dt.days
    q["max_gap_days"] = float(gaps.max())
    q["gaps_gt_10d"] = int((gaps > 10).sum())
    q["daily_ret_gt_limit"] = int((daily["ret"].abs() > lim).sum())

    # -- intraday profile (recent) ------------------------------------------
    rec = df[df["trading_day"] >= RECENT_START]
    prof = None
    if len(rec):
        mod = rec["date"].dt.hour * 60 + rec["date"].dt.minute
        dvol = rec.groupby("trading_day")["volume"].transform("sum").replace(0, np.nan)   # causal-ok: EDA intraday volume profile, descriptive only (never a feature)
        prof = (pd.DataFrame({"mod": mod.to_numpy(), "vshare": (rec["volume"] / dvol).to_numpy(),
                              "absret": rec["ret"].abs().to_numpy()})
                  .groupby("mod").agg(vshare=("vshare", "mean"), absret=("absret", "mean"),
                                      n=("vshare", "size")))
        prof = prof[prof["n"] >= 0.2 * prof["n"].max()].reset_index()
        prof["product"] = code

    # -- 5-min bars for multi-time-frame work (return, realised variance, volume, OI) --
    r5 = b5 = None
    ri = df[df["trading_day"] >= INTRADAY_START]
    if (INTRADAY_SET is None or code in INTRADAY_SET or code in LEADLAG_SET) and len(ri):
        g5 = ri.set_index("date").resample("5min", label="right", closed="right")
        b5 = pd.DataFrame({"ret": g5["ret"].sum(min_count=1), "rv2": g5["ret2"].sum(min_count=1),
                           "volume": g5["volume"].sum(min_count=1), "oi": g5["open_interest"].last(),
                           "hi": g5["high"].max(), "lo": g5["low"].min(), "cl": g5["close"].last(), "op": g5["open"].first(),
                           "money": g5["money"].sum(min_count=1), "rollb": g5["roll"].max()}).dropna(subset=["ret"])
        b5 = b5.astype("float32"); b5["product"] = code
        if code in LEADLAG_SET:
            r5 = b5["ret"].astype(float).rename(code)
        if not (INTRADAY_SET is None or code in INTRADAY_SET):
            b5 = None
    del df, rec, ri
    return q, daily, prof, r5, b5


# ---- smoke test: the whole ingestion function on a built-in sample of a real file (PF9999.XZCE, 1290 one-minute rows: night + day sessions,
# a contract roll PF2105 -> PF2107, duplicated timestamps, flat zero-volume night bars). It fails here, in seconds, if a column or a rule changes. ----
import gzip, base64, tempfile
SAMPLE_PF_B64 = (
    "..."
    "miOuaXCSO8xSUdho+OtOlfw//AexZ/aRAQA="
)
def smoke_test_ingestion():
    tmp = Path(tempfile.mkdtemp()) / "PF9999.XZCE.csv"; tmp.write_bytes(gzip.decompress(base64.b64decode(SAMPLE_PF_B64)))
    f0, e0 = dict(FILES), dict(EXCH)
    try:
        FILES["PF"], EXCH["PF"] = tmp, "XZCE"
        q, daily, prof, r5, b5 = process_product("PF")
    finally:
        FILES.clear(); FILES.update(f0); EXCH.clear(); EXCH.update(e0)
    need = ["open", "high", "low", "close", "volume", "money", "oi", "symbol", "ret", "rv", "roll", "roll_gap", "n_bars", "ret_night", "ret_day", "ret_open30", "ret_pre15", "vol_open15", "n_roll", "rv_night", "rv_day", "rv_open30", "n_day_bars", "pre_rv5_1", "pre_ret5_1"]
    miss = [c for c in need if c not in daily.columns]
    assert not miss, f"daily panel is missing columns {miss}"
    assert b5 is None or {"ret", "rv2", "volume", "oi", "hi", "lo", "cl", "money", "rollb"} <= set(b5.columns), "5-minute bars are missing columns"
    assert q["n_rolls"] >= 1 and q["dup_timestamps"] > 0, "the sample should contain a roll and duplicated timestamps"
    return q, daily
SMOKE_Q, SMOKE_D = smoke_test_ingestion()
print(f"smoke test on the built-in sample passed: {len(SMOKE_D)} trading days, {SMOKE_Q['n_rolls']} roll, {SMOKE_Q['dup_timestamps']} duplicated timestamps removed, "
      f"{SMOKE_Q['zero_vol_pct']}% zero-volume bars, all required daily columns present")

paths = {k: OUT_DIR / f for k, f in [("daily", "daily_panel.parquet"), ("q", "quality_raw.csv"),
                                      ("prof", "intraday_profiles.parquet"), ("r5", "ret5m.parquet"),
                                      ("b5", "bars5m.parquet")]}
todo = sorted(FILES) if UNIVERSE is None else [c for c in UNIVERSE if c in FILES]

def _cache_current():
    # an older cache without the opening-move or night / day variance columns is rebuilt automatically
    try:
        import pyarrow.parquet as pq
        return {"ret_open30", "ret_pre15", "n_roll", "rv_night", "rv_open30", "pre_rv5_1"} <= set(pq.ParquetFile(paths["daily"]).schema.names) and {"money", "rollb"} <= set(pq.ParquetFile(paths["b5"]).schema.names)
    except Exception:
        return False

if FORCE_REBUILD or not all(p.exists() for p in paths.values()) or not _cache_current():
    qs, dailies, profs, r5s, b5s = [], [], [], [], []
    t0 = time.time()
    for i, code in enumerate(todo, 1):
        try:
            q, d, pr, r5, b5 = process_product(code)
            qs.append(q); dailies.append(d)
            if pr is not None: profs.append(pr)
            if r5 is not None: r5s.append(r5)
            if b5 is not None: b5s.append(b5)
            print(f"[{i:>2}/{len(todo)}] {code:<3} rows={q['rows_raw']:>9,} days={q['n_days']:>5} "
                  f"rolls={q['n_rolls']:>3}  {time.time()-t0:6.0f}s")
        except Exception as e:
            print(f"[{i:>2}/{len(todo)}] {code}: FAILED -> {e!r}")
        gc.collect()
    quality_raw = pd.DataFrame(qs)
    panel = pd.concat(dailies).reset_index()
    profiles = pd.concat(profs, ignore_index=True) if profs else pd.DataFrame()
    ret5 = pd.concat(r5s, axis=1) if r5s else pd.DataFrame()
    quality_raw.to_csv(paths["q"], index=False)
    panel.to_parquet(paths["daily"], index=False)
    (OUT_DIR / "daily_panel_raw.parquet").unlink(missing_ok=True)          # Step 2f keeps an untreated copy; a fresh ingestion invalidates it
    profiles.to_parquet(paths["prof"], index=False)
    ret5.to_parquet(paths["r5"])
    (pd.concat(b5s).reset_index() if b5s else pd.DataFrame(columns=["date", "ret", "rv2", "volume", "oi", "product"])).to_parquet(paths["b5"], index=False)
    del b5s
    print(f"Done in {time.time()-t0:.0f}s; cache written to {OUT_DIR}")
else:
    print("Cache found — loading (set FORCE_REBUILD=True to rebuild)")

In [ ]:
# step 1
# Load cache (every later step starts from here)
quality_raw = pd.read_csv(paths["q"], parse_dates=["start", "end", "night_since"])
panel = pd.read_parquet(paths["daily"]).sort_values(["product", "trading_day"]).reset_index(drop=True)
profiles = pd.read_parquet(paths["prof"])
ret5 = pd.read_parquet(paths["r5"])
panel["sector"] = panel["product"].map(SECTOR).fillna("Other")
PRODUCTS = sorted(panel["product"].unique())
FOCUS = [c for c in FOCUS if c in PRODUCTS]
print(f"panel: {panel.shape}, products={len(PRODUCTS)}, "
      f"{panel.trading_day.min().date()} -> {panel.trading_day.max().date()}")
assert not panel.duplicated(["product", "trading_day"]).any(), "daily panel must be unique per (product, day)"
panel.head()

In [ ]:
# step 1z
# Pack the cache into ONE zip (Kaggle: after Save Version, Output tab > New Dataset creates a dataset from it with no download).
# Attach that Dataset to later runs: the notebook copies it in and Step 1 loads it (no rebuild). Re-run this cell at the end of the notebook to refresh the zip with the later-step files.
import zipfile
ZIP_P = (OUT_DIR.parent / "eda_cache.zip") if OUT_DIR.parent.exists() else Path("eda_cache.zip")
with zipfile.ZipFile(ZIP_P, "w", zipfile.ZIP_STORED) as _z:                 # parquet is already compressed
    for _f in sorted(OUT_DIR.iterdir()):
        if _f.is_file(): _z.write(_f, _f.name)
print(f"{ZIP_P}  {ZIP_P.stat().st_size/1e6:.0f} MB  ({len(list(OUT_DIR.iterdir()))} files)")
print("Kaggle serves no direct link to /kaggle/working. To keep this cache as an input: Save Version (Save & Run All), open the saved version, Output tab, then 'New Dataset' (no download needed). Or: right panel > Output > eda_cache.zip > three dots > Download.")

## Step 2 — Data quality checks
Each check becomes a flag. Thresholds are deliberately loose; the point is to list what needs a decision before modelling.

**Used later.** Quality flags decide which products and periods enter the universe (Step 9) and which bars are dropped (session opens, roll days, constant series).

In [ ]:
# step 2
qr = quality_raw.copy()
qr["mult_ratio"] = qr["implied_mult"] / qr["ref_mult"]

flags = {
    "duplicate timestamps":         qr["dup_timestamps"] > 0,
    "unparseable timestamps":       qr["bad_timestamps"] > 0,
    "missing values":               qr.filter(like="nan_").sum(axis=1) > 0,
    "non-positive prices":          qr["nonpos_price"] > 0,
    "OHLC inconsistent bars":       qr["ohlc_violations"] > 0,
    "negative volume":              qr["neg_volume"] > 0,
    "zero-volume bars > 5%":        qr["zero_vol_pct"] > 5,
    "stale (flat, 0-vol) bars > 2%":qr["stale_bar_pct"] > 2,
    "contract flip-flops":          qr["symbol_flipflops"] > 0,
    "1-min move > daily limit":     qr["min_ret_gt_limit"] > 0,
    "daily move > current limit":   qr["daily_ret_gt_limit"] > 0,
    "short days > 5%":              qr["short_day_pct"] > 5,
    "calendar gaps > 10 days":      qr["gaps_gt_10d"] > 0,
    "night bars without day session": qr["orphan_night_bars"] > 0,
    "turnover/multiplier mismatch": (qr["mult_ratio"] - 1).abs() > 0.25,
    "ended before 2021":            qr["end"] < "2021-01-01",
}
flag_df = pd.DataFrame(flags).astype(int)
flag_df.index = qr["product"]
summary = pd.DataFrame({"products_flagged": flag_df.sum(),
                        "examples": flag_df.apply(lambda s: ", ".join(s[s == 1].index[:8]))})
display(summary.sort_values("products_flagged", ascending=False))

quality_report = qr.set_index("product").join(flag_df.add_prefix("flag_"))
quality_report["n_flags"] = flag_df.sum(axis=1)
quality_report.to_csv(OUT_DIR / "quality_report.csv")
display(quality_report[["exchange", "sector", "start", "end", "n_days", "bars_per_day_med", "n_rolls",
                        "roll_jump_med_abs_pct", "zero_vol_pct", "ohlc_violations", "min_ret_gt_limit",
                        "mult_ratio", "n_flags"]].sort_values("n_flags", ascending=False))

In [ ]:
# 2b. Coverage timeline — which products exist when (and legacy codes that should be stitched)
cov = quality_report.sort_values("start")
fig, ax = plt.subplots(figsize=(14, max(6, 0.22 * len(cov))))
for i, (p, row) in enumerate(cov.iterrows()):
    color = "tab:red" if p in LEGACY else "tab:blue"
    ax.plot([row["start"], row["end"]], [i, i], lw=5, color=color, solid_capstyle="butt")
    if pd.notna(row["night_since"]):
        ax.plot([row["night_since"], row["end"]], [i, i], lw=2, color="orange")
ax.set_yticks(range(len(cov))); ax.set_yticklabels(cov.index, fontsize=7)
ax.set_title("Coverage per product (red = legacy code, orange = night session available)")
plt.tight_layout(); plt.show()

print("Legacy code pairs to stitch:")
for old, new in LEGACY.items():
    if old in quality_report.index and new in quality_report.index:
        print(f"  {old}: {quality_report.loc[old,'start'].date()}→{quality_report.loc[old,'end'].date()}   "
              f"{new}: {quality_report.loc[new,'start'].date()}→{quality_report.loc[new,'end'].date()}")

In [ ]:
# 2c. Missing trading days vs the exchange calendar (union of days traded on that exchange)
ex_days = panel.assign(exchange=panel["product"].map(lambda c: EXCH_NAME.get(EXCH[c], EXCH[c]))) \
               .groupby("exchange")["trading_day"].apply(lambda s: pd.DatetimeIndex(s.unique()))
rows = []
for p, d in panel.groupby("product"):
    ex = EXCH_NAME.get(EXCH[p], EXCH[p])
    cal = ex_days[ex]
    cal = cal[(cal >= d.trading_day.min()) & (cal <= d.trading_day.max())]
    miss = cal.difference(pd.DatetimeIndex(d.trading_day))
    rows.append({"product": p, "exchange": ex, "expected_days": len(cal),
                 "missing_days": len(miss), "missing_pct": round(100 * len(miss) / max(len(cal), 1), 2)})
missing_days = pd.DataFrame(rows).sort_values("missing_pct", ascending=False)
display(missing_days.head(15))

# 2d. Roll behaviour: how big the splice jump is, and bars-per-day regime changes
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
quality_report["roll_jump_med_abs_pct"].sort_values().plot.barh(ax=axes[0], fontsize=6)
axes[0].set_title("Median |price jump| at roll (%) — removed from returns")
(quality_report["mult_ratio"].dropna().sort_values()).plot.barh(ax=axes[1], fontsize=6)
axes[1].axvline(1, color="k", ls="--"); axes[1].set_title("Implied / reference multiplier (1 = consistent)")
plt.tight_layout(); plt.show()

In [ ]:
# 2e. Inspect the worst 1-min and daily outliers (are they data errors or real limit moves?)
out = panel.loc[panel["ret"].abs() > panel["product"].map(lambda c: LIMIT.get(c, 10) / 100),
                ["product", "trading_day", "symbol", "ret", "rv", "volume", "roll"]]
print(f"{len(out)} product-days with |daily return| above today's limit")
display(out.reindex(out["ret"].abs().sort_values(ascending=False).index).head(20))

**Decisions to record after reading Step 2** (fill in):
- Products to exclude (too short, illiquid, broken turnover): …
- Legacy codes to stitch: …
- Outlier policy (winsorize daily returns at k·σ? drop days?): …
- Start date for the modelling sample (night session/liquidity regime): …

## Step 2f — Data-quality treatment: one rule for every flag

Step 2 lists what is wrong. A researcher then decides, flag by flag, **what is an error (remove it), what is real (keep it, and say how it is handled), and what only needs a different estimator**. Every rule below uses only data up to the day it is applied (no look-ahead), is idempotent, and leaves the untreated panel in `daily_panel_raw.parquet`.

| Flag | What it usually is | Treatment | Why |
|---|---|---|---|
| duplicate timestamps | export artefact | already removed in Step 1 (last row kept); count reported | a duplicated minute would double-count volume |
| calendar gaps > 10 days | missing data, not a holiday (Chinese New Year / National Day are ≤ 10 days) | the first return after a gap is set to missing (it is a multi-week return, not a daily one) | a gap return would be a fake outlier and would shift every rolling window |
| daily move > current limit | real, from the years when limits were wider, or an expanded limit after a holiday | **kept**; vol-scaled returns are clipped at `CLIP_Z` later; days beyond `DQ_IMPOSSIBLE_X` × limit are errors → missing | a move 3× the exchange limit is a roll or a bad tick, a move slightly above is history |
| 1-min move > daily limit | the first bar after a night session or holiday carries the whole gap | **kept** (it is the gap), clipped in vol-scaled units; not cut | cutting it would delete real overnight risk |
| stale (flat, zero-volume) bars > 2 %, zero-volume bars > 5 % | illiquid hours or products | zero-volume 5-minute bars give a *missing* return in every intraday frame; a day with zero volume and zero return is a *no-trade day* (return missing) | a zero return from a market that did not trade is not information, and it manufactures autocorrelation in the intraday tests |
| contract flip-flops | the main-contract label flips back and forth | every symbol change already nulls that bar's return (Step 1); on days with two or more changes the roll gap (carry proxy) is set to missing | the carry proxy of a flipping day is noise |
| short days > 5 % | half sessions, first and last day, partial exports | volume, turnover, realised vol and opening-move columns are missing on days with fewer than `DQ_SHORT_FRAC` × the product's *expanding median* number of bars (the return is kept) | a partial day understates volume and variance; the expanding median is known at the time |
| ended before 2021 | delisted or legacy codes | **kept** until their last day (point-in-time universe, no survivorship); legacy duplicates stay in `EXCLUDE` | dropping them would keep only survivors |
| turnover / multiplier mismatch | wrong reference multiplier in the table | traded value is always taken from `money`; the implied multiplier replaces the reference value (`mult_used`) | a wrong multiplier would misstate contract value and capacity |

**Used later.** `panel` (treated) feeds Steps 3–26. The ledger table is reported in the findings summary (Step 26c) so that every result can be read against the data issues it was built on.

In [ ]:
# step 2f
DQ_GAP_DAYS, DQ_SHORT_FRAC, DQ_IMPOSSIBLE_X, DQ_MIN_HIST = 10, 0.5, 3.0, 20
raw_path = OUT_DIR / "daily_panel_raw.parquet"
if raw_path.exists():
    P0 = pd.read_parquet(raw_path)
else:
    P0 = pd.read_parquet(paths["daily"]); P0.to_parquet(raw_path, index=False)
P0 = P0.sort_values(["product", "trading_day"]).reset_index(drop=True)
P1 = P0.copy(); P1["sector"] = P1["product"].map(SECTOR).fillna("Other")
RETC = [c for c in P1.columns if c == "ret" or c.startswith("ret_")]
OPENC = [c for c in P1.columns if c.startswith(("ret_open", "ret_pre", "vol_open", "vol_pre", "rv_night", "rv_day", "rv_open", "pre_rv5_", "pre_ret5_"))]
LIMV = P1["product"].map(lambda c: LIMIT.get(c, 10) / 100.0)
n_all = len(P1); led = []
def note(issue, mask, cols, rule, how):
    k = int(mask.sum()); P1.loc[mask, cols] = np.nan
    led.append(dict(issue=issue, rule=rule, rows_affected=k, share_of_rows=k / n_all, products=int(P1.loc[mask, "product"].nunique()) if k else 0, applied=how))

# 1. calendar gaps: the first return after a gap longer than DQ_GAP_DAYS
gap = P1.groupby("product")["trading_day"].diff().dt.days > DQ_GAP_DAYS
note("calendar gaps > 10 days", gap, RETC + ["roll_gap"], f"first return after a gap > {DQ_GAP_DAYS} days → missing", "removed")
# 2. impossible daily moves (a roll residual or a bad tick), real limit moves are kept
imp = P1["ret"].abs() > DQ_IMPOSSIBLE_X * LIMV
note("daily move > current limit", imp, RETC + ["roll_gap"], f"|daily return| > {DQ_IMPOSSIBLE_X:g} × exchange limit → missing; moves between 1× and {DQ_IMPOSSIBLE_X:g}× are kept", "partly removed")
led.append(dict(issue="… of which kept (real, between 1x and 3x the limit)", rule="kept; clipped in vol-scaled units (CLIP_Z)", rows_affected=int((P1["ret"].abs() > LIMV).sum()), share_of_rows=float((P1["ret"].abs() > LIMV).mean()),
                products=int(P1.loc[P1["ret"].abs() > LIMV, "product"].nunique()), applied="kept"))
# 3. no-trade days: zero volume and an exactly zero return
nt = (P1["volume"] == 0) & (P1["ret"] == 0)
note("stale (flat, 0-vol) bars > 2%  /  zero-volume bars > 5%", nt, RETC, "day with zero volume and zero return → no-trade day, returns missing (5-minute zero-volume bars are masked when the intraday frames are loaded)", "removed")
# 4. flip-flop days: two or more symbol changes inside one trading day
ff = P1["n_roll"].fillna(0) >= 2
note("contract flip-flops", ff, ["roll_gap"], "two or more symbol changes in one day → carry proxy (roll gap) missing; each change already nulled its bar return", "partly removed")
# 5. short days: fewer bars than DQ_SHORT_FRAC × the expanding median known the day before
med = P1.groupby("product")["n_bars"].transform(lambda s: s.expanding(DQ_MIN_HIST).median().shift(1))
sh_ = P1["n_bars"] < DQ_SHORT_FRAC * med
note("short days > 5%", sh_, ["volume", "money", "rv"] + OPENC, f"bars < {DQ_SHORT_FRAC:g} × expanding median → volume, turnover, realised vol and opening-move columns missing (return kept)", "partly removed")
# 6. documented, nothing removed
dup = int(quality_raw["dup_timestamps"].sum()); n1 = int((quality_raw["min_ret_gt_limit"] > 0).sum())
led.append(dict(issue="duplicate timestamps", rule="last row of a duplicated minute kept in Step 1", rows_affected=dup, share_of_rows=np.nan, products=int((quality_raw["dup_timestamps"] > 0).sum()), applied="removed in Step 1"))
led.append(dict(issue="1-min move > daily limit", rule="kept (the first bar after a night session or holiday carries the gap); clipped in vol-scaled units", rows_affected=int(quality_raw["min_ret_gt_limit"].sum()), share_of_rows=np.nan, products=n1, applied="kept"))
dead = quality_raw[quality_raw["end"] < "2021-01-01"]["product"].tolist()
led.append(dict(issue="ended before 2021", rule=f"kept until the last day (point-in-time, no survivorship): {', '.join(dead) if dead else 'none'}", rows_affected=0, share_of_rows=np.nan, products=len(dead), applied="kept"))
mm = quality_report.index[((quality_report["mult_ratio"] - 1).abs() > 0.25)].tolist()
quality_report["mult_used"] = np.where((quality_report["mult_ratio"] - 1).abs() > 0.25, quality_report["implied_mult"], quality_report["ref_mult"])
led.append(dict(issue="turnover/multiplier mismatch", rule=f"traded value always from `money`; implied multiplier used for {', '.join(mm) if mm else 'none'}", rows_affected=0, share_of_rows=np.nan, products=len(mm), applied="replaced"))
DQ_LEDGER = pd.DataFrame(led)
DQ_LEDGER.to_csv(OUT2 / "data_quality_treatment.csv", index=False)

# effect on the data
def tail_stats(x):
    x = x.dropna(); z = x / x.groupby(P1.loc[x.index, "product"]).transform("std")
    return dict(n=len(x), kurtosis=float(x.kurt()), share_beyond_8_sd=float((z.abs() > 8).mean()))
eff = pd.DataFrame({"before": tail_stats(P0["ret"]), "after": tail_stats(P1["ret"])}).T
print("Treatment ledger (every flag of Step 2 has a rule):")
print(DQ_LEDGER[["issue", "applied", "rows_affected", "share_of_rows", "products", "rule"]].assign(share_of_rows=lambda d: (d.share_of_rows * 100).round(3)).to_string(index=False, max_colwidth=75))
print("\nDaily-return tails before / after:\n", eff.round(4).to_string())
panel = P1.copy(); panel.to_parquet(paths["daily"], index=False)           # every later step (and the cached Part B reader) sees the treated panel
fig, ax = plt.subplots(figsize=(9, 3.6)); d_ = DQ_LEDGER[DQ_LEDGER.rows_affected > 0].sort_values("rows_affected")
ax.barh(d_.issue.str.slice(0, 55), d_.rows_affected, color=np.where(d_.applied.str.contains("kept"), "#b9b8b2", "#2a78d6")); ax.set_xscale("log")
ax.set_title("Rows treated per data-quality flag (blue = changed, grey = kept as real)"); ax.set_xlabel("product-days / bars"); plt.tight_layout(); plt.show()

## Step 3 — Basic statistics and return distributions
Daily log returns (close-to-close by trading day, roll-adjusted) plus 1-min statistics from Step 1.

**Used later.** The return distributions justify the fat-tailed models of Steps 11–13 and the use of log returns in the relationship tests. Simple returns are used for P&L in Steps 21–25.

In [ ]:
# step 3
def ret_stats(r):
    r = r.dropna()
    if len(r) < 60:
        return pd.Series(dtype=float)
    jb = stats.jarque_bera(r)
    return pd.Series({"n_days": len(r), "ann_ret_%": r.mean() * ANN * 100, "ann_vol_%": r.std() * np.sqrt(ANN) * 100,
                      "sharpe": r.mean() / r.std() * np.sqrt(ANN), "skew": r.skew(), "ex_kurt": r.kurt(),
                      "jb_p": jb.pvalue, "q01_%": r.quantile(.01) * 100, "q99_%": r.quantile(.99) * 100,
                      "max_abs_%": r.abs().max() * 100, "ac1": r.autocorr(1),
                      "pct_days_gt_3sd": (r.abs() > 3 * r.std()).mean() * 100})

rstats = panel.groupby("product")["ret"].apply(ret_stats).unstack()
rstats = rstats.join(quality_report[["sector", "min_ret_std_bp", "min_ret_kurt", "min_ret_ac1"]])
recent = panel[panel.trading_day >= RECENT_START].groupby("product")["ret"].apply(ret_stats).unstack()
rstats["ann_vol_recent_%"] = recent["ann_vol_%"]
rstats.to_csv(OUT_DIR / "return_stats.csv")
display(rstats.round(3).sort_values("ann_vol_%", ascending=False))
print("Normal: ex_kurt=0, 3σ days=0.27%. "
      f"Median ex_kurt={rstats['ex_kurt'].median():.1f}, median 3σ frequency={rstats['pct_days_gt_3sd'].median():.2f}%")

In [ ]:
# 3b. Cross-section: vol and tail-fatness by product, coloured by sector
pal = dict(zip(sorted(rstats["sector"].unique()), sns.color_palette("tab20", rstats["sector"].nunique())))
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, col, title in [(axes[0], "ann_vol_%", "Annualised volatility (%)"), (axes[1], "ex_kurt", "Excess kurtosis (daily)")]:
    s = rstats.sort_values(col)
    ax.bar(s.index, s[col], color=[pal[x] for x in s["sector"]])
    ax.set_title(title); ax.tick_params(axis="x", rotation=90, labelsize=7)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in pal.values()]
axes[1].legend(handles, pal.keys(), fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

# 3c. Distribution shape for the focus set: standardised returns vs N(0,1), plus QQ
n = len(FOCUS); ncol = 5; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(18, 3.2 * nrow)); axes = np.atleast_1d(axes).ravel()
x = np.linspace(-6, 6, 200)
for ax, c in zip(axes, FOCUS):
    z = panel.loc[panel["product"] == c, "ret"].dropna()
    z = (z - z.mean()) / z.std()   # causal-ok: EDA chart scaling, descriptive only
    ax.hist(z.clip(-6, 6), bins=120, density=True, alpha=.6)
    ax.plot(x, stats.norm.pdf(x), "k--", lw=1); ax.set_yscale("log"); ax.set_ylim(1e-4, 1)
    ax.set_title(f"{c}  kurt={z.kurt():.1f}  skew={z.skew():.2f}", fontsize=9)
for ax in axes[n:]: ax.axis("off")
plt.suptitle("Standardised daily returns (log density) vs normal"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(nrow, ncol, figsize=(18, 3.2 * nrow)); axes = np.atleast_1d(axes).ravel()
for ax, c in zip(axes, FOCUS):
    stats.probplot(panel.loc[panel["product"] == c, "ret"].dropna(), dist="norm", plot=ax)
    ax.set_title(c, fontsize=9); ax.set_xlabel(""); ax.set_ylabel("")
for ax in axes[n:]: ax.axis("off")
plt.suptitle("QQ plots vs normal"); plt.tight_layout(); plt.show()

In [ ]:
# 3d. Return decomposition: night session vs day session vs the opening gap (products with night trading)
dec = panel.dropna(subset=["ret_night"]).groupby("product")[["ret", "ret_night", "ret_day_open", "ret_day_ex_open"]] \
           .agg(lambda s: s.std() * np.sqrt(ANN) * 100)
dec.columns = ["total_vol", "night_vol", "day_open_vol", "day_ex_open_vol"]
dec["night_var_share_%"] = (dec["night_vol"] ** 2 / dec["total_vol"] ** 2 * 100)
display(dec.round(2).sort_values("night_var_share_%", ascending=False))

### 3e — Log returns vs simple returns
Everything above uses **log returns** $r=\ln(P_t/P_{t-1})$: they add up over time, so a daily return is the sum of 1-min returns and a roll-adjusted price is a cumulative sum. **Simple returns** $R=e^{r}-1$ add up *across assets*, so portfolio P&L and the strategy backtests (Step 23) use them.

The two differ by about $\sigma^2/2$ per period (the Jensen / volatility drag): $E[R]\approx E[r]+\tfrac12\sigma^2$. Simple returns are also more right-skewed, because $e^r$ is convex. The table shows how much the choice matters per product.

In [ ]:
# step 3e
simple = panel.assign(R=np.expm1(panel["ret"]))
lr_vs_sr = pd.DataFrame({
    "mean_log_bp": simple.groupby("product")["ret"].mean() * 1e4,
    "mean_simple_bp": simple.groupby("product")["R"].mean() * 1e4,
    "half_var_bp": simple.groupby("product")["ret"].var() / 2 * 1e4,
    "skew_log": simple.groupby("product")["ret"].skew(), "skew_simple": simple.groupby("product")["R"].skew(),
    "kurt_log": simple.groupby("product")["ret"].apply(lambda x: x.kurt()),
    "kurt_simple": simple.groupby("product")["R"].apply(lambda x: x.kurt()),
    "geo_ann_%": simple.groupby("product")["ret"].mean() * ANN * 100,          # log mean = geometric growth
    "arith_ann_%": simple.groupby("product")["R"].mean() * ANN * 100,
})
lr_vs_sr["drag_check_bp"] = lr_vs_sr["mean_simple_bp"] - lr_vs_sr["mean_log_bp"] - lr_vs_sr["half_var_bp"]  # ≈ 0
lr_vs_sr.to_csv(OUT_DIR / "log_vs_simple.csv")
display(lr_vs_sr.round(3).sort_values("half_var_bp", ascending=False).head(20))

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
s = lr_vs_sr.sort_values("half_var_bp")
axes[0].bar(s.index, s["arith_ann_%"] - s["geo_ann_%"], color="#2a78d6")
axes[0].set_title("Volatility drag: arithmetic (simple) minus geometric (log) annual return, %")
axes[0].tick_params(axis="x", rotation=90, labelsize=7)
axes[1].scatter(lr_vs_sr["skew_log"], lr_vs_sr["skew_simple"], color="#2a78d6", s=20)
lim = np.nanmax(np.abs(lr_vs_sr[["skew_log", "skew_simple"]].values)) * 1.1
axes[1].plot([-lim, lim], [-lim, lim], color="#52514e", lw=1, ls="--")
axes[1].set_xlabel("skew of log returns"); axes[1].set_ylabel("skew of simple returns")
axes[1].set_title("Simple returns are more right-skewed (points above the diagonal)")
plt.tight_layout(); plt.show()

## Step 4 — Price, return, volatility, volume and open-interest analysis

**Used later.** Volatility, volume and OI profiles set the volatility-scaling window (Step 9), the time-of-day buckets (Steps 13 and 23a) and the liquidity ranks (Steps 17 and 24).

In [ ]:
# 4a. Deep dive: back-adjusted vs raw price, volume, OI for one product
panel["logp_adj"] = panel.groupby("product")["ret"].transform(lambda s: s.fillna(0).cumsum())
d = panel[panel["product"] == DEEP_DIVE].set_index("trading_day")
fig, axes = plt.subplots(4, 1, figsize=(16, 13), sharex=True)
axes[0].plot(d.index, d["close"], lw=.8, label="raw spliced close")
axes[0].plot(d.index, d["close"].iloc[0] * np.exp(d["logp_adj"]), lw=.8, label="roll-adjusted")
for t in d.index[d["roll"]]: axes[0].axvline(t, color="grey", alpha=.15, lw=.5)
axes[0].legend(); axes[0].set_title(f"{DEEP_DIVE}: price (grey lines = roll days)")
axes[1].plot(d.index, d["rv"].rolling(20).mean() * np.sqrt(ANN) * 100, lw=.8, color="tab:red")
axes[1].set_title("20d realized volatility (%, from 1-min returns)")
axes[2].bar(d.index, d["volume"], width=1, color="grey"); axes[2].set_title("Daily volume (lots)")
axes[3].plot(d.index, d["oi"], lw=.8, color="tab:green"); axes[3].set_title("Open interest (main contract)")
plt.tight_layout(); plt.show()

In [ ]:
# 4b. Realized volatility regimes across the focus set
rv_wide = panel.pivot(index="trading_day", columns="product", values="rv")[FOCUS]
rv_ann = rv_wide.rolling(60, min_periods=40).mean() * np.sqrt(ANN) * 100
ax = rv_ann.plot(figsize=(16, 6), lw=.8, colormap="tab20")
ax.set_title("60d realized vol (%)"); ax.legend(ncol=7, fontsize=8); plt.show()

# 4c. Volatility clustering: autocorrelation of returns vs |returns|
from statsmodels.tsa.stattools import acf
lags = 20
acf_rows = []
for c in PRODUCTS:
    r = panel.loc[panel["product"] == c, "ret"].dropna()
    if len(r) < 500: continue
    acf_rows.append(pd.DataFrame({"lag": range(1, lags + 1), "product": c,
                                  "acf_ret": acf(r, nlags=lags, fft=True)[1:],
                                  "acf_abs": acf(r.abs(), nlags=lags, fft=True)[1:]}))
acf_df = pd.concat(acf_rows)
m = acf_df.groupby("lag")[["acf_ret", "acf_abs"]].median()
ax = m.plot(kind="bar", figsize=(14, 4))
ax.axhline(1.96 / np.sqrt(2500), ls="--", c="k", lw=.7); ax.axhline(-1.96 / np.sqrt(2500), ls="--", c="k", lw=.7)
ax.set_title("Median ACF across products: returns (≈0) vs |returns| (persistent → vol is forecastable)"); plt.show()

In [ ]:
# 4d. Volume–volatility relationship and liquidity
vv = panel.assign(lvol=np.log(panel["volume"].where(panel["volume"] > 0)),
                  lrv=np.log(panel["rv"].where(panel["rv"] > 0)))
vv["dlvol"] = vv.groupby("product")["lvol"].diff()
vv["dlrv"] = vv.groupby("product")["lrv"].diff()
vvc = vv.groupby("product").apply(lambda x: pd.Series({
    "corr_level": x["lvol"].corr(x["lrv"]), "corr_change": x["dlvol"].corr(x["dlrv"]),
    "corr_vol_absret": x["volume"].corr(x["ret"].abs(), method="spearman")}))
display(vvc.describe().round(3))

# Liquidity ranking over the recent window -> candidate tradeable universe for MFT
rec = panel[panel.trading_day >= RECENT_START]
liq = rec.groupby("product").agg(med_turnover_bnCNY=("money", lambda s: s.median() / 1e9),
                                 med_volume=("volume", "median"), med_oi=("oi", "median"),
                                 days=("ret", "size"), ann_vol_pct=("ret", lambda s: s.std() * np.sqrt(ANN) * 100))
liq["sector"] = liq.index.map(SECTOR)
liq = liq.sort_values("med_turnover_bnCNY", ascending=False)
liq.to_csv(OUT_DIR / "liquidity_rank.csv")
display(liq.round(2).head(30))

yearly = panel.assign(year=panel.trading_day.dt.year).groupby(["year", "sector"])["money"].sum().unstack() / 1e12
ax = yearly.plot.area(figsize=(16, 5), colormap="tab20"); ax.set_title("Annual turnover by sector (trn CNY; CZCE may be double-counted)")
ax.legend(ncol=6, fontsize=8); plt.show()

In [ ]:
# 4e. Intraday seasonality: volume share and mean |1-min return| by clock time (recent window)
if len(profiles):
    pr = profiles[profiles["product"].isin(FOCUS)].copy()
    pr["order"] = (pr["mod"] - 20 * 60) % 1440          # start the axis at 20:00 (night session first)
    vs = pr.pivot_table(index="product", columns="order", values="vshare")
    ar = pr.pivot_table(index="product", columns="order", values="absret") * 1e4
    lab = [f"{((o + 1200) % 1440) // 60:02d}:{((o + 1200) % 1440) % 60:02d}" for o in vs.columns]
    for mat, title in [(vs.div(vs.max(axis=1), axis=0), "Volume share by minute (row-normalised)"),
                       (ar.div(ar.max(axis=1), axis=0), "Mean |1-min return| by minute (row-normalised)")]:
        fig, ax = plt.subplots(figsize=(18, 0.45 * len(mat) + 2))
        sns.heatmap(mat, ax=ax, cmap="viridis", cbar=False)
        step = max(1, len(lab) // 24)
        ax.set_xticks(np.arange(0, len(lab), step) + .5); ax.set_xticklabels(lab[::step], rotation=90)
        ax.set_title(title); ax.set_xlabel("clock time (trading-day order)"); plt.show()

    # share of daily volume in the first/last 30 minutes of the day session
    def window_share(g, a, b):
        return g.loc[(g["mod"] >= a) & (g["mod"] < b), "vshare"].sum()
    ses = profiles.groupby("product").apply(lambda g: pd.Series({
        "night_share": g.loc[(g["mod"] >= 20 * 60) | (g["mod"] < 4 * 60), "vshare"].sum(),
        "open30_share": window_share(g, 9 * 60, 9 * 60 + 31),
        "close30_share": window_share(g, 14 * 60 + 30, 15 * 60 + 16)}))
    display(ses.round(3).sort_values("night_share", ascending=False).head(20))

In [ ]:
# 4f. Calendar effects (pooled, vol-normalised returns)
cal = panel.dropna(subset=["ret"]).copy()
cal["z"] = cal["ret"] / cal.groupby("product")["ret"].transform("std")
cal["dow"] = cal["trading_day"].dt.day_name().str[:3]
cal["month"] = cal["trading_day"].dt.month
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
cal.groupby("dow")["z"].mean().reindex(["Mon", "Tue", "Wed", "Thu", "Fri"]).plot.bar(ax=axes[0], title="Mean z-return by weekday")
cal.groupby("month")["z"].mean().plot.bar(ax=axes[1], title="Mean z-return by month")
cal.assign(absz=cal["z"].abs()).groupby("dow")["absz"].mean().reindex(["Mon", "Tue", "Wed", "Thu", "Fri"]) \
   .plot.bar(ax=axes[2], title="Mean |z| by weekday (Mon includes weekend gap)")
plt.tight_layout(); plt.show()

## Step 5 — Correlation analysis
Daily returns over a common window, hierarchical clustering, correlation stability over time, and intraday lead–lag on 5-min returns.

**Used later.** Correlation clusters and lead–lag candidates seed the relationship search of Steps 14–15 and the sector structure of Step 13b.

In [ ]:
# step 5
R = panel.pivot(index="trading_day", columns="product", values="ret")    # unique by construction
R_rec = R[R.index >= RECENT_START]
R_rec = R_rec.loc[:, R_rec.notna().mean() > 0.8]                        # products alive in the window
C = R_rec.corr(min_periods=120)
C_s = R_rec.corr(method="spearman", min_periods=120)
C.to_csv(OUT_DIR / "corr_daily.csv")

cg = sns.clustermap(C.fillna(0), cmap="RdBu_r", vmin=-1, vmax=1, figsize=(16, 16),
                    xticklabels=True, yticklabels=True, method="average")
cg.ax_heatmap.tick_params(labelsize=7); cg.fig.suptitle(f"Daily return correlation since {RECENT_START} (clustered)", y=1.01)
plt.show()

# strongest pairs and Pearson vs Spearman gap (large gap = correlation driven by tail days)
iu = np.triu_indices_from(C, 1)
pairs = pd.DataFrame({"a": C.index[iu[0]], "b": C.columns[iu[1]], "pearson": C.values[iu], "spearman": C_s.values[iu]})
pairs["gap"] = pairs["pearson"] - pairs["spearman"]
display(pairs.sort_values("pearson", ascending=False).head(20).round(3))

# sector-level average correlation
sec = C.index.map(SECTOR)
S = C.groupby(sec).mean().T.groupby(sec).mean()
sns.heatmap(S, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1); plt.title("Average correlation by sector"); plt.show()

In [ ]:
# 5b. Correlation stability: average pairwise correlation per year, and rolling corr for economic pairs
def avg_offdiag(df):
    c = df.loc[:, df.notna().mean() > .8].corr(min_periods=60).values
    return np.nanmean(c[np.triu_indices_from(c, 1)])
yr = R.groupby(R.index.year).apply(avg_offdiag)
yr.plot.bar(figsize=(14, 3), title="Average pairwise daily correlation by year (market-wide co-movement)"); plt.show()

PAIRS = [("RB", "HC"), ("RB", "I"), ("J", "JM"), ("I", "J"), ("M", "RM"), ("Y", "P"), ("Y", "OI"),
         ("IF", "IH"), ("IF", "IC"), ("CU", "ZN"), ("AU", "AG"), ("TA", "EG"), ("L", "PP"), ("MA", "PP"), ("SC", "FU")]
PAIRS = [p for p in PAIRS if set(p) <= set(R.columns)]
roll_c = pd.DataFrame({f"{a}-{b}": R[a].rolling(120, min_periods=90).corr(R[b]) for a, b in PAIRS})
ax = roll_c.plot(figsize=(16, 6), lw=.9, colormap="tab20"); ax.set_title("120d rolling correlation"); ax.legend(ncol=5, fontsize=8); plt.show()

In [ ]:
# 5c. Lead–lag on 5-min returns: corr(a_t, b_{t+k}). Peak at k>0 means a leads b.
LAGS = range(-6, 7)
ll_rows = []
if len(ret5):
    X = ret5.copy()
    X = X[X.index.normalize() >= RECENT_START]
    for a, b in [(a, b) for i, a in enumerate(X.columns) for b in X.columns[i + 1:]]:
        cc = {k: X[a].corr(X[b].shift(-k)) for k in LAGS}   # causal-ok: EDA lead-lag cross-correlation (tests whether a leads b), not a feature
        s = pd.Series(cc)
        nz = s.drop(0)
        ll_rows.append({"a": a, "b": b, "corr_0": s[0], "best_lag": nz.abs().idxmax(), "corr_best": nz[nz.abs().idxmax()],
                        "ratio_lead_to_contemp": abs(nz[nz.abs().idxmax()]) / max(abs(s[0]), 1e-9)})
    leadlag = pd.DataFrame(ll_rows).sort_values("ratio_lead_to_contemp", ascending=False)
    leadlag.to_csv(OUT_DIR / "leadlag.csv", index=False)
    display(leadlag.head(15).round(4))
    print("Interpretation: best_lag=+1 means a's 5-min return correlates with b's NEXT 5-min return.")
    a0, b0 = leadlag.iloc[0][["a", "b"]]
    pd.Series({k: X[a0].corr(X[b0].shift(-k)) for k in LAGS}).plot.bar(figsize=(10, 3), title=f"Cross-correlation {a0}→{b0} (5-min)")   # causal-ok: EDA lead-lag chart
    plt.show()
else:
    leadlag = pd.DataFrame()
    print("No 5-min returns cached (LEADLAG_SET empty or no recent data).")

## Step 6 — Feature exploration
Features are computed **with information up to the close of day t only**; targets are forward returns starting at t+1.
Evaluation uses Spearman rank IC: cross-sectional (across products, per day) and time-series (per product).

**Used later.** Exploratory feature ICs decide which feature families are built properly in Step 18.

In [ ]:
# step 6
def build_features(panel):
    p = panel.sort_values(["product", "trading_day"]).copy()
    g = p.groupby("product", group_keys=False)
    roll_sum = lambda n, mp: (lambda s: s.rolling(n, min_periods=mp).sum())
    p["sym_change"] = g["symbol"].shift().ne(p["symbol"]) & g["symbol"].shift().notna()
    # momentum / reversal
    p["mom_1"] = p["ret"]
    for n in [5, 20, 60]:
        p[f"mom_{n}"] = g["ret"].transform(roll_sum(n, int(n * .8)))
    # volatility
    p["rv_20"] = g["rv"].transform(lambda s: np.sqrt((s ** 2).rolling(20, min_periods=15).mean()))
    p["mom_20_va"] = p["mom_20"] / (p["rv_20"] * np.sqrt(20))
    p["rv_ratio"] = p["rv"] / p["rv_20"]
    p["range_pk"] = np.log(p["high"] / p["low"]).where(~p["roll"])
    p["skew_20"] = g["ret"].transform(lambda s: s.rolling(20, min_periods=15).skew())
    # volume / open interest (OI changes across a contract switch are meaningless -> masked)
    p["_lv"] = np.log(p["volume"].where(p["volume"] > 0))
    p["vol_z"] = g["_lv"].transform(lambda s: (s - s.rolling(20, min_periods=15).mean().shift(1))
                                                / s.rolling(20, min_periods=15).std().shift(1))
    p["_loi"] = np.log(p["oi"].where(p["oi"] > 0))
    p["doi_1"] = g["_loi"].diff().where(~p["sym_change"])
    p["doi_5"] = p.groupby("product")["doi_1"].transform(lambda s: s.rolling(5, min_periods=4).sum())
    # session returns of day t
    p["night_ret"] = p["ret_night"]
    p["day_ex_open"] = p["ret_day_ex_open"]
    # targets
    p["fwd_1"] = g["ret"].shift(-1)
    for h in [5, 20]:
        p[f"fwd_{h}"] = g["ret"].transform(lambda s: s.rolling(h, min_periods=h).sum().shift(-h))
    p["fwd_1_z"] = p["fwd_1"] / p["rv_20"]
    return p.drop(columns=["_lv", "_loi"])

FEATURES = ["mom_1", "mom_5", "mom_20", "mom_60", "mom_20_va", "rv_20", "rv_ratio", "range_pk",
            "skew_20", "vol_z", "doi_1", "doi_5", "night_ret", "day_ex_open"]
TARGETS = {"fwd_1": 1, "fwd_5": 5, "fwd_20": 20}

feat = build_features(panel)
feat.to_parquet(OUT_DIR / "features.parquet", index=False)
display(feat[FEATURES].describe().T.round(4))

# feature-feature correlation (redundancy)
sns.heatmap(feat[FEATURES].corr(method="spearman"), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            annot_kws={"size": 7}); plt.title("Feature rank-correlation (redundancy check)"); plt.show()

In [ ]:
# step 6
def xs_ic(df, f, t, min_n=MIN_XS):
    # Per-date Spearman correlation between feature f and target t across products.
    d = df[["trading_day", f, t]].dropna()
    d = d[d.groupby("trading_day")[f].transform("size") >= min_n]
    if d.empty:
        return pd.Series(dtype=float)
    day = d["trading_day"]
    rk = d.groupby(day)[[f, t]].rank()
    a = rk[f] - rk.groupby(day)[f].transform("mean")   # causal-ok: same-date cross-sectional demeaning (all products at t)
    b = rk[t] - rk.groupby(day)[t].transform("mean")   # causal-ok: same-date cross-sectional demeaning of the target
    num = (a * b).groupby(day).sum()
    den = np.sqrt((a * a).groupby(day).sum() * (b * b).groupby(day).sum())
    return (num / den).replace([np.inf, -np.inf], np.nan).dropna()

ic_rows, ic_series = [], {}
for t, h in TARGETS.items():
    for f in FEATURES:
        ic = xs_ic(feat, f, t)
        if len(ic) < 50: continue
        ic_series[(f, t)] = ic
        ic_rows.append({"feature": f, "target": t, "n_days": len(ic), "mean_IC": ic.mean(), "IC_std": ic.std(),
                        "IR_ann": ic.mean() / ic.std() * np.sqrt(ANN / h),
                        "t_stat": ic.mean() / ic.std() * np.sqrt(len(ic) / h),   # /h: overlapping targets
                        "hit_rate": (ic > 0).mean(),
                        "IC_recent": ic[ic.index >= RECENT_START].mean()})
ic_summary = pd.DataFrame(ic_rows)
ic_summary.to_csv(OUT_DIR / "ic_summary.csv", index=False)
if len(ic_summary):
    display(ic_summary.pivot(index="feature", columns="target", values="t_stat").round(2)
            .style.background_gradient(cmap="RdBu_r", vmin=-4, vmax=4).set_caption("Cross-sectional IC t-stat (|t|>2 worth a look)"))
    display(ic_summary.sort_values("t_stat", key=abs, ascending=False).head(12).round(4))

    # cumulative IC over time for the strongest features (stability check)
    top = ic_summary[ic_summary.target == "fwd_1"].sort_values("t_stat", key=abs, ascending=False).feature.head(5)
    ax = pd.DataFrame({f: ic_series[(f, "fwd_1")].cumsum() for f in top}).plot(figsize=(16, 5))
    ax.set_title("Cumulative cross-sectional IC vs next-day return (straight line = stable signal)"); plt.show()
else:
    print(f"Too few products per date for cross-sectional IC (need ≥ {MIN_XS}).")

In [ ]:
# 6b. Time-series IC per product (does the feature predict that product's own next-day return?)
ts_ic = feat.groupby("product").apply(lambda x: pd.Series(
    {f: x[f].corr(x["fwd_1"], method="spearman") for f in FEATURES}))
ts_ic = ts_ic.loc[feat.groupby("product")["fwd_1"].count() > 500]
fig, ax = plt.subplots(figsize=(14, 0.25 * len(ts_ic) + 2))
sns.heatmap(ts_ic, cmap="RdBu_r", center=0, vmin=-0.08, vmax=0.08, ax=ax, yticklabels=True)
ax.tick_params(axis="y", labelsize=7); ax.set_title("Time-series rank IC vs next-day return, per product"); plt.show()
cons = pd.DataFrame({"median_TS_IC": ts_ic.median(), "share_positive": (ts_ic > 0).mean()})
display(cons.sort_values("median_TS_IC").round(4))

# 6c. Quintile spread for the best cross-sectional feature
if len(ic_summary):
    best = ic_summary[ic_summary.target == "fwd_1"].sort_values("t_stat", key=abs, ascending=False).iloc[0]["feature"]
    d = feat.dropna(subset=[best, "fwd_1"]).copy()
    d = d[d.groupby("trading_day")[best].transform("size") >= MIN_XS]
    d["q"] = d.groupby("trading_day")[best].transform(lambda s: pd.qcut(s.rank(method="first"), 5, labels=False)) + 1
    qret = d.groupby("q")["fwd_1"].mean() * 1e4
    qret.plot.bar(title=f"Mean next-day return (bp) by cross-sectional quintile of {best}", figsize=(8, 3)); plt.show()

### 6d — Sector-neutral IC
A raw cross-sectional IC can come from sector bets. For example, if ferrous products all trend together, momentum ranks pick "ferrous vs grains", not "RB vs HC". Two variants remove that:
* **sector-neutral target:** `fwd_1` minus the same-day sector mean (sectors with ≥ 2 products that day);
* **sector-neutral feature and target:** the feature is also demeaned within sector.

If the IC survives neutralisation, the signal picks products *within* a sector. If it vanishes, it was a sector-rotation signal (still tradable, but a different strategy). The Bonferroni bar accounts for testing every feature.

In [ ]:
# step 6d
sn = feat.copy()
sec_n = sn.groupby(["trading_day", "sector"])["fwd_1"].transform("count")   # causal-ok: same-date, same-sector cross-section
sn["fwd_1_sn"] = (sn["fwd_1"] - sn.groupby(["trading_day", "sector"])["fwd_1"].transform("mean")).where(sec_n >= 2)   # causal-ok: same-date, same-sector cross-section
for f in FEATURES:
    sn[f + "_sn"] = (sn[f] - sn.groupby(["trading_day", "sector"])[f].transform("mean")).where(sec_n >= 2)   # causal-ok: same-date, same-sector cross-section

rows = []
for f in FEATURES:
    for label, ff, tt in [("raw", f, "fwd_1"), ("target_sn", f, "fwd_1_sn"), ("both_sn", f + "_sn", "fwd_1_sn")]:
        ic = xs_ic(sn, ff, tt)
        if len(ic) >= 50:
            rows.append({"feature": f, "variant": label, "mean_IC": ic.mean(), "t_stat": ic.mean() / ic.std() * np.sqrt(len(ic)),
                         "n_days": len(ic)})
ic_sn = pd.DataFrame(rows)
if len(ic_sn):
    t_bonf = stats.norm.ppf(1 - 0.025 / max(len(FEATURES), 1))
    tab = ic_sn.pivot(index="feature", columns="variant", values="t_stat")[["raw", "target_sn", "both_sn"]]
    tab["survives_neutralisation"] = (tab["both_sn"].abs() >= t_bonf) & (np.sign(tab["both_sn"]) == np.sign(tab["raw"]))
    ic_sn.to_csv(OUT_DIR / "ic_sector_neutral.csv", index=False)
    display(tab.round(2).sort_values("raw", key=abs, ascending=False))
    fig, ax = plt.subplots(figsize=(14, 4))
    x = np.arange(len(tab)); w = 0.27
    for k, (col, c) in enumerate(zip(["raw", "target_sn", "both_sn"], ["#b9b8b2", "#86b6ef", "#2a78d6"])):
        ax.bar(x + (k - 1) * w, tab[col], width=w, color=c, label=col)
    for y in (t_bonf, -t_bonf):
        ax.axhline(y, color="#eb6834", lw=1, ls="--")
    ax.set_xticks(x); ax.set_xticklabels(tab.index, rotation=45, ha="right")
    ax.set_ylabel("IC t-stat vs fwd_1"); ax.legend(frameon=False)
    ax.set_title(f"Cross-sectional IC before and after sector neutralisation (dashed = Bonferroni |t| = {t_bonf:.2f})")
    plt.tight_layout(); plt.show()
else:
    print("Too few products per sector/date for sector-neutral IC.")

## Step 7 — Trading-signal hypotheses
Each hypothesis is reduced to one test statistic. Verdict rule: **promising** if |t| ≥ `t_needed` (2.5 by default, Bonferroni-raised when the row is the best of many tests, e.g. lead–lag pairs × lags or top IC features), **weak** if |t| ≥ 0.6·`t_needed`, **none** otherwise. These are in-sample, gross of costs — they decide what to research next, not what to trade.

**Used later.** These hypotheses become the factor families of Step 18 and the strategy templates of Step 23.

In [ ]:
# step 7
def verdict(t, t_needed=2.5):
    # t_needed > 2.5 when the row was picked as the best of many tests (multiple-testing guard)
    if pd.isna(t): return "n/a"
    return "promising" if abs(t) >= t_needed else ("weak" if abs(t) >= 0.6 * t_needed else "none")

def bonferroni_t(n_tests, alpha=0.05):
    return float(stats.norm.ppf(1 - alpha / (2 * max(n_tests, 1))))

H = []   # rows: id, hypothesis, metric, value, t, verdict, next_step

# H1 — serial dependence at daily horizons (variance ratio, Lo-MacKinlay homoskedastic z)
def variance_ratio(r, k):
    r = r.dropna().to_numpy(); n = len(r)
    if n < 10 * k: return np.nan, np.nan
    mu = r.mean(); v1 = ((r - mu) ** 2).sum() / (n - 1)
    rk = pd.Series(r).rolling(k).sum().dropna().to_numpy()
    vk = ((rk - k * mu) ** 2).sum() / (k * (n - k + 1) * (1 - k / n))
    vr = vk / v1
    return vr, (vr - 1) / np.sqrt(2 * (2 * k - 1) * (k - 1) / (3 * k * n))

vr_rows = []
for c, x in panel.groupby("product"):
    for k in [2, 5, 20]:
        vr, z = variance_ratio(x["ret"], k)
        vr_rows.append({"product": c, "k": k, "VR": vr, "z": z})
vr_df = pd.DataFrame(vr_rows).dropna()
vr_tab = vr_df.pivot(index="product", columns="k", values="VR")
display(vr_tab.describe().round(3))
for k in [5, 20]:
    s = vr_df[vr_df.k == k]
    H.append({"id": f"H1-{k}d", "hypothesis": f"Daily returns trend (VR>1) or mean-revert (VR<1) at {k}d",
              "metric": "median VR / #products z>2 / #z<-2", "value": f"{s.VR.median():.3f} / {(s.z > 2).sum()} / {(s.z < -2).sum()}",
              "t": s["z"].median() * np.sqrt(len(s)), "next_step": "TS momentum/reversal backtest on products with |z|>2"})

# H1b — 1-min bid/ask bounce
ac = quality_report["min_ret_ac1"].dropna()
H.append({"id": "H1b", "hypothesis": "1-min returns negatively autocorrelated (bounce / microstructure reversal)",
          "metric": "median lag-1 AC of 1-min returns", "value": f"{ac.median():.4f}",
          "t": ac.mean() / ac.std() * np.sqrt(len(ac)) if len(ac) > 2 else np.nan,
          "next_step": "Check whether it survives the tick size / spread (likely not tradable, but a cost-model input)"})

# H2 — night session predicts the day session (trade at 09:01 close)
nd = feat.dropna(subset=["night_ret", "day_ex_open"])
nd_ic = nd.groupby("product").apply(lambda x: x["night_ret"].corr(x["day_ex_open"], method="spearman")
                                    if len(x) > 250 else np.nan).dropna()
H.append({"id": "H2", "hypothesis": "Night-session return predicts the same trading day's day session (after 09:01)",
          "metric": "median per-product Spearman", "value": f"{nd_ic.median():.4f} ({len(nd_ic)} products)",
          "t": nd_ic.mean() / nd_ic.std() * np.sqrt(len(nd_ic)) if len(nd_ic) > 2 else np.nan,
          "next_step": "Intraday backtest: sign(night) entered 09:01, exit 15:00"})

# H3 — price/OI quadrant: new longs (r>0, ΔOI>0) / new shorts (r<0, ΔOI>0) continue; covering reverses
qd = feat.dropna(subset=["ret", "doi_1", "fwd_1_z"]).copy()
qd["quad"] = np.select([(qd.ret > 0) & (qd.doi_1 > 0), (qd.ret < 0) & (qd.doi_1 > 0),
                        (qd.ret > 0) & (qd.doi_1 < 0), (qd.ret < 0) & (qd.doi_1 < 0)],
                       ["up_OIup (new longs)", "down_OIup (new shorts)", "up_OIdown (short cover)", "down_OIdown (long liq.)"], "flat")
quad = qd.groupby("quad")["fwd_1_z"].agg(["mean", "std", "count"])
quad["t"] = quad["mean"] / quad["std"] * np.sqrt(quad["count"])
display(quad.round(4))
cont = qd["fwd_1_z"] * np.sign(qd["ret"]) * np.where(qd["doi_1"] > 0, 1, -1)
H.append({"id": "H3", "hypothesis": "OI-confirmed moves continue next day; OI-declining moves reverse",
          "metric": "mean vol-scaled fwd return of the combined signal", "value": f"{cont.mean():.4f}",
          "t": cont.mean() / cont.std() * np.sqrt(cont.count()), "next_step": "Condition on |ΔOI| size and test per sector"})

# H4 — volume shocks: continuation and higher next-day volatility
vs = feat.dropna(subset=["vol_z", "fwd_1_z", "ret"])
shock = vs[vs["vol_z"] > 2]
cont4 = shock["fwd_1_z"] * np.sign(shock["ret"])
absr = vs.groupby(vs["vol_z"] > 2)["fwd_1_z"].apply(lambda s: s.abs().mean())
H.append({"id": "H4", "hypothesis": "After a volume shock (z>2) the move continues next day",
          "metric": "mean sign(r_t)·z-fwd return | shock days", "value": f"{cont4.mean():.4f} (n={len(shock)})",
          "t": cont4.mean() / cont4.std() * np.sqrt(len(cont4)) if len(cont4) > 30 else np.nan,
          "next_step": f"Also: next-day |z| {absr.get(True, np.nan):.3f} on shock vs {absr.get(False, np.nan):.3f} otherwise → vol-timing input"})

In [ ]:
# step 7
# H5 — economically linked spreads: cointegration (Engle-Granger) and mean-reversion half-life
from statsmodels.tsa.stattools import coint

def half_life(s):
    s = s.dropna(); ds = s.diff().dropna(); lag = s.shift().dropna().loc[ds.index]
    b = np.polyfit(lag, ds, 1)[0]
    return -np.log(2) / b if b < 0 else np.inf

SPREADS = {"steel margin RB~I+J": ("RB", ["I", "J"]), "coke margin J~JM": ("J", ["JM"]),
           "crush B~M+Y": ("B", ["M", "Y"]), "crush A~M+Y": ("A", ["M", "Y"]), "RB~HC": ("RB", ["HC"]),
           "Y~P": ("Y", ["P"]), "Y~OI": ("Y", ["OI"]), "IF~IH": ("IF", ["IH"]), "AU~AG": ("AU", ["AG"]),
           "CU~ZN": ("CU", ["ZN"]), "L~PP": ("L", ["PP"]), "TA~EG": ("TA", ["EG"])}
LP = panel.pivot(index="trading_day", columns="product", values="logp_adj")
LP = LP[LP.index >= RECENT_START]
sp_rows = []
for name, (y, xs) in SPREADS.items():
    if not set([y] + xs) <= set(LP.columns): continue
    d = LP[[y] + xs].dropna()
    if len(d) < 250: continue
    X = np.column_stack([np.ones(len(d))] + [d[x] for x in xs])
    beta = np.linalg.lstsq(X, d[y].to_numpy(), rcond=None)[0]
    resid = pd.Series(d[y].to_numpy() - X @ beta, index=d.index)
    stat, pval, _ = coint(d[y], d[xs])
    sp_rows.append({"spread": name, "n": len(d), "betas": np.round(beta[1:], 3).tolist(),
                    "coint_p": pval, "half_life_days": half_life(resid), "resid_sd_%": resid.std() * 100})
spreads = pd.DataFrame(sp_rows)
display(spreads.round(4))
for _, s in spreads.iterrows():
    ok = s.coint_p < 0.05 and s.half_life_days < 60
    H.append({"id": f"H5 {s.spread}", "hypothesis": f"{s.spread} mean-reverts",
              "metric": "coint p / half-life (d)", "value": f"{s.coint_p:.3f} / {s.half_life_days:.1f}",
              "t": 3.0 if ok else (2.0 if s.coint_p < 0.10 else 0.0),
              "next_step": "Rolling-beta spread z-score backtest with costs; check stability by year"})

# H6 — cross-sectional anomalies from Step 6
t_need_ic = bonferroni_t(len(ic_summary))
for _, r in ic_summary.sort_values("t_stat", key=abs, ascending=False).head(4).iterrows():
    H.append({"t_needed": t_need_ic, "id": f"H6 {r.feature}/{r.target}", "hypothesis": f"Cross-sectional rank of {r.feature} predicts {r.target}",
              "metric": "mean rank IC (recent)", "value": f"{r.mean_IC:.4f} ({r.IC_recent:.4f})", "t": r.t_stat,
              "next_step": "Sector-neutral long/short quintile portfolio, vol-scaled, with turnover costs"})

# H7 — intraday lead-lag
if len(leadlag):
    top = leadlag.iloc[0]
    n5 = int(ret5[[top.a, top.b]].dropna().shape[0])
    n_tests = len(leadlag) * (len(LAGS) - 1)
    H.append({"t_needed": bonferroni_t(n_tests), "id": "H7", "hypothesis": f"{top.a}/{top.b} 5-min returns lead/lag (best lag {top.best_lag})",
              "metric": "corr at best non-zero lag", "value": f"{top.corr_best:.4f} (contemp {top.corr_0:.3f})",
              "t": top.corr_best * np.sqrt(n5), "next_step": "Test at 1-min with execution delay; check stability by month"})

hyp = pd.DataFrame(H)
if "t_needed" not in hyp: hyp["t_needed"] = np.nan
hyp["t_needed"] = hyp["t_needed"].fillna(2.5)
hyp["verdict"] = [verdict(t, tn) for t, tn in zip(hyp["t"], hyp["t_needed"])]
hyp = hyp.sort_values("t", key=abs, ascending=False)
hyp.to_csv(OUT_DIR / "hypotheses.csv", index=False)
display(hyp[["id", "hypothesis", "metric", "value", "t", "t_needed", "verdict", "next_step"]].round(2))

## Step 8 — Summary and hand-off
Writes the key numbers to `eda_summary.json` so the strategy notebook can load them.

**Used later.** The hand-off table lists the facts that Parts B–H test formally, with train / validation / test discipline.

In [ ]:
# step 8
summary = {
    "data_dir": str(DATA_DIR), "n_products": len(PRODUCTS),
    "span": [str(panel.trading_day.min().date()), str(panel.trading_day.max().date())],
    "quality_flags": {k: int(v) for k, v in flag_df.sum().items()},
    "products_with_3plus_flags": quality_report.index[quality_report["n_flags"] >= 3].tolist(),
    "median_ann_vol_pct": float(rstats["ann_vol_%"].median()),
    "median_excess_kurtosis": float(rstats["ex_kurt"].median()),
    "top_liquid_recent": liq.head(20).index.tolist(),
    "avg_pairwise_corr_recent": float(np.nanmean(C.values[np.triu_indices_from(C, 1)])),
    "promising_hypotheses": hyp.loc[hyp.verdict == "promising", "id"].tolist(),
    "weak_hypotheses": hyp.loc[hyp.verdict == "weak", "id"].tolist(),
}
with open(OUT_DIR / "eda_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))
print("\nFiles in", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print("  ", p.name)

---
# Part B — Univariate and multivariate properties (Steps 9–13d)
How each series behaves on its own, which is needed before modelling relationships between them:

| Step | Question |
|---|---|
| 9 | Splits; is the price or the return stationary? How much differencing is needed (fractional d*)? |
| 10 | Is there linear predictability (AR, ARMA = ARIMA(1,1,1), robust and penalised regressions)? |
| 11 | Which volatility model fits (GARCH family, HAR-RV)? Leverage? These residuals feed Steps 12–25 |
| 12 | Which distribution do the standardised residuals follow? |
| 11b / 11c | **Which volatility model survives every regime and time frame** (blocks, regimes, 5m ... 1W, walk-forward selection)? **Is volatility expanding or contracting** now? |
| 13 | How heavy are the tails (EVT), and **do extreme moves create opportunities** (continuation / reversal, time-of-day buckets, clustering, spillover), also **beyond a few days** (13.4)? |
| 13b | **Multivariate structure**: cross-section (PCA / random-matrix test), joint dynamics (correlation by regime, covariance estimators), Fama–MacBeth, sectors, between sectors |
| 13c | **Copulas**: which dependence family fits each pair (Gaussian, Student-t, Clayton, Gumbel), tail dependence, joint extremes |
| 13d | **Overnight return puzzle and the T+1 rule**: overnight vs intraday returns, pent-up pressure, gap fade, EquityIndex vs the rest |

### Toolbox
The functions used throughout Parts B–F are defined once, here.

In [ ]:
# step 8
# ======================= plotting helpers =======================
INK, INK2, GRID, MUTE = "#0b0b0b", "#52514e", "#e4e3df", "#b9b8b2"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#f4f8fd", "#86b6ef", "#2a78d6", "#104281"])
DIV = LinearSegmentedColormap.from_list("div", ["#1c5cab", "#86b6ef", "#f0efec", "#f3a383", "#c4471a"])
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.spines.top": False,
    "axes.spines.right": False, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.titlecolor": INK, "font.size": 9, "lines.linewidth": 1.6, "axes.prop_cycle": matplotlib.cycler(color=SERIES),
})
FIGS = []   # (step, title, path) for the report

def savefig(fig, name, title, part):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=110, bbox_inches="tight"); fig._kept = True
    FIGS.append((str(part), title, path))
    plt.show()

def heat(ax, M, cmap=SEQ, vmin=None, vmax=None, center=None, label="", annot=False, fmt="{:.2f}"):
    A = np.asarray(M, float)
    if A.size == 0 or not np.isfinite(A).any():
        ax.text(0.5, 0.5, "no data in this split", ha="center", va="center", transform=ax.transAxes); ax.set_axis_off(); ax.set_title(label, fontsize=8)
        return None
    if center is not None:
        lim = np.nanmax(np.abs(A - center)) if vmax is None else vmax
        lim = lim if np.isfinite(lim) and lim > 0 else 1
        vmin, vmax = center - lim, center + lim
    im = ax.imshow(A, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto", interpolation="nearest")
    ax.set_xticks(range(A.shape[1])); ax.set_xticklabels(M.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(A.shape[0])); ax.set_yticklabels(M.index, fontsize=7)
    ax.grid(False)
    if annot and A.size <= 400:
        for i in range(A.shape[0]):
            for j in range(A.shape[1]):
                if np.isfinite(A[i, j]):
                    ax.text(j, i, fmt.format(A[i, j]), ha="center", va="center", fontsize=6, color=INK)
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.01, label=label)
    return im

def sector_order(cols):
    return sorted(cols, key=lambda c: (SECTOR.get(c, "Other"), c))

In [ ]:
# step 8
# ======================= toolbox: regression, Granger, tests =======================
def lagmat(x, p):
    # row t holds x[t-1], ..., x[t-p]
    T = len(x); out = np.full((T, p), np.nan)
    for k in range(1, p + 1):
        out[k:, k - 1] = x[:-k]
    return out

def ols(X, y):
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X @ b
    return float(e @ e), b

def bh(p):
    # Benjamini-Hochberg q-values (NaN-safe)
    p = np.asarray(p, float); q = np.full_like(p, np.nan); ok = np.isfinite(p)
    ps = p[ok]; m = len(ps)
    if m == 0:
        return q
    o = np.argsort(ps); r = ps[o] * m / np.arange(1, m + 1)
    r = np.minimum.accumulate(r[::-1])[::-1]
    qq = np.empty(m); qq[o] = np.minimum(r, 1); q[ok] = qq
    return q

def dm_test(d):
    # Diebold-Mariano with Newey-West variance; one-sided: mean(d) > 0
    d = np.asarray(d, float); d = d[np.isfinite(d)]; T = len(d)
    if T < 30:
        return np.nan, np.nan
    h = int(np.floor(T ** (1 / 3))); dc = d - d.mean(); s = dc @ dc / T
    for k in range(1, h + 1):
        s += 2 * (1 - k / (h + 1)) * (dc[k:] @ dc[:-k] / T)
    t = d.mean() / np.sqrt(s / T)
    return t, stats.norm.sf(t)

def design(D, cause, effect, p, control=True):
    A = D.to_numpy(float); idx = {c: k for k, c in enumerate(D.columns)}
    T = len(A); y = A[:, idx[effect]]
    parts = [np.ones((T, 1)), lagmat(y, p)]
    if control:
        with np.errstate(all="ignore"):
            parts.append(lagmat(np.nanmean(A, axis=1), p))
    Xr = np.hstack(parts)
    return y, Xr, np.hstack([Xr, lagmat(A[:, idx[cause]], p)])

def granger_matrix(D, p, control=CONDITION_ON_MARKET, pairs=None):
    cols = list(D.columns); A = D.to_numpy(float); T = len(A); idx = {c: k for k, c in enumerate(cols)}
    L = {c: lagmat(A[:, k], p) for k, c in enumerate(cols)}
    with np.errstate(all="ignore"):
        Lm = lagmat(np.nanmean(A, axis=1), p)
    if pairs is None:
        pairs = [(c, e) for e in cols for c in cols if c != e]
    rows = []
    for cause, eff in pairs:
        y = A[:, idx[eff]]
        Xr = np.hstack([np.ones((T, 1)), L[eff]] + ([Lm] if control else []))
        Xu = np.hstack([Xr, L[cause]])
        m = np.isfinite(y) & np.isfinite(Xu).all(axis=1)
        nn = int(m.sum()); df2 = nn - Xu.shape[1]
        if nn < 10 * Xu.shape[1]:
            rows.append(dict(cause=cause, effect=eff, F=np.nan, p=np.nan, n=nn, partial_r2=np.nan, coef_sum=np.nan)); continue
        rss_r, _ = ols(Xr[m], y[m]); rss_u, b = ols(Xu[m], y[m])
        if not (rss_u > 1e-12 and rss_r > 1e-12 and df2 > 0):        # constant / perfectly fitted series (halted product, all-zero returns): no test possible
            rows.append(dict(cause=cause, effect=eff, F=np.nan, p=np.nan, n=nn, partial_r2=np.nan, coef_sum=np.nan)); continue
        F = ((rss_r - rss_u) / p) / (rss_u / df2)
        rows.append(dict(cause=cause, effect=eff, F=F, p=stats.f.sf(F, p, df2), n=nn,
                         partial_r2=(rss_r - rss_u) / rss_r, coef_sum=b[-p:].sum()))
    return pd.DataFrame(rows)

def to_matrix(G, val, cols):
    return G.pivot(index="cause", columns="effect", values=val).reindex(index=cols, columns=cols)

def arma_fit(y, order):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return ARIMA(np.asarray(y, float), order=(order[0], 0, order[1]), trend="c").fit()

def arma_pred(res, y):
    # one-step-ahead predictions on a new block with the parameters frozen
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return np.asarray(res.apply(np.asarray(y, float)).fittedvalues)

def ljung_box(x, lags=10):
    x = x[np.isfinite(x)] - np.nanmean(x); n = len(x); d = x @ x
    ac = np.array([x[k:] @ x[:-k] / d for k in range(1, lags + 1)])
    Q = n * (n + 2) * np.sum(ac ** 2 / (n - np.arange(1, lags + 1)))
    return Q, stats.chi2.sf(Q, lags)

def hill(x, frac=0.05):
    x = np.sort(x[np.isfinite(x) & (x > 0)])[::-1]; k = max(int(len(x) * frac), 10)
    if len(x) <= k:
        return np.nan
    return 1 / np.mean(np.log(x[:k] / x[k]))

In [ ]:
# step 8
# ======================= toolbox: distributions =======================
class GMM2:
    # 2-component Gaussian mixture with a scipy-like interface
    name = "gmm2"
    def fit(self, x):
        g = GaussianMixture(2, covariance_type="full", n_init=3, random_state=SEED).fit(np.asarray(x).reshape(-1, 1))
        return (g.weights_[0], g.means_[0, 0], np.sqrt(g.covariances_[0, 0, 0]), g.means_[1, 0], np.sqrt(g.covariances_[1, 0, 0]))
    def pdf(self, x, w, m1, s1, m2, s2):
        return w * stats.norm.pdf(x, m1, s1) + (1 - w) * stats.norm.pdf(x, m2, s2)
    def logpdf(self, x, *a):
        return np.log(np.maximum(self.pdf(x, *a), 1e-300))
    def cdf(self, x, w, m1, s1, m2, s2):
        return w * stats.norm.cdf(x, m1, s1) + (1 - w) * stats.norm.cdf(x, m2, s2)
    def ppf(self, q, *a):
        return np.array([optimize.brentq(lambda z: self.cdf(z, *a) - qq, -80, 80) for qq in np.atleast_1d(q)])
    def rvs(self, *a, size=1, random_state=None):
        w, m1, s1, m2, s2 = a; r = np.random.default_rng(random_state)
        return np.where(r.random(size) < w, r.normal(m1, s1, size), r.normal(m2, s2, size))

DISTS = {f: (GMM2() if f == "gmm2" else getattr(stats, f)) for f in FAMILIES}
NPAR = {f: (5 if f == "gmm2" else DISTS[f].numargs + 2) for f in FAMILIES}

def fit_family(f, x):
    d = DISTS[f]
    if f == "gmm2":
        return d.fit(x)
    if f == "t":
        return d.fit(x, 5, loc=np.median(x), scale=x.std() * 0.8)
    if f == "nct":
        return d.fit(x, 5, -0.1 * np.sign(stats.skew(x)), loc=np.median(x), scale=x.std() * 0.8)
    return d.fit(x)

def ad_stat(u):
    u = np.clip(np.sort(u), 1e-12, 1 - 1e-12); n = len(u); i = np.arange(1, n + 1)
    return -n - np.mean((2 * i - 1) * (np.log(u) + np.log(1 - u[::-1])))

def cvm_stat(u):
    u = np.sort(u); n = len(u); i = np.arange(1, n + 1)
    return 1 / (12 * n) + np.sum((u - (2 * i - 1) / (2 * n)) ** 2)

def gof(f, par, x):
    d = DISTS[f]; ll = float(np.sum(d.logpdf(x, *par))); k = NPAR[f]; n = len(x)
    u = d.cdf(np.sort(x), *par)
    ks = stats.kstest(x, lambda z: d.cdf(z, *par))
    return dict(loglik=ll, k=k, aic=2 * k - 2 * ll, bic=k * np.log(n) - 2 * ll, ks=ks.statistic, ks_p_naive=ks.pvalue,
                cvm=cvm_stat(u), ad=ad_stat(u))

def boot_ks_p(f, par, x, B=BOOT_B):
    n = min(len(x), 2000); ks0 = stats.kstest(x, lambda z: DISTS[f].cdf(z, *par)).statistic * np.sqrt(len(x))
    exceed = 0
    for b in range(B):
        xb = np.asarray(DISTS[f].rvs(*par, size=n, random_state=SEED + b), float)
        try:
            pb = tuple(np.atleast_1d(fit_family(f, xb)).astype(float))
            exceed += stats.kstest(xb, lambda z: DISTS[f].cdf(z, *pb)).statistic * np.sqrt(n) >= ks0
        except Exception:
            pass
    return (1 + exceed) / (B + 1)

def min_cvm_fit(f, par0, x):
    d = DISTS[f]; xs = np.sort(x)
    def obj(th):
        if f != "gmm2" and th[-1] <= 0:
            return 1e9
        if f == "gmm2" and not (0 < th[0] < 1 and th[2] > 0 and th[4] > 0):
            return 1e9
        with np.errstate(all="ignore"):
            v = cvm_stat(d.cdf(xs, *th))
        return v if np.isfinite(v) else 1e9
    r = optimize.minimize(obj, np.array(par0), method="Nelder-Mead", options={"maxiter": 2000, "xatol": 1e-4, "fatol": 1e-6})
    return tuple(r.x), r.fun

U_TAIL = (np.arange(200) + 0.5) / 200
def dist_tail(f, par, a):
    # (VaR, ES) of the LOSS (= -z) at confidence a for a fitted residual distribution
    q = float(np.atleast_1d(DISTS[f].ppf(1 - a, *par))[0])
    es = -float(np.mean(DISTS[f].ppf(U_TAIL * (1 - a), *par)))
    return -q, es

def dist_gain_tail_mean(f, par, a):
    return float(np.mean(DISTS[f].ppf(1 - U_TAIL * a, *par)))

In [ ]:
# step 8
# ======================= toolbox: risk backtests and performance =======================
def kupiec(I, a):
    # proportion-of-failures test; a = expected exceedance probability
    I = np.asarray(I, bool); T = len(I); x = I.sum()
    if T == 0:
        return np.nan
    ph = x / T
    lr = -2 * (xlogy(T - x, 1 - a) + xlogy(x, a) - xlogy(T - x, 1 - ph) - xlogy(x, ph))
    return stats.chi2.sf(lr, 1)

def christoffersen(I, a):
    # independence and conditional-coverage tests
    I = np.asarray(I, int)
    if len(I) < 3:
        return np.nan, np.nan
    s0, s1 = I[:-1], I[1:]
    n00 = np.sum((s0 == 0) & (s1 == 0)); n01 = np.sum((s0 == 0) & (s1 == 1))
    n10 = np.sum((s0 == 1) & (s1 == 0)); n11 = np.sum((s0 == 1) & (s1 == 1))
    p01 = n01 / max(n00 + n01, 1); p11 = n11 / max(n10 + n11, 1); p = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)
    ll = lambda pp, n0, n1: xlogy(n0, 1 - pp) + xlogy(n1, pp)
    lr_ind = -2 * (ll(p, n00 + n10, n01 + n11) - ll(p01, n00, n01) - ll(p11, n10, n11))
    T = len(I); x = I.sum(); ph = x / T
    lr_pof = -2 * (xlogy(T - x, 1 - a) + xlogy(x, a) - xlogy(T - x, 1 - ph) - xlogy(x, ph))
    return stats.chi2.sf(lr_ind, 1), stats.chi2.sf(lr_pof + lr_ind, 2)

def basel_zone(n_exc, T):
    k = n_exc * 250 / max(T, 1)
    return "green" if k <= 4 else ("yellow" if k <= 9 else "red")

def rachev(r, a=RACHEV_A):
    r = pd.Series(r).dropna()
    if len(r) < 40:
        return np.nan
    lo, hi = r.quantile(a), r.quantile(1 - a)
    return r[r >= hi].mean() / -r[r <= lo].mean()

def psr(sr, T, skew, kurt, sr0=0.0):
    # probabilistic Sharpe ratio (per-period SR, non-excess kurtosis)
    den = np.sqrt(max(1 - skew * sr + (kurt - 1) / 4 * sr ** 2, 1e-12))
    return stats.norm.cdf((sr - sr0) * np.sqrt(max(T - 1, 1)) / den)

def dsr_threshold(sr_trials):
    # expected maximum SR of N unskilled trials (Bailey & Lopez de Prado)
    sr_trials = np.asarray(sr_trials, float); sr_trials = sr_trials[np.isfinite(sr_trials)]; N = len(sr_trials)
    if N < 2:
        return 0.0
    g = 0.5772156649
    return np.sqrt(np.var(sr_trials)) * ((1 - g) * stats.norm.ppf(1 - 1 / N) + g * stats.norm.ppf(1 - 1 / (N * np.e)))

def perf(net, gross=None, turn=None):
    r = pd.Series(net).dropna()
    if len(r) < 20 or r.std() == 0:
        return dict(n=len(r), sharpe=np.nan)
    mu, sd = r.mean(), r.std(); dn = r[r < 0].std()
    eq = (1 + r).cumprod(); dd = (eq / eq.cummax() - 1).min()
    out = dict(n=len(r), ann_ret=mu * ANN, ann_vol=sd * np.sqrt(ANN), sharpe=mu / sd * np.sqrt(ANN),
               sortino=mu / dn * np.sqrt(ANN) if dn > 0 else np.nan, max_dd=dd, calmar=mu * ANN / abs(dd) if dd < 0 else np.nan,
               rachev=rachev(r), hit=(r[r != 0] > 0).mean(), t=mu / sd * np.sqrt(len(r)), skew=r.skew(), kurt=r.kurt(),
               psr0=psr(mu / sd, len(r), r.skew(), r.kurt() + 3))
    if gross is not None:
        g = pd.Series(gross).reindex(r.index)
        out["sharpe_gross"] = g.mean() / g.std() * np.sqrt(ANN) if g.std() > 0 else np.nan
        out["cost_drag"] = (g - r).mean() * ANN
    if turn is not None:
        out["turnover"] = pd.Series(turn).reindex(r.index).mean() * ANN
    return out

In [ ]:
# step 8
# ======================= toolbox: Granger pipeline (used daily in Step 14 and 5-min in Step 14) =======================
def granger_discover(Ztr, Zva, tag, p, min_pr2, sens=True):
    cols = list(Ztr.columns)
    G_tr = granger_matrix(Ztr, p)
    G_tr["q"] = bh(G_tr["p"]); G_tr["sig"] = (G_tr.q < FDR_Q) & (G_tr.partial_r2 >= min_pr2)
    n_pairs = int(G_tr.p.notna().sum())
    print(f"[{tag}] TRAIN: {n_pairs} pairs | raw p<0.05: {(G_tr.p < 0.05).sum()} (≈{0.05*n_pairs:.0f} by chance) | "
          f"BH q<{FDR_Q} & partial R² ≥ {min_pr2}: {int(G_tr.sig.sum())}")
    # lag sensitivity
    base = set(map(tuple, G_tr.loc[G_tr.sig, ["cause", "effect"]].values)); sens = []
    for pp in (sorted(set(SENS_LAGS) | {p}) if sens else [p]):
        G = G_tr if pp == p else granger_matrix(Ztr, pp)
        if pp != p:
            G["q"] = bh(G["p"]); G["sig"] = (G.q < FDR_Q) & (G.partial_r2 >= min_pr2)
        s = set(map(tuple, G.loc[G.sig, ["cause", "effect"]].values))
        sens.append(dict(p=pp, n_sig=len(s), jaccard_vs_base=len(s & base) / max(len(s | base), 1)))
        G_tr[f"sig_p{pp}"] = [tuple(x) in s for x in G_tr[["cause", "effect"]].values]
    G_tr["robust_lags"] = G_tr[[c for c in G_tr if c.startswith("sig_p")]].sum(axis=1)
    # validation: only train survivors
    pairs_tr = [tuple(x) for x in G_tr.loc[G_tr.sig, ["cause", "effect"]].values]
    vcols = ["cause", "effect", "F", "p", "n", "partial_r2", "coef_sum", "q", "coef_sum_tr", "partial_r2_tr", "p_tr",
             "robust_lags", "same_sign", "candidate"]
    if pairs_tr:
        G_va = granger_matrix(Zva, p, pairs=pairs_tr); G_va["q"] = bh(G_va["p"])
        G_va = G_va.merge(G_tr[["cause", "effect", "coef_sum", "partial_r2", "p", "robust_lags"]], on=["cause", "effect"], suffixes=("", "_tr"))
        G_va["same_sign"] = np.sign(G_va.coef_sum) == np.sign(G_va.coef_sum_tr)
        G_va["candidate"] = (G_va.q < VAL_Q) & G_va.same_sign
    else:
        G_va = pd.DataFrame(columns=vcols).astype({"candidate": bool, "same_sign": bool})
    CAND = G_va[G_va.candidate].sort_values("p")
    print(f"[{tag}] VALIDATION: {len(G_va)} train survivors re-tested → {len(CAND)} candidates (q<{VAL_Q}, same sign)")
    # freeze candidate models fitted on train+val
    frozen = {"tag": tag, "p": p, "control_market": CONDITION_ON_MARKET, "universe": cols, "models": []}
    for r in CAND.itertuples():
        ys, Xrs, Xus = [], [], []
        for D in (Ztr, Zva):
            y, Xr, Xu = design(D, r.cause, r.effect, p, CONDITION_ON_MARKET); ys.append(y); Xrs.append(Xr); Xus.append(Xu)
        y, Xr, Xu = np.concatenate(ys), np.vstack(Xrs), np.vstack(Xus)
        m = np.isfinite(y) & np.isfinite(Xu).all(axis=1)
        _, b_ar = ols(Xr[m], y[m]); _, b_arx = ols(Xu[m], y[m])
        frozen["models"].append({"cause": r.cause, "effect": r.effect, "beta_ar": b_ar.round(8).tolist(),
                                 "beta_arx": b_arx.round(8).tolist(), "n_fit": int(m.sum())})
    blob = json.dumps(frozen, sort_keys=True).encode(); sha = hashlib.sha256(blob).hexdigest()
    (OUT2 / f"frozen_models_{tag}.json").write_bytes(blob)
    G_tr.to_csv(OUT2 / f"granger_train_{tag}.csv", index=False); G_va.to_csv(OUT2 / f"granger_val_{tag}.csv", index=False)
    print(f"[{tag}] FROZEN: {len(frozen['models'])} models | SHA-256 {sha}")
    if len(CAND):
        print(CAND[["cause", "effect", "p_tr", "p", "q", "partial_r2_tr", "partial_r2", "coef_sum_tr", "coef_sum", "robust_lags"]]
              .round(4).to_string(index=False))
    return dict(tag=tag, p=p, min_pr2=min_pr2, cols=cols, G_tr=G_tr, sens=pd.DataFrame(sens), G_va=G_va, CAND=CAND,
                frozen=frozen, FROZEN_SHA=sha, n_pairs=n_pairs)

def granger_blind(res, Zte):
    tag, cols = res["tag"], res["cols"]
    perm = list(rng.permutation(cols)); codes = [f"T{k:02d}" for k in range(1, len(perm) + 1)]
    (SEALED / f"test_label_map_{tag}.json").write_text(json.dumps(dict(zip(codes, perm))))
    Zb = Zte[perm].copy(); Zb.columns = codes; Zb = Zb.reset_index(drop=True)
    del perm
    G_bl = granger_matrix(Zb, res["p"])
    G_bl["q"] = bh(G_bl["p"]); G_bl["sig"] = (G_bl.q < FDR_Q) & (G_bl.partial_r2 >= res["min_pr2"])
    G_bl.to_csv(OUT2 / f"blind_test_coded_{tag}.csv", index=False)
    sha = hashlib.sha256((OUT2 / f"blind_test_coded_{tag}.csv").read_bytes()).hexdigest()
    print(f"[{tag}] BLIND TEST: {G_bl.p.notna().sum()} coded pairs | q<{FDR_Q}: {int(G_bl.sig.sum())} | SHA-256 {sha[:16]}…")
    print(G_bl.sort_values("p").head(8)[["cause", "effect", "F", "p", "q", "partial_r2", "coef_sum"]].round(4).to_string(index=False))
    res.update(Z_blind=Zb, codes=codes, G_bl=G_bl, BLIND_SHA=sha)

def granger_score(res, Zte):
    tag = res["tag"]; label_map = json.loads((SEALED / f"test_label_map_{tag}.json").read_text())
    G_bl = res["G_bl"]
    G_te = G_bl.assign(cause=G_bl.cause.map(label_map), effect=G_bl.effect.map(label_map))
    Zn = res["Z_blind"].rename(columns=label_map)[res["cols"]]
    assert np.allclose(np.nan_to_num(Zn.to_numpy()), np.nan_to_num(Zte[res["cols"]].to_numpy())), "unseal mismatch"
    CAND = res["CAND"]
    base = CAND if len(CAND) else pd.DataFrame(columns=["cause", "effect", "coef_sum_tr", "partial_r2_tr", "p_tr", "p"])
    rep = base[["cause", "effect", "coef_sum_tr", "partial_r2_tr", "p_tr", "p"]].rename(columns={"p": "p_val"}).merge(
        G_te[["cause", "effect", "F", "p", "q", "partial_r2", "coef_sum"]].rename(
            columns={"p": "p_test", "q": "q_test", "partial_r2": "partial_r2_test", "coef_sum": "coef_sum_test"}),
        on=["cause", "effect"], how="left")
    rep["same_sign"] = np.sign(rep.coef_sum_test.astype(float)) == np.sign(rep.coef_sum_tr.astype(float))
    rep["replicated"] = (rep.same_sign & (rep.p_test.astype(float) / 2 < 0.05)).astype(bool)   # one-sided, frozen direction
    n_rep, m_ = int(rep.replicated.sum()), len(rep)
    binom_p = stats.binomtest(n_rep, m_, 0.025, alternative="greater").pvalue if m_ else np.nan
    G_all = res["G_tr"][["cause", "effect", "p", "sig"]].merge(G_te[["cause", "effect", "p", "sig"]], on=["cause", "effect"], suffixes=("_tr", "_te"))
    rho = stats.spearmanr(-np.log10(G_all.p_tr), -np.log10(G_all.p_te), nan_policy="omit").statistic
    both = int((G_all.sig_tr & G_all.sig_te).sum())
    oos, LOSS = [], {}
    for mdl in res["frozen"]["models"]:
        y, Xr, Xu = design(Zn, mdl["cause"], mdl["effect"], res["p"], CONDITION_ON_MARKET)
        m = np.isfinite(y) & np.isfinite(Xu).all(axis=1)
        e_ar = y[m] - Xr[m] @ np.array(mdl["beta_ar"]); f_arx = Xu[m] @ np.array(mdl["beta_arx"]); e_arx = y[m] - f_arx
        d = e_ar ** 2 - e_arx ** 2; t, pv = dm_test(d); LOSS[(mdl["cause"], mdl["effect"])] = d
        oos.append(dict(cause=mdl["cause"], effect=mdl["effect"], n_test=int(m.sum()),
                        r2_oos_vs_ar=1 - np.sum(e_arx ** 2) / np.sum(e_ar ** 2), dm_t=t, dm_p=pv,
                        hit_arx=np.mean(np.sign(f_arx) == np.sign(y[m])), hit_ar=np.mean(np.sign(y[m] - e_ar) == np.sign(y[m]))))
    oos = pd.DataFrame(oos)
    scored = rep.merge(oos, on=["cause", "effect"], how="left") if len(rep) and len(oos) else rep
    scored.to_csv(OUT2 / f"granger_scored_{tag}.csv", index=False)
    print(f"[{tag}] UNSEALED: candidates replicated {n_rep}/{m_} (chance ≈ {0.025*m_:.1f}, binomial p={binom_p:.3g}) | "
          f"Spearman(train, test strength) = {rho:.3f} | blind discoveries also train-significant: {both}/{int(G_all.sig_te.sum())}")
    cs = ["cause", "effect", "p_tr", "p_val", "p_test", "same_sign", "replicated", "r2_oos_vs_ar", "dm_t", "dm_p", "hit_arx", "hit_ar"]
    if len(scored):
        print(scored[[c for c in cs if c in scored]].round(4).to_string(index=False))
    res.update(G_te=G_te, rep=rep, scored=scored, oos=oos, LOSS=LOSS, G_all=G_all, rho=rho, n_rep=n_rep,
               binom_p=binom_p, both=both)

In [ ]:
# step 8
# ======================= toolbox: Granger figures =======================
def draw_network(G_sig, title, name, part, names=True):
    if nx is None or G_sig.empty:
        print("network skipped:", "networkx missing" if nx is None else "no significant links"); return
    g = nx.DiGraph()
    for r in G_sig.itertuples():
        g.add_edge(r.cause, r.effect, w=r.partial_r2, s=np.sign(r.coef_sum))
    fig, ax = plt.subplots(figsize=(9, 6.5))
    pos = nx.spring_layout(g, seed=SEED, k=1.2 / np.sqrt(max(len(g), 1)))
    secs = sorted({SECTOR.get(n, "Other") for n in g.nodes}) if names else ["coded"]
    colmap = {s: SERIES[k % len(SERIES)] for k, s in enumerate(secs)}
    ncol = [colmap[SECTOR.get(n, "Other")] if names else SERIES[0] for n in g.nodes]
    deg = dict(g.degree())
    nx.draw_networkx_nodes(g, pos, node_color=ncol, node_size=[200 + 80 * deg[n] for n in g.nodes], ax=ax, edgecolors="#fcfcfb", linewidths=2)
    w = np.array([g[u][v]["w"] for u, v in g.edges]); wmax = w.max() if len(w) else 1
    ecol = [SERIES[0] if g[u][v]["s"] > 0 else SERIES[1] for u, v in g.edges]
    nx.draw_networkx_edges(g, pos, width=0.6 + 3 * w / wmax, edge_color=ecol, arrows=True, arrowsize=12, ax=ax,
                           connectionstyle="arc3,rad=0.08", node_size=300)
    nx.draw_networkx_labels(g, pos, font_size=8, font_color=INK, ax=ax)
    if names:
        for s, c in colmap.items():
            ax.scatter([], [], color=c, label=s)
    ax.plot([], [], color=SERIES[0], label="positive"); ax.plot([], [], color=SERIES[1], label="negative (reversal)")
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1), frameon=False)
    ax.set_title(title); ax.axis("off")
    savefig(fig, name, title, part)

def plot_granger_train(res, part):
    tag, G_tr = res["tag"], res["G_tr"]; cols_o = sector_order(res["cols"])
    M = -np.log10(to_matrix(G_tr, "q", cols_o).astype(float).clip(lower=1e-12))
    if not SHOW_NAMES_TRAIN:
        M.index = M.columns = [f"S{k:02d}" for k in range(len(cols_o))]
    n = len(cols_o)
    fig, ax = plt.subplots(figsize=(max(6, 0.2 * n + 3), max(5, 0.2 * n + 1)))
    heat(ax, M, vmin=0, vmax=max(3, np.nanpercentile(M.values, 99.5)), label="-log10 q (BH)")
    ax.set_xlabel("effect (predicted)"); ax.set_ylabel("cause (predictor)")
    ax.set_title(f"[{tag}] Train Granger -log10 q, p={res['p']} (q<{FDR_Q} ⇔ >{-np.log10(FDR_Q):.1f})")
    savefig(fig, f"{part}_{tag}_train_heat", f"[{tag}] Granger train heatmap", part)
    draw_network(G_tr[G_tr.sig], f"[{tag}] Train: significant links (edge width = partial R²)", f"{part}_{tag}_train_net", part, SHOW_NAMES_TRAIN)
    fig, axs = plt.subplots(1, 3, figsize=(16, 3.8))
    axs[0].hist(G_tr["p"].dropna(), bins=40, color=SERIES[0], edgecolor="#fcfcfb", linewidth=1)
    axs[0].axhline(res["n_pairs"] / 40, color=INK2, ls="--", lw=1)
    axs[0].set_xlabel("train p-value"); axs[0].set_title("p-values (flat = no links)")
    sens = res["sens"]
    axs[1].plot(sens.p, sens.jaccard_vs_base, marker="o", color=SERIES[0]); axs[1].set_ylim(0, 1.05)
    for r in sens.itertuples():
        axs[1].annotate(f"{r.n_sig}", (r.p, r.jaccard_vs_base), xytext=(4, 4), textcoords="offset points", fontsize=8, color=INK2)
    axs[1].set_xlabel("lag order p"); axs[1].set_title(f"Lag sensitivity: overlap with p={res['p']} (labels = # sig.)")
    G_va = res["G_va"]; ax = axs[2]
    if len(G_va):
        c = np.where(G_va.candidate, SERIES[0], np.where(G_va.same_sign, MUTE, SERIES[1]))
        ax.scatter(-np.log10(G_va.p_tr.astype(float)), -np.log10(G_va.p.astype(float)), c=c, s=26, edgecolors="#fcfcfb", linewidths=1)
        for r in res["CAND"].head(10).itertuples():
            ax.annotate(f"{r.cause}→{r.effect}" if SHOW_NAMES_TRAIN else "", (-np.log10(r.p_tr), -np.log10(r.p)), fontsize=7, color=INK2,
                        xytext=(3, 3), textcoords="offset points")
        for lbl, col in [("validated", SERIES[0]), ("not sig. in val", MUTE), ("sign flipped", SERIES[1])]:
            ax.scatter([], [], color=col, label=lbl)
        ax.legend(frameon=False, fontsize=7)
    ax.axhline(-np.log10(0.05), color=INK2, ls="--", lw=1)
    ax.set_xlabel("train -log10 p"); ax.set_ylabel("validation -log10 p"); ax.set_title("Train survivors in validation")
    plt.tight_layout()
    savefig(fig, f"{part}_{tag}_train_diag", f"[{tag}] p-values, lag sensitivity, validation", part)

def plot_granger_blind(res, part):
    tag, codes = res["tag"], res["codes"]
    M = -np.log10(to_matrix(res["G_bl"], "q", codes).astype(float).clip(lower=1e-12)); n = len(codes)
    fig, ax = plt.subplots(figsize=(max(6, 0.2 * n + 3), max(5, 0.2 * n + 1)))
    heat(ax, M, vmin=0, vmax=max(3, np.nanpercentile(M.values, 99.5)), label="-log10 q (BH)")
    ax.set_xlabel("effect (code)"); ax.set_ylabel("cause (code)"); ax.set_title(f"[{tag}] Blind test on coded series")
    savefig(fig, f"{part}_{tag}_blind_heat", f"[{tag}] Blind test heatmap (coded)", part)
    G_bl = res["G_bl"]
    draw_network(G_bl[G_bl.sig], f"[{tag}] Blind test: significant links (codes only)", f"{part}_{tag}_blind_net", part, names=False)

def plot_granger_score(res, part):
    tag, G_all, rep, oos, LOSS = res["tag"], res["G_all"], res["rep"], res["oos"], res["LOSS"]
    fig, axs = plt.subplots(1, 3, figsize=(16, 4.3))
    ax = axs[0]
    ax.scatter(-np.log10(G_all.p_tr), -np.log10(G_all.p_te), s=6, color=MUTE, label="all pairs")
    if len(rep):
        rr = G_all.merge(rep[["cause", "effect", "replicated"]], on=["cause", "effect"])
        ax.scatter(-np.log10(rr.p_tr), -np.log10(rr.p_te), s=34, color=np.where(rr.replicated, SERIES[0], SERIES[1]), edgecolors="#fcfcfb", linewidths=1)
        ax.scatter([], [], color=SERIES[0], label="candidate, replicated"); ax.scatter([], [], color=SERIES[1], label="candidate, failed")
        for r in rr.itertuples():
            ax.annotate(f"{r.cause}→{r.effect}", (-np.log10(r.p_tr), -np.log10(r.p_te)), fontsize=7, color=INK2, xytext=(3, 2), textcoords="offset points")
    ax.axhline(-np.log10(0.05), color=INK2, ls="--", lw=1)
    ax.set_xlabel("train -log10 p"); ax.set_ylabel("test -log10 p (unsealed)"); ax.legend(frameon=False, fontsize=7)
    ax.set_title(f"Train vs test strength (Spearman {res['rho']:.2f})")
    ax = axs[1]
    if len(oos):
        o = oos.sort_values("r2_oos_vs_ar")
        ax.barh([f"{a}→{b}" for a, b in zip(o.cause, o.effect)], o.r2_oos_vs_ar * 100, color=np.where(o.dm_p < 0.05, SERIES[0], MUTE),
                height=min(0.6, 0.15 * len(o) + 0.2))
        ax.axvline(0, color=INK2, lw=1); ax.set_xlabel("OOS R² of frozen ARX vs AR (%)")
        ax.scatter([], [], color=SERIES[0], marker="s", label="DM p<0.05"); ax.scatter([], [], color=MUTE, marker="s", label="not significant")
        ax.legend(frameon=False, fontsize=7)
    ax.set_title("Forecast value on test")
    ax = axs[2]
    for k_, (key, d) in enumerate(sorted(LOSS.items(), key=lambda kv: -np.nansum(kv[1]))[:8]):
        ax.plot(np.nancumsum(d), label=f"{key[0]}→{key[1]}", color=SERIES[k_ % 8], lw=1.4)
    ax.axhline(0, color=INK2, lw=1); ax.set_xlabel("test observation"); ax.set_ylabel("cum. (AR − ARX) sq. error")
    ax.set_title("Where the gain comes from (rising = ARX better)")
    if LOSS:
        ax.legend(frameon=False, fontsize=7)
    plt.tight_layout()
    savefig(fig, f"{part}_{tag}_replication", f"[{tag}] Replication and forecast value", part)

In [ ]:
# step 8
# ======================= toolbox: López de Prado tools =======================
def ffd_weights(d, thresh=FD_THRESH, max_len=2000):
    w = [1.0]; k = 1
    while k < max_len:
        w_ = -w[-1] * (d - k + 1) / k
        if abs(w_) < thresh:
            break
        w.append(w_); k += 1
    return np.array(w)

def frac_diff(x, d, thresh=FD_THRESH):
    # fixed-width-window fractional differentiation (AFML ch. 5); x = log price Series
    w = ffd_weights(d, thresh); L = len(w); v = x.to_numpy(float); out = np.full(len(v), np.nan)
    if len(v) >= L:
        out[L - 1:] = np.convolve(v, w, mode="valid")
    return pd.Series(out, x.index)

def cusum_events(lp, h):
    # symmetric CUSUM filter (AFML 2.5.2.1): sample an event when drift since the last event exceeds h_t
    diff = lp.diff().to_numpy(); hv = np.asarray(h, float); ev = []; sp = sn = 0.0
    for i in range(1, len(diff)):
        if not (np.isfinite(diff[i]) and np.isfinite(hv[i])):
            continue
        sp = max(0.0, sp + diff[i]); sn = min(0.0, sn + diff[i])
        if sp > hv[i]:
            sp = 0.0; ev.append(i)
        elif sn < -hv[i]:
            sn = 0.0; ev.append(i)
    return np.array(ev, int)

def triple_barrier(lp, t_idx, width, H):
    # first touch of +width / -width within H bars (AFML 3.4); else the sign of the return at the vertical barrier
    v = np.asarray(lp, float); n = len(v); m = len(t_idx)
    lab = np.full(m, np.nan); t1 = np.asarray(t_idx).copy(); ret = np.full(m, np.nan)
    for k, i in enumerate(t_idx):
        j_end = min(i + H, n - 1); path = v[i + 1:j_end + 1] - v[i]
        if len(path) == 0 or not np.isfinite(width[k]) or not np.isfinite(v[i]):
            continue
        up = np.flatnonzero(path >= width[k]); dn = np.flatnonzero(path <= -width[k])
        fu = up[0] if len(up) else 10 ** 9; fd = dn[0] if len(dn) else 10 ** 9
        if fu == fd == 10 ** 9:
            fin = path[np.isfinite(path)]
            if len(fin):
                lab[k] = np.sign(fin[-1]); ret[k] = fin[-1]; t1[k] = j_end
        elif fu < fd:
            lab[k] = 1; ret[k] = path[fu]; t1[k] = i + 1 + fu
        else:
            lab[k] = -1; ret[k] = path[fd]; t1[k] = i + 1 + fd
    return lab, t1, ret

def avg_uniqueness(t0, t1):
    # average uniqueness of overlapping labels (AFML 4.4), on one product's time axis
    if len(t0) == 0:
        return np.array([])
    c = np.zeros(int(np.max(t1)) + 2)
    for a, b in zip(t0, t1):
        c[a:b + 1] += 1
    return np.array([np.mean(1 / c[a:b + 1]) for a, b in zip(t0, t1)])

def purged_kfold(t0, t1, n_splits=N_FOLDS, embargo=EMBARGO_PCT):
    # purged k-fold with embargo (AFML 7.4); t0/t1 = event start/end positions on a common time axis
    t0 = np.asarray(t0); t1 = np.asarray(t1); order = np.argsort(t0, kind="mergesort")
    t0s, t1s = t0[order], t1[order]; emb = int((t1s.max() + 1) * embargo)
    out = []
    for f in np.array_split(np.arange(len(order)), n_splits):
        a, b = t0s[f[0]], t1s[f].max()
        tr = np.flatnonzero((t1s < a) | (t0s > b + emb))
        out.append((order[tr], order[f]))
    return out

def cluster_features(X, max_k=15):
    # correlation clustering of features (for clustered MDA, López de Prado 2020)
    C = pd.DataFrame(X).corr(method="spearman").fillna(0).values
    D = np.sqrt(np.clip(0.5 * (1 - C), 0, None)); np.fill_diagonal(D, 0)
    Lk = hierarchy.linkage(squareform(D, checks=False), "average"); best = (np.arange(1, X.shape[1] + 1), -1)
    for k in range(2, min(max_k, X.shape[1] - 1) + 1):
        lab = hierarchy.fcluster(Lk, k, "maxclust")
        if len(set(lab)) > 1:
            sc = silhouette_score(D, lab, metric="precomputed")
            if sc > best[1]:
                best = (lab, sc)
    return best[0]

def hrp_weights(cov):
    # hierarchical risk parity (López de Prado 2016)
    cov = np.asarray(cov, float); sd = np.sqrt(np.diag(cov)); corr = cov / np.outer(sd, sd)
    D = np.sqrt(np.clip(0.5 * (1 - corr), 0, None)); np.fill_diagonal(D, 0)
    order = list(hierarchy.leaves_list(hierarchy.linkage(squareform(D, checks=False), "single")))
    w = np.ones(len(cov)); clusters = [order]
    def cvar(c):
        sub = cov[np.ix_(c, c)]; ivp = 1 / np.diag(sub); ivp /= ivp.sum()
        return ivp @ sub @ ivp
    while clusters:
        clusters = [c[j:k] for c in clusters for j, k in ((0, len(c) // 2), (len(c) // 2, len(c))) if len(c) > 1]
        for i in range(0, len(clusters), 2):
            c0, c1 = clusters[i], clusters[i + 1]; v0, v1 = cvar(c0), cvar(c1); a = 1 - v0 / (v0 + v1)
            w[c0] *= a; w[c1] *= 1 - a
    return w / w.sum()

def pbo_cscv(M, S=PBO_S):
    # probability of backtest overfitting, combinatorially symmetric cross-validation (Bailey, Borwein, López de Prado, Zhu)
    M = np.asarray(M, float); T, N = M.shape; blocks = np.array_split(np.arange(T), S)
    sums = np.array([np.nansum(M[b], 0) for b in blocks]); sq = np.array([np.nansum(M[b] ** 2, 0) for b in blocks])
    cnt = np.array([np.isfinite(M[b]).sum(0) for b in blocks]); logits = []
    def sr(mask):
        n = np.maximum(cnt[mask].sum(0), 1); m = sums[mask].sum(0) / n; v = sq[mask].sum(0) / n - m ** 2
        return m / np.sqrt(np.maximum(v, 1e-18))
    for comb in itertools.combinations(range(S), S // 2):
        ins = np.zeros(S, bool); ins[list(comb)] = True
        best = int(np.nanargmax(sr(ins))); r = stats.rankdata(sr(~ins))[best] / (N + 1)
        logits.append(np.log(r / (1 - r)))
    logits = np.array(logits)
    return float(np.mean(logits <= 0)), logits

def bet_size(p):
    # bet size from a classifier probability (AFML 10.3)
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    return 2 * stats.norm.cdf((p - 0.5) / np.sqrt(p * (1 - p))) - 1

def icss_breaks(x, crit=ICSS_CRIT, min_seg=ICSS_MIN_SEG):
    # variance break points: binary segmentation with the kurtosis-robust Inclan-Tiao kappa-1 statistic (Sansó et al. 2004)
    x = np.asarray(x, float); x = x - np.mean(x); out = []
    def seg(a, b):
        e = x[a:b]; T = len(e)
        if T <= 2 * min_seg + 1:
            return
        e2 = e ** 2; C = np.cumsum(e2); k = np.arange(1, T + 1); s2 = C[-1] / T; w4 = np.mean(e2 ** 2) - s2 ** 2
        if w4 <= 0:
            return
        B = (C - k / T * C[-1]) / np.sqrt(T * w4)
        j = int(np.argmax(np.abs(B[min_seg:T - min_seg]))) + min_seg
        if abs(B[j]) > crit:
            out.append(a + j + 1); seg(a, a + j + 1); seg(a + j + 1, b)
    seg(0, len(x))
    return sorted(out)

def rolling_tstat(Y, n):
    # t-stat of the slope of Y on time over a rolling window of n bars (trend strength)
    tt = pd.Series(np.arange(len(Y), dtype=float), index=Y.index)
    mt = tt.rolling(n).mean(); vt = tt.rolling(n).var(ddof=0)
    my = Y.rolling(n).mean(); mty = Y.mul(tt, axis=0).rolling(n).mean(); vy = Y.rolling(n).var(ddof=0)
    b = (mty - my.mul(mt, axis=0)).div(vt, axis=0)
    res = (vy - (b ** 2).mul(vt, axis=0)).clip(lower=1e-18) * n / (n - 2)
    return b / np.sqrt(res.div(vt * n, axis=0))

def xs_ic_fast(F, Y, min_n=None):
    # cross-sectional Spearman IC per row (vectorised)
    min_n = MIN_XS if min_n is None else min_n
    m = F.notna() & Y.notna(); Fr = F.where(m).rank(axis=1); Yr = Y.where(m).rank(axis=1)
    a = Fr.sub(Fr.mean(axis=1), axis=0); b = Yr.sub(Yr.mean(axis=1), axis=0)
    ic = (a * b).sum(axis=1) / np.sqrt((a * a).sum(axis=1) * (b * b).sum(axis=1))
    return ic[m.sum(axis=1) >= min_n].replace([np.inf, -np.inf], np.nan).dropna()

def ts_ic_pooled(F, Y, max_n=300_000):
    # pooled Spearman correlation over all (time, product) cells
    s = pd.concat([F.stack(), Y.stack()], axis=1).dropna()
    if len(s) > max_n:
        s = s.sample(max_n, random_state=SEED)
    return s.iloc[:, 0].corr(s.iloc[:, 1], method="spearman") if len(s) > 50 else np.nan

def ic_t(ic, h=1):
    ic = pd.Series(ic).dropna()
    return ic.mean() / ic.std() * np.sqrt(len(ic) / h) if len(ic) > 10 and ic.std() > 0 else np.nan

In [ ]:
# step 8
# ======================= toolbox: multi-time-frame data =======================
def to_trading_day(idx):
    # night bars (20:00-04:00) belong to the next day session, as in Step 1
    idx = pd.DatetimeIndex(idx); h = idx.hour; night = (h >= 20) | (h < 4)
    return session_day(idx)                     # calendar rule of Step 0c (no backfill from the next bar)

_P5 = {}
def load_p5():
    if not _P5:
        p = CACHE / "bars5m.parquet"
        if p.exists():
            b = pd.read_parquet(p); b = b[~b["product"].isin(EXCLUDE)]
            for c in ["ret", "rv2", "volume", "oi"]:
                _P5[c] = b.pivot(index="date", columns="product", values=c).sort_index().astype(float)
            del b
            for c in ("ret", "rv2"):                   # data-quality rule (Step 2f): a bar without volume is not a return
                _P5[c] = _P5[c].where(_P5["volume"] > 0)
    return _P5

def tf_frames(tf):
    # wide frames (ret, rv2, volume, oi) at time frame tf: intraday from 5-min bars, 1D/1W from the daily panel
    if tf in ("1D", "1W"):
        F = {"ret": RAW, "rv2": RV ** 2, "volume": VOLU, "oi": OIW}
        if tf == "1W":
            F = {"ret": RAW.resample("W-FRI").sum(min_count=1), "rv2": (RV ** 2).resample("W-FRI").sum(min_count=1),
                 "volume": VOLU.resample("W-FRI").sum(min_count=1), "oi": OIW.resample("W-FRI").last()}
        return {k: v.copy() for k, v in F.items()}
    P = load_p5()
    if not P:
        return None
    if tf == "5m":
        F = {k: v.copy() for k, v in P.items()}
    else:
        rule = f"{TF_MIN[tf]}min"
        F = {c: P[c].resample(rule, label="right", closed="right").sum(min_count=1) for c in ("ret", "rv2", "volume")}
        F["oi"] = P["oi"].resample(rule, label="right", closed="right").last()
    keep = F["ret"].notna().any(axis=1); F = {k: v[keep] for k, v in F.items()}
    R = F["ret"]
    for c in R.columns:                       # drop session-open bars: they carry the overnight / lunch gap
        s = R[c].dropna(); gap = (s.index.to_series().diff() > pd.Timedelta(minutes=TF_MIN[tf] + 10)).to_numpy()
        R.loc[s.index[gap], c] = np.nan
    return F

def split_index(idx, emb=EMB):
    # ONE calendar for every frequency: the boundaries are the dates of the daily split (Step 9), so a 5-minute bar, a daily bar and a weekly bar
    # that fall on the same date are always in the same period. Fractions of each frame's own length would put the validation / test dates of
    # one frame inside the training period of another (the frames start at different dates) and leak the test period through cross-frame steps.
    idx = pd.DatetimeIndex(idx)
    tr = idx[idx <= SPL["train"][-1]]; va = idx[(idx >= SPL["val"][0]) & (idx <= SPL["val"][-1])]; te = idx[idx >= SPL["test"][0]]
    return {"train": tr[:len(tr) - emb] if emb and len(tr) > 3 * emb else tr, "val": va[:len(va) - emb] if emb and len(va) > 3 * emb else va, "test": te}

def tf_z(F, tf):
    # intraday: divide by the train time-of-day volatility profile; then ex-ante EWMA vol scaling
    R = F["ret"]; itr = split_index(R.index)["train"]
    if tf in ("1D", "1W"):
        hl = VOL_HL if tf == "1D" else 4
    else:
        tod = pd.Index(R.index.hour * 60 + R.index.minute); mtr = R.index.isin(itr)
        seas = R[mtr].abs().groupby(tod[mtr]).mean(); seas = seas / seas.mean()
        R = R / seas.reindex(tod).set_axis(R.index)
        bpd = R.notna().any(axis=1).groupby(to_trading_day(R.index)).sum().median()
        hl = max(int(VOL_HL_DAYS * bpd), 20)
    return (R / R.ewm(halflife=hl, min_periods=min(hl, 100)).std().shift(1)).clip(-CLIP_Z, CLIP_Z)

def tod_bucket(idx):
    m = np.asarray(idx.hour * 60 + idx.minute); m = np.where(m < 4 * 60, m + 24 * 60, m)
    out = np.full(len(m), "other", dtype=object)
    for name, a, b in TOD_BUCKETS:
        out[(m > a) & (m <= b)] = name
    return out

## Step 9 — Data, splits, stationarity and fractional differentiation

**Used later.** `SPL` (one calendar for all frequencies) and the stationary variable (return or fractionally differenced price) are used by every step from 10 to 26. `UNIV` defines the universe.

In [ ]:
# step 9
# ======================= data =======================
CACHE = OUT_DIR
dp = pd.read_parquet(CACHE / "daily_panel.parquet")
wide = lambda col: dp.pivot(index="trading_day", columns="product", values=col).sort_index()
RAW = wide("ret")
RAW = RAW.drop(columns=[c for c in EXCLUDE if c in RAW.columns]).replace([np.inf, -np.inf], np.nan)
SIMPLE = np.expm1(RAW)                                        # simple returns (P&L, portfolios)
RV = wide("rv").reindex_like(RAW)                             # realised vol from 1-min returns (log units)
CLOSE = wide("close").reindex_like(RAW)
ROLL = wide("roll").reindex_like(RAW).astype(float).fillna(0).astype(bool)
VOLU = wide("volume").reindex_like(RAW); OIW = wide("oi").reindex_like(RAW)
SEC_MAP = pd.Series({p: SECTOR.get(p, "Other") for p in RAW.columns})

# ex-ante vol scaling: sigma_t uses returns up to t-1 only
sig = RAW.ewm(halflife=VOL_HL, min_periods=20).std().shift(1)
Z = (RAW / sig).clip(-CLIP_Z, CLIP_Z)

dates = Z.index[Z.notna().sum(axis=1) >= 3]
n = len(dates); i1, i2 = int(n * SPLIT[0]), int(n * (SPLIT[0] + SPLIT[1]))
SPL = {"train": dates[:i1], "val": dates[i1 + EMB:i2], "test": dates[i2 + EMB:]}
counts = pd.DataFrame({k: Z.loc[v].notna().sum() for k, v in SPL.items()})
UNIV = counts.index[(counts >= MIN_OBS_SPLIT).all(axis=1)].tolist()
dropped = counts.index.difference(UNIV).tolist()
Ztr, Zva, Zte = (Z.loc[SPL[k], UNIV] for k in ("train", "val", "test"))
_i0 = pd.Timestamp(INTRADAY_START)
print(f"calendar splits (used by EVERY frequency): train to {SPL['train'][-1].date()} | validation {SPL['val'][0].date()} to {SPL['val'][-1].date()} | test from {SPL['test'][0].date()}")
if _i0 > SPL["train"][-1] - pd.Timedelta(days=365):
    print(f"WARNING: 5-minute bars start on {_i0.date()} but the training period ends on {SPL['train'][-1].date()}: intraday frames have little or no training data. "
          f"Set INTRADAY_START (Step 0) to about {(SPL['train'][-1] - pd.Timedelta(days=3*365)).date()} or earlier and re-run Step 1 (more memory), or drop the intraday frames.")
split_info = {k: {"start": str(v[0].date()), "end": str(v[-1].date()), "n": len(v)} for k, v in SPL.items()}
split_info.update(universe=UNIV, dropped_short_history=dropped, embargo=EMB)
json.dump(split_info, open(OUT2 / "splits.json", "w"), indent=2, default=str)
print(pd.DataFrame({k: split_info[k] for k in SPL}).T)
print(f"\nuniverse: {len(UNIV)} series | dropped (too few obs in some period): {dropped}")

avail = Z[UNIV].notna()
fig, ax = plt.subplots(figsize=(11, max(3, 0.16 * len(UNIV))))
order = avail.idxmax().sort_values().index
for k, c in enumerate(order):
    ax.fill_between(avail.index, k - 0.35, k + 0.35, where=avail[c].values, color=SERIES[0], step="mid", linewidth=0)
for name, col in zip(SPL, SERIES[:3]):
    ax.axvspan(SPL[name][0], SPL[name][-1], color=col, alpha=0.08)
    ax.text(SPL[name][0], len(order) + 0.3, f" {name} (n={len(SPL[name])})", color=INK2, fontsize=8, va="bottom")
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=6); ax.set_ylim(-1, len(order) + 1.5); ax.grid(False)
ax.set_title("Coverage per series with train / validation / test (gaps = embargo)")
savefig(fig, "9_coverage", "Coverage and splits", "9")

### 9.1 — Stationarity: which variable is the right one to model?
* **ADF** (null: unit root) and **KPSS** (null: stationary) are run on the train period. The two together give a verdict: *stationary* (ADF rejects, KPSS does not), *unit root* (the reverse), or *inconclusive*.
* The roll-adjusted **log price** should come out as a unit root. That is why ARIMA uses d = 1 on log prices, which is the same thing as an ARMA on log returns (Step 10).
* **Log** and **simple** returns should both come out stationary, so either can be modelled. The models use log returns; P&L uses simple returns.

In [ ]:
# step 9
st_rows = []
for s in UNIV:
    r = RAW.loc[SPL["train"], s].dropna()
    for name, x, reg in [("log price (roll-adj.)", r.cumsum(), "ct"), ("log return", r, "c"), ("simple return", np.expm1(r), "c")]:
        try:
            adf_p = adfuller(x.values, autolag="AIC", regression=reg)[1]
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                kpss_p = kpss(x.values, regression=reg, nlags="auto")[1]
        except Exception:
            adf_p = kpss_p = np.nan
        st_rows.append(dict(series=s, variable=name, adf_p=adf_p, kpss_p=kpss_p))
ST = pd.DataFrame(st_rows)
ST["verdict"] = np.select([(ST.adf_p < 0.05) & (ST.kpss_p > 0.05), (ST.adf_p >= 0.05) & (ST.kpss_p <= 0.05)],
                          ["stationary", "unit root"], "inconclusive")
ST.to_csv(OUT2 / "stationarity.csv", index=False)
tab = pd.crosstab(ST.variable, ST.verdict).reindex(columns=["stationary", "inconclusive", "unit root"], fill_value=0)
print(tab.to_string())
print("\nKPSS p-values are bounded to [0.01, 0.10] by the lookup table, so 0.1 means ≥ 0.1.")

fig, ax = plt.subplots(figsize=(8, 2.6))
left = np.zeros(len(tab))
for v, c in zip(tab.columns, [SERIES[0], MUTE, SERIES[1]]):
    ax.barh(tab.index, tab[v], left=left, color=c, label=v, height=0.55, edgecolor="#fcfcfb", linewidth=2); left += tab[v].values
ax.set_xlabel("number of series"); ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower right", bbox_to_anchor=(1, 1))
ax.set_title("ADF + KPSS verdicts on the train period")
savefig(fig, "9_stationarity", "Stationarity verdicts", "9")

### 9.2 — Fractional differentiation (López de Prado, AFML ch. 5)
Returns (d = 1) are stationary but have thrown away all memory of the price level. Prices (d = 0) keep the memory but are not stationary. For each product this finds the **smallest d\*** at which the fixed-width fractionally differenced log price passes ADF on train. That series is the most memory-preserving stationary version of the price, and Step 18 uses it as a mean-reversion feature (`mr_fd`). A high correlation with the price at d\* means a lot of memory was preserved.

In [ ]:
# step 9
LP = RAW.fillna(0).cumsum().where(RAW.notna().cumsum() > 0)          # roll-adjusted log price
fd_rows, FD = [], {}
for s in UNIV:
    lp = LP[s].dropna(); dstar, corr = np.nan, np.nan
    for d in FD_GRID:
        fd = frac_diff(lp, d).dropna(); fdt = fd[fd.index <= SPL["train"][-1]]
        if len(fdt) < 100:
            continue
        if adfuller(fdt.values, maxlag=1, regression="c", autolag=None)[1] < 0.05:
            dstar = d; corr = np.corrcoef(fd.values, lp.reindex(fd.index).values)[0, 1]; FD[s] = frac_diff(lp, d); break
    fd_rows.append(dict(series=s, d_star=dstar, corr_with_price=corr, window=len(ffd_weights(dstar)) if np.isfinite(dstar) else np.nan))
FDT = pd.DataFrame(fd_rows).set_index("series"); FD = pd.DataFrame(FD).reindex(index=RAW.index)
FDT.to_csv(OUT2 / "fracdiff.csv")
print(f"d* (median) = {FDT.d_star.median():.2f}; memory kept (corr with log price, median) = {FDT.corr_with_price.median():.2f}")
print(FDT.round(3).sort_values("d_star").to_string())
fig, axs = plt.subplots(1, 2, figsize=(13, 3.6))
o = FDT.sort_values("d_star")
axs[0].bar(o.index, o.d_star, color=SERIES[0]); axs[0].tick_params(axis="x", rotation=90, labelsize=7)
axs[0].set_ylabel("d*"); axs[0].set_title("Minimum differencing for stationarity (1 = returns)")
axs[1].scatter(FDT.d_star, FDT.corr_with_price, color=SERIES[0], s=28, edgecolors="#fcfcfb", linewidths=1)
for s_, r in FDT.iterrows():
    axs[1].annotate(s_, (r.d_star, r.corr_with_price), fontsize=6, color=INK2, xytext=(2, 2), textcoords="offset points")
axs[1].set_xlabel("d*"); axs[1].set_ylabel("corr(fractionally differenced, log price)"); axs[1].set_title("Memory preserved at d*")
plt.tight_layout()
savefig(fig, "09_fracdiff", "Fractional differentiation", "9")

## Step 10 — Mean models: error-minimising regressions, ARMA / ARIMA

Fitted on **train**, hyper-parameters chosen on **validation**, **test** scored once; all models are evaluated on the same rows. The target is the vol-scaled log return `Z`.

| Model | Loss minimised | Regressors |
|---|---|---|
| `zero` | — | benchmark: forecast 0 |
| `mean` | squared error | train mean |
| `ar` | squared error (OLS) | own lags 1..p, p ≤ 5 by BIC |
| `arma11` | Gaussian likelihood (≈ squared error) | **ARMA(1,1) on log returns ≡ ARIMA(1,1,1) on log price** (d = 1 from Step 9.1) |
| `arma_bic` | same | ARMA(p,q), (p,q) from `ARMA_GRID` by BIC |
| `huber` | Huber loss (robust to outliers) | own lags + market lags |
| `lad` | absolute error (median regression) | own lags + market lags |
| `ridge` | squared error + L2, α on validation | lags 1–2 of **all** series |
| `lasso` | squared error + L1, α on validation | lags 1–2 of all series (selects predictors) |

$R^2_{oos} = 1-\sum(y-\hat y)^2/\sum y^2$ against the zero forecast. For daily futures, +0.5% is economically meaningful.

**Used later.** The mean-model results say how much linear own-series predictability exists (little), which is why Step 14 tests links *between* series and Step 18 predicts through cross-sectional ranks.

In [ ]:
# step 10
BLK = {k: D.to_numpy(float) for k, D in zip(["train", "val", "test"], [Ztr, Zva, Zte])}
with np.errstate(all="ignore"):
    MKT = {k: np.nanmean(A, axis=1) for k, A in BLK.items()}
LALL = {k: np.nan_to_num(np.hstack([lagmat(A[:, i], P_ALL) for i in range(A.shape[1])])) for k, A in BLK.items()}

def X_of(k, j, kind, p):
    y = BLK[k][:, j]
    if kind == "own":
        return lagmat(y, p)
    if kind == "own+mkt":
        return np.hstack([lagmat(y, p), lagmat(MKT[k], P_ALL)])
    return LALL[k]

def mask_of(k, j):
    y = BLK[k][:, j]
    return np.isfinite(y) & np.isfinite(lagmat(y, 5)).all(1) & np.isfinite(lagmat(MKT[k], P_ALL)).all(1)

def metr(y, f):
    return dict(mse=np.mean((y - f) ** 2), mae=np.mean(np.abs(y - f)), r2=1 - np.sum((y - f) ** 2) / np.sum(y ** 2))

ALPHAS_R = [1, 10, 100, 1000, 10000]; ALPHAS_L = [0.001, 0.003, 0.01, 0.03, 0.1]
rows, ARMA_ORD = [], {}
t0 = time.time()
for j, s in enumerate(UNIV):
    M = {k: mask_of(k, j) for k in BLK}; Y = {k: BLK[k][:, j] for k in BLK}
    nf = lambda k, kind, p: np.nan_to_num(X_of(k, j, kind, p))
    P = {}
    bics = {}
    for p in range(1, 6):
        X = X_of("train", j, "own", p)[M["train"]]; y = Y["train"][M["train"]]
        rss, _ = ols(np.c_[np.ones(len(X)), X], y); bics[p] = len(y) * np.log(rss / len(y)) + (p + 1) * np.log(len(y))
    p_ar = min(bics, key=bics.get)
    _, b = ols(np.c_[np.ones(M["train"].sum()), X_of("train", j, "own", p_ar)[M["train"]]], Y["train"][M["train"]])
    P["ar"] = {k: np.c_[np.ones(len(Y[k])), nf(k, "own", p_ar)] @ b for k in BLK}
    mu = Y["train"][M["train"]].mean()
    P["mean"] = {k: np.full(len(Y[k]), mu) for k in BLK}; P["zero"] = {k: np.zeros(len(Y[k])) for k in BLK}
    arma = {}
    for o in ARMA_GRID:
        try:
            arma[o] = arma_fit(Y["train"], o)
        except Exception:
            pass
    if arma:
        best_o = min(arma, key=lambda o: arma[o].bic); ARMA_ORD[s] = best_o
        for name, o in [("arma11", (1, 1)), ("arma_bic", best_o)]:
            if o in arma:
                P[name] = {"train": np.asarray(arma[o].fittedvalues), "val": arma_pred(arma[o], Y["val"]), "test": arma_pred(arma[o], Y["test"])}
    X = X_of("train", j, "own+mkt", p_ar)[M["train"]][-ROBUST_MAX_N:]; y = Y["train"][M["train"]][-ROBUST_MAX_N:]
    hub = HuberRegressor(epsilon=1.35, alpha=1e-4, max_iter=500).fit(X, y)
    lad = QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs").fit(X, y)
    P["huber"] = {k: hub.predict(nf(k, "own+mkt", p_ar)) for k in BLK}
    P["lad"] = {k: lad.predict(nf(k, "own+mkt", p_ar)) for k in BLK}
    Xt, yt = LALL["train"][M["train"]], Y["train"][M["train"]]; Xv, yv = LALL["val"][M["val"]], Y["val"][M["val"]]
    lasso_pick = ""
    for name, grid in [("ridge", ALPHAS_R), ("lasso", ALPHAS_L)]:
        best = None
        for a in grid:
            mdl = Lasso(alpha=a, max_iter=5000).fit(Xt, yt) if name == "lasso" else Ridge(alpha=a).fit(Xt, yt)
            v = np.mean((yv - mdl.predict(Xv)) ** 2)
            if best is None or v < best[0]:
                best = (v, a, mdl)
        P[name] = {k: best[2].predict(LALL[k]) for k in BLK}
        if name == "lasso":
            lasso_pick = ";".join(f"{UNIV[k // P_ALL]}_L{k % P_ALL + 1}" for k in np.flatnonzero(best[2].coef_)[:10])
    for name, pr in P.items():
        for k in BLK:
            m = M[k] & np.isfinite(pr[k])
            rows.append(dict(series=s, model=name, split=k, n=int(m.sum()), p_ar=p_ar, arma_order=str(ARMA_ORD.get(s)),
                             lasso_selected=lasso_pick if name == "lasso" else "", **metr(Y[k][m], pr[k][m])))
SCORES = pd.DataFrame(rows); SCORES.to_csv(OUT2 / "model_scores.csv", index=False)
print(f"{len(UNIV)} series × {SCORES.model.nunique()} models in {time.time()-t0:.0f}s")
R2 = SCORES.pivot_table(index="series", columns=["split", "model"], values="r2")
summ = SCORES.groupby(["model", "split"]).r2.median().unstack()[["train", "val", "test"]] * 100
summ["share_test_r2>0"] = (R2["test"] > 0).mean() * 100
print("\nMedian R²_oos across series (%), and % of series with positive test R²:")
print(summ.round(3).sort_values("val", ascending=False).to_string())
best_by_val = SCORES[SCORES.split == "val"].loc[lambda d: d.groupby("series").mse.idxmin()][["series", "model"]]
best_by_val["test_r2"] = [R2.loc[s, ("test", m)] for s, m in best_by_val.values]
print("\nModel chosen by validation MSE:", best_by_val.model.value_counts().to_dict())
print("ARMA orders chosen by BIC:", pd.Series({k: str(v) for k, v in ARMA_ORD.items()}).value_counts().to_dict())

In [ ]:
# step 10
MODELS = ["zero", "mean", "ar", "arma11", "arma_bic", "huber", "lad", "ridge", "lasso"]
MODELS = [m for m in MODELS if m in SCORES.model.unique()]
MCOL = dict(zip(MODELS, [MUTE] + SERIES))
fig, axs = plt.subplots(1, 2, figsize=(14, max(4, 0.17 * len(UNIV))), gridspec_kw={"width_ratios": [1.2, 1]})
T2 = (R2["test"][MODELS] * 100).loc[sector_order(UNIV)]
heat(axs[0], T2, cmap=DIV, center=0, vmax=max(1, np.nanpercentile(np.abs(T2.values), 97)), label="test R²_oos (%)")
axs[0].set_title("Test R²_oos by series and model (orange = beats zero)")
ax = axs[1]
data = [R2[sp][m].dropna() * 100 for m in MODELS for sp in ("val", "test")]
pos = [k * 3 + o for k in range(len(MODELS)) for o in (0, 1)]
bp = ax.boxplot(data, positions=pos, widths=0.8, patch_artist=True, showfliers=False, vert=False)
for k, b_ in enumerate(bp["boxes"]):
    b_.set_facecolor(SERIES[0] if k % 2 == 0 else SERIES[1]); b_.set_edgecolor("#fcfcfb")
for med in bp["medians"]:
    med.set_color(INK)
ax.set_yticks([k * 3 + 0.5 for k in range(len(MODELS))]); ax.set_yticklabels(MODELS)
ax.axvline(0, color=INK2, lw=1); ax.set_xlabel("R²_oos (%) across series")
ax.scatter([], [], color=SERIES[0], marker="s", label="validation"); ax.scatter([], [], color=SERIES[1], marker="s", label="test")
ax.legend(frameon=False, fontsize=8); ax.set_title("Distribution of R²_oos across series")
plt.tight_layout()
savefig(fig, "10_model_r2", "Mean models: R² by series", "10")

fig, ax = plt.subplots(figsize=(11, 3.5))
b_ = best_by_val.sort_values("test_r2")
ax.bar(b_.series, b_.test_r2 * 100, color=[MCOL[m] for m in b_.model], width=0.7)
for m in b_.model.unique():
    ax.bar([], [], color=MCOL[m], label=m)
ax.axhline(0, color=INK2, lw=1); ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.set_ylabel("test R²_oos (%)"); ax.legend(frameon=False, fontsize=8, ncol=9)
ax.set_title("Honest result: test R² of the model each series would have picked on validation")
savefig(fig, "10_best_by_val", "Best-by-validation model, test R²", "10")

## Step 11 — Volatility models

**Do we need a volatility model, and which kind?**
* **ARCH-LM** (Engle) on raw train returns: is there volatility clustering at all?
* **Sign-bias test** (Engle–Ng) on GARCH(1,1) residuals: do negative and positive shocks move volatility differently? If so, an asymmetric model (GJR / EGARCH / TARCH / APARCH) is needed.
* **Leverage sign.** GJR γ > 0 means falls raise vol more than rises (equity-like). γ < 0 means *inverse leverage* (rallies raise vol), which is common in commodities in supply squeezes.

**Candidates.** Every model is fitted by maximum likelihood on train with `mu` constant. Their conditional variances on validation and test use the frozen parameters, so they are true one-step-ahead forecasts.

| Model | What it adds |
|---|---|
| EWMA (λ = 0.94) | RiskMetrics: no estimation |
| GARCH(1,1) normal / t / skew-t | clustering; fat-tailed or skewed innovations |
| GJR-GARCH t / GED | asymmetric response to negative shocks (threshold on squared shocks) |
| EGARCH t | asymmetry in log-variance; no positivity constraints |
| TARCH (Zakoian) t | asymmetry on absolute shocks (power 1) |
| APARCH t | estimates the power δ as well as the asymmetry |
| FIGARCH t | long memory in volatility (hyperbolic decay) |
| HAR-RV | Corsi: daily / weekly / monthly **realised** variance from 1-min data |

**Evaluation** is against the realised-variance proxy from 1-min returns:
* **QLIKE** loss: robust to proxy noise, and it penalises under-prediction.
* **MSE.**
* **Mincer–Zarnowitz** regression $RV = a + b\,\hat h$ (an unbiased forecast has a = 0, b = 1).
* **Diebold–Mariano** of the validation-selected model against GARCH-n.

Selection is by **validation QLIKE**. The chosen model's standardised residuals feed Steps 12, 13 and 25, and its σ drives the volatility regimes in Step 20.

**Used later.** `BEST_VOL` (made robust in Step 11b) standardises the residuals used by Steps 12–13 and the VaR / ES models of Step 25. The conditional variances also give the volatility scaling of the risk tests.

In [ ]:
# step 11
def qlike(rv, h):
    x = rv / h
    return x - np.log(x) - 1

def sign_bias_p(z):
    z = np.asarray(z, float); z = z[np.isfinite(z)]
    y = z[1:] ** 2; zl = z[:-1]; Sm = (zl < 0).astype(float)
    X = np.c_[np.ones(len(y)), Sm, Sm * zl, (1 - Sm) * zl]
    rss_u, _ = ols(X, y); rss_r = float(((y - y.mean()) ** 2).sum()); n_, k_ = X.shape
    F = ((rss_r - rss_u) / 3) / (rss_u / (n_ - k_))
    return stats.f.sf(F, 3, n_ - k_)

train_end = SPL["train"][-1]
VOLFIT, vrows, drows = {}, [], []
VOL_FAIL, VOL_RETRY, VOL_ERR = {}, {}, {}
def fit_arch_robust(y, kw, n_fit):
    # maximum likelihood with up to 3 attempts: (1) per-cent returns as they are, (2) returns scaled to unit variance, (3) scaled again with a tighter tolerance.
    # Variance, mean and log-likelihood are mapped back to per-cent units, so all models stay comparable.
    sd = float(y.iloc[:n_fit].std())
    for k_, (sc, opts, extra) in enumerate(((1.0, dict(maxiter=1000), {}), (1.0 / sd, dict(maxiter=2000), {}), (1.0 / sd, dict(maxiter=3000, ftol=1e-9), {"tol": 1e-9}))):
        try:
            am = arch_model(y * sc, mean="Constant", rescale=False, **kw)
            r = am.fit(last_obs=n_fit, disp="off", options=opts, **extra)
            h = am.fix(r.params).conditional_volatility ** 2 / sc ** 2
            if not np.isfinite(h.iloc[n_fit:]).all():
                raise ValueError("non-finite variance")
            if k_ > 0:
                VOL_RETRY[kw.get("vol", "?")] = VOL_RETRY.get(kw.get("vol", "?"), 0) + 1
            return dict(h=h, mu=float(r.params["mu"]) / sc, params=r.params, tvals=r.tvalues, ll=r.loglikelihood + n_fit * np.log(sc),
                        k=len(r.params), conv=int(r.convergence_flag))
        except Exception as e_:
            VOL_ERR[kw.get("vol", "?")] = f"{type(e_).__name__}: {e_}"[:200]      # the last error is shown below if a model cannot be fitted
    return None
t0 = time.time()
for s in UNIV:
    y = RAW[s].dropna() * 100                                  # % returns
    n_fit = int((y.index <= train_end).sum())
    if n_fit < 300:
        continue
    rvp = (RV[s].reindex(y.index) * 100) ** 2                  # realised variance proxy, %²
    fits = {}
    if arch_model is not None:
        for name, kw in VOL_SPECS.items():
            if name in SLOW_VOL and not RUN_SLOW_VOL:
                continue
            res_ = fit_arch_robust(y, kw, n_fit)
            if res_ is None:
                VOL_FAIL[name] = VOL_FAIL.get(name, 0) + 1
            else:
                fits[name] = res_
    mu_tr = float(y.iloc[:n_fit].mean()); e2 = ((y - mu_tr) ** 2).to_numpy()
    h = np.empty(len(y)); h[0] = e2[:n_fit].mean()
    for t in range(1, len(y)):
        h[t] = EWMA_LAMBDA * h[t - 1] + (1 - EWMA_LAMBDA) * e2[t - 1]
    fits["EWMA"] = dict(h=pd.Series(h, y.index), mu=mu_tr, ll=np.nan, k=0, conv=0)
    lrv = np.log(rvp.where(rvp > 0))
    Xh = pd.concat({"d": lrv.shift(1), "w": lrv.rolling(5).mean().shift(1), "m": lrv.rolling(22).mean().shift(1)}, axis=1)
    mt = (Xh.notna().all(1) & lrv.notna() & (y.index <= train_end)).to_numpy()
    if mt.sum() > 100:
        _, bhar = ols(np.c_[np.ones(mt.sum()), Xh.values[mt]], lrv.values[mt])
        fit_ = np.c_[np.ones(len(Xh)), Xh.values] @ bhar
        hr = np.exp(fit_ + np.var(lrv.values[mt] - fit_[mt]) / 2)
        scale = np.nanmean(e2[:n_fit]) / np.nanmean(hr[:n_fit])   # map realised-variance forecast to return variance
        fits["HAR-RV"] = dict(h=pd.Series(hr * scale, y.index).ffill(), mu=mu_tr, ll=np.nan, k=4, conv=0, har_beta=bhar)
    VOLFIT[s] = fits
    for name, F in fits.items():
        row = dict(series=s, model=name, ll=F["ll"], k=F["k"], conv=F["conv"])
        if np.isfinite(F["ll"]):
            row.update(aic=2 * F["k"] - 2 * F["ll"], bic=F["k"] * np.log(n_fit) - 2 * F["ll"])
        for sp in ("train", "val", "test"):
            idx = y.index.intersection(SPL[sp]); pv = rvp.reindex(idx); hh = F["h"].reindex(idx)
            ok = (pv > 0) & (hh > 0) & hh.notna()
            row[f"qlike_{sp}"] = qlike(pv[ok], hh[ok]).mean(); row[f"mse_{sp}"] = ((pv[ok] - hh[ok]) ** 2).mean()
            if sp == "test" and ok.sum() > 30:
                X1 = np.c_[np.ones(ok.sum()), hh[ok].values]; rss, bmz = ols(X1, pv[ok].values)
                row.update(mz_a=bmz[0], mz_b=bmz[1], mz_r2=1 - rss / np.sum((pv[ok].values - pv[ok].values.mean()) ** 2))
        vrows.append(row)
    # diagnostics
    z_tr = (y.iloc[:n_fit] - mu_tr).to_numpy()
    d = dict(series=s, sector=SECTOR.get(s, "Other"), archlm_p_raw=het_arch(z_tr, nlags=5)[1])
    if "GARCH-n" in fits:
        zg = ((y - fits["GARCH-n"]["mu"]) / np.sqrt(fits["GARCH-n"]["h"])).iloc[:n_fit].to_numpy()
        d.update(archlm_p_garch=het_arch(zg, nlags=5)[1], sign_bias_p=sign_bias_p(zg))
    if "GJR-t" in fits:
        pr, tv = fits["GJR-t"]["params"], fits["GJR-t"]["tvals"]
        pers = pr["alpha[1]"] + pr["beta[1]"] + pr["gamma[1]"] / 2
        d.update(gjr_gamma=pr["gamma[1]"], gjr_gamma_t=tv["gamma[1]"], persistence=pers,
                 half_life_days=np.log(0.5) / np.log(pers) if 0 < pers < 1 else np.inf, nu_t=pr.get("nu", np.nan))
    if "EGARCH-t" in fits:
        d.update(egarch_gamma=fits["EGARCH-t"]["params"]["gamma[1]"], egarch_gamma_t=fits["EGARCH-t"]["tvals"]["gamma[1]"])
    if "FIGARCH-t" in fits:
        d.update(figarch_d=fits["FIGARCH-t"]["params"].get("d", np.nan))
    if "APARCH-t" in fits:
        d.update(aparch_delta=fits["APARCH-t"]["params"].get("delta", np.nan))
    drows.append(d)
VOLS = pd.DataFrame(vrows); VDIAG = pd.DataFrame(drows).set_index("series")
print(f"volatility models fitted in {time.time()-t0:.0f}s" + (f" | fits that needed a retry on rescaled data: {sum(VOL_RETRY.values())}" if VOL_RETRY else "") +
      (f" | failed after retries (excluded from the comparison): {VOL_FAIL}" if VOL_FAIL else " | no fit failed"))

if arch_model is None or not VOLS.get("ll", pd.Series(dtype=float)).notna().any():
    print("WARNING: no GARCH-family model could be fitted (package 'arch' missing or every fit failed). Step 11 continues with EWMA and HAR-RV only; "
          "to get the full comparison turn Internet ON in the Kaggle notebook settings (or run: pip install arch) and re-run Step 0b, then Step 11."
          + (f" Last errors: {VOL_ERR}" if VOL_ERR else ""))
for c_ in ("aic", "bic"):
    if c_ not in VOLS: VOLS[c_] = np.nan                     # no log-likelihood without a fitted model: keep the columns so the BIC step still works
BENCH = "GARCH-n" if (VOLS.model == "GARCH-n").any() else "EWMA"
BEST_VOL = VOLS.loc[VOLS.groupby("series").qlike_val.idxmin()].set_index("series").model
VDIAG["best_vol_val"] = BEST_VOL
VDIAG["best_vol_test"] = VOLS.loc[VOLS.groupby("series").qlike_test.idxmin()].set_index("series").model
bic_ok = VOLS.dropna(subset=["bic"])
VDIAG["bic_winner"] = bic_ok.loc[bic_ok.groupby("series").bic.idxmin()].set_index("series").model if len(bic_ok) else np.nan
VDIAG["dm_t_vs_bench"] = np.nan; VDIAG["dm_p_vs_bench"] = np.nan
for s in VDIAG.index:
    y = RAW[s].dropna() * 100; idx = y.index.intersection(SPL["test"]); rvp = ((RV[s].reindex(idx) * 100) ** 2)
    hb, h0 = VOLFIT[s][BEST_VOL[s]]["h"].reindex(idx), VOLFIT[s][BENCH]["h"].reindex(idx)
    ok = (rvp > 0) & (hb > 0) & (h0 > 0)
    VDIAG.loc[s, ["dm_t_vs_bench", "dm_p_vs_bench"]] = dm_test(qlike(rvp[ok], h0[ok]) - qlike(rvp[ok], hb[ok]))
VOLS.to_csv(OUT2 / "vol_models.csv", index=False); VDIAG.to_csv(OUT2 / "vol_diagnostics.csv")

# standardised residuals and sigma of the validation-selected model (used by Steps 14-16)
STD, SIGMA, MU = {}, {}, {}
for s, mname in BEST_VOL.items():
    F = VOLFIT[s][mname]; y = RAW[s].dropna() * 100
    SIGMA[s] = np.sqrt(F["h"]); MU[s] = F["mu"]; STD[s] = (y - F["mu"]) / SIGMA[s]
STD = pd.DataFrame(STD).clip(-15, 15); SIGMA = pd.DataFrame(SIGMA)
STD.to_parquet(OUT2 / "std_resid.parquet")
UNIV_V = list(STD.columns)

print(f"\nARCH effects in raw returns (ARCH-LM p<0.05): {(VDIAG.archlm_p_raw < 0.05).sum()}/{len(VDIAG)}")
if "archlm_p_garch" in VDIAG:
    print(f"  …left after GARCH(1,1): {(VDIAG.archlm_p_garch < 0.05).sum()} | sign bias (p<0.05): {(VDIAG.sign_bias_p < 0.05).sum()}")
if "gjr_gamma_t" in VDIAG:
    print(f"GJR leverage γ>0 significant: {(VDIAG.gjr_gamma_t > 1.96).sum()} | inverse leverage γ<0 significant: {(VDIAG.gjr_gamma_t < -1.96).sum()}")
print("\nBest by validation QLIKE:", BEST_VOL.value_counts().to_dict())
print("Best by test QLIKE:      ", VDIAG.best_vol_test.value_counts().to_dict())
print("Best by train BIC:       ", VDIAG.bic_winner.value_counts().to_dict())
print(f"Validation-selected model beats {BENCH} on test (DM p<0.05): {(VDIAG.dm_p_vs_bench < 0.05).sum()}/{len(VDIAG)}")
qt = VOLS.pivot(index="series", columns="model", values="qlike_test")
print("\nMedian test QLIKE by model (lower = better):\n", qt.median().sort_values().round(4).to_string())
mz = VOLS.groupby("model")[["mz_a", "mz_b", "mz_r2"]].median().round(3)
print("\nMincer–Zarnowitz on test (median across series; unbiased: a=0, b=1):\n", mz.to_string())

In [ ]:
# step 11
VMODELS = [m for m in ["EWMA", "HAR-RV"] + list(VOL_SPECS) if m in qt.columns]
fig, axs = plt.subplots(1, 2, figsize=(15, max(4, 0.17 * len(qt))), gridspec_kw={"width_ratios": [1.3, 1]})
imp = (1 - qt[VMODELS].div(qt[BENCH], axis=0)) * 100
heat(axs[0], imp.loc[sector_order(imp.index)], cmap=DIV, center=0, vmax=max(5, np.nanpercentile(np.abs(imp.values), 95)),
     label=f"test QLIKE improvement vs {BENCH} (%)")
axs[0].set_title(f"Test QLIKE improvement over {BENCH} (orange = better)")
ax = axs[1]
cnt = pd.DataFrame({"best on validation": BEST_VOL.value_counts(), "best on test": VDIAG.best_vol_test.value_counts(),
                    "BIC winner (train)": VDIAG.bic_winner.value_counts()}).reindex(VMODELS).fillna(0)
yy = np.arange(len(cnt))
for k, (col, c) in enumerate(zip(cnt.columns, [SERIES[0], SERIES[1], MUTE])):
    ax.barh(yy + (k - 1) * 0.27, cnt[col], height=0.25, color=c, label=col)
ax.set_yticks(yy); ax.set_yticklabels(cnt.index); ax.invert_yaxis(); ax.legend(frameon=False, fontsize=8)
ax.set_xlabel("number of series"); ax.set_title("Which volatility model wins")
plt.tight_layout()
savefig(fig, "11_vol_compare", "Volatility model comparison", "11")

if "gjr_gamma_t" in VDIAG:
    fig, axs = plt.subplots(1, 3, figsize=(17, 4.3))
    ax = axs[0]
    o = VDIAG.sort_values("gjr_gamma_t")
    ax.bar(o.index, o.gjr_gamma_t, color=np.where(o.gjr_gamma_t > 1.96, SERIES[0], np.where(o.gjr_gamma_t < -1.96, SERIES[1], MUTE)))
    for yv in (1.96, -1.96):
        ax.axhline(yv, color=INK2, lw=1, ls="--")
    ax.tick_params(axis="x", rotation=90, labelsize=7); ax.set_ylabel("t-stat of GJR γ")
    ax.set_title("Leverage (blue: falls raise vol) vs inverse leverage (orange)")
    # news impact curves for the most asymmetric series
    s0 = VDIAG.gjr_gamma_t.abs().idxmax(); ax = axs[1]; zg = np.linspace(-4, 4, 200)
    for name, col in [("GARCH-n", MUTE), ("GJR-t", SERIES[0]), ("EGARCH-t", SERIES[1]), ("TARCH-t", SERIES[2])]:
        if name not in VOLFIT[s0]:
            continue
        pr = VOLFIT[s0][name]["params"]; hbar = float(VOLFIT[s0][name]["h"].iloc[:int((RAW[s0].dropna().index <= train_end).sum())].mean())
        if name == "EGARCH-t":
            lh = pr["omega"] + pr["alpha[1]"] * (np.abs(zg) - np.sqrt(2 / np.pi)) + pr["gamma[1]"] * zg + pr["beta[1]"] * np.log(hbar)
            hz = np.exp(lh)
        elif name == "TARCH-t":
            sd = pr["omega"] + (pr["alpha[1]"] + pr["gamma[1]"] * (zg < 0)) * np.abs(zg) * np.sqrt(hbar) + pr["beta[1]"] * np.sqrt(hbar)
            hz = sd ** 2
        else:
            eps2 = zg ** 2 * hbar; g_ = pr.get("gamma[1]", 0.0)
            hz = pr["omega"] + (pr["alpha[1]"] + g_ * (zg < 0)) * eps2 + pr["beta[1]"] * hbar
        ax.plot(zg, np.sqrt(hz), color=col, label=name)
    ax.set_xlabel("yesterday's shock (in sd)"); ax.set_ylabel("today's volatility (%)"); ax.legend(frameon=False, fontsize=8)
    ax.set_title(f"News impact curves: {s0}")
    ax = axs[2]; idx = RAW[s0].dropna().index.intersection(SPL["test"])
    ax.plot(idx, RV[s0].reindex(idx) * 100, color=MUTE, lw=0.8, label="realised vol (1-min)")
    for name, col in [(BENCH, SERIES[1]), (BEST_VOL[s0], SERIES[0])]:
        ax.plot(idx, np.sqrt(VOLFIT[s0][name]["h"].reindex(idx)), color=col, lw=1.3, label=f"{name} forecast")
    ax.legend(frameon=False, fontsize=8); ax.set_ylabel("daily vol (%)"); ax.set_title(f"{s0}: test-period forecasts vs realised")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    savefig(fig, "11_leverage_nic", "Leverage, news impact, forecasts", "11")

## Step 11b — Which volatility model survives every regime and every time frame?

Step 11 picked a model per series on **one** validation number. A single validation window can favour whichever model happens to suit the volatility level of that window, so the winner may disappear in the next regime. A researcher therefore checks **stability** before relying on a model:

1. **Slices.** The validation and test periods are cut into blocks of `VOL_SLICE_DAYS` trading days. Every model's QLIKE is computed in every block. Parameters are frozen on train, so each block is out-of-sample.
2. **Regimes.** A block is tagged LOW / MID / HIGH volatility by comparing its realised variance with the same-size blocks of the **train** period (terciles). A model is *robust* if its worst regime is not bad: the criterion is the **worst-regime mean rank** (minimax), not the average.
3. **Time frames.** The light models (constant, EWMA, GARCH-t, GJR-t, HAR-RV) are re-run on 5m, 15m, 30m, 60m, 1D and 1W bars for the most liquid products, with the same calendar splits, to see whether one model works at all horizons.
4. **Walk-forward selection (the validation + test hybrid done safely).** Choosing a model on test data would leak. What *is* allowed: at every block, choose the model that did best on **all earlier blocks** of validation and test, and score it on the current block. Every block is scored out of sample, validation and test data are both used to learn which model is stable, and nothing from the future is seen.

**Used later.** The robust choice (`BEST_VOL`) replaces the single-validation winner. It standardises the residuals for Step 12 (distributions), Step 13 (EVT) and the VaR / ES models of Step 25. The per-time-frame winner (`VOL_TF_BEST`) says which volatility model to use when a step runs on 5m ... 1W data. The walk-forward comparison says whether re-selecting through time is worth it.

In [ ]:
# step 11b
# ---- 11b.1 slice-level QLIKE of every fitted model (daily, all series) ----
def blocks(idx, n):
    # consecutive blocks of n observations; a short last block is merged into the previous one
    k = np.arange(len(idx)) // n
    if len(k) and (k == k.max()).sum() < n / 2 and k.max() > 0:
        k[k == k.max()] = k.max() - 1
    return k
SLQ_ROWS = []
for s, fits in VOLFIT.items():
    y = RAW[s].dropna(); rvp = (RV[s].reindex(y.index) * 100) ** 2
    trn = y.index.intersection(SPL["train"]); tr_blk = pd.Series(rvp.reindex(trn).values, index=trn).groupby(blocks(trn, VOL_SLICE_DAYS)).mean()
    q1, q2 = tr_blk.quantile([1 / 3, 2 / 3]) if len(tr_blk) >= 6 else (np.nan, np.nan)
    for sp in ("val", "test"):
        idx = y.index.intersection(SPL[sp]); b = blocks(idx, VOL_SLICE_DAYS)
        for bi in np.unique(b):
            ii = idx[b == bi]; pv = rvp.reindex(ii); mlevel = pv.mean(); reg = "LOW" if mlevel <= q1 else ("HIGH" if mlevel > q2 else "MID")
            for m, F in fits.items():
                hh = F["h"].reindex(ii); ok = (pv > 0) & (hh > 0)
                if ok.sum() >= 10:
                    SLQ_ROWS.append(dict(series=s, model=m, split=sp, slice=f"{sp}{bi}", order=(0 if sp == "val" else 1) * 1000 + int(bi), regime=reg, qlike=qlike(pv[ok], hh[ok]).mean()))
SLQ = pd.DataFrame(SLQ_ROWS)
SLQ["rank"] = SLQ.groupby(["series", "slice"]).qlike.rank()                                  # 1 = best model in that block
print(f"{SLQ.slice.nunique()} blocks of {VOL_SLICE_DAYS} trading days (validation + test) × {SLQ.series.nunique()} series × {SLQ.model.nunique()} models")

val = SLQ[SLQ.split == "val"]; tst = SLQ[SLQ.split == "test"]
def reg_table(D):
    t = D.pivot_table(index="model", columns="regime", values="rank", aggfunc="mean")
    t["all"] = D.groupby("model")["rank"].mean(); t["worst_regime"] = t[[c for c in ("LOW", "MID", "HIGH") if c in t]].max(axis=1)
    return t.sort_values("worst_regime")
RT_VAL, RT_TEST = reg_table(val), reg_table(tst)
print("\nMean rank of each volatility model by regime (1 = best; validation blocks; sorted by the worst regime):\n", RT_VAL.round(2).to_string())
print("\nSame on TEST blocks (confirmation only, nothing is selected here):\n", RT_TEST.reindex(RT_VAL.index).round(2).to_string())
rho_ = stats.spearmanr(RT_VAL["worst_regime"], RT_TEST["worst_regime"].reindex(RT_VAL.index), nan_policy="omit").statistic
print(f"\nRank agreement of the robustness score, validation vs test: Spearman {rho_:.2f}")
ROBUST_MODEL = RT_VAL.index[0]
rel = (SLQ[SLQ.split == "val"].pivot_table(index="model", columns="regime", values="qlike", aggfunc="median") / SLQ[(SLQ.split == "val") & (SLQ.model == "EWMA")].pivot_table(columns="regime", values="qlike", aggfunc="median").iloc[0]).round(3)
print("\nMedian QLIKE relative to EWMA (validation; below 1 = better than EWMA):\n", rel.reindex(RT_VAL.index).to_string())

In [ ]:
# step 11b
# ---- 11b.2 per-series robust choice, and walk-forward (expanding) re-selection ----
def robust_pick(D):
    out = {}
    for s_ in D.series.unique():
        g = D[D.series == s_]; sc = g.groupby(["model", "regime"])["rank"].agg(["mean", "size"]).reset_index()
        sc = sc[sc["size"] >= 2]                                                                # a regime needs two blocks to count
        worst = sc.groupby("model")["mean"].max(); allm = g.groupby("model")["rank"].mean()
        out[s_] = (worst + 0.01 * allm.reindex(worst.index)).idxmin() if len(worst) else allm.idxmin()
    return pd.Series(out)
BEST_VOL_VAL = BEST_VOL.copy()
BEST_VOL_ROBUST = robust_pick(val)
BEST_VOL = BEST_VOL_ROBUST.reindex(BEST_VOL_VAL.index).fillna(BEST_VOL_VAL)
chg = (BEST_VOL != BEST_VOL_VAL).sum()
VDIAG["best_vol_val"] = BEST_VOL                                                                 # downstream steps read this
print(f"Robust choice (worst-regime rank on validation blocks) differs from the single-window validation winner for {chg} of {len(BEST_VOL)} series")
print("Robust choice:", BEST_VOL.value_counts().to_dict(), "| single-window winner:", BEST_VOL_VAL.value_counts().to_dict())

# walk-forward: before block k, pick the model with the best mean QLIKE over all earlier blocks of validation + test; score block k
allb = SLQ.sort_values("order"); wf_rows = []
for s_, g in allb.groupby("series"):
    P = g.pivot_table(index="order", columns="model", values="qlike"); sp_of = g.drop_duplicates("order").set_index("order").split
    for j, o in enumerate(P.index):
        past = P.iloc[:j]
        pick = past.mean().idxmin() if j >= VOL_WF_MIN_BLOCKS else BEST_VOL_VAL.get(s_, "EWMA")
        if pick not in P.columns:
            continue
        wf_rows.append(dict(series=s_, split=sp_of[o], wf=P.loc[o, pick], static_val=P.loc[o].get(BEST_VOL_VAL.get(s_), np.nan), robust=P.loc[o].get(BEST_VOL.get(s_), np.nan),
                            **{m: P.loc[o].get(m, np.nan) for m in ("GARCH-n", "EWMA", "HAR-RV")}))
WF = pd.DataFrame(wf_rows)
WFS = WF.groupby("split")[["wf", "static_val", "robust", "GARCH-n", "EWMA", "HAR-RV"]].mean().T.rename(columns=str)
WFS["wf_beats"] = [np.nan] * len(WFS)
for c in ("static_val", "robust", "GARCH-n", "EWMA", "HAR-RV"):
    d_ = (WF[WF.split == "test"].groupby("series").apply(lambda g: (g.wf - g[c]).mean()))
    WFS.loc[c, "wf_beats"] = float((d_ < 0).mean())
print("\nMean QLIKE by selection procedure (blocks scored out of sample; lower is better). wf_beats = share of series where walk-forward selection beats the row's procedure on test blocks:")
print(WFS.round(4).to_string())
SLQ.to_csv(OUT2 / "vol_model_slices.csv", index=False)

In [ ]:
# step 11b
# ---- 11b.3 time frames: do the same light models work from 5m to 1W? ----
VOL_TF_LIST = [t for t in VOL_TFS if t in ("1D", "1W") or (len(load_p5()) > 0)]
_turn = dp[dp["trading_day"] <= SPL["train"][-1]].groupby("product")["money"].median()      # liquidity measured on the training period only
VOL_TF_PRODS = [p for p in _turn.reindex(UNIV).sort_values(ascending=False).index[:VOL_TF_N]]
TF_PAR = {"5m": None, "15m": None, "30m": None, "60m": None, "1D": dict(hl=VOL_HL, w=5, m=22, blk=VOL_SLICE_DAYS), "1W": dict(hl=4, w=4, m=13, blk=max(VOL_SLICE_DAYS // 5, 4))}

def tf_models(p, tf, F):
    R = F["ret"][p]; V2 = F["rv2"][p]; ok0 = R.notna() & V2.notna() & (V2 > 0)
    R, V2 = R[ok0], V2[ok0]
    if len(R) < (150 if tf == "1W" else 600):
        return None
    sp = split_index(R.index)
    if min(len(sp["train"]), len(sp["val"]), len(sp["test"])) < 150:
        return None
    if tf in ("1D", "1W"):
        seas = 1.0; day = R.index; par = TF_PAR[tf]; hl, wl, ml, blk = par["hl"], par["w"], par["m"], par["blk"]
    else:
        tod = pd.Index(R.index.hour * 60 + R.index.minute); mtr = R.index.isin(sp["train"])
        seas_s = R[mtr].abs().groupby(tod[mtr]).mean(); seas_s = seas_s / seas_s.mean(); seas = seas_s.reindex(tod).fillna(1.0).to_numpy()
        day = to_trading_day(R.index); bpd = max(int(pd.Series(1, index=R.index).groupby(day).sum().median()), 1)
        hl, wl, ml, blk = max(int(VOL_HL_DAYS * bpd), 20), bpd, 5 * bpd, VOL_SLICE_DAYS
    Rn = (R / seas) * 100; V2n = V2 / np.asarray(seas) ** 2 * 1e4
    ntr = len(sp["train"]); e2 = Rn.to_numpy() ** 2
    H = {"constant": pd.Series(e2[:ntr].mean(), index=R.index), "EWMA": pd.Series(e2, index=R.index).ewm(halflife=hl, min_periods=5).mean().shift(1).fillna(e2[:ntr].mean())}   # warm-up: fixed training variance, never a backfill
    lv = np.log(V2n); Xh = pd.concat({"d": lv.shift(1), "w": lv.rolling(wl).mean().shift(1), "m": lv.rolling(ml).mean().shift(1)}, axis=1)
    mt = (Xh.notna().all(axis=1)).to_numpy() & (np.arange(len(R)) < ntr)
    if mt.sum() > 100:
        _, b = ols(np.c_[np.ones(mt.sum()), Xh.values[mt]], lv.values[mt]); fit_ = np.c_[np.ones(len(Xh)), Xh.values] @ b
        hr = np.exp(fit_ + np.var(lv.values[mt] - fit_[mt]) / 2); sc = np.nanmean(e2[:ntr]) / np.nanmean(hr[:ntr])
        H["HAR-RV"] = pd.Series(hr * sc, index=R.index).ffill().fillna(e2[:ntr].mean())
    if arch_model is not None and VOL_TF_ARCH:
        a0 = max(0, ntr - VOL_TF_MAX_FIT); ys = Rn.iloc[a0:]
        for nm_, kw in (("GARCH-t", dict(vol="GARCH", p=1, q=1, dist="t")), ("GJR-t", dict(vol="GARCH", p=1, o=1, q=1, dist="t"))):
            try:
                am = arch_model(ys, mean="Zero", rescale=False, **kw); r_ = am.fit(last_obs=ntr - a0, disp="off", options={"maxiter": 500})
                h_ = (am.fix(r_.params).conditional_volatility ** 2).reindex(R.index)
                if np.isfinite(h_.iloc[ntr:]).all():
                    H[nm_] = h_.ffill().fillna(e2[:ntr].mean())
            except Exception:
                pass
    rows = []
    trn = R.index[:ntr]; tdays = pd.Index(day[:ntr]) if tf not in ("1D", "1W") else trn
    tr_lbl = pd.Series(np.arange(len(pd.unique(tdays))), index=pd.unique(tdays)); trb = pd.Series(V2n.iloc[:ntr].values, index=trn).groupby(pd.Series(tr_lbl.reindex(tdays).values // blk, index=trn).values).mean()
    q1, q2 = trb.quantile([1 / 3, 2 / 3]) if len(trb) >= 6 else (np.nan, np.nan)
    for spn in ("val", "test"):
        ix = sp[spn]; dd = pd.Series(day[R.index.isin(ix)] if tf not in ("1D", "1W") else ix, index=ix)
        lab = pd.Series(np.arange(dd.nunique()), index=pd.unique(dd)); bi = lab.reindex(dd.values).values // blk
        for b_ in np.unique(bi):
            ii = ix[bi == b_]; pv = V2n.reindex(ii); reg = "LOW" if pv.mean() <= q1 else ("HIGH" if pv.mean() > q2 else "MID")
            for m_, h_ in H.items():
                hh = h_.reindex(ii); ok = (pv > 0) & (hh > 0)
                if ok.sum() >= 10:
                    rows.append(dict(tf=tf, product=p, model=m_, split=spn, slice=f"{spn}{b_}", regime=reg, qlike=qlike(pv[ok], hh[ok]).mean()))
    return rows

TFQ_ROWS = []; t0 = time.time()
for tf in VOL_TF_LIST:
    F_ = tf_frames(tf)
    if F_ is None:
        continue
    for p in VOL_TF_PRODS:
        if p in F_["ret"].columns:
            r_ = tf_models(p, tf, F_)
            if r_:
                TFQ_ROWS += r_
    del F_
TFQ = pd.DataFrame(TFQ_ROWS)
if len(TFQ):
    TFQ["rank"] = TFQ.groupby(["tf", "product", "slice"]).qlike.rank()
    rel_tf = TFQ[TFQ.split == "val"].groupby(["tf", "model"]).qlike.median().unstack("tf")
    rel_tf = rel_tf / rel_tf.loc["constant"] if "constant" in rel_tf.index else rel_tf
    rk_tf = TFQ[TFQ.split == "val"].pivot_table(index="model", columns=["tf", "regime"], values="rank", aggfunc="mean")
    worst_tf = rk_tf.max(axis=1).sort_values()
    VOL_TF_BEST = TFQ[TFQ.split == "val"].groupby(["tf", "model"])["rank"].mean().unstack("tf").idxmin()
    VOL_TF_TABLE = rel_tf.reindex(columns=[t for t in VOL_TFS if t in rel_tf.columns])
    VOL_TF_TABLE.round(3).to_csv(OUT2 / "vol_models_by_timeframe.csv")
    tt = TFQ[TFQ.split == "test"].groupby(["tf", "model"])["rank"].mean().unstack("tf")
    print(f"time-frame check on {len(VOL_TF_PRODS)} most liquid products, light models, in {time.time()-t0:.0f}s")
    print("\nMedian QLIKE relative to the constant-variance model (validation; < 1 = the model adds value at that frequency):\n", VOL_TF_TABLE.round(3).to_string())
    print("\nBest model per time frame (mean block rank on validation):", VOL_TF_BEST.to_dict())
    print("Same on TEST blocks:", tt.idxmin().to_dict())
    print("Worst (tf, regime) mean rank on validation — the minimax model across ALL time frames and regimes:\n", worst_tf.round(2).to_string())
    VOL_UNIVERSAL = worst_tf.index[0]
    print(f"\n→ model that works across all time frames and regimes: {VOL_UNIVERSAL}")
else:
    VOL_TF_BEST, VOL_UNIVERSAL = pd.Series(dtype=object), None
    print("no time-frame results (not enough bars)")

In [ ]:
# step 11b
fig, axs = plt.subplots(1, 3, figsize=(18, 4.8))
ax = axs[0]
heat(ax, RT_VAL[[c for c in ("LOW", "MID", "HIGH", "all") if c in RT_VAL]], cmap=SEQ, label="mean rank (1 = best)", annot=True, fmt="{:.1f}")
ax.set_title("Volatility models by regime (validation blocks)")
ax = axs[1]
if len(TFQ):
    heat(ax, VOL_TF_TABLE.clip(0.4, 1.6), cmap=DIV, center=1, vmax=0.6, label="QLIKE vs constant (< 1 better)", annot=True, fmt="{:.2f}")
ax.set_title("Same model, different time frame")
ax = axs[2]
vals = WFS[["test"]].dropna() if "test" in WFS else WFS.iloc[:, :1]
ax.barh(vals.index, vals.iloc[:, 0].values, color=[SERIES[0] if i == "wf" else MUTE for i in vals.index], height=0.6); ax.set_xlabel("mean QLIKE on test blocks")
ax.set_title("Walk-forward re-selection vs fixed choices")
plt.tight_layout()
savefig(fig, "11b_vol_robustness", "Volatility models: regimes, time frames, walk-forward selection", "11")

## Step 11d — Overnight realised variance in the volatility forecast, and the volatility regime

**Question.** A Chinese futures day has two very different parts: the **night session** (it carries the overseas news that arrived while the day market was shut) and the **day session**. Does the realised variance of the night session improve the forecast of the **day-session** variance, and is the gain concentrated in high-volatility regimes? Nothing below is assumed; the result is reported as found, per regime.

**Notation.** For product $i$ and trading day $t$, $RV^{N}_t$ and $RV^{D}_t$ are the realised variances of the night and day sessions, each a sum of squared clean 1-minute returns (a roll never enters). The night session of day $t$ is finished **before** the day session opens, so $RV^{N}_t$ is known when the day-session forecast is made.

**Regime (ex-ante).** $\;v_{t}=\sqrt{\tfrac1{20}\sum_{j=1}^{20}RV_{t-j}}\;$, $\;p_t=\text{rank of }v_t\text{ inside its trailing }W=500\text{ days}\;$, and
$$S_t=\text{HIGH if }p_t>\tfrac23,\quad \text{LOW if }p_t\le\tfrac13,\quad \text{MID otherwise.}$$
It uses data up to $t-1$ only (truncation-tested below). The same $S_t$ is used in Steps 13e and 23d.

**Models** (all in logs, fitted on **train** by least squares, scored on validation and test with QLIKE as in Step 11):
$$\begin{aligned}
\text{HAR-D:}\quad & \ln RV^D_t=\beta_0+\beta_d\ln RV^D_{t-1}+\beta_w\,\overline{\ln RV^D}_{t-1}^{(5)}+\beta_m\,\overline{\ln RV^D}_{t-1}^{(22)}+\varepsilon_t\\
\text{HAR-D+N:}\quad & \text{HAR-D}\;+\;\delta\,\ln RV^N_t\\
\text{HAR-D+N}\times\text{HIGH:}\quad & \text{HAR-D+N}\;+\;\gamma\,\mathbf 1[S_t=\text{HIGH}]\ln RV^N_t+\eta\,\mathbf 1[S_t=\text{HIGH}]
\end{aligned}$$
Forecast $\hat h_t=\exp(\hat\ln RV^D_t+\hat\sigma^2_\varepsilon/2)$; loss $\;\text{QLIKE}=x-\ln x-1,\;x=RV^D_t/\hat h_t$.

**Claim and test.** The claim is $\delta>0$ with a gain concentrated where $S_t=$ HIGH. Measured as the QLIKE improvement $1-\text{QLIKE}_{\text{model}}/\text{QLIKE}_{\text{HAR-D}}$ by regime, with a Diebold–Mariano-style $t$ (date-clustered) on the loss difference. **Supported** only if the improvement is positive in HIGH on validation **and** test, $t>2$ on test, and larger than in LOW.

**Needs** the night / day variance columns from Step 1 (`rv_night`, `rv_day`). A cache built before this step lacks them and is rebuilt once, automatically.

**Used later.** The regime $S_t$ gates the opening-reversal strategy (Step 23d). The result says whether a night-session term belongs in the volatility forecast that sizes day-session trades.

In [ ]:
# step 11d
# ---- 11d.1 ex-ante volatility regime ----
RVN_OK = all(c in dp.columns for c in ("rv_night", "rv_day"))
def vol_regime(RV_):
    # regime of day t from realised volatility up to t-1: percentile of the 20-day vol inside its trailing REG_WIN days; -1 LOW (bottom third), 0 MID, +1 HIGH (top third)
    v20 = (RV_ ** 2).rolling(20, min_periods=15).mean().pow(0.5)
    pc = v20.rolling(REG_WIN, min_periods=REG_MIN).rank(pct=True).shift(1)
    return pd.DataFrame(np.where(pc.isna(), np.nan, np.where(pc > 2 / 3, 1.0, np.where(pc <= 1 / 3, -1.0, 0.0))), index=pc.index, columns=pc.columns)
causal_test("Step 11d volatility regime (LOW / MID / HIGH)", vol_regime, RV[UNIV], step="11d")
VREG = vol_regime(RV[UNIV]); REGN = {-1.0: "LOW", 0.0: "MID", 1.0: "HIGH"}
print("share of product-days by regime:", {REGN[k]: round(float((VREG == k).sum().sum() / max(VREG.notna().sum().sum(), 1)), 3) for k in REGN})
if not RVN_OK:
    print("the cached daily panel has no night / day variance columns: set FORCE_REBUILD = True in Step 0 and re-run Step 1 (the HAR-RV-ON comparison is skipped)")
    HARON_SUMMARY = dict(available=False); HARON_TAB = pd.DataFrame()

In [ ]:
# step 11d
# ---- 11d.2 HAR-RV with a night-session term: QLIKE by regime on validation and test ----
if RVN_OK:
    import statsmodels.api as sm
    RVN, RVD = (wide(c).reindex(index=RAW.index, columns=UNIV) for c in ("rv_night", "rv_day"))
    lg_ = lambda x: np.log(x.where(x > 0))
    HM = {"HAR-D": ["d", "w", "m"], "HAR-D+N": ["d", "w", "m", "n"], "HAR-D+N×HIGH": ["d", "w", "m", "n", "nH", "H"]}
    QD, HCOEF, n_fit = {}, [], 0
    SPL_OF = pd.Series("", index=RAW.index); SPL_OF.loc[SPL["val"]] = "val"; SPL_OF.loc[SPL["test"]] = "test"
    for s in UNIV:
        yv, nv = (RVD[s] * 100) ** 2, (RVN[s] * 100) ** 2; ly, ln_ = lg_(yv), lg_(nv)
        Hh = (VREG[s] == 1).astype(float).where(VREG[s].notna())
        B = pd.concat({"y": ly, "d": ly.shift(1), "w": ly.rolling(5).mean().shift(1), "m": ly.rolling(22).mean().shift(1), "n": ln_, "nH": ln_ * Hh, "H": Hh, "yv": yv}, axis=1)
        ok = B.notna().all(axis=1); tr = ok & (B.index <= train_end); te = ok & (B.index > train_end)
        if tr.sum() < HARON_MIN or te.sum() < 60: continue
        n_fit += 1
        for nm_, cols in HM.items():
            Xtr = np.c_[np.ones(int(tr.sum())), B.loc[tr, cols].to_numpy()]; ytr = B.loc[tr, "y"].to_numpy()
            b = np.linalg.lstsq(Xtr, ytr, rcond=None)[0]; s2 = float(np.var(ytr - Xtr @ b))
            h = pd.Series(np.exp(np.c_[np.ones(int(te.sum())), B.loc[te, cols].to_numpy()] @ b + s2 / 2), index=B.index[te])
            QD[(nm_, s)] = qlike(B.loc[te, "yv"], h)
            if nm_ != "HAR-D": HCOEF.append(dict(series=s, model=nm_, delta=b[1 + cols.index("n")], gamma=b[1 + cols.index("nH")] if "nH" in cols else np.nan))
    print(f"{n_fit} of {len(UNIV)} products have enough days with a night session (>= {HARON_MIN} in train)")
    rows = []
    for nm_ in ("HAR-D+N", "HAR-D+N×HIGH"):
        D = pd.DataFrame({s: QD[(nm_, s)] - QD[("HAR-D", s)] for (m_, s) in QD if m_ == nm_}); B0 = pd.DataFrame({s: QD[("HAR-D", s)] for (m_, s) in QD if m_ == "HAR-D"})
        RG_ = VREG.reindex(index=D.index, columns=D.columns)
        for sp in ("val", "test"):
            ix = D.index[SPL_OF.reindex(D.index).to_numpy() == sp]
            for k, rn in REGN.items():
                m_ = (RG_.loc[ix] == k); d_ = D.loc[ix].where(m_).stack().dropna(); b_ = B0.loc[ix].where(m_).stack().dropna()
                if len(d_) < 100: continue
                r_ = sm.OLS(d_.to_numpy(), np.ones(len(d_))).fit(cov_type="cluster", cov_kwds=dict(groups=pd.factorize(d_.index.get_level_values(0))[0]))
                pp = d_.groupby(level=1).mean(); pb = b_.groupby(level=1).mean()
                rows.append(dict(model=nm_, split=sp, regime=rn, n=len(d_), qlike_base=float(b_.mean()), qlike_model=float(b_.mean() + d_.mean()), improvement_pct=float(-d_.mean() / b_.mean() * 100),
                                 share_products_better=float((pp < 0).mean()), dm_t=float(-r_.tvalues[0])))
    HARON_TAB = pd.DataFrame(rows); HARON_TAB.round(4).to_csv(OUT2 / "har_on_by_regime.csv", index=False)
    HCF = pd.DataFrame(HCOEF); HCF.round(4).to_csv(OUT2 / "har_on_coefficients.csv", index=False)
    print("QLIKE improvement of a night-session term over HAR-D, by regime (positive = better; dm_t > 0 = the gain is significant):")
    print(HARON_TAB.pivot_table(index=["model", "regime"], columns="split", values=["improvement_pct", "dm_t", "share_products_better"]).reindex(columns=["improvement_pct", "dm_t", "share_products_better"], level=0).round(2).to_string())
    print("\nmedian training coefficient on ln RV(night): delta =", round(float(HCF[HCF.model == "HAR-D+N"].delta.median()), 3), "| extra slope in HIGH: gamma =", round(float(HCF[HCF.model == "HAR-D+N×HIGH"].gamma.median()), 3))
    def _g(mod, sp, rg, col):
        v = HARON_TAB[(HARON_TAB.model == mod) & (HARON_TAB.split == sp) & (HARON_TAB.regime == rg)][col]; return float(v.iloc[0]) if len(v) else np.nan
    best_m = max(("HAR-D+N", "HAR-D+N×HIGH"), key=lambda m_: np.nan_to_num(_g(m_, "val", "HIGH", "improvement_pct"), nan=-9))
    HARON_SUPPORTED = bool(_g(best_m, "val", "HIGH", "improvement_pct") > 0 and _g(best_m, "test", "HIGH", "improvement_pct") > 0 and _g(best_m, "test", "HIGH", "dm_t") > 2
                           and _g(best_m, "test", "HIGH", "improvement_pct") > _g(best_m, "test", "LOW", "improvement_pct"))
    _rd = (f"A night-session variance term ({best_m}, chosen on validation) changes day-session QLIKE by {_g(best_m, 'val', 'HIGH', 'improvement_pct'):+.1f}% (validation) and {_g(best_m, 'test', 'HIGH', 'improvement_pct'):+.1f}% (test, t {_g(best_m, 'test', 'HIGH', 'dm_t'):+.1f}) "
           f"in HIGH-volatility regimes against {_g(best_m, 'test', 'LOW', 'improvement_pct'):+.1f}% in LOW on test. Claim {'SUPPORTED' if HARON_SUPPORTED else 'NOT supported'}: it needs a gain in HIGH on validation and test, t > 2, and more than in LOW.")
    print("\n" + _rd)
    HARON_SUMMARY = dict(available=True, model=best_m, supported=HARON_SUPPORTED, n_products=int(n_fit), reading=_rd,
                         high_val=_g(best_m, "val", "HIGH", "improvement_pct"), high_test=_g(best_m, "test", "HIGH", "improvement_pct"), high_test_t=_g(best_m, "test", "HIGH", "dm_t"), low_test=_g(best_m, "test", "LOW", "improvement_pct"),
                         delta_median=float(HCF[HCF.model == "HAR-D+N"].delta.median()))

In [ ]:
# step 11d
if RVN_OK and len(HARON_TAB):
    fig, axs = plt.subplots(1, 3, figsize=(18, 4.8)); regs_ = ["LOW", "MID", "HIGH"]
    ax = axs[0]; w_ = 0.2
    for j, (mod, sp, c_) in enumerate((("HAR-D+N", "val", SERIES[0]), ("HAR-D+N", "test", SERIES[2]), ("HAR-D+N×HIGH", "val", SERIES[1]), ("HAR-D+N×HIGH", "test", SERIES[3]))):
        v_ = HARON_TAB[(HARON_TAB.model == mod) & (HARON_TAB.split == sp)].set_index("regime")["improvement_pct"].reindex(regs_); ax.bar(np.arange(3) + (j - 1.5) * w_, v_.values, w_, color=c_, label=f"{mod} {sp}")
    ax.set_xticks(range(3)); ax.set_xticklabels(regs_); ax.axhline(0, color=INK2, lw=1); ax.set_ylabel("QLIKE improvement over HAR-D, %"); ax.legend(frameon=False, fontsize=7); ax.set_title("Does the night session help the day-session forecast? By regime")
    ax = axs[1]
    for j, (sp, c_) in enumerate((("val", SERIES[0]), ("test", SERIES[2]))):
        v_ = HARON_TAB[(HARON_TAB.model == "HAR-D+N") & (HARON_TAB.split == sp)].set_index("regime")["share_products_better"].reindex(regs_); ax.bar(np.arange(3) + (j - 0.5) * 0.35, v_.values, 0.35, color=c_, label=sp)
    ax.axhline(0.5, color=INK2, ls=":", lw=1); ax.set_xticks(range(3)); ax.set_xticklabels(regs_); ax.set_ylim(0, 1); ax.set_ylabel("share of products where it helps"); ax.legend(frameon=False, fontsize=8); ax.set_title("Breadth of the gain (HAR-D+N)")
    ax = axs[2]; hc = HCF[HCF.model == "HAR-D+N"].delta.dropna(); ax.hist(hc, bins=20, color=SERIES[0]); ax.axvline(0, color=INK2, lw=1); ax.axvline(hc.median(), color=SERIES[1], ls="--", label=f"median {hc.median():+.2f}")
    ax.set_xlabel("train coefficient δ on ln RV(night)"); ax.set_ylabel("products"); ax.legend(frameon=False, fontsize=8); ax.set_title("Weight of the night session in the day forecast")
    plt.tight_layout(); savefig(fig, "11d_har_overnight", "HAR-RV with a night-session term, by volatility regime", "11")

## Step 11c — Volatility expansion and contraction monitor

Volatility clusters and mean-reverts, so its *state* is information in itself: where the market is in the cycle changes position sizes, stops, and which edges to trust. The monitor uses only **rolling realised volatility** (from 1-minute returns), no model:

| Quantity | Definition |
|---|---|
| short, long vol | $\sigma_S$ over `VOLMON_S` days and $\sigma_L$ over `VOLMON_L` days (root mean square of daily realised vol) |
| ratio | $\rho = \sigma_S / \sigma_L$ |
| level percentile | rank of $\sigma_L$ within its last 250 days |
| **EXPANSION** | $\rho >$ `VOLMON_UP` and $\sigma_S$ still rising over 5 days |
| **CONTRACTION** | $\rho <$ `VOLMON_DN` and $\sigma_S$ still falling |
| **SQUEEZE** | contraction with the level percentile below 15%: volatility is low *and* falling, the setting for a breakout |

The tests ask whether the state **carries information**: does volatility mean-revert from each state, how often does a squeeze end in an expansion within 10 days (against the unconditional frequency), do trends persist more in expansion or in contraction, and does it behave the same in train, validation and test?

**Used later.** `VOLSTATE_NOW` (state, ratio, percentile, days in state) is a column of the opportunity board (Step 26). The market-wide share of products in expansion is added to the regime and state-variable tables (Steps 20 and 21). Squeeze lists are alerts. The conditional results say whether a state should change position size.

In [ ]:
# step 11c
# ---- 11c.1 states ----
def vol_states(RV_):
    # trailing windows only: every state at t uses realised volatility up to t
    rv2 = RV_ ** 2
    v_s = rv2.rolling(VOLMON_S, min_periods=VOLMON_S).mean().pow(0.5); v_l = rv2.rolling(VOLMON_L, min_periods=VOLMON_L).mean().pow(0.5)
    vratio = v_s / v_l; vpct = v_l.rolling(250, min_periods=120).rank(pct=True); vslope = np.log(v_s) - np.log(v_s.shift(5))
    EXPN = (vratio > VOLMON_UP) & (vslope > 0); CONT = (vratio < VOLMON_DN) & (vslope < 0); SQZ = CONT & (vpct <= 0.15)
    VSTATE = pd.DataFrame("NORMAL", index=vratio.index, columns=vratio.columns).mask(CONT, "CONTRACTION").mask(EXPN, "EXPANSION").mask(SQZ, "SQUEEZE").where(vratio.notna())
    return dict(state=VSTATE, ratio=vratio, pct=vpct, v_s=v_s, v_l=v_l, expn=EXPN.astype(float).where(vratio.notna()))
causal_test("Step 11c volatility states (ratio, percentile, state)", vol_states, RV[UNIV], step="11c")
_vs = vol_states(RV[UNIV]); rv2d = RV[UNIV] ** 2
VSTATE, vratio, vpct, v_s, v_l = _vs["state"], _vs["ratio"], _vs["pct"], _vs["v_s"], _vs["v_l"]
EXPN = (_vs["expn"] == 1); CONT = VSTATE.isin(["CONTRACTION", "SQUEEZE"]); SQZ = VSTATE.eq("SQUEEZE")
tr_ix = RAW.index.intersection(SPL["train"])
print("share of product-days in each state (train):", (VSTATE.loc[tr_ix].stack().value_counts(normalize=True)).round(3).to_dict())

# forward volatility and trend persistence by state
def fwd_vol(h):
    return rv2d.rolling(h, min_periods=h).mean().shift(-h).pow(0.5)   # causal-ok: target: forward realised volatility (evaluation only)
rows = []
past5 = RAW[UNIV].rolling(5).sum(); fwd5 = RAW[UNIV].rolling(5).sum().shift(-5); sd5 = RAW[UNIV].ewm(halflife=VOL_HL, min_periods=20).std() * np.sqrt(5)
for sp in ("train", "val", "test"):
    ix = RAW.index.intersection(SPL[sp])
    for h in (5, 20):
        chg = np.log(fwd_vol(h) / v_l).loc[ix]
        for st_ in ("EXPANSION", "CONTRACTION", "SQUEEZE", "NORMAL"):
            m = (VSTATE.loc[ix] == st_); v = chg.where(m).stack().dropna()
            rows.append(dict(split=sp, state=st_, h=h, n=len(v), fwd_vol_vs_long=np.expm1(v.mean()) if len(v) else np.nan, t=v.mean() / v.std() * np.sqrt(len(v) / h) if len(v) > 10 else np.nan))
    tr_ = (np.sign(past5) * fwd5 / sd5).loc[ix]
    for st_ in ("EXPANSION", "CONTRACTION", "NORMAL"):
        v = tr_.where(VSTATE.loc[ix] == st_).stack().dropna()
        rows.append(dict(split=sp, state=st_, h=-5, n=len(v), fwd_vol_vs_long=v.mean(), t=v.mean() / v.std() * np.sqrt(len(v) / 5) if len(v) > 10 else np.nan))
VSTAT = pd.DataFrame(rows)
print("\nForward realised volatility relative to the 60-day level, by state (h = 5 / 20 days); row h = -5 is trend persistence: mean of sign(past 5d) × next 5d return in vol units:")
print(VSTAT.pivot_table(index=["h", "state"], columns="split", values=["fwd_vol_vs_long", "t"]).round(3).to_string())

# squeeze → expansion within 10 days, against the unconditional frequency
exp_next = EXPN.astype(float).rolling(10).max().shift(-10)
sq_rows = []
for sp in ("train", "val", "test"):
    ix = RAW.index.intersection(SPL[sp]); a = exp_next.loc[ix]; sq = SQZ.loc[ix]
    pc = a.where(sq).stack().dropna(); pu = a.stack().dropna()
    sq_rows.append(dict(split=sp, n_squeeze=len(pc), p_expansion_after_squeeze=pc.mean(), p_expansion_unconditional=pu.mean(),
                        lift=pc.mean() / pu.mean() if len(pc) else np.nan, p_binom=stats.binomtest(int(pc.sum()), len(pc), pu.mean(), alternative="greater").pvalue if len(pc) > 5 else np.nan))
SQ = pd.DataFrame(sq_rows); print("\nDoes a squeeze end in an expansion within 10 days?\n", SQ.round(3).to_string(index=False))
# is the ratio itself predictive of the next 5-day volatility change? (rank IC, daily cross-section)
ic_v = {sp: xs_ic_fast(np.log(vratio).replace([np.inf, -np.inf], np.nan).loc[RAW.index.intersection(SPL[sp])], np.log(fwd_vol(5) / v_s).replace([np.inf, -np.inf], np.nan).loc[RAW.index.intersection(SPL[sp])]) for sp in ("train", "val", "test")}
print("Rank IC of log(ratio) against the next 5-day volatility change (negative = volatility mean-reverts):", {k: round(float(v.mean()), 3) for k, v in ic_v.items()})

In [ ]:
# step 11c
# ---- 11c.2 now, market and sector breadth, figure ----
last_ = vratio.dropna(how="all").index[-1]
dstate = VSTATE.apply(lambda c: c.groupby((c != c.shift()).cumsum()).cumcount() + 1)
VOLSTATE_NOW = pd.DataFrame({"sector": SEC_MAP.reindex(UNIV), "state": VSTATE.loc[last_], "ratio": vratio.loc[last_], "level_pctile": vpct.loc[last_],
                             "vol_short_ann_pct": v_s.loc[last_] * np.sqrt(ANN) * 100, "vol_long_ann_pct": v_l.loc[last_] * np.sqrt(ANN) * 100, "days_in_state": dstate.loc[last_]})
VOLSTATE_NOW = VOLSTATE_NOW.sort_values("ratio", ascending=False); VOLSTATE_NOW.round(3).to_csv(OUT2 / "volatility_state_now.csv")
BREADTH_EXP = (VSTATE == "EXPANSION").mean(axis=1).where(vratio.notna().any(axis=1)); BREADTH_CON = VSTATE.isin(["CONTRACTION", "SQUEEZE"]).mean(axis=1).where(vratio.notna().any(axis=1))
print(f"as of {last_.date()}: {int((VOLSTATE_NOW.state == 'EXPANSION').sum())} products in EXPANSION, {int((VOLSTATE_NOW.state == 'CONTRACTION').sum())} in CONTRACTION, {int((VOLSTATE_NOW.state == 'SQUEEZE').sum())} in SQUEEZE | "
      f"market breadth of expansion {BREADTH_EXP.loc[last_]:.0%}")
print("Squeezes (low and falling volatility, breakout candidates):", ", ".join(VOLSTATE_NOW.index[VOLSTATE_NOW.state == "SQUEEZE"]) or "none")
print("Most expanded:\n", VOLSTATE_NOW.head(8).round(2).to_string())
sec_exp = (VSTATE == "EXPANSION").T.groupby(SEC_MAP.reindex(UNIV)).mean().T
fig, axs = plt.subplots(1, 3, figsize=(18, 4.6))
ax = axs[0]
ax.plot(BREADTH_EXP.index, BREADTH_EXP.rolling(5).mean(), color=SERIES[1], lw=1, label="share in EXPANSION"); ax.plot(BREADTH_CON.index, BREADTH_CON.rolling(5).mean(), color=SERIES[0], lw=1, label="share in CONTRACTION / SQUEEZE")
for sp in ("val", "test"):
    ax.axvline(SPL[sp][0], color=INK2, ls="--", lw=1)
ax.legend(frameon=False, fontsize=8); ax.set_title("Market-wide volatility cycle (5-day mean of breadth)")
ax = axs[1]
pv_ = VSTAT[(VSTAT.h == 5) & VSTAT.state.isin(["EXPANSION", "CONTRACTION", "SQUEEZE", "NORMAL"])].pivot(index="state", columns="split", values="fwd_vol_vs_long")[["train", "val", "test"]]
x = np.arange(len(pv_)); w = 0.27
for k, sp in enumerate(pv_.columns):
    ax.bar(x + (k - 1) * w, pv_[sp].values * 100, width=w, color=[MUTE, "#86b6ef", SERIES[0]][k], label=sp)
ax.set_xticks(x); ax.set_xticklabels(pv_.index, fontsize=8); ax.axhline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=8)
ax.set_ylabel("next-5-day vol vs 60-day vol, %"); ax.set_title("Volatility mean-reverts from each state")
ax = axs[2]
v_ = VOLSTATE_NOW.sort_values("ratio"); ax.barh(v_.index, v_.ratio.values - 1, left=1, color=np.where(v_.state.isin(["EXPANSION"]), SERIES[1], np.where(v_.state.isin(["CONTRACTION", "SQUEEZE"]), SERIES[0], MUTE)), height=0.7)
ax.axvline(1, color=INK2, lw=1); ax.tick_params(axis="y", labelsize=5); ax.set_xlabel("short / long volatility"); ax.set_title(f"Where each product stands ({last_.date()})")
plt.tight_layout()
savefig(fig, "11c_vol_monitor", "Volatility expansion / contraction monitor", "11")

## Step 11e — Is the volatility model adequate? Residual diagnostics, in statistical order

**Why this step exists.** Steps 10 and 11 follow the usual order (mean model, then volatility model), but three checks were missing, and one wiring problem existed:
1. ARCH-LM in Step 11 is run on **raw** returns. The textbook test is on the **mean-model residuals**, because a serial-correlation structure in the mean can create or hide apparent ARCH effects.
2. Step 11 uses a **constant** mean. Does an AR mean change the volatility model?
3. There was **no Ljung–Box** test, either on returns or on squared standardised residuals, and no out-of-sample adequacy check of the chosen model.
4. Step 11b replaces `BEST_VOL` by the robust choice, but `STD`, `SIGMA` and `MU` (which Steps 12, 13 and 25 read) were still built from the **single-window** validation winner. Tail and distribution estimates inherited that choice. This step rebuilds them from the final `BEST_VOL`.

**Tests.** Let $z_t=(r_t-\hat\mu)/\hat\sigma_t$ be the standardised residual of the chosen model.
* **Ljung–Box** $Q(m)=n(n+2)\sum_{k=1}^{m}\hat\rho_k^2/(n-k)$ on $r_t$ (or the mean-model residual), on $r_t^2$, and on $z_t$ and $z_t^2$, with $m=10$ and $20$. For $z_t^2$ a pass means the model has absorbed the clustering.
* **ARCH-LM** (Engle): regress $z_t^2$ on its 5 lags, $LM=nR^2\sim\chi^2_5$.
* **Engle–Ng sign-bias** on $z_t$, as in Step 11.
* **Mean and variance of $z$**: should be near 0 and 1.

**Order in this step.** (1) ARCH effects after the mean model; (2) rebuild the residuals from the final choice; (3) in-sample and **out-of-sample** (validation + test, frozen parameters) adequacy; (4) constant mean against an AR mean for the volatility model.

**Decision rule.** A series is *adequate* if, on out-of-sample data, Ljung–Box on $z^2$ ($m=10$) and ARCH-LM both give $p>0.05$. Inadequate series are listed and flagged (`STD_ADEQUATE`) and stay in the study. The reading says how far EVT and VaR tail estimates for them can be trusted.

**Used later.** `STD`, `SIGMA` and `MU` (rebuilt here) feed Steps 12, 13, 14–16 and 25. The adequacy flags and counts go into the findings report.

In [ ]:
# step 11e
# ---- 11e.1 volatility clustering after the mean model; Ljung-Box on returns and squared returns ----
from statsmodels.stats.diagnostic import acorr_ljungbox
def lb_p(x, m):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < 5 * m: return np.nan
    return float(acorr_ljungbox(x, lags=[m], return_df=True)["lb_pvalue"].iloc[0])
def arch_p(x, m=5):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return float(het_arch(x, nlags=m)[1]) if len(x) > 10 * m else np.nan
rows = []
for s in UNIV:
    y = RAW[s].dropna() * 100; ytr = y[y.index <= train_end]
    if len(ytr) < 300: continue
    o = ARMA_ORD.get(s, (1, 1)); r = dict(series=s, arma_order=str(o), n=len(ytr), lb10_ret=lb_p(ytr, 10), lb10_ret2=lb_p(ytr ** 2, 10), archlm_raw=arch_p(ytr - ytr.mean()))
    try:
        res = ARIMA(ytr.to_numpy(), order=(o[0], 0, o[1]), trend="c").fit(); e = np.asarray(res.resid)[max(o):]
        r.update(lb10_resid=lb_p(e, 10), lb10_resid2=lb_p(e ** 2, 10), archlm_resid=arch_p(e))
    except Exception as ex_:
        r["note"] = f"ARMA refit failed: {type(ex_).__name__}"
    rows.append(r)
ARCH_AFTER_MEAN = pd.DataFrame(rows).set_index("series"); ARCH_AFTER_MEAN.round(4).to_csv(OUT2 / "arch_after_mean_model.csv")
cnt = lambda c: int((ARCH_AFTER_MEAN[c] < 0.05).sum()) if c in ARCH_AFTER_MEAN else 0; N11 = len(ARCH_AFTER_MEAN)
print(f"Train period, {N11} series (p < 0.05 means the null of no serial dependence is rejected):")
print(f"  Ljung-Box(10) on returns: {cnt('lb10_ret')} | on squared returns: {cnt('lb10_ret2')} | ARCH-LM(5) on demeaned returns: {cnt('archlm_raw')}")
print(f"  after the ARMA mean model (order from Step 10): Ljung-Box(10) on residuals {cnt('lb10_resid')} | on squared residuals {cnt('lb10_resid2')} | ARCH-LM(5) on residuals {cnt('archlm_resid')}")
_ch = int(((ARCH_AFTER_MEAN.archlm_raw < 0.05) != (ARCH_AFTER_MEAN.archlm_resid < 0.05)).sum()) if "archlm_resid" in ARCH_AFTER_MEAN else 0
print(f"  the ARCH-LM verdict changes for {_ch} of {N11} series when the test is run on mean-model residuals instead of raw returns")

In [ ]:
# step 11e
# ---- 11e.2 standardised residuals rebuilt from the FINAL volatility choice; in-sample and out-of-sample adequacy ----
STD_VALWIN = STD.copy()                                                                      # as built in Step 11 (single-window validation winner)
_chg = [s for s in BEST_VOL.index if s in STD_VALWIN.columns and BEST_VOL[s] != BEST_VOL_VAL.get(s)]
_S2, _G2, _M2 = {}, {}, {}
for s, mname in BEST_VOL.items():
    F = VOLFIT[s][mname]; y = RAW[s].dropna() * 100
    _G2[s] = np.sqrt(F["h"]); _M2[s] = F["mu"]; _S2[s] = (y - F["mu"]) / _G2[s]
STD = pd.DataFrame(_S2).clip(-15, 15); SIGMA = pd.DataFrame(_G2); MU = _M2; UNIV_V = list(STD.columns); STD.to_parquet(OUT2 / "std_resid.parquet")
print(f"standardised residuals rebuilt from the final choice (BEST_VOL after Step 11b) for {len(STD.columns)} series; {len(_chg)} of them use a different model than the single-window winner used before: {_chg[:12]}{' ...' if len(_chg) > 12 else ''}")
def std_diag(STD_, label):
    rows = []
    for s in STD_.columns:
        z_all = STD_[s].dropna(); r = dict(series=s, model=BEST_VOL.get(s))
        for nm_, ix in (("train", z_all.index.intersection(SPL["train"])), ("oos", z_all.index.intersection(SPL["val"]).union(z_all.index.intersection(SPL["test"])))):
            z = z_all.loc[ix]
            r.update({f"{nm_}_n": len(z), f"{nm_}_mean": float(z.mean()), f"{nm_}_var": float(z.var()), f"{nm_}_lb10_z": lb_p(z, 10), f"{nm_}_lb10_z2": lb_p(z ** 2, 10), f"{nm_}_lb20_z2": lb_p(z ** 2, 20), f"{nm_}_archlm": arch_p(z - z.mean()),
                      f"{nm_}_signbias": float(sign_bias_p(z.to_numpy())) if len(z) > 50 else np.nan})
        r["adequate_oos"] = bool(r["oos_lb10_z2"] > 0.05 and r["oos_archlm"] > 0.05); r["adequate_train"] = bool(r["train_lb10_z2"] > 0.05 and r["train_archlm"] > 0.05); rows.append(r)
    D_ = pd.DataFrame(rows).set_index("series"); D_["version"] = label; return D_
STD_DIAG = std_diag(STD, "final"); STD_DIAG["sector"] = [SECTOR.get(s, "Other") for s in STD_DIAG.index]; STD_ADEQUATE = STD_DIAG.adequate_oos
STD_DIAG.round(4).to_csv(OUT2 / "std_resid_diagnostics.csv")
n_ = len(STD_DIAG); f_ = lambda c: int((STD_DIAG[c] < 0.05).sum())
print(f"\nFinal standardised residuals, {n_} series (a count is the number with p < 0.05, i.e. dependence left):")
print(pd.DataFrame({"Ljung-Box(10) on z": [f_("train_lb10_z"), f_("oos_lb10_z")], "Ljung-Box(10) on z²": [f_("train_lb10_z2"), f_("oos_lb10_z2")], "Ljung-Box(20) on z²": [f_("train_lb20_z2"), f_("oos_lb20_z2")], "ARCH-LM(5) on z": [f_("train_archlm"), f_("oos_archlm")],
                    "sign bias": [f_("train_signbias"), f_("oos_signbias")]}, index=["train (in-sample)", "validation + test (frozen model)"]).to_string())
print(f"\nmean of z: train {STD_DIAG.train_mean.median():+.3f}, out of sample {STD_DIAG.oos_mean.median():+.3f} | variance of z (median; should be 1): train {STD_DIAG.train_var.median():.2f}, out of sample {STD_DIAG.oos_var.median():.2f}")
print(f"adequate out of sample (Ljung-Box on z² and ARCH-LM both p > 0.05): {int(STD_DIAG.adequate_oos.sum())} of {n_}")
print("\nShare adequate out of sample by chosen model:\n", STD_DIAG.groupby("model").adequate_oos.agg(["size", "mean"]).rename(columns={"size": "series", "mean": "share_adequate"}).round(2).to_string())
print("\nBy sector:\n", STD_DIAG.groupby("sector").adequate_oos.agg(["size", "mean"]).rename(columns={"size": "series", "mean": "share_adequate"}).round(2).to_string())
_bad = STD_DIAG[~STD_DIAG.adequate_oos]; print("\nNot adequate out of sample (tail estimates for these are less reliable):", {s: r.model for s, r in _bad.iterrows()})
if _chg:
    old_ = std_diag(STD_VALWIN[[s for s in _chg if s in STD_VALWIN.columns]], "single-window winner")
    print(f"\nThe {len(_chg)} series whose model changed: adequate out of sample with the single-window winner {int(old_.adequate_oos.sum())}, with the robust choice {int(STD_DIAG.loc[_chg, 'adequate_oos'].sum())}")

In [ ]:
# step 11e
# ---- 11e.3 does an AR mean change the volatility model? (constant mean versus AR(1) mean, same variance model) ----
AR_MEAN_TAB = pd.DataFrame()
if arch_model is not None:
    rows = []
    for s in UNIV_V:
        mname = BEST_VOL[s]
        if mname not in VOL_SPECS or mname not in VOLFIT[s] or not np.isfinite(VOLFIT[s][mname].get("ll", np.nan)): continue
        y = RAW[s].dropna() * 100; n_fit = int((y.index <= train_end).sum()); rvp = (RV[s].reindex(y.index) * 100) ** 2
        try:
            am = arch_model(y, mean="AR", lags=1, rescale=False, **VOL_SPECS[mname]); r_ = am.fit(last_obs=n_fit, disp="off", options=dict(maxiter=1000)); h = am.fix(r_.params).conditional_volatility ** 2
        except Exception as e_:
            continue
        F = VOLFIT[s][mname]; row = dict(series=s, model=mname, d_bic=(len(r_.params) * np.log(n_fit) - 2 * r_.loglikelihood) - (F["k"] * np.log(n_fit) - 2 * F["ll"]), ar1=float(next((r_.params[k] for k in r_.params.index if k.endswith("[1]") and not k.startswith(("alpha", "beta", "gamma", "delta", "eta"))), np.nan)))
        for sp in ("val", "test"):
            idx = y.index.intersection(SPL[sp]); pv = rvp.reindex(idx); h1, h0 = h.reindex(idx), F["h"].reindex(idx); ok = (pv > 0) & (h1 > 0) & (h0 > 0)
            row[f"qlike_{sp}_ar"] = qlike(pv[ok], h1[ok]).mean(); row[f"qlike_{sp}_const"] = qlike(pv[ok], h0[ok]).mean()
        rows.append(row)
    AR_MEAN_TAB = pd.DataFrame(rows)
    if len(AR_MEAN_TAB):
        AR_MEAN_TAB.round(4).to_csv(OUT2 / "ar_mean_vs_constant.csv", index=False)
        print(f"{len(AR_MEAN_TAB)} series with a fitted ARCH-family choice, AR(1) mean added to the same variance model:")
        print(f"  BIC lower with the AR mean (negative d_bic): {int((AR_MEAN_TAB.d_bic < 0).sum())} | validation QLIKE better: {int((AR_MEAN_TAB.qlike_val_ar < AR_MEAN_TAB.qlike_val_const).sum())} | test QLIKE better: {int((AR_MEAN_TAB.qlike_test_ar < AR_MEAN_TAB.qlike_test_const).sum())}")
        print(f"  median AR(1) coefficient {AR_MEAN_TAB.ar1.median():+.3f}; median relative QLIKE change on test {100*float((AR_MEAN_TAB.qlike_test_ar / AR_MEAN_TAB.qlike_test_const - 1).median()):+.2f}%")
else:
    print("arch not available: the constant-mean versus AR-mean comparison is skipped")
_o = lambda c: int(STD_DIAG[c].lt(0.05).sum())
_ar = (f" An AR(1) mean lowered BIC for {int((AR_MEAN_TAB.d_bic < 0).sum())} of {len(AR_MEAN_TAB)} series and improved test QLIKE for {int((AR_MEAN_TAB.qlike_test_ar < AR_MEAN_TAB.qlike_test_const).sum())}; the constant mean is {'adequate' if len(AR_MEAN_TAB) and (AR_MEAN_TAB.qlike_test_ar < AR_MEAN_TAB.qlike_test_const).mean() < 0.6 else 'worth revisiting'}." if len(AR_MEAN_TAB) else "")
DIAG11E_SUMMARY = dict(n_series=int(len(STD_DIAG)), adequate_oos=int(STD_DIAG.adequate_oos.sum()), not_adequate=list(STD_DIAG.index[~STD_DIAG.adequate_oos]), lb_z2_oos_fail=_o("oos_lb10_z2"), archlm_oos_fail=_o("oos_archlm"), rebuilt_series=len(_chg),
                       arch_after_mean=dict(raw=int((ARCH_AFTER_MEAN.archlm_raw < 0.05).sum()), resid=int((ARCH_AFTER_MEAN.get("archlm_resid", pd.Series(dtype=float)) < 0.05).sum())),
                       reading=(f"ARCH effects: {int((ARCH_AFTER_MEAN.archlm_raw < 0.05).sum())} of {len(ARCH_AFTER_MEAN)} series on raw returns and {int((ARCH_AFTER_MEAN.get('archlm_resid', pd.Series(dtype=float)) < 0.05).sum())} on ARMA residuals, so the mean model does not remove the clustering. "
                                f"After the final volatility model, {int(STD_DIAG.adequate_oos.sum())} of {len(STD_DIAG)} series are adequate out of sample (Ljung-Box on z² and ARCH-LM both p > 0.05); dependence in z² is left in {_o('oos_lb10_z2')} (Ljung-Box) and {_o('oos_archlm')} (ARCH-LM). "
                                f"Residuals were rebuilt from the robust Step 11b choice for {len(_chg)} series whose model had changed." + _ar))
print("\n" + DIAG11E_SUMMARY["reading"])

## Step 12 — Which distribution fits the standardised residuals?

Inputs are the train residuals of the validation-selected volatility model from Step 11. Every family is fitted by maximum likelihood; the winner is the one with the lowest BIC.

| Family | Params | Captures |
|---|---|---|
| `norm` | 2 | benchmark |
| `t` | 3 | symmetric power-law tails |
| `laplace`, `logistic`, `hypsecant` | 2 | exponential tails |
| `gennorm` (GED) | 3 | tail shape β (2 = normal, 1 = Laplace) |
| `skewnorm` | 3 | skew, thin tails |
| `johnsonsu`, `nct` | 4 | skew **and** fat tails |
| `gmm2` | 5 | calm/turbulent mixture, no shape assumed |

**Checks:**
* KS, Cramér–von Mises and Anderson–Darling statistics against the fitted CDF.
* A **parametric-bootstrap** KS p-value for the winner. The naive p-value is too lenient once parameters have been fitted.
* A **minimum-CvM-distance** refit, to see whether a few outliers drag the MLE.
* **Validation** KS against the frozen fit. Step 25 uses it to choose between the winner and FHS.
* **Test** KS against the frozen fit, and a train-vs-test two-sample KS.

**Used later.** The winning distribution gives the innovations for VaR / ES (Step 25) and for the tail studies of Step 13.

In [ ]:
# step 12
RES = {k: STD.loc[STD.index.intersection(SPL[k])] for k in SPL}
fit_rows, PARAMS = [], {}
t0 = time.time()
for s in UNIV_V:
    x = RES["train"][s].dropna().to_numpy()
    if len(x) > MAX_FIT_N:
        x = rng.choice(x, MAX_FIT_N, replace=False)
    for f in FAMILIES:
        try:
            par = tuple(np.atleast_1d(fit_family(f, x)).astype(float)); g = gof(f, par, x)
            if not np.isfinite(g["loglik"]):
                raise ValueError("non-finite loglik")
            PARAMS[(s, f)] = par
            fit_rows.append(dict(series=s, family=f, n=len(x), params=np.round(par, 4).tolist(), **g))
        except Exception as e:
            fit_rows.append(dict(series=s, family=f, n=len(x), params=None, loglik=np.nan, bic=np.nan, error=type(e).__name__))
FITS = pd.DataFrame(fit_rows)
FITS["dBIC"] = FITS.bic - FITS.groupby("series").bic.transform("min")
FITS.to_csv(OUT2 / "dist_fits.csv", index=False)
WIN = FITS.loc[FITS.groupby("series").bic.idxmin()].set_index("series")
print(f"{len(FITS)} fits in {time.time()-t0:.0f}s | BIC winners: {WIN.family.value_counts().to_dict()}")

win_rows = []
for s, w in WIN.iterrows():
    f = w.family; par = PARAMS[(s, f)]
    x = RES["train"][s].dropna().to_numpy(); x_fit = x if len(x) <= MAX_FIT_N else rng.choice(x, MAX_FIT_N, replace=False)
    pb = boot_ks_p(f, par, x_fit)
    par_md, _ = min_cvm_fit(f, par, x_fit)
    shift = np.max(np.abs(np.array(par_md) - np.array(par)) / (np.abs(np.array(par)) + 0.1))
    xv, xt = RES["val"][s].dropna().to_numpy(), RES["test"][s].dropna().to_numpy()
    cdf = lambda z, f=f, par=par: DISTS[f].cdf(z, *par)
    ll_oos = {ff: np.mean(DISTS[ff].logpdf(xt, *PARAMS[(s, ff)])) for ff in FAMILIES if (s, ff) in PARAMS}
    win_rows.append(dict(series=s, sector=SECTOR.get(s, "Other"), winner=f, params=np.round(par, 3).tolist(),
                         dBIC_norm=float(FITS.query("series==@s and family=='norm'").dBIC.iloc[0]),
                         dBIC_t=float(FITS.query("series==@s and family=='t'").dBIC.iloc[0]),
                         ks=w.ks, ks_p_naive=w.ks_p_naive, ks_p_boot=pb, cvm=w.cvm, ad=w.ad,
                         mde_params=np.round(par_md, 3).tolist(), mle_mde_shift=shift,
                         ks_val_p=stats.kstest(xv, cdf).pvalue, ks_oos_p=stats.kstest(xt, cdf).pvalue,
                         ks_oos_norm_p=stats.kstest(xt, lambda z, m=x.mean(), sd=x.std(): stats.norm.cdf(z, m, sd)).pvalue,
                         ks_2samp_p=stats.ks_2samp(x, xt).pvalue, best_oos_loglik=max(ll_oos, key=ll_oos.get),
                         oos_ll_gain_vs_norm=ll_oos[f] - ll_oos.get("norm", np.nan)))
WINS = pd.DataFrame(win_rows).set_index("series"); WINS.to_csv(OUT2 / "dist_winners.csv")
print(WINS[["winner", "dBIC_norm", "dBIC_t", "ks_p_boot", "mle_mde_shift", "ks_val_p", "ks_oos_p", "ks_oos_norm_p", "best_oos_loglik"]].round(3).to_string())
print(f"\nwinner not rejected in-sample (bootstrap KS p>0.05): {(WINS.ks_p_boot > 0.05).sum()}/{len(WINS)}")
print(f"frozen winner fits validation: {(WINS.ks_val_p > 0.05).sum()} | fits test: {(WINS.ks_oos_p > 0.05).sum()} | normal fits test: {(WINS.ks_oos_norm_p > 0.05).sum()}")

samples_sub = [RES["train"][s].dropna().to_numpy() for s in UNIV_V]
samples_sub = [x if len(x) <= 1000 else rng.choice(x, 1000, replace=False) for x in samples_sub]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ak = stats.anderson_ksamp(samples_sub)
print(f"\nk-sample Anderson–Darling across {len(UNIV_V)} series: p={ak.pvalue:.3g} (scipy floors at 0.001, caps at 0.25)")
KSD = pd.DataFrame(0.0, index=UNIV_V, columns=UNIV_V)
for a in range(len(UNIV_V)):
    for b in range(a + 1, len(UNIV_V)):
        KSD.iat[a, b] = KSD.iat[b, a] = stats.ks_2samp(samples_sub[a], samples_sub[b]).statistic
pooled = np.concatenate(samples_sub); pool_rows = []
for f in FAMILIES:
    try:
        par = tuple(np.atleast_1d(fit_family(f, pooled)).astype(float)); pool_rows.append(dict(family=f, params=np.round(par, 3).tolist(), **gof(f, par, pooled)))
    except Exception:
        pass
POOL = pd.DataFrame(pool_rows).sort_values("bic"); POOL["dBIC"] = POOL.bic - POOL.bic.min()
print("\nPooled residuals:\n", POOL[["family", "params", "dBIC", "ks", "ad"]].round(3).to_string(index=False))

In [ ]:
# step 12
D_ = FITS.pivot(index="series", columns="family", values="dBIC").reindex(index=sector_order(UNIV_V), columns=FAMILIES)
fig, axs = plt.subplots(1, 2, figsize=(14, max(4, 0.17 * len(UNIV_V))), gridspec_kw={"width_ratios": [1.4, 1]})
heat(axs[0], D_.clip(upper=60), vmin=0, vmax=60, label="ΔBIC vs best (capped at 60)")
axs[0].set_title("ΔBIC per series and family (0 = winner, >10 = strongly worse)")
ax = axs[1]
wc = WINS.winner.value_counts().reindex(FAMILIES).fillna(0)
okc = WINS[WINS.ks_oos_p > 0.05].winner.value_counts().reindex(FAMILIES).fillna(0)
ax.barh(wc.index, wc.values, color=MUTE, height=0.6, label="BIC winner (train)")
ax.barh(okc.index, okc.values, color=SERIES[0], height=0.35, label="…and still fits test (KS p>0.05)")
ax.invert_yaxis(); ax.legend(frameon=False, fontsize=8); ax.set_xlabel("number of series"); ax.set_title("Winning family counts")
plt.tight_layout()
savefig(fig, "12_bic_winners", "Distribution selection", "12")

def grid_plot(series, kind, name, title):
    nc = 6; nr = int(np.ceil(len(series) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(2.6 * nc, 2.2 * nr), squeeze=False)
    for ax, s in zip(axs.flat, series):
        x = RES["train"][s].dropna().to_numpy(); f = WINS.loc[s, "winner"]; par = PARAMS[(s, f)]
        if kind == "hist":
            h, e = np.histogram(x, bins=60, density=True); c = (e[:-1] + e[1:]) / 2
            ax.bar(c, h, width=e[1] - e[0], color="#cfdcee", linewidth=0)
            g = np.linspace(e[0], e[-1], 400)
            ax.plot(g, DISTS[f].pdf(g, *par), color=SERIES[0], lw=1.5)
            ax.plot(g, stats.norm.pdf(g, x.mean(), x.std()), color=SERIES[1], lw=1, ls="--")
            ax.set_yscale("log"); ax.set_ylim(max(h[h > 0].min() / 3, 1e-5), h.max() * 2)
        else:
            q = (np.arange(1, len(x) + 1) - 0.5) / len(x); xs = np.sort(x)
            sub = np.unique(np.r_[np.linspace(0, len(x) - 1, 300).astype(int), np.arange(min(15, len(x))), np.arange(max(0, len(x) - 15), len(x))])
            ax.scatter(stats.norm.ppf(q[sub], x.mean(), x.std()), xs[sub], s=4, color=SERIES[1])
            ax.scatter(DISTS[f].ppf(q[sub], *par), xs[sub], s=4, color=SERIES[0])
            lo, hi = xs.min(), xs.max(); ax.plot([lo, hi], [lo, hi], color=INK2, lw=0.8)
        ax.set_title(f"{s}: {f}", fontsize=8); ax.tick_params(labelsize=6)
    for ax in list(axs.flat)[len(series):]:
        ax.axis("off")
    axs.flat[0].legend(handles=[plt.Line2D([], [], color=SERIES[0], label="BIC winner"),
                                plt.Line2D([], [], color=SERIES[1], ls="--", label="normal")], frameon=False, fontsize=6)
    fig.suptitle(title, fontsize=11, fontweight="bold", color=INK); plt.tight_layout()
    savefig(fig, name, title, "12")

ser_o = sector_order(UNIV_V)
for k in range(0, len(ser_o), 36):
    grid_plot(ser_o[k:k + 36], "hist", f"12_hist_{k//36}", f"Residual density (log scale) with BIC winner — page {k//36+1}")
    grid_plot(ser_o[k:k + 36], "qq", f"12_qq_{k//36}", f"QQ plots (blue = winner, orange = normal) — page {k//36+1}")

if sns is not None and len(UNIV_V) > 2:
    link = hierarchy.linkage(squareform(KSD.values, checks=False), method="average")
    cg = sns.clustermap(KSD, cmap=SEQ, row_linkage=link, col_linkage=link, figsize=(max(7, 0.18 * len(UNIV_V) + 3),) * 2,
                        xticklabels=True, yticklabels=True, cbar_kws={"label": "2-sample KS distance"})
    cg.ax_heatmap.tick_params(labelsize=6); cg.fig.suptitle("Series with similarly shaped residual distributions", y=1.01, fontweight="bold")
    savefig(cg.fig, "12_shape_cluster", "Shape-distance clustermap", "12")

## Step 13 — Extreme value theory

Step 12 describes the whole distribution, but risk lives in the far tail, where a full-distribution fit is dominated by the centre. EVT models the tail directly.

| Test | What it answers |
|---|---|
| **GPD peaks-over-threshold** (Pickands–Balkema–de Haan) on exceedances over the `EVT_Q` quantile of losses (−z) and gains (z) | Shape ξ for each tail: ξ > 0 is a power law with tail index 1/ξ; ξ = 0 is exponential; ξ < 0 has a finite end-point |
| ξ standard error ≈ (1+ξ)/√Nᵤ, plus a **parametric-bootstrap Anderson–Darling** test of the GPD fit | Is the tail heavy, and does the GPD describe it? |
| **Threshold stability**: ξ across `EVT_QS` | A valid threshold gives a flat ξ. The mean-excess plot should be linear above it |
| **Hill estimator** vs GPD ξ, and vs 1/ν of a Student-t winner | Do three tail estimators agree? |
| **GEV on block maxima** (monthly maxima of raw daily losses), likelihood-ratio test against Gumbel (ξ = 0) | Fréchet / Gumbel / Weibull domain; **10-year return level** of the daily loss |
| **Extremal index** θ (runs estimator) of raw losses vs GARCH residuals | Do extremes cluster (θ < 1)? A good volatility model should make θ ≈ 1 |
| Out-of-sample: test exceedance rate over the train threshold; KS of test exceedances against the train GPD | Does the tail model hold on new data? |

The GPD fits give **EVT (McNeil–Frey) VaR and ES** (Step 25). Section 13.3 then uses the tails to look for **opportunities**.

**Used later.** GPD tails give VaR / ES in Step 25. The tail events of 13.3 feed strategy C3 (Step 23) and the tail-spillover links of the relationship library (Step 17).

In [ ]:
# step 13
def gpd_fit(exc):
    c, _, sc = stats.genpareto.fit(exc, floc=0)
    return c, sc

def gpd_var_es(u, xi, beta, n, nu, a):
    # VaR / ES at confidence a of the variable whose upper tail is modelled (McNeil-Frey)
    pu = nu / n
    if abs(xi) < 1e-6:
        var = u - beta * np.log((1 - a) / pu); es = var + beta
    else:
        var = u + beta / xi * (((1 - a) / pu) ** (-xi) - 1)
        es = (var + beta - xi * u) / (1 - xi) if xi < 1 else np.inf
    return var, es

def extremal_index(x, q=EI_Q, r=EI_RUN):
    x = np.asarray(x, float); x = x[np.isfinite(x)]; idx = np.flatnonzero(x > np.quantile(x, q))
    return (1 + np.sum(np.diff(idx) > r)) / len(idx) if len(idx) > 1 else np.nan

EVT_PAR, evt_rows, stab_rows = {}, [], []
t0 = time.time()
for s in UNIV_V:
    z_tr = RES["train"][s].dropna().to_numpy(); z_te = RES["test"][s].dropna().to_numpy()
    for tail, x, xt in [("loss", -z_tr, -z_te), ("gain", z_tr, z_te)]:
        u = np.quantile(x, EVT_Q); exc = x[x > u] - u; nu = len(exc)
        xi, beta = gpd_fit(exc)
        A2 = ad_stat(stats.genpareto.cdf(exc, xi, 0, beta)); exceed = 0
        for b in range(EVT_B):
            eb = stats.genpareto.rvs(xi, 0, beta, size=nu, random_state=SEED + b)
            try:
                xb, sb = gpd_fit(eb); exceed += ad_stat(stats.genpareto.cdf(eb, xb, 0, sb)) >= A2
            except Exception:
                pass
        exc_te = xt[xt > u] - u
        row = dict(series=s, tail=tail, u=u, n_exc=nu, xi=xi, xi_se=(1 + xi) / np.sqrt(nu), beta=beta, ad=A2,
                   ad_p_boot=(1 + exceed) / (EVT_B + 1), xi_hill=1 / hill(x, frac=1 - EVT_Q),
                   test_exc_rate=np.mean(xt > u), test_exc_binom_p=stats.binomtest(int(np.sum(xt > u)), len(xt), 1 - EVT_Q).pvalue if len(xt) else np.nan,
                   test_gpd_ks_p=stats.kstest(exc_te, lambda e: stats.genpareto.cdf(e, xi, 0, beta)).pvalue if len(exc_te) >= 5 else np.nan)
        for a in (0.99, 0.995):
            row[f"var_{a}"], row[f"es_{a}"] = gpd_var_es(u, xi, beta, len(x), nu, a)
        if s in WINS.index and WINS.loc[s, "winner"] == "t":
            row["xi_from_t"] = 1 / PARAMS[(s, "t")][0]
        evt_rows.append(row); EVT_PAR[(s, tail)] = (u, xi, beta, len(x), nu)
        for q in EVT_QS:
            uq = np.quantile(x, q); eq = x[x > uq] - uq
            if len(eq) >= 15:
                xq, _ = gpd_fit(eq); stab_rows.append(dict(series=s, tail=tail, q=q, xi=xq))
EVT = pd.DataFrame(evt_rows); STAB = pd.DataFrame(stab_rows)
EVT.to_csv(OUT2 / "evt_gpd.csv", index=False)

gev_rows = []
for s in UNIV_V:
    L = -RAW.loc[SPL["train"], s].dropna().to_numpy() * 100
    nb_ = len(L) // BLOCK
    if nb_ < 20:
        continue
    Mx = L[:nb_ * BLOCK].reshape(nb_, BLOCK).max(axis=1)
    c, loc, sc = stats.genextreme.fit(Mx); ll_gev = np.sum(stats.genextreme.logpdf(Mx, c, loc, sc))
    lg, sg = stats.gumbel_r.fit(Mx); ll_gum = np.sum(stats.gumbel_r.logpdf(Mx, lg, sg))
    lr_p = stats.chi2.sf(max(2 * (ll_gev - ll_gum), 0), 1); xi_g = -c                   # scipy uses c = -xi
    per_year = ANN / BLOCK
    gev_rows.append(dict(series=s, n_blocks=nb_, xi_gev=xi_g, loc=loc, scale=sc, lr_p_vs_gumbel=lr_p,
                         domain=("Fréchet (heavy)" if xi_g > 0 else "Weibull (bounded)") if lr_p < 0.05 else "Gumbel (exp. tail)",
                         rl_10y_pct=stats.genextreme.ppf(1 - 1 / (10 * per_year), c, loc, sc),
                         rl_10y_gumbel_pct=stats.gumbel_r.ppf(1 - 1 / (10 * per_year), lg, sg),
                         max_train_loss_pct=L.max(), max_test_loss_pct=(-RAW.loc[SPL["test"], s].dropna() * 100).max(),
                         ei_raw=extremal_index(L), ei_resid=extremal_index(-RES["train"][s].dropna().to_numpy())))
GEV = pd.DataFrame(gev_rows).set_index("series"); GEV.to_csv(OUT2 / "evt_gev.csv")
print(f"EVT done in {time.time()-t0:.0f}s")
summ_evt = EVT.groupby("tail").agg(median_xi=("xi", "median"), share_xi_sig_pos=("xi", lambda v: np.nan),
                                   gpd_ok=("ad_p_boot", lambda v: (v > 0.05).mean()), test_tail_ok=("test_gpd_ks_p", lambda v: (v > 0.05).mean()))
summ_evt["share_xi_sig_pos"] = EVT.assign(sig=EVT.xi - 1.96 * EVT.xi_se > 0).groupby("tail").sig.mean()
print("\nGPD tail shape (standardised residuals):\n", summ_evt.round(3).to_string())
print(EVT[["series", "tail", "n_exc", "xi", "xi_se", "xi_hill", "ad_p_boot", "test_exc_rate", "test_gpd_ks_p", "var_0.99", "es_0.99"]]
      .round(3).sort_values(["tail", "xi"], ascending=[True, False]).to_string(index=False))
print("\nGEV on monthly maxima of daily losses:\n", GEV[["xi_gev", "lr_p_vs_gumbel", "domain", "rl_10y_pct", "max_train_loss_pct", "max_test_loss_pct", "ei_raw", "ei_resid"]].round(3).to_string())
print(f"\nextremal index: raw losses median θ={GEV.ei_raw.median():.2f}, GARCH residuals θ={GEV.ei_resid.median():.2f} (1 = no clustering)")

In [ ]:
# step 13
fig, axs = plt.subplots(1, 3, figsize=(17, max(4.5, 0.15 * len(UNIV_V))))
ax = axs[0]
for k, (tail, col) in enumerate([("loss", SERIES[1]), ("gain", SERIES[0])]):
    e = EVT[EVT["tail"] == tail].set_index("series").reindex(sector_order(UNIV_V))
    yy = np.arange(len(e)) + (k - 0.5) * 0.3
    ax.errorbar(e.xi, yy, xerr=1.96 * e.xi_se, fmt="o", ms=4, color=col, ecolor=col, elinewidth=1, label=f"{tail} tail")
ax.axvline(0, color=INK2, lw=1); ax.set_yticks(range(len(e))); ax.set_yticklabels(e.index, fontsize=6)
ax.set_xlabel("GPD shape ξ (±95% CI)"); ax.legend(frameon=False, fontsize=8); ax.set_title("Tail shape per series (ξ>0 = power law)")
ax = axs[1]
for tail, col in [("loss", SERIES[1]), ("gain", SERIES[0])]:
    g = STAB[STAB["tail"] == tail].groupby("q").xi
    ax.plot(g.median().index, g.median().values, marker="o", color=col, label=f"{tail}: median ξ")
    ax.fill_between(g.median().index, g.quantile(0.25).values, g.quantile(0.75).values, color=col, alpha=0.15, linewidth=0)
ax.axvline(EVT_Q, color=INK2, lw=1, ls="--"); ax.set_xlabel("threshold quantile"); ax.set_ylabel("ξ")
ax.legend(frameon=False, fontsize=8); ax.set_title("Threshold stability (flat = good threshold; band = IQR)")
ax = axs[2]
e = EVT[EVT["tail"] == "loss"]
ax.scatter(e.xi, e.xi_hill, color=SERIES[0], s=26, edgecolors="#fcfcfb", linewidths=1, label="Hill")
if "xi_from_t" in e:
    ax.scatter(e.xi, e.xi_from_t, color=SERIES[2], s=26, marker="s", edgecolors="#fcfcfb", linewidths=1, label="1/ν of t winner")
lim = [min(e.xi.min(), 0) - 0.05, max(e.xi.max(), e.xi_hill.max()) + 0.05]
ax.plot(lim, lim, color=INK2, lw=1, ls="--"); ax.set_xlabel("GPD ξ (loss tail)"); ax.set_ylabel("other tail estimate")
ax.legend(frameon=False, fontsize=8); ax.set_title("Do tail estimators agree?")
plt.tight_layout()
savefig(fig, "13_evt_gpd", "GPD tail shape, threshold stability, estimator agreement", "13")

fig, axs = plt.subplots(1, 3, figsize=(17, 4.2))
s0 = EVT[EVT["tail"] == "loss"].sort_values("xi").series.iloc[-1]; x = np.sort(-RES["train"][s0].dropna().to_numpy())
us = np.quantile(x, np.linspace(0.5, 0.98, 40)); me = [np.mean(x[x > u_] - u_) for u_ in us]
axs[0].plot(us, me, marker="o", ms=3, color=SERIES[0]); axs[0].axvline(EVT_PAR[(s0, "loss")][0], color=INK2, ls="--", lw=1)
axs[0].set_xlabel("threshold u"); axs[0].set_ylabel("mean excess e(u)"); axs[0].set_title(f"Mean-excess plot, {s0} losses (upward = heavy tail)")
g_ = GEV.sort_values("rl_10y_pct")
axs[1].barh(g_.index, g_.rl_10y_pct, color=SERIES[0], height=0.6, label="GEV 10-year return level")
axs[1].scatter(g_.max_train_loss_pct, g_.index, color=INK, s=12, zorder=3, label="largest train loss")
axs[1].scatter(g_.max_test_loss_pct, g_.index, color=SERIES[1], s=12, zorder=3, label="largest test loss")
axs[1].tick_params(axis="y", labelsize=6); axs[1].set_xlabel("daily loss (%)"); axs[1].legend(frameon=False, fontsize=7)
axs[1].set_title("Once-in-10-years daily loss (GEV, monthly maxima)")
axs[2].scatter(GEV.ei_raw, GEV.ei_resid, color=SERIES[0], s=26, edgecolors="#fcfcfb", linewidths=1)
for s_, r in GEV.iterrows():
    axs[2].annotate(s_, (r.ei_raw, r.ei_resid), fontsize=6, color=INK2, xytext=(2, 2), textcoords="offset points")
axs[2].plot([0, 1], [0, 1], color=INK2, ls="--", lw=1); axs[2].axhline(1, color=MUTE, lw=1)
axs[2].set_xlabel("extremal index θ, raw losses"); axs[2].set_ylabel("θ, standardised residuals")
axs[2].set_title("Extreme clustering removed by the vol model (θ→1)")
plt.tight_layout()
savefig(fig, "13_evt_gev", "Mean excess, return levels, extremal index", "13")

### 13.3 — EVT as an opportunity detector
A tail event here is a standardised move beyond the `EVT_EVENT_Q` quantile of |z| (train threshold, per product). The questions:

| | Test | Use |
|---|---|---|
| a | **Daily tail events**: average vol-scaled return over the next 1–10 days, split into up and down shocks; train vs test; by sector | continuation (trend-following after shocks) or reversal (fade the shock) → strategy C3 in Step 23 |
| b | **Intraday time bucketing** (5-min): where extremes cluster (night, open, morning, afternoon, close; day of week) and how they resolve over the next 1–12 bars | when to act on, or avoid, intraday extremes |
| c | **Clustering**: probability of another extreme within k bars, against the no-clustering benchmark 1−(1−p)ᵏ | position sizing after shocks |
| d | **Tail spillover**: an extreme in X today → an extreme in Y tomorrow, as a lift over the base rate; same or opposite sign | cross-product relationships (added to the library in Step 17) |
| e | **Tail heaviness by time of day**: GPD ξ per intraday bucket | which sessions carry jump risk |

In [ ]:
# step 13
# (a) daily tail events → continuation or reversal
U_EV = RES["train"][UNIV_V].abs().quantile(EVT_EVENT_Q)
E_D = (np.sign(STD[UNIV_V]) * (STD[UNIV_V].abs() > U_EV)).reindex(RAW.index).fillna(0)
Zd = Z[UNIV_V]
ev_rows = []
for h in EVT_H:
    fwd = Zd.rolling(h, min_periods=h).sum().shift(-h) / np.sqrt(h)
    for sp in ("train", "val", "test"):
        idx = RAW.index.intersection(SPL[sp])
        for sgn, name in [(1, "up shock"), (-1, "down shock")]:
            v = fwd.loc[idx].where(E_D.loc[idx] == sgn).stack().dropna()
            ev_rows.append(dict(h=h, split=sp, event=name, n=len(v), mean_fwd=v.mean(),
                                t=v.mean() / v.std() * np.sqrt(len(v) / h) if len(v) > 5 else np.nan, continuation=sgn * v.mean()))
EVS = pd.DataFrame(ev_rows); EVS.to_csv(OUT2 / "evt_event_study.csv", index=False)
print("Daily tail events — mean vol-scaled forward return (continuation > 0, reversal < 0):")
print(EVS.pivot_table(index=["event", "h"], columns="split", values=["continuation", "t"]).round(3).to_string())
# by sector, train, at the strategy holding period → orientation for strategy C3
fwdH = Zd.rolling(EVT_HOLD, min_periods=EVT_HOLD).sum().shift(-EVT_HOLD) / np.sqrt(EVT_HOLD)
sec_rows = []
for sec in sorted(set(SEC_MAP[UNIV_V])):
    cols = [c for c in UNIV_V if SEC_MAP[c] == sec]
    for sgn, name in [(1, "up shock"), (-1, "down shock")]:
        for sp in ("train", "test"):
            idx = RAW.index.intersection(SPL[sp]); v = fwdH.loc[idx, cols].where(E_D.loc[idx, cols] == sgn).stack().dropna()
            sec_rows.append(dict(sector=sec, event=name, split=sp, n=len(v), cont=sgn * v.mean() if len(v) else np.nan,
                                 t=sgn * v.mean() / v.std() * np.sqrt(len(v) / EVT_HOLD) if len(v) > 5 else np.nan))
EVSEC = pd.DataFrame(sec_rows)
EVT_ORIENT = {(r.sector, 1 if r.event == "up shock" else -1): (np.sign(r.cont) if abs(r.t) > 1 else 0.0)
              for r in EVSEC[EVSEC.split == "train"].itertuples() if np.isfinite(r.t)}
print("\nBy sector (h = EVT_HOLD days; +1 = continuation, −1 = reversal; used by strategy C3):")
print(EVSEC.pivot_table(index=["sector", "event"], columns="split", values=["cont", "t"]).round(3).to_string())

In [ ]:
# step 13
# (b)-(e) intraday time bucketing, clustering, spillover, tail heaviness by bucket
F5 = tf_frames("5m"); HAVE_5M = F5 is not None and len(F5["ret"]) > 1000
if HAVE_5M:
    Z5 = tf_z(F5, "5m"); SPL5 = split_index(Z5.index)
    u5 = Z5.loc[SPL5["train"]].abs().quantile(1 - (1 - EVT_EVENT_Q) / 5)          # 99th pct for 5-min
    E5 = np.sign(Z5) * (Z5.abs() > u5)
    bucket = pd.Series(tod_bucket(Z5.index), index=Z5.index); dow = pd.Series(to_trading_day(Z5.index).dayofweek, index=Z5.index)
    b_rows = []
    for sp in ("train", "test"):
        idx = Z5.index.intersection(SPL5[sp]); ev = (E5.loc[idx] != 0); avail = Z5.loc[idx].notna()
        base = ev.values.sum() / avail.values.sum()
        for k in EVT_H_BARS:
            fwd = Z5.rolling(k, min_periods=k).sum().shift(-k).loc[idx] / np.sqrt(k)
            for bk in sorted(set(bucket.loc[idx])):
                mb = (bucket.loc[idx] == bk).values
                rate = ev.values[mb].sum() / max(avail.values[mb].sum(), 1)
                for sgn, name in [(1, "up"), (-1, "down")]:
                    v = fwd[mb].where(E5.loc[idx][mb] == sgn).stack().dropna()
                    b_rows.append(dict(split=sp, bucket=bk, k=k, shock=name, lift=rate / base, n=len(v),
                                       cont=sgn * v.mean() if len(v) else np.nan,
                                       t=sgn * v.mean() / v.std() * np.sqrt(len(v) / k) if len(v) > 5 else np.nan))
    EVB = pd.DataFrame(b_rows); EVB.to_csv(OUT2 / "evt_intraday_buckets.csv", index=False)
    print("Intraday extremes by time-of-day bucket (lift = event rate / overall rate; cont > 0 continuation):")
    print(EVB[EVB.k == EVT_H_BARS[1]].pivot_table(index=["bucket", "shock"], columns="split", values=["lift", "cont", "t", "n"]).round(3).to_string())
    idx = Z5.index.intersection(SPL5["train"]); evtr = (E5.loc[idx] != 0)
    dow_lift = (evtr.groupby(dow.loc[idx]).sum().sum(1) / Z5.loc[idx].notna().groupby(dow.loc[idx]).sum().sum(1))
    dow_lift = dow_lift / (evtr.values.sum() / Z5.loc[idx].notna().values.sum())
    print("\nDay-of-week lift of intraday extremes (train):", dow_lift.round(2).to_dict())
    # (e) GPD shape by bucket, pooled
    xi_b = {}
    for bk in sorted(set(bucket.loc[idx])):
        x = Z5.loc[idx][(bucket.loc[idx] == bk).values].abs().stack().dropna().values
        if len(x) > 2000:
            u_ = np.quantile(x, 0.95); xi_b[bk] = stats.genpareto.fit(x[x > u_] - u_, floc=0)[0]
    print("GPD ξ of |z| by bucket (train, pooled):", {k: round(v, 3) for k, v in xi_b.items()})

def cluster_curve(E, ks=range(1, 21)):
    # P(next extreme within k bars) vs the no-clustering benchmark
    gaps, p_ev = [], []
    for c in E.columns:
        e = E[c].dropna().to_numpy(); pos = np.flatnonzero(e != 0)
        if len(pos) > 5:
            gaps.extend(np.diff(pos)); p_ev.append(len(pos) / len(e))
    gaps = np.array(gaps); p = np.mean(p_ev) if p_ev else np.nan
    return pd.DataFrame({"empirical": [np.mean(gaps <= k) for k in ks], "no clustering": [1 - (1 - p) ** k for k in ks]}, index=list(ks))
CLU_D = cluster_curve(E_D.loc[RAW.index.intersection(SPL["train"])])
CLU_5 = cluster_curve(E5.loc[SPL5["train"]]) if HAVE_5M else None

# (d) tail spillover (daily): extreme in X at t → extreme in Y at t+1
def spill(Ew):
    A = (Ew != 0).astype(float).values; Pp = (Ew > 0).astype(float).values; Nn = (Ew < 0).astype(float).values
    co = A[:-1].T @ A[1:]; same = Pp[:-1].T @ Pp[1:] + Nn[:-1].T @ Nn[1:]; nx_ = A[:-1].sum(0); py = A[1:].mean(0)
    lift = co / np.maximum(nx_[:, None], 1) / np.maximum(py[None, :], 1e-9)
    return co, same, nx_, py, lift
sp_rows = []
idx_tr, idx_te = RAW.index.intersection(SPL["train"]), RAW.index.intersection(SPL["test"])
co, same, nx_, py, lift = spill(E_D.loc[idx_tr]); _, _, _, _, lift_te = spill(E_D.loc[idx_te])
for a_, ca in enumerate(UNIV_V):
    for b_, cb in enumerate(UNIV_V):
        if a_ != b_ and co[a_, b_] >= 5:
            sp_rows.append(dict(cause=ca, effect=cb, n_co=int(co[a_, b_]), lift_train=lift[a_, b_], lift_test=lift_te[a_, b_],
                                same_sign_share=same[a_, b_] / co[a_, b_], p_binom=stats.binomtest(int(co[a_, b_]), int(nx_[a_]), min(py[b_], 0.999), alternative="greater").pvalue))
TAIL_SPILL = pd.DataFrame(sp_rows)
if len(TAIL_SPILL):
    TAIL_SPILL["q"] = bh(TAIL_SPILL.p_binom); TAIL_SPILL = TAIL_SPILL.sort_values("lift_train", ascending=False)
    TAIL_SPILL.to_csv(OUT2 / "evt_tail_spillover.csv", index=False)
    print(f"\nTail spillover pairs with BH q<0.05 on train: {(TAIL_SPILL.q < 0.05).sum()} "
          f"(of these, lift>1 in test: {((TAIL_SPILL.q < 0.05) & (TAIL_SPILL.lift_test > 1)).sum()})")
    print(TAIL_SPILL.head(10).round(3).to_string(index=False))

In [ ]:
# step 13
fig, axs = plt.subplots(1, 3, figsize=(17, 4.2))
ax = axs[0]
for k, (ev, col) in enumerate([("up shock", SERIES[0]), ("down shock", SERIES[1])]):
    for sp, ls in [("train", "-"), ("test", "--")]:
        d_ = EVS[(EVS.event == ev) & (EVS.split == sp)]
        ax.plot(d_.h, d_.continuation, ls=ls, marker="o", color=col, label=f"{ev}, {sp}")
ax.axhline(0, color=INK2, lw=1); ax.set_xlabel("days after the tail event"); ax.set_ylabel("continuation (+) / reversal (−), vol-scaled")
ax.legend(frameon=False, fontsize=7); ax.set_title("Daily tail events: what happens next")
ax = axs[1]
if HAVE_5M:
    pv = EVB[(EVB.split == "train")].pivot_table(index=["bucket", "shock"], columns="k", values="cont")
    heat(ax, pv, cmap=DIV, center=0, label="continuation (train)", annot=True)
    ax.set_title("5-min extremes by time of day: continuation (+) / reversal (−)")
else:
    ax.axis("off")
ax = axs[2]
ax.plot(CLU_D.index, CLU_D["empirical"], color=SERIES[0], marker="o", ms=3, label="daily, empirical")
ax.plot(CLU_D.index, CLU_D["no clustering"], color=SERIES[0], ls="--", label="daily, no clustering")
if CLU_5 is not None:
    ax.plot(CLU_5.index, CLU_5["empirical"], color=SERIES[1], marker="o", ms=3, label="5-min, empirical")
    ax.plot(CLU_5.index, CLU_5["no clustering"], color=SERIES[1], ls="--", label="5-min, no clustering")
ax.set_xlabel("k bars"); ax.set_ylabel("P(next extreme within k bars)"); ax.legend(frameon=False, fontsize=7)
ax.set_title("Extremes cluster (solid above dashed)")
plt.tight_layout()
savefig(fig, "13_evt_opportunities", "EVT opportunities: event study, time buckets, clustering", "13")
if HAVE_5M and xi_b:
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.4))
    lf = EVB[(EVB.split == "train") & (EVB.k == EVT_H_BARS[0]) & (EVB.shock == "up")].set_index("bucket").lift
    axs[0].bar(lf.index, lf.values, color=SERIES[0]); axs[0].axhline(1, color=INK2, lw=1)
    axs[0].set_title("Where intraday extremes happen (lift vs average rate)")
    axs[1].bar(list(xi_b), list(xi_b.values()), color=SERIES[1]); axs[1].axhline(0, color=INK2, lw=1)
    axs[1].set_title("Tail heaviness (GPD ξ) by time of day")
    plt.tight_layout()
    savefig(fig, "13_evt_buckets", "EVT by time-of-day bucket", "13")

### 13.4 — Do the links in 13.3 and the lead–lags survive beyond a few days?

Section 13.3 looked at tail events up to 10 days ahead and at spillover to the **next day** only. A researcher also asks whether an effect lives at weeks and months, because that decides the holding period.

* The **event study** of 13.3 now runs to `EVT_H` (up to 60 days).
* **Tail spillover over longer windows.** An extreme move in $X$ on day $t$: does $Y$ have an extreme move at any time in the next 1, 5, 20 trading days, more often than its base rate for such a window? (`TAIL_SPILL_W`)
* **Return lead–lag at long horizons.** Rank correlation, on non-overlapping blocks, of the trailing $h$-day return of a *driver* (own sector ex the product, the rest of the market, the product itself) with the product's next $h$-day return, for $h$ = 5, 20, 60 days (`LR_H`). Estimated in the three periods separately.
* Weekly Granger links are tested in Step 14 (time frame `1W`), with lag decay out to 60 days.

**Used later.** Horizons where an effect still holds in validation and test set the holding period of the event strategy (C3) and tell Step 21 which momentum / reversal lookbacks are worth searching. Long-horizon sector links feed the sector-rotation tests in Step 21.

In [ ]:
# step 13
# ---- 13.4a tail spillover over windows of 1, 5, 20 days ----
A_ = (E_D != 0).astype(float).loc[:, UNIV_V]; ix_tr, ix_va, ix_te = (RAW.index.intersection(SPL[k]) for k in ("train", "val", "test"))
sw_rows = []
for W_ in TAIL_SPILL_W:
    Yw = A_.rolling(W_, min_periods=W_).max().shift(-W_)                     # an extreme in Y at any time in t+1 .. t+W
    res_ = {}
    for sp, ix in (("train", ix_tr), ("val", ix_va), ("test", ix_te)):
        a = A_.loc[ix]; y = Yw.loc[ix]; ok = y.notna().all(axis=1); a, y = a[ok].to_numpy(), y[ok].to_numpy()
        co = a.T @ y; nx_ = a.sum(0); py = y.mean(0); lift = co / np.maximum(nx_[:, None], 1) / np.maximum(py[None, :], 1e-9)
        res_[sp] = (co, nx_, py, lift)
    co, nx_, py, lift = res_["train"]
    for i, ca in enumerate(UNIV_V):
        for j, cb in enumerate(UNIV_V):
            if i != j and co[i, j] >= 8 and nx_[i] >= 10:
                pb_ = stats.binomtest(int(co[i, j]), int(nx_[i]), float(min(max(py[j], 1e-6), 0.999)), alternative="greater").pvalue
                sw_rows.append(dict(window=W_, cause=ca, effect=cb, n_events=int(nx_[i]), lift_train=lift[i, j], lift_val=res_["val"][3][i, j], lift_test=res_["test"][3][i, j], p=pb_))
TSW = pd.DataFrame(sw_rows)
if len(TSW):
    TSW["q"] = TSW.groupby("window").p.transform(bh)
    TSW["replicates"] = (TSW.q < 0.05) & (TSW.lift_val > 1) & (TSW.lift_test > 1)
    TSW.sort_values(["window", "lift_train"], ascending=[True, False]).to_csv(OUT2 / "evt_tail_spillover_windows.csv", index=False)
    summ = TSW.groupby("window").agg(pairs=("p", "size"), sig_train=("q", lambda q: int((q < 0.05).sum())), replicate_val_test=("replicates", "sum"))
    summ["expected_by_chance"] = (summ.pairs * 0.05).round(0)
    print("Tail spillover (extreme in X at t → extreme in Y within the next W days): pairs, BH q<0.05 on train, and also lift>1 in validation AND test\n", summ.to_string())
    print(TSW[TSW.replicates].sort_values("lift_train", ascending=False).head(10).round(3).to_string(index=False) if TSW.replicates.any() else "no spillover pair replicates in validation and test at any window")

In [ ]:
# step 13
# ---- 13.4b return lead-lag at 5 / 20 / 60 days (non-overlapping blocks) ----
LRR = []
sec_of = SEC_MAP.reindex(UNIV_V)
for h in LR_H:
    cum = RAW[UNIV_V].fillna(0).rolling(h).sum(); fwdh = cum.shift(-h)
    mkt_ex = (cum.sum(axis=1).to_numpy()[:, None] - cum.to_numpy()) / (len(UNIV_V) - 1)
    MKT_EX = pd.DataFrame(mkt_ex, index=cum.index, columns=cum.columns)
    sec_sum = cum.T.groupby(sec_of).transform("sum").T; sec_cnt = cum.notna().T.groupby(sec_of).transform("sum").T
    SEC_EX = ((sec_sum - cum) / (sec_cnt - 1)).where(sec_cnt > 1)
    for sp, ix in (("train", ix_tr), ("val", ix_va), ("test", ix_te)):
        pick = ix[::h]                                                         # non-overlapping
        for nm_, drv in (("own sector ex product", SEC_EX), ("rest of market", MKT_EX), ("product itself", cum)):
            ic = xs_ic_fast(drv.loc[pick], fwdh.loc[pick], min_n=5)
            LRR.append(dict(h=h, driver=nm_, split=sp, ic=ic.mean() if len(ic) else np.nan, t=ic.mean() / ic.std() * np.sqrt(len(ic)) if len(ic) > 5 and ic.std() > 0 else np.nan, n_blocks=len(ic)))
LRR = pd.DataFrame(LRR); LRR.to_csv(OUT2 / "lead_lag_long_horizon.csv", index=False)
print("Rank IC of the trailing h-day return of a driver with the product's next h-day return (non-overlapping blocks; cross-sectional, mean over blocks):")
print(LRR.pivot_table(index=["driver", "h"], columns="split", values=["ic", "t"])[[("ic", "train"), ("ic", "val"), ("ic", "test"), ("t", "train"), ("t", "val"), ("t", "test")]].round(3).to_string())
rep_h = LRR.pivot_table(index=["driver", "h"], columns="split", values="ic")
rep_h = rep_h[(np.sign(rep_h.train) == np.sign(rep_h.val)) & (np.sign(rep_h.val) == np.sign(rep_h.test)) & (rep_h.abs() > 0.01).all(axis=1)]
print("\nSame sign in train, validation and test (and |IC| > 0.01):", rep_h.round(3).to_dict("index") if len(rep_h) else "none")

## Step 13b — Multivariate structure: cross-section, joint dynamics, sectors

Steps 9–13 describe **one series at a time**. Everything that follows (relationships, factors, portfolios, risk) works on **many series at once**, so a researcher measures that structure before building on it. Five layers, each on the same calendar splits, using data already in memory (no per-pair model fits):

| Layer | Question | Method | Used later by |
|---|---|---|---|
| **1 Cross-section** (all products on one date) | How many common factors drive the 70+ products? How dispersed are they? | PCA of the train correlation matrix, **Marchenko–Pastur** edge for the number of real factors, loading stability across periods, cross-sectional dispersion | Step 18 (what XS / SEC levels can capture), Step 21 (dispersion state), Step 24 (risk model) |
| **2 Multi-product** (joint dynamics) | Does diversification fail in stress? Which covariance estimator forecasts best? | rolling average correlation by volatility regime and its persistence; horse race of 6 covariance estimators on validation and test (variance forecast QLIKE, realised volatility of the minimum-variance portfolio) | `COV_BEST` → sector risk in Step 24, volatility targets |
| **3 Multi-product, cross-sectional** | Which characteristics predict returns *jointly*, controlling for each other? | Fama–MacBeth: daily cross-sectional regressions of next-day vol-scaled returns on lagged return, volatility ratio, volume and OI characteristics; Newey–West t | Step 18 (which families add information beyond the others) |
| **4 Sectoral** (within a sector) | Are sectors cohesive, and how much of a product's variance is idiosyncratic? | sector series battery (ADF, ARCH, kurtosis, autocorrelation), within-sector correlation and first-PC share, R² of market + own sector | Step 18 within-sector (WS) factors and Step 23 pairs: room exists only where idiosyncratic variance is large |
| **5 Multi-sectoral** (between sectors) | Do sectors lead each other? How stable is the sector correlation structure? | sector PCA, train-vs-test correlation drift, Granger between sector series (daily and weekly) with BH correction | Step 21 sector rotation, Step 22 monitor |

In [ ]:
# step 13b
# ---- 13b.1 cross-section: factor structure, random-matrix test, loading stability, dispersion ----
ZU = Z[UNIV]; PC = {}
for sp in ("train", "val", "test"):
    X = ZU.loc[ZU.index.intersection(SPL[sp])]; X = X.loc[:, X.notna().mean() > 0.8].fillna(0.0)
    C = np.corrcoef(X.to_numpy().T); w, V = np.linalg.eigh(C); o = np.argsort(-w); PC[sp] = (X.columns, w[o], V[:, o], len(X))
tr_cols, w_tr, V_tr, T_tr = PC["train"]; N_ = len(tr_cols)
mp_edge = (1 + np.sqrt(N_ / T_tr)) ** 2; n_sig = int((w_tr > mp_edge).sum())
share = w_tr / w_tr.sum()
def cos_pc(sp, k):
    c2, _, V2, _ = PC[sp]; common = [c for c in tr_cols if c in set(c2)]
    a = V_tr[[list(tr_cols).index(c) for c in common], k]; b = V2[[list(c2).index(c) for c in common], k]
    return abs(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"{N_} products, {T_tr} train days: Marchenko–Pastur edge {mp_edge:.2f} → {n_sig} eigenvalues above noise; PC1 explains {share[0]:.1%}, first {max(n_sig,1)} explain {share[:max(n_sig,1)].sum():.1%}")
print("loading stability (|cosine| of the PC loading, train vs period):", {f"PC{k+1} {sp}": round(cos_pc(sp, k), 2) for k in range(min(3, max(n_sig, 1))) for sp in ("val", "test")})
print("PC1 share by period:", {sp: round(float(PC[sp][1][0] / PC[sp][1].sum()), 3) for sp in PC})
pc1 = pd.Series(V_tr[:, 0] * np.sign(V_tr[:, 0].sum()), index=tr_cols)
PC_TAB = pd.DataFrame({"pc1_loading": pc1, "pc2_loading": pd.Series(V_tr[:, 1], index=tr_cols), "sector": SEC_MAP.reindex(tr_cols)})
# dispersion and cross-sectional higher moments
disp = ZU.sub(ZU.mean(axis=1), axis=0).abs().mean(axis=1).where(ZU.notna().sum(axis=1) >= MIN_XS)
xs_skew = ZU.skew(axis=1).where(ZU.notna().sum(axis=1) >= MIN_XS); xs_kurt = ZU.kurt(axis=1).where(ZU.notna().sum(axis=1) >= MIN_XS)
DISP = pd.DataFrame({"dispersion": disp, "xs_skew": xs_skew, "xs_kurt": xs_kurt})
mkt_ret = RAW[UNIV].mean(axis=1)
print(f"cross-sectional dispersion: median {disp.median():.2f}, persistence (AR1) {disp.autocorr(1):.2f}; corr(dispersion, |market return|) = {disp.corr(mkt_ret.abs()):.2f}; median XS excess kurtosis {xs_kurt.median():.2f}")

In [ ]:
# step 13b
# ---- 13b.2 joint dynamics: correlation by regime, persistence, covariance estimators ----
from sklearn.covariance import LedoitWolf
avgc = avg_corr(ZU, 60) if "avg_corr" in globals() else None
if avgc is None:
    def _avg_corr(X, win):
        out = pd.Series(np.nan, index=X.index)
        for i in range(win, len(X)):
            C = np.corrcoef(X.iloc[i - win:i].fillna(0).to_numpy().T); out.iloc[i] = (C.sum() - len(C)) / (len(C) * (len(C) - 1))
        return out
    avgc = _avg_corr(ZU, 60)
mvol = RAW[UNIV].mean(axis=1).rolling(20).std()
q_ = mvol.loc[ZU.index.intersection(SPL["train"])].quantile([1 / 3, 2 / 3])
reg_c = pd.cut(mvol, [-np.inf, q_.iloc[0], q_.iloc[1], np.inf], labels=["LOW", "MID", "HIGH"])
CORR_REG = pd.DataFrame({sp: avgc.loc[avgc.index.intersection(SPL[sp])].groupby(reg_c).mean() for sp in ("train", "val", "test")})
print("average pairwise correlation (60-day) by market-volatility regime:\n", CORR_REG.round(3).to_string())
ac = avgc.dropna(); phi = ac.autocorr(1) if len(ac) > 30 else np.nan
print(f"correlation level persistence AR(1) = {phi:.3f} → half-life {np.log(0.5)/np.log(phi):.0f} days" if 0 < phi < 1 else f"AR(1) = {phi}")
print(f"diversification fails in stress if HIGH > LOW: train {CORR_REG.loc['HIGH','train']-CORR_REG.loc['LOW','train']:+.3f} | val {CORR_REG.loc['HIGH','val']-CORR_REG.loc['LOW','val']:+.3f} | test {CORR_REG.loc['HIGH','test']-CORR_REG.loc['LOW','test']:+.3f}")

# covariance horse race on daily vol-scaled-free returns (log returns of the universe), re-estimated every COV_REBAL days from the past only
RU = RAW[UNIV].fillna(0.0); dates_ = RU.index; Nn = RU.shape[1]
def cov_est(X, name):
    if name == "sample":
        return np.cov(X.T)
    if name == "EWMA94":
        w = 0.94 ** np.arange(len(X))[::-1]; w /= w.sum(); mu = (X * w[:, None]).sum(0); Xc = X - mu; return (Xc * w[:, None]).T @ Xc
    if name == "EWMA97":
        w = 0.97 ** np.arange(len(X))[::-1]; w /= w.sum(); mu = (X * w[:, None]).sum(0); Xc = X - mu; return (Xc * w[:, None]).T @ Xc
    if name == "LedoitWolf":
        return LedoitWolf().fit(X).covariance_
    if name == "const-corr":
        S = np.cov(X.T); sd = np.sqrt(np.diag(S)); C = S / np.outer(sd, sd); rbar = (C.sum() - len(C)) / (len(C) * (len(C) - 1))
        w = 0.94 ** np.arange(len(X))[::-1]; w /= w.sum(); sd_e = np.sqrt(((X - X.mean(0)) ** 2 * w[:, None]).sum(0))
        Cc = np.full_like(C, rbar); np.fill_diagonal(Cc, 1.0); return Cc * np.outer(sd_e, sd_e)
    if name == "1-factor":
        S = np.cov(X.T); m = X.mean(axis=1); b = np.array([np.cov(X[:, j], m)[0, 1] for j in range(X.shape[1])]) / m.var(); res = X - np.outer(m, b)
        return np.outer(b, b) * m.var() + np.diag(res.var(0))
EST = ["sample", "EWMA94", "EWMA97", "LedoitWolf", "const-corr", "1-factor"]
reb_ = list(range(COV_WIN, len(RU) - 1, COV_REBAL)); recs = []
SP_OF = {d: sp for sp in ("train", "val", "test") for d in SPL[sp]}
Rv = RU.to_numpy()
for i in reb_:
    X = Rv[i - COV_WIN:i]; ok = (X != 0).mean(0) > 0.8
    if ok.sum() < 10:
        continue
    X = X[:, ok]; fwd = Rv[i:i + COV_REBAL][:, ok]
    if len(fwd) < 5:
        continue
    for e in EST:
        try:
            S = cov_est(X, e) + np.eye(X.shape[1]) * 1e-10; ew = np.ones(X.shape[1]) / X.shape[1]
            Si = np.linalg.pinv(S); wg = Si.sum(1); wg = np.clip(wg / wg.sum(), 0, None); wg /= wg.sum()               # long-only-ish minimum variance
            pf_ew, pf_gm = fwd @ ew, fwd @ wg; h_ew = ew @ S @ ew
            recs.append(dict(date=dates_[i], est=e, qlike_ew=float(qlike(max(float(pf_ew.var()), 1e-12), float(h_ew))) if h_ew > 0 else np.nan, vol_gmv=pf_gm.std() * np.sqrt(ANN), sp=SP_OF.get(dates_[i], "gap")))
        except Exception:
            pass
COVR = pd.DataFrame(recs)
COVT = COVR[COVR.sp.isin(["val", "test"])].groupby(["est", "sp"])[["qlike_ew", "vol_gmv"]].mean().unstack("sp")
COVT["score_val"] = COVT[("qlike_ew", "val")].rank() + COVT[("vol_gmv", "val")].rank()
COV_BEST = COVT["score_val"].idxmin()
print("\nCovariance estimators out of sample (rebalanced every %d days; QLIKE of the equal-weight portfolio variance forecast, realised volatility of the minimum-variance book):\n" % COV_REBAL, COVT.round(4).to_string())
print(f"→ chosen on validation: {COV_BEST} (test value in the same table)")
COV_FN = lambda X, name=None: cov_est(np.asarray(X, float), name or COV_BEST)

In [ ]:
# step 13b
# ---- 13b.3 multi-product cross-sectional: Fama-MacBeth with several characteristics at once ----
def nw_t(x, L=5):
    x = pd.Series(x).dropna().to_numpy(); n = len(x)
    if n < 20:
        return np.nan, np.nan
    m = x.mean(); e = x - m; g0 = e @ e / n; s = g0 + 2 * sum((1 - l / (L + 1)) * (e[l:] @ e[:-l] / n) for l in range(1, L + 1))
    return m, m / np.sqrt(max(s, 1e-18) / n)
CH = {"ret_1d": RAW[UNIV], "ret_5d": RAW[UNIV].rolling(5).sum(), "ret_20d": RAW[UNIV].rolling(20).sum(),
      "vol_ratio": np.log(vratio).replace([np.inf, -np.inf], np.nan) if "vratio" in globals() else np.log(RV[UNIV].rolling(10).mean() / RV[UNIV].rolling(60).mean()),
      "volume_z": (np.log1p(VOLU[UNIV]) - np.log1p(VOLU[UNIV]).rolling(60).mean()) / np.log1p(VOLU[UNIV]).rolling(60).std(),
      "oi_chg_20d": np.log(OIW[UNIV].where(OIW[UNIV] > 0)).diff(20)}
CHR = {k: v.rank(axis=1, pct=True) - 0.5 for k, v in CH.items()}                       # ranks: comparable, robust
fwd1 = Z[UNIV].shift(-1)                                                                # next-day vol-scaled return
names_ = list(CHR); betas = {}
for dt in RAW.index:
    y = fwd1.loc[dt]; X = pd.concat([CHR[k].loc[dt] for k in names_], axis=1, keys=names_).assign(y=y).dropna()
    if len(X) >= MIN_XS + 4:
        A = np.c_[np.ones(len(X)), X[names_].to_numpy()]; b, *_ = np.linalg.lstsq(A, X["y"].to_numpy(), rcond=None); betas[dt] = b[1:]
FMB = pd.DataFrame(betas, index=names_).T
fm_rows = []
for k in names_:
    r = {"characteristic": k}
    for sp in ("train", "val", "test"):
        m, t = nw_t(FMB[k].loc[FMB.index.intersection(SPL[sp])]); r[f"slope_{sp}"], r[f"t_{sp}"] = m, t
    fm_rows.append(r)
FMT = pd.DataFrame(fm_rows).set_index("characteristic"); FMT["same_sign_all"] = (np.sign(FMT.slope_train) == np.sign(FMT.slope_val)) & (np.sign(FMT.slope_val) == np.sign(FMT.slope_test))
FMT.round(4).to_csv(OUT2 / "fama_macbeth.csv")
print("Fama–MacBeth (rank-standardised characteristics, joint regression; slope = effect of a move from the lowest to the highest rank on the next day's vol-scaled return; Newey–West t):\n", FMT.round(3).to_string())

In [ ]:
# step 13b
# ---- 13b.4 sectoral: sector series battery, cohesion, idiosyncratic share ----
SECL = sorted(set(SEC_MAP.reindex(UNIV)))
SECZ = pd.DataFrame({s: Z[[c for c in UNIV if SEC_MAP[c] == s]].mean(axis=1) for s in SECL if (SEC_MAP.reindex(UNIV) == s).sum() >= 3})
trs = Z.index.intersection(SPL["train"])
sec_rows = []
for s in SECZ:
    x = SECZ[s].dropna(); xt = x.loc[x.index.intersection(SPL["train"])]; mem = [c for c in UNIV if SEC_MAP[c] == s]
    C = Z.loc[trs, mem].corr().to_numpy(); n_ = len(mem); within = (C.sum() - n_) / (n_ * (n_ - 1))
    w_ = np.linalg.eigvalsh(C)[::-1]
    sec_rows.append(dict(sector=s, n=n_, within_corr_train=within, within_corr_test=(lambda Ct: (Ct.sum() - n_) / (n_ * (n_ - 1)))(Z.loc[Z.index.intersection(SPL["test"]), mem].corr().to_numpy()),
                         pc1_share=w_[0] / w_.sum(), ac1=xt.autocorr(1), arch_lm_p=het_arch(xt.to_numpy(), nlags=5)[1], excess_kurt=stats.kurtosis(xt),
                         vr5=float(xt.rolling(5).sum().var() / (5 * xt.var())), beta_to_market=float(np.cov(xt, Z.loc[xt.index, UNIV].mean(axis=1))[0, 1] / Z.loc[xt.index, UNIV].mean(axis=1).var())))
SECB = pd.DataFrame(sec_rows).set_index("sector"); SECB.round(3).to_csv(OUT2 / "sector_series_battery.csv")
print("Sector series (equal-risk average of vol-scaled returns): cohesion and univariate facts\n", SECB.round(3).to_string())
# idiosyncratic share: R² of a product on the market and on its own sector excluding itself (train)
mk = Z[UNIV].mean(axis=1); idio = {}
for c in UNIV:
    mem = [m for m in UNIV if SEC_MAP[m] == SEC_MAP[c] and m != c]
    d = pd.concat([Z[c], mk.rename("mkt"), (Z[mem].mean(axis=1) if len(mem) else pd.Series(np.nan, index=Z.index)).rename("sec")], axis=1).loc[trs].dropna(subset=[c, "mkt"])
    cols = ["mkt"] + (["sec"] if len(mem) and d["sec"].notna().all() else [])
    rss, b = ols(np.c_[np.ones(len(d)), d[cols].to_numpy()], d[c].to_numpy()); idio[c] = rss / np.sum((d[c].to_numpy() - d[c].mean()) ** 2)
IDIO = pd.Series(idio).sort_values(); SEC_IDIO = IDIO.groupby(SEC_MAP.reindex(IDIO.index)).mean().sort_values()
print(f"\nIdiosyncratic share of variance after market and own-sector (train): median {IDIO.median():.2f}; by sector (low = products move together, little room for within-sector relative value):\n", SEC_IDIO.round(2).to_string())

In [ ]:
# step 13b
# ---- 13b.5 multi-sectoral: sector PCA, correlation drift, sector lead-lag (daily and weekly) ----
Cs_tr = SECZ.loc[SECZ.index.intersection(SPL["train"])].corr(); Cs_te = SECZ.loc[SECZ.index.intersection(SPL["test"])].corr()
w_s = np.linalg.eigvalsh(Cs_tr.to_numpy())[::-1]
print(f"{SECZ.shape[1]} sectors: PC1 of the sector correlation matrix explains {w_s[0]/w_s.sum():.1%}; mean off-diagonal correlation train {(Cs_tr.values.sum()-len(Cs_tr))/(len(Cs_tr)*(len(Cs_tr)-1)):.2f} → test {(Cs_te.values.sum()-len(Cs_te))/(len(Cs_te)*(len(Cs_te)-1)):.2f}; "
      f"max |change| of a pair correlation {np.nanmax(np.abs((Cs_te - Cs_tr).values)):.2f}")
SG_ROWS = []
for lab, S_ in (("1D", SECZ), ("1W", SECZ.resample("W-FRI").sum(min_count=3))):
    spl_ = {"train": S_.index[S_.index <= SPL["train"][-1]], "val": S_.index[(S_.index >= SPL["val"][0]) & (S_.index <= SPL["val"][-1])], "test": S_.index[S_.index >= SPL["test"][0]]}
    if min(len(v) for v in spl_.values()) < 60:
        continue
    G = {sp: granger_matrix(S_.loc[spl_[sp]].dropna(how="all"), 2) for sp in spl_}
    G["train"]["q"] = bh(G["train"]["p"])
    mrg = G["train"][["cause", "effect", "p", "q", "partial_r2", "coef_sum"]].merge(G["val"][["cause", "effect", "p", "coef_sum"]], on=["cause", "effect"], suffixes=("", "_val")).merge(
        G["test"][["cause", "effect", "p", "coef_sum"]].rename(columns={"p": "p_test", "coef_sum": "coef_test"}), on=["cause", "effect"])
    mrg["tf"] = lab; mrg["sig_train"] = mrg.q < 0.10
    mrg["replicates"] = mrg.sig_train & (np.sign(mrg.coef_sum) == np.sign(mrg.coef_sum_val)) & (np.sign(mrg.coef_sum) == np.sign(mrg.coef_test)) & (mrg.p_val < 0.10) & (mrg.p_test < 0.10)
    SG_ROWS.append(mrg)
SECG = pd.concat(SG_ROWS) if SG_ROWS else pd.DataFrame()
if len(SECG):
    SECG.to_csv(OUT2 / "sector_lead_lag.csv", index=False)
    print("\nSector → sector lead–lag (Granger, 2 lags; BH q<0.10 on train; replicates = same sign and p<0.10 in validation and test):\n",
          SECG.groupby("tf").agg(pairs=("p", "size"), sig_train=("sig_train", "sum"), replicates=("replicates", "sum")).to_string())
    print(SECG[SECG.replicates].sort_values("p").head(8)[["tf", "cause", "effect", "p", "p_val", "p_test", "coef_sum"]].round(4).to_string(index=False) if SECG.replicates.any() else "no sector link replicates in both later periods")
STRUCT = dict(n_pc_above_noise=n_sig, pc1_share=float(share[0]), cov_best=COV_BEST, idio_median=float(IDIO.median()))

In [ ]:
# step 13b
fig, axs = plt.subplots(2, 3, figsize=(18, 9))
k_pc = min(15, len(share)); ax = axs[0, 0]; ax.bar(range(1, k_pc + 1), share[:k_pc] * 100, color=[SERIES[0] if w > mp_edge else MUTE for w in w_tr[:k_pc]])
ax.set_xlabel("principal component"); ax.set_ylabel("% of variance"); ax.set_title(f"Cross-section: {n_sig} components above the random-matrix edge (blue)")
ax = axs[0, 1]; hp = PC_TAB.copy(); hp["sector"] = hp["sector"].fillna("Other")
for k, (s_, g) in enumerate(hp.groupby("sector")):
    ax.scatter(g.pc1_loading, g.pc2_loading, s=22, color=SERIES[k % 8], label=s_)
ax.axhline(0, color=INK2, lw=1); ax.axvline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=6, ncol=2); ax.set_xlabel("PC1 (market)"); ax.set_ylabel("PC2"); ax.set_title("Loadings coloured by sector")
ax = axs[0, 2]; ax.plot(avgc.index, avgc.values, color=SERIES[0], lw=0.8); ax2 = ax.twinx(); ax2.plot(mvol.index, mvol.values * np.sqrt(ANN), color=SERIES[1], lw=0.6, alpha=0.7)
ax.set_title("Average pairwise correlation (blue) and market volatility (orange)")
for sp in ("val", "test"):
    ax.axvline(SPL[sp][0], color=INK2, ls="--", lw=1)
ax = axs[1, 0]; cb = COVT[[("vol_gmv", "val"), ("vol_gmv", "test")]]; x_ = np.arange(len(cb)); ax.bar(x_ - 0.2, cb.iloc[:, 0] * 100, 0.4, color=MUTE, label="validation"); ax.bar(x_ + 0.2, cb.iloc[:, 1] * 100, 0.4, color=SERIES[0], label="test")
ax.set_xticks(x_); ax.set_xticklabels(cb.index, rotation=30, fontsize=8); ax.legend(frameon=False, fontsize=8); ax.set_ylabel("realised vol of min-variance book, % p.a."); ax.set_title("Which covariance estimator? (lower is better)")
ax = axs[1, 1]; heat(ax, SECB[["within_corr_train", "within_corr_test", "pc1_share", "beta_to_market"]], cmap=SEQ, label="", annot=True, fmt="{:.2f}"); ax.set_title("Sector cohesion")
ax = axs[1, 2]; heat(ax, Cs_te - Cs_tr, cmap=DIV, center=0, vmax=0.3, label="correlation, test − train"); ax.set_title("Sector correlation drift")
plt.tight_layout()
savefig(fig, "13b_multivariate", "Multivariate structure: cross-section, covariance, sectors", "13")

## Step 13c — Copulas: the dependence structure that correlation does not show

Correlation measures the *average linear* link. A copula separates the **dependence** from the **marginal distributions** (already studied in Steps 11–13): $F(x,y)=C\big(F_X(x),F_Y(y)\big)$. It answers the questions that matter for risk and for relative-value trades: *do two products crash together more than they rally together?* (tail asymmetry), *how heavy is the joint tail?* (degrees of freedom), and *which functional form describes the pair out of sample?*

**Method (all parameters on train, selection on validation, confirmation on test).**
1. **Pseudo-observations**: the rank of each vol-scaled return within its period, divided by $n+1$.
2. **Families per pair**, parameters by inversion of Kendall's $\tau$ (Student-$t$: $\nu$ by likelihood on a grid): Gaussian (no tail dependence), Student-$t$ (symmetric tails), Clayton (lower tail), Gumbel (upper tail). A pair with $\tau<0$ is flipped first.
3. **Selection**: AIC on train; **out-of-sample log-likelihood** of the frozen parameters on validation (decides) and test (confirms). The *gain over Gaussian* is the evidence that a tail-aware copula is needed.
4. **Empirical tail dependence** $\lambda_L(q)=P(V<q\mid U<q)$ and $\lambda_U(q)=P(V>1-q\mid U>1-q)$ per pair and period; independence gives $q$.
5. **Whole universe**: Gaussian versus Student-$t$ copula ($\nu$ by profile likelihood), out-of-sample log-likelihood per day, and **how often many products are in their worst 5% on the same day** — observed against what the Gaussian and the $t$ copula (train parameters) predict. This is the diversification test.

**Used later.** The preferred family and $\nu$ feed the copula portfolio risk of Step 24b and the joint-tail limits of the board (Step 26). Pairs with stable lower-tail dependence are crash-sensitive pairs (Step 25 stress). Step 15's tail dependence (on GARCH residuals) is the complementary view.

In [ ]:
# step 13c
# ---- 13c.1 copula toolbox ----
from scipy.special import gammaln
_NU_GRID = [3, 4, 5, 6, 8, 12, 20, 40]
def pseudo_obs(x):
    return stats.rankdata(x) / (len(x) + 1.0)
def _clip(u): return np.clip(u, 1e-6, 1 - 1e-6)
def ll_gauss(u, v, rho):
    s, t = stats.norm.ppf(_clip(u)), stats.norm.ppf(_clip(v)); r2 = 1 - rho ** 2
    return -0.5 * np.log(r2) - (rho ** 2 * (s ** 2 + t ** 2) - 2 * rho * s * t) / (2 * r2)
def ll_t(u, v, rho, nu):
    x, y = stats.t.ppf(_clip(u), nu), stats.t.ppf(_clip(v), nu); r2 = 1 - rho ** 2
    l2 = gammaln((nu + 2) / 2) - gammaln(nu / 2) - np.log(np.pi * nu) - 0.5 * np.log(r2) - (nu + 2) / 2 * np.log1p((x ** 2 - 2 * rho * x * y + y ** 2) / (nu * r2))
    return l2 - stats.t.logpdf(x, nu) - stats.t.logpdf(y, nu)
def ll_clayton(u, v, th):
    u, v = _clip(u), _clip(v)
    return np.log1p(th) - (1 + th) * (np.log(u) + np.log(v)) - (2 + 1 / th) * np.log(u ** -th + v ** -th - 1)
def ll_gumbel(u, v, th):
    u, v = _clip(u), _clip(v); a, b = -np.log(u), -np.log(v); A = a ** th + b ** th
    return -A ** (1 / th) + (th - 1) * (np.log(a) + np.log(b)) + (2 / th - 2) * np.log(A) + np.log1p((th - 1) * A ** (-1 / th)) + a + b
COP_FAMS = ["Gaussian", "Student-t", "Clayton (lower tail)", "Gumbel (upper tail)"]
def fit_cop_pair(u, v):
    tau = stats.kendalltau(u, v)[0]
    if not np.isfinite(tau): return None
    sgn = 1.0 if tau >= 0 else -1.0
    if sgn < 0: v = 1 - v; tau = -tau
    tau = min(max(tau, 1e-3), 0.95); rho = np.sin(np.pi * tau / 2)
    P = {"sign": sgn, "tau": tau, "Gaussian": (rho,)}
    ll_nu = [(ll_t(u, v, rho, nu).sum(), nu) for nu in _NU_GRID]; P["Student-t"] = (rho, max(ll_nu)[1])
    P["Clayton (lower tail)"] = (2 * tau / (1 - tau),); P["Gumbel (upper tail)"] = (1 / (1 - tau),)
    return P
def cop_ll(P, fam, u, v):
    if P["sign"] < 0: v = 1 - v
    pr = P[fam]
    return {"Gaussian": lambda: ll_gauss(u, v, *pr), "Student-t": lambda: ll_t(u, v, *pr), "Clayton (lower tail)": lambda: ll_clayton(u, v, *pr), "Gumbel (upper tail)": lambda: ll_gumbel(u, v, *pr)}[fam]()
def tail_emp(u, v, q):
    lo = (u < q); hi = (u > 1 - q)
    return (float(((v < q) & lo).sum() / max(lo.sum(), 1)), float(((v > 1 - q) & hi).sum() / max(hi.sum(), 1)))
print("copula toolbox ready:", ", ".join(COP_FAMS))

In [ ]:
# step 13c
# ---- 13c.2 pairs: family selection (train AIC, validation OOS), tail dependence by period ----
ZU = Z[UNIV]; ZPER = {sp: ZU.loc[ZU.index.intersection(SPL[sp])] for sp in ("train", "val", "test")}
cand = []
for a, b in itertools.combinations(UNIV, 2):
    x = ZPER["train"][[a, b]].dropna()
    if len(x) >= 150: cand.append((a, b, "within" if SEC_MAP[a] == SEC_MAP[b] else "cross", abs(stats.spearmanr(x[a], x[b])[0])))
CP = pd.DataFrame(cand, columns=["a", "b", "scope", "abs_rho"])
CP = pd.concat([CP[CP.scope == "within"], CP[CP.scope == "cross"].nlargest(max(COPULA_MAX_PAIRS - int((CP.scope == "within").sum()), 50), "abs_rho")]).head(COPULA_MAX_PAIRS * 2)
rows = []; t0 = time.time()
for a, b, scope, _ in CP.itertuples(index=False):
    U = {}
    for sp in ("train", "val", "test"):
        x = ZPER[sp][[a, b]].dropna()
        if len(x) >= 100: U[sp] = (pseudo_obs(x[a].to_numpy()), pseudo_obs(x[b].to_numpy()))
    if len(U) < 3: continue
    P = fit_cop_pair(*U["train"])
    if P is None: continue
    r = dict(a=a, b=b, scope=scope, sector=SEC_MAP[a] if scope == "within" else "cross", tau_train=P["sign"] * P["tau"], nu=P["Student-t"][1], n_train=len(U["train"][0]))
    for fam in COP_FAMS:
        k = 2 if fam == "Student-t" else 1
        r[f"aic_{fam}"] = 2 * k - 2 * cop_ll(P, fam, *U["train"]).sum()
        for sp in ("val", "test"):
            r[f"oos_{sp}_{fam}"] = float(np.mean(cop_ll(P, fam, *U[sp])))
    for sp in ("train", "val", "test"):
        lo, hi = tail_emp(*U[sp], COPULA_Q); r[f"lamL_{sp}"], r[f"lamU_{sp}"] = lo, hi
        r[f"tau_{sp}"] = float(stats.kendalltau(*U[sp])[0])
    rows.append(r)
COPP = pd.DataFrame(rows)
COPP["best_aic_train"] = COPP[[f"aic_{f}" for f in COP_FAMS]].idxmin(axis=1).str.replace("aic_", "", regex=False)
for sp in ("val", "test"):
    COPP[f"best_{sp}"] = COPP[[f"oos_{sp}_{f}" for f in COP_FAMS]].idxmax(axis=1).str.replace(f"oos_{sp}_", "", regex=False)
    for f in COP_FAMS[1:]:
        COPP[f"gain_{sp}_{f}"] = COPP[f"oos_{sp}_{f}"] - COPP[f"oos_{sp}_Gaussian"]
COPP["asym_train"] = COPP.lamL_train - COPP.lamU_train; COPP["asym_test"] = COPP.lamL_test - COPP.lamU_test
COPP.round(5).to_csv(OUT2 / "copula_pairs.csv", index=False)
print(f"{len(COPP)} pairs ({int((COPP.scope == 'within').sum())} within-sector) fitted in {time.time()-t0:.0f}s; dependence is positive in {(COPP.tau_train > 0).mean():.0%} of them")
share = pd.DataFrame({"train AIC": COPP.best_aic_train.value_counts(normalize=True), "validation OOS": COPP.best_val.value_counts(normalize=True), "test OOS": COPP.best_test.value_counts(normalize=True)}).reindex(COP_FAMS).fillna(0)
print("\nShare of pairs for which each family is best:\n", (share * 100).round(1).to_string())
gain = pd.DataFrame({sp: {f: COPP[f"gain_{sp}_{f}"].mean() for f in COP_FAMS[1:]} for sp in ("val", "test")})
print("\nMean out-of-sample log-likelihood gain over the Gaussian copula (per observation; > 0 = the family describes unseen data better):\n", gain.round(4).to_string())
agree = (COPP.best_val == COPP.best_test).mean()
both = ((COPP[[f"gain_val_{f}" for f in COP_FAMS[1:]]].max(axis=1) > 0) & (COPP[[f"gain_test_{f}" for f in COP_FAMS[1:]]].max(axis=1) > 0)).mean()
print(f"\nthe family chosen on validation is also best on test for {agree:.0%} of pairs; pairs where a tail-aware copula beats Gaussian on BOTH validation and test: {both:.0%}")
asy = COPP.groupby("sector").agg(pairs=("a", "size"), lamL_train=("lamL_train", "mean"), lamU_train=("lamU_train", "mean"), lamL_test=("lamL_test", "mean"), lamU_test=("lamU_test", "mean"), nu_median=("nu", "median"))
asy["asym_train"] = asy.lamL_train - asy.lamU_train; asy["asym_test"] = asy.lamL_test - asy.lamU_test
print(f"\nTail dependence at q={COPULA_Q:.2f} (independence = {COPULA_Q:.2f}) by sector; asym = lower minus upper (> 0: crash together more than rally together):\n", asy.round(3).to_string())
print(f"\nasymmetry persistence: corr(train, test) of lower-minus-upper = {COPP[['asym_train','asym_test']].corr().iloc[0,1]:.2f} | median ν {COPP.nu.median():.0f}")
COPULA_BY_SECTOR = asy

In [ ]:
# step 13c
# ---- 13c.3 the whole universe: Gaussian vs Student-t copula, and simultaneous extremes ----
from scipy.stats import multivariate_t
KC = ZPER["train"].notna().sum().sort_values(ascending=False).index[:COPULA_K]
UP = {}
for sp in ("train", "val", "test"):
    X = ZPER[sp][KC]; UP[sp] = X.rank(pct=True).mul(X.notna().sum().div(X.notna().sum() + 1)).fillna(0.5)     # per-product pseudo-observations; a missing value is neutral (0.5)
def uni_ll(Uf, R, nu):
    if nu is None:
        S = stats.norm.ppf(_clip(Uf.to_numpy())); Ri = np.linalg.inv(R); return -0.5 * np.linalg.slogdet(R)[1] - 0.5 * (np.einsum("ij,jk,ik->i", S, Ri - np.eye(len(R)), S))
    S = stats.t.ppf(_clip(Uf.to_numpy()), nu)
    return multivariate_t(loc=np.zeros(len(R)), shape=R, df=nu).logpdf(S) - stats.t.logpdf(S, nu).sum(axis=1)
Str = stats.norm.ppf(_clip(UP["train"].to_numpy())); R0 = np.corrcoef(Str.T)
R0 = 0.9 * R0 + 0.1 * np.eye(len(R0))
prof = {nu: float(np.mean(uni_ll(UP["train"], R0, nu))) for nu in _NU_GRID}; NU_U = max(prof, key=prof.get)
ll_g = {sp: float(np.mean(uni_ll(UP[sp], R0, None))) for sp in UP}; ll_tt = {sp: float(np.mean(uni_ll(UP[sp], R0, NU_U))) for sp in UP}
COPULA_NU = NU_U
print(f"{len(KC)} products with the most data | Student-t copula: ν = {NU_U} (profile log-lik per day: " + ", ".join(f"{k}:{v:.1f}" for k, v in prof.items()) + ")")
print("log-likelihood per day (higher is better): " + " | ".join(f"{sp}: Gaussian {ll_g[sp]:.2f}, t {ll_tt[sp]:.2f}, gain {ll_tt[sp]-ll_g[sp]:+.2f}" for sp in UP))
# simultaneous lower extremes: number of products in their worst COPULA_Q on the same day
rng_c = np.random.default_rng(SEED); NS = COPULA_SIM; L = np.linalg.cholesky(R0)
Xg = rng_c.standard_normal((NS, len(KC))) @ L.T; Ug = stats.norm.cdf(Xg)
W_ = rng_c.chisquare(NU_U, NS) / NU_U; Xt = (rng_c.standard_normal((NS, len(KC))) @ L.T) / np.sqrt(W_)[:, None]; Ut = stats.t.cdf(Xt, NU_U)
cnt = lambda U, q=COPULA_Q: (U < q).sum(axis=1)
ks = [2, 3, 5, 8, 12]; ks = [k for k in ks if k <= len(KC)]
SIMX = pd.DataFrame({"independent": [1 - stats.binom.cdf(k - 1, len(KC), COPULA_Q) for k in ks], "Gaussian copula": [(cnt(Ug) >= k).mean() for k in ks], "Student-t copula": [(cnt(Ut) >= k).mean() for k in ks]}, index=ks)
for sp in ("train", "val", "test"):
    SIMX[f"observed {sp}"] = [float((cnt(UP[sp].to_numpy()) >= k).mean()) for k in ks]
SIMX.index.name = "≥ k products in their worst 10% on the same day"; SIMX.to_csv(OUT2 / "copula_joint_extremes.csv")
print(f"\nP(at least k of {len(KC)} products are in their worst {COPULA_Q:.0%} on the same day):\n", SIMX.round(4).to_string())
tail_k = ks[-2] if len(ks) > 1 else ks[0]
better = "Student-t copula" if abs(np.log((SIMX.loc[tail_k, "Student-t copula"] + 1e-9) / (SIMX.loc[tail_k, "observed val"] + 1e-9))) < abs(np.log((SIMX.loc[tail_k, "Gaussian copula"] + 1e-9) / (SIMX.loc[tail_k, "observed val"] + 1e-9))) else "Gaussian copula"
print(f"closest to the observed frequency of {tail_k}+ simultaneous extremes on validation: {better} (Gaussian {SIMX.loc[tail_k, 'Gaussian copula']:.4f}, t {SIMX.loc[tail_k, 'Student-t copula']:.4f}, observed {SIMX.loc[tail_k, 'observed val']:.4f}, independent {SIMX.loc[tail_k, 'independent']:.4f})")
COPULA_SUMMARY = dict(nu=NU_U, ll_gain_val=ll_tt["val"] - ll_g["val"], ll_gain_test=ll_tt["test"] - ll_g["test"], best_family_val=COPP.best_val.mode().iloc[0], best_family_test=COPP.best_test.mode().iloc[0],
                      lower_minus_upper_test=float(COPP.asym_test.mean()), asym_persistence=float(COPP[['asym_train','asym_test']].corr().iloc[0, 1]), closest_joint_tail=better, pairs=len(COPP))

In [ ]:
# step 13c
fig, axs = plt.subplots(2, 3, figsize=(18, 9))
ax = axs[0, 0]; x_ = np.arange(len(COP_FAMS)); wd = 0.27
for k, c_ in enumerate(share.columns): ax.bar(x_ + (k - 1) * wd, share[c_].values * 100, wd, color=[MUTE, SERIES[0], SERIES[1]][k], label=c_)
ax.set_xticks(x_); ax.set_xticklabels([f.replace(" (", "\n(") for f in COP_FAMS], fontsize=8); ax.set_ylabel("% of pairs"); ax.legend(frameon=False, fontsize=8); ax.set_title("Which copula family is best?")
ax = axs[0, 1]; secs = sorted(COPP.sector.unique())
for k, s_ in enumerate(secs):
    g_ = COPP[COPP.sector == s_]; ax.scatter(g_.lamU_test, g_.lamL_test, s=16, color=SERIES[k % 8], label=s_, alpha=0.8)
lim_ = [0, max(COPP.lamL_test.max(), COPP.lamU_test.max()) * 1.05]; ax.plot(lim_, lim_, color=INK2, lw=1, ls="--"); ax.axhline(COPULA_Q, color=MUTE, lw=0.8); ax.axvline(COPULA_Q, color=MUTE, lw=0.8)
ax.set_xlabel("upper-tail dependence (test)"); ax.set_ylabel("lower-tail dependence (test)"); ax.legend(frameon=False, fontsize=6, ncol=2); ax.set_title("Crash together (above the line) or rally together (below)?")
ax = axs[0, 2]; gg = pd.DataFrame({f"{f.split(' (')[0]} {sp}": COPP[f"gain_{sp}_{f}"] for f in COP_FAMS[1:] for sp in ("val", "test")})
ax.boxplot([gg[c].dropna() for c in gg], showfliers=False); ax.set_xticks(range(1, len(gg.columns) + 1)); ax.set_xticklabels(gg.columns, rotation=60, ha="right", fontsize=7)
ax.axhline(0, color=INK2, lw=1); ax.set_title("Out-of-sample log-likelihood gain over Gaussian (per pair)")
ax = axs[1, 0]; ax.plot(list(prof), list(prof.values()), marker="o", color=SERIES[0]); ax.axvline(NU_U, color=SERIES[1], ls="--"); ax.set_xlabel("degrees of freedom ν"); ax.set_ylabel("train log-lik per day"); ax.set_title(f"Student-t copula of {len(KC)} products: ν = {NU_U}")
ax = axs[1, 1]
for c_, col in (("independent", MUTE), ("Gaussian copula", SERIES[2]), ("Student-t copula", SERIES[1]), ("observed val", INK), ("observed test", SERIES[0])):
    ax.plot(SIMX.index, SIMX[c_], marker="o", color=col, label=c_, lw=1.4)
ax.set_yscale("log"); ax.set_xlabel("k products in their worst 10% on the same day"); ax.set_ylabel("probability of k or more"); ax.legend(frameon=False, fontsize=8); ax.set_title("Joint crashes: observed vs copula models")
ax = axs[1, 2]; a_ = COPULA_BY_SECTOR[["asym_train", "asym_test"]].sort_values("asym_test"); ax.barh(np.arange(len(a_)) - 0.18, a_.asym_train, 0.36, color=MUTE, label="train"); ax.barh(np.arange(len(a_)) + 0.18, a_.asym_test, 0.36, color=SERIES[0], label="test")
ax.set_yticks(range(len(a_))); ax.set_yticklabels(a_.index, fontsize=8); ax.axvline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=8); ax.set_title("Lower minus upper tail dependence by sector")
plt.tight_layout()
savefig(fig, "13c_copulas", "Copulas: families, tail asymmetry, joint extremes", "13")

## Step 13d — The overnight return puzzle and the "T+1" effect

**The claim.** In China's A-share market a stock bought today cannot be sold until tomorrow (T+1). Short-term buyers therefore cannot exit intraday, and the selling they would have done pushes into the **next open**. Two observable consequences: (i) most of the return is earned **overnight** (close → open) and the intraday part is weak or negative (the *overnight return puzzle*); (ii) an intraday rally is followed by selling pressure at the next open (a gap fade, or a weaker overnight return after strong days).

**What this dataset can and cannot say.** These are **futures**: they trade T+0 (a position can be closed the same day, and shorted). The rule itself does not bind them. The mechanism can only reach a futures product through a T+1 underlying: the **equity-index futures** (IF, IH, IC, IM), whose price is tied to the stock index by arbitrage. So the test is a *comparison*: does the pattern appear in `EquityIndex` and **not** in commodities and bonds? If it appears everywhere, it is an overnight risk-premium or night-session effect, not a T+1 effect. Results are reported as found.

**Decomposition** (every return is a sum of clean 1-minute returns, so rolls never enter):
$$ r_t \;=\; \underbrace{r^{\text{night}}_t + r^{\text{open bar}}_t}_{\text{ON: previous close → first bar}} \;+\; \underbrace{r^{\text{rest of day}}_t}_{\text{ID: first bar → close}} $$
For products with a night session, ON includes the night session. Returns are also divided by the ex-ante daily volatility (vol units), so products are comparable.

| Test | Question |
|---|---|
| 1. ON vs ID by sector and period | Is the return earned overnight? Mean in bp / day, Newey–West $t$, ON share of the total |
| 2. Pent-up pressure | Does the intraday move of yesterday (up to 30 minutes before the close, so it is known to an entry before the close) predict tonight's overnight return? And the last-30-minute move? |
| 3. Gap fade | After the open, does the first 30-minute move reverse or continue the overnight gap? |
| 4. Holidays | After a long closure ($\ge 4$ calendar days) pent-up orders accumulate: is ON different? |
| 5. T+1 comparison | `EquityIndex` against all other sectors: same sign in train, validation and test? |

**Used later.** Step 13e repeats the split by volatility regime and tests the gap fade there. The sector pattern decides where the overnight strategy (Step 23c) is allowed to trade; the sign of the pent-up-pressure coefficient is the starting direction of its cross-sectional variant. The summary report (Step 26c) states whether the T+1 explanation is supported.

In [ ]:
# step 13d
# ---- 13d.1 decomposition: overnight (ON) vs intraday (ID) ----
import statsmodels.api as sm
OVN_NEED = ["ret_night", "ret_day_open", "ret_day_ex_open", "ret_day", "ret_open1", "ret_open5", "ret_open15", "ret_open30", "ret_open60",
            "ret_pre5", "ret_pre15", "ret_pre30", "vol_open15", "vol_pre30", "volume"]
OVN_OK = all(c in dp.columns for c in OVN_NEED)
if not OVN_OK:
    print("the cached daily panel has no opening-move columns: set FORCE_REBUILD = True in Step 0 and re-run Step 1 (steps 13d and 23c are skipped)")
else:
    OV = {c: wide(c).reindex(index=RAW.index, columns=RAW.columns) for c in OVN_NEED}
    SIGd = RAW.ewm(halflife=VOL_HL, min_periods=20).std().shift(1)          # ex-ante daily volatility
    ON = (OV["ret_night"].fillna(0) + OV["ret_day_open"]).where(RAW.notna()); ID = OV["ret_day_ex_open"].where(RAW.notna())
    ONz, IDz = (ON / SIGd).clip(-CLIP_Z, CLIP_Z), (ID / SIGd).clip(-CLIP_Z, CLIP_Z)
    PRE30z = (OV["ret_pre30"] / SIGd).clip(-CLIP_Z, CLIP_Z); IDx30z = ((ID - OV["ret_pre30"]) / SIGd).clip(-CLIP_Z, CLIP_Z)       # intraday move up to 30 minutes before the close
    OPX30z = ((OV["ret_open30"] - OV["ret_day_open"]) / SIGd).clip(-CLIP_Z, CLIP_Z)                                                   # first 30 bars after the opening bar
    SECN = SEC_MAP.reindex(RAW.columns)
    def sec_series(F):
        g = F.T.groupby(SECN).mean().T; out = pd.DataFrame({"ALL": F.mean(axis=1)}); return pd.concat([out, g], axis=1)
    def hac(y, X, L=5):
        d = pd.concat([y, X], axis=1).dropna()
        if len(d) < 80: return None
        return sm.OLS(d.iloc[:, 0], sm.add_constant(d.iloc[:, 1:])).fit(cov_type="HAC", cov_kwds=dict(maxlags=L))
    def mean_t(x, L=5):
        x = x.dropna()
        if len(x) < 80: return np.nan, np.nan
        r = sm.OLS(x.to_numpy(), np.ones(len(x))).fit(cov_type="HAC", cov_kwds=dict(maxlags=L)); return float(r.params[0]), float(r.tvalues[0])
    IDX = {sp: RAW.index.intersection(SPL[sp]) for sp in ("train", "val", "test")}
    S_ON, S_ID, S_ONz, S_IDz = sec_series(ON * 1e4), sec_series(ID * 1e4), sec_series(ONz), sec_series(IDz)
    rows = []
    for s in S_ON.columns:
        for sp, ix in IDX.items():
            m1, t1_ = mean_t(S_ON.loc[ix, s]); m2, t2_ = mean_t(S_ID.loc[ix, s])
            rows.append(dict(sector=s, split=sp, ON_bp=m1, ON_t=t1_, ID_bp=m2, ID_t=t2_, ON_share=m1 / (m1 + m2) if abs(m1 + m2) > 1e-9 else np.nan, n=int(S_ON.loc[ix, s].count())))
    OVN_DEC = pd.DataFrame(rows); OVN_DEC.round(3).to_csv(OUT2 / "overnight_decomposition.csv", index=False)
    print("Mean return in bp per day, overnight (ON) vs intraday (ID), Newey-West t (sector = equal-weight mean of its products):")
    print(OVN_DEC.pivot_table(index="sector", columns="split", values=["ON_bp", "ON_t", "ID_bp", "ID_t"]).reindex(columns=["ON_bp", "ON_t", "ID_bp", "ID_t"], level=0).round(2).to_string())
    pz = OVN_DEC[OVN_DEC.sector == "ALL"].set_index("split")
    OVN_PUZZLE = bool((pz.ON_bp > 0).all() and (pz.ID_bp < pz.ON_bp).all())
    print(f"\nall products: overnight mean {pz.ON_bp.round(2).to_dict()} bp vs intraday {pz.ID_bp.round(2).to_dict()} bp | overnight beats intraday in every period: {OVN_PUZZLE}")

In [ ]:
# step 13d
# ---- 13d.2 pent-up pressure, gap fade, holidays, T+1 comparison ----
if OVN_OK:
    X1 = sec_series(IDx30z.shift(1)); X2 = sec_series(PRE30z.shift(1)); Yg = sec_series(ONz)
    rows = []
    for s in Yg.columns:
        for sp, ix in IDX.items():
            m = hac(Yg.loc[ix, s].rename("ON"), pd.concat([X1.loc[ix, s].rename("ID_prev"), X2.loc[ix, s].rename("PRE30_prev")], axis=1))
            if m is not None: rows.append(dict(sector=s, split=sp, b_ID_prev=m.params["ID_prev"], t_ID_prev=m.tvalues["ID_prev"], b_PRE30_prev=m.params["PRE30_prev"], t_PRE30_prev=m.tvalues["PRE30_prev"], n=int(m.nobs)))
    OVN_PRED = pd.DataFrame(rows); OVN_PRED.round(4).to_csv(OUT2 / "overnight_pent_up.csv", index=False)
    print("Overnight return (vol units) on yesterday's intraday move (up to 30 min before the close) and on yesterday's last-30-minute move; HAC t:")
    print(OVN_PRED.pivot_table(index="sector", columns="split", values=["b_ID_prev", "t_ID_prev", "t_PRE30_prev"]).reindex(columns=["b_ID_prev", "t_ID_prev", "t_PRE30_prev"], level=0).round(2).to_string())
    # gap fade: first 30 bars after the opening bar on the overnight gap of the same morning
    Xg_ = sec_series(ONz); Yo = sec_series(OPX30z); rows = []
    for s in Xg_.columns:
        for sp, ix in IDX.items():
            m = hac(Yo.loc[ix, s].rename("open30"), Xg_.loc[ix, s].rename("gap"))
            if m is not None: rows.append(dict(sector=s, split=sp, b_gap=m.params["gap"], t_gap=m.tvalues["gap"], n=int(m.nobs)))
    OVN_OPEN = pd.DataFrame(rows); OVN_OPEN.round(4).to_csv(OUT2 / "overnight_gap_fade.csv", index=False)
    print("\nFirst 30 bars after the open on the overnight gap (negative = the gap fades, positive = it continues):")
    print(OVN_OPEN.pivot_table(index="sector", columns="split", values=["b_gap", "t_gap"]).reindex(columns=["b_gap", "t_gap"], level=0).round(2).to_string())
    # holidays: previous close at least 4 calendar days earlier
    hol = pd.Series(RAW.index.to_series().diff().dt.days.ge(4).to_numpy(), index=RAW.index)
    rows = []
    for s in S_ONz.columns:
        for sp, ix in IDX.items():
            a, b = S_ONz.loc[ix, s][hol.reindex(ix)].mean(), S_ONz.loc[ix, s][~hol.reindex(ix)].mean(); n_h = int(hol.reindex(ix).sum())
            ta = stats.ttest_ind(S_ONz.loc[ix, s][hol.reindex(ix)].dropna(), S_ONz.loc[ix, s][~hol.reindex(ix)].dropna(), equal_var=False).statistic if n_h > 5 else np.nan
            rows.append(dict(sector=s, split=sp, after_holiday=a, other_days=b, diff_t=ta, n_holiday=n_h))
    OVN_HOL = pd.DataFrame(rows); OVN_HOL.round(4).to_csv(OUT2 / "overnight_holiday.csv", index=False)
    print(f"\nOvernight return (vol units) after a closure of 4+ days vs other days, ALL products:\n", OVN_HOL[OVN_HOL.sector == "ALL"].set_index("split")[["after_holiday", "other_days", "diff_t", "n_holiday"]].round(3).to_string())
    # T+1 comparison: EquityIndex against the rest (sector-mean series of the two groups)
    EQ = [s for s in OVN_DEC.sector.unique() if s.lower().startswith("equity")]; eq_cols = [c for c in RAW.columns if SECN[c] in EQ]; ot_cols = [c for c in RAW.columns if SECN[c] not in EQ]
    T1 = []
    for nm, cols in (("EquityIndex futures (tied to T+1 stocks)", eq_cols), ("all other products (T+0 underlyings)", ot_cols)):
        if len(cols) == 0: continue
        for sp, ix in IDX.items():
            y = ONz[cols].mean(axis=1).loc[ix]; x = IDx30z[cols].shift(1).mean(axis=1).loc[ix]; mo, to_ = mean_t(y); mi, ti_ = mean_t(IDz[cols].mean(axis=1).loc[ix]); m = hac(y.rename("ON"), x.rename("ID_prev"))
            g = hac(OPX30z[cols].mean(axis=1).loc[ix].rename("o"), ONz[cols].mean(axis=1).loc[ix].rename("gap"))
            T1.append(dict(group=nm, split=sp, ON_z=mo, ON_t=to_, ID_z=mi, ID_t=ti_, b_pent_up=m.params["ID_prev"] if m is not None else np.nan, t_pent_up=m.tvalues["ID_prev"] if m is not None else np.nan,
                           b_gap_fade=g.params["gap"] if g is not None else np.nan, t_gap_fade=g.tvalues["gap"] if g is not None else np.nan))
    OVN_T1 = pd.DataFrame(T1); OVN_T1.round(4).to_csv(OUT2 / "overnight_t1_comparison.csv", index=False)
    print("\nT+1 comparison (vol units):\n", OVN_T1.round(3).to_string(index=False))
    def _sg(g, c): v = OVN_T1[OVN_T1.group.str.startswith(g)].set_index("split")[c]; return bool(len(v) == 3 and (np.sign(v) == np.sign(v.iloc[0])).all() and v.iloc[0] != 0)
    OVN_SUMMARY = dict(puzzle_all=OVN_PUZZLE, equity_pent_up_consistent=_sg("EquityIndex", "b_pent_up"), equity_gap_fade_consistent=_sg("EquityIndex", "b_gap_fade"),
                       other_pent_up_consistent=_sg("all other", "b_pent_up"),
                       equity_on_test_bp=float(OVN_DEC[(OVN_DEC.sector.isin(EQ)) & (OVN_DEC.split == "test")].ON_bp.mean()) if EQ else np.nan,
                       equity_pent_up_test_t=float(OVN_T1[(OVN_T1.group.str.startswith("EquityIndex")) & (OVN_T1.split == "test")].t_pent_up.mean()) if EQ else np.nan,
                       equity_gap_fade_test_t=float(OVN_T1[(OVN_T1.group.str.startswith("EquityIndex")) & (OVN_T1.split == "test")].t_gap_fade.mean()) if EQ else np.nan)
    print(f"\nT+1 reading: pent-up coefficient has the same sign in train, validation and test for EquityIndex: {OVN_SUMMARY['equity_pent_up_consistent']} | gap fade consistent: {OVN_SUMMARY['equity_gap_fade_consistent']} | "
          f"the same sign in the other products: {OVN_SUMMARY['other_pent_up_consistent']}")

In [ ]:
# step 13d
if OVN_OK:
    fig, axs = plt.subplots(2, 3, figsize=(18, 9))
    ax = axs[0, 0]; piv = OVN_DEC[OVN_DEC.split.isin(["train", "test"])]; secs_ = [s for s in OVN_DEC.sector.unique()]
    x_ = np.arange(len(secs_)); wd = 0.2
    for k, (sp, col, c_) in enumerate((("train", "ON_bp", SERIES[0]), ("test", "ON_bp", SERIES[2]), ("train", "ID_bp", SERIES[1]), ("test", "ID_bp", SERIES[3]))):
        v_ = piv[piv.split == sp].set_index("sector")[col].reindex(secs_); ax.bar(x_ + (k - 1.5) * wd, v_.values, wd, color=c_, label=f"{col.split('_')[0]} {sp}")
    ax.set_xticks(x_); ax.set_xticklabels(secs_, rotation=40, ha="right", fontsize=8); ax.axhline(0, color=INK2, lw=1); ax.set_ylabel("bp per day"); ax.legend(frameon=False, fontsize=7, ncol=2); ax.set_title("Overnight (ON) vs intraday (ID) mean return by sector")
    ax = axs[0, 1]
    for k, (nm, cols) in enumerate((("all products", list(RAW.columns)), ("EquityIndex", eq_cols))):
        if cols:
            ax.plot(RAW.index, ON[cols].mean(axis=1).fillna(0).cumsum() * 100, color=SERIES[k * 2], lw=1.3, label=f"{nm}: overnight"); ax.plot(RAW.index, ID[cols].mean(axis=1).fillna(0).cumsum() * 100, color=SERIES[k * 2 + 1], lw=1.3, ls="--", label=f"{nm}: intraday")
    for sp in ("val", "test"): ax.axvline(SPL[sp][0], color=INK2, ls=":", lw=1)
    ax.legend(frameon=False, fontsize=7); ax.set_ylabel("cumulative log return, %"); ax.set_title("Where is the return earned? (equal-weight, cumulative)")
    ax = axs[0, 2]; pp = OVN_PRED.pivot(index="sector", columns="split", values="b_ID_prev").reindex(columns=["train", "val", "test"]); pp.plot.barh(ax=ax, color=[MUTE, SERIES[0], SERIES[1]], width=0.8); ax.axvline(0, color=INK2, lw=1)
    ax.set_title("Overnight return on yesterday's intraday move (pent-up pressure)"); ax.legend(frameon=False, fontsize=8); ax.set_xlabel("coefficient (vol units)")
    ax = axs[1, 0]; gg = OVN_OPEN.pivot(index="sector", columns="split", values="b_gap").reindex(columns=["train", "val", "test"]); gg.plot.barh(ax=ax, color=[MUTE, SERIES[0], SERIES[1]], width=0.8); ax.axvline(0, color=INK2, lw=1)
    ax.set_title("First 30 minutes after the open on the overnight gap (< 0: gap fades)"); ax.legend(frameon=False, fontsize=8); ax.set_xlabel("coefficient")
    ax = axs[1, 1]; hh = OVN_HOL[OVN_HOL.sector == "ALL"].set_index("split").reindex(["train", "val", "test"]); x_ = np.arange(3)
    ax.bar(x_ - 0.2, hh.after_holiday, 0.4, color=SERIES[1], label="after a 4+ day closure"); ax.bar(x_ + 0.2, hh.other_days, 0.4, color=MUTE, label="other days"); ax.set_xticks(x_); ax.set_xticklabels(["train", "val", "test"]); ax.axhline(0, color=INK2, lw=1)
    ax.legend(frameon=False, fontsize=8); ax.set_ylabel("overnight return, vol units"); ax.set_title("Holidays: do pent-up orders show up at the next open?")
    ax = axs[1, 2]; t1v = OVN_T1.pivot(index="group", columns="split", values="t_pent_up").reindex(columns=["train", "val", "test"]); t1v.index = [i.split(" (")[0] for i in t1v.index]
    t1v.plot.barh(ax=ax, color=[MUTE, SERIES[0], SERIES[1]], width=0.8); ax.axvline(0, color=INK2, lw=1); ax.axvline(2, color=MUTE, ls=":", lw=1); ax.axvline(-2, color=MUTE, ls=":", lw=1)
    ax.set_title("T+1 test: pent-up coefficient t-stat, EquityIndex vs the rest"); ax.legend(frameon=False, fontsize=8); ax.set_xlabel("HAC t")
    plt.tight_layout()
    savefig(fig, "13d_overnight_t1", "Overnight returns and the T+1 effect", "13")

## Step 13e — Overnight versus day by volatility regime, and the opening reversal

**Why here.** Step 13d asked whether the return is earned overnight (ON) or intraday (ID), pooled over all days. A regime view asks **when**: the working hypothesis is that the overnight leg carries the return when volatility is high (global shocks arrive while the market is shut, and the gap is large) and the intraday leg carries it when volatility is low (smooth price discovery). The regime $S_t$ is the ex-ante state of Step 11d (LOW / MID / HIGH).

**Notation.** $z^{ON}_t=(r^{night}_t+r^{open\,bar}_t)/\sigma_t$ and $z^{ID}_t=r^{rest\,of\,day}_t/\sigma_t$, with $\sigma_t$ the ex-ante daily volatility (as in 13d). $z^{o30}_t=(r^{open\,30}_t-r^{open\,bar}_t)/\sigma_t$ is the move of bars 2–30 after the gap.

| Hypothesis | Formula | Predicted sign | Test |
|---|---|---|---|
| H1 overnight pays more in HIGH | $\bar z^{ON}_{HIGH}-\bar z^{ON}_{LOW}$ | $>0$ | date-clustered $t$, same sign in train, validation, test |
| H2 intraday pays more in LOW | $\bar z^{ID}_{HIGH}-\bar z^{ID}_{LOW}$ | $<0$ | same |
| H3 night share of variance rises | $\;\overline{RV^{N\,2}/(RV^{N\,2}+RV^{D\,2})}_{HIGH}-(\ldots)_{LOW}$ | $>0$ | same (needs Step 1 night / day variance) |
| H4 the gap fades in HIGH | $z^{o30}_t=a_S+b_S\,z^{ON}_t+\varepsilon_t$, $b_{HIGH}$ vs $b_{LOW}$ | $b_{HIGH}<0$ and $<b_{LOW}$ | clustered $t$ on $b_{HIGH}-b_{LOW}$ |
| H5 the first 30 minutes carry the day's variance peak | $\;\rho=\dfrac{\overline{RV_{2..30}^2}/29}{\overline{RV^2_{rest}}/(n_{day}-30)}$ | $\rho>1$, larger in HIGH | level by regime |

**Execution reality (no order book here).** Depth is proxied by the first 15 minutes' volume relative to the trailing 20-day mean volume, by regime; half-spreads come from Step 23a by time of day. Quote depth, queue position and slippage on gaps cannot be measured with 1-minute bars, so the regime view of liquidity is a **proxy**, not a measurement.

**Used later.** If H4 holds out of sample in the HIGH regime, the gap reversal is traded in Step 23d, gated by the same regime.

In [ ]:
# step 13e
# ---- 13e.1 overnight (ON) versus intraday (ID) by volatility regime; variance shares; depth proxy ----
if OVN_OK:
    def stk(F, nm):
        s_ = F.reindex(index=RAW.index, columns=UNIV).stack(); s_.index.names = ["date", "product"]; return s_.rename(nm)
    cols_ = {"on": ONz, "id": IDz, "op30": OPX30z, "reg": VREG, "dep": (OV["vol_open15"] / VOLU.rolling(20, min_periods=10).mean().shift(1)).clip(0, 5)}
    OVSQ_OK = RVN_OK and all(c in dp.columns for c in ("rv_open30", "n_day_bars"))
    if RVN_OK:
        _rnn, _rdd = (wide(c).reindex(index=RAW.index, columns=UNIV) for c in ("rv_night", "rv_day")); cols_["nvs"] = _rnn ** 2 / (_rnn ** 2 + _rdd ** 2)
    if OVSQ_OK:
        _ow = wide("rv_open30").reindex(index=RAW.index, columns=UNIV); _nb = wide("n_day_bars").reindex(index=RAW.index, columns=UNIV)
        cols_["ow2"] = _ow ** 2; cols_["nbm"] = _nb - 30; cols_["rest2"] = (_rdd ** 2 - _ow ** 2 - OV["ret_day_open"].reindex(columns=UNIV) ** 2).clip(lower=0)
    LONG = pd.concat([stk(F, k) for k, F in cols_.items()], axis=1).dropna(subset=["on", "id", "reg"])
    LONG["split"] = np.select([LONG.index.get_level_values("date").isin(IDX[sp]) for sp in ("train", "val", "test")], ["train", "val", "test"], "")
    LONG = LONG[LONG.split != ""]
    grp_ = lambda d: pd.factorize(d.index.get_level_values("date"))[0]
    def clmean(y, g):
        if len(y) < 80: return np.nan, np.nan
        r_ = sm.OLS(np.asarray(y, float), np.ones(len(y))).fit(cov_type="cluster", cov_kwds=dict(groups=g)); return float(r_.params[0]), float(r_.tvalues[0])
    rows = []
    for sp in ("train", "val", "test"):
        for k, rn in REGN.items():
            d_ = LONG[(LONG.split == sp) & (LONG.reg == k)]
            if len(d_) < 80: continue
            g_ = grp_(d_); mo, to_ = clmean(d_.on, g_); mi, ti_ = clmean(d_["id"], g_)
            r = dict(split=sp, regime=rn, n=len(d_), ON_z=mo, ON_t=to_, ID_z=mi, ID_t=ti_, carrier="ON" if mo > mi else "ID", on_var_share=float((d_.on ** 2).mean() / ((d_.on ** 2).mean() + (d_["id"] ** 2).mean())), open_depth_median=float(d_.dep.median()))
            if "nvs" in d_: r["night_var_share"] = float(d_.nvs.mean())
            if OVSQ_OK:
                dd = d_.dropna(subset=["ow2", "rest2", "nbm"]); dd = dd[dd.nbm > 0]
                r["open_intensity"] = float((dd.ow2.mean() / 29) / (dd.rest2.mean() / dd.nbm.mean())) if len(dd) > 50 and dd.rest2.mean() > 0 else np.nan
            rows.append(r)
    REG_ONID = pd.DataFrame(rows); REG_ONID.round(4).to_csv(OUT2 / "regime_on_id.csv", index=False)
    print("Mean overnight (ON) and intraday (ID) return in vol units by regime, clustered t; ON variance share; night variance share; opening depth proxy; open-window variance intensity:")
    print(REG_ONID.set_index(["split", "regime"]).round(3).to_string())
    def hl_diff(sp, col, fn=None):
        d_ = LONG[(LONG.split == sp) & (LONG.reg != 0)].copy()
        if col not in d_ or d_[col].notna().sum() < 200: return np.nan, np.nan
        d_ = d_.dropna(subset=[col]); X = sm.add_constant((d_.reg == 1).astype(float).to_numpy())
        r_ = sm.OLS(d_[col].to_numpy(float), X).fit(cov_type="cluster", cov_kwds=dict(groups=grp_(d_))); return float(r_.params[1]), float(r_.tvalues[1])
    HYP = []
    for tag, col, sign, txt in (("H1", "on", 1, "overnight return larger in HIGH than LOW"), ("H2", "id", -1, "intraday return smaller in HIGH than LOW"), ("H3", "nvs", 1, "night share of variance larger in HIGH than LOW")):
        r = dict(hypothesis=tag, statement=txt, predicted_sign=sign)
        for sp in ("train", "val", "test"): r[f"diff_{sp}"], r[f"t_{sp}"] = hl_diff(sp, col)
        d3 = np.array([r["diff_train"], r["diff_val"], r["diff_test"]]); r["same_sign_all_periods"] = bool(np.isfinite(d3).all() and (np.sign(d3) == sign).all())
        r["supported"] = bool(r["same_sign_all_periods"] and abs(r["t_test"]) > 2); HYP.append(r)
    REG_HYP = pd.DataFrame(HYP)

In [ ]:
# step 13e
# ---- 13e.2 the opening reversal by regime: does the gap fade, and where? ----
if OVN_OK:
    rows = []
    for sp in ("train", "val", "test"):
        d_ = LONG[LONG.split == sp].dropna(subset=["op30"])
        if len(d_) < 300: continue
        X = pd.DataFrame({"g_LOW": d_.on * (d_.reg == -1), "g_MID": d_.on * (d_.reg == 0), "g_HIGH": d_.on * (d_.reg == 1), "d_LOW": (d_.reg == -1) * 1.0, "d_HIGH": (d_.reg == 1) * 1.0}).astype(float)
        r_ = sm.OLS(d_.op30.to_numpy(float), sm.add_constant(X)).fit(cov_type="cluster", cov_kwds=dict(groups=grp_(d_)))
        dif = r_.t_test("g_HIGH - g_LOW = 0")
        rows.append(dict(split=sp, n=len(d_), **{f"b_{n}": float(r_.params[f"g_{n}"]) for n in ("LOW", "MID", "HIGH")}, **{f"t_{n}": float(r_.tvalues[f"g_{n}"]) for n in ("LOW", "MID", "HIGH")},
                         b_HIGH_minus_LOW=float(np.squeeze(dif.effect)), t_HIGH_minus_LOW=float(np.squeeze(dif.tvalue))))
    REG_FADE = pd.DataFrame(rows); REG_FADE.round(4).to_csv(OUT2 / "regime_gap_fade.csv", index=False); REG_HYP.round(4).to_csv(OUT2 / "regime_hypotheses.csv", index=False)
    print("Bars 2-30 after the open on the overnight gap, by regime (negative b = the gap fades):\n", REG_FADE.set_index("split").round(3).to_string())
    f3 = REG_FADE.set_index("split").reindex(["train", "val", "test"])
    h4 = bool(f3.b_HIGH.notna().all() and (f3.b_HIGH < 0).all() and (f3.b_HIGH_minus_LOW < 0).all() and f3.loc["test", "t_HIGH_minus_LOW"] < -2) if len(f3) == 3 else False
    REG_HYP = pd.concat([REG_HYP, pd.DataFrame([dict(hypothesis="H4", statement="gap fades in HIGH, more than in LOW (bars 2-30)", predicted_sign=-1, diff_train=f3.get("b_HIGH_minus_LOW", pd.Series()).get("train", np.nan),
                  diff_val=f3.get("b_HIGH_minus_LOW", pd.Series()).get("val", np.nan), diff_test=f3.get("b_HIGH_minus_LOW", pd.Series()).get("test", np.nan), t_train=f3.get("t_HIGH_minus_LOW", pd.Series()).get("train", np.nan),
                  t_val=f3.get("t_HIGH_minus_LOW", pd.Series()).get("val", np.nan), t_test=f3.get("t_HIGH_minus_LOW", pd.Series()).get("test", np.nan), same_sign_all_periods=h4, supported=h4)])], ignore_index=True)
    REG_HYP.round(4).to_csv(OUT2 / "regime_hypotheses.csv", index=False)
    print("\nHypotheses (difference HIGH minus LOW, clustered t):\n", REG_HYP[["hypothesis", "statement", "diff_train", "diff_val", "diff_test", "t_test", "same_sign_all_periods", "supported"]].round(3).to_string(index=False))
    h5 = REG_ONID.set_index(["split", "regime"])["open_intensity"] if "open_intensity" in REG_ONID else None
    car = REG_ONID[REG_ONID.split == "test"].set_index("regime").carrier.to_dict()
    sup = REG_HYP.set_index("hypothesis").supported.to_dict()
    _rd = (f"By regime on test, the larger mean leg is: {', '.join(k + ' ' + str(car.get(k, 'n/a')) for k in ('LOW', 'MID', 'HIGH'))}. H1 overnight pays more in HIGH: {sup.get('H1')}; H2 intraday pays less in HIGH: {sup.get('H2')}; H3 night variance share rises in HIGH: {sup.get('H3', 'n/a')}; "
           f"H4 the gap fades in HIGH more than in LOW: {sup.get('H4')}. 'Supported' needs the predicted sign in train, validation and test and |t| > 2 on test."
           + (f" First-30-minute variance per bar is {float(h5.xs('test').mean()):.1f}x the rest of the day on average (test)." if h5 is not None and h5.notna().any() else ""))
    print("\n" + _rd)
    REGON_SUMMARY = dict(hypotheses={r.hypothesis: dict(supported=bool(r.supported), t_test=float(r.t_test) if np.isfinite(r.t_test) else None) for r in REG_HYP.itertuples()}, carrier_test=car, reading=_rd,
                         b_gap_HIGH_test=float(f3.loc["test", "b_HIGH"]) if "test" in f3.index else None, b_gap_LOW_test=float(f3.loc["test", "b_LOW"]) if "test" in f3.index else None)

In [ ]:
# step 13e
if OVN_OK and len(REG_ONID):
    fig, axs = plt.subplots(2, 3, figsize=(18, 9)); regs_ = ["LOW", "MID", "HIGH"]; spl3 = ["train", "val", "test"]; cs3 = [MUTE, SERIES[0], SERIES[1]]
    for ax, col, ttl in ((axs[0, 0], "ON_z", "Overnight return by regime (vol units)"), (axs[0, 1], "ID_z", "Intraday return by regime (vol units)")):
        for j, (sp, c_) in enumerate(zip(spl3, cs3)):
            v_ = REG_ONID[REG_ONID.split == sp].set_index("regime")[col].reindex(regs_); ax.bar(np.arange(3) + (j - 1) * 0.27, v_.values, 0.27, color=c_, label=sp)
        ax.axhline(0, color=INK2, lw=1); ax.set_xticks(range(3)); ax.set_xticklabels(regs_); ax.legend(frameon=False, fontsize=8); ax.set_title(ttl)
    ax = axs[0, 2]; colv = "night_var_share" if "night_var_share" in REG_ONID else "on_var_share"
    for j, (sp, c_) in enumerate(zip(spl3, cs3)):
        v_ = REG_ONID[REG_ONID.split == sp].set_index("regime")[colv].reindex(regs_); ax.bar(np.arange(3) + (j - 1) * 0.27, v_.values, 0.27, color=c_, label=sp)
    ax.set_xticks(range(3)); ax.set_xticklabels(regs_); ax.legend(frameon=False, fontsize=8); ax.set_title("Night-session share of daily variance" if colv == "night_var_share" else "Overnight share of squared moves (night variance not available)")
    ax = axs[1, 0]
    for j, (sp, c_) in enumerate(zip(spl3, cs3)):
        v_ = REG_FADE.set_index("split").reindex([sp])[["b_LOW", "b_MID", "b_HIGH"]].to_numpy().ravel(); ax.bar(np.arange(3) + (j - 1) * 0.27, v_, 0.27, color=c_, label=sp)
    ax.axhline(0, color=INK2, lw=1); ax.set_xticks(range(3)); ax.set_xticklabels(regs_); ax.legend(frameon=False, fontsize=8); ax.set_title("Bars 2-30 on the overnight gap by regime (< 0: the gap fades)")
    ax = axs[1, 1]
    if "open_intensity" in REG_ONID:
        for j, (sp, c_) in enumerate(zip(spl3, cs3)):
            v_ = REG_ONID[REG_ONID.split == sp].set_index("regime")["open_intensity"].reindex(regs_); ax.bar(np.arange(3) + (j - 1) * 0.27, v_.values, 0.27, color=c_, label=sp)
        ax.axhline(1, color=INK2, ls=":", lw=1); ax.legend(frameon=False, fontsize=8); ax.set_title("Variance per bar, first 30 minutes vs rest of day")
    else:
        ax.text(0.5, 0.5, "open-window variance not in the cache", ha="center", va="center", transform=ax.transAxes); ax.set_title("Opening-window intensity")
    ax.set_xticks(range(3)); ax.set_xticklabels(regs_)
    ax = axs[1, 2]; hy = REG_HYP.set_index("hypothesis")[["t_train", "t_val", "t_test"]].astype(float); hy.plot.barh(ax=ax, color=cs3, width=0.8); ax.axvline(0, color=INK2, lw=1); ax.axvline(2, color=MUTE, ls=":", lw=1); ax.axvline(-2, color=MUTE, ls=":", lw=1)
    ax.set_xlabel("clustered t of HIGH minus LOW"); ax.legend(frameon=False, fontsize=8); ax.set_title("Hypotheses H1-H4: sign and strength")
    plt.tight_layout(); savefig(fig, "13e_regime_overnight", "Overnight versus day returns by volatility regime and the opening reversal", "13")

---
# Part C — Relationships between products (Steps 14–17)
| Step | Question |
|---|---|
| 14 | **Multi-time-frame Granger** (5m, 15m, 30m, 60m, 1D): who leads whom, at which frequency, with a blind test at each one; **how long a relationship survives**, both across lags (minutes / days ahead) and through time (rolling windows) |
| 15 | Level relationships: **cointegration** (within and across sectors, plus economic spreads), correlation stability, **tail dependence**, **sector → product** lead |
| 16 | **Fundamental-linked relationships**: economic chains (processing margins, product spreads, input costs) tested for cointegration, mean-reversion of the spread, and whether drivers lead |
| 17 | **Relationship library**: every relationship scored for strength, out-of-sample confirmation, stability, survival and tradability, then ranked, plus the **current opportunities** it implies (bull / bear) |

## Step 13f — Regimes first: regime atlas, regime-balanced train/validation, overnight by sector, positive versus negative overnight days

**Why this step exists.** The earlier steps split the calendar once (train, then validation, then test) and looked for one rule over all of it. If pattern-recognition rules differ by regime, a chronological split gives a validation period with only the regimes that happened to occur then (see the coverage table), and a rule that works in one regime is averaged away or judged in a regime it never meant to cover. This step looks at the regimes first, then allocates them to train and validation, then studies the overnight return inside them.

**1. Regime atlas.** Three dimensions, each defined ex ante: a day's regime is the tercile of *yesterday's* value of the statistic among *all earlier* values (expanding window, so no later data defines it; truncation-tested).

| Dimension | Statistic | Labels |
|---|---|---|
| Volatility | 20-day standard deviation of the equal-weight market return | low / mid / high |
| Trend | 120-day sum of the market return | bear / range / bull |
| Co-movement | share of the 60-day variance that is common to all products | dispersed / mixed / co-moving |

The nine joint volatility × trend cells are the allocation unit. The tables show how many days, how long each regime lasts, how the market behaves in it and how the overnight (ON) and intraday (ID) returns look in it. Night-session start dates by product are printed too: before a night session exists, "overnight" is only a close-to-open gap.

**2. Regime-balanced train / validation (`SPL_R`).** The pre-test period is cut into 60-day blocks; each block is labelled with its dominant joint regime; inside each regime every third block (in time order) is validation, the rest training. Every regime with at least two blocks is therefore in both sets, and early and late history are mixed. 20 training days (`EMB`) on each side of every validation block are purged because features and labels overlap. **The test stays chronological and is read once.** Caveats: validation blocks sit between training blocks, so validation cannot show drift forward in time (only the test can); a regime missing or thin in the test is flagged in the coverage table and its rules cannot be confirmed out of sample. The global `SPL` of the other steps is unchanged; this split is used here and is available as `SPL_R` for re-running other steps by regime.

**3. Overnight by sector.** `EquityIndex` is studied alone and every other sector alone (and "OTHERS" = their equal-weight mean). For each: ON and ID in bp/day with Newey–West *t*; the night-session and opening-bar parts of ON; ON on days with versus without a night session; ON and ID in each of the nine regime cells and each single dimension. Cumulative ON versus ID by sector is plotted (the picture in the request). *Supported* = same sign in train_R, val_R and test and |t| above the Bonferroni level for all cells tested. Is `EquityIndex` different from the rest? Tested as a daily difference with the same rule.

**4. Positive versus negative overnight days.** Frequency (binomial *z* against one half), average size of positive and negative days, skew, share of the gross overnight move from positive days; the anatomy of each kind of day (night session vs opening bar, what the intraday does next, whether the next overnight has the same sign); then *why* and *when*:
- **Why (drivers):** regression of the overnight return (vol units) on features known before the night session (yesterday's intraday, overnight and last-30-minute moves, close location in yesterday's range, volume surprise, open-interest change, 5-day return, long-closure, Monday/Friday, month start/end, and the three regime levels). Coefficients in train_R, val_R and test; *supported* needs same sign, |t| > Bonferroni on test and |t| > 2 on validation. A logistic model on train_R gives the out-of-sample AUC of the sign (0.5 = no skill).
- **When (conditions):** lift in P(overnight > 0) over the period's base rate, by weekday, closure, month position, regime and quintile of yesterday's intraday return (quintile edges from the train/validation pool); supported = same sign in all periods and |z| above the Bonferroni level on test.

**Reading rules.** Sector means are equal-weight; regime-cell t-statistics use a Newey–West lag on days that are not always consecutive, so they are approximate; the test period's positive drift is not removed here (the overnight mean itself is the object); nothing contemporaneous is used as a predictor. All thresholds are tests, not assumptions: the 60-day block, the tercile cuts and the 20/60/120-day windows are the framework and are meant to be varied.

**Used later.** `RG` (the regime labels), `SPL_R` / `RG_IDX` (the regime-balanced split) feed Steps 13g, 18l and 18m; the overnight tables by sector and the positive-versus-negative-day results feed the overnight strategies (Steps 23c and 23d) and the board.

In [ ]:
# step 13f
RG_OK = bool(OVN_OK) and all(k in globals() for k in ('ON','ID','ONz','IDz','SIGd','SECN','OV','mean_t','IDX'))
if not RG_OK:
    print('Step 13f needs Step 13d (overnight decomposition): run it first; skipped')
else:
    # ---- 13f.1 regime atlas: every regime is defined ex-ante, then looked at, then allocated to train and validation ----
    RG_MINHIST, RG_BLK = 250, 60
    RG_VN, RG_TN, RG_CN = ("low", "mid", "high"), ("bear", "range", "bull"), ("dispersed", "mixed", "co-moving")
    def rg_label(x, minp=RG_MINHIST):
        # regime 0 / 1 / 2 = tercile of yesterday's value among ALL EARLIER values (expanding, shifted one day): no later data defines today's regime
        def f(a):
            if np.isnan(a[-1]): return np.nan
            b = a[:-1]; b = b[~np.isnan(b)]
            return float((b < a[-1]).mean()) if len(b) >= 30 else np.nan
        r = x.shift(1).expanding(minp).apply(f, raw=True)
        return pd.Series(np.where(r.isna(), np.nan, np.digitize(r.fillna(0), [1 / 3, 2 / 3])), index=x.index)
    def rg_all(R):
        M = R.mean(axis=1); cm = M.rolling(60, min_periods=40).var() / R.rolling(60, min_periods=40).var().mean(axis=1)
        out = pd.DataFrame(dict(V=rg_label(M.rolling(20, min_periods=15).std()), T=rg_label(M.rolling(120, min_periods=100).sum()), C=rg_label(cm))); out["S"] = out["V"] * 3 + out["T"]; return out
    _t = pd.Series(np.r_[np.zeros(300), np.arange(300) * 0.01]); _l = rg_label(_t); assert _l.iloc[-1] == 2 and np.isnan(_l.iloc[10]), "regime label broken"
    print("unit check passed: the label is the rank among earlier values only")
    causal_test("regime_labels", lambda R: rg_all(R), RAW.iloc[-1500:], step="13f")
    RG = rg_all(RAW); RG_SN = {s: f"{RG_VN[s // 3]}-vol / {RG_TN[s % 3]}" for s in range(9)}; MKT = RAW.mean(axis=1)
    RG_IX = RG.index[RG["S"].notna()]
    print(f"regimes defined for {len(RG_IX)} of {len(RG)} days (the first {RG_MINHIST} days have no history); volatility = 20-day market volatility, trend = 120-day market return, co-movement = share of variance that is common")
    # night-session availability per product (a structural regime: before it exists, 'overnight' is a close-to-open gap)
    NSF = (OV["ret_night"].notna().astype(float).rolling(60, min_periods=30).mean() >= 0.5).reindex(columns=RAW.columns)
    NS_START = NSF.idxmax().where(NSF.any()); print("night session first seen (products by year):", pd.Series(NS_START.dropna().dt.year).value_counts().sort_index().to_dict(), "| never:", int(NSF.any().eq(False).sum()))
    # ---- the regime calendar, regime table (what each regime looks like) ----
    def rg_spell(s):
        d = s.dropna(); g = (d != d.shift()).cumsum(); return d.groupby(g).size()
    RG_TAB = []
    for nm, col, names in (("volatility", "V", RG_VN), ("trend", "T", RG_TN), ("co-movement", "C", RG_CN)):
        for k, lab in enumerate(names):
            m = RG[col] == k; lv = RG[col].dropna(); gg = (lv != lv.shift()).cumsum(); spells = lv.groupby(gg).agg(["first", "size"]); sp_k = spells[spells["first"] == k]["size"]
            RG_TAB.append(dict(dimension=nm, regime=lab, days=int(m.sum()), share=float(m.mean()), mean_spell_days=float(sp_k.mean()) if len(sp_k) else np.nan, market_ann_vol=float(MKT[m].std() * np.sqrt(252)), market_mean_bp=float(MKT[m].mean() * 1e4),
                               ON_mean_bp=float(ON.mean(axis=1)[m].mean() * 1e4), ID_mean_bp=float(ID.mean(axis=1)[m].mean() * 1e4)))
    RG_TAB = pd.DataFrame(RG_TAB); RG_TAB.round(3).to_csv(OUT2 / "regime_table.csv", index=False)
    print("What each regime looks like (equal-weight market; ON = overnight, ID = intraday; bp per day):"); print(RG_TAB.round(2).to_string(index=False))
    RG_TR = pd.crosstab(RG["V"].shift(1), RG["V"], normalize="index"); print("\nVolatility regime: probability of tomorrow's regime given today's (rows today):"); print(RG_TR.round(2).to_string())
    RG_YEAR = pd.crosstab(RG.index.year, RG["S"].map(RG_SN)); RG_YEAR.to_csv(OUT2 / "regime_by_year.csv")
    # ---- regime-balanced allocation of the pre-test period to train and validation ----
    # Blocks of RG_BLK days are labelled by their dominant joint regime; inside each regime every third block (in time order) is validation, so every regime that has
    # at least two blocks appears in both sets and early and late history are mixed. EMB training days on each side of every validation block are dropped (purge: features and labels overlap by up to EMB days).
    def rg_allocate(S, last, blk=RG_BLK, emb=EMB):
        pre = S.index[S.index <= last]; blocks = [pre[i:i + blk] for i in range(0, len(pre), blk)]
        lab = [S.loc[b].dropna().mode().iloc[0] if S.loc[b].notna().mean() >= 0.5 else np.nan for b in blocks]; assign = np.array(["train"] * len(blocks), dtype=object)
        for st in pd.Series(lab).dropna().unique():
            ids = [i for i, l in enumerate(lab) if l == st]
            for j, i in enumerate(ids):
                if len(ids) >= 2 and j % 3 == 1: assign[i] = "val"
        keep = {b[0]: True for b in blocks}; purged = []
        for i in range(len(blocks)):
            if assign[i] == "val":
                if i > 0 and assign[i - 1] == "train": purged += list(blocks[i - 1][-emb:])
                if i + 1 < len(blocks) and assign[i + 1] == "train": purged += list(blocks[i + 1][:emb])
        pur = pd.DatetimeIndex(sorted(set(purged)))
        sets = {k: pd.DatetimeIndex([d for i, b in enumerate(blocks) if assign[i] == k for d in b]).difference(pur) for k in ("train", "val")}
        B = pd.DataFrame(dict(start=[b[0] for b in blocks], end=[b[-1] for b in blocks], regime=[RG_SN.get(l, "unlabelled") if pd.notna(l) else "unlabelled" for l in lab], assigned=assign)); return sets, pur, B
    SPL_R, RG_PURGED, RG_BLOCKS = rg_allocate(RG["S"], SPL["val"][-1]); SPL_R["test"] = SPL["test"]; RG_BLOCKS.to_csv(OUT2 / "regime_split_blocks.csv", index=False)
    RG_IDX = {k: RAW.index.intersection(v) for k, v in SPL_R.items()}; RG_POOL = RG_IDX["train"].union(RG_IDX["val"])
    cov = []
    for s in range(9):
        r = dict(regime=RG_SN[s])
        for k, ix in (("train_R", RG_IDX["train"]), ("val_R", RG_IDX["val"]), ("test", RG_IDX["test"]), ("train_chrono", IDX["train"]), ("val_chrono", IDX["val"])): r[k] = int((RG["S"].reindex(ix) == s).sum())
        r["note"] = ("absent from the test period: rules for it cannot be checked out of sample" if r["test"] == 0 else ("thin in the test (<30 days): read its test result with caution" if r["test"] < 30 else "")) + ("; absent from chronological validation" if r["val_chrono"] == 0 else "") + ("; not in validation_R (single block)" if r["val_R"] == 0 and r["train_R"] > 0 else ""); cov.append(r)
    RG_COV = pd.DataFrame(cov); RG_COV.to_csv(OUT2 / "regime_coverage.csv", index=False)
    print(f"\nBlocks of {RG_BLK} days: train {int((RG_BLOCKS.assigned == 'train').sum())}, validation {int((RG_BLOCKS.assigned == 'val').sum())}; purged {len(RG_PURGED)} training days next to validation blocks; the test period stays chronological from {SPL['test'][0].date()}")
    print("Days of each joint regime in the regime-balanced sets against the chronological ones:"); print(RG_COV.to_string(index=False))
    print("Caveat: validation blocks sit between training blocks, so validation cannot reveal drift forward in time; only the chronological test can. Every decision below is made on train_R / val_R and read once on the test.")

In [ ]:
# step 13f
if RG_OK:
    # ---- 13f.2 overnight by sector and regime: EquityIndex separately from every other sector ----
    from scipy import stats as _rgst
    OR_SP = {"train_R": RG_IDX["train"], "val_R": RG_IDX["val"], "test": RG_IDX["test"]}
    def rg_nw(x, L=5, minn=30):
        x = pd.Series(x).dropna().to_numpy(float); n = len(x)
        if n < minn: return np.nan, np.nan
        m = x.mean(); e = x - m; v = e @ e / n
        for l in range(1, min(L, n - 1) + 1): v += 2 * (1 - l / (L + 1)) * (e[l:] @ e[:-l]) / n
        return float(m), float(m / np.sqrt(max(v, 1e-18) / n))
    _rgx = pd.Series(np.r_[np.ones(50), -np.ones(50)] * 0.0 + np.random.default_rng(0).normal(0.5, 1, 100)); assert abs(rg_nw(_rgx)[0] - _rgx.mean()) < 1e-12 and np.isnan(rg_nw(_rgx.iloc[:10])[0]); print("unit check passed: Newey-West mean and t")
    OR_SECS = ["EquityIndex"] + sorted([s for s in SECN.dropna().unique() if s != "EquityIndex"])
    def rg_sec(F):
        g = F.T.groupby(SECN).mean().T; return g.reindex(columns=OR_SECS)
    O_ON, O_ID = rg_sec(ON * 1e4), rg_sec(ID * 1e4)
    O_NI, O_OP = rg_sec(OV["ret_night"] * 1e4), rg_sec(OV["ret_day_open"] * 1e4)
    O_ONn, O_IDn = rg_sec((ON * 1e4).where(NSF)), rg_sec((ID * 1e4).where(NSF)); O_ONg, O_IDg = rg_sec((ON * 1e4).where(~NSF)), rg_sec((ID * 1e4).where(~NSF))
    OT_OTH = [s for s in OR_SECS if s != "EquityIndex"]; O_ON["OTHERS"] = O_ON[OT_OTH].mean(axis=1); O_ID["OTHERS"] = O_ID[OT_OTH].mean(axis=1)
    print("Sectors:", {s: int((SECN == s).sum()) for s in OR_SECS})
    # table 1: the decomposition by split, with and without a night session
    rows = []
    for s in OR_SECS + ["OTHERS"]:
        for sp, ix in OR_SP.items():
            r = dict(sector=s, split=sp, n=int(O_ON.loc[ix, s].count()))
            r["ON_bp"], r["ON_t"] = rg_nw(O_ON.loc[ix, s]); r["ID_bp"], r["ID_t"] = rg_nw(O_ID.loc[ix, s])
            if s != "OTHERS":
                r["night_bp"], _ = rg_nw(O_NI.loc[ix, s]); r["openbar_bp"], _ = rg_nw(O_OP.loc[ix, s])
                r["ON_withnight_bp"], r["ON_withnight_t"] = rg_nw(O_ONn.loc[ix, s]); r["ON_gap_only_bp"], r["ON_gap_only_t"] = rg_nw(O_ONg.loc[ix, s])
            rows.append(r)
    OR_TAB = pd.DataFrame(rows); OR_TAB.round(3).to_csv(OUT2 / "overnight_by_sector_regime_split.csv", index=False)
    print("\nOvernight (ON) and intraday (ID) mean return, bp per day, Newey-West t, regime-balanced train/validation (_R) and chronological test:")
    print(OR_TAB[["sector", "split", "n", "ON_bp", "ON_t", "ID_bp", "ID_t", "night_bp", "openbar_bp"]].round(2).to_string(index=False))
    print("\nSame, split by whether the product had a night session that day (night-session days: ON = night session + opening bar; others: ON = close-to-open gap):")
    print(OR_TAB[OR_TAB.sector != "OTHERS"][["sector", "split", "ON_withnight_bp", "ON_withnight_t", "ON_gap_only_bp", "ON_gap_only_t"]].round(2).to_string(index=False))
    # table 2: by regime. Every cell is one test; supported = same sign in train_R, val_R and test and |t_test| above the Bonferroni level
    LEGS = dict(ON=O_ON, ID=O_ID); rows = []
    for leg, ORFm in LEGS.items():
        for s in OR_SECS + ["OTHERS"]:
            for dim, names in (("V", RG_VN), ("T", RG_TN), ("C", RG_CN)):
                for k, nm in enumerate(names):
                    r = dict(leg=leg, sector=s, dimension=dim, regime=nm)
                    for sp, ix in OR_SP.items():
                        sel = ix[(RG[dim].reindex(ix) == k).to_numpy()]; r[sp + "_n"] = len(sel); r[sp + "_bp"], r[sp + "_t"] = rg_nw(ORFm.loc[sel, s], minn=25)
                    rows.append(r)
    OR_REG = pd.DataFrame(rows); NT_OR = int(OR_REG[["train_R_bp", "val_R_bp", "test_bp"]].notna().all(axis=1).sum()); OR_TB = bonferroni_t(max(NT_OR, 1))
    OR_REG["same_sign"] = (np.sign(OR_REG.train_R_bp) == np.sign(OR_REG.val_R_bp)) & (np.sign(OR_REG.val_R_bp) == np.sign(OR_REG.test_bp)) & OR_REG.test_bp.notna()
    OR_REG["supported"] = OR_REG.same_sign & (OR_REG.test_t.abs() > OR_TB); OR_REG.round(3).to_csv(OUT2 / "overnight_by_regime_cells.csv", index=False)
    print(f"\nRegime cells tested: {NT_OR}; Bonferroni |t| > {OR_TB:.2f} on test; supported cells: {int(OR_REG.supported.sum())}")
    if OR_REG.supported.any(): print(OR_REG[OR_REG.supported][["leg", "sector", "dimension", "regime", "train_R_bp", "val_R_bp", "test_bp", "test_t"]].round(2).sort_values("test_t", key=abs, ascending=False).head(25).to_string(index=False))
    # table 3: is EquityIndex different from the rest? difference of daily means, same-sign rule
    rows = []
    for leg, ORFm in LEGS.items():
        ORdff = ORFm["EquityIndex"] - ORFm["OTHERS"]; r = dict(leg=leg)
        for sp, ix in OR_SP.items(): r[sp + "_bp"], r[sp + "_t"] = rg_nw(ORdff.loc[ix])
        r["EquityIndex_differs"] = bool(np.sign(r["train_R_bp"]) == np.sign(r["val_R_bp"]) == np.sign(r["test_bp"]) and abs(r["test_t"]) > 2); rows.append(r)
    OR_DIFF = pd.DataFrame(rows); OR_DIFF.round(3).to_csv(OUT2 / "overnight_equity_vs_others.csv", index=False)
    print("\nEquityIndex minus the equal-weight mean of all other sectors (bp per day):"); print(OR_DIFF.round(2).to_string(index=False))
    # figures: cumulative ON versus ID by sector (the picture in the request) and the regime heatmap
    _rgn = len(OR_SECS); _rgc = 3; _rgr = int(np.ceil(_rgn / _rgc)); fig, axs = plt.subplots(_rgr, _rgc, figsize=(15, 3.2 * _rgr), squeeze=False)
    for a, s in zip(axs.ravel(), OR_SECS):
        a.plot(O_ON[s].fillna(0).cumsum() / 100, label="overnight", color="#1b6ca8"); a.plot(O_ID[s].fillna(0).cumsum() / 100, label="intraday", color="#c0392b")
        for d in (SPL["val"][0], SPL["test"][0]): a.axvline(d, color="grey", ls=":", lw=1)
        a.set_title(s + f" ({int((SECN == s).sum())} products)", fontsize=10); a.set_ylabel("cumulative %"); a.grid(alpha=.3)
    for a in axs.ravel()[_rgn:]: a.axis("off")
    axs.ravel()[0].legend(); fig.suptitle("Cumulative overnight and intraday return by sector (sum of daily equal-weight returns; dotted = validation and test starts)"); fig.tight_layout(); fig.savefig(OUT2 / "13f_cumulative_by_sector.png", dpi=110); plt.show()
    _rgh = OR_REG[(OR_REG.leg == "ON") & (OR_REG.sector != "OTHERS")].assign(cell=lambda d: d.dimension + ":" + d.regime).pivot(index="sector", columns="cell", values="test_bp")
    _rgs = OR_REG[(OR_REG.leg == "ON") & (OR_REG.sector != "OTHERS")].assign(cell=lambda d: d.dimension + ":" + d.regime).pivot(index="sector", columns="cell", values="supported").reindex(index=_rgh.index, columns=_rgh.columns)
    fig, a = plt.subplots(figsize=(13, 0.5 * len(_rgh) + 2)); im = a.imshow(_rgh.values.astype(float), cmap="RdBu", vmin=-np.nanmax(np.abs(_rgh.values)), vmax=np.nanmax(np.abs(_rgh.values)), aspect="auto")
    a.set_xticks(range(_rgh.shape[1])); a.set_xticklabels(_rgh.columns, rotation=60, ha="right"); a.set_yticks(range(_rgh.shape[0])); a.set_yticklabels(_rgh.index)
    for i in range(_rgh.shape[0]):
        for j in range(_rgh.shape[1]):
            v = _rgh.values[i, j]
            if np.isfinite(v): a.text(j, i, f"{v:.0f}" + ("*" if _rgs.values[i, j] == True else ""), ha="center", va="center", fontsize=8)
    fig.colorbar(im, label="mean overnight bp (test)"); a.set_title("Overnight return by sector and regime on the test period (* = same sign in train_R, val_R, test and Bonferroni-significant)"); fig.tight_layout(); fig.savefig(OUT2 / "13f_overnight_regime_heatmap.png", dpi=110); plt.show()

In [ ]:
# step 13f
if RG_OK:
    # ---- 13f.3 positive versus negative overnight days: how often, how big, why and when ----
    PN_SECS = ["EquityIndex", "OTHERS"] + OT_OTH; PN_ON = O_ON[PN_SECS]
    rows = []
    for s in PN_SECS:
        for sp, ix in OR_SP.items():
            y = PN_ON.loc[ix, s].dropna(); p = y[y > 0]; q = y[y < 0]; n = len(y)
            if n < 30: continue
            rows.append(dict(sector=s, split=sp, n=n, share_positive=len(p) / n, z_vs_half=(len(p) / n - .5) / np.sqrt(.25 / n), mean_pos_bp=p.mean(), mean_neg_bp=q.mean(), size_ratio=p.mean() / abs(q.mean()), skew=float(_rgst.skew(y)), share_of_gross_move_from_positive_days=p.sum() / (p.sum() + abs(q.sum()))))
    PN_TAB = pd.DataFrame(rows); PN_TAB.round(3).to_csv(OUT2 / "overnight_sign_frequency_size.csv", index=False)
    print("Positive versus negative overnight days (sector = equal-weight mean of its products; bp):"); print(PN_TAB.round(2).to_string(index=False))
    # features known before the overnight begins (dated t-1 or earlier) and calendar facts
    def rg_z(F): return rg_sec(F)
    _rgH, _rgL, _rgC, _rgV, _rgO = (wide(c).reindex(index=RAW.index, columns=RAW.columns) for c in ("high", "low", "close", "volume", "oi"))
    PN_F = {}
    PN_F["prev_intraday"] = rg_z(IDz.shift(1)); PN_F["prev_overnight"] = rg_z(ONz.shift(1)); PN_F["prev_last30"] = rg_z(PRE30z.shift(1))
    PN_F["prev_close_location"] = rg_z((((_rgC - _rgL) / (_rgH - _rgL).replace(0, np.nan)) - .5).shift(1))
    PN_F["prev_volume_surprise"] = rg_z(np.log(_rgV.shift(1) / _rgV.shift(2).rolling(20, min_periods=10).median()).clip(-2, 2))
    PN_F["prev_oi_change"] = rg_z(np.log(_rgO.shift(1) / _rgO.shift(2)).clip(-.2, .2))
    PN_F["prev_5day_return"] = rg_z((RAW.rolling(5, min_periods=4).sum() / (SIGd * np.sqrt(5))).clip(-6, 6).shift(1))
    _rgd = pd.Series(RAW.index, index=RAW.index); _rggap = (_rgd - _rgd.shift(1)).dt.days
    PN_CAL = pd.DataFrame(dict(long_gap=(_rggap >= 4).astype(float), monday=(RAW.index.dayofweek == 0).astype(float), friday=(RAW.index.dayofweek == 4).astype(float),
                               month_end=(RAW.index.to_series().groupby([RAW.index.year, RAW.index.month]).cumcount(ascending=False).to_numpy() < 3).astype(float),
                               month_start=(RAW.index.to_series().groupby([RAW.index.year, RAW.index.month]).cumcount().to_numpy() < 3).astype(float),
                               vol_level=RG["V"], trend_level=RG["T"], comove_level=RG["C"]), index=RAW.index)
    PN_FEATS = list(PN_F) + list(PN_CAL.columns); PN_SC = pool_ix = RG_POOL
    # regression of the overnight return (vol units) on the features, per sector and period; features standardised on the train/validation pool
    S_ONz = rg_sec(ONz); S_ONz["OTHERS"] = S_ONz[OT_OTH].mean(axis=1); rows = []; pn_hold = {}
    for s in PN_SECS:
        PNX = pd.concat([PN_F[f][s].rename(f) for f in PN_F] + [PN_CAL[c] for c in PN_CAL.columns], axis=1) if s != "OTHERS" else pd.concat([pd.concat([PN_F[f][o] for o in OT_OTH], axis=1).mean(axis=1).rename(f) for f in PN_F] + [PN_CAL[c] for c in PN_CAL.columns], axis=1)
        PNmu, PNsd = PNX.loc[pool_ix].mean(), PNX.loc[pool_ix].std().replace(0, np.nan); PNXs = ((PNX - PNmu) / PNsd).fillna(0.0); pn_hold[s] = PNXs
        for sp, ix in OR_SP.items():
            m = hac(S_ONz.loc[ix, s].rename("y"), PNXs.loc[ix])
            if m is None: continue
            for f in PN_FEATS: rows.append(dict(sector=s, split=sp, feature=f, b=m.params[f], t=m.tvalues[f]))
    PN_COEF = pd.DataFrame(rows); PNWp = PN_COEF.pivot_table(index=["sector", "feature"], columns="split", values=["b", "t"])
    PN_W = pd.DataFrame(dict(b_train=PNWp[("b", "train_R")], t_train=PNWp[("t", "train_R")], b_val=PNWp[("b", "val_R")], t_val=PNWp[("t", "val_R")], b_test=PNWp[("b", "test")], t_test=PNWp[("t", "test")])).reset_index()
    PN_TB = bonferroni_t(max(len(PN_W), 1)); PN_W["same_sign"] = (np.sign(PN_W.b_train) == np.sign(PN_W.b_val)) & (np.sign(PN_W.b_val) == np.sign(PN_W.b_test))
    PN_W["supported"] = PN_W.same_sign & (PN_W.t_test.abs() > PN_TB) & (PN_W.t_val.abs() > 2); PN_W.round(4).to_csv(OUT2 / "overnight_drivers_regression.csv", index=False)
    print(f"\nDrivers of the overnight return (vol units), {len(PN_W)} sector x feature tests, Bonferroni |t| > {PN_TB:.2f} on test and |t| > 2 on validation:"); print(PN_W[PN_W.supported].round(3).sort_values("t_test", key=abs, ascending=False).head(25).to_string(index=False) if PN_W.supported.any() else "none supported")
    print("Features dated t-1 or earlier or calendar facts are known before the night session; they are usable. Nothing contemporaneous is in this table.")
    # sign prediction: logistic model fitted on train_R, area under the curve on validation_R and test
    def rg_auc(y, p):
        r = _rgst.rankdata(p); n1 = y.sum(); n0 = len(y) - n1
        return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 > 0 and n0 > 0 else np.nan
    _ry = np.r_[0, 0, 1, 1, 1]; assert abs(rg_auc(_ry, np.array([.1, .2, .3, .4, .5])) - 1) < 1e-12 and abs(rg_auc(_ry, np.array([.5, .4, .3, .2, .1]))) < 1e-12; print("unit check passed: AUC")
    rows = []
    for s in PN_SECS:
        PNyb = (PN_ON[s] > 0).astype(float).where(PN_ON[s].notna()); PNXs = pn_hold[s]; itr = RG_IDX["train"].intersection(PNyb.dropna().index); kc = [c for c in PNXs.columns if PNXs.loc[itr, c].std() > 0]
        try:
            lg = sm.Logit(PNyb.loc[itr], sm.add_constant(PNXs.loc[itr, kc])).fit(disp=0, maxiter=200); r = dict(sector=s)
            for sp in ("val_R", "test"):
                ix = OR_SP[sp].intersection(PNyb.dropna().index); r["auc_" + sp] = rg_auc(PNyb.loc[ix].to_numpy(), lg.predict(sm.add_constant(PNXs.loc[ix, kc], has_constant="add")).to_numpy())
            rows.append(r)
        except Exception as e: print("logit failed for", s, type(e).__name__)
    PN_AUC = pd.DataFrame(rows); PN_AUC.round(4).to_csv(OUT2 / "overnight_sign_auc.csv", index=False)
    print("\nCan the sign of the next overnight return be predicted from these features? AUC (0.5 = no skill):"); print(PN_AUC.round(3).to_string(index=False))
    # why: what happens inside positive and negative overnight days, and do they persist or fade during the day
    rows = []; S_IDx = O_ID[PN_SECS]; S_NIx, S_OPx = O_NI.reindex(columns=PN_SECS), O_OP.reindex(columns=PN_SECS)
    for s in PN_SECS:
        sg = np.sign(PN_ON[s])
        for sp, ix in OR_SP.items():
            for lab, v in (("positive", 1), ("negative", -1)):
                dd = ix[(sg.reindex(ix) == v).to_numpy()]; nxt = sg.shift(-1).reindex(dd)
                r = dict(sector=s, split=sp, overnight_sign=lab, days=len(dd), ON_bp=PN_ON.loc[dd, s].mean(), night_session_bp=S_NIx.loc[dd, s].mean() if s in S_NIx else np.nan, opening_bar_bp=S_OPx.loc[dd, s].mean() if s in S_OPx else np.nan,
                         same_day_intraday_bp=S_IDx.loc[dd, s].mean(), p_next_overnight_positive=float((nxt > 0).sum() / max(nxt.notna().sum(), 1)))
                rows.append(r)
    PN_WHY = pd.DataFrame(rows); PN_WHY.round(3).to_csv(OUT2 / "overnight_sign_anatomy.csv", index=False)
    print("\nInside positive and negative overnight days (night-session and opening-bar parts, the intraday that follows, persistence of sign):"); print(PN_WHY[PN_WHY.sector.isin(["EquityIndex", "OTHERS"])].round(2).to_string(index=False))
    # when: probability of a positive overnight by condition, lift over the period's base rate, stable if same sign of lift in all three periods
    def pn_bin(s):
        f = PN_F["prev_intraday"][s] if s != "OTHERS" else pd.concat([PN_F["prev_intraday"][o] for o in OT_OTH], axis=1).mean(axis=1)
        qs = f.loc[RG_POOL].quantile([.2, .4, .6, .8]).to_numpy(); return pd.Series(np.where(f.isna(), np.nan, np.digitize(f.fillna(0), qs)), index=f.index)
    rows = []
    for s in PN_SECS:
        PNyb = (PN_ON[s] > 0).astype(float).where(PN_ON[s].notna()); PNqb = pn_bin(s)
        conds = {"Monday (weekend gap)": PN_CAL.monday == 1, "Friday": PN_CAL.friday == 1, "after a closure of 4+ calendar days": PN_CAL.long_gap == 1, "last 3 trading days of month": PN_CAL.month_end == 1, "first 3 trading days of month": PN_CAL.month_start == 1}
        for dim, names in (("V", RG_VN), ("T", RG_TN)):
            for k, nm in enumerate(names): conds[("volatility " if dim == "V" else "trend ") + nm] = RG[dim] == k
        for k, nm in enumerate(("lowest", "low", "middle", "high", "highest")): conds["yesterday's intraday return " + nm + " quintile"] = PNqb == k
        for cn, cm in conds.items():
            r = dict(sector=s, condition=cn)
            for sp, ix in OR_SP.items():
                yy = PNyb.reindex(ix).dropna(); mm = cm.reindex(yy.index).fillna(False).to_numpy(bool); n = int(mm.sum())
                r[sp + "_n"] = n; r[sp + "_lift"] = float(yy[mm].mean() - yy.mean()) if n >= 20 else np.nan; r[sp + "_z"] = float((yy[mm].mean() - yy.mean()) / np.sqrt(yy.mean() * (1 - yy.mean()) / n)) if n >= 20 else np.nan
            rows.append(r)
    PN_WHEN = pd.DataFrame(rows); PN_WHEN["same_sign"] = (np.sign(PN_WHEN.train_R_lift) == np.sign(PN_WHEN.val_R_lift)) & (np.sign(PN_WHEN.val_R_lift) == np.sign(PN_WHEN.test_lift)) & PN_WHEN.test_lift.notna()
    PN_TW = bonferroni_t(max(len(PN_WHEN), 1)); PN_WHEN["supported"] = PN_WHEN.same_sign & (PN_WHEN.test_z.abs() > PN_TW); PN_WHEN.round(4).to_csv(OUT2 / "overnight_sign_when.csv", index=False)
    print(f"\nWhen is the overnight return positive? Lift = P(positive | condition) - P(positive), {len(PN_WHEN)} tests, Bonferroni |z| > {PN_TW:.2f} on test, same sign in all three periods:")
    print(PN_WHEN[PN_WHEN.supported].round(3).sort_values("test_z", key=abs, ascending=False).head(25)[["sector", "condition", "train_R_lift", "val_R_lift", "test_lift", "test_z"]].to_string(index=False) if PN_WHEN.supported.any() else "no condition survives")
    # figures
    fig, axs = plt.subplots(1, 2, figsize=(15, 4.2)); _pw = PN_WHEN[PN_WHEN.sector.isin(["EquityIndex", "OTHERS"])]
    for a, s in zip(axs, ("EquityIndex", "OTHERS")):
        d = _pw[_pw.sector == s].set_index("condition")[["train_R_lift", "val_R_lift", "test_lift"]] * 100; d.plot.barh(ax=a, width=.8); a.axvline(0, color="k", lw=.8); a.set_title(("EquityIndex" if s == "EquityIndex" else "All other sectors") + ": change in P(overnight > 0), points"); a.grid(alpha=.3); a.tick_params(axis="y", labelsize=7)
    fig.tight_layout(); fig.savefig(OUT2 / "13f_overnight_sign_when.png", dpi=110); plt.show()
    _pc = PN_W.pivot(index="feature", columns="sector", values="t_test").reindex(PN_FEATS); fig, a = plt.subplots(figsize=(11, 6)); im = a.imshow(_pc.values.astype(float), cmap="RdBu", vmin=-4, vmax=4, aspect="auto")
    a.set_xticks(range(_pc.shape[1])); a.set_xticklabels(_pc.columns, rotation=45, ha="right"); a.set_yticks(range(len(_pc))); a.set_yticklabels(_pc.index); fig.colorbar(im, label="t on test")
    _ps = PN_W.pivot(index="feature", columns="sector", values="supported").reindex(index=_pc.index, columns=_pc.columns)
    for i in range(_pc.shape[0]):
        for j in range(_pc.shape[1]):
            if _ps.values[i, j] == True: a.text(j, i, "*", ha="center", va="center", fontsize=14)
    a.set_title("Drivers of the overnight return: t on test (* = supported)"); fig.tight_layout(); fig.savefig(OUT2 / "13f_overnight_drivers.png", dpi=110); plt.show()
    # summary
    def _rgl(df, c, cols): return [dict(zip(cols, r)) for r in df[df[c]][cols].round(3).itertuples(index=False, name=None)][:12]
    RGS_SUMMARY = dict(regime_days=int(len(RG_IX)), regime_dimensions=dict(volatility=list(RG_VN), trend=list(RG_TN), comovement=list(RG_CN)),
                       coverage=RG_COV.to_dict("records"), blocks=dict(train=int((RG_BLOCKS.assigned == "train").sum()), val=int((RG_BLOCKS.assigned == "val").sum()), purged_days=int(len(RG_PURGED))),
                       equity_vs_others=OR_DIFF.to_dict("records"), overnight_regime_supported=_rgl(OR_REG, "supported", ["leg", "sector", "dimension", "regime", "train_R_bp", "val_R_bp", "test_bp", "test_t"]),
                       sign_drivers_supported=_rgl(PN_W, "supported", ["sector", "feature", "b_train", "b_val", "b_test", "t_test"]), sign_when_supported=_rgl(PN_WHEN, "supported", ["sector", "condition", "train_R_lift", "val_R_lift", "test_lift", "test_z"]),
                       sign_auc=PN_AUC.round(3).to_dict("records"), frequency=PN_TAB.round(3).to_dict("records"), tests=dict(regime_cells=NT_OR, drivers=len(PN_W), when=len(PN_WHEN)))
    RGS_SUMMARY["reading"] = ("Regimes are defined ex ante from earlier data only; train and validation are interleaved blocks inside each regime with a purge, the test is chronological and read once. "
                              + "EquityIndex differs from the other sectors overnight in " + str([(r["leg"]) for r in OR_DIFF.to_dict("records") if r["EquityIndex_differs"]] or "no leg") + "; "
                              + str(len(RGS_SUMMARY["overnight_regime_supported"])) + " regime cells, " + str(len(RGS_SUMMARY["sign_drivers_supported"])) + " drivers and " + str(len(RGS_SUMMARY["sign_when_supported"])) + " conditions survive the Bonferroni and same-sign rules.")
    print("\n" + RGS_SUMMARY["reading"])

## Step 14 — Granger causality across time frames, with survival

The pipeline from earlier, unchanged at every frequency:
* train with names → BH-FDR → validation re-test → freeze → **blind test on coded series** → unseal and score;
* intraday returns have session-open bars removed, are divided by the train time-of-day profile, and are EWMA-scaled.

**Survival** has two dimensions:
1. **Across lags (horizon).** For each validated link, the t-stat of $x_{t-k}$ in $y_t = c + a\,y_{t-1} + m\,\bar r_{t-1} + b\,x_{t-k}$, one k at a time. The *survival horizon* is the last k (in `SURV_LAGS`, consecutive from k = 1) where |t| > 2 with the frozen sign. It is shown in minutes or days.
2. **Through time.** The Granger test is repeated in rolling windows (`SURV_WIN_DAYS`) over the whole sample. *Share alive* is the fraction of windows significant with the same sign, *typical life* is the mean length of consecutive significant runs, and *alive now* means the last window is significant.

**Used later.** Replicated links go to the relationship library (Step 17), to strategies C1 / C4 (Step 23) and to the monitor (Step 22). Survival gives the holding period.

In [ ]:
# step 14
GRES, TFDATA = {}, {}
for tf in GRANGER_TFS:
    if tf == "1D":
        Zs, spl, univ = Z, SPL, UNIV
    else:
        F_ = tf_frames(tf)
        if F_ is None:
            print(f"{tf}: no 5-min bars in the cache — skipped"); continue
        Zs = tf_z(F_, tf); spl = split_index(Zs.index); del F_
        cnt = pd.DataFrame({k: Zs.loc[v].notna().sum() for k, v in spl.items()})
        var_ok = pd.DataFrame({k: Zs.loc[v].std() > 1e-9 for k, v in spl.items()})        # drop series that are constant in any period (halted products)
        univ = cnt.index[((cnt >= MIN_OBS_SPLIT_TF.get(tf, MIN_OBS_SPLIT_INTRA)) & var_ok).all(axis=1)].tolist()
        if len(univ) < 3:
            print(f"{tf}: fewer than 3 products with enough bars — skipped"); continue
    TFDATA[tf] = dict(Z=Zs[univ], SPL=spl, UNIV=univ)
    print(f"\n==================== {tf}: {len(univ)} series | train {len(spl['train'])} / val {len(spl['val'])} / test {len(spl['test'])} bars ====================")
    res = granger_discover(Zs.loc[spl["train"], univ], Zs.loc[spl["val"], univ], tf, P_LAG_TF[tf], MIN_PR2_TF[tf], sens=(tf == "1D"))
    granger_blind(res, Zs.loc[spl["test"], univ]); granger_score(res, Zs.loc[spl["test"], univ])
    GRES[tf] = res

GSUM = pd.DataFrame([dict(tf=tf, series=len(r["cols"]), pairs=r["n_pairs"], train_sig=int(r["G_tr"].sig.sum()), candidates=len(r["CAND"]),
                          replicated=r["n_rep"], binom_p=r["binom_p"], blind_sig=int(r["G_bl"].sig.sum()),
                          spearman_train_test=r["rho"], median_pr2_sig=r["G_tr"][r["G_tr"].sig].partial_r2.median())
                     for tf, r in GRES.items()]).set_index("tf")
GSUM.to_csv(OUT2 / "granger_by_timeframe.csv")
print("\nGranger by time frame:\n", GSUM.round(4).to_string())

In [ ]:
# step 14
# detailed figures for the daily and finest intraday frame; a cross-frame summary for all
for tf in [t for t in ("1D", "5m") if t in GRES]:
    plot_granger_train(GRES[tf], "14"); plot_granger_blind(GRES[tf], "14"); plot_granger_score(GRES[tf], "14")

links = sorted({(c, e) for r in GRES.values() for c, e in r["G_tr"].loc[r["G_tr"].sig, ["cause", "effect"]].values})
if links:
    LQ = pd.DataFrame(index=[f"{c}→{e}" for c, e in links], columns=list(GRES), dtype=float)
    for tf, r in GRES.items():
        g = r["G_tr"].set_index(["cause", "effect"])
        for c, e in links:
            if (c, e) in g.index:
                LQ.loc[f"{c}→{e}", tf] = -np.log10(max(g.loc[(c, e), "q"], 1e-12))
    LQ = LQ.loc[LQ.max(axis=1).sort_values(ascending=False).index[:40]]
    fig, axs = plt.subplots(1, 2, figsize=(15, max(4, 0.22 * len(LQ))), gridspec_kw={"width_ratios": [1, 1.2]})
    heat(axs[0], LQ, vmin=0, vmax=max(3, np.nanpercentile(LQ.values, 95)), label="-log10 q (train)", annot=True, fmt="{:.1f}")
    axs[0].set_title("Same link, different time frames (train strength)")
    ax = axs[1]; x = np.arange(len(GSUM)); w = 0.27
    for k, (col, c) in enumerate(zip(["train_sig", "candidates", "replicated"], [MUTE, "#86b6ef", SERIES[0]])):
        ax.bar(x + (k - 1) * w, GSUM[col], width=w, color=c, label=col)
    ax.set_xticks(x); ax.set_xticklabels(GSUM.index); ax.legend(frameon=False, fontsize=8)
    ax.set_title("Links found (train), validated, and replicated blind (test)")
    plt.tight_layout()
    savefig(fig, "14_mtf_summary", "Granger across time frames", "14")

In [ ]:
# step 14
# survival 1: lag decay (horizon) for validated links
def lag_tstats(Zs, cause, eff, lags):
    y = Zs[eff].to_numpy(); x = Zs[cause].to_numpy()
    with np.errstate(all="ignore"):
        mk = np.nanmean(Zs.to_numpy(), axis=1)
    out = []
    for k in lags:
        X = np.column_stack([np.ones(len(y)), lagmat(y, 1)[:, 0], lagmat(mk, 1)[:, 0], lagmat(x, k)[:, k - 1]])
        m = np.isfinite(y) & np.isfinite(X).all(1)
        if m.sum() < 60:
            out.append(np.nan); continue
        b, *_ = np.linalg.lstsq(X[m], y[m], rcond=None); e = y[m] - X[m] @ b
        cov = (e @ e / (m.sum() - 4)) * np.linalg.pinv(X[m].T @ X[m]); out.append(b[3] / np.sqrt(cov[3, 3]))
    return np.array(out)

# survival 2: rolling windows through time
def rolling_survival(Zs, cause, eff, p, sign, tf):
    win, step = SURV_WIN_DAYS[tf if tf in ("1D", "1W") else "intra"]
    td = to_trading_day(Zs.index) if tf not in ("1D", "1W") else Zs.index
    days = pd.DatetimeIndex(sorted(set(td.dropna())))
    rows = []
    for s0 in range(0, max(len(days) - win, 1), step):
        d0, d1 = days[s0], days[min(s0 + win, len(days) - 1)]
        m = (td >= d0) & (td <= d1)
        g = granger_matrix(Zs.loc[m], p, pairs=[(cause, eff)])
        pv, cs = g.p.iloc[0], g.coef_sum.iloc[0]
        rows.append(dict(start=d0, end=d1, p=pv, sig=bool(pv < 0.05 and np.sign(cs) == sign) if np.isfinite(pv) else False,
                         signed=np.sign(cs) * -np.log10(max(pv, 1e-12)) if np.isfinite(pv) else np.nan))
    return pd.DataFrame(rows)

surv_rows, ROLLSURV = [], {}
for tf, r in GRES.items():
    D = TFDATA[tf]; Zpre = D["Z"].loc[D["SPL"]["train"].append(D["SPL"]["val"])]
    for c, e, cs in r["CAND"][["cause", "effect", "coef_sum"]].values[:15]:
        lags_ = SURV_LAGS_D if tf in ("1D", "1W") else SURV_LAGS; tv = lag_tstats(Zpre, c, e, lags_); sgn = np.sign(cs)
        alive = [(np.isfinite(t) and abs(t) > 2 and np.sign(t) == sgn) for t in tv]
        hb = 0
        for k, a in zip(lags_, alive):
            if a:
                hb = k
            else:
                break
        rs = rolling_survival(D["Z"], c, e, r["p"], sgn, tf); ROLLSURV[(tf, c, e)] = rs
        runs = [len(list(g)) for v, g in itertools.groupby(rs.sig) if v]
        step = SURV_WIN_DAYS[tf if tf in ("1D", "1W") else "intra"][1] * (5 if tf == "1W" else 1)
        rep = r["rep"].set_index(["cause", "effect"]).replicated.get((c, e), False) if len(r["rep"]) else False
        surv_rows.append(dict(tf=tf, cause=c, effect=e, coef_sum=cs, replicated=bool(rep), horizon_bars=hb,
                              horizon=f"{hb} weeks" if tf == "1W" else (f"{hb} days" if tf == "1D" else f"{hb * TF_MIN[tf]} min"),
                              horizon_minutes=hb * TF_MIN[tf], share_alive=rs.sig.mean() if len(rs) else np.nan,
                              typical_life_days=(np.mean(runs) * step) if runs else 0.0, alive_now=bool(rs.sig.iloc[-1]) if len(rs) else False,
                              **{f"t_lag{k}": t for k, t in zip(lags_, tv)}))
SURV = pd.DataFrame(surv_rows)
if len(SURV):
    SURV.to_csv(OUT2 / "granger_survival.csv", index=False)
    print(SURV[["tf", "cause", "effect", "replicated", "horizon", "share_alive", "typical_life_days", "alive_now"]].round(3).to_string(index=False))
    fig, axs = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [1, 1.4]})
    ax = axs[0]
    for k, r_ in enumerate(SURV.itertuples()):
        lg_ = SURV_LAGS_D if r_.tf in ("1D", "1W") else SURV_LAGS
        tv = np.array([getattr(r_, f"t_lag{l}") for l in lg_]) * np.sign(r_.coef_sum)
        ax.plot(np.array(lg_) * TF_MIN[r_.tf] / (1 if r_.tf != "1D" else TF_MIN["1D"] / 1440), tv, marker="o", ms=3,
                color=SERIES[list(GRES).index(r_.tf) % 8], alpha=0.8, label=r_.tf if r_.tf not in [s.tf for s in list(SURV.itertuples())[:k]] else None)
    ax.axhline(2, color=INK2, ls="--", lw=1); ax.axhline(0, color=INK2, lw=1); ax.set_xscale("log")
    ax.set_xlabel("how far ahead (minutes; daily links: days × 1440)"); ax.set_ylabel("t-stat in the frozen direction")
    ax.legend(frameon=False, fontsize=8, title="time frame"); ax.set_title("Lag decay: how far ahead a link still predicts")
    ax = axs[1]
    keys = [k for k in ROLLSURV if len(ROLLSURV[k])][:20]
    if keys:
        L = max(len(ROLLSURV[k]) for k in keys); M = np.full((len(keys), L), np.nan)
        for i, k in enumerate(keys):
            v = ROLLSURV[k].signed.to_numpy() * np.sign(SURV.set_index(["tf", "cause", "effect"]).coef_sum.get(k, 1)); M[i, :len(v)] = v
        Mdf = pd.DataFrame(M, index=[f"[{k[0]}] {k[1]}→{k[2]}" for k in keys], columns=range(L))
        heat(ax, Mdf.clip(-6, 6), cmap=DIV, center=0, vmax=6, label="signed -log10 p (+ = frozen direction)")
        ax.set_xticks([]); ax.set_xlabel("rolling window (time →)")
    ax.set_title("Survival through time (orange = alive in the frozen direction)")
    plt.tight_layout()
    savefig(fig, "14_survival", "Relationship survival: lag decay and through time", "14")

# aggregational statistics across frequencies
agg = []
for tf, D in TFDATA.items():
    x = D["Z"].loc[D["SPL"]["train"]]
    agg.append(dict(tf=tf, minutes=TF_MIN[tf], exkurt=np.nanmedian([stats.kurtosis(x[c].dropna()) for c in x]),
                    ac1=np.nanmedian([x[c].autocorr(1) for c in x]), hill=np.nanmedian([hill(np.abs(x[c].dropna().values)) for c in x])))
AGG = pd.DataFrame(agg).set_index("tf"); print("\nStylised facts by frequency (median across series):\n", AGG.round(3).to_string())

### 14.x — Every relationship found, in one table

The figures above show the **significant** links. This table lists **every link that was significant at any stage** (train discovery, or the blind test) across all time frames, with its full history: effect size and sign, validation, blind-test replication, out-of-sample forecast gain, lag decay and whether it is alive now. `verdict` is the one-word summary. The full list of all tested pairs is in `granger_train_<tf>.csv`.

| verdict | meaning |
|---|---|
| `REPLICATED` | significant on train, confirmed on validation, same sign and significant in the blind test |
| `VALIDATED` | confirmed on validation, not significant in the blind test (kept as a watch-list item) |
| `TRAIN ONLY` | significant on train, failed validation: treated as a chance finding |
| `BLIND ONLY` | not found on train but significant in the blind test: a new finding that has **not** been validated |

**Used later.** `REPLICATED` and `VALIDATED` links are the Granger entries of the relationship library (Step 17) and the daily ones become strategy C1 (Step 23). Their survival columns decide the maximum holding period.

In [ ]:
# step 14
tab_rows = []
for tf, r in GRES.items():
    G_tr = r["G_tr"].set_index(["cause", "effect"]); G_va = r["G_va"].set_index(["cause", "effect"]) if len(r["G_va"]) else pd.DataFrame()
    G_te = r["G_te"].set_index(["cause", "effect"]); rep = r["rep"].set_index(["cause", "effect"]) if len(r["rep"]) else pd.DataFrame()
    sc = r["scored"].set_index(["cause", "effect"]) if len(r.get("scored", [])) else pd.DataFrame()
    sv = SURV[SURV.tf == tf].set_index(["cause", "effect"]) if len(SURV) else pd.DataFrame()
    keys = set(G_tr.index[G_tr.sig]) | set(G_te.index[G_te.sig.fillna(False).astype(bool)])
    for k in keys:
        in_tr = k in G_tr.index and bool(G_tr.loc[k, "sig"]); va = G_va.loc[k] if len(G_va) and k in G_va.index else None
        te = G_te.loc[k] if k in G_te.index else None; rp = rep.loc[k] if len(rep) and k in rep.index else None
        validated = bool(va.candidate) if va is not None else False; replicated = bool(rp.replicated) if rp is not None else False
        verdict = "REPLICATED" if replicated else "VALIDATED" if validated else "TRAIN ONLY" if in_tr else "BLIND ONLY"
        trr = G_tr.loc[k] if k in G_tr.index else None
        tab_rows.append(dict(tf=tf, cause=k[0], effect=k[1], sectors=f"{SEC_MAP.get(k[0], '?')} → {SEC_MAP.get(k[1], '?')}", sign="+" if (trr is not None and trr.coef_sum > 0) else "−",
                             partial_r2_train=trr.partial_r2 if trr is not None else np.nan, q_train=trr.q if trr is not None else np.nan, robust_lags=trr.robust_lags if trr is not None else np.nan,
                             p_val=va.p if va is not None else np.nan, p_test=te.p if te is not None else np.nan, q_test=te.q if te is not None else np.nan,
                             oos_r2_vs_ar=sc.loc[k, "r2_oos_vs_ar"] if len(sc) and k in sc.index and "r2_oos_vs_ar" in sc else np.nan,
                             dm_p=sc.loc[k, "dm_p"] if len(sc) and k in sc.index and "dm_p" in sc else np.nan,
                             horizon=sv.loc[k, "horizon"] if len(sv) and k in sv.index else "", share_alive=sv.loc[k, "share_alive"] if len(sv) and k in sv.index else np.nan,
                             alive_now=bool(sv.loc[k, "alive_now"]) if len(sv) and k in sv.index else np.nan, verdict=verdict))
REL14 = pd.DataFrame(tab_rows)
if len(REL14):
    order_ = {"REPLICATED": 0, "VALIDATED": 1, "BLIND ONLY": 2, "TRAIN ONLY": 3}
    REL14 = REL14.assign(_o=REL14.verdict.map(order_)).sort_values(["_o", "tf", "q_train"]).drop(columns="_o").reset_index(drop=True)
    REL14.round(5).to_csv(OUT2 / "granger_all_links.csv", index=False)
    print("Links by time frame and verdict:\n", REL14.pivot_table(index="tf", columns="verdict", values="cause", aggfunc="count", fill_value=0).to_string())
    print(f"\nAll {len(REL14)} links (full table: granger_all_links.csv):")
    with pd.option_context("display.max_rows", 400, "display.width", 250):
        print(REL14.round(4).to_string(index=False))
else:
    print("no significant link in any time frame")

## Step 15 — Level relationships: cointegration, correlation, tail dependence, sector lead
* **Cointegration** (Engle–Granger) on train log prices for every **within-sector** pair, the most correlated **cross-sector** pairs (`N_CROSS_PAIRS`) and the economic spreads (crush, coke margin, steel margin …). The hedge ratio is fixed on train. Validation and test check that the spread still mean-reverts (ADF), and the AR(1) half-life gives the trade horizon.
* **Correlation stability**: the train vs test correlation of vol-scaled returns, which shows the hedges that hold.
* **Tail dependence**: $\lambda_L = P(z_b < q_{5\%} \mid z_a < q_{5\%})$ and $\lambda_U$ for the upper tail, on GARCH residuals. It is compared with the λ a **Gaussian copula with the same correlation** would give, and the *excess* is contagion that correlation misses (diversification fails exactly when it is needed).
* **Sector → product**: does the rest of the sector (equal-weight, excluding the product) Granger-lead the product? This is the "laggard catches up with its sector" effect.

**Used later.** Cointegrated pairs and tail dependence go to the relationship library (Step 17) and to strategy C2 (Step 23). The sector lead–lag feeds Step 21.

In [ ]:
# step 15
def half_life(s):
    s = pd.Series(s).dropna(); ds = s.diff().dropna(); x = s.shift(1).loc[ds.index]
    if len(ds) < 30:
        return np.nan
    b = np.polyfit(x.values, ds.values, 1)[0]
    return -np.log(2) / np.log(1 + b) if -1 < b < 0 else np.inf

lp_tr = LP.loc[SPL["train"], UNIV]
pair_set = set()
for sec in sorted(set(SEC_MAP[UNIV])):
    mem = sorted(c for c in UNIV if SEC_MAP[c] == sec)
    pair_set |= {(a, (b,), "within") for a, b in itertools.combinations(mem, 2)}
C_ = Ztr.corr(min_periods=100)
cross = [(a, b) for a, b in itertools.combinations(UNIV, 2) if SEC_MAP[a] != SEC_MAP[b]]
cross = sorted(cross, key=lambda ab: -abs(C_.loc[ab[0], ab[1]]) if np.isfinite(C_.loc[ab[0], ab[1]]) else 0)[:N_CROSS_PAIRS]
pair_set |= {(a, (b,), "cross") for a, b in cross}
for name, (y, xs) in SPREADS.items():
    if y in UNIV and all(x in UNIV for x in xs):
        pair_set.add((y, tuple(xs), "spread"))
co_rows = []
for a, bs, scope in sorted(pair_set):
    d = lp_tr[[a, *bs]].dropna()
    if len(d) < 250:
        continue
    try:
        p_eg = coint(d[a].values, d[list(bs)].values)[1]
    except Exception:
        continue
    _, beta = ols(np.c_[np.ones(len(d)), d[list(bs)].values], d[a].values)
    spread = LP[a] - LP[list(bs)] @ beta[1:] - beta[0]
    row = dict(type="coint", a=a, b="+".join(bs), scope=scope, p_train=p_eg, hl_train=half_life(spread.loc[SPL["train"]]), beta=np.round(beta[1:], 3).tolist())
    for sp in ("val", "test"):
        sv = spread.loc[RAW.index.intersection(SPL[sp])].dropna()
        row[f"adf_p_{sp}"] = adfuller(sv.values, maxlag=1, autolag=None)[1] if len(sv) > 60 else np.nan; row[f"hl_{sp}"] = half_life(sv)
    co_rows.append(row)
COINT = pd.DataFrame(co_rows)
COINT["q_train"] = bh(COINT.p_train)
COINT["valid"] = (COINT.p_train < COINT_P) & (COINT.adf_p_val < 0.10) & (COINT.hl_train < 60)
COINT["test_ok"] = COINT.valid & (COINT.adf_p_test < 0.10)
COINT.to_csv(OUT2 / "cointegration.csv", index=False)
print(f"{len(COINT)} pairs/spreads tested | cointegrated on train (p<{COINT_P}): {(COINT.p_train < COINT_P).sum()} | "
      f"still mean-reverting in validation: {COINT.valid.sum()} | and in test: {COINT.test_ok.sum()}")
print(COINT.sort_values("p_train").head(15).round(3).to_string(index=False))

# correlation stability
Cte = Zte.corr(min_periods=50)
corr_rows = [dict(a=a, b=b, scope="within" if SEC_MAP[a] == SEC_MAP[b] else "cross", rho_train=C_.loc[a, b], rho_test=Cte.loc[a, b])
             for a, b in itertools.combinations(UNIV, 2)]
CORR = pd.DataFrame(corr_rows); CORR["delta"] = CORR.rho_test - CORR.rho_train

# tail dependence on GARCH residuals
def tail_dep(Sx, q=TAILDEP_Q):
    lo = (Sx < Sx.quantile(1 - q)).astype(float); hi = (Sx > Sx.quantile(q)).astype(float)
    lo = lo.where(Sx.notna(), 0); hi = hi.where(Sx.notna(), 0)
    lamL = (lo.T @ lo) / np.maximum(lo.sum().values[:, None], 1); lamU = (hi.T @ hi) / np.maximum(hi.sum().values[:, None], 1)
    return lamL, lamU
LL_tr, LU_tr = tail_dep(RES["train"][UNIV_V]); LL_te, LU_te = tail_dep(RES["test"][UNIV_V])
td_rows = [dict(a=a, b=b, scope="within" if SEC_MAP[a] == SEC_MAP[b] else "cross", lamL_train=LL_tr.loc[a, b], lamL_test=LL_te.loc[a, b],
                lamU_train=LU_tr.loc[a, b], lamU_test=LU_te.loc[a, b], rho_train=C_.loc[a, b] if a in C_ and b in C_ else np.nan)
           for a, b in itertools.combinations(UNIV_V, 2)]
TAILDEP = pd.DataFrame(td_rows)
# excess tail dependence: empirical λ_L minus what a Gaussian copula with the same correlation implies
def lam_gauss(rho, q=1 - TAILDEP_Q):
    if not np.isfinite(rho):
        return np.nan
    z = stats.norm.ppf(q)
    return stats.multivariate_normal(mean=[0, 0], cov=[[1, rho], [rho, 1]]).cdf([z, z]) / q
C_te_ = Zte.corr(min_periods=50)
TAILDEP["lamL_gauss_train"] = [lam_gauss(r) for r in TAILDEP.rho_train]
TAILDEP["lamL_gauss_test"] = [lam_gauss(C_te_.loc[a, b]) if a in C_te_ and b in C_te_ else np.nan for a, b in zip(TAILDEP.a, TAILDEP.b)]
TAILDEP["excess_train"] = TAILDEP.lamL_train - TAILDEP.lamL_gauss_train
TAILDEP["excess_test"] = TAILDEP.lamL_test - TAILDEP.lamL_gauss_test
TAILDEP = TAILDEP.sort_values("excess_train", ascending=False)
TAILDEP.to_csv(OUT2 / "tail_dependence.csv", index=False)
print(f"\nTail dependence (independence would give {1 - TAILDEP_Q:.2f}): median λ_L train {TAILDEP.lamL_train.median():.3f}, "
      f"median excess over Gaussian copula {TAILDEP.excess_train.median():.3f}, corr(excess train, test) = {TAILDEP[['excess_train', 'excess_test']].corr().iloc[0, 1]:.2f}")
print(TAILDEP.head(10).round(3).to_string(index=False))

# sector -> product lead (daily)
sl_rows = []
for i in UNIV:
    peers = [c for c in UNIV if SEC_MAP[c] == SEC_MAP[i] and c != i]
    if len(peers) < 2:
        continue
    D_ = pd.DataFrame({"SEC": Z[peers].mean(axis=1), i: Z[i]}).join(Z[UNIV].drop(columns=[i] + peers).mean(axis=1).rename("MKT_REST"))
    row = dict(type="sector_lead", a=f"{SEC_MAP[i]} ex-{i}", b=i, scope="sector")
    for sp in ("train", "val", "test"):
        g = granger_matrix(D_.loc[RAW.index.intersection(SPL[sp])], P_LAG_TF["1D"], pairs=[("SEC", i)])
        row[f"p_{sp}"] = g.p.iloc[0]; row[f"pr2_{sp}"] = g.partial_r2.iloc[0]; row[f"coef_{sp}"] = g.coef_sum.iloc[0]
    sl_rows.append(row)
SECLEAD = pd.DataFrame(sl_rows)
if len(SECLEAD):
    SECLEAD["q_train"] = bh(SECLEAD.p_train)
    SECLEAD["val_ok"] = (SECLEAD.q_train < FDR_Q) & (SECLEAD.p_val < 0.05) & (np.sign(SECLEAD.coef_val) == np.sign(SECLEAD.coef_train))
    SECLEAD["test_ok"] = SECLEAD.val_ok & (SECLEAD.p_test < 0.05) & (np.sign(SECLEAD.coef_test) == np.sign(SECLEAD.coef_train))
    print(f"\nSector → product lead: train-significant {(SECLEAD.q_train < FDR_Q).sum()}, validated {SECLEAD.val_ok.sum()}, test-confirmed {SECLEAD.test_ok.sum()}")

In [ ]:
# step 15
fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
ax = axs[0]
cc = COINT.sort_values("p_train").head(20).iloc[::-1]
ax.barh([f"{a}~{b}" for a, b in zip(cc.a, cc.b)], -np.log10(cc.p_train.clip(lower=1e-6)),
        color=np.where(cc.test_ok, SERIES[0], np.where(cc.valid, "#86b6ef", MUTE)), height=0.6)
ax.axvline(-np.log10(COINT_P), color=INK2, ls="--", lw=1); ax.tick_params(axis="y", labelsize=7)
ax.set_xlabel("-log10 Engle-Granger p (train)"); ax.set_title("Cointegration (dark = holds in val and test)")
ax = axs[1]
ax.scatter(CORR.rho_train, CORR.rho_test, s=8, c=np.where(CORR.scope == "within", SERIES[0], MUTE))
ax.plot([-1, 1], [-1, 1], color=INK2, ls="--", lw=1); ax.set_xlabel("correlation, train"); ax.set_ylabel("correlation, test")
ax.scatter([], [], color=SERIES[0], label="within sector"); ax.scatter([], [], color=MUTE, label="cross sector"); ax.legend(frameon=False, fontsize=8)
ax.set_title("Correlation stability")
ax = axs[2]
ax.scatter(TAILDEP.rho_train, TAILDEP.lamL_train, s=10, color=SERIES[1], label="λ_L (crashes together)")
ax.scatter(TAILDEP.rho_train, TAILDEP.lamU_train, s=10, color=SERIES[0], label="λ_U (spikes together)")
rr_ = np.linspace(max(-0.2, np.nanmin(TAILDEP.rho_train)), min(0.95, np.nanmax(TAILDEP.rho_train)), 30)
ax.plot(rr_, [lam_gauss(r) for r in rr_], color=INK, lw=1.2, label="Gaussian copula λ (no extra contagion)")
ax.axhline(1 - TAILDEP_Q, color=INK2, ls="--", lw=1); ax.set_xlabel("correlation (train)"); ax.set_ylabel("tail dependence (train)")
ax.legend(frameon=False, fontsize=8); ax.set_title("Tail dependence vs correlation")
plt.tight_layout()
savefig(fig, "15_level_relationships", "Cointegration, correlation stability, tail dependence", "15")

## Step 16 — Fundamental-linked relationships (economic chains)

Steps 14–15 *search* for relationships statistically: thousands of candidate pairs, so a high bar after multiple-testing correction. Here the relationships come from **economics first**, so the hypotheses are few and each has a reason to exist (`FUND_CHAINS` in Step 0b):

| Kind | Example | Why it should hold |
|---|---|---|
| processing margin | rebar vs iron ore + coke; crush (soybean vs meal + oil); PTA vs crude | producers shut or expand when the margin is out of line, which pulls the product price back |
| product spread | HRC vs rebar, soy oil vs palm oil, PP vs polyethylene | substitutes |
| input cost | methanol and urea vs thermal coal, glass vs soda ash | cost push |
| ratio / relative value | copper vs aluminium, silver vs gold, small-cap vs large-cap index | same driver, different beta |

For each chain (dependent product $y$ and driver products $x$):
1. **Level relation.** On train, $\ln P_y = a + \sum_j b_j \ln P_{x_j} + s_t$ (Engle–Granger). The hedge ratios are **frozen on train**. $s_t$ is the *fundamental spread*; its 120-day z-score $z_t$ uses past data only.
2. **Does it stay a relation?** ADF on the spread in validation and in test with the frozen $b$; AR(1) half-life.
3. **Does the spread carry information?** Rank IC of $-z_t$ against the dependent product's forward return (1, 5, 20 days): rich versus fundamentals should be followed by weakness. Train, validation and test separately.
4. **Does the driver lead?** Granger *driver → dependent* (market-controlled) in all three periods, and the *reverse* direction for contrast. A cost-push chain should lead one way.

A chain is **valid** if it is cointegrated on train, has a half-life below 60 days, and the pull-back signal ($-z$ predicting the dependent's 5-day return) is positive on train and positive with t > 1.5 on validation. ADF of the frozen-beta spread is reported for information: it has little power in a 300-day window. Valid chains join the relationship library (Step 17), the strategy library (C5, Step 23) and the drift monitor (Step 22).

**Used later.** Valid chains join the relationship library (Step 17), the strategy library as C5 (Step 23) and the monitor (Step 22).

In [ ]:
# step 16
IDX = {sp: RAW.index.intersection(SPL[sp]) for sp in ("train", "val", "test")}
FWDR = {h: RAW.rolling(h, min_periods=h).sum().shift(-h) for h in FUND_H}          # forward log return from t+1 .. t+h

def ic_t_pooled(x, y, idx, h):
    d = pd.concat([x.loc[idx], y.loc[idx]], axis=1).dropna()
    if len(d) < 60:
        return np.nan, np.nan
    c = d.iloc[:, 0].corr(d.iloc[:, 1], method="spearman"); n_eff = max(len(d) / h, 5)
    return c, c * np.sqrt(n_eff - 2) / np.sqrt(max(1 - c * c, 1e-9))

fund_rows, FUND_Z, FUND_SKIP = [], {}, []
for name, dep, drv in FUND_CHAINS:
    legs = [dep] + drv
    if any(l not in UNIV for l in legs):
        FUND_SKIP.append(name); continue
    d = LP.loc[IDX["train"], legs].dropna()
    if len(d) < FUND_MIN_N:
        FUND_SKIP.append(name); continue
    try:
        p_eg = coint(d[dep].values, d[drv].values)[1]
    except Exception:
        FUND_SKIP.append(name); continue
    _, beta = ols(np.c_[np.ones(len(d)), d[drv].values], d[dep].values)
    spread = LP[dep] - LP[drv] @ beta[1:] - beta[0]
    z = (spread - spread.rolling(FUND_Z_WIN).mean()) / spread.rolling(FUND_Z_WIN).std()
    FUND_Z[name] = z
    row = dict(chain=name, dep=dep, drivers=tuple(drv), scope="within" if all(SEC_MAP[l] == SEC_MAP[dep] for l in drv) else "cross",
               p_eg_train=p_eg, beta=np.round(beta[1:], 3).tolist(), hl_train=half_life(spread.loc[IDX["train"]]))
    for sp in ("val", "test"):
        sv = spread.loc[IDX[sp]].dropna()
        row[f"adf_p_{sp}"] = adfuller(sv.values, maxlag=1, autolag=None)[1] if len(sv) > 60 else np.nan
    for h in FUND_H:
        for sp in ("train", "val", "test"):
            row[f"ic{h}_{sp}"], row[f"t{h}_{sp}"] = ic_t_pooled(-z, FWDR[h][dep], IDX[sp], h)
    # does a driver lead the dependent product? (market-controlled Granger on vol-scaled daily returns)
    rest = [c for c in UNIV if c not in legs]
    leads = []
    for dv in drv:
        D_ = Z[[dv, dep]].join(Z[rest].mean(axis=1).rename("MKT_REST")) if rest else Z[[dv, dep]]
        g = {sp: granger_matrix(D_.loc[IDX[sp]], P_LAG_TF["1D"], pairs=[(dv, dep), (dep, dv)]) for sp in ("train", "val", "test")}
        r = dict(driver=dv)
        for sp in g:
            r[f"p_{sp}"], r[f"pr2_{sp}"], r[f"coef_{sp}"] = g[sp].p.iloc[0], g[sp].partial_r2.iloc[0], g[sp].coef_sum.iloc[0]
            r[f"rev_p_{sp}"] = g[sp].p.iloc[1]
        r["lead_ok"] = (r["p_train"] < 0.05) and (r["p_val"] < 0.10) and np.sign(r["coef_val"]) == np.sign(r["coef_train"])
        r["lead_test_ok"] = r["lead_ok"] and (r["p_test"] < 0.10) and np.sign(r["coef_test"]) == np.sign(r["coef_train"])
        r["one_way"] = r["p_train"] < r["rev_p_train"]
        leads.append(r)
    row["lead_ok"] = any(r["lead_ok"] for r in leads); row["lead_test_ok"] = any(r["lead_test_ok"] for r in leads)
    row["leaders"] = ",".join(r["driver"] for r in leads if r["lead_ok"])
    row["lead_p_train"] = min(r["p_train"] for r in leads)
    fund_rows.append(row)

FUND = pd.DataFrame(fund_rows)
if len(FUND):
    # valid = cointegrated on train, tradable half-life, and the spread's pull-back signal is confirmed on validation (ADF on a frozen-beta spread has little power in a short window, so it is reported but not required)
    FUND["valid"] = (FUND.p_eg_train < COINT_P) & (FUND.hl_train < 60) & (FUND.ic5_train > 0) & (FUND.ic5_val > 0) & (FUND.t5_val > 1.5)
    FUND["test_ok"] = FUND.valid & (FUND.ic5_test > 0) & (FUND.t5_test > 1.0)
    FUND["signal_val_ok"] = FUND.valid; FUND["signal_test_ok"] = FUND.test_ok
    FUND.drop(columns=["drivers"]).assign(drivers=FUND.drivers.map("+".join)).to_csv(OUT2 / "fundamental_chains.csv", index=False)
    print(f"{len(FUND)} chains tested ({len(FUND_SKIP)} skipped: product missing or too short) | cointegrated on train: {(FUND.p_eg_train < COINT_P).sum()} | "
          f"valid (pull-back signal confirmed in validation): {FUND.valid.sum()} | and in test: {FUND.test_ok.sum()}")
    print(f"a driver Granger-leads the dependent product (train, validation, same sign): {FUND.lead_ok.sum()}, and on test: {FUND.lead_test_ok.sum()}")
    show = ["chain", "p_eg_train", "hl_train", "adf_p_val", "adf_p_test", "ic5_train", "ic5_val", "ic5_test", "t5_test", "leaders", "valid", "test_ok", "signal_test_ok"]
    print(FUND.sort_values("p_eg_train")[show].round(3).to_string(index=False))
    if FUND_SKIP:
        print("\nskipped:", "; ".join(FUND_SKIP))
else:
    FUND = pd.DataFrame(columns=["chain", "dep", "drivers", "valid"])
    print("no chain has all its products in the universe")

# current state: the latest z of every chain
now_rows = []
for name, z in FUND_Z.items():
    zl = z.dropna()
    if len(zl):
        r_ = FUND[FUND.chain == name].iloc[0]
        now_rows.append(dict(chain=name, dep=r_.dep, z=zl.iloc[-1], side="BEAR (rich vs fundamentals)" if zl.iloc[-1] > PAIR_Z_IN else "BULL (cheap vs fundamentals)" if zl.iloc[-1] < -PAIR_Z_IN else "—",
                             valid=bool(r_.valid)))
FUND_NOW = pd.DataFrame(now_rows)
if len(FUND_NOW):
    FUND_NOW = FUND_NOW.reindex(FUND_NOW.z.abs().sort_values(ascending=False).index); FUND_NOW.to_csv(OUT2 / "fundamental_chains_now.csv", index=False)
    print("\nChains furthest from their fundamental value now (|z| > %.1f = signal):\n" % PAIR_Z_IN, FUND_NOW.head(12).round(2).to_string(index=False))

In [ ]:
# step 16
if len(FUND):
    fig, axs = plt.subplots(1, 3, figsize=(18, 5.2), gridspec_kw={"width_ratios": [1.2, 1.2, 1.3]})
    ax = axs[0]; f_ = FUND.sort_values("p_eg_train").head(22).iloc[::-1]
    col = np.where(f_.test_ok, SERIES[0], np.where(f_.valid, "#86b6ef", MUTE))
    ax.barh(f_.chain.str.slice(0, 38), -np.log10(f_.p_eg_train.clip(lower=1e-6)), color=col, height=0.65)
    ax.axvline(-np.log10(COINT_P), color=INK2, ls="--", lw=1); ax.tick_params(axis="y", labelsize=7)
    ax.set_xlabel("−log10 Engle–Granger p (train)"); ax.set_title("Fundamental chains (dark = still holds in validation and test)")
    ax = axs[1]; v_ = FUND[FUND.valid].sort_values("ic5_test", ascending=False).head(14)
    if len(v_):
        x = np.arange(len(v_)); w = 0.27
        for k, sp in enumerate(("train", "val", "test")):
            ax.bar(x + (k - 1) * w, v_[f"ic5_{sp}"], width=w, color=[MUTE, "#86b6ef", SERIES[0]][k], label=sp)
        ax.set_xticks(x); ax.set_xticklabels(v_.chain.str.slice(0, 22), rotation=75, ha="right", fontsize=6)
        ax.axhline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=8); ax.set_ylabel("rank IC of −z vs 5-day forward return")
    ax.set_title("Does 'rich vs fundamentals' predict weakness?")
    ax = axs[2]
    best = FUND[FUND.valid].sort_values("p_eg_train").head(1)
    if len(best) == 0:
        best = FUND.sort_values("p_eg_train").head(1)
    nm = best.chain.iloc[0]; z = FUND_Z[nm].dropna()
    ax.plot(z.index, z.values, color=SERIES[0], lw=0.8); ax.axhline(0, color=INK2, lw=1)
    for lv in (PAIR_Z_IN, -PAIR_Z_IN):
        ax.axhline(lv, color=SERIES[1], ls="--", lw=1)
    for sp in ("val", "test"):
        ax.axvline(SPL[sp][0], color=INK2, ls="--", lw=1)
    ax.set_title(f"Fundamental spread z-score: {nm}"); ax.set_ylabel("z (120-day)")
    plt.tight_layout()
    savefig(fig, "16_fundamental_chains", "Fundamental-linked relationships", "16")

## Step 16b — Fundamental chains again: rules by length of history, a rolling hedge ratio, no cointegration gate

Step 16 fits each chain on the **train block only** (to Feb 2015) and demands an Engle–Granger rejection, so chains whose products listed later were skipped and well-known pairs failed a test with little power. Step 16b changes four things and **keeps the calendar split** (validation and test dates are the same as everywhere else):

1. **The hedge ratio is estimated from the chain's own past.** For each date $t$, $\beta_t$ comes from the previous `FUND_BETA_WIN` = 250 joint days ending at $t-1$ (a one-day lag), and the spread is $s_t = \ln P^{dep}_t - a_t - \beta_t'\ln P^{drv}_t$. A **Kalman filter** (random-walk hedge ratio) is run alongside as a check. A chain's clock starts at its first 250 joint days; its z-score needs 60 more days of spread. The first signal is dated after both.
2. **No cointegration gate.** The Engle–Granger, Johansen and Phillips–Ouliaris results are reported, but a chain is judged by the **pull-back signal**: the rank IC of $-z_t$ against the dependent product's forward 5-day return, and a spread half-life below `FUND_HL_MAX` = 60 days.
3. **What a chain may claim depends on how much history it has** (rules below).
4. **Evidence is pooled across chains** (chain fixed effects, standard errors clustered by week, false-discovery-rate correction), and a **skip log** (`fundamental_skip_reasons.csv`) says for every chain whether Step 16 skipped it, which leg and which rule did it, and what it can do now.

### The rules, by case

| Case | History of the chain (signal days = days with a z-score) | What is tested | What it may be called |
|---|---|---|---|
| **A — long** | ≥ 500 signal days in train **and** ≥ 250 in validation | pull-back IC > 0 on train and validation (t > `FUND_T_VAL`); half-life < 60 d estimated before validation; then test (IC > 0, t > `FUND_T_TEST`) | `VALID`, `VALID + TEST OK` |
| **B — medium** | ≥ 250 signal days in validation, fewer than 500 in train | same, without the train condition | `VALID`, `VALID + TEST OK` |
| **C — short (about 3 years or less)** | ≥ 120 signal days but < 250 in validation (mostly or wholly in test) | descriptive only: IC over all signal days, half-life over the whole spread | `WATCH (promising)` if IC > 0 over all days and in test with t > 1 and half-life < 60; else `WATCH (no evidence yet)`; **never VALID** |
| **D — too short** | < 120 signal days | nothing | `WAIT`, with days until it can be evaluated |
| **E — leg missing** | a product is not in the data | nothing; a **stand-in** (`FUND_STANDIN`) may replace a short-history leg if the two co-move (correlation of daily returns ≥ 0.5 over their overlap) | `SKIP`, or a stand-in row marked as such |

**Size rule.** A chain trades at risk proportional to its years of signal: `size_cap = min(1, years / FUND_FULL_SIZE_YEARS)`. A case-C chain therefore never gets more than about a third of the risk of a long one, and the drift monitor (Step 22) stops it when its rolling Sharpe turns negative.

**Short-history tier.** Products that have too little history for the global universe `UNIV` (so they stay out of every cross-sectional step) can still sit in a chain, because a chain needs only its own legs.

**Used later.** `FUND2`, the rolling z-scores `FUND_Z2` and `fundamental_skip_reasons.csv` feed the findings report (Step 26b) and the summary. Step 16 is kept as the legacy fixed-train version for comparison; the strategies of Step 23 still read it, so the new `VALID` chains are not traded there yet.

In [ ]:
# step 16b
# ---- 16b.1 the skip log: every chain, which leg, which rule, and what the new rules do with it ----
_G0 = dict(globals())          # names present before this step; temporaries are removed again at the end of 16b.4 so later steps keep their own variables
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from scipy import stats as _st
EMBD = EMB
IDXb = {sp: RAW.index.intersection(SPL[sp]) for sp in ("train", "val", "test")}
FWDb = {h: RAW.rolling(h, min_periods=h).sum().shift(-h) for h in FUND_H}
old_skipped = set(globals().get("FUND_SKIP", []))
def _ts(d): return "" if d is None or pd.isna(d) else str(pd.Timestamp(d).date())
def _first(p): return RAW[p].first_valid_index() if p in RAW.columns else None

# stand-in rows: a short-history leg is replaced by a long-history product that co-moves with it (flagged)
CH = [dict(name=n, dep=d, drv=list(v), standin="") for n, d, v in FUND_CHAINS]
for n, d, v in FUND_CHAINS:
    for leg in [d] + list(v):
        sub = FUND_STANDIN.get(leg)
        if sub is None or sub not in RAW.columns or leg not in RAW.columns or sub in ([d] + list(v)): continue
        both = RAW[[leg, sub]].dropna()
        if len(both) < FUND_BETA_WIN + 60 or both[leg].corr(both[sub]) < FUND_STANDIN_MIN_CORR: continue
        nd, nv = (sub if d == leg else d), [sub if x == leg else x for x in v]
        CH.append(dict(name=f"{n} [stand-in {sub} for {leg}]", dep=nd, drv=nv, standin=f"{sub} for {leg} (corr {both[leg].corr(both[sub]):.2f})"))

skip_rows = []
for c in CH:
    legs = [c["dep"]] + c["drv"]
    miss = [l for l in legs if l not in RAW.columns]
    not_u = [l for l in legs if l in RAW.columns and l not in UNIV]
    firsts = {l: _first(l) for l in legs}
    J = LP[legs].dropna().index if not miss else pd.DatetimeIndex([])
    jdays = len(J); jfirst = J[0] if jdays else None
    sig_start = J[FUND_BETA_WIN + FUND_Z_WIN // 2] if jdays > FUND_BETA_WIN + FUND_Z_WIN // 2 else None
    n_by = {sp: (int(((IDXb[sp] >= sig_start)).sum()) if sig_start is not None else 0) for sp in IDXb}
    n_tr, n_va, n_te = n_by["train"], n_by["val"], n_by["test"]
    n_sig = n_tr + n_va + n_te
    tr_overlap = int(LP.loc[IDXb["train"], legs].dropna().shape[0]) if not miss else 0
    if miss: old, rule = "skipped", "E: leg not in the data"
    else:
        old = "skipped" if c["name"] in old_skipped else ("new row" if c["standin"] else "tested")
        if not_u and old == "skipped": old_rule = "a leg is not in the global universe"
        elif tr_overlap < FUND_MIN_N and old == "skipped": old_rule = f"fewer than {FUND_MIN_N} joint days in the train block ({tr_overlap})"
        else: old_rule = ""
    if miss: case = "E"
    elif n_sig < FUND_MIN_SIGNAL: case = "D"
    elif n_tr >= FUND_CASE_A_TRAIN and n_va >= FUND_CASE_B_VAL: case = "A"
    elif n_va >= FUND_CASE_B_VAL: case = "B"
    else: case = "C"
    skip_rows.append(dict(chain=c["name"], dep=c["dep"], drivers="+".join(c["drv"]), step16=old if not miss else "skipped",
                          step16_rule=("" if miss else old_rule) if old == "skipped" else "", legs_missing=",".join(miss), legs_not_in_universe=",".join(not_u),
                          first_dates="; ".join(f"{l} {_ts(firsts[l])}" for l in legs), joint_first=_ts(jfirst), joint_days=jdays,
                          signal_start=_ts(sig_start), signal_days=n_sig, train_days=n_tr, val_days=n_va, test_days=n_te,
                          new_case=case, standin=c["standin"], fixable=case in ("A", "B", "C"),
                          days_until_evaluable=(max(FUND_BETA_WIN + FUND_Z_WIN // 2 + FUND_MIN_SIGNAL - jdays, 0) if case == "D" else 0)))
SKIPLOG = pd.DataFrame(skip_rows)
SKIPLOG.to_csv(OUT2 / "fundamental_skip_reasons.csv", index=False)
cnt = SKIPLOG[~SKIPLOG.standin.astype(bool)]
print(f"{len(cnt)} chains: Step 16 tested {(cnt.step16 == 'tested').sum()} and skipped {(cnt.step16 == 'skipped').sum()}; "
      f"under the 16b rules: case A {(cnt.new_case == 'A').sum()}, B {(cnt.new_case == 'B').sum()}, C {(cnt.new_case == 'C').sum()}, D (too short) {(cnt.new_case == 'D').sum()}, E (leg missing) {(cnt.new_case == 'E').sum()}")
print(f"stand-in rows added: {int(SKIPLOG.standin.astype(bool).sum())}")
print("\nWhy Step 16 skipped them (rule that fired):\n", cnt[cnt.step16 == "skipped"].step16_rule.replace("", "leg missing").str.replace(r"\(\d+\)", "", regex=True).value_counts().to_string())
show = ["chain", "step16", "step16_rule", "legs_missing", "joint_first", "joint_days", "signal_days", "train_days", "val_days", "test_days", "new_case"]
print("\n", SKIPLOG[SKIPLOG.step16 == "skipped"][show].to_string(index=False, max_colwidth=46))

In [ ]:
# step 16b
# ---- 16b.2 rolling hedge ratio (lagged one day), Kalman check, spread, z-score ----
def rolling_beta(y, X, win):
    # beta_t from rows t-win .. t-1 (lagged one day); y (T,), X (T,k) -> (T, k+1) array of [a, b...]
    Xc = np.c_[np.ones(len(y)), X]; k1 = Xc.shape[1]
    CXX = np.cumsum(np.einsum("ti,tj->tij", Xc, Xc), axis=0); CXy = np.cumsum(Xc * y[:, None], axis=0)
    B = np.full((len(y), k1), np.nan)
    for t in range(win, len(y)):
        a_, b_ = t - 1, t - 1 - win
        A = CXX[a_] - (CXX[b_] if b_ >= 0 else 0); v = CXy[a_] - (CXy[b_] if b_ >= 0 else 0)
        try: B[t] = np.linalg.solve(A + 1e-10 * np.eye(k1), v)
        except np.linalg.LinAlgError: pass
    return B

def kalman_z(y, X, delta, ve):
    # random-walk hedge ratio; the innovation e_t = y_t - x_t'beta_(t|t-1) uses only the past; returns standardised innovations
    Xc = np.c_[np.ones(len(y)), X]; k1 = Xc.shape[1]
    beta = np.zeros(k1); P = np.eye(k1) * 1.0; Vw = np.eye(k1) * delta / (1 - delta); z = np.full(len(y), np.nan)
    beta[0] = y[0]
    for t in range(len(y)):
        x = Xc[t]; Pp = P + Vw; e = y[t] - x @ beta; Q = x @ Pp @ x + ve
        z[t] = e / np.sqrt(Q); K = Pp @ x / Q; beta = beta + K * e; P = Pp - np.outer(K, x) @ Pp
    return z

def ic_t16(x, y, idx, h):
    d = pd.concat([x.reindex(idx), y.reindex(idx)], axis=1).dropna()
    if len(d) < 60: return np.nan, np.nan, len(d)
    c = d.iloc[:, 0].corr(d.iloc[:, 1], method="spearman"); n_eff = max(len(d) / h, 5)
    return c, c * np.sqrt(n_eff - 2) / np.sqrt(max(1 - c * c, 1e-9)), len(d)

def jo_flag(M):
    try:
        from statsmodels.tsa.vector_ar.vecm import coint_johansen
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore"); r = coint_johansen(M, det_order=0, k_ar_diff=1)
        return bool(np.real(r.lr1[0]) > r.cvt[0, 1])      # trace statistic above its 5% critical value
    except Exception: return np.nan
def po_p(y, X):
    try:
        from arch.unitroot.cointegration import phillips_ouliaris
        return float(phillips_ouliaris(y, X).pvalue)
    except Exception: return np.nan

rows16, FUND_Z2, FUND_PNL, FUND_BETAS = [], {}, {}, {}
for c, sk in zip(CH, SKIPLOG.itertuples()):
    if sk.new_case in ("D", "E"):
        rows16.append(dict(chain=c["name"], dep=c["dep"], drivers=tuple(c["drv"]), case=sk.new_case, status=("WAIT" if sk.new_case == "D" else "SKIP (leg missing)"))); continue
    legs = [c["dep"]] + c["drv"]; D_ = LP[legs].dropna(); y = D_[c["dep"]].values; X = D_[c["drv"]].values
    B = rolling_beta(y, X, FUND_BETA_WIN)
    s = pd.Series(y - np.einsum("ti,ti->t", np.c_[np.ones(len(y)), X], B), index=D_.index)
    z = (s - s.rolling(FUND_Z_WIN, min_periods=FUND_Z_WIN // 2).mean()) / s.rolling(FUND_Z_WIN, min_periods=FUND_Z_WIN // 2).std()
    z[z.index < pd.Timestamp(sk.signal_start)] = np.nan
    zk = pd.Series(kalman_z(y, X, FUND_KALMAN_DELTA, FUND_KALMAN_VE), index=D_.index); zk[zk.index < pd.Timestamp(sk.signal_start)] = np.nan
    FUND_Z2[c["name"]] = z.reindex(RAW.index); FUND_BETAS[c["name"]] = pd.DataFrame(B, index=D_.index, columns=["a"] + c["drv"])
    row = dict(chain=c["name"], dep=c["dep"], drivers=tuple(c["drv"]), case=sk.new_case, standin=c["standin"], signal_start=sk.signal_start, signal_years=round(sk.signal_days / 250, 2),
               scope="within" if all(SEC_MAP.get(l) == SEC_MAP.get(c["dep"]) for l in c["drv"]) else "cross")
    # which window was available before the first judged day (for the half-life and the cointegration tests, reported only)
    judge0 = SPL["val"][0] if sk.new_case in ("A", "B") else None
    pre = s.dropna() if judge0 is None else s.dropna().loc[:judge0 - pd.Timedelta(days=1)]
    row["hl_days"] = half_life(pre); row["hl_basis"] = "before validation" if judge0 is not None else "all signal days (descriptive)"
    Dpre = D_.loc[: (judge0 - pd.Timedelta(days=1)) if judge0 is not None else D_.index[-1]]
    if len(Dpre) > FUND_BETA_WIN:
        try: row["eg_p"] = coint(Dpre[c["dep"]].values, Dpre[c["drv"]].values)[1]
        except Exception: row["eg_p"] = np.nan
        row["johansen_5pct"] = jo_flag(Dpre.values) if len(legs) >= 2 else np.nan; row["po_p"] = po_p(Dpre[c["dep"]], Dpre[c["drv"]])
    for h in FUND_H:
        for sp in ("train", "val", "test"):
            row[f"ic{h}_{sp}"], row[f"t{h}_{sp}"], row[f"n{h}_{sp}"] = ic_t16(-z, FWDb[h][c["dep"]], IDXb[sp], h)
    allidx = RAW.index[(RAW.index >= pd.Timestamp(sk.signal_start))]
    row["ic5_all"], row["t5_all"], _ = ic_t16(-z, FWDb[5][c["dep"]], allidx, 5)
    for sp in ("val", "test"): row[f"ic5_{sp}_kalman"] = ic_t16(-zk, FWDb[5][c["dep"]], IDXb[sp], 5)[0]
    # gross pull-back P&L (position known at the close of t, earned on t+1; hedge ratio at t)
    pos = (-z.clip(-2, 2) / 2).reindex(RAW.index); Bd = FUND_BETAS[c["name"]].reindex(RAW.index)
    spr_ret = RAW[c["dep"]] - sum(Bd[d_] * RAW[d_] for d_ in c["drv"])
    pnl = (pos * spr_ret.shift(-1)).dropna()
    FUND_PNL[c["name"]] = pnl
    for sp in ("train", "val", "test"):
        q = pnl.reindex(IDXb[sp]).dropna(); row[f"sharpe_gross_{sp}"] = q.mean() / q.std() * np.sqrt(ANN) if len(q) > 60 and q.std() > 0 else np.nan
    row["turnover_per_year"] = float(pos.diff().abs().mean() * ANN)
    # p-values from t (for the false-discovery correction)
    for sp in ("val", "test"):
        t_ = row[f"t5_{sp}"]; row[f"p_{sp}"] = float(_st.norm.sf(t_)) if np.isfinite(t_) else np.nan      # one-sided: pull-back IC > 0
    hl, a_ = row["hl_days"], row["ic5_all"]
    row["size_cap"] = round(min(1.0, sk.signal_days / 250 / FUND_FULL_SIZE_YEARS), 2)
    if sk.new_case in ("A", "B"):
        tr_ok = (row["ic5_train"] > 0) if sk.new_case == "A" else True
        row["valid"] = bool(tr_ok and row["ic5_val"] > 0 and row["t5_val"] > FUND_T_VAL and np.isfinite(hl) and hl < FUND_HL_MAX)
        row["test_ok"] = bool(row["valid"] and row["ic5_test"] > 0 and row["t5_test"] > FUND_T_TEST)
        row["status"] = "VALID + TEST OK" if row["test_ok"] else ("VALID" if row["valid"] else "NOT VALID")
    else:
        prom = bool(a_ > 0 and row["t5_all"] > FUND_T_TEST and (not np.isfinite(row["ic5_test"]) or row["ic5_test"] > 0) and np.isfinite(hl) and hl < FUND_HL_MAX)
        row["valid"] = False; row["test_ok"] = False; row["status"] = "WATCH (promising)" if prom else "WATCH (no evidence yet)"
    rows16.append(row)

FUND2 = pd.DataFrame(rows16)
ev = FUND2[FUND2.case.isin(["A", "B", "C"])].copy()
for sp in ("val", "test"):                       # false-discovery rate across the chains that have data in that period
    m = ev[f"p_{sp}"].notna()
    if m.sum() > 1:
        FUND2.loc[ev.index[m], f"q_{sp}"] = multipletests(ev.loc[m, f"p_{sp}"], method="fdr_bh")[1]
FUND2["fdr_ok_test"] = (FUND2.get("q_test", np.nan) < FUND_FDR_Q) & (FUND2.get("ic5_test", np.nan) > 0)
FUND2.drop(columns=["drivers"]).assign(drivers=FUND2.drivers.map(lambda d: "+".join(d))).to_csv(OUT2 / "fundamental_chains_16b.csv", index=False)
print("status counts:", FUND2.status.value_counts().to_dict())
cols = ["chain", "case", "signal_years", "hl_days", "eg_p", "ic5_train", "ic5_val", "t5_val", "ic5_test", "t5_test", "sharpe_gross_val", "sharpe_gross_test", "turnover_per_year", "size_cap", "q_test", "status"]
print(FUND2[FUND2.case.isin(["A", "B", "C"])].sort_values(["case", "t5_test"], ascending=[True, False])[[c_ for c_ in cols if c_ in FUND2.columns]].round(3).to_string(index=False, max_colwidth=42))
print("\nCointegration tests are reported, not required: EG p < 0.05 in %d of %d chains; Johansen trace above 5%% in %d; Phillips-Ouliaris p < 0.05 in %d." % (
      (ev.eg_p < 0.05).sum(), ev.eg_p.notna().sum(), (ev.johansen_5pct == True).sum(), (ev.po_p < 0.05).sum()))
if "ic5_val_kalman" in ev:
    print("Kalman hedge ratio, same signal: mean IC validation %.3f vs rolling %.3f | test %.3f vs %.3f" % (ev.ic5_val_kalman.mean(), ev.ic5_val.mean(), ev.ic5_test_kalman.mean(), ev.ic5_test.mean()))

In [ ]:
# step 16b
# ---- 16b.3 pooled evidence: chain fixed effects, week-clustered standard errors, by case ----
def pooled16(names, idx_list, h=5, step=5):
    X, Y, G, W = [], [], [], []
    for n in names:
        d = pd.concat([-FUND_Z2[n], FWDb[h][FUND2.set_index("chain").loc[n, "dep"]]], axis=1).dropna()
        d = d.loc[d.index.isin(idx_list)].iloc[::step]
        if len(d) < 20: continue
        u = d.iloc[:, 0].rank(pct=True) - 0.5; v = d.iloc[:, 1].rank(pct=True) - 0.5          # ranks inside the chain: chain effects drop out
        X.append(u); Y.append(v); G.append(pd.Series(n, index=d.index)); W.append(pd.Series(d.index.to_period("W").astype(str), index=d.index))
    if not X: return None
    x, y, w = pd.concat(X), pd.concat(Y), pd.concat(W)
    m = sm.OLS(y.values, x.values).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(w.values)[0]})
    return dict(slope=float(m.params[0]), t=float(m.tvalues[0]), p_one_sided=float(_st.norm.sf(m.tvalues[0])), chains=len(X), obs=int(len(y)))
groups = {"all evaluable (A+B+C)": FUND2[FUND2.case.isin(["A", "B", "C"])].chain.tolist(), "long and medium (A+B)": FUND2[FUND2.case.isin(["A", "B"])].chain.tolist(),
          "short (C)": FUND2[FUND2.case == "C"].chain.tolist()}
periods = {"validation": IDXb["val"], "test": IDXb["test"], "validation + test": IDXb["val"].union(IDXb["test"])}
pool_rows = []
for g, names in groups.items():
    for pn, ix in periods.items():
        r = pooled16([n for n in names if n in FUND_Z2], ix)
        if r: pool_rows.append(dict(group=g, period=pn, **r))
FUND_POOL = pd.DataFrame(pool_rows); FUND_POOL.round(4).to_csv(OUT2 / "fundamental_pooled_16b.csv", index=False)
print("Pooled pull-back IC (slope of rank(5-day forward return) on rank(-z) inside each chain; t clustered by week; one-sided p):")
print(FUND_POOL.round(4).to_string(index=False))
nS = (FUND2.fdr_ok_test == True).sum()
print(f"\nchains whose test IC is positive after the false-discovery correction (q < {FUND_FDR_Q}): {nS} of {len(ev)}")

In [ ]:
# step 16b
# ---- 16b.4 the chains now, with the size each may carry; figure ----
now = []
for n, z in FUND_Z2.items():
    zl = z.dropna(); r_ = FUND2.set_index("chain").loc[n]
    if len(zl) and r_.status not in ("NOT VALID",):
        now.append(dict(chain=n, dep=r_.dep, z=zl.iloc[-1], side="BEAR (rich vs fundamentals)" if zl.iloc[-1] > PAIR_Z_IN else "BULL (cheap vs fundamentals)" if zl.iloc[-1] < -PAIR_Z_IN else "—",
                        status=r_.status, size_cap=r_.size_cap, last_date=zl.index[-1].date()))
FUND_NOW2 = pd.DataFrame(now)
if len(FUND_NOW2):
    FUND_NOW2 = FUND_NOW2.reindex(FUND_NOW2.z.abs().sort_values(ascending=False).index); FUND_NOW2.to_csv(OUT2 / "fundamental_chains_now_16b.csv", index=False)
    print("Chains away from their fundamental value now (|z| > %.1f), with the risk size each may carry:\n" % PAIR_Z_IN, FUND_NOW2[FUND_NOW2.z.abs() > PAIR_Z_IN].round(2).to_string(index=False))
fig, axs = plt.subplots(1, 3, figsize=(19, 7), gridspec_kw={"width_ratios": [1.25, 1.1, 0.9]})
ax = axs[0]; e_ = FUND2[FUND2.case.isin(["A", "B", "C"])].sort_values("signal_years")
cc = {"A": SERIES[0], "B": "#86b6ef", "C": SERIES[1]}
ax.barh(e_.chain.str.slice(0, 40), e_.signal_years, color=[cc[c_] for c_ in e_.case], height=0.65)
ax.axvline(FUND_CASE_A_TRAIN / 250, color=INK2, ls="--", lw=0.8); ax.axvline(FUND_FULL_SIZE_YEARS, color=INK2, ls=":", lw=0.8)
ax.tick_params(axis="y", labelsize=6.5); ax.set_xlabel("years of signal (case A blue, B light blue, C orange; dotted = full-size line)"); ax.set_title("History each chain has under the new rules")
ax = axs[1]; x = np.arange(len(e_)); w = 0.4
ax.barh(x - w / 2, e_.ic5_val.fillna(0), height=w, color="#86b6ef", label="validation"); ax.barh(x + w / 2, e_.ic5_test.fillna(0), height=w, color=SERIES[0], label="test")
ax.set_yticks(x); ax.set_yticklabels([""] * len(e_)); ax.axvline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=8); ax.set_xlabel("rank IC of -z vs 5-day forward return"); ax.set_title("Pull-back signal by period")
ax = axs[2]
if len(FUND_POOL):
    pp = FUND_POOL.copy(); pp["lab"] = pp.group.str.slice(0, 22) + " | " + pp.period.str.slice(0, 5); pp = pp.iloc[::-1]
    ax.barh(pp.lab, pp.slope, xerr=1.96 * pp.slope.abs() / pp.t.abs().clip(lower=0.1), color=[SERIES[0] if t_ > 1.64 else MUTE for t_ in pp.t], height=0.6, ecolor=INK2)
    ax.axvline(0, color=INK2, lw=1); ax.tick_params(axis="y", labelsize=7); ax.set_xlabel("pooled slope (dark: one-sided p < 0.05)"); ax.set_title("Pooled across chains")
plt.tight_layout()
savefig(fig, "16b_fundamental_rules", "Fundamental chains 16b: history, pull-back signal, pooled test", "16")
_OUT16 = {"SKIPLOG", "FUND2", "FUND_Z2", "FUND_PNL", "FUND_BETAS", "FUND_POOL", "FUND_NOW2", "IDXb", "FWDb", "rolling_beta", "kalman_z", "ic_t16", "jo_flag", "po_p", "pooled16", "CH"}
for k_ in list(globals()):
    if k_ in _OUT16 or k_.startswith("_") or k_ == "k_": continue
    if k_ not in _G0: del globals()[k_]
    elif globals()[k_] is not _G0[k_]: globals()[k_] = _G0[k_]

## Step 17 — The relationship library and its scorecard

Every relationship from Steps 13–16 goes into one table, scored 0–100. Directional relationships (Granger, pairs, sector lead, spillover) are tagged **trade**. Tail dependence has no direction, so it is tagged **risk / hedge** and is ranked separately; it tells you which positions crash together:

| Component | Weight (`REL_W`) | Measured as |
|---|---|---|
| strength | 0.35 | percentile within its type: partial R² (Granger), −log p (cointegration), excess λ over Gaussian (tail), lift (spillover) |
| out-of-sample | 0.30 | 1 = confirmed on the blind / test period, 0.6 = validation only, 0.2 = train only |
| stability | 0.15 | share of rolling windows alive (Granger); test/train ratio of the statistic otherwise |
| survival | 0.10 | how far ahead it predicts (Granger lag decay, capped at 1 day) or 1/half-life (pairs) |
| tradability | 0.10 | liquidity percentile of the less liquid leg (median turnover) |

The **current opportunity** of each relationship comes from the latest data:
* **Fundamental chain**: the spread z-score beyond ±`PAIR_Z_IN` (rich versus fundamentals is bearish for the dependent product, cheap is bullish).
* **Granger**: the sign of the frozen cause term on the latest bars, i.e. bull or bear for the *effect*.
* **Pairs**: the spread z-score beyond ±`PAIR_Z_IN` (short the rich leg, long the cheap one).
* **Sector lead**: the sector's latest move times the frozen coefficient.
* **Spillover**: a fresh tail event in the cause.

Strength × signal gives a bull or bear score per product, which feeds the final board in Step 26.

**Used later.** The scored library feeds the opportunity board (Step 26) and the monitor (Step 22).

In [ ]:
# step 17
TURN = dp[dp["trading_day"] <= SPL["train"][-1]].groupby("product")["money"].median(); LIQ = TURN.rank(pct=True)     # training-period liquidity (no later data)
liq = lambda prods: float(np.nanmin([LIQ.get(p, np.nan) for p in prods])) if len(prods) else np.nan
rows = []
for tf, r in GRES.items():
    g = r["G_tr"][r["G_tr"].sig]; cand = set(map(tuple, r["CAND"][["cause", "effect"]].values))
    repd = set(map(tuple, r["rep"].loc[r["rep"].replicated, ["cause", "effect"]].values)) if len(r["rep"]) else set()
    sv = SURV[SURV.tf == tf].set_index(["cause", "effect"]) if len(SURV) else pd.DataFrame()
    for x in g.itertuples():
        k = (x.cause, x.effect); s_ = sv.loc[k] if len(sv) and k in sv.index else None
        rows.append(dict(type=f"granger_{tf}", a=x.cause, b=x.effect, scope="within" if SEC_MAP.get(x.cause) == SEC_MAP.get(x.effect) else "cross",
                         tf=tf, direction=np.sign(x.coef_sum), strength=x.partial_r2, p_train=x.p,
                         oos=1.0 if k in repd else (0.6 if k in cand else 0.2),
                         stability=s_.share_alive if s_ is not None else np.nan,
                         survival=min(1.0, s_.horizon_minutes / 1440) if s_ is not None else np.nan, legs=[x.cause, x.effect]))
for x in COINT[COINT.p_train < COINT_P].itertuples():
    rows.append(dict(type="coint", a=x.a, b=x.b, scope=x.scope, tf="1D", direction=0, strength=-np.log10(max(x.p_train, 1e-12)), p_train=x.p_train,
                     oos=1.0 if x.test_ok else (0.6 if x.valid else 0.2), stability=float(x.adf_p_test < 0.10) if np.isfinite(x.adf_p_test) else np.nan,
                     survival=min(1.0, 5 / x.hl_train) if np.isfinite(x.hl_train) and x.hl_train > 0 else 0, legs=[x.a, *x.b.split("+")]))
for x in TAILDEP[TAILDEP.excess_train > 0.05].itertuples():          # contagion beyond what correlation explains
    rows.append(dict(type="tail_dep", a=x.a, b=x.b, scope=x.scope, tf="1D", direction=1, strength=x.excess_train, p_train=np.nan,
                     oos=1.0 if x.excess_test > 0.05 else (0.6 if x.excess_test > 0 else 0.2),
                     stability=float(np.clip(x.excess_test / x.excess_train, 0, 1)) if x.excess_train > 0 else np.nan,
                     survival=np.nan, legs=[x.a, x.b]))
if len(TAIL_SPILL):
    for x in TAIL_SPILL[TAIL_SPILL.q < 0.05].itertuples():
        rows.append(dict(type="tail_spill", a=x.cause, b=x.effect, scope="within" if SEC_MAP.get(x.cause) == SEC_MAP.get(x.effect) else "cross",
                         tf="1D", direction=1 if x.same_sign_share >= 0.5 else -1, strength=x.lift_train, p_train=x.p_binom,
                         oos=1.0 if x.lift_test > 1.5 else 0.2, stability=min(1.0, x.lift_test / x.lift_train) if x.lift_train > 0 else np.nan,
                         survival=np.nan, legs=[x.cause, x.effect]))
if len(SECLEAD):
    for x in SECLEAD[SECLEAD.q_train < FDR_Q].itertuples():
        rows.append(dict(type="sector_lead", a=x.a, b=x.b, scope="sector", tf="1D", direction=np.sign(x.coef_train), strength=x.pr2_train, p_train=x.p_train,
                         oos=1.0 if x.test_ok else (0.6 if x.val_ok else 0.2), stability=np.nan, survival=np.nan, legs=[x.b]))
if len(FUND):          # economic chains (Step 16): same scoring as other level relationships
    for x in FUND[FUND.p_eg_train < COINT_P].itertuples():
        rows.append(dict(type="fundamental", a="+".join(x.drivers), b=x.dep, scope=x.scope, tf="1D", direction=0, strength=-np.log10(max(x.p_eg_train, 1e-12)),
                         p_train=x.p_eg_train, oos=1.0 if x.test_ok else (0.6 if x.valid else 0.2), key=x.chain,
                         stability=float(x.adf_p_test < 0.10) if np.isfinite(x.adf_p_test) else np.nan,
                         survival=min(1.0, 5 / x.hl_train) if np.isfinite(x.hl_train) and x.hl_train > 0 else 0, legs=[x.dep, *x.drivers]))
REL = pd.DataFrame(rows)
if len(REL):
    REL["strength_pct"] = REL.groupby("type").strength.rank(pct=True)
    REL["tradability"] = [liq(l) for l in REL.legs]
    comp = {"strength": REL.strength_pct, "oos": REL.oos, "stability": REL.stability.fillna(0.5),
            "survival": REL.survival.fillna(0.5), "tradability": REL.tradability.fillna(0.5)}
    REL["score"] = 100 * sum(REL_W[k] * comp[k] for k in REL_W)
    REL["use"] = np.where(REL.type == "tail_dep", "risk / hedge", "trade")      # tail dependence has no direction: it informs sizing and hedging
    REL = REL.sort_values("score", ascending=False).reset_index(drop=True); REL.index.name = "rank"
    REL.drop(columns=["legs"]).to_csv(OUT2 / "relationship_library.csv")
    print(f"Relationship library: {len(REL)} relationships | by type: {REL.type.value_counts().to_dict()}")
    cols_ = ["type", "a", "b", "scope", "direction", "strength", "oos", "stability", "survival", "tradability", "score"]
    print("\nTop TRADE relationships:\n", REL[REL.use == "trade"][cols_].head(20).round(3).to_string())
    print("\nTop RISK / HEDGE relationships (excess tail dependence):\n", REL[REL.use != "trade"][cols_].head(10).round(3).to_string())

In [ ]:
# step 17
# current opportunities implied by each relationship (latest data)
opp = []
if len(REL):
    last_z = {tf: D["Z"] for tf, D in TFDATA.items()}
    for x in REL.itertuples():
        sig, side_prod, via = 0.0, x.b, x.type
        mkey = {"granger_1D": f"gr:{x.a}→{x.b}", "coint": f"pair:{x.a}~{x.b}"}.get(x.type, f"chain:{str(x.key)[:34]}" if x.type == "fundamental" else None)
        if x.type.startswith("granger_") and x.tf in GRES:
            fm = {(m["cause"], m["effect"]): m for m in GRES[x.tf]["frozen"]["models"]}
            if (x.a, x.b) in fm:
                p_ = GRES[x.tf]["p"]; b_ = np.array(fm[(x.a, x.b)]["beta_arx"][-p_:])
                xc = last_z[x.tf][x.a].dropna().iloc[-p_:][::-1].to_numpy()
                if len(xc) == p_:
                    sig = float(b_ @ xc) / max(np.abs(b_).sum(), 1e-9)
        elif x.type == "coint":
            legs = x.b.split("+"); row_ = COINT[(COINT.a == x.a) & (COINT.b == x.b)].iloc[0]
            spread = LP[x.a] - LP[legs] @ np.array(row_.beta)
            zsp = ((spread - spread.rolling(PAIR_Z_WIN).mean()) / spread.rolling(PAIR_Z_WIN).std()).dropna()
            if len(zsp) and abs(zsp.iloc[-1]) > PAIR_Z_IN:
                sig = -np.sign(zsp.iloc[-1]) * min(abs(zsp.iloc[-1]) / PAIR_Z_IN, 3); side_prod = x.a; via = f"spread vs {x.b}"
        elif x.type == "fundamental":
            zf = FUND_Z[x.key].dropna()
            if len(zf) and abs(zf.iloc[-1]) > PAIR_Z_IN:
                sig = -np.sign(zf.iloc[-1]) * min(abs(zf.iloc[-1]) / PAIR_Z_IN, 3); via = f"fundamental ({x.key})"
        elif x.type == "sector_lead":
            peers = [c for c in UNIV if SEC_MAP[c] == SEC_MAP[x.b] and c != x.b]
            sig = float(x.direction * Z[peers].mean(axis=1).dropna().iloc[-1])
        elif x.type == "tail_spill":
            zl = STD[x.a].dropna()
            if len(zl) and abs(zl.iloc[-1]) > U_EV.get(x.a, np.inf):
                sig = float(x.direction * np.sign(zl.iloc[-1]))
        if sig != 0 and np.isfinite(sig):
            opp.append(dict(product=side_prod, side="BULL" if sig > 0 else "BEAR", rel_score=x.score, signal=sig,
                            opp=x.score * min(abs(sig), 3) / 3, mkey=mkey, via=f"{via}: {x.a}→{x.b}" if x.type not in ("coint", "fundamental") else f"{via}"))
REL_OPP = pd.DataFrame(opp)
if len(REL_OPP):
    REL_OPP = REL_OPP.sort_values("opp", ascending=False); REL_OPP.to_csv(OUT2 / "relationship_opportunities_now.csv", index=False)
    print("Current relationship-driven opportunities:\n", REL_OPP.head(20).round(2).to_string(index=False))
# the daily board only takes relationships whose signal lives at least a session: 5m / 15m Granger signals are stale within minutes
_stale = REL_OPP.via.str.contains(r"granger_(?:5m|15m)") if len(REL_OPP) else pd.Series(dtype=bool)
REL_BOARD = REL_OPP[~_stale] if len(REL_OPP) else REL_OPP
REL_BULL = REL_BOARD[REL_BOARD.side == "BULL"].groupby("product").opp.sum() if len(REL_OPP) else pd.Series(dtype=float)
REL_BEAR = REL_BOARD[REL_BOARD.side == "BEAR"].groupby("product").opp.sum() if len(REL_OPP) else pd.Series(dtype=float)

if len(REL):
    fig, axs = plt.subplots(1, 2, figsize=(16, max(4.5, 0.26 * min(len(REL), 25))), gridspec_kw={"width_ratios": [1.5, 1]})
    top = REL[REL.use == "trade"].head(25).iloc[::-1]; left = np.zeros(len(top))
    comps = {"strength": top.strength_pct, "oos": top.oos, "stability": top.stability.fillna(0.5), "survival": top.survival.fillna(0.5),
             "tradability": top.tradability.fillna(0.5)}
    for k, (name, v) in enumerate(comps.items()):
        val = 100 * REL_W[name] * v.values
        axs[0].barh(range(len(top)), val, left=left, color=SERIES[k], height=0.65, label=name, edgecolor="#fcfcfb", linewidth=1); left += val
    axs[0].set_yticks(range(len(top))); axs[0].set_yticklabels([f"[{t}] {a}→{b}" for t, a, b in zip(top.type, top.a, top.b)], fontsize=7)
    axs[0].legend(frameon=False, fontsize=8, ncol=5, loc="lower right", bbox_to_anchor=(1, 1)); axs[0].set_xlabel("score (0–100)")
    axs[0].set_title("Top tradable relationships and what their score is made of")
    ct = pd.crosstab(REL.type, REL.scope)
    heat(axs[1], ct, label="count", annot=True, fmt="{:.0f}"); axs[1].set_title("Library by type and scope")
    plt.tight_layout()
    savefig(fig, "17_relationship_scorecard", "Relationship library scorecard", "17")

---
# Part D — Factors and opportunity ranks (Steps 18–19)

## Step 18 — Factor library (López de Prado principles)

**Factor families** (daily), each computed from information up to the close of day t:

| Family | Base factors |
|---|---|
| Trend | price vs 50-day EMA (vol-scaled), 60-day slope t-stat, signed efficiency ratio |
| Momentum | 20 / 60-day vol-scaled return, 120-day skipping the last 5, acceleration (20 − 60) |
| Volatility | low-vol level (−z of 20-day RV vs its 250-day history), vol contraction (−log RV₅/RV₆₀), up/down semivariance skew |
| Mean reversion | −z of price vs 20-day mean, 5-day reversal, −z of the **fractionally differenced** price (Step 9.2), variance ratio |
| Volume / OI | volume z-score, price–volume confirmation, OI-confirmed trend |
| Carry | annual roll yield from the calendar-spread jump at each roll (contango = negative carry for a long) and its 60-day change; **term-structure slope**: the same jump divided by the months between the two contracts' expiries (annualised, comparable across products that roll at different intervals), and the **steepening** between the last two rolls. Only the main contract is in the data, so the slope has one point per roll; the full curve needs the non-main files |
| Breadth | share of the market and of the sector in an uptrend |

**Four levels** for every base factor:
* **TS** (the product against its own 250-day history);
* **XS** (cross-sectional rank across all products);
* **SEC** (the sector's average ranked across sectors, i.e. sector rotation);
* **WS** (rank *within* the sector, i.e. relative value against peers).

**Evaluation, following López de Prado (AFML / MLAM):**
1. **IC and IC decay** against forward vol-scaled returns at 1, 5 and 20 days (train), with validation confirmation and yearly sign stability.
2. **Event sampling** with a symmetric **CUSUM filter**, so samples are not dominated by quiet days and overlapping labels.
3. **Triple-barrier labels** (±`TB_K`·σ·√H, vertical barrier `TB_H` days), weighted by **average uniqueness**.
4. A **random forest** with **purged k-fold CV and an embargo**, so no leakage through overlapping labels.
5. **Clustered MDA** importance: features are clustered by correlation and permuted together, which avoids substitution effects. Also **MDI**, and **SFI** (single-feature importance) for the top features.
6. An **opportunity rank** per feature: |IC t| + validation confirmation + stability + MDA.

**Used later.** Orientations and scores feed the ranks of Step 19, the dynamics of Step 21, the B strategies of Step 23, and the family groups of Step 18b.

In [ ]:
# step 18
ROLLGAP = wide("roll_gap").reindex_like(RAW) if "roll_gap" in dp.columns else None
if ROLLGAP is None or not ROLLGAP.notna().any().any():
    print("no roll_gap column in the cached daily panel (cache written by an older Step 1): carry family skipped; delete eda_cache/daily_panel.parquet and re-run Step 1 to add it")
def _expiry(sym, day):
    # contract code -> expiry month (first day). 4 digits = YYMM (SHFE / DCE / CFFEX / INE); 3 digits = YMM (older CZCE codes): choose the decade closest to the trading day
    m = re.search(r"(\d{3,4})$", str(sym))
    if not m:
        return pd.NaT
    s_ = m.group(1)
    if len(s_) == 4:
        return pd.Timestamp(2000 + int(s_[:2]), int(s_[2:]), 1) if 1 <= int(s_[2:]) <= 12 else pd.NaT
    mm = int(s_[1:])
    if not 1 <= mm <= 12:
        return pd.NaT
    cands = [pd.Timestamp(day.year // 10 * 10 + k * 10 + int(s_[0]), mm, 1) for k in (-1, 0, 1)]
    ok = [c for c in cands if -45 <= (c - day).days <= 900]
    return min(ok, key=lambda c: abs((c - day).days)) if ok else pd.NaT
def build_slope(dp_, cols, idx):
    # annualised calendar-spread slope at each roll: (new - old contract) / months between the two expiries * 12; known on the roll day
    d = dp_.sort_values(["product", "trading_day"])
    out = pd.DataFrame(np.nan, index=idx, columns=cols)
    for p, g in d.groupby("product"):
        if p not in out.columns or "roll_gap" not in g:
            continue
        prev = g["symbol"].shift(1)
        rr = g.loc[g["roll"].astype(bool)].assign(ps=prev)
        for day, sym, ps, gap in zip(rr["trading_day"], rr["symbol"], rr["ps"], rr["roll_gap"]):
            e1, e0 = _expiry(sym, day), _expiry(ps, day)
            if pd.isna(e1) or pd.isna(e0) or pd.isna(gap) or day not in out.index:
                continue
            months = (e1.year - e0.year) * 12 + (e1.month - e0.month)
            if months >= 1:
                out.at[day, p] = gap * 12.0 / months
    return out
SLOPE = build_slope(dp, RAW.columns, RAW.index) if ROLLGAP is not None else None
if SLOPE is not None:
    print(f"term-structure slope: {int(SLOPE.notna().sum().sum())} roll events with a parseable pair of contract codes, {int((SLOPE.notna().sum() > 0).sum())} products" +
          ("" if SLOPE.notna().any().any() else " — contract codes could not be parsed: slope carry skipped"))
def make_base_factors(RAW_, LP_, RV_, VOLU_, OIW_, FD_, ROLLGAP_, SLOPE_=None):
    # every base factor from information up to the close of day t only (the point-in-time audit below re-runs this on truncated data)
    SD_D = RAW_.ewm(halflife=VOL_HL, min_periods=20).std()
    ema = lambda x, n: x.ewm(span=n, min_periods=n // 2).mean()
    lrv = np.log((RV_ ** 2).where(RV_ > 0))
    B, BFAM = {}, {}
    def addb(name, fam, val):
        B[name] = val.replace([np.inf, -np.inf], np.nan).astype("float32"); BFAM[name] = fam
    addb("trend_ema50", "trend", (LP_ - ema(LP_, 50)) / (SD_D * np.sqrt(50)))
    addb("trend_tstat60", "trend", rolling_tstat(LP_, 60))
    chg = LP_ - LP_.shift(20); addb("trend_er20", "trend", chg / RAW_.abs().rolling(20).sum())
    addb("mom_20", "momentum", RAW_.rolling(20).sum() / (SD_D * np.sqrt(20)))
    addb("mom_60", "momentum", RAW_.rolling(60).sum() / (SD_D * np.sqrt(60)))
    addb("mom_120_5", "momentum", (RAW_.rolling(120).sum() - RAW_.rolling(5).sum()) / (SD_D * np.sqrt(115)))
    addb("mom_accel", "momentum", B["mom_20"] - B["mom_60"])
    lr20 = lrv.rolling(20, min_periods=10).mean()
    addb("vol_low", "volatility", -(lr20 - lr20.rolling(250, min_periods=60).mean()) / lr20.rolling(250, min_periods=60).std())
    addb("vol_contract", "volatility", -(lrv.rolling(5).mean() - lrv.rolling(60).mean()))
    up_, dn_ = (RAW_.clip(lower=0) ** 2).rolling(60).sum(), (RAW_.clip(upper=0) ** 2).rolling(60).sum()
    addb("vol_upskew", "volatility", (up_ - dn_) / (up_ + dn_))
    addb("mr_z20", "mean_reversion", -(LP_ - LP_.rolling(20).mean()) / (SD_D * np.sqrt(20)))
    addb("mr_rev5", "mean_reversion", -RAW_.rolling(5).sum() / (SD_D * np.sqrt(5)))
    if len(FD_.columns):
        addb("mr_fracdiff", "mean_reversion", (-(FD_ - FD_.rolling(60).mean()) / FD_.rolling(60).std()).reindex(columns=RAW_.columns))
    addb("mr_varratio", "mean_reversion", -((RAW_.rolling(5).sum()).rolling(120).var() / (5 * RAW_.rolling(120).var()) - 1))
    lv = np.log1p(VOLU_)
    addb("volu_z", "volume_oi", (lv - lv.rolling(60).mean()) / lv.rolling(60).std())
    addb("pv_confirm", "volume_oi", np.sign(RAW_.rolling(5).sum()) * B["volu_z"].rolling(5).mean())
    addb("oi_trend", "volume_oi", np.log(OIW_.where(OIW_ > 0)).diff(20) * np.sign(RAW_.rolling(20).sum()))
    if ROLLGAP_ is not None and ROLLGAP_.notna().any().any():
        ryr = ROLLGAP_.fillna(0).rolling(250, min_periods=250).sum()
        addb("carry_roll", "carry", -ryr.where(RAW_.notna().rolling(250).sum() >= 200))
        addb("carry_chg", "carry", -(ryr - ryr.shift(60)))
    if SLOPE_ is not None and SLOPE_.notna().any().any():
        sl = SLOPE_.ffill(limit=130)                                     # latest roll's annualised slope, valid for ~half a year (point in time: known since the roll day)
        addb("carry_slope", "carry", -sl)                                # backwardation (negative slope) = positive carry for a long
        last2 = SLOPE_.apply(lambda c: c.dropna().rolling(2).apply(lambda v: v[1] - v[0], raw=True).reindex(c.index))
        addb("carry_steepen", "carry", last2.ffill(limit=130))           # curve flattening / steepening between the last two rolls: a rise in slope = curve steepening into contango
    upt = (B["trend_ema50"] > 0).astype(float).where(B["trend_ema50"].notna())
    addb("breadth_mkt", "breadth", pd.DataFrame(np.repeat(upt.mean(axis=1).to_numpy()[:, None], len(RAW_.columns), 1), index=RAW_.index, columns=RAW_.columns) - 0.5)
    addb("breadth_sec", "breadth", upt.T.groupby(SEC_MAP).transform("mean").T - 0.5)
    return B, BFAM, SD_D
B, BFAM, SD_D = make_base_factors(RAW, LP, RV, VOLU, OIW, FD, ROLLGAP, SLOPE)

sec_n = SEC_MAP.value_counts()
def build_levels(B_):
    # every level transform is trailing (TS z-score) or a same-date cross-section (XS / WS / SEC ranks)
    FEATS, FMETA = {}, []
    for name, F_ in B_.items():
        fam = BFAM[name]
        lv_ = {"TS": (F_ - F_.rolling(TS_Z_WIN, min_periods=60).mean()) / F_.rolling(TS_Z_WIN, min_periods=60).std()}
        if fam != "breadth":
            lv_["XS"] = F_.rank(axis=1, pct=True) - 0.5
            ok_cols = [c for c in F_.columns if sec_n[SEC_MAP[c]] >= MIN_SEC_WS]
            lv_["WS"] = (F_.T.groupby(SEC_MAP).rank(pct=True).T - 0.5)[ok_cols].reindex(columns=F_.columns) if ok_cols else None
            secm = F_.T.groupby(SEC_MAP).mean().T
            if secm.shape[1] >= 3:
                sr = secm.rank(axis=1, pct=True) - 0.5
                lv_["SEC"] = sr.reindex(columns=SEC_MAP.reindex(F_.columns).values).set_axis(F_.columns, axis=1).where(F_.notna())
        else:
            lv_["XS"] = F_
        for lev, X_ in lv_.items():
            if X_ is None:
                continue
            FEATS[f"{name}|{lev}"] = X_.astype("float32"); FMETA.append(dict(feature=f"{name}|{lev}", base=name, family=fam, level=lev))
    return FEATS, pd.DataFrame(FMETA).set_index("feature")
FEATS, FMETA = build_levels(B)
causal_test("Step 18 feature levels (TS / XS / WS / SEC) of 8 base factors", lambda B_: build_levels(B_)[0], {k: B[k] for k in list(B)[:8]}, step="18")
FWD = {h: (RAW.rolling(h, min_periods=h).sum().shift(-h) / (SD_D * np.sqrt(h))) for h in FWD_H}
print(f"{len(B)} base factors × levels → {len(FEATS)} features")

# ---- point-in-time audit (look-ahead test): rebuild every base factor on data cut at an earlier date; values up to the cut must not change ----
_cuts = [RAW.index[int(len(RAW) * cf)] for cf in (0.45, 0.75)]
causal_test("Step 18 base factors (all families, incl. carry and slope)", make_base_factors, RAW, LP, RV, VOLU, OIW, FD, ROLLGAP, SLOPE, cuts=_cuts, step="18")
pit_rows = [dict(cut=r["cut"].date(), factor=r["output"], max_abs_diff=r["max_abs_diff"], availability_mismatch=0 if r["passed"] else 1) for r in CAUSAL_LOG if r["test"].startswith("Step 18 base factors")]
PIT = pd.DataFrame(pit_rows)
assert len(PIT) and PIT.max_abs_diff.max() < 1e-4 and PIT.availability_mismatch.max() == 0, PIT[(PIT.max_abs_diff >= 1e-4) | (PIT.availability_mismatch > 0)]
c0, p0_ = RAW.columns[0], 100
assert np.isclose(FWD[5].iloc[p0_][c0] * SD_D.iloc[p0_][c0] * np.sqrt(5), RAW.iloc[p0_ + 1:p0_ + 6][c0].sum(), equal_nan=True), "forward return must start the day after the signal"
print(f"point-in-time audit passed: {len(B)} base factors × 2 cut dates unchanged by later data (max |diff| {PIT.max_abs_diff.max():.1e}); forward returns start at t+1; "
      f"embargo between periods = {EMB} days (longest forward horizon {max(FWD_H)})")

In [ ]:
# step 18
idx_tr, idx_va, idx_te = (RAW.index.intersection(SPL[k]) for k in ("train", "val", "test"))
fs_rows, ICSER = [], {}
t0 = time.time()
for f, X_ in FEATS.items():
    row = dict(feature=f)
    for h in FWD_H:
        ic = xs_ic_fast(X_.loc[idx_tr], FWD[h].loc[idx_tr]) if FMETA.loc[f, "family"] != "breadth" else pd.Series(dtype=float)
        if len(ic) < 30:      # no cross-sectional variation (breadth): pooled time-series IC
            row[f"ic{h}"] = ts_ic_pooled(X_.loc[idx_tr], FWD[h].loc[idx_tr]); row[f"t{h}"] = row[f"ic{h}"] * np.sqrt(len(idx_tr) / h)
        else:
            row[f"ic{h}"] = ic.mean(); row[f"t{h}"] = ic_t(ic, h)
            if h == 5:
                ICSER[f] = xs_ic_fast(X_, FWD[5]); yrs = ic.groupby(ic.index.year).mean()
                row["stable"] = float((np.sign(yrs) == np.sign(ic.mean())).mean())
    for sp, idx in (("val", idx_va), ("test", idx_te)):
        ic = xs_ic_fast(X_.loc[idx], FWD[5].loc[idx])
        row[f"{sp}_ic5"] = ic.mean() if len(ic) >= 30 else ts_ic_pooled(X_.loc[idx], FWD[5].loc[idx]); row[f"{sp}_t5"] = ic_t(ic, 5) if len(ic) >= 30 else np.nan
    row["ts_ic5"] = ts_ic_pooled(X_.loc[idx_tr], FWD[5].loc[idx_tr])
    fs_rows.append(row)
FSCORE = pd.DataFrame(fs_rows).set_index("feature").join(FMETA)
FSCORE["orient"] = pd.Series(np.where(FSCORE.level == "TS", np.sign(FSCORE.ts_ic5), np.sign(FSCORE.ic5)), index=FSCORE.index).replace(0, 1).fillna(1)
FSCORE["val_confirm"] = np.where(np.sign(FSCORE.val_ic5) == FSCORE.orient, np.where(FSCORE.val_t5.abs() > 1, 1.0, 0.5), 0.0)
print(f"IC table in {time.time()-t0:.0f}s")

In [ ]:
# step 18
# López de Prado pipeline: CUSUM events → triple-barrier labels → uniqueness weights → purged CV random forest
ev = []
pos_of = {d: i for i, d in enumerate(RAW.index)}
for j, s in enumerate(UNIV):
    lp = LP[s]; sd = SD_D[s]
    e_idx = cusum_events(lp, CUSUM_K * sd)
    e_idx = e_idx[np.isfinite(sd.to_numpy()[e_idx])] if len(e_idx) else e_idx
    width = TB_K * sd.to_numpy()[e_idx] * np.sqrt(TB_H)
    lab, t1, r_ = triple_barrier(lp.to_numpy(), e_idx, width, TB_H)
    ok = np.isfinite(lab) & (lab != 0)
    u = avg_uniqueness(e_idx[ok], t1[ok])
    ev.append(pd.DataFrame({"t0": e_idx[ok], "t1": t1[ok], "product": s, "label": lab[ok], "ret": r_[ok], "w": u}))
EV = pd.concat(ev, ignore_index=True)
EV["date"] = RAW.index[EV.t0]; EV["date1"] = RAW.index[np.minimum(EV.t1.astype(int).values, len(RAW) - 1)]
# purge: an event belongs to a period only if its whole label window [t0, t1] lies inside it (no label peeks into the next period)
EV["period"] = np.select([EV.date1 <= SPL["train"][-1], (EV.date >= SPL["val"][0]) & (EV.date1 <= SPL["val"][-1]), EV.date >= SPL["test"][0]],
                         ["train", "val", "test"], "gap")
fnames = list(FEATS)
colpos = {c: i for i, c in enumerate(RAW.columns)}
X_all = np.column_stack([FEATS[f].to_numpy()[EV.t0.values, EV["product"].map(colpos).values] for f in fnames]).astype(float)
X_all = np.nan_to_num(X_all, nan=0.0, posinf=0.0, neginf=0.0)
y_all = (EV.label.values > 0).astype(int)
tr = np.flatnonzero(EV.period.values == "train")
if len(tr) > MAX_EVENTS:
    tr = np.sort(rng.choice(tr, MAX_EVENTS, replace=False))
print(f"CUSUM events: {len(EV)} ({EV.period.value_counts().to_dict()}; 'gap' = label window crosses a split, purged), train used: {len(tr)} | "
      f"up-label share {y_all[tr].mean():.3f} | mean uniqueness {EV.w.values[tr].mean():.2f}")
Xtr, ytr, wtr = X_all[tr], y_all[tr], EV.w.values[tr]
CL = cluster_features(Xtr); clusters = {k: [fnames[i] for i in np.flatnonzero(CL == k)] for k in sorted(set(CL))}
rf_kw = dict(n_estimators=RF_TREES, max_features=1, min_samples_leaf=50, class_weight="balanced_subsample",
             max_samples=float(np.clip(wtr.mean(), 0.05, 1.0)), n_jobs=-1, random_state=SEED)
folds = purged_kfold(EV.t0.values[tr], EV.t1.values[tr])
cv_auc, mda = [], {k: [] for k in clusters}
for trn, tst in folds:
    if len(set(ytr[trn])) < 2 or len(set(ytr[tst])) < 2:
        continue
    m_ = RandomForestClassifier(**rf_kw).fit(Xtr[trn], ytr[trn], sample_weight=wtr[trn])
    pr = m_.predict_proba(Xtr[tst])[:, 1]; base = log_loss(ytr[tst], pr, sample_weight=wtr[tst], labels=[0, 1])
    cv_auc.append(roc_auc_score(ytr[tst], pr, sample_weight=wtr[tst]))
    for k, fl in clusters.items():
        Xp = Xtr[tst].copy(); ii = [fnames.index(f) for f in fl]; Xp[:, ii] = Xp[rng.permutation(len(Xp))][:, ii]
        mda[k].append(log_loss(ytr[tst], m_.predict_proba(Xp)[:, 1], sample_weight=wtr[tst], labels=[0, 1]) - base)
full_rf = RandomForestClassifier(**rf_kw).fit(Xtr, ytr, sample_weight=wtr)
oos_auc = {}
for sp in ("val", "test"):
    ii = np.flatnonzero(EV.period.values == sp)
    if len(ii) > 50 and len(set(y_all[ii])) == 2:
        oos_auc[sp] = roc_auc_score(y_all[ii], full_rf.predict_proba(X_all[ii])[:, 1])
print(f"purged {N_FOLDS}-fold CV AUC: {np.mean(cv_auc):.3f} ± {np.std(cv_auc):.3f} | out-of-sample AUC: {oos_auc}  (0.5 = no skill)")
MDA_C = pd.DataFrame({"mda_mean": {k: np.mean(v) for k, v in mda.items()}, "mda_std": {k: np.std(v) for k, v in mda.items()},
                      "features": {k: ", ".join(v[:6]) + (" …" if len(v) > 6 else "") for k, v in clusters.items()}}).sort_values("mda_mean", ascending=False)
FSCORE["cluster"] = [int(CL[fnames.index(f)]) for f in FSCORE.index]
FSCORE["mda_cluster"] = FSCORE.cluster.map(MDA_C.mda_mean)
FSCORE["mdi"] = pd.Series(full_rf.feature_importances_, index=fnames)
# single-feature importance for the top features by |t|
sfi = {}
for f in FSCORE.t5.abs().sort_values(ascending=False).index[:25]:
    j = fnames.index(f); sc = []
    for trn, tst in folds:
        if len(set(ytr[trn])) < 2 or len(set(ytr[tst])) < 2:
            continue
        m1 = RandomForestClassifier(n_estimators=50, max_depth=3, min_samples_leaf=100, n_jobs=-1, random_state=SEED).fit(Xtr[trn][:, [j]], ytr[trn], sample_weight=wtr[trn])
        sc.append(roc_auc_score(ytr[tst], m1.predict_proba(Xtr[tst][:, [j]])[:, 1], sample_weight=wtr[tst]))
    sfi[f] = np.mean(sc) if sc else np.nan
FSCORE["sfi_auc"] = pd.Series(sfi)
pct = lambda s: s.rank(pct=True).fillna(0.5)
FSCORE["opportunity"] = 100 * (0.40 * pct(FSCORE.t5.abs()) + 0.30 * FSCORE.val_confirm + 0.15 * FSCORE.stable.fillna(0.5) + 0.15 * pct(FSCORE.mda_cluster))
FSCORE = FSCORE.sort_values("opportunity", ascending=False); FSCORE.to_csv(OUT2 / "factor_scorecard.csv")
print("\nClustered MDA (log-loss increase when the cluster is permuted; > 0 = useful):\n", MDA_C.round(5).to_string())
print("\nFactor opportunity ranking (top 25):")
print(FSCORE[["family", "level", "ic1", "ic5", "ic20", "t5", "val_ic5", "test_ic5", "stable", "val_confirm", "mda_cluster", "sfi_auc", "opportunity"]].head(25).round(4).to_string())
fam_lvl = FSCORE.pivot_table(index="family", columns="level", values="t5", aggfunc=lambda v: v.abs().max())
print("\nStrongest |t| (5-day IC, train) by family × level:\n", fam_lvl.round(2).to_string())

In [ ]:
# step 18
fig, axs = plt.subplots(1, 3, figsize=(18, 4.8))
signed = FSCORE.pivot_table(index="family", columns="level", values="t5", aggfunc=lambda v: v.iloc[np.argmax(np.abs(v.values))])
heat(axs[0], signed, cmap=DIV, center=0, label="best IC t-stat (5d, train)", annot=True, fmt="{:.1f}")
axs[0].set_title("Where the opportunity is: family × level")
ax = axs[1]
for k, fam in enumerate(sorted(FSCORE.family.unique())):
    d_ = FSCORE[FSCORE.family == fam]
    ax.plot(FWD_H, [d_[f"ic{h}"].abs().mean() for h in FWD_H], marker="o", color=SERIES[k % 8], label=fam)
ax.set_xscale("log"); ax.set_xticks(FWD_H); ax.set_xticklabels(FWD_H); ax.set_xlabel("horizon (days)"); ax.set_ylabel("mean |IC|")
ax.legend(frameon=False, fontsize=8); ax.set_title("IC decay by family (train)")
ax = axs[2]; mm = MDA_C.iloc[::-1]
ax.barh(range(len(mm)), mm.mda_mean, xerr=mm.mda_std, color=np.where(mm.mda_mean > 0, SERIES[0], MUTE), height=0.6, ecolor=INK2)
ax.set_yticks(range(len(mm))); ax.set_yticklabels([f"C{k}: {t[:38]}" for k, t in zip(mm.index, mm.features)], fontsize=6)
ax.axvline(0, color=INK2, lw=1); ax.set_xlabel("clustered MDA (log-loss increase)")
ax.set_title(f"Feature-cluster importance (purged CV AUC {np.mean(cv_auc):.3f})")
plt.tight_layout()
savefig(fig, "18_factor_library", "Factor library: family × level, IC decay, clustered MDA", "18")

# current factor opportunity per product: IC-weighted sum of oriented, validated XS/WS/SEC/TS features
SEL_F = FSCORE[(FSCORE.val_confirm > 0) & (FSCORE.t5.abs() > 2)].index.tolist() or FSCORE.index[:10].tolist()
wF = (FSCORE.loc[SEL_F, "t5"].abs() * FSCORE.loc[SEL_F, "orient"])
FACTOR_SCORE = sum(FEATS[f].rank(axis=1, pct=True).sub(0.5) * wF[f] for f in SEL_F) / wF.abs().sum()
FACTOR_NOW = FACTOR_SCORE.ffill().iloc[-1].rank(pct=True)
print(f"multi-factor score uses {len(SEL_F)} validated features: {SEL_F[:12]}{' …' if len(SEL_F) > 12 else ''}")

## Step 18b — Factor families together: pairs, triplets, quadruplets

Step 18 scored each feature on its own. Factors are never traded alone for long, so a researcher asks **how the families relate** and whether combining them adds anything beyond the best member:

| Question | Measure |
|---|---|
| **Are two families the same bet?** | average cross-sectional rank correlation of the two composite signals (train) |
| **Does B add information beyond A?** | IC of A *after removing its cross-sectional dependence on B* (per date), and the reverse |
| **Is the effect conditional?** | IC of A inside the top and the bottom tercile of B (an interaction: A works only when B is high or low) |
| **Do groups of 2, 3, 4 families beat the best member?** | equal-weight composite of the oriented family signals; IC at 5 days in train, validation and test; **synergy** = IC(group) minus the best IC among its sub-groups one size smaller |

A family composite at a level (`XS` cross-section, `WS` within sector) is the mean of its oriented features (Step 18 orientation, fixed on train). `breadth` has no cross-sectional variation and is excluded. With `n` families there are $\binom{n}{2}+\binom{n}{3}+\binom{n}{4}$ groups per level, so the table is read with the multiple-testing count in mind: **a group counts only if train, validation and test agree**.

**Used later.** The best group confirmed on validation becomes the strategy `B family combo` (Step 23). Redundant pairs (correlation above `COMBO_REDUNDANT`) are merged in the multi-factor score so that the same bet is not counted twice. The conditional-IC table suggests regimes in which a family should be switched off (Step 22).

In [ ]:
# step 18b
# ---- 18b.1 family composites by level, pairwise relationships ----
LV_COMBO = [l for l in COMBO_LEVELS]
FAMS = [f for f in sorted(FSCORE.family.unique()) if f != "breadth" and (FSCORE.family == f).sum() >= 1]
idx_tr, idx_va, idx_te = (RAW.index.intersection(SPL[k]) for k in ("train", "val", "test"))
def make_famc(FE):
    # family composite = mean of its features times the orientation fixed on the training period (no later data enter the weights)
    out = {}
    for lv in LV_COMBO:
        for f in FAMS:
            fl = [x for x in FSCORE[(FSCORE.family == f) & (FSCORE.level == lv)].index if x in FE]
            if len(fl):
                out[(f, lv)] = sum(FE[x].astype(float) * FSCORE.loc[x, "orient"] for x in fl) / len(fl)
    return out
FAMC = make_famc(FEATS)
causal_test("Step 18b family composites", lambda FE: {f"{k[0]}|{k[1]}": v for k, v in make_famc(FE).items()}, {k: FEATS[k] for k in FSCORE.index if k in FEATS}, step="18b")
F5 = FWD[5]

def mean_rank_corr(A, B, ix):
    ic = xs_ic_fast(A.loc[ix], B.loc[ix]); return ic.mean() if len(ic) else np.nan
def residualise(A, B):
    # per date: remove the part of A explained by B (cross-sectional regression through the origin on centred ranks)
    Ar = A.rank(axis=1, pct=True) - 0.5; Br = B.rank(axis=1, pct=True) - 0.5; m = Ar.notna() & Br.notna()
    beta = (Ar.where(m) * Br.where(m)).sum(axis=1) / (Br.where(m) ** 2).sum(axis=1)
    return (Ar - Br.mul(beta, axis=0)).where(m)
def cond_ic(A, B, ix):
    Br = B.rank(axis=1, pct=True); hi = Br > 2 / 3; lo = Br <= 1 / 3
    return (xs_ic_fast(A.loc[ix].where(hi.loc[ix]), F5.loc[ix], min_n=5).mean(), xs_ic_fast(A.loc[ix].where(lo.loc[ix]), F5.loc[ix], min_n=5).mean())
pair_rows = []
for lv in LV_COMBO:
    for a, b in itertools.permutations(FAMS, 2):
        if (a, lv) not in FAMC or (b, lv) not in FAMC:
            continue
        fa_, fb_ = FAMC[(a, lv)], FAMC[(b, lv)]
        r = dict(level=lv, A=a, B=b, signal_corr_train=mean_rank_corr(fa_, fb_, idx_tr), ic_A_train=xs_ic_fast(fa_.loc[idx_tr], F5.loc[idx_tr]).mean())
        Ares = residualise(fa_, fb_)
        for sp, ix in (("train", idx_tr), ("val", idx_va), ("test", idx_te)):
            ic_r = xs_ic_fast(Ares.loc[ix], F5.loc[ix]); r[f"ic_A_given_B_{sp}"] = ic_r.mean(); r[f"t_A_given_B_{sp}"] = ic_t(ic_r, 5)
        r["ic_A_when_B_high"], r["ic_A_when_B_low"] = cond_ic(fa_, fb_, idx_tr)
        pair_rows.append(r)
PAIRS = pd.DataFrame(pair_rows)
PAIRS["adds_beyond"] = (PAIRS.ic_A_given_B_train * PAIRS.ic_A_train > 0) & (np.sign(PAIRS.ic_A_given_B_val) == np.sign(PAIRS.ic_A_given_B_train)) & (np.sign(PAIRS.ic_A_given_B_test) == np.sign(PAIRS.ic_A_given_B_train)) & (PAIRS.t_A_given_B_train.abs() > 1.5)
PAIRS["interaction"] = PAIRS.ic_A_when_B_high - PAIRS.ic_A_when_B_low
PAIRS.round(4).to_csv(OUT2 / "factor_family_pairs.csv", index=False)
xs = PAIRS[PAIRS.level == LV_COMBO[0]]
print(f"{len(FAMS)} families: {', '.join(FAMS)} | levels: {LV_COMBO}")
print(f"\nSignal correlation between families ({LV_COMBO[0]}, train; near 0 = different bets, high = same bet):")
M_ = xs.pivot(index="A", columns="B", values="signal_corr_train").reindex(index=FAMS, columns=FAMS); M_ = pd.DataFrame(np.where(np.eye(len(FAMS), dtype=bool), 1.0, M_.to_numpy()), index=FAMS, columns=FAMS); print(M_.round(2).to_string())
print(f"\nDoes A add information beyond B? ({LV_COMBO[0]}; IC of A after removing B, same sign in train / validation / test and |t|>1.5 on train)")
print(xs[xs.adds_beyond].sort_values("t_A_given_B_train", key=abs, ascending=False)[["A", "B", "ic_A_train", "ic_A_given_B_train", "ic_A_given_B_val", "ic_A_given_B_test", "t_A_given_B_train"]].round(4).to_string(index=False) if xs.adds_beyond.any() else "none")
print(f"\nConditional effects: largest |IC(A | B high) − IC(A | B low)| ({LV_COMBO[0]}, train):")
print(xs.reindex(xs.interaction.abs().sort_values(ascending=False).index).head(6)[["A", "B", "ic_A_when_B_high", "ic_A_when_B_low", "interaction"]].round(4).to_string(index=False))

In [ ]:
# step 18b
# ---- 18b.2 groups of 2, 3, 4 families ----
def comp_of(keys, lv):
    return sum(FAMC[(f, lv)].rank(axis=1, pct=True) - 0.5 for f in keys) / len(keys)
def ic_sp(C, h=5):
    out = {}
    for sp, ix in (("train", idx_tr), ("val", idx_va), ("test", idx_te)):
        ic = xs_ic_fast(C.loc[ix], FWD[h].loc[ix]); out[sp] = (ic.mean(), ic_t(ic, h))
    return out
G_ROWS, CACHE_IC = [], {}
for lv in LV_COMBO:
    fl_ = [f for f in FAMS if (f, lv) in FAMC]
    for size in range(1, min(COMBO_MAX, len(fl_)) + 1):
        for keys in itertools.combinations(fl_, size):
            C = comp_of(keys, lv); r = ic_sp(C)
            row = dict(level=lv, size=size, families="+".join(keys), ic_train=r["train"][0], t_train=r["train"][1], ic_val=r["val"][0], ic_test=r["test"][0], t_test=r["test"][1])
            CACHE_IC[(lv, keys)] = r["train"][0]
            if size > 1:
                subs = [CACHE_IC[(lv, tuple(s))] for s in itertools.combinations(keys, size - 1)]
                sing = [CACHE_IC[(lv, (k,))] for k in keys]
                row["best_sub_ic_train"] = max(subs, key=abs); row["synergy_train"] = abs(r["train"][0]) - max(abs(x) for x in subs)
                cm = [mean_rank_corr(FAMC[(a, lv)], FAMC[(b, lv)], idx_tr) for a, b in itertools.combinations(keys, 2)]; row["mean_member_corr"] = float(np.nanmean(cm))
            G_ROWS.append(row)
GR = pd.DataFrame(G_ROWS)
GR["agree"] = (np.sign(GR.ic_train) == np.sign(GR.ic_val)) & (np.sign(GR.ic_val) == np.sign(GR.ic_test)) & (GR.t_train.abs() > 2)
GR.round(4).to_csv(OUT2 / "factor_family_groups.csv", index=False)
print(f"{len(GR)} groups tested ({GR.groupby('level').size().to_dict()}); agree in train, validation and test with |t_train|>2: {int(GR.agree.sum())} "
      f"(expected by chance if signs were random: about {len(GR) * 0.25 * 0.1:.0f})")
for sz in range(1, COMBO_MAX + 1):
    g = GR[(GR["size"] == sz)].copy()
    g["score"] = np.where(g.agree, g.ic_train.abs(), 0) + 0.0 * g.ic_train.abs()
    print(f"\nbest groups of {sz} (by |IC| in train among those that agree out of sample):")
    cols = ["level", "families", "ic_train", "t_train", "ic_val", "ic_test"] + (["synergy_train", "mean_member_corr"] if sz > 1 else [])
    gg = g[g.agree].sort_values("ic_train", key=abs, ascending=False).head(4)
    print(gg[cols].round(4).to_string(index=False) if len(gg) else "none agree out of sample")
print("\nAverage synergy (|IC| of the group minus the best sub-group), train:", GR[GR["size"] > 1].groupby("size").synergy_train.mean().round(4).to_dict(),
      "| share of groups that beat their best sub-group:", GR[GR["size"] > 1].groupby("size").apply(lambda g: float((g.synergy_train > 0).mean())).round(2).to_dict())
# the best validated group → composite score used by strategy 'B family combo'
cand = GR[GR.agree & (GR["size"] >= 2)].copy()
if len(cand):
    cand["rank_key"] = cand.ic_val.abs()                                                         # selection on validation, test is confirmation only
    best = cand.sort_values("rank_key", ascending=False).iloc[0]
    COMBO_BEST = (best.level, tuple(best.families.split("+")))
    COMBO_BEST_SCORE = comp_of(COMBO_BEST[1], COMBO_BEST[0]) * np.sign(best.ic_train)
    print(f"\nbest validated group: {best.families} at level {best.level} (train IC {best.ic_train:+.4f}, validation {best.ic_val:+.4f}, test {best.ic_test:+.4f}) → strategy 'B family combo' in Step 23")
else:
    COMBO_BEST, COMBO_BEST_SCORE = None, None
    print("\nno group of 2+ families agrees in train, validation and test: no combo strategy is built")

In [ ]:
# step 18b
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
ax = axs[0]; heat(ax, M_, cmap=DIV, center=0, vmax=1, label="signal correlation", annot=True, fmt="{:.2f}"); ax.set_title(f"How alike are the families? ({LV_COMBO[0]}, train)")
ax = axs[1]; piv = xs.pivot(index="A", columns="B", values="ic_A_given_B_train").reindex(index=FAMS, columns=FAMS)
heat(ax, piv, cmap=DIV, center=0, label="IC of A after removing B", annot=True, fmt="{:.3f}"); ax.set_title("Does A add information beyond B? (rows A, columns B)")
ax = axs[2]
for k, sz in enumerate(range(1, COMBO_MAX + 1)):
    g = GR[GR["size"] == sz]; ax.scatter(g.ic_train, g.ic_test, s=16 + 6 * sz, color=SERIES[k], alpha=0.75, label=f"{sz} family" if sz == 1 else f"{sz} families")
ax.axhline(0, color=INK2, lw=1); ax.axvline(0, color=INK2, lw=1); lim = max(GR.ic_train.abs().max(), GR.ic_test.abs().max()); ax.plot([-lim, lim], [-lim, lim], color=INK2, ls="--", lw=1)
ax.set_xlabel("IC (5-day), train"); ax.set_ylabel("IC (5-day), test"); ax.legend(frameon=False, fontsize=8); ax.set_title("Groups: does the train IC survive to test?")
plt.tight_layout()
savefig(fig, "18b_factor_groups", "Factor families in pairs, triplets and quadruplets", "18")

## Step 18c — Price–volume geometry: how price and volume move together in phase space

**Why here.** Steps 4d and 18 treated volume as a level (a z-score, a confirmation flag). This step looks at the **shape of the joint path**. Over a rolling window of $W=20$ days every product traces a curve $(x_i,y_i)$ in the price–volume plane:

$$x_i=\frac{1}{\sqrt W}\sum_{j\le i} z_j\;-\;x_1,\qquad y_i=\tilde v_i-\tilde v_1,\qquad z_j=\frac{r_j}{\sigma_j},\;\;\tilde v_j=\mathrm{clip}\!\left(\frac{\ln V_j-\overline{\ln V}_{j-1}^{(60)}}{s_{j-1}^{(60)}},\pm5\right)$$

($z$ is the ex-ante vol-scaled return of Step 3 and $\tilde v$ the log-volume z-score against its own trailing 60-day history, shifted one day.) Everything at date $t$ uses data up to and including $t$ only; the labels are $z_{t+1}$, $\sum_{k=1}^5 z_{t+k}$ and $|z_{t+1}|$.

| Geometric object | Formula | What it says |
|---|---|---|
| Signed (Lévy) area | $A=\frac{1}{2W}\sum_i\big(\bar x_i\,\Delta y_i-\bar y_i\,\Delta x_i\big)$, midpoints $\bar x_i,\bar y_i$ | $A>0$: counter-clockwise loop, **price leads volume**; $A<0$: clockwise, **volume leads price** |
| Efficiency | $ER=\|(x_W,y_W)\|\,/\sum_i\sqrt{\Delta x_i^2+\Delta y_i^2}$ | 1 = straight line (price and volume move together in one direction); near 0 = wandering |
| Turning | $\bar\kappa=\frac1{W-2}\sum_i\big\|\Delta\theta_i\big\|$ and winding $\frac1{2\pi}\sum\Delta\theta_i$ | Curvature of the path; winding counts net rotations |
| Co-movement | $\rho_{abs}=\mathrm{corr}(\|z\|,\tilde v)$, $\rho_{sgn}=\mathrm{corr}(z_t,\Delta\tilde v_t)$ | Volume–volatility and volume-change–return coupling |
| Volume tilt | $\sum e^{\tilde v_i}\mathrm{sgn}(z_i)\big/\sum e^{\tilde v_i}$ | Share of the window's volume on up days minus down days |
| Quadrant | sign of window return $\sum z$ × sign of window mean $\tilde v$ | Q1 confirmed advance, Q2 thin advance, Q3 confirmed decline, Q4 thin decline |

**Tests.** (1) Cross-sectional rank IC of each feature against the three labels, by split, Newey–West $t$, Benjamini–Hochberg on the training rows; a feature counts as *confirmed* only if val and test keep the sign and the test $|t|>2$. (2) Volume confirmation premium $c$ in $z_{t+1}=a+b\,s_t+c\,s_t\,u_t+\varepsilon$ with $s_t=\mathrm{sgn}(\sum z)$, $u_t=\mathrm{sgn}(\overline{\tilde v})$, date-clustered. (3) Lead–lag by day: $\mathrm{corr}(|z_t|,\Delta\tilde v_{t+k})$. (4) **Geometry map:** every product gets a profile (mean area, $|A|$, $ER$, $\bar\kappa$, $\rho_{abs}$, $\rho_{sgn}$, tilt, quadrant shares) on the training period; distances between standardised profiles give a 2-D classical-MDS map and a Ward clustering. The map is *stable* only if the distance matrix measured on val+test agrees with the training one (Spearman/Mantel correlation).

**Limits.** Daily bars only; a $W=20$ window has heavy overlap, so rolling features are strongly autocorrelated (hence NW lags and the 5-day label overlap handling). The map describes behaviour, it is not a signal by itself.

In [ ]:
# step 18c
# ---- 18c.1 geometric features: signed area, efficiency, turning, co-movement, tilt, quadrant ----
from numpy.lib.stride_tricks import sliding_window_view
PV_W, PV_VW, PV_VMIN = 20, 60, 30

def pv_geometry(Zr, V, W=PV_W):
    # every feature at row t is computed from the window of the W days ending at t (no future data)
    lv = np.log(V.where(V > 0))
    mu = lv.rolling(PV_VW, min_periods=PV_VMIN).mean().shift(1); sd = lv.rolling(PV_VW, min_periods=PV_VMIN).std().shift(1)
    vz = ((lv - mu) / sd).clip(-5, 5).where(Zr.notna())
    zr = Zr.where(vz.notna())
    def win(F):
        a = np.vstack([np.full((W - 1, F.shape[1]), np.nan), F.to_numpy(float)])
        return sliding_window_view(a, W, axis=0)                      # (date, product, W)
    Zw, Vw = win(zr), win(vz)
    bad = np.isnan(Zw).any(-1) | np.isnan(Vw).any(-1)
    Zw, Vw = np.nan_to_num(Zw), np.nan_to_num(Vw)
    x = np.cumsum(Zw, -1) / np.sqrt(W); x = x - x[..., :1]; y = Vw - Vw[..., :1]
    dx, dy = np.diff(x, axis=-1), np.diff(y, axis=-1)
    xm, ym = (x[..., :-1] + x[..., 1:]) / 2, (y[..., :-1] + y[..., 1:]) / 2
    area = 0.5 * (xm * dy - ym * dx).sum(-1) / W
    step = np.hypot(dx, dy); Lp = step.sum(-1)
    er = np.hypot(x[..., -1], y[..., -1]) / np.where(Lp > 0, Lp, np.nan)
    th = np.arctan2(dy, dx); dth = (np.diff(th, axis=-1) + np.pi) % (2 * np.pi) - np.pi
    turn = np.abs(dth).mean(-1); wind = dth.sum(-1) / (2 * np.pi)
    def corr(a, b):
        a = a - a.mean(-1, keepdims=True); b = b - b.mean(-1, keepdims=True)
        return (a * b).sum(-1) / np.sqrt((a * a).sum(-1) * (b * b).sum(-1))
    rho_abs = corr(np.abs(Zw), Vw); rho_sgn = corr(Zw[..., 1:], np.diff(Vw, axis=-1))
    e = np.exp(Vw); tilt = (e * np.sign(Zw)).sum(-1) / e.sum(-1)
    net = Zw.sum(-1); vbar = Vw.mean(-1)
    out = dict(area=area, er=er, turn=turn, wind=wind, rho_abs=rho_abs, rho_sgn=rho_sgn, tilt=tilt, net=net, vbar=vbar)
    out = {k: pd.DataFrame(np.where(bad, np.nan, v), index=zr.index, columns=zr.columns) for k, v in out.items()}
    s = np.sign(out["net"]); out["area_dir"] = out["area"] * s; out["er_dir"] = out["er"] * s
    out["quad"] = (1 + (out["net"] < 0) * 2 + (out["vbar"] < 0) * 1).where(out["net"].notna())
    out["vz"] = vz
    return out

PVG = pv_geometry(Z.reindex(columns=UNIV), VOLU.reindex(index=Z.index, columns=UNIV))
PV_FEAT = ["area", "area_dir", "er", "er_dir", "turn", "wind", "rho_abs", "rho_sgn", "tilt", "vbar", "net"]
print(f"geometry features: {len(PV_FEAT)} x {PVG['area'].shape[1]} products, {int(PVG['area'].notna().any(axis=1).sum())} dates with a full window")

# sanity: a counter-clockwise loop (price leads volume) must give positive area; a straight line gives ER = 1 and zero area
_t = np.arange(PV_W); _x = np.cos(2 * np.pi * _t / PV_W); _y = np.sin(2 * np.pi * _t / PV_W)
_dx, _dy = np.diff(_x), np.diff(_y); _a = 0.5 * ((((_x[:-1] + _x[1:]) / 2) * _dy) - (((_y[:-1] + _y[1:]) / 2) * _dx)).sum()
assert _a > 0, "area sign convention broken"
_line = np.hypot(np.diff(np.arange(PV_W) * 1.0), np.diff(np.arange(PV_W) * 2.0)).sum(); assert abs(np.hypot(19, 38) / _line - 1) < 1e-9
# causal check: truncating the data at T must not change any feature on or before T
causal_test("pv_geometry", lambda Zr, V: {k: PVG_ for k, PVG_ in pv_geometry(Zr, V).items() if k in ("area", "er", "turn", "rho_abs", "tilt", "quad")},
            Z.reindex(columns=UNIV).iloc[-1200:], VOLU.reindex(index=Z.index, columns=UNIV).iloc[-1200:], step="18c")

In [ ]:
# ---- 18c.2 do the geometric features predict? rank IC by split, volume confirmation premium, lead-lag by day ----
PV_LAB = {"fwd1": Z.shift(-1), "fwd5": Z.rolling(5).sum().shift(-5), "fwdabs1": Z.abs().shift(-1)}
PV_IDX = {sp: Z.index.intersection(SPL[sp]) for sp in ("train", "val", "test")}
rows = []
for f in PV_FEAT:
    F = PVG[f].reindex(columns=UNIV)
    for lb, Y in PV_LAB.items():
        ic = xs_ic_fast(F, Y.reindex(columns=UNIV))
        for sp, ix in PV_IDX.items():
            m_, t_ = mean_t(ic.reindex(ix), L=10 if lb == "fwd5" else 5)
            rows.append(dict(feature=f, label=lb, split=sp, ic=m_, t=t_))
PV_IC = pd.DataFrame(rows)
PV_ICW = PV_IC.pivot_table(index=["feature", "label"], columns="split", values=["ic", "t"]); PV_ICW.columns = [f"{a}_{b}" for a, b in PV_ICW.columns]
PV_ICW = PV_ICW.reindex(columns=[f"{a}_{b}" for a in ("ic", "t") for b in ("train", "val", "test")])
p_ = 2 * (1 - stats.norm.cdf(PV_ICW["t_train"].abs())); ok_ = np.isfinite(p_); q_ = np.full(len(p_), np.nan)
if ok_.any():                                                          # Benjamini-Hochberg on the training rows only
    o_ = np.argsort(p_[ok_]); pp = p_[ok_][o_]; m_ = len(pp); adj = np.minimum.accumulate((pp * m_ / np.arange(1, m_ + 1))[::-1])[::-1]
    tmp = np.empty(m_); tmp[o_] = np.clip(adj, 0, 1); q_[ok_] = tmp
PV_ICW["q_train"] = q_
PV_ICW["confirmed"] = ((PV_ICW.q_train < 0.10) & (np.sign(PV_ICW.ic_train) == np.sign(PV_ICW.ic_val)) & (np.sign(PV_ICW.ic_train) == np.sign(PV_ICW.ic_test)) & (PV_ICW.t_test.abs() > 2))
PV_ICW.round(4).to_csv(OUT2 / "pv_geometry_ic.csv")
print("Cross-sectional rank IC of each geometric feature (NW t); q = BH-adjusted on train; confirmed = val and test keep the sign and |t_test| > 2:")
print(PV_ICW.round(3).to_string())

# volume confirmation premium: z(t+1) = a + b s + c s u, s = sign(window return), u = sign(window mean volume z)
def stk_pv(F, nm):
    s_ = F.reindex(index=Z.index, columns=UNIV).stack(); s_.index.names = ["date", "product"]; return s_.rename(nm)
PVL = pd.concat([stk_pv(PV_LAB["fwd1"], "y"), stk_pv(np.sign(PVG["net"]), "s"), stk_pv(np.sign(PVG["vbar"]), "u"), stk_pv(PVG["quad"], "q")], axis=1).dropna()
PVL["su"] = PVL.s * PVL.u
PVL["split"] = np.select([PVL.index.get_level_values("date").isin(PV_IDX[sp]) for sp in ("train", "val", "test")], ["train", "val", "test"], "")
rows = []
for sp in ("train", "val", "test"):
    d_ = PVL[PVL.split == sp]
    if len(d_) < 500: continue
    g_ = pd.factorize(d_.index.get_level_values("date"))[0]
    r_ = sm.OLS(d_.y.to_numpy(float), sm.add_constant(d_[["s", "su"]].to_numpy(float))).fit(cov_type="cluster", cov_kwds=dict(groups=g_))
    row = dict(split=sp, n=len(d_), b_trend=r_.params[1], t_trend=r_.tvalues[1], c_confirm=r_.params[2], t_confirm=r_.tvalues[2])
    for qq in (1, 2, 3, 4): row[f"mean_z_Q{qq}"] = float(d_.y[d_.q == qq].mean()); row[f"share_Q{qq}"] = float((d_.q == qq).mean())
    rows.append(row)
PV_CONF = pd.DataFrame(rows); PV_CONF.round(4).to_csv(OUT2 / "pv_confirmation_premium.csv", index=False)
print("\nNext-day z by quadrant and the volume confirmation premium c (positive c: a price move on high volume continues more than one on thin volume):")
print(PV_CONF.set_index("split").round(3).to_string())

# lead-lag by day, training period: corr(|z_t|, dv_{t+k}) and corr(z_t, dv_{t+k})
dv_ = PVG["vz"].diff(); LAGS = list(range(-5, 6)); tr_ = PV_IDX["train"]
def xc(a, b, k, ix):
    return a.loc[ix].corrwith(b.shift(-k).loc[ix])
PV_XC_ABS = pd.DataFrame({k: xc(Z.abs().reindex(columns=UNIV), dv_.reindex(columns=UNIV), k, tr_) for k in LAGS})
PV_XC_SGN = pd.DataFrame({k: xc(Z.reindex(columns=UNIV), dv_.reindex(columns=UNIV), k, tr_) for k in LAGS})
PV_XC_ABS.round(4).to_csv(OUT2 / "pv_xcorr_abs.csv"); PV_XC_SGN.round(4).to_csv(OUT2 / "pv_xcorr_signed.csv")
print("\nMean over products of corr(|z_t|, d volume_{t+k}) (k>0: volume follows the move):\n", PV_XC_ABS.mean().round(3).to_string())

In [ ]:
# ---- 18c.3 geometry map of the products: profile -> distance -> 2-D map and clusters, with a stability check ----
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist, squareform
def pv_profile(ix):
    cols = {}
    for nm, F in (("area", PVG["area"]), ("abs_area", PVG["area"].abs()), ("er", PVG["er"]), ("turn", PVG["turn"]), ("rho_abs", PVG["rho_abs"]), ("rho_sgn", PVG["rho_sgn"]), ("tilt", PVG["tilt"])):
        cols[nm] = F.reindex(index=ix, columns=UNIV).mean()
    for qq in (1, 2, 3, 4):
        qd_ = PVG["quad"].reindex(index=ix, columns=UNIV); cols[f"Q{qq}"] = (qd_ == qq).astype(float).where(qd_.notna()).mean()
    return pd.DataFrame(cols).dropna()
def pv_dist(P, ref=None):
    mu_, sd_ = (P.mean(), P.std().replace(0, np.nan)) if ref is None else (ref.mean(), ref.std().replace(0, np.nan))
    S_ = ((P - mu_) / sd_).dropna(axis=1)
    return S_, squareform(pdist(S_.to_numpy(), "euclidean"))
def pv_mds(D, k=2):
    n_ = len(D); J = np.eye(n_) - 1.0 / n_; B = -0.5 * J @ (D ** 2) @ J; w, V = np.linalg.eigh(B); o = np.argsort(w)[::-1][:k]
    return V[:, o] * np.sqrt(np.clip(w[o], 0, None)), w[o] / np.clip(w[w > 0].sum(), 1e-12, None)
PV_PTR = pv_profile(PV_IDX["train"]); PV_POOS = pv_profile(PV_IDX["val"].append(PV_IDX["test"]))
PV_COM = PV_PTR.index.intersection(PV_POOS.index)
PV_PTR, PV_POOS = PV_PTR.loc[PV_COM], PV_POOS.loc[PV_COM]
PV_STR, PV_DTR = pv_dist(PV_PTR); PV_SOOS, PV_DOOS = pv_dist(PV_POOS, ref=PV_PTR)     # OOS profile standardised with the TRAIN mean / sd
PV_XY, PV_EVR = pv_mds(PV_DTR)
PV_K = 4; PV_CL = pd.Series(fcluster(linkage(pdist(PV_STR.to_numpy()), "ward"), PV_K, "maxclust"), index=PV_COM, name="cluster")
PV_CL_OOS = pd.Series(fcluster(linkage(pdist(PV_SOOS.to_numpy()), "ward"), PV_K, "maxclust"), index=PV_COM)
iu_ = np.triu_indices(len(PV_COM), 1)
PV_MANTEL = float(stats.spearmanr(PV_DTR[iu_], PV_DOOS[iu_])[0]) if len(PV_COM) > 4 else np.nan
rng_ = np.random.default_rng(SEED); perm_ = []
for _ in range(500):
    p_ = rng_.permutation(len(PV_COM)); Dp = PV_DOOS[np.ix_(p_, p_)]; perm_.append(stats.spearmanr(PV_DTR[iu_], Dp[iu_])[0])
PV_MANTEL_P = float((np.sum(np.array(perm_) >= PV_MANTEL) + 1) / (len(perm_) + 1)) if np.isfinite(PV_MANTEL) else np.nan
try:
    from sklearn.metrics import adjusted_rand_score
    PV_ARI = float(adjusted_rand_score(PV_CL.to_numpy(), PV_CL_OOS.to_numpy()))
except Exception:
    PV_ARI = np.nan
PV_SECT = SEC_MAP.reindex(PV_COM)
PV_SAME = float(np.mean([PV_SECT.iloc[i] == PV_SECT.iloc[j] for i, j in zip(*iu_) if PV_CL.iloc[i] == PV_CL.iloc[j]])) if len(PV_COM) > 4 else np.nan
PV_BASE = float(np.mean([PV_SECT.iloc[i] == PV_SECT.iloc[j] for i, j in zip(*iu_)])) if len(PV_COM) > 4 else np.nan
PV_MAP = pd.concat([PV_PTR.round(3), pd.DataFrame(PV_XY, index=PV_COM, columns=["mds1", "mds2"]).round(3), PV_CL, PV_SECT.rename("sector")], axis=1)
PV_MAP.to_csv(OUT2 / "pv_geometry_map.csv")
print(f"Geometry map: {len(PV_COM)} products, 2-D MDS keeps {PV_EVR.sum():.0%} of the profile variance; Ward clusters k={PV_K}")
print(f"Stability (distance matrix train vs val+test): Spearman/Mantel = {PV_MANTEL:.2f} (permutation p = {PV_MANTEL_P:.3f}); cluster agreement ARI = {PV_ARI:.2f}")
print(f"Pairs in the same cluster that also share a sector: {PV_SAME:.0%}  (any pair: {PV_BASE:.0%})")
print(PV_MAP.groupby("cluster")[["area", "er", "turn", "rho_abs", "rho_sgn", "tilt", "Q1", "Q3"]].mean().round(3).to_string())
print("members:", {int(k): list(v.index) for k, v in PV_MAP.groupby("cluster")})

In [ ]:
# ---- 18c.4 figures and summary ----
fig, axs = plt.subplots(3, 5, figsize=(17, 10.5))
foc_ = [c for c in FOCUS if c in UNIV][:14]
for ax, c in zip(axs.ravel(), foc_ + ["__MKT__"]):
    if c == "__MKT__":
        zs, vs = Z[UNIV].mean(axis=1), PVG["vz"][UNIV].mean(axis=1); ttl = "Market (equal weight)"
    else:
        zs, vs = Z[c], PVG["vz"][c]; ttl = c
    d_ = pd.concat([zs, vs], axis=1).dropna().iloc[-PV_W:]
    if len(d_) < PV_W: ax.set_axis_off(); continue
    x_ = d_.iloc[:, 0].cumsum().to_numpy() / np.sqrt(PV_W); y_ = d_.iloc[:, 1].to_numpy(); x_, y_ = x_ - x_[0], y_ - y_[0]
    ax.scatter(x_, y_, c=np.arange(PV_W), cmap=SEQ, s=26, zorder=3); ax.plot(x_, y_, color=MUTE, lw=1)
    ax.annotate("", xy=(x_[-1], y_[-1]), xytext=(x_[-2], y_[-2]), arrowprops=dict(arrowstyle="->", color=INK2))
    ar_ = PVG["area"][c].iloc[-1] if c != "__MKT__" else np.nan; er_ = PVG["er"][c].iloc[-1] if c != "__MKT__" else np.nan
    ax.set_title(ttl + ("" if c == "__MKT__" else f"   A={ar_:+.2f}  ER={er_:.2f}"), fontsize=9); ax.axhline(0, color="#c9c7c2", lw=.6); ax.axvline(0, color="#c9c7c2", lw=.6)
    ax.set_xlabel("cum. return (σ units)", fontsize=7); ax.set_ylabel("volume z", fontsize=7); ax.tick_params(labelsize=7)
for ax in axs.ravel()[len(foc_) + 1:]: ax.set_axis_off()
fig.suptitle(f"Price–volume phase portraits, last {PV_W} days (dark = latest). Counter-clockwise: price leads volume", y=1.0)
savefig(fig, "18c_phase_portraits", "Price–volume phase portraits of the deep-dive products", "18c")

fig, axs = plt.subplots(1, 3, figsize=(17, 5))
ax = axs[0]; T_ = PV_ICW.reset_index(); T_["name"] = T_.feature + " → " + T_.label; T_ = T_.reindex(T_.t_test.abs().sort_values(ascending=False).index).head(14)[::-1]
yy = np.arange(len(T_)); w_ = 0.27
for k, (sp, col) in enumerate((("train", SERIES[0]), ("val", SERIES[1]), ("test", SERIES[2]))):
    ax.barh(yy + (k - 1) * w_, T_[f"ic_{sp}"], height=w_, color=col, label=sp)
ax.set_yticks(yy); ax.set_yticklabels(T_.name, fontsize=7); ax.axvline(0, color="#c9c7c2"); ax.legend(fontsize=7); ax.set_title("Rank IC (14 largest |t| on test)", fontsize=10)
ax = axs[1]
qx = np.arange(4)
for k, (sp, col) in enumerate(zip(("train", "val", "test"), SERIES)):
    r_ = PV_CONF[PV_CONF.split == sp]
    if len(r_): ax.bar(qx + (k - 1) * .27, [r_[f"mean_z_Q{q}"].iloc[0] for q in (1, 2, 3, 4)], width=.27, color=col, label=sp)
ax.set_xticks(qx); ax.set_xticklabels(["Q1 up\nhigh vol", "Q2 up\nthin", "Q3 down\nhigh vol", "Q4 down\nthin"], fontsize=8); ax.axhline(0, color="#c9c7c2"); ax.legend(fontsize=7)
ax.set_title("Next-day z by quadrant", fontsize=10)
ax = axs[2]
ax.plot(LAGS, PV_XC_ABS.mean(), color=SERIES[0], marker="o", label="corr(|z_t|, Δvolume_{t+k})"); ax.plot(LAGS, PV_XC_SGN.mean(), color=SERIES[1], marker="s", label="corr(z_t, Δvolume_{t+k})")
ax.axhline(0, color="#c9c7c2"); ax.axvline(0, color="#c9c7c2", lw=.6); ax.set_xlabel("k (days; k>0: volume after the move)"); ax.legend(fontsize=7); ax.set_title("Lead–lag of price and volume (train, mean over products)", fontsize=10)
plt.tight_layout(); savefig(fig, "18c_ic_quadrant_leadlag", "Price–volume geometry: predictive power, quadrants, lead–lag", "18c")

fig, axs = plt.subplots(1, 2, figsize=(15, 6.2))
ax = axs[0]; secs_ = sorted(PV_SECT.unique()); cm_ = {s: SERIES[i % len(SERIES)] for i, s in enumerate(secs_)}
for c, (a_, b_) in zip(PV_COM, PV_XY):
    ax.scatter(a_, b_, s=70, color=cm_[PV_SECT[c]], edgecolor=INK2, lw=.5, zorder=3, marker=["o", "s", "^", "D", "v", "P"][(PV_CL[c] - 1) % 6]); ax.annotate(c, (a_, b_), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("MDS 1"); ax.set_ylabel("MDS 2"); ax.axhline(0, color="#c9c7c2", lw=.6); ax.axvline(0, color="#c9c7c2", lw=.6)
ax.set_title(f"Geometry map (colour = sector, marker = cluster); Mantel train vs OOS = {PV_MANTEL:.2f}", fontsize=10)
ax = axs[1]; iu2 = np.triu_indices(len(PV_COM), 1); ax.scatter(PV_DTR[iu2], PV_DOOS[iu2], s=8, alpha=.4, color=SERIES[0]); lim_ = max(PV_DTR[iu2].max(), PV_DOOS[iu2].max())
ax.plot([0, lim_], [0, lim_], color=MUTE, lw=1); ax.set_xlabel("distance, training profile"); ax.set_ylabel("distance, val+test profile"); ax.set_title("Stability of the geometry: pairwise distances", fontsize=10)
plt.tight_layout(); savefig(fig, "18c_geometry_map", "Geometry map of the products and its stability", "18c")

def _g(sp, c):
    r_ = PV_CONF[PV_CONF.split == sp]; return float(r_[c].iloc[0]) if len(r_) else np.nan
_cf = PV_ICW[PV_ICW.confirmed]
PV_SUMMARY = dict(window_days=PV_W, n_products=int(len(PV_COM)), n_features=len(PV_FEAT),
                  confirmed_features=[f"{a} -> {b}" for a, b in _cf.index], n_confirmed=int(len(_cf)), n_tested=int(PV_ICW.q_train.notna().sum()),
                  confirm_premium={sp: dict(c=_g(sp, "c_confirm"), t=_g(sp, "t_confirm")) for sp in ("train", "val", "test")},
                  mean_area=float(PVG["area"].loc[PV_IDX["train"]].stack().mean()), mean_er=float(PVG["er"].loc[PV_IDX["train"]].stack().mean()),
                  volume_follows_move=bool(PV_XC_ABS.mean().loc[1:3].mean() > PV_XC_ABS.mean().loc[-3:-1].mean()),
                  map=dict(mantel=PV_MANTEL, mantel_p=PV_MANTEL_P, ari=PV_ARI, k=PV_K, same_sector_in_cluster=PV_SAME, same_sector_any_pair=PV_BASE, clusters={int(k): list(v.index) for k, v in PV_MAP.groupby("cluster")}))
_sg = "positive" if PV_SUMMARY["mean_area"] > 0 else "negative"
PV_SUMMARY["reading"] = (f"Average signed area is {_sg} (price {'leads' if PV_SUMMARY['mean_area'] > 0 else 'follows'} volume on average), mean efficiency {PV_SUMMARY['mean_er']:.2f}; "
    f"volume {'follows' if PV_SUMMARY['volume_follows_move'] else 'precedes'} large moves in the day-lag profile. {PV_SUMMARY['n_confirmed']} of {PV_SUMMARY['n_tested']} feature-label pairs are confirmed out of sample (BH on train, sign kept in val and test, |t|>2). "
    f"Volume confirmation premium c: train {_g('train','c_confirm'):+.3f} (t={_g('train','t_confirm'):.1f}), val {_g('val','c_confirm'):+.3f} (t={_g('val','t_confirm'):.1f}), test {_g('test','c_confirm'):+.3f} (t={_g('test','t_confirm'):.1f}). "
    f"The geometry map is {'stable' if (np.isfinite(PV_MANTEL) and PV_MANTEL > 0.5 and PV_MANTEL_P < 0.05) else 'not clearly stable'} (Mantel {PV_MANTEL:.2f}, p={PV_MANTEL_P:.3f}).")
print(PV_SUMMARY["reading"])

## Step 18d — Topology of price and volume: loops, holes and how the state changes

**Why this step.** Step 18c measured *geometry* (areas, angles, distances). Geometry changes when you stretch the picture; **topology** asks what survives stretching: how many separate pieces the data falls into ($H_0$), and how many independent loops/holes it contains ($H_1$). A loop in price–volume space means the system **returns to where it was by a different route** (volume builds, price moves, volume decays, price reverts) — recurrent dynamics rather than drift. A cloud that fragments into several components means the market is split into separate states. Persistent homology measures this at *every* scale at once and keeps only features that last.

**Construction (Vietoris–Rips filtration).** For a point cloud $X$ let $\mathrm{VR}_\varepsilon(X)$ be the complex that joins points closer than $\varepsilon$. As $\varepsilon$ grows, a hole appears at $b$ ("birth") and is filled at $d$ ("death"); its **persistence** is $d-b$. All windows end at $t$ (no future data); distances are divided by the window's median pairwise distance so the shape and the size are measured separately.

| Cloud | Points | Question |
|---|---|---|
| **Product cloud** | Takens delay embedding of the pair $(z,\tilde v)$ over $W=40$ days: $p_i=(z_i,\tilde v_i,z_{i-1},\tilde v_{i-1})\in\mathbb R^4$ | Does one product's price–volume dynamics circle back on itself (recurrence), and how fragmented is it? |
| **Market cloud** | the $W=60$ days, each a point in $\mathbb R^{2N}$ (all $z$ and all $\tilde v$ of the day) | Does the whole market cycle between states, as in Gidea–Katz's crash-warning work? |

| Summary of a diagram | Formula |
|---|---|
| size | median pairwise distance $s$ |
| fragmentation | $\sum (d-b)_{H_0}/s$ |
| strongest loop | $\max (d-b)_{H_1}/s$; count of loops with $(d-b)>0.2\,s$ |
| loop entropy | $-\sum p_k\ln p_k$, $p_k=(d_k-b_k)/\sum(d-b)$ |
| landscape | $\int\lambda_1(u)\,du/s^2$, $\lambda_1(u)=\max_k\max\!\big(0,\min(u-b_k,\,d_k-u)\big)$ |

**Tests (same discipline as the rest of the notebook).** (1) *Null*: shuffling volume inside each window destroys price–volume coupling; the share of real windows whose strongest loop beats the 95th percentile of the shuffled windows says whether the loops are more than chance. (2) *Prediction*: market topology vs next-5-day market volatility with trailing volatility as control (HAC $t$ by split), and per-product rank IC of topology, **after removing trailing volatility** cross-sectionally, with Benjamini–Hochberg on train and a sign/|t| check on validation and test. (3) *Event study*: how topology moves in the 20 days before and after volatility shocks, and how often the topology was already elevated the day before. (4) **Mapper**: the market's state space (volatility level, volume level, 20-day return) is covered by overlapping bins and clustered into nodes joined when they share days; built on train, test days are assigned to the nearest train node; the loops of that graph and the *flows between nodes* show how the market moves from one state to the next, and a node's forward volatility on train is checked against validation + test.

**Limits.** Daily data, 40–60 points per cloud: only loops that clearly outlive the noise matter, and with that few points $H_1$ is fragile; the null and the out-of-sample checks are there for that. The package `ripser` is used when installed; otherwise a slower built-in implementation runs on a thinner grid of windows. Topology describes the state; it is not a trading rule until it passes the later strategy steps.

In [ ]:
# step 18d
# ---- 18d.1 persistent homology toolkit (ripser if available, otherwise a built-in Vietoris-Rips for H0 and H1) ----
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster
TD_W, TD_WM, TD_LAG = 40, 60, 1                    # window of the product cloud, of the market cloud, Takens delay
try:
    from ripser import ripser as _ripser; TD_BACKEND = "ripser"
except Exception:
    try:                                                    # best effort: works when Internet is enabled on Kaggle
        import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ripser"], check=True, timeout=300)
        from ripser import ripser as _ripser; TD_BACKEND = "ripser"
    except Exception:
        _ripser = None; TD_BACKEND = "built-in"
TD_STRIDE = 5 if _ripser is not None else 20       # windows are computed every TD_STRIDE days and carried forward (carry-forward is causal)
TD_KEYS = ["scale", "h0_tot", "h1_max", "h1_tot", "h1_n", "h1_ent", "land"]
print(f"persistent homology backend: {TD_BACKEND}; stride {TD_STRIDE} days" + ("" if _ripser is not None else "  (pip install ripser makes this about 20x faster and uses every 5th day)"))

def _ph_builtin(X):
    D = squareform(pdist(X)); n = len(D); iu = np.triu_indices(n, 1); w = D[iu]; o = np.argsort(w, kind="stable")
    ei, ej, ew = iu[0][o], iu[1][o], w[o]; rank = {(int(a), int(b)): r for r, (a, b) in enumerate(zip(ei, ej))}
    par = list(range(n))
    def find(a):
        while par[a] != a: par[a] = par[par[a]]; a = par[a]
        return a
    h0 = []
    for r, (a, b) in enumerate(zip(ei, ej)):
        ra, rb = find(int(a)), find(int(b))
        if ra != rb: par[ra] = rb; h0.append((0.0, float(ew[r])))
    tris = []
    for a in range(n):
        for b in range(a + 1, n):
            rab = rank[(a, b)]
            for c in range(b + 1, n):
                e = (rab, rank[(a, c)], rank[(b, c)]); tris.append((max(e), e))
    tris.sort(); piv = {}; h1 = []
    for f, cols in tris:
        col = set(cols)
        while col:
            low = max(col)
            if low in piv: col ^= piv[low]
            else: piv[low] = col; break
        if col:
            low = max(col)
            if ew[f] > ew[low]: h1.append((float(ew[low]), float(ew[f])))
    return np.array(h0).reshape(-1, 2), np.array(h1).reshape(-1, 2)

def ph01(X, force_builtin=False):
    X = np.asarray(X, float)
    if _ripser is not None and not force_builtin:
        d = _ripser(X, maxdim=1)["dgms"]; h0 = d[0][np.isfinite(d[0][:, 1])]; h1 = d[1]
        return h0, h1
    return _ph_builtin(X)

def ph_stats(h0, h1, X):
    Dm = pdist(X); s = float(np.median(Dm)) if len(Dm) else np.nan
    if not np.isfinite(s) or s <= 0: return {k: np.nan for k in TD_KEYS}
    l0 = (h0[:, 1] - h0[:, 0]) / s if len(h0) else np.zeros(0); l1 = (h1[:, 1] - h1[:, 0]) / s if len(h1) else np.zeros(0)
    p = l1 / l1.sum() if l1.sum() > 0 else l1
    ent = float(-(p * np.log(p)).sum()) if len(l1) else 0.0
    land = 0.0
    if len(h1):
        u = np.linspace(0, float(Dm.max()) / s, 64); b, d = (h1[:, 0] / s)[:, None], (h1[:, 1] / s)[:, None]
        land = float(np.clip(np.minimum(u[None, :] - b, d - u[None, :]), 0, None).max(0).sum() * (u[1] - u[0]))
    return dict(scale=s, h0_tot=float(l0.sum()), h1_max=float(l1.max()) if len(l1) else 0.0, h1_tot=float(l1.sum()), h1_n=float((l1 > 0.2).sum()), h1_ent=ent, land=land)

def td_cloud(z, v, lag=TD_LAG):
    return np.column_stack([z[lag:], v[lag:], z[:-lag], v[:-lag]])

# ---- unit checks: a circle has one long loop; a blob has none; two far clusters have one long H0 bar; built-in agrees with ripser ----
_r = np.random.default_rng(0); _th = np.linspace(0, 2 * np.pi, 40, endpoint=False)
_circ = np.c_[np.cos(_th), np.sin(_th)] + 0.02 * _r.normal(size=(40, 2)); _blob = _r.normal(size=(40, 2))
_c0, _c1 = ph01(_circ); _b0, _b1 = ph01(_blob)
assert (_c1[:, 1] - _c1[:, 0]).max() > 0.8, "circle: loop not found"
assert len(_b1) == 0 or (_b1[:, 1] - _b1[:, 0]).max() < 0.5, "blob: spurious loop"
_two = np.vstack([_r.normal(size=(20, 2)) * 0.1, _r.normal(size=(20, 2)) * 0.1 + 5]); _t0, _ = ph01(_two); assert (_t0[:, 1] - _t0[:, 0]).max() > 4, "two clusters: split not found"
if _ripser is not None:
    for _X in (_circ, _blob, _r.normal(size=(30, 4))):
        _a0, _a1 = ph01(_X); _bb0, _bb1 = _ph_builtin(_X)
        assert abs(np.sort(_a0[:, 1]).sum() - np.sort(_bb0[:, 1]).sum()) < 1e-5 and abs((_a1[:, 1] - _a1[:, 0]).sum() - (_bb1[:, 1] - _bb1[:, 0]).sum()) < 1e-5, "built-in persistent homology disagrees with ripser"
    print("built-in implementation agrees with ripser on circle, blob and a 4-D cloud")
print("topology unit checks passed: circle -> one long loop, blob -> none, two clusters -> one long H0 bar")

# ---- window features, every TD_STRIDE days, causal (the window ends at the date) ----
def td_product_feats(Zr, VZ, W=TD_W, stride=TD_STRIDE, cols=None):
    cols = list(Zr.columns) if cols is None else cols
    za, va = Zr[cols].to_numpy(float), VZ.reindex(index=Zr.index, columns=cols).to_numpy(float); T = len(Zr)
    out = {k: np.full((T, len(cols)), np.nan) for k in TD_KEYS}
    for j in range(len(cols)):
        ok = np.isfinite(za[:, j]) & np.isfinite(va[:, j])
        for t in range(W - 1, T, stride):
            sl = slice(t - W + 1, t + 1)
            if not ok[sl].all(): continue
            X = td_cloud(za[sl, j], va[sl, j]); st = ph_stats(*ph01(X), X)
            for k in TD_KEYS: out[k][t, j] = st[k]
    return {k: pd.DataFrame(a, index=Zr.index, columns=cols).ffill(limit=stride - 1) for k, a in out.items()}

def td_market_feats(Zf, Vf, W=TD_WM, stride=TD_STRIDE):
    za, va = Zf.to_numpy(float), Vf.to_numpy(float); T = len(Zf); out = {k: np.full(T, np.nan) for k in TD_KEYS}
    for t in range(W - 1, T, stride):
        sl = slice(t - W + 1, t + 1); X = np.hstack([za[sl], va[sl]])
        if np.isnan(X).mean() > 0.3: continue
        X = np.nan_to_num(X); st = ph_stats(*ph01(X), X)
        for k in TD_KEYS: out[k][t] = st[k]
    return pd.DataFrame(out, index=Zf.index).ffill(limit=stride - 1)

TD_Z = Z.reindex(columns=UNIV); TD_VZ = PVG["vz"].reindex(columns=UNIV)          # volume z-score of Step 18c (trailing 60-day baseline, shifted one day)
t0_ = time.time()
TDP = td_product_feats(TD_Z, TD_VZ, cols=UNIV if _ripser is not None else [c for c in FOCUS if c in UNIV])
TDM = td_market_feats(TD_Z.fillna(0).where(TD_Z.notna().mean(axis=1) > 0.7), TD_VZ.fillna(0).where(TD_Z.notna().mean(axis=1) > 0.7))
print(f"topology features: product clouds {list(TDP['h1_max'].columns)[:3]}...({TDP['h1_max'].shape[1]} products), market cloud; {time.time() - t0_:.0f}s")
causal_test("td_market_feats", lambda a, b: td_market_feats(a, b), TD_Z.fillna(0).iloc[-420:], TD_VZ.fillna(0).iloc[-420:], step="18d")

In [ ]:
# ---- 18d.2 are the loops real? prediction, event study ----
TD_IDX = {sp: Z.index.intersection(SPL[sp]) for sp in ("train", "val", "test")}
# (1) null: volume shuffled inside each window
rng_ = np.random.default_rng(SEED); rows = []; za_, va_ = TD_Z.to_numpy(float), TD_VZ.to_numpy(float)
pcols_ = list(TDP["h1_max"].columns)
for j, c in enumerate(pcols_):
    jj = list(TD_Z.columns).index(c); real, null = [], []
    ts_ = [t for t in range(TD_W - 1, len(TD_Z), TD_STRIDE * 4) if np.isfinite(za_[t - TD_W + 1:t + 1, jj]).all() and np.isfinite(va_[t - TD_W + 1:t + 1, jj]).all() and TD_Z.index[t] in TD_IDX["train"]]
    for t in ts_[:60]:
        sl = slice(t - TD_W + 1, t + 1); zz, vv = za_[sl, jj], va_[sl, jj]
        X = td_cloud(zz, vv); real.append(ph_stats(*ph01(X), X)["h1_max"])
        Xn = td_cloud(zz, rng_.permutation(vv)); null.append(ph_stats(*ph01(Xn), Xn)["h1_max"])
    if len(real) >= 10: rows.append(dict(product=c, n=len(real), real_mean=np.mean(real), null_mean=np.mean(null), share_above_null95=float(np.mean(np.array(real) > np.quantile(null, 0.95)))))
TD_NULL = pd.DataFrame(rows).set_index("product") if rows else pd.DataFrame(columns=["n", "real_mean", "null_mean", "share_above_null95"])
TD_NULL.round(3).to_csv(OUT2 / "topology_null.csv")
print("Strongest H1 loop (relative to window scale), real vs volume-shuffled windows, train period; share of real windows above the 95th percentile of the null (5% = pure chance):")
print(TD_NULL.round(3).to_string()); print(f"mean share above null: {TD_NULL.share_above_null95.mean():.3f}")

# (2a) market topology vs next-5-day market volatility, trailing volatility as control
MABS = Z[UNIV].abs().mean(axis=1)
TD_LAB = pd.DataFrame({"fvol5": MABS.rolling(5).mean().shift(-5), "vol5": MABS.rolling(5).mean(), "vol20": MABS.rolling(20).mean()})
rows = []
for f in TD_KEYS:
    x = TDM[f]; mu_, sd_ = x.loc[TD_IDX["train"]].mean(), x.loc[TD_IDX["train"]].std()
    if not np.isfinite(sd_) or sd_ < 1e-9:                      # constant feature (e.g. a loop count that is always 0): nothing to test
        rows += [dict(feature=f, split=sp, n=0, beta=np.nan, t=np.nan, t_alone=np.nan) for sp in TD_IDX]; continue
    xs_ = (x - mu_) / sd_
    for sp, ix in TD_IDX.items():
        d_ = pd.concat([TD_LAB.fvol5, xs_.rename("f"), TD_LAB.vol5, TD_LAB.vol20], axis=1).loc[ix].dropna()
        d_ = d_.iloc[::5]                                           # non-overlapping 5-day labels
        if len(d_) < 60: rows.append(dict(feature=f, split=sp, n=len(d_), beta=np.nan, t=np.nan, t_alone=np.nan)); continue
        r_ = sm.OLS(d_.fvol5, sm.add_constant(d_[["f", "vol5", "vol20"]])).fit(cov_type="HAC", cov_kwds=dict(maxlags=3))
        r0_ = sm.OLS(d_.fvol5, sm.add_constant(d_[["f"]])).fit(cov_type="HAC", cov_kwds=dict(maxlags=3))
        rows.append(dict(feature=f, split=sp, n=len(d_), beta=float(r_.params["f"]), t=float(r_.tvalues["f"]), t_alone=float(r0_.tvalues["f"])))
TD_MKT = pd.DataFrame(rows); TD_MKTW = TD_MKT.pivot(index="feature", columns="split", values=["beta", "t", "t_alone"]); TD_MKTW.columns = [f"{a}_{b}" for a, b in TD_MKTW.columns]
TD_MKTW = TD_MKTW.reindex(index=TD_KEYS, columns=[f"{a}_{b}" for a in ("beta", "t", "t_alone") for b in ("train", "val", "test")])
TD_MKTW["controls_ok"] = (np.sign(TD_MKTW.beta_train) == np.sign(TD_MKTW.beta_val)) & (np.sign(TD_MKTW.beta_train) == np.sign(TD_MKTW.beta_test)) & (TD_MKTW.t_test.abs() > 2)
TD_MKTW.round(4).to_csv(OUT2 / "topology_market_vol.csv")
print("\nMarket topology -> next-5-day market volatility (standardised feature; 't' controls for trailing 5/20-day volatility; 't_alone' does not); controls_ok = same sign in all periods and |t_test| > 2:")
print(TD_MKTW.round(3).to_string())

# (2b) per product: rank IC of topology (trailing volatility removed cross-sectionally) vs next-day and next-5-day absolute move
def td_resid(F, V):
    m = F.notna() & V.notna(); Fr = F.where(m).rank(axis=1); Vr = V.where(m).rank(axis=1)
    a = Fr.sub(Fr.mean(axis=1), axis=0); b = Vr.sub(Vr.mean(axis=1), axis=0); beta = (a * b).sum(axis=1) / (b * b).sum(axis=1)
    return a - b.mul(beta, axis=0)
TD_LABP = {"abs1": Z.abs().shift(-1), "abs5": Z.rolling(5).sum().shift(-5).abs()}
vol20_ = Z.abs().rolling(20).mean()
rows = []
if TDP["h1_max"].shape[1] >= 5:
    for f in TD_KEYS:
        F = TDP[f].reindex(columns=pcols_); R_ = td_resid(F, vol20_.reindex(columns=pcols_))
        for lb, Y in TD_LABP.items():
            ic = xs_ic_fast(R_, Y.reindex(columns=pcols_), min_n=5)
            for sp, ix in TD_IDX.items():
                m_, t_ = mean_t(ic.reindex(ix), L=10 if lb == "abs5" else 5); rows.append(dict(feature=f, label=lb, split=sp, ic=m_, t=t_))
TD_IC = pd.DataFrame(rows)
if len(TD_IC):
    TD_ICW = TD_IC.pivot_table(index=["feature", "label"], columns="split", values=["ic", "t"]); TD_ICW.columns = [f"{a}_{b}" for a, b in TD_ICW.columns]
    TD_ICW = TD_ICW.reindex(columns=[f"{a}_{b}" for a in ("ic", "t") for b in ("train", "val", "test")])
    p_ = 2 * (1 - stats.norm.cdf(TD_ICW["t_train"].abs())); ok_ = np.isfinite(p_); q_ = np.full(len(p_), np.nan)
    if ok_.any():
        o_ = np.argsort(p_[ok_]); pp = p_[ok_][o_]; m_ = len(pp); adj = np.minimum.accumulate((pp * m_ / np.arange(1, m_ + 1))[::-1])[::-1]; tmp = np.empty(m_); tmp[o_] = np.clip(adj, 0, 1); q_[ok_] = tmp
    TD_ICW["q_train"] = q_
    TD_ICW["confirmed"] = (TD_ICW.q_train < 0.10) & (np.sign(TD_ICW.ic_train) == np.sign(TD_ICW.ic_val)) & (np.sign(TD_ICW.ic_train) == np.sign(TD_ICW.ic_test)) & (TD_ICW.t_test.abs() > 2)
    TD_ICW.round(4).to_csv(OUT2 / "topology_product_ic.csv")
    print("\nPer-product rank IC of topology features, trailing volatility removed (confirmed = BH q<0.10 on train, sign kept in val and test, |t_test|>2):")
    print(TD_ICW.round(3).to_string())
else:
    TD_ICW = pd.DataFrame(columns=["confirmed"]); print("\nper-product IC skipped (fewer than 5 products have topology features; install ripser for the full universe)")

# (3) event study: volatility shocks
thr_ = MABS.loc[TD_IDX["train"]].quantile(0.975); ev_ = []; last_ = -99
for d_ in MABS.index[MABS > thr_]:
    p_ = MABS.index.get_loc(d_)
    if p_ - last_ >= 10 and p_ >= 25 and p_ + 20 < len(MABS): ev_.append(p_); last_ = p_
TD_EV_N = {sp: sum(MABS.index[p] in TD_IDX[sp] for p in ev_) for sp in TD_IDX}
TD_EVF = ["h1_max", "land", "h0_tot", "scale"]; TD_PATH = {}; lead_ = []
for f in TD_EVF:
    x = TDM[f].ffill(); rel = x / x.rolling(250, min_periods=100).median().shift(1); q80 = x.rolling(250, min_periods=100).quantile(0.8).shift(1); fl = (x > q80).where(q80.notna())
    for grp, sps in (("train", ("train",)), ("val+test", ("val", "test"))):
        ps_ = [p for p in ev_ if any(MABS.index[p] in TD_IDX[s] for s in sps)]
        if len(ps_) < 5: continue
        M_ = np.vstack([rel.iloc[p - 20:p + 21].to_numpy() for p in ps_]); TD_PATH[(f, grp)] = pd.Series(np.nanmean(M_, axis=0), index=range(-20, 21))
        flags = np.array([fl.iloc[p - 1] for p in ps_], float); flags = flags[np.isfinite(flags)]; base = float(fl.dropna().mean())
        if len(flags) >= 5:
            pv = float(stats.binomtest(int(flags.sum()), len(flags), base, alternative="greater").pvalue); lead_.append(dict(feature=f, group=grp, events=len(flags), share_elevated_day_before=float(flags.mean()), base_rate=base, p_value=pv))
TD_LEAD = pd.DataFrame(lead_); TD_LEAD.round(4).to_csv(OUT2 / "topology_event_lead.csv", index=False)
print(f"\nVolatility shocks (market mean |z| above train 97.5th pct, >=10 days apart): {TD_EV_N}")
print("Share of shocks for which the feature was already in its own top quintile the day before (base rate = chance):")
print(TD_LEAD.round(3).to_string(index=False) if len(TD_LEAD) else "too few events")

In [ ]:
# ---- 18d.3 Mapper: the market's state space as a graph; how the market moves from state to state ----
import networkx as nx
TD_MIN, TD_NI, TD_OV = 20, 7, 0.35
ZM_ = Z[UNIV].mean(axis=1)
TD_STATE = pd.DataFrame({"vol20": MABS.rolling(20).mean(), "vbar20": TD_VZ.mean(axis=1).rolling(20).mean(), "net20": ZM_.rolling(20).sum()}).dropna()
TD_FUT = pd.DataFrame({"fvol5": MABS.rolling(5).mean().shift(-5), "fret5": ZM_.rolling(5).sum().shift(-5)})
_tr = TD_STATE.index.intersection(TD_IDX["train"]); mu_, sd_ = TD_STATE.loc[_tr].mean(), TD_STATE.loc[_tr].std()
TD_S = (TD_STATE - mu_) / sd_                                                                       # standardised with TRAIN moments
A = TD_S.loc[_tr].to_numpy(); nnd = np.sort(squareform(pdist(A[::3])), axis=1)[:, 5]; TD_EPS = float(2.0 * np.median(nnd))
def td_cover(lens, n_int=TD_NI, ov=TD_OV):
    lo, hi = lens.min(0), lens.max(0); L = (hi - lo) / (n_int - (n_int - 1) * ov); step = L * (1 - ov); cubes = []
    for i in range(n_int):
        for j in range(n_int):
            a0, b0 = lo[0] + i * step[0], lo[1] + j * step[1]
            cubes.append(np.where((lens[:, 0] >= a0) & (lens[:, 0] <= a0 + L[0]) & (lens[:, 1] >= b0) & (lens[:, 1] <= b0 + L[1]))[0])
    return cubes
nodes = []                                                                                           # list of index arrays into the train matrix
for idx_ in td_cover(A[:, [0, 1]]):                                                                  # lens: volatility level, volume level
    if len(idx_) < TD_MIN: continue
    lab = fcluster(linkage(A[idx_], "single"), TD_EPS, "distance")
    for k in np.unique(lab):
        m_ = idx_[lab == k]
        if len(m_) >= TD_MIN: nodes.append(m_)
TD_G = nx.Graph(); sets = [set(n_.tolist()) for n_ in nodes]
for i, n_ in enumerate(nodes): TD_G.add_node(i)
for i in range(len(nodes)):
    for j in range(i + 1, len(nodes)):
        if sets[i] & sets[j]: TD_G.add_edge(i, j)
TD_NN, TD_NE, TD_NC = TD_G.number_of_nodes(), TD_G.number_of_edges(), nx.number_connected_components(TD_G) if TD_G.number_of_nodes() else 0
def td_gf2_rank(cols):
    piv = {}; r = 0
    for c in cols:
        while c:
            h = c.bit_length() - 1
            if h in piv: c ^= piv[h]
            else: piv[h] = c; r += 1; break
    return r
_eid = {tuple(sorted(e)): k for k, e in enumerate(TD_G.edges())}
_tri = [sum(1 << _eid[tuple(sorted(p))] for p in ((a_, b_), (a_, c_), (b_, c_))) for a_, b_, c_ in (t_ for t_ in (c for c in nx.enumerate_all_cliques(TD_G) if len(c) == 3))]
TD_BETTI1 = TD_NE - TD_NN + TD_NC - td_gf2_rank(_tri)                                                # independent loops of the state graph with its triangles filled (H1 of the nerve)
cent = np.array([A[n_].mean(0) for n_ in nodes]) if nodes else np.zeros((0, 3))
fut_tr = TD_FUT.reindex(TD_S.loc[_tr].index)
TD_NODES = pd.DataFrame({"n_train": [len(n_) for n_ in nodes], "vol20": cent[:, 0] * sd_["vol20"] + mu_["vol20"] if len(cent) else [], "vbar20": cent[:, 1] * sd_["vbar20"] + mu_["vbar20"] if len(cent) else [],
                         "net20": cent[:, 2] * sd_["net20"] + mu_["net20"] if len(cent) else [],
                         "fvol5_train": [fut_tr.fvol5.iloc[n_].mean() for n_ in nodes], "fret5_train": [fut_tr.fret5.iloc[n_].mean() for n_ in nodes]})
# assign every date to the nearest TRAIN node centroid; forward behaviour of each node on train vs val+test
Aall = TD_S.to_numpy(); dd_ = ((Aall[:, None, :] - cent[None, :, :]) ** 2).sum(-1) if len(cent) else np.zeros((len(Aall), 0))
TD_ASSIGN = pd.Series(dd_.argmin(1) if dd_.shape[1] else np.nan, index=TD_S.index)
oos_ = TD_S.index.intersection(TD_IDX["val"].append(TD_IDX["test"])); fo_ = TD_FUT.reindex(oos_)
TD_NODES["n_oos"] = [int((TD_ASSIGN.loc[oos_] == i).sum()) for i in range(TD_NN)]
TD_NODES["fvol5_oos"] = [fo_.fvol5[TD_ASSIGN.loc[oos_] == i].mean() for i in range(TD_NN)]; TD_NODES["fret5_oos"] = [fo_.fret5[TD_ASSIGN.loc[oos_] == i].mean() for i in range(TD_NN)]
TD_NODES.round(4).to_csv(OUT2 / "topology_mapper_nodes.csv")
_ok = TD_NODES[(TD_NODES.n_oos >= 10)]
TD_MAP_RHO = float(stats.spearmanr(_ok.fvol5_train, _ok.fvol5_oos)[0]) if len(_ok) >= 6 else np.nan
fit = np.polyfit(TD_S.loc[_tr, "vol20"], TD_FUT.reindex(_tr).fvol5.fillna(TD_FUT.fvol5.mean()), 1)
y_ = fo_.fvol5; ok2 = y_.notna() & TD_ASSIGN.loc[oos_].notna()
pred_node = TD_ASSIGN.loc[oos_].map(TD_NODES.fvol5_train); pred_base = np.polyval(fit, TD_S.loc[oos_, "vol20"])
den = ((y_[ok2] - y_[ok2].mean()) ** 2).sum()
TD_R2_NODE = float(1 - ((y_[ok2] - pred_node[ok2]) ** 2).sum() / den) if ok2.sum() > 50 else np.nan
TD_R2_BASE = float(1 - ((y_[ok2] - pred_base[ok2]) ** 2).sum() / den) if ok2.sum() > 50 else np.nan
# flows between nodes: consecutive-day transitions of the assigned node
a_ = TD_ASSIGN.dropna().astype(int); T_ = pd.crosstab(a_.shift(1).dropna().astype(int), a_.iloc[1:].astype(int)).reindex(index=range(TD_NN), columns=range(TD_NN), fill_value=0)
TD_STAY = float(np.trace(T_.to_numpy()) / max(T_.to_numpy().sum(), 1)); TD_FLOW = (T_ - np.diag(np.diag(T_)))
fl_ = TD_FLOW.stack().sort_values(ascending=False); TD_TOP = [(int(i), int(j), int(v)) for (i, j), v in fl_.head(12).items() if v > 0]
T_.to_csv(OUT2 / "topology_mapper_transitions.csv")
# shock entry: which node is the market in 1..5 days before a volatility shock?
TD_PRE = pd.Series([TD_ASSIGN.get(MABS.index[p - 1]) for p in ev_]).dropna().astype(int).value_counts()
print(f"Mapper: {TD_NN} nodes, {TD_NE} edges, {TD_NC} components, {TD_BETTI1} independent loops (H1 of the Mapper complex, triangles filled); epsilon={TD_EPS:.2f}")
print(f"Forward volatility of a node, train vs val+test: Spearman {TD_MAP_RHO:.2f}; out-of-sample R2 from node means {TD_R2_NODE:.3f} vs from trailing volatility alone {TD_R2_BASE:.3f}")
print(f"Day-to-day: the market stays in the same node {TD_STAY:.0%} of the time; strongest moves between nodes (from, to, count): {TD_TOP[:6]}")
print("Nodes occupied the day before volatility shocks (node: count):", TD_PRE.head(5).to_dict())
print(TD_NODES.round(3).head(20).to_string())

In [ ]:
# ---- 18d.4 figures and summary ----
def td_panel(axs, X2, h0, h1, title):
    ax = axs[0]; ax.scatter(X2[:, 0], X2[:, 1], c=np.arange(len(X2)), cmap=SEQ, s=24, zorder=3); ax.plot(X2[:, 0], X2[:, 1], color=MUTE, lw=.7)
    ax.set_title(title, fontsize=9); ax.tick_params(labelsize=7)
    ax = axs[1]; mx = max([h1.max() if len(h1) else 0, h0.max() if len(h0) else 0, 1e-6]); ax.plot([0, mx], [0, mx], color=MUTE, lw=1)
    if len(h0): ax.scatter(h0[:, 0], h0[:, 1], s=14, color=SERIES[1], label="H0 components")
    if len(h1): ax.scatter(h1[:, 0], h1[:, 1], s=26, color=SERIES[0], label="H1 loops")
    ax.set_xlabel("birth", fontsize=7); ax.set_ylabel("death", fontsize=7); ax.legend(fontsize=6); ax.set_title("persistence diagram", fontsize=9); ax.tick_params(labelsize=7)
    ax = axs[2]; b1 = h1[np.argsort(-(h1[:, 1] - h1[:, 0]))][:8] if len(h1) else np.zeros((0, 2)); b0 = h0[np.argsort(-(h0[:, 1] - h0[:, 0]))][:6]
    for k, (b, d) in enumerate(b0): ax.hlines(k, b, d, color=SERIES[1], lw=3)
    for k, (b, d) in enumerate(b1): ax.hlines(len(b0) + 1 + k, b, d, color=SERIES[0], lw=3)
    ax.set_title("barcode (orange H0, blue H1)", fontsize=9); ax.set_yticks([]); ax.tick_params(labelsize=7)
def _win(c, date):
    jj = list(TD_Z.columns).index(c); t = TD_Z.index.get_loc(date); sl = slice(t - TD_W + 1, t + 1); return td_cloud(TD_Z[c].to_numpy()[sl], TD_VZ[c].to_numpy()[sl])
pc_ = "RB" if "RB" in pcols_ else pcols_[0]; hm_ = TDP["h1_max"][pc_].loc[TD_IDX["train"]].dropna()
if len(hm_) > 20:
    calm_d, rich_d = hm_.index[(hm_ - hm_.quantile(0.5)).abs().argmin()], hm_.idxmax(); fig, axs = plt.subplots(2, 3, figsize=(13, 7.5))
    for row, (d_, nm) in enumerate(((calm_d, "median loop strength"), (rich_d, "strongest loop in training"))):
        X = _win(pc_, d_); h0, h1 = ph01(X); td_panel(axs[row], X[:, :2], h0, h1, f"{pc_} {d_.date()} — {nm}\nreturn (x) vs volume z (y), dark = latest")
    fig.suptitle(f"Topology of one product's price–volume dynamics ({TD_W}-day Takens cloud)", y=1.0); plt.tight_layout(); savefig(fig, "18d_product_topology", "Persistence of price–volume dynamics: typical vs loop-rich window", "18d")
mk_ = TDM["h1_max"].loc[TD_IDX["train"]].dropna()
if len(mk_) > 20:
    mc_, mr_ = mk_.index[(mk_ - mk_.quantile(0.5)).abs().argmin()], mk_.idxmax(); fig, axs = plt.subplots(2, 3, figsize=(13, 7.5)); okm = TD_Z.notna().mean(axis=1) > 0.7
    for row, (d_, nm) in enumerate(((mc_, "median loop strength"), (mr_, "strongest loop in training"))):
        t = TD_Z.index.get_loc(d_); sl = slice(t - TD_WM + 1, t + 1); X = np.nan_to_num(np.hstack([TD_Z.to_numpy(float)[sl], TD_VZ.to_numpy(float)[sl]])); h0, h1 = ph01(X)
        U, S_, Vt = np.linalg.svd(X - X.mean(0), full_matrices=False); td_panel(axs[row], U[:, :2] * S_[:2], h0, h1, f"Market {d_.date()} — {nm}\nprincipal plane of the {TD_WM}-day cloud, dark = latest")
    fig.suptitle("Topology of the whole market's daily states (all returns and all volumes together)", y=1.0); plt.tight_layout(); savefig(fig, "18d_market_topology", "Persistence of the market state cloud", "18d")
fig, axs = plt.subplots(3, 1, figsize=(14, 8.5), sharex=True); tt_ = TD_IDX["val"].append(TD_IDX["test"])
axs[0].plot(MABS.loc[tt_], color=MUTE, lw=.8); axs[0].axhline(thr_, color=SERIES[4], lw=1, ls="--"); axs[0].set_title("Market mean |z| (dashed: shock threshold)", fontsize=10)
axs[1].plot(TDM["h1_max"].loc[tt_], color=SERIES[0], lw=1); axs[1].set_title("Strongest loop (H1) of the market state cloud, relative to scale", fontsize=10)
axs[2].plot(TDM["h0_tot"].loc[tt_], color=SERIES[1], lw=1); axs[2].set_title("Fragmentation of the cloud (H0 total persistence)", fontsize=10)
for p in ev_:
    d_ = MABS.index[p]
    if d_ in tt_:
        for a in axs: a.axvline(d_, color=SERIES[4], lw=.6, alpha=.6)
plt.tight_layout(); savefig(fig, "18d_market_topology_time", "Market topology over time (validation + test) with shock dates", "18d")
fig, axs = plt.subplots(1, 4, figsize=(17, 4)); xs_ = range(-20, 21)
for ax, f in zip(axs, TD_EVF):
    for grp, col in (("train", SERIES[0]), ("val+test", SERIES[1])):
        if (f, grp) in TD_PATH: ax.plot(xs_, TD_PATH[(f, grp)], color=col, label=grp)
    ax.axvline(0, color=MUTE, lw=.8); ax.axhline(1, color=MUTE, lw=.8, ls=":"); ax.set_title(f"{f} / trailing median", fontsize=9); ax.set_xlabel("days from shock", fontsize=8); ax.legend(fontsize=7)
plt.tight_layout(); savefig(fig, "18d_event_study", "How topology moves around volatility shocks", "18d")
if TD_NN > 1:
    fig, axs = plt.subplots(1, 2, figsize=(16, 7)); ax = axs[0]; pos = nx.spring_layout(TD_G, seed=SEED, k=1.2 / np.sqrt(max(TD_NN, 1)))
    nx.draw_networkx_edges(TD_G, pos, ax=ax, edge_color=MUTE, width=.8)
    sc = nx.draw_networkx_nodes(TD_G, pos, ax=ax, node_size=TD_NODES.n_train.to_numpy() * 0.8 + 40, node_color=TD_NODES.fvol5_train.to_numpy(), cmap=SEQ, edgecolors=INK2, linewidths=.6)
    for i, j, v in TD_TOP[:8]:
        if i in pos and j in pos: ax.annotate("", xy=pos[j], xytext=pos[i], arrowprops=dict(arrowstyle="-|>", color=SERIES[1], lw=1 + 3 * v / max(TD_TOP[0][2], 1), alpha=.8, connectionstyle="arc3,rad=0.15"))
    plt.colorbar(sc, ax=ax, shrink=.7, label="next-5-day market mean |z| (train)"); ax.set_axis_off()
    ax.set_title(f"Mapper graph of the market state (size = days; orange = strongest day-to-day flows)\n{TD_NN} nodes, {TD_NC} components, {TD_BETTI1} independent loops", fontsize=10)
    ax = axs[1]; ax.scatter(_ok.fvol5_train, _ok.fvol5_oos, s=np.clip(_ok.n_oos, 10, 300), color=SERIES[0], alpha=.7); lim = [min(_ok.fvol5_train.min(), _ok.fvol5_oos.min()), max(_ok.fvol5_train.max(), _ok.fvol5_oos.max())] if len(_ok) else [0, 1]
    ax.plot(lim, lim, color=MUTE); ax.set_xlabel("node's forward volatility, train"); ax.set_ylabel("same node, validation + test"); ax.set_title(f"Do the states behave the same out of sample? Spearman {TD_MAP_RHO:.2f}", fontsize=10)
    plt.tight_layout(); savefig(fig, "18d_mapper", "Mapper graph of the market state space and its out-of-sample check", "18d")

_cm = TD_MKTW[TD_MKTW.controls_ok].index.tolist() if len(TD_MKTW) else []; _cp = [f"{a} -> {b}" for a, b in TD_ICW.index[TD_ICW.confirmed.fillna(False)]] if "confirmed" in TD_ICW and len(TD_ICW) else []
_ld = TD_LEAD[(TD_LEAD.p_value < 0.05)] if len(TD_LEAD) else pd.DataFrame()
TD_SUMMARY = dict(backend=TD_BACKEND, windows=dict(product=TD_W, market=TD_WM, stride=TD_STRIDE), n_products=len(pcols_),
                  null_share_above95=float(TD_NULL.share_above_null95.mean()) if len(TD_NULL) else None,
                  market_vol_confirmed=_cm, product_ic_confirmed=_cp, shock_leading=[f"{r.feature} ({r.group})" for r in _ld.itertuples()] if len(_ld) else [],
                  mapper=dict(nodes=TD_NN, edges=TD_NE, components=TD_NC, cycles=TD_BETTI1, rho_oos=TD_MAP_RHO, r2_node=TD_R2_NODE, r2_vol_only=TD_R2_BASE, stay=TD_STAY, top_flows=TD_TOP[:6]))
_nl = TD_SUMMARY["null_share_above95"]
TD_SUMMARY["reading"] = (("Strongest-loop test against volume-shuffled windows: " + (f"{_nl:.0%} of real windows beat the 95th percentile of the null (chance = 5%), so the loops {'carry price–volume structure' if _nl > 0.10 else 'are close to what chance produces'}. " if _nl is not None else "not available. "))
    + f"Market topology predicts next-5-day volatility beyond trailing volatility out of sample for: {', '.join(_cm) if _cm else 'no feature'}. Per-product topology adds beyond volatility for: {', '.join(_cp) if _cp else 'no feature-label pair'}. "
    + f"Before volatility shocks the topology was already elevated for: {', '.join(TD_SUMMARY['shock_leading']) if TD_SUMMARY['shock_leading'] else 'no feature (no lead beyond chance)'}. "
    + f"The Mapper state graph has {TD_NN} nodes and {TD_BETTI1} independent loops (triangles filled); the market stays in the same node {TD_STAY:.0%} of days; a node's forward volatility has Spearman {TD_MAP_RHO:.2f} between train and out-of-sample, and node means explain {TD_R2_NODE:.1%} of out-of-sample next-5-day volatility against {TD_R2_BASE:.1%} for trailing volatility alone.")
print(TD_SUMMARY["reading"])

## Step 18e — Volume and open interest together: what they do to price

**Question.** Volume says how much traded; open interest (OI) says whether positions were opened or closed. Together they separate four kinds of price move that look identical in a price chart:

| Price | OI | Reading | Volume high / low |
|---|---|---|---|
| up | up | **new longs** | conviction / thin |
| up | down | **short covering** | forced / drift |
| down | up | **new shorts** | conviction / thin |
| down | down | **long liquidation** | forced / drift |

**Variables** (all known at the close of day $t$): $z_t$ ex-ante vol-scaled return (Step 3); $\tilde v_t$ log-volume z-score against the trailing 60 days (Step 18c); $\tilde o_t=\Delta\ln OI_t\,/\,s^{(60)}_{t-1}$, the OI change in units of its own trailing volatility (clipped at ±5, which also tames contract-roll jumps); and the same over 5 days. Labels: $z_{t+1}$ and $\sum_{k=1}^{5}z_{t+k}$.

**What is measured**

1. *State table.* The 8 states (4 above × high/low volume) with mean next-day and next-5-day $z$, date-clustered $t$, share of days, by train / validation / test.
2. *Pooled regression* $z_{t+1}=a+b_1z+b_2\tilde o+b_3\tilde v+b_4 z\tilde o+b_5 z\tilde v+b_6\tilde o\tilde v+b_7 z\tilde o\tilde v+\varepsilon$ (date-clustered). $b_4>0$ means moves that build OI continue; $b_5>0$ means moves on high volume continue. A second regression explains the *size* of the move, $|z_t|$, by $\tilde v,|\tilde o|$ and their product (the contemporaneous price impact).
3. *Directional scores*, ranked across products: new positioning $NP=\mathrm{sgn}(z)\max(\tilde o,0)$, unwinding $UW=\mathrm{sgn}(z)\max(-\tilde o,0)$, volume pressure $VP=\mathrm{sgn}(z)\max(\tilde v,0)$; rank IC vs both labels, Benjamini–Hochberg on train.
4. *Product consistency.* Do products agree on a state's behaviour, and does a product's behaviour in train carry to test?

**How the test period is used.** Behaviour is *decided on train and validation only*: a state is called **continuation** if its next-5-day mean has the same sign as its price move in both periods, **reversal** if the opposite sign in both, otherwise **no edge**. The test period is then used once, as the descriptive study you asked for and as the check: a call is *confirmed* only if the test mean has the same sign and $|t|>2$. Nothing in the call is changed after looking at test.

**Limits.** OI on a continuous futures series jumps at rolls; the clip limits but does not remove that. Daily states overlap in time (a 5-day state shares days with its neighbours), so inference uses clustered errors and non-overlapping 5-day labels.

In [ ]:
# step 18e
# ---- 18e.1 volume x open interest features and the 8 states ----
OV_NAMES = {1: "new longs", 2: "short covering", 3: "new shorts", 4: "long liquidation"}
OV_DIR = {1: 1, 2: 1, 3: -1, 4: -1}                                     # direction of the price move that defines the state
def ov_feats(Zr, VZ, OI, h=1):
    lo = np.log(OI.where(OI > 0)); d = lo.diff(); sd = d.rolling(60, min_periods=30).std().shift(1)
    oz = (d / sd).clip(-5, 5)
    zz = Zr.rolling(h).sum() / np.sqrt(h) if h > 1 else Zr
    ozh = oz.rolling(h).sum() / np.sqrt(h) if h > 1 else oz
    vzh = VZ.rolling(h).mean() if h > 1 else VZ
    m = zz.notna() & ozh.notna() & vzh.notna()
    zz, ozh, vzh = zz.where(m), ozh.where(m), vzh.where(m)
    price_up, oi_up = zz > 0, ozh > 0
    st = (np.where(price_up, np.where(oi_up, 1, 2), np.where(oi_up, 3, 4)) * 1.0)
    st = pd.DataFrame(st, index=zz.index, columns=zz.columns).where(m)
    hv = (vzh > 0).where(m)
    return dict(z=zz, o=ozh, v=vzh, state=st, hv=hv, np_=np.sign(zz) * ozh.clip(lower=0), uw=np.sign(zz) * (-ozh).clip(lower=0), vp=np.sign(zz) * vzh.clip(lower=0))
OV1 = ov_feats(Z.reindex(columns=UNIV), PVG["vz"].reindex(columns=UNIV), OIW.reindex(index=Z.index, columns=UNIV), h=1)
OV5 = ov_feats(Z.reindex(columns=UNIV), PVG["vz"].reindex(columns=UNIV), OIW.reindex(index=Z.index, columns=UNIV), h=5)
OV_IDX = {sp: Z.index.intersection(SPL[sp]) for sp in ("train", "val", "test")}
OV_LAB = {"fwd1": Z.shift(-1).reindex(columns=UNIV), "fwd5": Z.rolling(5).sum().shift(-5).reindex(columns=UNIV)}
print("coverage of open interest (share of product-days with a valid OI change):", f"{float(OV1['o'].notna().mean().mean()):.0%}")
# the state of day t uses only data to t; a truncation test on the OI features
causal_test("ov_feats", lambda a, b, c: {k: v for k, v in ov_feats(a, b, c, h=5).items() if k in ("z", "o", "state")}, Z.reindex(columns=UNIV).iloc[-900:], PVG["vz"].reindex(columns=UNIV).iloc[-900:], OIW.reindex(index=Z.index, columns=UNIV).iloc[-900:], step="18e")

In [ ]:
# ---- 18e.2 state table, decisions on train+val, check on test; regressions; directional scores ----
def ov_cmean(y, dates):
    y = np.asarray(y, float); n = len(y)
    if n < 30: return np.nan, np.nan
    m = y.mean(); s = pd.Series(y - m).groupby(np.asarray(dates)).sum(); se = np.sqrt((s ** 2).sum()) / n
    return float(m), float(m / se) if se > 0 else np.nan
def ov_long(F):
    out = {}
    for k in ("z", "o", "v", "state", "hv"):
        s_ = F[k].stack(); s_.index.names = ["date", "product"]; out[k] = s_
    d_ = pd.DataFrame(out)
    for lb, Y in OV_LAB.items():
        s_ = Y.stack(); s_.index.names = ["date", "product"]; d_[lb] = s_
    d_ = d_.dropna(); d_["split"] = np.select([d_.index.get_level_values("date").isin(OV_IDX[sp]) for sp in ("train", "val", "test")], ["train", "val", "test"], "")
    return d_[d_.split != ""]
OVL = {1: ov_long(OV1), 5: ov_long(OV5)}
rows = []
for h, L in OVL.items():
    for sp in ("train", "val", "test"):
        d_ = L[L.split == sp]
        if h == 5: d_ = d_[d_.index.get_level_values("date").isin(d_.index.get_level_values("date").unique()[::5])]            # non-overlapping 5-day states
        for stt in (1, 2, 3, 4):
            for hv in (1.0, 0.0):
                x = d_[(d_.state == stt) & (d_.hv == hv)]
                m1, t1 = ov_cmean(x.fwd1, x.index.get_level_values("date")); m5, t5 = ov_cmean(x.fwd5, x.index.get_level_values("date"))
                rows.append(dict(horizon=h, split=sp, state=stt, name=OV_NAMES[stt], volume="high" if hv else "low", n=len(x), share=len(x) / max(len(d_), 1), mean_fwd1=m1, t_fwd1=t1, mean_fwd5=m5, t_fwd5=t5, price_dir=OV_DIR[stt]))
OV_TAB = pd.DataFrame(rows); OV_TAB.round(4).to_csv(OUT2 / "oi_volume_states.csv", index=False)
# decisions use train and validation only
dec = []
for h in (1, 5):
    for (stt, vol), g in OV_TAB[OV_TAB.horizon == h].groupby(["state", "volume"]):
        g = g.set_index("split"); sgn = lambda v: np.sign(v) * OV_DIR[stt]                      # +1: moved the way the price had moved
        a, b = sgn(g.loc["train", "mean_fwd5"]), sgn(g.loc["val", "mean_fwd5"])
        call = "continuation" if (a > 0 and b > 0) else "reversal" if (a < 0 and b < 0) else "no edge"
        te = g.loc["test"]; ts = sgn(te.mean_fwd5)
        conf = (call == "continuation" and ts > 0 and abs(te.t_fwd5) > 2) or (call == "reversal" and ts < 0 and abs(te.t_fwd5) > 2)
        dec.append(dict(horizon=h, state=OV_NAMES[stt], volume=vol, call=call, train=g.loc["train", "mean_fwd5"], val=g.loc["val", "mean_fwd5"], test=te.mean_fwd5, t_test=te.t_fwd5, n_test=int(te.n), confirmed_on_test=bool(conf) if call != "no edge" else None))
OV_DEC = pd.DataFrame(dec); OV_DEC.round(4).to_csv(OUT2 / "oi_volume_decisions.csv", index=False)
print("Next-5-day z by state. 'call' is decided on train and validation only; confirmed_on_test = same sign and |t| > 2 on the test period:")
print(OV_DEC.round(3).to_string(index=False))
print("\nTest-period profile of the 1-day states (share of days, next-day z, t):")
print(OV_TAB[(OV_TAB.horizon == 1) & (OV_TAB.split == "test")][["name", "volume", "n", "share", "mean_fwd1", "t_fwd1"]].round(4).to_string(index=False))

# pooled regressions, date-clustered
rows = []
for sp in ("train", "val", "test"):
    d_ = OVL[1][OVL[1].split == sp].copy()
    if len(d_) < 1000: continue
    g_ = pd.factorize(d_.index.get_level_values("date"))[0]; d_["zo"] = d_.z * d_.o; d_["zv"] = d_.z * d_.v; d_["ov"] = d_.o * d_.v; d_["zov"] = d_.z * d_.o * d_.v
    X = sm.add_constant(d_[["z", "o", "v", "zo", "zv", "ov", "zov"]]); r_ = sm.OLS(d_.fwd1.to_numpy(float), X.to_numpy(float)).fit(cov_type="cluster", cov_kwds=dict(groups=g_))
    row = dict(model="next-day z", split=sp, n=len(d_)); row.update({f"b_{c}": r_.params[i] for i, c in enumerate(X.columns)}); row.update({f"t_{c}": r_.tvalues[i] for i, c in enumerate(X.columns)}); rows.append(row)
    d2 = pd.DataFrame({"y": d_.z.abs(), "v": d_.v, "ao": d_.o.abs()}); d2["vao"] = d2.v * d2.ao
    r2 = sm.OLS(d2.y.to_numpy(float), sm.add_constant(d2[["v", "ao", "vao"]]).to_numpy(float)).fit(cov_type="cluster", cov_kwds=dict(groups=g_))
    rows.append(dict(model="|z| (size of move)", split=sp, n=len(d2), **{f"b_{c}": r2.params[i] for i, c in enumerate(["const", "v", "ao", "vao"])}, **{f"t_{c}": r2.tvalues[i] for i, c in enumerate(["const", "v", "ao", "vao"])}))
OV_REG = pd.DataFrame(rows); OV_REG.round(4).to_csv(OUT2 / "oi_volume_regression.csv", index=False)
print("\nPooled regressions (date-clustered t). zo = price x OI change, zv = price x volume, zov = all three:")
print(OV_REG.round(3).to_string(index=False))

# directional scores: cross-sectional rank IC with BH on train
rows = []
for h, F in ((1, OV1), (5, OV5)):
    for nm, k in (("new positioning", "np_"), ("unwinding", "uw"), ("volume pressure", "vp")):
        for lb, Y in OV_LAB.items():
            ic = xs_ic_fast(F[k], Y, min_n=5)
            for sp, ix in OV_IDX.items():
                m_, t_ = mean_t(ic.reindex(ix), L=10 if (lb == "fwd5" or h == 5) else 5); rows.append(dict(score=nm, horizon=h, label=lb, split=sp, ic=m_, t=t_))
OV_IC = pd.DataFrame(rows).pivot_table(index=["score", "horizon", "label"], columns="split", values=["ic", "t"]); OV_IC.columns = [f"{a}_{b}" for a, b in OV_IC.columns]
OV_IC = OV_IC.reindex(columns=[f"{a}_{b}" for a in ("ic", "t") for b in ("train", "val", "test")])
p_ = 2 * (1 - stats.norm.cdf(OV_IC["t_train"].abs())); ok_ = np.isfinite(p_); q_ = np.full(len(p_), np.nan)
if ok_.any():
    o_ = np.argsort(p_[ok_]); pp = p_[ok_][o_]; m_ = len(pp); adj = np.minimum.accumulate((pp * m_ / np.arange(1, m_ + 1))[::-1])[::-1]; tmp = np.empty(m_); tmp[o_] = np.clip(adj, 0, 1); q_[ok_] = tmp
OV_IC["q_train"] = q_
OV_IC["confirmed"] = (OV_IC.q_train < 0.10) & (np.sign(OV_IC.ic_train) == np.sign(OV_IC.ic_val)) & (np.sign(OV_IC.ic_train) == np.sign(OV_IC.ic_test)) & (OV_IC.t_test.abs() > 2)
OV_IC.round(4).to_csv(OUT2 / "oi_volume_ic.csv"); print("\nDirectional scores, rank IC across products:"); print(OV_IC.round(3).to_string())

In [ ]:
# ---- 18e.3 product consistency, figures, summary ----
# do products agree? per product mean next-5-day z by 1-day state; train vs test
cons = []
Ld = OVL[1]
for stt in (1, 2, 3, 4):
    mt = {sp: Ld[(Ld.split == sp) & (Ld.state == stt)].groupby(level="product").fwd5.mean() for sp in ("train", "val", "test")}
    com = mt["train"].index.intersection(mt["test"].index).intersection(mt["val"].index)
    if len(com) >= 8: cons.append(dict(state=OV_NAMES[stt], products=len(com), sign_agree_train_test=float((np.sign(mt["train"][com]) == np.sign(mt["test"][com])).mean()), rank_corr_train_test=float(stats.spearmanr(mt["train"][com], mt["test"][com])[0]), share_continuing_test=float((np.sign(mt["test"][com]) == OV_DIR[stt]).mean())))
OV_CONS = pd.DataFrame(cons); OV_CONS.round(3).to_csv(OUT2 / "oi_volume_consistency.csv", index=False)
print("Across products (1-day states, next-5-day z): agreement of the train and test sign, rank correlation of the per-product means, and share of products continuing in test:")
print(OV_CONS.round(3).to_string(index=False) if len(OV_CONS) else "too few products")

# figure 1: next-5-day z by state and split
fig, axs = plt.subplots(1, 2, figsize=(15, 5)); lab = [f"{OV_NAMES[s]}\n{v} vol" for s in (1, 2, 3, 4) for v in ("high", "low")]
for ax, h in zip(axs, (1, 5)):
    T_ = OV_TAB[OV_TAB.horizon == h]
    for k, (sp, col) in enumerate(zip(("train", "val", "test"), SERIES)):
        g = T_[T_.split == sp].set_index(["state", "volume"]).reindex([(s, v) for s in (1, 2, 3, 4) for v in ("high", "low")])
        ax.bar(np.arange(8) + (k - 1) * .27, g.mean_fwd5.to_numpy(), width=.27, color=col, label=sp, yerr=np.abs(g.mean_fwd5.to_numpy() / np.where(np.abs(g.t_fwd5.to_numpy()) > 0, g.t_fwd5.to_numpy(), np.nan)), error_kw=dict(lw=.8, ecolor=INK2))
    ax.axhline(0, color="#c9c7c2"); ax.set_xticks(range(8)); ax.set_xticklabels(lab, fontsize=7); ax.legend(fontsize=7); ax.set_title(f"Next-5-day z after each {h}-day state (bars ± 1 standard error)", fontsize=10)
plt.tight_layout(); savefig(fig, "18e_state_returns", "What each volume x open-interest state does to price", "18e")
# figure 2: price x OI map, high vs low volume, train vs test (bins frozen on train)
trn = OVL[1][OVL[1].split == "train"]; qz = trn.z.quantile(np.linspace(0, 1, 6)).to_numpy().copy(); qo = trn.o.quantile(np.linspace(0, 1, 6)).to_numpy().copy(); qz[0] = qo[0] = -np.inf; qz[-1] = qo[-1] = np.inf
fig, axs = plt.subplots(2, 2, figsize=(11, 9)); vmax = 0.0; mats = {}
for sp in ("train", "test"):
    for hv in (1.0, 0.0):
        d_ = OVL[1][(OVL[1].split == sp) & (OVL[1].hv == hv)]; bz = pd.cut(d_.z, qz, labels=False); bo = pd.cut(d_.o, qo, labels=False)
        mats[(sp, hv)] = d_.groupby([bz, bo]).fwd1.mean().unstack().reindex(index=range(5), columns=range(5)); vmax = max(vmax, np.nanmax(np.abs(mats[(sp, hv)].to_numpy())))
for ax, ((sp, hv), M) in zip(axs.ravel(), mats.items()):
    im = ax.imshow(M.to_numpy(), origin="lower", cmap=DIV, vmin=-vmax, vmax=vmax, aspect="auto")
    for i in range(5):
        for j in range(5):
            if np.isfinite(M.iat[i, j]): ax.text(j, i, f"{M.iat[i, j]:+.2f}", ha="center", va="center", fontsize=8)
    ax.set_xticks(range(5)); ax.set_yticks(range(5)); ax.set_xticklabels(["OI ↓↓", "OI ↓", "OI ~", "OI ↑", "OI ↑↑"], fontsize=8); ax.set_yticklabels(["price ↓↓", "price ↓", "~", "price ↑", "price ↑↑"], fontsize=8)
    ax.set_title(f"{sp}, {'high' if hv else 'low'}-volume days: next-day z", fontsize=10)
plt.tight_layout(); savefig(fig, "18e_price_oi_map", "Next-day return by price move and OI change, high vs low volume", "18e")

def _r(model, sp, c):
    g = OV_REG[(OV_REG.model == model) & (OV_REG.split == sp)]; return (float(g[f"b_{c}"].iloc[0]), float(g[f"t_{c}"].iloc[0])) if len(g) else (np.nan, np.nan)
cf = OV_DEC[OV_DEC.confirmed_on_test == True]; called = OV_DEC[OV_DEC.call != "no edge"]
OV_SUMMARY = dict(n_products=int(OV1["o"].notna().any().sum()), states_called=int(len(called)), states_confirmed=int(len(cf)),
                  confirmed=[f"{r.state} ({r.volume} volume, {r.horizon}-day): {r.call}" for r in cf.itertuples()],
                  oi_confirmation=dict({sp: dict(b=_r("next-day z", sp, "zo")[0], t=_r("next-day z", sp, "zo")[1]) for sp in ("train", "val", "test")}),
                  volume_confirmation=dict({sp: dict(b=_r("next-day z", sp, "zv")[0], t=_r("next-day z", sp, "zv")[1]) for sp in ("train", "val", "test")}),
                  size_of_move=dict({sp: dict(b_interaction=_r("|z| (size of move)", sp, "vao")[0], t=_r("|z| (size of move)", sp, "vao")[1]) for sp in ("train", "val", "test")}),
                  scores_confirmed=[f"{a} ({b}-day) -> {c}" for a, b, c in OV_IC.index[OV_IC.confirmed.fillna(False)]],
                  consistency=OV_CONS.to_dict("records") if len(OV_CONS) else [])
_oc = OV_SUMMARY["oi_confirmation"]; _vc = OV_SUMMARY["volume_confirmation"]; _sz = OV_SUMMARY["size_of_move"]
OV_SUMMARY["reading"] = (f"Of {len(called)} state calls made on train and validation, {len(cf)} are confirmed on the test period (same sign, |t|>2): {', '.join(OV_SUMMARY['confirmed']) if OV_SUMMARY['confirmed'] else 'none'}. "
    f"Price x OI coefficient on next-day z: train {_oc['train']['b']:+.3f} (t={_oc['train']['t']:.1f}), val {_oc['val']['b']:+.3f} (t={_oc['val']['t']:.1f}), test {_oc['test']['b']:+.3f} (t={_oc['test']['t']:.1f}); "
    f"price x volume: train {_vc['train']['b']:+.3f}, val {_vc['val']['b']:+.3f}, test {_vc['test']['b']:+.3f}. "
    f"Volume and OI together on the size of the move (interaction): train t={_sz['train']['t']:.1f}, val t={_sz['val']['t']:.1f}, test t={_sz['test']['t']:.1f}. "
    f"Directional scores confirmed across products: {', '.join(OV_SUMMARY['scores_confirmed']) if OV_SUMMARY['scores_confirmed'] else 'none'}.")
print("\n" + OV_SUMMARY["reading"])

## Step 18f — Trend phases with volume and open interest: before, during, after (rise and fall separately)

**What 18e did and did not do.** Step 18e classified each day (or 5-day window) into a price × OI × volume state. That covers only part of the rule set you pasted (new longs, short covering, new shorts, long liquidation). It did not follow a *trend episode* through its three phases, and it did not test the coil, the exhaustion divergence, or the Vol/OI spike. This step does, using the pasted rules (saved as the skill `volume-oi-momentum-phases`) as pre-registered hypotheses with the expected signs fixed in advance.

**Two kinds of evidence, kept apart.**

*A. Event study (labelled with hindsight, descriptive).* Per product, take the cumulative vol-scaled return $c_t=\sum z$ and find swings with a zigzag of threshold $H=4$ (σ units). Every swing from a low pivot to the next high pivot is a **rise** episode, high to low a **fall** episode; keep episodes lasting 5–40 days. Each episode is laid out on one time axis: 10 days **before** the onset, the episode itself in 5 progress bins (**during**), 10 days **after** the end. *Coil present* means the 10 days before the onset moved less than $0.5H$ (a consolidation, not the tail of the previous leg). Features, all standardised against trailing history and shifted by one day:

| Feature | Definition | Rule expects |
|---|---|---|
| `rng` | $\ln\big(\sqrt{RV_t}\,/\,\mathrm{median}_{60}\big)$, range/spread proxy | before: down (narrowing); during: up |
| `vz` | log-volume z (18c) | before: down (dries up); during: up |
| `oz` | $\Delta\ln OI$ in units of its trailing volatility | before: up or flat; **during: up (key)**; **after: down** |
| `evr` | effort-versus-result: $\tilde v_t-\mathrm{z}(|z_t|)$ | after: up (volume without progress) |
| `voz` | $\ln(V/OI)$ against its own trailing mean/sd | spikes mark repricing days |

OHLC bars are not in the panel, so wicks and closes-off-lows are not measured; `rng` and `evr` stand in for them. During a rush, range and volume rise by construction because the episode is defined by big moves; the informative parts are `oz`, the before phase and the after phase.

*B. Causal rule tests (tradable information only).* At every date, trailing features only; the rule's expected sign comes from the file; outcome is the next 5 (or 10) days:

| Rule | Condition | Outcome | Expected |
|---|---|---|---|
| 1 Trend fuel | strong 5-day move, OI up and volume up | continuation of the move | better than other strong moves |
| 2 Exhaustion divergence | new 20-day extreme in the move's direction, OI falling | continuation | worse than with OI rising |
| 3a Short-covering trap | strong rise, volume high, OI falling | next-5-day return | lower than other strong rises |
| 3b Long-liquidation flush | strong fall, volume high, OI falling | next-5-day return | higher than other strong falls |
| 4 Vol/OI spike | $\ln(V/OI)$ in its top decile (threshold from train) | size of the next-5-day move and continuation of today's move | larger / continues |
| 5 Coil → rush | 10-day range compressed (bottom 30% of train), volume dry, OI creeping up | size of the next-10-day move; direction = sign of the coil's net move | larger / direction agrees |

A hypothesis counts as **supported** only if the expected sign holds in train, validation *and* test and $|t|>2$ on test (errors clustered by onset date or by date). Rises and falls are always reported separately. Thresholds that are data-driven (Vol/OI decile, range percentile) use the training period only. Note that the file's Vol/OI threshold of 0.5 is not used: in Chinese futures daily turnover is often above 1, so each product's own trailing percentile is used.

In [ ]:
# step 18f
assert "OV1" in globals() and "ov_cmean" in globals(), "Step 18f needs the variables of Step 18e (and 18c): run the Step 18e cells first"
# ---- 18f.1 trend episodes (zigzag, hindsight) and event-time features ----
F_P, F_A, F_NB, F_H, F_MINDUR, F_MAXDUR, F_COIL = 10, 10, 5, 4.0, 5, 40, 0.5
def zz_pivots(c, H):
    n = len(c); piv = []; trend = 0; hi = lo = 0
    for t in range(1, n):
        if trend == 0:
            if c[t] > c[hi]: hi = t
            if c[t] < c[lo]: lo = t
            if c[t] - c[lo] >= H: piv.append((lo, -1)); trend = 1; hi = t
            elif c[hi] - c[t] >= H: piv.append((hi, 1)); trend = -1; lo = t
        elif trend == 1:
            if c[t] > c[hi]: hi = t
            elif c[hi] - c[t] >= H: piv.append((hi, 1)); trend = -1; lo = t
        else:
            if c[t] < c[lo]: lo = t
            elif c[t] - c[lo] >= H: piv.append((lo, -1)); trend = 1; hi = t
    return piv
_c = np.array([0, 1, 2, 3, 4, 3, 2, 1, 0, 1, 2, 3, 4, 5, 4, 3, 2, 1, 0, 1, 2.0]); assert zz_pivots(_c, 3.0) == [(0, -1), (4, 1), (8, -1), (13, 1)], zz_pivots(_c, 3.0)   # known swings are recovered

Zu = Z.reindex(columns=UNIV)
_rv = np.sqrt(RV.reindex(index=Z.index, columns=UNIV).where(RV.reindex(index=Z.index, columns=UNIV) > 0))
_az = Zu.abs(); _azz = ((_az - _az.rolling(60, min_periods=30).mean().shift(1)) / _az.rolling(60, min_periods=30).std().shift(1)).clip(-5, 5)
_vo = np.log((VOLU.reindex(index=Z.index, columns=UNIV).where(VOLU > 0)) / OIW.reindex(index=Z.index, columns=UNIV).where(OIW > 0))
OF = dict(rng=np.log(_rv / _rv.rolling(60, min_periods=30).median().shift(1)), vz=PVG["vz"].reindex(columns=UNIV), oz=OV1["o"], evr=PVG["vz"].reindex(columns=UNIV) - _azz,
          voz=(_vo - _vo.rolling(250, min_periods=60).mean().shift(1)) / _vo.rolling(250, min_periods=60).std().shift(1))
OF_FEATS = list(OF)
_lab = np.full(len(Z), "", dtype=object)
for _sp in ("train", "val", "test"): _lab[Z.index.isin(SPL[_sp])] = _sp
def _nm(x): return float(np.nanmean(x)) if np.isfinite(x).any() else np.nan
rows, bins = [], {f: [] for f in OF_FEATS}
FA = {f: OF[f].to_numpy(float) for f in OF_FEATS}
for j, c in enumerate(UNIV):
    z = Zu[c].to_numpy(float); cum = np.cumsum(np.where(np.isfinite(z), z, 0.0)); piv = zz_pivots(cum, F_H)
    for (i0, k0), (i1, k1) in zip(piv[:-1], piv[1:]):
        d = i1 - i0
        if d < F_MINDUR or d > F_MAXDUR or i0 - F_P + 1 < 0 or i1 + F_A >= len(z): continue
        if _lab[i0 - F_P + 1] == "" or _lab[i0 - F_P + 1] != _lab[i1 + F_A]: continue
        path = np.abs(z[i0 + 1:i1 + 1]).sum(); net = cum[i1] - cum[i0]; pre_net = cum[i0] - cum[i0 - F_P]
        rows.append(dict(product=c, dir="rise" if k0 == -1 else "fall", onset=Z.index[i0], end=Z.index[i1], dur=d, net=net, eff=abs(net) / path if path > 0 else np.nan, coil=bool(abs(pre_net) < F_COIL * F_H), split=_lab[i0 - F_P + 1]))
        idx = (np.arange(d) * F_NB) // d
        for f in OF_FEATS:
            a = FA[f][:, j]; dur = a[i0 + 1:i1 + 1]
            bins[f].append(np.concatenate([a[i0 - F_P + 1:i0 + 1], [_nm(dur[idx == b]) for b in range(F_NB)], a[i1 + 1:i1 + 1 + F_A]]))
OF_EP = pd.DataFrame(rows); OF_BIN = {f: np.array(v) for f, v in bins.items()}
OF_EP.to_csv(OUT2 / "trend_episodes.csv", index=False)
print(f"{len(OF_EP)} episodes (zigzag threshold {F_H} sigma, {F_MINDUR}-{F_MAXDUR} days):")
print(OF_EP.groupby(["dir", "split"]).agg(n=("dur", "size"), mean_days=("dur", "mean"), mean_move=("net", "mean"), efficiency=("eff", "mean"), coil_share=("coil", "mean")).round(2).to_string())

In [ ]:
# ---- 18f.2 event study: what each phase looks like, tested against the sign the rules expect ----
OF_SL = {"pre": slice(0, F_P), "during": slice(F_P, F_P + F_NB), "after": slice(F_P + F_NB, F_P + F_NB + F_A)}
OF_EXP = {("pre", "vz"): -1, ("pre", "rng"): -1, ("pre", "oz"): +1, ("during", "vz"): +1, ("during", "rng"): +1, ("during", "oz"): +1, ("after", "oz"): -1, ("after", "evr"): +1}
rows = []
for d_ in ("rise", "fall"):
    for (ph, f), ex in OF_EXP.items():
        for sub in (("all", "coil") if ph == "pre" else ("all",)):
            for sp in ("train", "val", "test"):
                m = ((OF_EP.dir == d_) & (OF_EP.split == sp) & (OF_EP.coil if sub == "coil" else True)).to_numpy()
                v = np.array([_nm(r) for r in OF_BIN[f][m][:, OF_SL[ph]]]) if m.any() else np.array([])
                ok = np.isfinite(v)
                mean_, t_ = ov_cmean(v[ok], OF_EP.onset[m].dt.strftime("%Y-%m-%d").to_numpy()[ok]) if ok.sum() >= 15 else (np.nan, np.nan)
                rows.append(dict(dir=d_, phase=ph if sub == "all" else "pre (coil only)", feature=f, expected=ex, split=sp, n=int(ok.sum()), mean=mean_, t=t_))
OF_HYP = pd.DataFrame(rows); OFH = OF_HYP.pivot_table(index=["dir", "phase", "feature", "expected"], columns="split", values=["mean", "t", "n"]); OFH.columns = [f"{a}_{b}" for a, b in OFH.columns]
OFH = OFH.reindex(columns=[f"{a}_{b}" for a in ("mean", "t", "n") for b in ("train", "val", "test")]).reset_index()
OFH["supported"] = [bool(np.sign(r.mean_train) == r.expected and np.sign(r.mean_val) == r.expected and np.sign(r.mean_test) == r.expected and abs(r.t_test) > 2) for r in OFH.itertuples()]
OFH.round(3).to_csv(OUT2 / "trend_phase_hypotheses.csv", index=False)
print("Event study. 'expected' is the sign the pasted rules predict (decided before looking); supported = that sign in train, val and test and |t_test| > 2 (errors clustered by onset date):")
print(OFH[["dir", "phase", "feature", "expected", "mean_train", "mean_val", "mean_test", "t_test", "n_test", "supported"]].round(3).to_string(index=False))
# the share of episodes preceded by a coil, and what the OI does during the move
cs = OF_EP.groupby(["dir", "split"]).coil.mean().unstack().round(2); print("\nShare of episodes preceded by a coil (10 days that moved less than half the zigzag threshold):\n", cs.to_string())
OFD = OF_EP.copy(); OFD["oi_during_up"] = [bool(_nm(OF_BIN["oz"][i][OF_SL["during"]]) > 0) for i in range(len(OFD))]
print("\nShare of episodes whose OI rose during the move ('trend fuel'):\n", OFD.groupby(["dir", "split"]).oi_during_up.mean().unstack().round(2).to_string())

In [ ]:
# ---- 18f.3 causal rule tests: trailing features at each date, forward outcome ----
c_ = Zu.fillna(0).cumsum(); hi20, lo20 = c_.rolling(20).max().shift(1), c_.rolling(20).min().shift(1)
z5, o5, v5 = OV5["z"], OV5["o"], OV5["v"]; strong = z5.abs() >= 1.3                                                   # |5-day sum| >= 2.9 sigma units
F5, F1, F10 = OV_LAB["fwd5"], OV_LAB["fwd1"], Zu.rolling(10).sum().shift(-10)
rng10, vz10, oz10, net10 = OF["rng"].rolling(10).mean(), OF["vz"].rolling(10).mean(), OF["oz"].rolling(10).mean(), Zu.rolling(10).sum()
_tr = Zu.index.intersection(SPL["train"]); thr_voz = float(OF["voz"].loc[_tr].stack().quantile(0.90)); thr_rng = float(rng10.loc[_tr].stack().quantile(0.30))
dir5 = np.sign(z5); ext = ((c_ > hi20) & (z5 > 0)) | ((c_ < lo20) & (z5 < 0))
spl_dates = {sp: Zu.index.intersection(SPL[sp])[::5] for sp in ("train", "val", "test")}                              # every 5th day: non-overlapping labels
RULES = [
 ("1 trend fuel", "continuation of a strong move", strong, (o5 > 0.5) & (v5 > 0), dir5 * F5, dir5, +1, None),
 ("2 exhaustion divergence", "continuation at a new 20-day extreme", ext & strong, o5 < 0, dir5 * F5, dir5, -1, None),
 ("3a short-covering trap", "next-5-day return after a strong rise", strong & (z5 > 0), (v5 > 0.5) & (o5 < -0.5), F5, dir5, -1, 1),
 ("3b long-liquidation flush", "next-5-day return after a strong fall", strong & (z5 < 0), (v5 > 0.5) & (o5 < -0.5), F5, dir5, +1, -1),
 ("4 Vol/OI spike (size)", "size of the next-5-day move", OF["voz"].notna(), OF["voz"] > thr_voz, F5.abs(), np.sign(Zu), +1, None),
 ("4 Vol/OI spike (direction)", "continuation of today's move", OF["voz"].notna(), OF["voz"] > thr_voz, np.sign(Zu) * F1, np.sign(Zu), +1, None),
 ("5 coil (size)", "size of the next-10-day move", rng10.notna() & vz10.notna() & oz10.notna(), (rng10 < thr_rng) & (vz10 < 0) & (oz10 > 0), F10.abs(), np.sign(net10), +1, None),
 ("5 coil (direction)", "move continues the coil's net direction", rng10.notna() & vz10.notna() & oz10.notna(), (rng10 < thr_rng) & (vz10 < 0) & (oz10 > 0), np.sign(net10) * F10, np.sign(net10), +1, None)]
rows = []
for nm, outc, uni, flag, y, dr, ex, only in RULES:
    for dname, dval in (("rise", 1), ("fall", -1)):
        if only is not None and only != dval: continue
        for sp, ds in spl_dates.items():
            U = (uni & (dr == dval)).reindex(index=ds).stack(); U = U[U.fillna(False).astype(bool)].index
            if len(U) < 200: rows.append(dict(rule=nm, dir=dname, split=sp, n_flag=0, diff=np.nan, t=np.nan, expected=ex)); continue
            Y = y.reindex(index=ds).stack().reindex(U); Fg = flag.reindex(index=ds).stack().reindex(U).astype(float); d_ = pd.DataFrame({"y": Y, "f": Fg}).dropna()
            if (d_.f == 1).sum() < 30 or (d_.f == 0).sum() < 30: rows.append(dict(rule=nm, dir=dname, split=sp, n_flag=int((d_.f == 1).sum()), diff=np.nan, t=np.nan, expected=ex)); continue
            g_ = pd.factorize(d_.index.get_level_values(0))[0]; r_ = sm.OLS(d_.y.to_numpy(float), sm.add_constant(d_.f.to_numpy(float))).fit(cov_type="cluster", cov_kwds=dict(groups=g_))
            rows.append(dict(rule=nm, dir=dname, split=sp, n_flag=int((d_.f == 1).sum()), share_flag=float(d_.f.mean()), mean_flag=float(d_.y[d_.f == 1].mean()), mean_other=float(d_.y[d_.f == 0].mean()), diff=float(r_.params[1]), t=float(r_.tvalues[1]), expected=ex))
OF_RULES = pd.DataFrame(rows)
W = OF_RULES.pivot_table(index=["rule", "dir", "expected"], columns="split", values=["diff", "t", "n_flag", "mean_flag", "mean_other"]); W.columns = [f"{a}_{b}" for a, b in W.columns]; W = W.reset_index()
for c in ("diff", "t", "n_flag"):
    for sp in ("train", "val", "test"):
        if f"{c}_{sp}" not in W: W[f"{c}_{sp}"] = np.nan
W["supported"] = [bool(all(np.sign(getattr(r, f"diff_{sp}")) == r.expected for sp in ("train", "val", "test")) and abs(r.t_test) > 2) for r in W.itertuples()]
W.round(4).to_csv(OUT2 / "trend_rule_tests.csv", index=False)
print(f"Causal rule tests (Vol/OI spike threshold {thr_voz:.2f} and coil range threshold {thr_rng:.2f} come from train). diff = flagged minus other days; expected = sign the pasted rule predicts:")
print(W[["rule", "dir", "expected", "diff_train", "diff_val", "diff_test", "t_test", "n_flag_test", "supported"]].round(3).to_string(index=False))

In [ ]:
# ---- 18f.4 figures and summary ----
xs_ = np.arange(F_P + F_NB + F_A); lab_x = [f"{k}" for k in range(-F_P + 1, 1)] + [f"{(b + .5) / F_NB:.0%}" for b in range(F_NB)] + [f"+{k}" for k in range(1, F_A + 1)]
fig, axs = plt.subplots(2, 4, figsize=(18, 7.5), sharex=True)
for r_, d_ in enumerate(("rise", "fall")):
    for c_i, (f, ttl) in enumerate((("rng", "range proxy (log ratio to 60-day median)"), ("vz", "volume z"), ("oz", "OI change (z)"), ("evr", "effort versus result"))):
        ax = axs[r_, c_i]
        for sp, col in zip(("train", "val", "test"), SERIES):
            m = ((OF_EP.dir == d_) & (OF_EP.split == sp)).to_numpy()
            if m.sum() >= 5: ax.plot(xs_, np.nanmean(OF_BIN[f][m], axis=0), color=col, lw=1.6, label=f"{sp} (n={int(m.sum())})")
        ax.axhline(0, color="#c9c7c2", lw=.8); ax.axvspan(F_P - .5, F_P + F_NB - .5, color=MUTE, alpha=.12); ax.set_title(f"{d_}: {ttl}", fontsize=9)
        ax.set_xticks(xs_[::3]); ax.set_xticklabels(np.array(lab_x)[::3], fontsize=7); ax.legend(fontsize=6)
fig.suptitle("Event-time profile of rises and falls: 10 days before, the episode in 5 progress bins (shaded), 10 days after", y=1.0)
plt.tight_layout(); savefig(fig, "18f_phase_profiles", "Volume, OI and range before, during and after trend episodes", "18f")
fig, axs = plt.subplots(1, 2, figsize=(16, 5.5))
for ax, d_ in zip(axs, ("rise", "fall")):
    T_ = W[W.dir == d_].reset_index(drop=True); yy = np.arange(len(T_))
    for k, (sp, col) in enumerate(zip(("train", "val", "test"), SERIES)): ax.barh(yy + (k - 1) * .27, T_[f"diff_{sp}"] * T_.expected, height=.27, color=col, label=sp)
    ax.set_yticks(yy); ax.set_yticklabels([f"{r.rule}{'  ✓' if r.supported else ''}" for r in T_.itertuples()], fontsize=8); ax.axvline(0, color="#c9c7c2"); ax.invert_yaxis(); ax.legend(fontsize=7)
    ax.set_title(f"{d_}: effect of the rule's condition, signed so that positive = the rule's prediction (✓ = supported)", fontsize=9)
plt.tight_layout(); savefig(fig, "18f_rule_tests", "Do the volume/OI rules hold? Effect in the direction each rule predicts", "18f")

_sup = OFH[OFH.supported]; _rs = W[W.supported]
OF_SUMMARY = dict(episodes=int(len(OF_EP)), by_dir=OF_EP.groupby("dir").size().to_dict(), coil_share=OF_EP.groupby("dir").coil.mean().round(3).to_dict(),
                  phase_supported=[f"{r.dir} {r.phase} {r.feature} ({'+' if r.expected > 0 else '-'})" for r in _sup.itertuples()], phase_tested=int(len(OFH)),
                  rules_supported=[f"{r.rule} ({r.dir})" for r in _rs.itertuples()], rules_tested=int(len(W)),
                  rules_wrong_sign_on_test=[f"{r.rule} ({r.dir})" for r in W.itertuples() if np.sign(r.diff_test) == -r.expected and abs(r.t_test) > 2])
OF_SUMMARY["reading"] = (f"{OF_SUMMARY['episodes']} trend episodes (rises {OF_SUMMARY['by_dir'].get('rise', 0)}, falls {OF_SUMMARY['by_dir'].get('fall', 0)}). "
    f"Phase signatures supported in train, validation and test: {', '.join(OF_SUMMARY['phase_supported']) if OF_SUMMARY['phase_supported'] else 'none'} (of {OF_SUMMARY['phase_tested']} tested). "
    f"Causal rules supported: {', '.join(OF_SUMMARY['rules_supported']) if OF_SUMMARY['rules_supported'] else 'none'} (of {OF_SUMMARY['rules_tested']}). "
    f"Rules that point the opposite way on test with |t|>2: {', '.join(OF_SUMMARY['rules_wrong_sign_on_test']) if OF_SUMMARY['rules_wrong_sign_on_test'] else 'none'}. Share of episodes preceded by a coil: {OF_SUMMARY['coil_share']}.")
print("\n" + OF_SUMMARY["reading"])

## Step 18g — Quantile flow: how the distribution of returns is migrating (Probability Mass Flow Tracker and Quantile Flow Field)

**Idea.** Price moves are the visible part. Underneath, the *distribution* of returns moves first or at least separately: the centre shifts, one tail grows faster than the other, layers of the distribution migrate together or against each other. Moments (mean, skew, kurtosis) are very noisy in a 10–60-day window and their rates of change are worse, so everything here is built from **quantiles** $Q_u$ of the vol-scaled daily return $z$ in a trailing window (levels $u\in\{.05,.10,.25,.50,.75,.90,.95\}$), which are robust.

**A. Probability Mass Flow Tracker** (per product; fast window $f$, slow window $s$)

| Component | Definition |
|---|---|
| Centre | $m^{(1)}=Q_{.50}^{f}/\hat s$, with $\hat s=(Q^{s}_{.75}-Q^{s}_{.25})/1.349$ |
| Body asymmetry (Bowley) | $m^{(2)}=\dfrac{Q_{.75}+Q_{.25}-2Q_{.50}}{Q_{.75}-Q_{.25}}$ (window $f$) |
| Tail asymmetry (Kelley-type) | $m^{(3)}=\dfrac{Q_{.95}+Q_{.05}-2Q_{.50}}{Q_{.95}-Q_{.05}}$ (window $f$) |
| Moment evolution | $d^{(j)}_t=\mathrm{EMA}_5\big(m^{(j)}_t-m^{(j)}_{t-k}\big)$, $k=\max(2,\lfloor f/4\rfloor)$ |
| Flow | $F_t=\overline{\,d^{(j)}_t/\mathrm{sd}_s(d^{(j)})\,}$ (clipped at ±4); acceleration $F_t-F_{t-k}$ |
| Confidence | $C_t=\big|\sum_j \mathrm{sgn}\,d^{(j)}_t\big|/3$ (1 = all three components agree) |
| Early signal | $|F|>1$, $C=1$ and price **not yet confirming**: the 5-day return is small ($|\sum_5 z|<1.5$) or has the opposite sign |

**B. Quantile Flow Field** (same $f,s$)

| Component | Definition |
|---|---|
| Layer displacement | $d_u=(Q_u^{f}-Q_u^{s})/\hat s$ for each of the 7 layers |
| Alignment | $A_t=\frac17\sum_u \mathrm{sgn}\,d_u\in[-1,1]$; layer groups: lower tail (5, 10), body (25, 50, 75), upper tail (90, 95) |
| Conviction | $|A|\ge 5/7$: most layers migrate the same way; mixed layers = conflict |
| Multi-timeframe | share of the configurations below whose alignment has the same sign as the ensemble |

**Windows are not assumed.** Six (fast, slow) pairs are computed: (10,40), (10,60), (20,60), (20,120), (40,120), (40,250). The choice follows the rule used elsewhere in this notebook: *selected on the validation period only*, by the **worst-case information coefficient across volatility regimes** (low / mid / high), averaged with the neighbouring pairs so that a lone lucky window cannot win (plateau rule); an **equal-weight ensemble of all pairs** is reported next to it; the test period is read once; and the chosen pair's test rank among all six shows how much luck was involved. With six pairs tried, a Bonferroni-adjusted threshold ($|t|>2.64$) is used for "confirmed".

**Tests.** Cross-sectional rank IC of $F$ and $A$ against the next-1 and next-5-day returns by train/validation/test; IC after removing 20-day momentum cross-sectionally (does it add anything that momentum does not already contain?); IC on days where price has *not* confirmed versus days where it has (the "before price confirms" claim); mean forward return of early signals; lead–lag against momentum.

**Limits.** Daily data only (20–250 samples per quantile estimate). A later upgrade is to store daily quantiles of the 1-minute returns in Step 1, which would give each day hundreds of observations instead of one. The three flow indicators in this step and the Wasserstein indicator from the first reference measure related things, so their correlation should be checked before treating them as independent evidence.

In [ ]:
# step 18g
assert "ov_cmean" in globals() and "OV_LAB" in globals(), "Step 18g needs Step 18e (helpers and labels): run the Step 18e cells first"
# ---- 18g.1 quantile layers, Probability Mass Flow Tracker (PMF) and Quantile Flow Field (QFF) ----
QG_U = (0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95)
QG_CFG = [(10, 40), (10, 60), (20, 60), (20, 120), (40, 120), (40, 250)]
QG_EARLY_F, QG_CONF_PRICE = 1.0, 1.5
def qg_signals(Zr, cfgs=QG_CFG, layers=False):
    W = sorted({w for c in cfgs for w in c}); Q = {w: {u: Zr.rolling(w, min_periods=int(0.8 * w)).quantile(u) for u in QG_U} for w in W}
    out = {}
    for f, s in cfgs:
        qf, qs = Q[f], Q[s]; sh = (qs[0.75] - qs[0.25]) / 1.349
        iqr_f = qf[0.75] - qf[0.25]; k = max(2, f // 4)
        m = [qf[0.5] / sh, (qf[0.75] + qf[0.25] - 2 * qf[0.5]) / iqr_f, (qf[0.95] + qf[0.05] - 2 * qf[0.5]) / (qf[0.95] - qf[0.05])]
        d = [(x - x.shift(k)).ewm(span=5, min_periods=3).mean() for x in m]
        zc = [(x / x.rolling(s, min_periods=int(0.6 * s)).std()).clip(-4, 4) for x in d]
        F = sum(zc) / 3.0; C = abs(sum(np.sign(x) for x in zc)) / 3.0
        disp = {u: (qf[u] - qs[u]) / sh for u in QG_U}; A = sum(np.sign(disp[u]) for u in QG_U) / len(QG_U)
        out[("pmf", (f, s))] = F; out[("pmfc", (f, s))] = C; out[("qff", (f, s))] = A
        out[("qffw", (f, s))] = sum(disp.values()) / sum(abs(v) for v in disp.values())
        if layers: out[("layers", (f, s))] = disp
    return out
# unit check: a recent upward shift of the whole distribution must give positive flow and full upward alignment
_r = np.random.default_rng(1); _x = _r.normal(size=400); _x[-12:] += 1.5; _s = pd.DataFrame({"a": _x})
_o = qg_signals(_s, [(20, 60)]); assert _o[("qff", (20, 60))]["a"].iloc[-1] > 0.7 and _o[("pmf", (20, 60))]["a"].iloc[-1] > 0, "flow sign convention broken"
_x = _r.normal(size=400); _x[-12:] -= 1.5; _o = qg_signals(pd.DataFrame({"a": _x}), [(20, 60)]); assert _o[("qff", (20, 60))]["a"].iloc[-1] < -0.7 and _o[("pmf", (20, 60))]["a"].iloc[-1] < 0
print("unit checks passed: an upward shift of the distribution gives positive flow and alignment, a downward shift negative")
Zq = Z.reindex(columns=UNIV)
QGS = qg_signals(Zq)
QG_ENS = {"pmf": sum(QGS[("pmf", c)].clip(-3, 3) for c in QG_CFG) / len(QG_CFG), "qff": sum(QGS[("qff", c)] for c in QG_CFG) / len(QG_CFG)}
QG_AGREE = sum((np.sign(QGS[("qff", c)]) == np.sign(QG_ENS["qff"])).astype(float) for c in QG_CFG) / len(QG_CFG)       # multi-timeframe reconciliation
causal_test("qg_signals", lambda a: {f"{k[0]}{k[1]}": v for k, v in qg_signals(a, [(10, 40), (20, 60)]).items()}, Zq.iloc[-700:], step="18g")
print(f"computed {len(QGS)} signal frames for {len(QG_CFG)} window pairs; ensemble = equal-weight mean of all pairs")

In [ ]:
# ---- 18g.2 window selection (validation only, worst-case over volatility regimes, plateau rule) and test ----
QG_IDX = {sp: Z.index.intersection(SPL[sp]) for sp in ("train", "val", "test")}
mabs_ = Z[UNIV].abs().mean(axis=1).rolling(20).mean(); pct_ = mabs_.rolling(500, min_periods=250).apply(lambda w: (w[:-1] < w[-1]).mean(), raw=True)
QG_REG = pd.cut(pct_, [-0.01, 1 / 3, 2 / 3, 1.01], labels=["low", "mid", "high"])                                      # ex-ante market volatility regime
QG_Y = {"fwd1": OV_LAB["fwd1"], "fwd5": OV_LAB["fwd5"]}
def qg_ic(sig, lb): return xs_ic_fast(sig.reindex(columns=UNIV), QG_Y[lb], min_n=5)
rows, ICS = [], {}
sigs = {("pmf", c): QGS[("pmf", c)] for c in QG_CFG} | {("qff", c): QGS[("qff", c)] for c in QG_CFG} | {("pmf", "ens"): QG_ENS["pmf"], ("qff", "ens"): QG_ENS["qff"]}
for key, sg in sigs.items():
    for lb in QG_Y:
        ic = qg_ic(sg, lb); ICS[(key, lb)] = ic
        row = dict(indicator=key[0], cfg=str(key[1]), label=lb)
        for sp, ix in QG_IDX.items():
            m_, t_ = mean_t(ic.reindex(ix), L=10 if lb == "fwd5" else 5); row[f"ic_{sp}"] = m_; row[f"t_{sp}"] = t_
        v_ = ic.reindex(QG_IDX["val"]); rg = QG_REG.reindex(QG_IDX["val"])
        row["val_ic_worst_regime"] = min([v_[rg == r].mean() for r in ("low", "mid", "high") if (rg == r).sum() >= 40] or [np.nan]); rows.append(row)
QG_TAB = pd.DataFrame(rows)
# selection on validation only, label fwd5, plateau rule: average the worst-regime score with the neighbouring pairs
QG_CHOSEN = {}
for ind in ("pmf", "qff"):
    t5 = QG_TAB[(QG_TAB.indicator == ind) & (QG_TAB.label == "fwd5") & (QG_TAB.cfg != "ens")].set_index("cfg").val_ic_worst_regime
    sc = {}
    for c in QG_CFG:
        nb = [d for d in QG_CFG if d != c and (d[0] == c[0] or d[1] == c[1])]
        sc[c] = np.nanmean([t5[str(c)]] + [t5[str(d)] for d in nb])
    QG_CHOSEN[ind] = max(sc, key=lambda c: (-np.inf if not np.isfinite(sc[c]) else sc[c]))
QG_TAB["chosen"] = [c == str(QG_CHOSEN.get(i)) for i, c in zip(QG_TAB.indicator, QG_TAB.cfg)]
QG_TAB.round(4).to_csv(OUT2 / "quantile_flow_ic.csv", index=False)
print("Window pairs: rank IC by period (label fwd5 and fwd1); val_ic_worst_regime = the selection score; chosen = picked on validation only:")
print(QG_TAB[QG_TAB.label == "fwd5"][["indicator", "cfg", "ic_train", "ic_val", "ic_test", "t_test", "val_ic_worst_regime", "chosen"]].round(4).to_string(index=False))
print("chosen pairs (validation, worst-regime, plateau rule):", QG_CHOSEN)
BONF = 2.64                                                                                                          # |t| for alpha = 0.05 with 6 pairs tried
rows = []
for ind in ("pmf", "qff"):
    for lb in ("fwd1", "fwd5"):
        g = QG_TAB[(QG_TAB.indicator == ind) & (QG_TAB.label == lb)]; ch = g[g.chosen].iloc[0]; en = g[g.cfg == "ens"].iloc[0]; allp = g[g.cfg != "ens"]
        same = lambda r: bool(np.sign(r.ic_train) == np.sign(r.ic_val) == np.sign(r.ic_test))
        rows.append(dict(indicator=ind, label=lb, chosen=ch.cfg, ic_test_chosen=ch.ic_test, t_test_chosen=ch.t_test, chosen_rank_on_test=int((allp.ic_test * np.sign(ch.ic_test) > ch.ic_test * np.sign(ch.ic_test)).sum() + 1), pairs=len(allp),
                         ic_test_ens=en.ic_test, t_test_ens=en.t_test, confirmed_chosen=bool(same(ch) and abs(ch.t_test) > BONF), confirmed_ens=bool(same(en) and abs(en.t_test) > 2.0)))
QG_SEL = pd.DataFrame(rows); QG_SEL.round(4).to_csv(OUT2 / "quantile_flow_selection.csv", index=False)
print("\nTest-period check of the chosen pair against the ensemble. chosen_rank_on_test: 1 = best of the 6 pairs on test (a high rank only by luck would be near 1 for every run):")
print(QG_SEL.round(3).to_string(index=False))

In [ ]:
# ---- 18g.3 does it add to momentum? does it lead price? early signals; charts; summary ----
mom20 = Zq.rolling(20).sum(); price_conf = lambda sg: (np.sign(Zq.rolling(5).sum()) == np.sign(sg)) & (Zq.rolling(5).sum().abs() >= QG_CONF_PRICE)
rows = []
for ind in ("pmf", "qff"):
    c = QG_CHOSEN[ind]; sg = QGS[(ind, c)]
    for nm, s_ in (("chosen", sg), ("ensemble", QG_ENS[ind])):
        res = td_resid(s_.reindex(columns=UNIV), mom20)
        for lb in ("fwd5",):
            ic_r = xs_ic_fast(res, QG_Y[lb], min_n=5); ic_u = ICS[((ind, c) if nm == "chosen" else (ind, "ens"), lb)]
            conf = price_conf(s_); ic_c = xs_ic_fast(s_.where(conf).reindex(columns=UNIV), QG_Y[lb], min_n=5); ic_n = xs_ic_fast(s_.where(~conf).reindex(columns=UNIV), QG_Y[lb], min_n=5)
            r = dict(indicator=ind, version=nm, pair=str(c) if nm == "chosen" else "ensemble")
            for sp, ix in QG_IDX.items():
                for tag, ic_ in (("ic", ic_u), ("ic_ex_momentum", ic_r), ("ic_price_confirmed", ic_c), ("ic_price_not_confirmed", ic_n)):
                    m_, t_ = mean_t(ic_.reindex(ix), L=10); r[f"{tag}_{sp}"] = m_; r[f"t_{tag}_{sp}"] = t_
            rows.append(r)
QG_ADD = pd.DataFrame(rows); QG_ADD.round(4).to_csv(OUT2 / "quantile_flow_vs_momentum.csv", index=False)
print("Incremental value (label fwd5): IC after removing 20-day momentum, and IC split by whether price already confirms the flow direction:")
print(QG_ADD[["indicator", "version", "pair", "ic_test", "ic_ex_momentum_val", "ic_ex_momentum_test", "t_ic_ex_momentum_test", "ic_price_not_confirmed_test", "ic_price_confirmed_test"]].round(4).to_string(index=False))
# early signals of the tracker: strong, fully agreeing components, price not confirming
cp_ = QG_CHOSEN["pmf"]; F_, C_ = QGS[("pmf", cp_)], QGS[("pmfc", cp_)]
early = (F_.abs() > QG_EARLY_F) & (C_ >= 0.99) & ~price_conf(F_); strongflow = (F_.abs() > QG_EARLY_F) & (C_ >= 0.99) & price_conf(F_)
rows = []
for nm, flag in (("early (price not yet confirming)", early), ("flow with price already confirming", strongflow)):
    for sp in ("train", "val", "test"):
        ds = QG_IDX[sp][::5]; y = (np.sign(F_) * OV_LAB["fwd5"]).reindex(index=ds); fl = flag.reindex(index=ds)
        s_ = y.where(fl).stack().dropna(); m_, t_ = ov_cmean(s_.to_numpy(), s_.index.get_level_values(0)) if len(s_) >= 30 else (np.nan, np.nan)
        rows.append(dict(signal=nm, split=sp, n=int(len(s_)), mean_signed_fwd5=m_, t=t_, base=float(y.stack().dropna().mean())))
QG_EARLY = pd.DataFrame(rows); QG_EARLY.round(4).to_csv(OUT2 / "quantile_flow_early.csv", index=False)
print("\nForward 5-day return in the direction of the flow (z units) when the tracker fires; base = all days:"); print(QG_EARLY.round(3).to_string(index=False))
# lead-lag: flow vs momentum against returns k days ahead (test period, cross-sectional rank IC)
LAG = list(range(1, 11)); ll = {}
for nm, s_ in (("PMF flow", QGS[("pmf", cp_)]), ("QFF alignment", QGS[("qff", QG_CHOSEN["qff"])]), ("20-day momentum", mom20)):
    ll[nm] = [mean_t(xs_ic_fast(s_.reindex(columns=UNIV), Zq.shift(-k), min_n=5).reindex(QG_IDX["test"]), L=5)[0] for k in LAG]
QG_LL = pd.DataFrame(ll, index=LAG); QG_LL.round(4).to_csv(OUT2 / "quantile_flow_leadlag.csv")
print("\nRank IC (test) of each signal against the return k days later:"); print(QG_LL.round(4).T.to_string())

# charts: a TradingView-style view of one product over the last 250 test days
pc = [c for c in FOCUS if c in UNIV][0]; ix = QG_IDX["test"][-250:]; cq = QG_CHOSEN["qff"]; LY = qg_signals(Zq[[pc]], [cq], layers=True)[("layers", cq)]
fig, axs = plt.subplots(3, 1, figsize=(15, 9), sharex=True, gridspec_kw=dict(height_ratios=[2, 2, 1.3]))
cum = Zq[pc].fillna(0).cumsum().reindex(ix); axs[0].plot(cum, color=INK2, lw=1.2); axs[0].set_title(f"{pc}: cumulative vol-scaled return", fontsize=10)
Fv = F_[pc].reindex(ix); Cv = C_[pc].reindex(ix); GREEN, RED = "#1baf7a", "#e34948"
axs[1].fill_between(ix, 0, Fv.clip(lower=0), color=GREEN, alpha=.75, step="mid"); axs[1].fill_between(ix, 0, Fv.clip(upper=0), color=RED, alpha=.75, step="mid"); axs[1].axhline(0, color="#c9c7c2")
for d_, f_, c_ in zip(ix, Fv, Cv):
    if np.isfinite(f_) and c_ >= .99 and abs(f_) > QG_EARLY_F: axs[1].axvspan(d_, d_ + pd.Timedelta(days=1), color=GREEN if f_ > 0 else RED, alpha=.12, lw=0)
ee = early[pc].reindex(ix); axs[1].scatter(ix[ee.fillna(False).to_numpy()], np.where(Fv[ee.fillna(False)] > 0, Fv.min() * 1.05, Fv.max() * 1.05), marker="^", s=36, color=INK2, zorder=4)
axs[1].set_title(f"Probability Mass Flow Tracker, pair {cp_}: flow F (shading: all three components agree and |F|>1; triangles: early signals, price not yet confirming)", fontsize=10)
Av = QGS[("qff", cq)][pc].reindex(ix); axs[2].bar(ix, Av, color=np.where(Av >= 0, GREEN, RED), width=1.0); axs[2].axhline(5 / 7, color=MUTE, ls="--", lw=.8); axs[2].axhline(-5 / 7, color=MUTE, ls="--", lw=.8)
axs[2].set_title(f"Quantile Flow Field alignment, pair {cq} (dashed: 5 of 7 layers move the same way)", fontsize=10); axs[2].set_ylim(-1.05, 1.05)
plt.tight_layout(); savefig(fig, "18g_flow_tracker", "Probability Mass Flow Tracker and Quantile Flow Field for one product", "18g")
fig, axs = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw=dict(width_ratios=[1.6, 1]))
M_ = pd.DataFrame({u: LY[u][pc] for u in QG_U}).reindex(ix).T; axs[0].imshow(M_.to_numpy(float), aspect="auto", cmap=DIV, vmin=-1.5, vmax=1.5, extent=[0, len(ix), -.5, len(QG_U) - .5], origin="lower")
axs[0].set_yticks(range(len(QG_U))); axs[0].set_yticklabels([f"Q{int(u * 100)}" for u in QG_U], fontsize=8); axs[0].set_title(f"{pc}: how each layer of the return distribution moves (fast minus slow window, in scale units), pair {cq}", fontsize=10); axs[0].set_xlabel("last 250 test days")
for k, (nm, col) in enumerate(zip(QG_LL.columns, SERIES)): axs[1].plot(QG_LL.index, QG_LL[nm], marker="o", color=col, label=nm)
axs[1].axhline(0, color="#c9c7c2"); axs[1].set_xlabel("days ahead"); axs[1].set_ylabel("rank IC (test)"); axs[1].legend(fontsize=7); axs[1].set_title("Does flow lead? Rank IC against the return k days later", fontsize=10)
plt.tight_layout(); savefig(fig, "18g_quantile_field", "Layers of the return distribution and lead-lag of the flow signals", "18g")

_cf = QG_SEL[QG_SEL.confirmed_chosen | QG_SEL.confirmed_ens]
QG_SUMMARY = dict(pairs=[str(c) for c in QG_CFG], chosen={k: str(v) for k, v in QG_CHOSEN.items()}, bonferroni_t=BONF,
                  selection=QG_SEL.round(4).to_dict("records"), confirmed=[f"{r.indicator} {r.label}: " + ("chosen pair" if r.confirmed_chosen else "") + (" ensemble" if r.confirmed_ens else "") for r in _cf.itertuples()],
                  early=QG_EARLY.round(4).to_dict("records"), ex_momentum_test={f"{r.indicator} {r.version}": dict(ic=r.ic_ex_momentum_test, t=r.t_ic_ex_momentum_test) for r in QG_ADD.itertuples()})
_e = QG_EARLY[(QG_EARLY.signal.str.startswith("early")) & (QG_EARLY.split == "test")]
QG_SUMMARY["reading"] = (f"Windows chosen on validation only (worst-regime IC, plateau rule): tracker {QG_CHOSEN['pmf']}, field {QG_CHOSEN['qff']}. "
    f"Confirmed on test (same sign in all periods; |t|>{BONF} for the chosen pair, >2 for the ensemble): {', '.join(QG_SUMMARY['confirmed']) if QG_SUMMARY['confirmed'] else 'none'}. "
    f"IC after removing momentum on test: " + "; ".join(f"{k} {v['ic']:+.3f} (t={v['t']:.1f})" for k, v in QG_SUMMARY["ex_momentum_test"].items()) + ". "
    + (f"Early signals on test: n={int(_e.n.iloc[0])}, mean forward 5-day z in the flow direction {_e.mean_signed_fwd5.iloc[0]:+.3f} (t={_e.t.iloc[0]:.1f})." if len(_e) else ""))
print("\n" + QG_SUMMARY["reading"])

## Step 18h — Wasserstein probability flow, and are the three flow indicators independent evidence?

**Why.** Step 18g built two distribution-flow indicators (Probability Mass Flow Tracker, Quantile Flow Field). A third one was shown in the first reference chart (Probability Flow Momentum, based on the Wasserstein distance). All three are computed from rolling quantiles of the same return series, so they may be largely the *same* information three times. Counting them as three confirmations would overstate the evidence. This step builds the third indicator and then measures the overlap before any of them is combined.

**C. Probability Flow Momentum (Wasserstein).** Recent window $R$ of $f$ days against the reference window $B$ of $b$ days that ends where $R$ begins (no overlap); quantile levels $u\in\{.05,.15,\dots,.95\}$:

$$d_u=\frac{Q^{R}_u-Q^{B}_u}{\hat s_B},\quad \hat s_B=\frac{Q^B_{.75}-Q^B_{.25}}{1.349},\qquad W_1=\frac1{10}\sum_u |d_u|,\quad D=\frac1{10}\sum_u d_u,\quad E=\frac{D}{W_1}\in[-1,1]$$

$W_1$ is the 1-D Wasserstein distance (how much the distribution changed), $D$ the transport direction, $E$ the flow efficiency. Velocity $V=\mathrm{EMA}_5(W_{1,t}-W_{1,t-5})$; momentum $PFM=\mathrm{EMA}_9(D\cdot W_1)$ divided by its trailing 250-day standard deviation. Regime: **TRANS** when $W_1$ is above its trailing mean by 0.5 sd and $V>0$; **BULL / BEAR** when TRANS and $|E|>0.5$ with $D>0$ / $D<0$. The window pair is chosen exactly as in 18g (validation only, worst-regime IC, plateau rule); $W_1$ is also tested as a *volatility* signal (next-5-day absolute move, trailing volatility removed).

**Overlap tests.** The five direction signals compared are: PMF and QFF (18g, chosen pairs), PFM (this step), and two classic references, 20-day momentum and 5-day reversal.

| Test | What it answers |
|---|---|
| Cross-sectional rank correlation per date (averaged by period and by volatility regime) | do they rank products the same way? |
| Time-series correlation per product | do they switch on and off together? |
| Correlation of their daily IC series | do they win and lose on the same days? |
| Participation ratio $(\sum\lambda)^2/\sum\lambda^2$ of the correlation matrix, first-component share | how many independent signals are there in effect? |
| IC after removing the others (per-date regression of the signal's ranks on the other signals' ranks) | what does each add beyond the rest? |
| Combination gain $=IC_{combo}/\overline{IC}_{single}$ (1 = same information, $\sqrt3\approx1.73$ = three independent signals) | is combining worth it? |

**Verdict rule (fixed in advance).** The three flow indicators count as separate evidence only if the average absolute pairwise correlation is below 0.30 *and* each indicator that has an IC of at least 0.005 on validation keeps at least half of it after the others are removed. Otherwise they are one family and are reported and combined as one.

In [ ]:
# step 18h
assert "QGS" in globals() and "QG_CHOSEN" in globals(), "Step 18h needs Step 18g: run the Step 18g cells first"
# ---- 18h.1 Wasserstein probability flow momentum (PFM) ----
PW_U = (0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95); PW_CFG = [(10, 60), (20, 60), (20, 120), (40, 120), (40, 250)]
def pw_signals(Zr, cfgs=PW_CFG):
    W = sorted({w for c in cfgs for w in c}); Q = {w: {u: Zr.rolling(w, min_periods=int(0.8 * w)).quantile(u) for u in PW_U} for w in W}; out = {}
    for f, b in cfgs:
        ref = {u: Q[b][u].shift(f) for u in PW_U}; sref = (ref[0.75] - ref[0.25]) / 1.349                      # reference window ends where the recent one begins
        dif = {u: (Q[f][u] - ref[u]) / sref for u in PW_U}; n = len(PW_U)
        W1 = sum(abs(v) for v in dif.values()) / n; D = sum(dif.values()) / n; E = D / W1.where(W1 > 0)
        V = (W1 - W1.shift(5)).ewm(span=5, min_periods=3).mean(); P = (D * W1).ewm(span=9, min_periods=5).mean(); P = P / P.rolling(250, min_periods=100).std()
        zW = (W1 - W1.rolling(250, min_periods=100).mean()) / W1.rolling(250, min_periods=100).std(); trans = (zW > 0.5) & (V > 0)
        out[("pfm", (f, b))] = P; out[("w1", (f, b))] = W1; out[("zw1", (f, b))] = zW; out[("v", (f, b))] = V; out[("e", (f, b))] = E
        out[("regime", (f, b))] = (trans * np.where(E.abs() > 0.5, np.sign(D), 0.0)).where(zW.notna())
    return out
_r = np.random.default_rng(2); _x = _r.normal(size=500); _x[-12:] += 1.5; _o = pw_signals(pd.DataFrame({"a": _x}), [(10, 60)])
assert _o[("pfm", (10, 60))]["a"].iloc[-1] > 0 and _o[("e", (10, 60))]["a"].iloc[-1] > 0.8 and _o[("regime", (10, 60))]["a"].iloc[-1] == 1, "Wasserstein flow sign convention broken"
_x = _r.normal(size=500); _x[-12:] -= 1.5; _o = pw_signals(pd.DataFrame({"a": _x}), [(10, 60)]); assert _o[("pfm", (10, 60))]["a"].iloc[-1] < 0 and _o[("regime", (10, 60))]["a"].iloc[-1] == -1
_x = _r.normal(size=500); _x[-12:] = np.tile([-3.0, 3.0], 6); _o = pw_signals(pd.DataFrame({"a": _x}), [(10, 60)]); assert _o[("w1", (10, 60))]["a"].iloc[-1] > 1.0 and abs(_o[("e", (10, 60))]["a"].iloc[-1]) < 0.5   # symmetric widening: distance up, direction not
print("unit checks passed: an upward shift gives BULL, a downward shift BEAR, a pure widening raises the distance without a direction")
PWS = pw_signals(Zq)
causal_test("pw_signals", lambda a: {f"{k[0]}{k[1]}": v for k, v in pw_signals(a, [(10, 60)]).items()}, Zq.iloc[-700:], step="18h")
# window pair: validation only, worst-regime IC, plateau rule (same rule as 18g)
rows = []
for c in PW_CFG:
    ic = xs_ic_fast(PWS[("pfm", c)].reindex(columns=UNIV), QG_Y["fwd5"], min_n=5); v_ = ic.reindex(QG_IDX["val"]); rg = QG_REG.reindex(QG_IDX["val"])
    r = dict(pair=str(c), val_worst_regime=min([v_[rg == k].mean() for k in ("low", "mid", "high") if (rg == k).sum() >= 40] or [np.nan]))
    for sp, ix in QG_IDX.items(): r[f"ic_{sp}"], r[f"t_{sp}"] = mean_t(ic.reindex(ix), L=10)
    rows.append(r)
PW_TAB = pd.DataFrame(rows).set_index("pair"); sc = {}
for c in PW_CFG:
    nb = [d for d in PW_CFG if d != c and (d[0] == c[0] or d[1] == c[1])]; sc[c] = np.nanmean([PW_TAB.val_worst_regime[str(c)]] + [PW_TAB.val_worst_regime[str(d)] for d in nb])
PW_CHOSEN = max(sc, key=lambda c: (-np.inf if not np.isfinite(sc[c]) else sc[c])); PW_TAB["chosen"] = [p == str(PW_CHOSEN) for p in PW_TAB.index]
PW_TAB.round(4).to_csv(OUT2 / "wasserstein_flow_ic.csv"); print("Wasserstein flow, rank IC against the next-5-day return by window pair; chosen on validation only:"); print(PW_TAB.round(4).to_string()); print("chosen:", PW_CHOSEN)
# W1 as a volatility signal: IC against the next-5-day absolute move after removing trailing volatility
vol20_ = Zq.abs().rolling(20).mean(); absf = Zq.rolling(5).sum().shift(-5).abs(); rows = []
for nm in ("w1", "zw1", "v"):
    res = td_resid(PWS[(nm, PW_CHOSEN)].reindex(columns=UNIV), vol20_); ic = xs_ic_fast(res, absf, min_n=5); r = dict(signal=nm)
    for sp, ix in QG_IDX.items(): r[f"ic_{sp}"], r[f"t_{sp}"] = mean_t(ic.reindex(ix), L=10)
    rows.append(r)
PW_VOL = pd.DataFrame(rows); PW_VOL.round(4).to_csv(OUT2 / "wasserstein_flow_volatility.csv", index=False)
print("\nDistance W1, its z-score and velocity as predictors of the SIZE of the next-5-day move (trailing volatility removed):"); print(PW_VOL.round(3).to_string(index=False))

In [ ]:
# ---- 18h.2 are the flow indicators independent evidence? ----
SIGS = {"PMF (18g)": QGS[("pmf", QG_CHOSEN["pmf"])].reindex(columns=UNIV), "QFF (18g)": QGS[("qff", QG_CHOSEN["qff"])].reindex(columns=UNIV), "PFM Wasserstein (18h)": PWS[("pfm", PW_CHOSEN)].reindex(columns=UNIV),
        "momentum 20d": Zq.rolling(20).sum(), "reversal 5d": -Zq.rolling(5).sum()}
NM = list(SIGS); FLOW = NM[:3]; QG_REGL = QG_REG
def xs_corr(A, B):
    m = A.notna() & B.notna(); a = A.where(m).rank(axis=1); b = B.where(m).rank(axis=1); a = a.sub(a.mean(axis=1), axis=0); b = b.sub(b.mean(axis=1), axis=0)
    return (a * b).sum(axis=1) / np.sqrt((a * a).sum(axis=1) * (b * b).sum(axis=1))
XC = {(i, j): xs_corr(SIGS[i], SIGS[j]) for k, i in enumerate(NM) for j in NM[k + 1:]}
def cmat(f):
    M = pd.DataFrame(np.eye(len(NM)), index=NM, columns=NM)
    for (i, j), s in XC.items(): M.loc[i, j] = M.loc[j, i] = f(s)
    return M
CM = {sp: cmat(lambda s, ix=ix: float(s.reindex(ix).mean())) for sp, ix in QG_IDX.items()}
CMR = {rg: cmat(lambda s, rg=rg: float(s[QG_REG.reindex(s.index) == rg].mean())) for rg in ("low", "mid", "high")}
TS = {}
for sp, ix in QG_IDX.items():
    M = pd.DataFrame(np.eye(len(NM)), index=NM, columns=NM)
    for k, i in enumerate(NM):
        for j in NM[k + 1:]: M.loc[i, j] = M.loc[j, i] = float(SIGS[i].loc[ix].corrwith(SIGS[j].loc[ix]).mean())
    TS[sp] = M
ICSET = {n: xs_ic_fast(SIGS[n], QG_Y["fwd5"], min_n=5) for n in NM}
ICC = {sp: pd.DataFrame({n: ICSET[n].reindex(ix) for n in NM}).corr() for sp, ix in QG_IDX.items()}
print("Cross-sectional rank correlation between the signals, averaged over dates (train / validation / test):")
for sp in ("train", "val", "test"): print(f"\n[{sp}]\n", CM[sp].round(2).to_string())
print("\nTime-series correlation per product (test):\n", TS["test"].round(2).to_string()); print("\nCorrelation of the daily IC series (test):\n", ICC["test"].round(2).to_string())
print("\nBy volatility regime (the three flow indicators only):", {rg: {f"{a[:3]}-{b[:3]}": round(float(CMR[rg].loc[a, b]), 2) for k, a in enumerate(FLOW) for b in FLOW[k + 1:]} for rg in CMR})
def pr(M):
    w = np.clip(np.linalg.eigvalsh(M.to_numpy()), 0, None)[::-1]; return float(w.sum() ** 2 / (w ** 2).sum()), float(w[0] / w.sum())
PRS = {sp: pr(CM[sp].loc[FLOW, FLOW]) for sp in CM}; print("\nEffective number of independent signals among the three flow indicators (participation ratio, share of first component):", {sp: (round(a, 2), round(b, 2)) for sp, (a, b) in PRS.items()})
# IC after removing the other signals (per-date regression of ranks)
def orth(name, others):
    A = SIGS[name].rank(axis=1); Bs = [SIGS[o].rank(axis=1) for o in others]; out = pd.DataFrame(np.nan, index=A.index, columns=A.columns)
    a_ = A.to_numpy(float); b_ = np.stack([b.to_numpy(float) for b in Bs], axis=-1)
    for t in range(len(a_)):
        y = a_[t]; X = b_[t]; ok = np.isfinite(y) & np.isfinite(X).all(axis=1)
        if ok.sum() < len(others) + 4: continue
        Xo = np.column_stack([np.ones(ok.sum()), X[ok]]); beta = np.linalg.lstsq(Xo, y[ok], rcond=None)[0]; r = np.full(len(y), np.nan); r[ok] = y[ok] - Xo @ beta; out.iloc[t] = r
    return out
rows = []
for n in FLOW:
    res = orth(n, [o for o in NM if o != n]); ic = xs_ic_fast(res, QG_Y["fwd5"], min_n=5); r = dict(signal=n)
    for sp, ix in QG_IDX.items():
        raw = mean_t(ICSET[n].reindex(ix), L=10); rem = mean_t(ic.reindex(ix), L=10); r[f"ic_{sp}"] = raw[0]; r[f"ic_after_others_{sp}"] = rem[0]; r[f"t_after_others_{sp}"] = rem[1]; r[f"kept_{sp}"] = rem[0] / raw[0] if abs(raw[0]) >= 0.005 else np.nan                  # only meaningful where the signal has an IC to keep
    rows.append(r)
PW_ORTH = pd.DataFrame(rows); PW_ORTH.round(4).to_csv(OUT2 / "flow_indicators_orthogonal_ic.csv", index=False)
print("\nIC against the next-5-day return before and after removing the other four signals (kept = share of IC that survives):"); print(PW_ORTH.round(3).to_string(index=False))

In [ ]:
# ---- 18h.3 combination, verdict, figures, summary ----
def xs_norm(F): r = F.rank(axis=1); return r.sub(r.mean(axis=1), axis=0).div(r.std(axis=1), axis=0)
COMBO = sum(xs_norm(SIGS[n]) for n in FLOW) / 3.0; ICSET["combo"] = xs_ic_fast(COMBO, QG_Y["fwd5"], min_n=5)
rows = []
for sp, ix in QG_IDX.items():
    singles = [mean_t(ICSET[n].reindex(ix), L=10)[0] for n in FLOW]; cm_, ct_ = mean_t(ICSET["combo"].reindex(ix), L=10); gain = cm_ / np.mean(singles) if abs(np.mean(singles)) >= 0.005 else np.nan          # undefined when the single signals have no IC
    rows.append(dict(split=sp, mean_single_ic=float(np.mean(singles)), combo_ic=cm_, combo_t=ct_, gain=gain, effective_breadth=gain ** 2 if np.isfinite(gain) else np.nan))
PW_COMBO = pd.DataFrame(rows); PW_COMBO.round(4).to_csv(OUT2 / "flow_indicators_combination.csv", index=False)
print("Combining the three (equal-weight rank average): gain 1 = they carry the same information, 1.73 = three independent signals:"); print(PW_COMBO.round(3).to_string(index=False))
iu = [(a, b) for k, a in enumerate(FLOW) for b in FLOW[k + 1:]]; mabs = {sp: float(np.mean([abs(CM[sp].loc[a, b]) for a, b in iu])) for sp in CM}
kept_val = PW_ORTH.set_index("signal").kept_val; kept_test = PW_ORTH.set_index("signal").kept_test
kv_ = kept_val.dropna(); independent = bool(mabs["val"] < 0.30 and (len(kv_) == 0 or (kv_ >= 0.5).all())); VERDICT = "separate evidence" if independent else "one family (do not count them as three confirmations)"
print(f"\nVerdict (rule fixed in advance: mean |correlation| < 0.30 and each indicator keeps >= 50% of its IC on validation): mean |correlation| val {mabs['val']:.2f}; IC kept on validation {kept_val.round(2).to_dict()} -> {VERDICT}" + ("  (IC retention could not be evaluated for indicators with no validation IC; decided on correlation alone)" if len(kv_) < len(kept_val) else ""))
fig, axs = plt.subplots(1, 3, figsize=(18, 5.3))
for ax, sp in zip(axs[:2], ("train", "test")):
    im = ax.imshow(CM[sp].to_numpy(), cmap=DIV, vmin=-1, vmax=1)
    for i in range(len(NM)):
        for j in range(len(NM)): ax.text(j, i, f"{CM[sp].iat[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_xticks(range(len(NM))); ax.set_yticks(range(len(NM))); ax.set_xticklabels(NM, rotation=35, ha="right", fontsize=7); ax.set_yticklabels(NM, fontsize=7); ax.set_title(f"Cross-sectional rank correlation, {sp}", fontsize=10)
ax = axs[2]; x = np.arange(len(FLOW))
for k, (sp, col) in enumerate(zip(("val", "test"), SERIES[:2])):
    ax.bar(x + (k - .5) * .38, PW_ORTH.set_index("signal").loc[FLOW, f"ic_{sp}"], width=.19, color=col, label=f"{sp}: raw IC"); ax.bar(x + (k - .5) * .38 + .19, PW_ORTH.set_index("signal").loc[FLOW, f"ic_after_others_{sp}"], width=.19, color=col, alpha=.45, label=f"{sp}: after removing the others")
ax.set_xticks(x); ax.set_xticklabels(FLOW, fontsize=8); ax.axhline(0, color="#c9c7c2"); ax.legend(fontsize=7); ax.set_title("What each flow indicator adds beyond the others (IC vs next-5-day)", fontsize=10)
plt.tight_layout(); savefig(fig, "18h_flow_overlap", "How much do the three flow indicators overlap?", "18h")
PW_SUMMARY = dict(chosen_pair=str(PW_CHOSEN), mean_abs_corr=mabs, kept_val={k: float(v) for k, v in kept_val.items()}, kept_test={k: float(v) for k, v in kept_test.items()}, participation={sp: dict(effective=a, first_share=b) for sp, (a, b) in PRS.items()},
                  corr_test={f"{a} | {b}": float(CM['test'].loc[a, b]) for a, b in iu}, combination=PW_COMBO.round(4).to_dict("records"), verdict=VERDICT)
PW_SUMMARY["reading"] = (f"Wasserstein flow pair chosen on validation: {PW_CHOSEN}. Mean absolute cross-sectional correlation among the three flow indicators: train {mabs['train']:.2f}, validation {mabs['val']:.2f}, test {mabs['test']:.2f}; effective number of independent signals {PRS['test'][0]:.2f} of 3 (test). "
    f"IC kept after removing the other signals (validation): {', '.join(f'{k} {v:.0%}' if np.isfinite(v) else f'{k} n/a (no IC)' for k, v in PW_SUMMARY['kept_val'].items())}. Combination gain on test {PW_COMBO[PW_COMBO.split == 'test'].gain.iloc[0]:.2f} (1 = same information, 1.73 = independent). Verdict: {VERDICT}.")
print("\n" + PW_SUMMARY["reading"])

## Step 18i — ECHO: a field-momentum model of displacement and volatility (injection and dissipation)

**Idea.** Treat the market as a two-variable system and follow its *total balance*. When the balance jumps, something external has pushed the system (an injection of momentum); when it drains while the system is displaced, momentum is leaking away (dissipation, exhaustion). The description of the original indicator gives no formulas, so this is an explicit, testable version of the stated ideas.

**Dual-state coordinates** (per product, smoothing span $a$; all known at the close of day $t$):

$$u_t=\frac{\mathrm{EMA}_a(z_t)}{\mathrm{sd}_{250}},\qquad v_t=\frac{\mathrm{EMA}_a\!\big(\ln(R_t/\mathrm{med}_{60}(R)_{t-1})\big)}{\mathrm{sd}_{250}},\qquad R_t=\ln\frac{High_t}{Low_t}$$

$u$ is the *displacement* (directional drift in σ units), $v$ the *volatility state* (daily range against its own history; roll days are excluded because the contract switch creates artificial ranges). Daily high and low come from the daily panel.

**Balance and force.** $B_t=u_t^2+v_t^2$ (the system's total "energy"); the **power** is $I_t=\dfrac{B_t-B_{t-2}}{\mathrm{sd}_{250}(B_t-B_{t-2})}$. Injection: $I_t>1.5$; dissipation: $I_t<-1.5$. The field-momentum oscillator is
$$FM_t=\mathrm{sgn}(u_t)\cdot\mathrm{clip}(I_t,-4,4)$$
so an injection while displaced up is bullish, an injection while displaced down is bearish, and a dissipation while displaced up (the push draining away) is bearish, while a dissipation while displaced down is bullish.

**Pre-registered hypotheses (signs fixed before looking):**

| | Condition | Outcome | Expected |
|---|---|---|---|
| H1 | injection | continuation $\mathrm{sgn}(u)\cdot z_{t+1..t+5}$ | higher than on other days |
| H2 | dissipation with $|u|>1$ | continuation | **lower** than on other days (exhaustion) |
| H3 | injection | size of the next-5-day move $|\sum z|$, trailing volatility as control | larger |

**Tests.** Rank IC of $FM$ against the next-1 and next-5-day returns by train/validation/test; the smoothing span $a\in\{5,10,20\}$ chosen on validation only (worst-regime IC, plateau rule, as in 18g) with an equal-weight ensemble beside it; IC after removing 20-day momentum; correlation with the earlier flow indicators (so that it is not counted as new evidence if it is the old one again). A hypothesis is *supported* if the sign holds in all three periods and $|t|>2.13$ on test (Bonferroni for three spans).

**Limits.** "Conservation" is an analogy; nothing in a market conserves $B$, so $I$ is simply the standardised change of a volatility-and-displacement score. If $FM$ adds nothing beyond momentum and volatility, the verdict says so.

**Used later.** ECHO's size-of-move result is re-tested with a trailing-volatility control in Step 13g; only that part is a candidate for position sizing.

In [ ]:
# step 18i
assert "QG_REG" in globals() and "ov_cmean" in globals() and "td_resid" in globals(), "Step 18i needs Steps 18d, 18e and 18g: run them first"
# ---- 18i.1 ECHO: dual-state coordinates, balance, injection and dissipation ----
HI_ = wide("high").reindex_like(RAW); LO_ = wide("low").reindex_like(RAW)
RNG = np.log(HI_ / LO_).where((HI_ > 0) & (LO_ > 0) & (HI_ >= LO_) & ~ROLL).reindex(index=Z.index, columns=UNIV)     # roll days carry contract-switch gaps
RNG = RNG.where(RNG > 0)
EC_A = (5, 10, 20)
def echo_signals(Zr, Rg, spans=EC_A, thr=1.5):
    base = np.log(Rg / Rg.rolling(60, min_periods=30).median().shift(1)); out = {}
    for a in spans:
        u = Zr.ewm(span=a, min_periods=a).mean(); u = u / u.rolling(250, min_periods=100).std()
        v = base.ewm(span=a, min_periods=a, ignore_na=True).mean(); v = v / v.rolling(250, min_periods=100).std()
        B = u ** 2 + v ** 2; dB = B - B.shift(2); I = dB / dB.rolling(250, min_periods=100).std()
        out[("u", a)], out[("v", a)], out[("B", a)], out[("I", a)] = u, v, B, I; out[("FM", a)] = np.sign(u) * I.clip(-4, 4)
        out[("inj", a)] = (I > thr).where(I.notna()); out[("dis", a)] = (I < -thr).where(I.notna())
    return out
# unit check: a sudden run up with expanding range is an injection with positive field momentum; the sign flips for a run down
_r = np.random.default_rng(3); _z = _r.normal(size=420); _z[-5:] += 2.5; _g = np.exp(0.3 * _r.normal(size=420)); _g[-5:] *= 2.5
_o = echo_signals(pd.DataFrame({"a": _z}), pd.DataFrame({"a": _g}), (5,)); assert _o[("I", 5)]["a"].iloc[-3:].max() > 1.5 and _o[("FM", 5)]["a"].iloc[-3:].max() > 1.5, "ECHO injection not detected"
_z = _r.normal(size=420); _z[-5:] -= 2.5; _o = echo_signals(pd.DataFrame({"a": _z}), pd.DataFrame({"a": _g}), (5,)); assert _o[("FM", 5)]["a"].iloc[-3:].min() < -1.5, "ECHO sign convention broken"
print("unit checks passed: a run up with expanding range gives injection and positive field momentum, a run down negative")
Zq = Z.reindex(columns=UNIV); ECS = echo_signals(Zq, RNG)
causal_test("echo_signals", lambda a, b: {f"{k[0]}{k[1]}": v for k, v in echo_signals(a, b, (5, 10)).items()}, Zq.iloc[-700:], RNG.iloc[-700:], step="18i")
EC_ENS = sum((ECS[("FM", a)]).clip(-3, 3) for a in EC_A) / len(EC_A)
print(f"ECHO computed for spans {EC_A} on {Zq.shape[1]} products; share of product-days with a valid range: {float(RNG.notna().mean().mean()):.0%}; injections {float(ECS[('inj', 10)].mean().mean()):.1%}, dissipations {float(ECS[('dis', 10)].mean().mean()):.1%} of days (span 10)")


In [ ]:
# ---- 18i.2 span selection (validation only), IC, momentum control ----
mom20 = Zq.rolling(20).sum(); vol20_ = Zq.abs().rolling(20).mean()
rows, ICE = [], {}
for key, sg in {**{a: ECS[("FM", a)] for a in EC_A}, "ens": EC_ENS}.items():
    for lb in ("fwd1", "fwd5"):
        ic = xs_ic_fast(sg, QG_Y[lb], min_n=5); ICE[(key, lb)] = ic; r = dict(span=str(key), label=lb)
        for sp, ix in QG_IDX.items(): r[f"ic_{sp}"], r[f"t_{sp}"] = mean_t(ic.reindex(ix), L=10 if lb == "fwd5" else 5)
        v_ = ic.reindex(QG_IDX["val"]); rg = QG_REG.reindex(QG_IDX["val"]); r["val_worst_regime"] = min([v_[rg == k].mean() for k in ("low", "mid", "high") if (rg == k).sum() >= 40] or [np.nan])
        rows.append(r)
EC_TAB = pd.DataFrame(rows)
t5 = EC_TAB[(EC_TAB.label == "fwd5") & (EC_TAB.span != "ens")].set_index("span").val_worst_regime; sc = {}
for a in EC_A:
    nb = [b for b in EC_A if abs(EC_A.index(b) - EC_A.index(a)) == 1]; sc[a] = np.nanmean([t5[str(a)]] + [t5[str(b)] for b in nb])
EC_CHOSEN = max(sc, key=lambda a: (-np.inf if not np.isfinite(sc[a]) else sc[a])); EC_TAB["chosen"] = [s == str(EC_CHOSEN) for s in EC_TAB.span]
EC_TAB.round(4).to_csv(OUT2 / "echo_ic.csv", index=False)
print("ECHO field momentum, rank IC by span (selection score = worst-regime validation IC); chosen on validation only:"); print(EC_TAB[["span", "label", "ic_train", "ic_val", "ic_test", "t_test", "val_worst_regime", "chosen"]].round(4).to_string(index=False)); print("chosen span:", EC_CHOSEN)
rows = []
for nm, sg, ickey in (("chosen", ECS[("FM", EC_CHOSEN)], (EC_CHOSEN, "fwd5")), ("ensemble", EC_ENS, ("ens", "fwd5"))):
    res = td_resid(sg, mom20); ic_r = xs_ic_fast(res, QG_Y["fwd5"], min_n=5); r = dict(version=nm)
    for sp, ix in QG_IDX.items():
        r[f"ic_{sp}"], r[f"t_{sp}"] = mean_t(ICE[ickey].reindex(ix), L=10); r[f"ic_ex_momentum_{sp}"], r[f"t_ex_momentum_{sp}"] = mean_t(ic_r.reindex(ix), L=10)
    r["confirmed"] = bool(np.sign(r["ic_train"]) == np.sign(r["ic_val"]) == np.sign(r["ic_test"]) and abs(r["t_test"]) > (2.13 if nm == "chosen" else 2.0)); rows.append(r)
EC_ADD = pd.DataFrame(rows); EC_ADD.round(4).to_csv(OUT2 / "echo_vs_momentum.csv", index=False)
print("\nECHO vs the next-5-day return, with and without 20-day momentum removed; confirmed = same sign in all periods and |t_test| above the threshold:"); print(EC_ADD.round(3).to_string(index=False))
# ---- hypotheses H1-H3 on the chosen span ----
a = EC_CHOSEN; U, I_, inj, dis = ECS[("u", a)], ECS[("I", a)], ECS[("inj", a)], ECS[("dis", a)]; sU = np.sign(U)
F5, F1 = QG_Y["fwd5"], QG_Y["fwd1"]; spl_dates = {sp: Zq.index.intersection(SPL[sp])[::5] for sp in ("train", "val", "test")}
HYP = [("H1 injection continues the displacement", sU * F5, inj, None, +1), ("H2 dissipation at large displacement reverses", sU * F5, dis & (U.abs() > 1), None, -1), ("H3 injection enlarges the next move", F5.abs(), inj, vol20_, +1)]
rows = []
for nm, y, flag, ctrl, ex in HYP:
    for sp, ds in spl_dates.items():
        d_ = pd.DataFrame({"y": y.reindex(index=ds).stack(), "f": flag.reindex(index=ds).stack().astype(float)})
        if ctrl is not None: d_["c"] = ctrl.reindex(index=ds).stack()
        d_ = d_.dropna()
        if (d_.f == 1).sum() < 30 or (d_.f == 0).sum() < 30: rows.append(dict(hypothesis=nm, split=sp, n_flag=int((d_.f == 1).sum()), diff=np.nan, t=np.nan, expected=ex)); continue
        X = d_[["f"] + (["c"] if ctrl is not None else [])]; g_ = pd.factorize(d_.index.get_level_values(0))[0]
        r_ = sm.OLS(d_.y.to_numpy(float), sm.add_constant(X.to_numpy(float))).fit(cov_type="cluster", cov_kwds=dict(groups=g_)); rows.append(dict(hypothesis=nm, split=sp, n_flag=int((d_.f == 1).sum()), mean_flag=float(d_.y[d_.f == 1].mean()), mean_other=float(d_.y[d_.f == 0].mean()), diff=float(r_.params[1]), t=float(r_.tvalues[1]), expected=ex))
H = pd.DataFrame(rows).pivot_table(index=["hypothesis", "expected"], columns="split", values=["diff", "t", "n_flag"]); H.columns = [f"{x}_{y}" for x, y in H.columns]; H = H.reset_index()
for c in ("diff", "t", "n_flag"):
    for sp in ("train", "val", "test"):
        if f"{c}_{sp}" not in H: H[f"{c}_{sp}"] = np.nan
H["supported"] = [bool(all(np.sign(getattr(r, f"diff_{sp}")) == r.expected for sp in ("train", "val", "test")) and abs(r.t_test) > 2.13) for r in H.itertuples()]
H.round(4).to_csv(OUT2 / "echo_hypotheses.csv", index=False)
print(f"\nPre-registered hypotheses (span {a}); diff = flagged minus other days (continuation or move size, in z units); supported = expected sign in all periods and |t_test| > 2.13:"); print(H[["hypothesis", "expected", "diff_train", "diff_val", "diff_test", "t_test", "n_flag_test", "supported"]].round(3).to_string(index=False))


In [ ]:
# ---- 18i.3 overlap with the earlier flow indicators, charts, summary ----
EC_OVER = None
if "SIGS" in globals() and "xs_corr" in globals():
    S2 = {"ECHO FM (18i)": ECS[("FM", EC_CHOSEN)].reindex(columns=UNIV), **{k: SIGS[k] for k in ("PMF (18g)", "QFF (18g)", "PFM Wasserstein (18h)", "momentum 20d")}}
    nm_ = list(S2); EC_OVER = pd.DataFrame({sp: {k: float(xs_corr(S2[nm_[0]], S2[k]).reindex(ix).mean()) for k in nm_[1:]} for sp, ix in QG_IDX.items()})
    EC_OVER.round(3).to_csv(OUT2 / "echo_overlap.csv"); print("Cross-sectional rank correlation of ECHO with the earlier signals:"); print(EC_OVER.round(2).to_string())
pc = [c for c in FOCUS if c in UNIV][0]; ix = QG_IDX["test"][-250:]; a = EC_CHOSEN; GREEN, RED = "#1baf7a", "#e34948"
fig, axs = plt.subplots(3, 1, figsize=(15, 9), sharex=True, gridspec_kw=dict(height_ratios=[2, 1.6, 1.6]))
axs[0].plot(Zq[pc].fillna(0).cumsum().reindex(ix), color=INK2, lw=1.2); axs[0].set_title(f"{pc}: cumulative vol-scaled return", fontsize=10)
Iv = ECS[("I", a)][pc].reindex(ix); Bv = ECS[("B", a)][pc].reindex(ix); Uv = ECS[("u", a)][pc].reindex(ix)
axs[1].plot(Bv, color=SERIES[0], lw=1.2); axs[1].set_title(f"Total balance B = u² + v² (span {a}); markers: injection (green) and dissipation (red) in the direction of the displacement", fontsize=10)
for d_, b_, i_, u_ in zip(ix, Bv, Iv, Uv):
    if np.isfinite(i_) and i_ > 1.5: axs[1].scatter(d_, b_, marker="^" if u_ > 0 else "v", color=GREEN, s=26, zorder=3)
    elif np.isfinite(i_) and i_ < -1.5: axs[1].scatter(d_, b_, marker="v" if u_ > 0 else "^", color=RED, s=26, zorder=3)
Fv = ECS[("FM", a)][pc].reindex(ix); axs[2].bar(ix, Fv, color=np.where(Fv >= 0, GREEN, RED), width=1.0); axs[2].axhline(0, color="#c9c7c2"); axs[2].set_title("ECHO field momentum FM = sign(u) × power", fontsize=10)
plt.tight_layout(); savefig(fig, "18i_echo", "ECHO: balance, injection and dissipation for one product", "18i")
fig, axs = plt.subplots(1, 3, figsize=(17, 5))
ax = axs[0]; uu, vv = ECS[("u", a)][pc].reindex(ix[-120:]).to_numpy(), ECS[("v", a)][pc].reindex(ix[-120:]).to_numpy()
th = np.linspace(0, 2 * np.pi, 200)
for rr in (1, 2, 3): ax.plot(rr * np.cos(th), rr * np.sin(th), color="#c9c7c2", lw=.7)
ax.scatter(uu, vv, c=np.arange(len(uu)), cmap=SEQ, s=22, zorder=3); ax.plot(uu, vv, color=MUTE, lw=.6); ax.set_xlabel("displacement u"); ax.set_ylabel("volatility state v"); ax.set_title(f"{pc}: dual-state trajectory, last 120 days (circles: constant balance)", fontsize=9); ax.set_aspect("equal", adjustable="datalim")
ax = axs[1]; T_ = H.reset_index(drop=True); yy = np.arange(len(T_))
for k, (sp, col) in enumerate(zip(("train", "val", "test"), SERIES)): ax.barh(yy + (k - 1) * .27, T_[f"diff_{sp}"] * T_.expected, height=.27, color=col, label=sp)
ax.set_yticks(yy); ax.set_yticklabels([f"{r.hypothesis[:2]}{'  ✓' if r.supported else ''}" for r in T_.itertuples()]); ax.axvline(0, color="#c9c7c2"); ax.invert_yaxis(); ax.legend(fontsize=7); ax.set_title("Hypothesis effects (positive = as predicted; ✓ supported)", fontsize=9)
ax = axs[2]; x = np.arange(len(EC_ADD))
for k, (sp, col) in enumerate(zip(("val", "test"), SERIES[:2])): ax.bar(x + (k - .5) * .38, EC_ADD[f"ic_{sp}"], width=.19, color=col, label=f"{sp}: raw IC"); ax.bar(x + (k - .5) * .38 + .19, EC_ADD[f"ic_ex_momentum_{sp}"], width=.19, color=col, alpha=.45, label=f"{sp}: momentum removed")
ax.set_xticks(x); ax.set_xticklabels(EC_ADD.version); ax.axhline(0, color="#c9c7c2"); ax.legend(fontsize=7); ax.set_title("IC vs next-5-day return, with and without momentum", fontsize=9)
plt.tight_layout(); savefig(fig, "18i_echo_tests", "ECHO: phase portrait, hypotheses, and value beyond momentum", "18i")
EC_SUMMARY = dict(span_chosen=int(EC_CHOSEN), ic=EC_ADD.round(4).to_dict("records"), hypotheses=H.round(4).to_dict("records"), supported=[r.hypothesis for r in H.itertuples() if r.supported],
                  overlap_test=(EC_OVER["test"].round(3).to_dict() if EC_OVER is not None else None))
_c = EC_ADD[EC_ADD.version == "chosen"].iloc[0]
EC_SUMMARY["reading"] = (f"ECHO span chosen on validation: {EC_CHOSEN}. IC against the next-5-day return: train {_c.ic_train:+.3f}, validation {_c.ic_val:+.3f}, test {_c.ic_test:+.3f} (t={_c.t_test:.1f}); after removing 20-day momentum, test {_c.ic_ex_momentum_test:+.3f} (t={_c.t_ex_momentum_test:.1f}); confirmed: {bool(_c.confirmed)}. "
    f"Hypotheses supported in all three periods: {', '.join(EC_SUMMARY['supported']) if EC_SUMMARY['supported'] else 'none'}. " + (f"Correlation with earlier signals on test: {EC_SUMMARY['overlap_test']}." if EC_OVER is not None else ""))
print("\n" + EC_SUMMARY["reading"])


## Step 18k — Candle anatomy: range, body, wicks, rejection, volume, volatility — and how long it lasts

The rulebook (range, body-to-range ratio, wick ratios, rejection, breakout, fakeout) is turned into measurements and tested. **Every number in it is only a suggestion**: thresholds are learned from the training period as quantiles *inside each volatility regime* (the regime of the product itself: trailing 20-day mean |return|, terciles from train), the quantile and the look-back are chosen on validation only (worst world × regime cell, plateau rule over neighbouring settings), and the rulebook values (BRR ≥ 0.7, wick ≥ 0.5, range 1.5× normal, volume 1.5× normal) are reported next to the learned ones.

**Measurements** (all point-in-time; "normal" = the median of the same time-of-day slot over the previous 20 days, so the opening minutes are not compared with the lunch break): range = log(High/Low) relative to normal; BRR = |Close−Open| / range; upper and lower wick / range; the two rejection ratios as written, |High−Open| / |High−Close| and |Low−Open| / |Low−Close| (log, 2% of the range added to avoid 0/0); volume relative to normal; the key level = the highest high of the previous 12, 36 or 72 bars.

**Structures** (expected sign fixed before looking): upper-wick rejection of a prior high → fade (−); the same without a level; breakout candle (close above the prior high, strong body, expanded range, with and without high volume) → continuation (+); fakeout (failed probe, then a strong down candle) → reversal (−); range expansion → larger next range (+); range contraction → larger next range (+, the rulebook claim; volatility persistence says the opposite).

**Time limit**: forward windows are non-overlapping — 0–5, 5–10, 10–15, 15–30, 30–60, 60–120 minutes, and the daily candle for next day / week / month. "Inside 30 minutes" and "beyond 30 minutes" are tested separately, and the regression of the forward range on today's range keeps the trailing range level as control, so persistence of volatility is not mistaken for information in the candle.

**Negated series**: every structure is also run on the series with price → −price (highs become lows, up becomes down, return → −return). A symmetric effect shows the same sign-aligned size in both worlds; a difference is an asymmetry (long bias in the test period, or a genuine bull / bear difference). The unconditional drift of both worlds is reported, and all effects are measured **in excess of the unconditional mean of the same split and regime**, so drift does not masquerade as an event effect.

Supported = expected sign in validation and test and |t| above the Bonferroni level on test (28 event × bar-size tests); "suggestive" = |t| > 2. Date-clustered standard errors.

**Used later.** The candle-anatomy helpers (`kc_base`, `kc_world`, `kc_wins`) and the learned-threshold machinery are reused by Steps 18l and 18m; the decay and negated-series results define what is kept beyond 30 minutes.

In [ ]:
# step 18k
assert "ov_cmean" in globals() and "session_day" in globals() and "causal_test" in globals() and "QG_REG" in globals(), "Step 18k needs Steps 0c, 18e and 18g: run them first"
# ---- 18k.1 candle anatomy: bars, ex-ante scales, features, forward windows ----
# Bars: 5, 15 and 30 minutes (from the cached 5-minute high / low / close) and the daily candle. Every threshold is learned (train quantiles by volatility regime);
# the rulebook values are only the starting suggestion and are reported next to the learned ones.
K_BARS = (5, 15, 30); K_EDGES = (0, 5, 10, 15, 30, 60, 120)          # forward windows in minutes; the question is what is left beyond 30
K_LB = (12, 36, 72); K_Q = (0.6, 0.75, 0.9); K_LBD = (10, 20, 60)     # look-back (bars) that defines the level a candle probes; quantile grid
K_RULEBOOK = dict(uw=0.5, brr=0.7, rng=float(np.log(1.5)), vz=float(np.log(1.5)), rng_lo=float(np.log(0.7)), L=36)
def kc_slot_med(x, td, slot, win=20, minp=8):
    # median of the same time-of-day slot over the previous `win` trading days (shifted: today is never in its own scale)
    di = pd.factorize(pd.DatetimeIndex(td), sort=True)[0]; si = pd.factorize(slot, sort=True)[0]
    W = np.full((di.max() + 1, si.max() + 1), np.nan); W[di, si] = x
    return pd.DataFrame(W).rolling(win, min_periods=minp).median().shift(1).to_numpy()[di, si]
def kc_base(g, bar_min, daily=False):
    # g: columns op, hi, lo, cl, volume (levels), index = bar end time (daily: date), optional bad (1 = unusable, e.g. roll day)
    g = g[~g.index.duplicated()].sort_index(); g = g[(g[["op", "hi", "lo", "cl"]] > 0).all(axis=1)]
    ts = g.index; td = pd.DatetimeIndex(ts.normalize()) if daily else pd.DatetimeIndex(session_day(ts))
    slot = np.zeros(len(g), int) if daily else (ts.hour * 60 + ts.minute).to_numpy()
    o, h, l, c = (np.log(g[k].to_numpy(float)) for k in ("op", "hi", "lo", "cl")); h = np.maximum.reduce([h, o, c]); l = np.minimum.reduce([l, o, c])
    rg = h - l; win, minp = (60, 20) if daily else (20, 8); vol = g["volume"].to_numpy(float); vol = np.where(vol > 0, vol, np.nan)
    sc = kc_slot_med(np.where(rg > 0, rg, np.nan), td, slot, win, minp); vm = kc_slot_med(vol, td, slot, win, minp)
    tdi = ((td - pd.Timestamp("2000-01-01")).days).to_numpy()
    if daily: seg = np.zeros(len(g), int)
    else:
        gap = np.r_[0.0, np.diff(ts.values).astype("timedelta64[s]").astype(float)]; seg = np.cumsum((gap > bar_min * 60 * 1.01) | np.r_[False, np.diff(tdi) != 0])
    bad = g["bad"].to_numpy(float) if "bad" in g else np.zeros(len(g))
    return pd.DataFrame(dict(o=o, h=h, l=l, c=c, vol=vol, sc=sc, vm=vm, tdi=tdi, seg=seg, bad=bad), index=ts)
def kc_world(b, s, Ls=K_LB):
    # candle anatomy in the 'world' s: s = +1 the series as it is, s = -1 the negated series (price -> -price: highs become lows, up becomes down)
    o, h, l, c = b["o"].to_numpy(), b["h"].to_numpy(), b["l"].to_numpy(), b["c"].to_numpy()
    if s < 0: o, h, l, c = -o, -l, -h, -c
    sc = b["sc"].to_numpy(); sg = b["seg"].to_numpy(); rg = h - l; ok = (rg > 1e-7) & (b["bad"].to_numpy() == 0) & np.isfinite(sc)
    f = {}
    with np.errstate(all="ignore"):
        r = np.where(ok, rg, np.nan); f["rng"] = np.log(r / sc); f["brr"] = np.abs(c - o) / r
        f["uw"] = (h - np.maximum(o, c)) / r; f["lw"] = (np.minimum(o, c) - l) / r
        e = 0.02 * r; f["rh"] = np.log((h - o + e) / (h - c + e)); f["rl"] = np.log((o - l + e) / (c - l + e))        # |high-open| / |high-close| and |low-open| / |low-close| (2% of the range added to avoid 0/0)
        f["rz"] = (c - o) / sc; f["vz"] = np.log(b["vol"].to_numpy() / b["vm"].to_numpy())
        cc = pd.Series(c); f["tr6"] = ((cc - cc.shift(6)) / (sc * np.sqrt(6))).to_numpy()
        f["trr"] = pd.Series(f["rng"]).rolling(11, min_periods=6).mean().shift(1).to_numpy()          # trailing range level (previous 11 bars): the volatility control
        f["uwp"] = np.r_[np.nan, f["uw"][:-1]]; same = np.r_[False, sg[1:] == sg[:-1]]
        for L in Ls:
            pm = pd.Series(h).rolling(L, min_periods=L).max().shift(1).to_numpy(); nan = np.isnan(pm)
            brk = c > pm; br = (h > pm) & (c <= pm); f[f"brk{L}"] = np.where(nan, np.nan, brk.astype(float)); f[f"br{L}"] = np.where(nan, np.nan, br.astype(float))
            prev = np.r_[np.nan, f[f"br{L}"][:-1]]; lp = np.r_[np.nan, l[:-1]]; f[f"fk{L}"] = np.where(np.isnan(prev), np.nan, ((prev == 1) & (c < lp) & (c < o) & same).astype(float))
    out = pd.DataFrame(f, index=b.index); out.loc[~ok, :] = np.nan
    return out
def kc_swv_max(a, w, off, fn):
    n = len(a); out = np.full(n, np.nan)
    if w > n: return out
    v = fn(np.lib.stride_tricks.sliding_window_view(a, w), axis=1); idx = np.arange(n) + off; m = idx <= n - w; out[m] = v[idx[m]]; return out
def kc_wins(b, wins, short_bars, daily=False):
    # forward outcomes (labels: use the future on purpose). Window (k1, k2] = bars i+k1+1 .. i+k2: return and range, in units of the ex-ante scale * sqrt(width)
    n = len(b); c, h, l, sc, td, sg = (b[k].to_numpy() for k in ("c", "h", "l", "sc", "tdi", "seg")); ix = np.arange(n); out = {}
    for nm, (k1, k2) in wins.items():
        w = k2 - k1; i1 = np.minimum(ix + k1, n - 1); i2 = np.minimum(ix + k2, n - 1); ok = ix + k2 < n
        ok &= (td[i2] - td <= 12 * k2) if daily else (td[i2] == td)
        if not daily and k2 <= short_bars: ok &= sg[i2] == sg
        with np.errstate(all="ignore"):
            ret = np.where(ok, c[i2] - c[i1], np.nan) / (sc * np.sqrt(w)); rg = (kc_swv_max(h, w, k1 + 1, np.max) - kc_swv_max(l, w, k1 + 1, np.min)) / (sc * np.sqrt(w))
        out[nm] = (ret.astype(np.float32), np.where(ok, rg, np.nan).astype(np.float32))
    return out
def kc_windows_for(bar_min):
    ed = [e for e in K_EDGES if e % bar_min == 0]; W = {"0-30m": (0, 30 // bar_min)}
    for a, b_ in zip(ed[:-1], ed[1:]): W[f"{a}-{b_}m"] = (a // bar_min, b_ // bar_min)
    return W
# ---- unit checks ----
_kcg = pd.DataFrame(dict(op=[1.0, 1.0], hi=[np.exp(2.0), np.exp(2.0)], lo=[np.exp(-1.0), np.exp(-1.0)], cl=[np.exp(1.0)] * 2, volume=[10.0, 10.0]), index=pd.to_datetime(["2020-01-02 09:05", "2020-01-02 09:10"]))
_kcg["op"] = 1.0; _kcb = kc_base(_kcg, 5); _kcb["sc"] = 3.0; _kcb["bad"] = 0.0
_kcf1, _kcf2 = kc_world(_kcb, 1, (12,)), kc_world(_kcb, -1, (12,))
assert abs(_kcf1["brr"].iloc[0] - 1 / 3) < 1e-6 and abs(_kcf1["uw"].iloc[0] - 1 / 3) < 1e-6 and abs(_kcf1["lw"].iloc[0] - 1 / 3) < 1e-6, "candle anatomy wrong"
_kcb2 = _kcb.copy(); _kcb2["h"] = [2.0, 2.0]; _kcb2["l"] = [0.0, 0.0]; _kcb2["c"] = [0.5, 0.5]; _kcb2["o"] = [0.0, 0.0]          # upper wick 1.5/2, lower wick 0
_kca, _kcm = kc_world(_kcb2, 1, (12,)), kc_world(_kcb2, -1, (12,)); assert abs(_kca["uw"].iloc[0] - 0.75) < 1e-9 and abs(_kcm["lw"].iloc[0] - 0.75) < 1e-9 and abs(_kca["lw"].iloc[0]) < 1e-9, "the negated series must swap the wicks"
_kcw = kc_wins(pd.DataFrame(dict(c=np.arange(10.0), h=np.arange(10.0) + 1, l=np.arange(10.0) - 1, sc=1.0, tdi=0, seg=0), index=pd.RangeIndex(10)), {"x": (0, 2), "y": (2, 5)}, 6)
assert abs(_kcw["x"][0][0] - 2 / np.sqrt(2)) < 1e-6 and abs(_kcw["y"][0][0] - 3 / np.sqrt(3)) < 1e-6 and abs(_kcw["y"][1][0] - 4 / np.sqrt(3)) < 1e-6, "forward windows wrong"
assert np.isnan(_kcw["y"][0][8]), "a window that leaves the sample must be NaN"
print("unit checks passed: body / wick ratios, the negated series swaps upper and lower wick, forward windows")
# ---- data: 5-minute bars (cached by Step 1) and the daily candle ----
def kc_load5():
    p = CACHE / "bars5m.parquet"
    if not p.exists(): return None
    b = pd.read_parquet(p)
    if not {"hi", "lo", "cl"} <= set(b.columns): return None
    return b[~b["product"].isin(EXCLUDE)]
KC_RAW5 = kc_load5(); KC_HAS_OPEN = KC_RAW5 is not None and "op" in KC_RAW5.columns
KC_PRODS = [p for p in FOCUS if p in UNIV] if KC_RAW5 is None else [p for p in FOCUS if p in UNIV and p in set(KC_RAW5["product"])]
KC_SPL = pd.Series(np.select([RAW.index.isin(SPL[k]) for k in ("train", "val", "test")], [0, 1, 2], -1), index=RAW.index)
def kc_regime(p):
    # ex-ante volatility regime of the product: trailing 20-day mean |return| (shifted), terciles from the training period only
    v = RAW[p].abs().rolling(20, min_periods=15).mean().shift(1); cu = v.loc[SPL["train"]].quantile([1 / 3, 2 / 3]).to_numpy()
    return pd.Series(np.where(v.isna(), -1, np.digitize(v, cu)), index=RAW.index)
KC_REG = {p: kc_regime(p) for p in KC_PRODS}
if KC_RAW5 is None: print("no 5-minute high / low / close in the cache (older Step 1): the intraday part is skipped, the daily candle is still studied; delete eda_cache/bars5m.parquet and re-run Step 1 to add them")
else: print(f"5-minute bars: {len(KC_RAW5):,} rows, {len(KC_PRODS)} products; opening price {'stored' if KC_HAS_OPEN else 'taken as the previous close (cl * exp(-ret))'}")
def kc_prep5(p):
    g = KC_RAW5[KC_RAW5["product"] == p].set_index("date").sort_index(); g = g[~g.index.duplicated()]
    g = pd.DataFrame(dict(op=g["op"] if KC_HAS_OPEN else g["cl"] * np.exp(-g["ret"].astype(float)), hi=g["hi"], lo=g["lo"], cl=g["cl"], volume=g["volume"])).astype(float)
    return g
def kc_agg(g, B):
    if B == 5: return g
    r = g.resample(f"{B}min", label="right", closed="right")
    return pd.DataFrame(dict(op=r["op"].first(), hi=r["hi"].max(), lo=r["lo"].min(), cl=r["cl"].last(), volume=r["volume"].sum(min_count=1))).dropna(subset=["cl"])
def kc_daily(p):
    O, H, L, Cc = (wide(k)[p] for k in ("open", "high", "low", "close")); V = VOLU[p]; bad = ROLL[p].reindex(O.index).fillna(False).astype(float)
    d = pd.DataFrame(dict(op=O, hi=H, lo=L, cl=Cc, volume=V, bad=bad)).dropna(subset=["op", "hi", "lo", "cl"]); return d
# causality: the point-in-time part (scales, anatomy, levels) at a bar must not change when later bars are removed
if KC_RAW5 is not None and KC_PRODS:
    _kcg5 = kc_prep5(KC_PRODS[0]).iloc[-6000:]
    def _kc_pit(g):
        bb = kc_base(g, 5); w = kc_world(bb, 1); w2 = kc_world(bb, -1, (12,)); return pd.concat([bb[["sc", "vm"]], w, w2.add_suffix("_kcm")], axis=1)
    causal_test("kc_point_in_time_5m", _kc_pit, _kcg5, step="18k")
_kcgd = kc_daily(KC_PRODS[0] if KC_PRODS else UNIV[0]).iloc[-900:]
def _kc_pit_d(g):
    bb = kc_base(g, 1440, daily=True); return pd.concat([bb[["sc", "vm"]], kc_world(bb, 1, K_LBD), kc_world(bb, -1, (10,)).add_suffix("_kcm")], axis=1)
causal_test("kc_point_in_time_daily", _kc_pit_d, _kcgd, step="18k")


In [ ]:
# ---- 18k.2 events with learned thresholds, selection on validation, decay beyond 30 minutes, bull versus negated-series asymmetry ----
from scipy.stats import norm as _norm
KC_EV = {"UWR_L": (-1, "r", True), "UWR_0": (-1, "r", False), "BRK_L": (1, "r", True), "BRKV_L": (1, "r", True), "FAKE_L": (-1, "r", True), "EXP": (1, "R", False), "CON": (1, "R", False)}
KC_NAMES = {"UWR_L": "upper-wick rejection of a prior high (fade)", "UWR_0": "upper-wick rejection anywhere (fade)", "BRK_L": "breakout candle: close above the prior high, strong body, expanded range", "BRKV_L": "breakout candle with high volume as well",
            "FAKE_L": "fakeout: failed probe of the prior high followed by a strong down candle", "EXP": "range expansion -> next-window range", "CON": "range contraction -> next-window range"}
def kc_assemble(bases, wins, short_bars, daily, Ls):
    parts = {1: [], -1: []}; sh = []
    for k, (p, b) in enumerate(bases.items()):
        dts = pd.Timestamp("2000-01-01") + pd.to_timedelta(b["tdi"].to_numpy(), unit="D")
        sp = KC_SPL.reindex(dts).fillna(-1).to_numpy().astype(int); rg = KC_REG[p].reindex(dts).fillna(-1).to_numpy().astype(int); W = kc_wins(b, wins, short_bars, daily)
        for s in (1, -1):
            f = kc_world(b, s, Ls); keep = (sp >= 0) & (rg >= 0) & f["rng"].notna().to_numpy()
            d = {c: f[c].to_numpy()[keep].astype(np.float32) for c in f.columns}
            for nm, (r, R) in W.items(): d["r_" + nm] = (s * r[keep]).astype(np.float32); d["R_" + nm] = R[keep]
            parts[s].append(d)
        sh.append(dict(spl=sp[keep], reg=rg[keep], day=b["tdi"].to_numpy()[keep], pid=np.full(int(keep.sum()), k), sc=b["sc"].to_numpy()[keep].astype(np.float32)))
    A = {s: {c: np.concatenate([d[c] for d in parts[s]]) for c in parts[s][0]} for s in (1, -1)}; S = {c: np.concatenate([d[c] for d in sh]) for c in sh[0]}
    S["day"] = np.unique(S["day"], return_inverse=True)[1].astype(np.int64); S["cell9"] = S["spl"] * 3 + S["reg"]; S["sc_med"] = float(np.nanmedian(S["sc"]))
    return A, S
def kc_cellstats(ye, mask, cell, ncell, day, nday):
    sel = mask & np.isfinite(ye) & (cell >= 0); c = cell[sel]; d = day[sel]; y = ye[sel].astype(float)
    n = np.bincount(c, minlength=ncell).astype(float); m = np.bincount(c, weights=y, minlength=ncell) / np.maximum(n, 1)
    sres = np.bincount(c * nday + d, weights=y - m[c], minlength=ncell * nday).reshape(ncell, nday); se = np.sqrt((sres ** 2).sum(1)) / np.maximum(n, 1)
    return n, m, np.where((n >= 30) & (se > 0), m / se, np.nan)
def kc_excess(y, S, vol=False):
    ok = np.isfinite(y); n = np.bincount(S["cell9"][ok], minlength=9).astype(float); mu = np.bincount(S["cell9"][ok], weights=y[ok].astype(float), minlength=9) / np.maximum(n, 1)
    return (y / mu[S["cell9"]] - 1 if vol else y - mu[S["cell9"]]).astype(np.float32), mu
def kc_cuts(Aw, S):
    C = {}
    for f in ("uw", "brr", "rng", "vz"):
        C[f] = {}
        for q in K_Q:
            C[f][q] = np.array([np.quantile(x[np.isfinite(x)], q) if np.isfinite(x).sum() >= 200 else np.nan for x in (Aw[f][(S["spl"] == 0) & (S["reg"] == r)] for r in range(3))])
    C["rng_lo"] = {q: np.array([np.quantile(x[np.isfinite(x)], 1 - q) if np.isfinite(x).sum() >= 200 else np.nan for x in (Aw["rng"][(S["spl"] == 0) & (S["reg"] == r)] for r in range(3))]) for q in K_Q}
    return C
def kc_event(Aw, S, C, ev, q, L, rule=None):
    T = (lambda f: C[f][q][S["reg"]]) if rule is None else (lambda f: rule["lo" if f == "rng_lo" else f])
    with np.errstate(invalid="ignore"):
        if ev == "UWR_L": return (Aw[f"br{L}"] == 1) & (Aw["uw"] >= T("uw"))
        if ev == "UWR_0": return Aw["uw"] >= T("uw")
        if ev == "BRK_L": return (Aw[f"brk{L}"] == 1) & (Aw["brr"] >= T("brr")) & (Aw["rng"] >= T("rng"))
        if ev == "BRKV_L": return (Aw[f"brk{L}"] == 1) & (Aw["brr"] >= T("brr")) & (Aw["rng"] >= T("rng")) & (Aw["vz"] >= T("vz"))
        if ev == "FAKE_L": return (Aw[f"fk{L}"] == 1) & (Aw["uwp"] >= T("uw")) & (Aw["brr"] >= T("brr"))
        if ev == "EXP": return Aw["rng"] >= T("rng")
        if ev == "CON": return Aw["rng"] <= T("rng_lo")
KC_RULE = dict(uw=K_RULEBOOK["uw"], brr=K_RULEBOOK["brr"], rng=K_RULEBOOK["rng"], vz=K_RULEBOOK["vz"], lo=K_RULEBOOK["rng_lo"])
def kc_run(tag, A, S, prim, Ls, wins):
    nday = int(S["day"].max()) + 1; spn = np.array(["train", "val", "test"]); rgn = np.array(["low", "mid", "high"]); rows, YE = [], {}
    for w in (1, -1):
        for kind in ("r", "R"): YE[(w, kind)] = kc_excess(A[w][f"{kind}_{prim}"], S, vol=(kind == "R"))[0]
    CUT = {w: kc_cuts(A[w], S) for w in (1, -1)}
    def add(ev, ver, q, L, w, mask):
        kind = KC_EV[ev][1]; ye = YE[(w, kind)]
        n, m, t = kc_cellstats(ye, mask, S["cell9"], 9, S["day"], nday)
        for c in range(9): rows.append(dict(bar=tag, event=ev, version=ver, q=q, L=L, world=w, split=spn[c // 3], regime=rgn[c % 3], n=n[c], mean=m[c], t=t[c]))
        n, m, t = kc_cellstats(ye, mask, S["spl"], 3, S["day"], nday)
        for c in range(3): rows.append(dict(bar=tag, event=ev, version=ver, q=q, L=L, world=w, split=spn[c], regime="all", n=n[c], mean=m[c], t=t[c]))
    for ev, (sg, kind, lev) in KC_EV.items():
        for q in K_Q:
            for L in (Ls if lev else (None,)):
                for w in (1, -1): add(ev, "learned", q, L, w, kc_event(A[w], S, CUT[w], ev, q, L))
        L0 = K_RULEBOOK["L"] if lev else None
        if lev and L0 not in Ls: L0 = Ls[len(Ls) // 2]
        for w in (1, -1): add(ev, "rulebook", np.nan, L0, w, kc_event(A[w], S, CUT[w], ev, None, L0, rule=KC_RULE))
    R = pd.DataFrame(rows); R["expected"] = R["event"].map(lambda e: KC_EV[e][0]); R["aligned_t"] = R["t"] * R["expected"]; R["aligned_mean"] = R["mean"] * R["expected"]
    # ---- selection: validation only, worst (world, regime) cell, plateau rule over neighbouring (q, L) ----
    sel = []
    for ev, (sg, kind, lev) in KC_EV.items():
        Rl = R[(R.event == ev) & (R.version == "learned")]; cfg = [(q, L) for q in K_Q for L in (Ls if lev else (None,))]; sc = {}
        for q, L in cfg:
            x = Rl[(Rl.q == q) & ((Rl.L == L) if L is not None else Rl.L.isna()) & (Rl.split == "val") & (Rl.regime != "all")]["aligned_t"].to_numpy(); sc[(q, L)] = np.nanmin(x) if np.isfinite(x).sum() >= 4 else np.nan
        for (q, L) in cfg:
            qi = K_Q.index(q); nb = [(K_Q[j], L) for j in (qi - 1, qi + 1) if 0 <= j < len(K_Q)] + ([(q, Ls[j]) for j in (Ls.index(L) - 1, Ls.index(L) + 1) if 0 <= j < len(Ls)] if lev else [])
            pl = np.nanmean([sc[(q, L)]] + [sc[k] for k in nb]) if np.isfinite(sc[(q, L)]) else np.nan
            te = Rl[(Rl.q == q) & ((Rl.L == L) if L is not None else Rl.L.isna()) & (Rl.split == "test") & (Rl.regime == "all")]["aligned_t"].mean()
            sel.append(dict(bar=tag, event=ev, q=q, L=L, val_worst_cell_t=sc[(q, L)], plateau_score=pl, test_aligned_t_mean_worlds=te))
    SEL = pd.DataFrame(sel); SEL["chosen"] = False; SEL["test_rank"] = np.nan
    for ev in KC_EV:
        m = SEL.event == ev
        if SEL.loc[m, "plateau_score"].notna().any(): SEL.loc[SEL.loc[m, "plateau_score"].idxmax(), "chosen"] = True
        SEL.loc[m, "test_rank"] = SEL.loc[m, "test_aligned_t_mean_worlds"].rank(ascending=False)
    # ---- decay: the chosen configuration and the rulebook values, every forward window ----
    D = []
    for ev, (sg, kind, lev) in KC_EV.items():
        ch = SEL[(SEL.event == ev) & SEL.chosen]
        if not len(ch): continue
        q, L = float(ch.q.iloc[0]), (None if pd.isna(ch.L.iloc[0]) else int(ch.L.iloc[0]))
        for ver in ("learned", "rulebook"):
            for w in (1, -1):
                L1 = L if ver == "learned" else (K_RULEBOOK["L"] if K_RULEBOOK["L"] in Ls else Ls[len(Ls) // 2]) if lev else None
                mask = kc_event(A[w], S, CUT[w], ev, q if ver == "learned" else None, L1, rule=None if ver == "learned" else KC_RULE)
                for nm in wins:
                    ye = kc_excess(A[w][f"{kind}_{nm}"], S, vol=(kind == "R"))[0]; n, m_, t = kc_cellstats(ye, mask, S["spl"], 3, S["day"], nday)
                    for c in range(3): D.append(dict(bar=tag, event=ev, version=ver, world=w, window=nm, split=spn[c], n=n[c], mean=m_[c], t=t[c], expected=sg, bp=(m_[c] * sg * S["sc_med"] * np.sqrt(wins[nm][1] - wins[nm][0]) * 1e4) if kind == "r" else np.nan))
    D = pd.DataFrame(D); D["aligned_t"] = D["t"] * D["expected"]; D["aligned_mean"] = D["mean"] * D["expected"]
    thr = pd.DataFrame([dict(bar=tag, world=w, feature=f, q=q, regime=rgn[r], cut=CUT[w][f][q][r]) for w in (1, -1) for f in ("uw", "brr", "rng", "vz", "rng_lo") for q in K_Q for r in range(3)])
    drift = pd.DataFrame([dict(bar=tag, world=w, split=spn[c], mean_fwd=float(np.nanmean(A[w]["r_" + prim][S["spl"] == c]))) for w in (1, -1) for c in range(3)])
    return dict(R=R, SEL=SEL, D=D, thr=thr, drift=drift)
def kc_beyond(D, ev, ver="learned", thr_t=2.0):
    # supported inside 30 minutes: expected sign in validation and test and |t| above the bar on test; persists beyond 30 minutes: the same for any window that starts at 30 minutes or later
    out = []
    for w in (1, -1):
        d = D[(D.event == ev) & (D.version == ver) & (D.world == w)]; ex = d.expected.iloc[0] if len(d) else np.nan
        def ok(win):
            x = d[d.window == win].set_index("split"); 
            if not {"val", "test"} <= set(x.index): return False
            return bool(np.sign(x.loc["val", "mean"]) == ex and np.sign(x.loc["test", "mean"]) == ex and abs(x.loc["test", "t"]) > thr_t)
        late = [wn for wn in d.window.unique() if wn.endswith("m") and int(wn.split("-")[0]) >= 30]
        out.append(dict(event=ev, version=ver, world=w, short_30m=ok("0-30m"), beyond_30m=any(ok(wn) for wn in late) if late else np.nan))
    return pd.DataFrame(out)
KC_RES, KC_A5 = {}, None
if KC_RAW5 is not None and KC_PRODS:
    for KCB in K_BARS:
        t0 = time.time(); bases = {p: kc_base(kc_agg(kc_prep5(p), KCB), KCB) for p in KC_PRODS}; bases = {p: b for p, b in bases.items() if len(b) > 2000}
        if not bases: continue
        wins = kc_windows_for(KCB); A, S = kc_assemble(bases, wins, 30 // KCB, False, K_LB); res = kc_run(f"{KCB}m", A, S, "0-30m", K_LB, wins); res["rows"] = int(len(S["spl"])); KC_RES[f"{KCB}m"] = res
        if KCB == 5: KC_A5 = (A, S, wins)
        else: del A
        print(f"{KCB}-minute bars: {len(S['spl']):,} candles in {len(bases)} products, {time.time() - t0:.0f}s")
# daily candle: the 'beyond 30 minutes' horizon (next day, next week, next month)
KC_DW = {"d1": (0, 1), "d2-5": (1, 5), "d6-20": (5, 20)}
_kcbd = {p: kc_base(kc_daily(p), 1440, daily=True) for p in (KC_PRODS or UNIV[:12])}; _kcbd = {p: b for p, b in _kcbd.items() if len(b) > 800}
for p in _kcbd:
    if p not in KC_REG: KC_REG[p] = kc_regime(p)
_kcAd, _kcSd = kc_assemble(_kcbd, KC_DW, 0, True, K_LBD); KC_RES["1D"] = kc_run("1D", _kcAd, _kcSd, "d1", K_LBD, KC_DW); KC_RES["1D"]["rows"] = int(len(_kcSd["spl"])); del _kcAd
print(f"daily candles: {KC_RES['1D']['rows']:,} in {len(_kcbd)} products")
KC_SELALL = pd.concat([r["SEL"] for r in KC_RES.values()]); KC_EFFALL = pd.concat([r["R"] for r in KC_RES.values()]); KC_DECAY = pd.concat([r["D"] for r in KC_RES.values()]); KC_THR = pd.concat([r["thr"] for r in KC_RES.values()]); KC_DRIFT = pd.concat([r["drift"] for r in KC_RES.values()])
KC_SELALL.round(4).to_csv(OUT2 / "candle_event_selection.csv", index=False); KC_EFFALL.round(4).to_csv(OUT2 / "candle_event_effects.csv", index=False); KC_DECAY.round(4).to_csv(OUT2 / "candle_decay.csv", index=False); KC_THR.round(4).to_csv(OUT2 / "candle_thresholds.csv", index=False); KC_DRIFT.round(5).to_csv(OUT2 / "candle_drift_worlds.csv", index=False)
print("\nConfiguration chosen on validation (worst world x regime cell, plateau rule); test_rank = rank of the chosen one on test among all configurations (1 = best):")
_kcc = KC_SELALL[KC_SELALL.chosen][["bar", "event", "q", "L", "val_worst_cell_t", "plateau_score", "test_aligned_t_mean_worlds", "test_rank"]]; print(_kcc.round(2).to_string(index=False))

In [ ]:
# ---- 18k.3 continuous view (regressions), what survives beyond 30 minutes, charts, summary ----
def kc_ols(X, y, day, nday):
    ok = np.isfinite(y) & np.isfinite(X).all(1); X1 = np.c_[np.ones(int(ok.sum())), X[ok].astype(float)]; y = y[ok].astype(float); d = day[ok]
    if len(y) < 200: return None
    XtX = X1.T @ X1; b = np.linalg.solve(XtX, X1.T @ y); e = y - X1 @ b; sc = X1 * e[:, None]
    G = np.stack([np.bincount(d, weights=sc[:, j], minlength=nday) for j in range(X1.shape[1])], 1); Vi = np.linalg.inv(XtX); V = Vi @ (G.T @ G) @ Vi
    return b, b / np.sqrt(np.diag(V)), len(y)
KC_RDIR, KC_RVOL = None, None
if KC_A5 is not None:
    A5, S5, W5_ = KC_A5; a = A5[1]; nday = int(S5["day"].max()) + 1; sgn = np.sign(a["rz"])
    cols = {"bar move (rz)": a["rz"], "lower minus upper wick": a["lw"] - a["uw"], "signed body (sign x BRR)": sgn * a["brr"], "range vs normal": a["rng"], "volume vs normal": a["vz"], "volume x signed body": a["vz"] * sgn * a["brr"],
            "trailing 6-bar move": a["tr6"], "log|high-open|/|high-close|": a["rh"], "log|low-open|/|low-close|": a["rl"]}
    names = list(cols); X = np.c_[[np.clip(v, *np.nanpercentile(v[S5["spl"] == 0], [1, 99])) for v in cols.values()]].T
    mu_, sd_ = np.nanmean(X[S5["spl"] == 0], 0), np.nanstd(X[S5["spl"] == 0], 0); X = (X - mu_) / sd_; rows = []
    for sp_, spn in enumerate(("train", "val", "test")):
        for rg_, rgn_ in [(None, "all")] + [(r, n_) for r, n_ in enumerate(("low", "mid", "high"))]:
            m = (S5["spl"] == sp_) & ((S5["reg"] == rg_) if rg_ is not None else True); o = kc_ols(X[m], a["r_0-30m"][m], S5["day"][m], nday)
            if o is None: continue
            for j, nm in enumerate(names): rows.append(dict(split=spn, regime=rgn_, variable=nm, beta=o[0][j + 1], t=o[1][j + 1], n=o[2]))
    KC_RDIR = pd.DataFrame(rows); KC_RDIR.round(4).to_csv(OUT2 / "candle_regression_direction.csv", index=False)
    print("Next-30-minute return on the candle, 5-minute bars (standardised regressors, day-clustered t); regime 'all':"); print(KC_RDIR[KC_RDIR.regime == "all"].pivot(index="variable", columns="split", values="t").reindex(names).round(2).to_string())
    # volatility: log of the forward range in each window on today's candle, controlling for the trailing range level
    vcols = {"range vs normal": a["rng"], "body (BRR)": a["brr"], "wicks (upper + lower)": a["uw"] + a["lw"], "volume vs normal": a["vz"], "trailing range level": a["trr"]}; vn = list(vcols)
    XV = np.c_[[np.clip(v, *np.nanpercentile(v[S5["spl"] == 0], [1, 99])) for v in vcols.values()]].T; XV = (XV - np.nanmean(XV[S5["spl"] == 0], 0)) / np.nanstd(XV[S5["spl"] == 0], 0); rows = []
    for nm in W5_:
        yv = np.log(np.where(a["R_" + nm] > 0, a["R_" + nm], np.nan))
        for sp_, spn in enumerate(("train", "val", "test")):
            o = kc_ols(XV[S5["spl"] == sp_], yv[S5["spl"] == sp_], S5["day"][S5["spl"] == sp_], nday)
            if o is None: continue
            for j, v_ in enumerate(vn): rows.append(dict(window=nm, split=spn, variable=v_, beta=o[0][j + 1], t=o[1][j + 1], n=o[2]))
    KC_RVOL = pd.DataFrame(rows); KC_RVOL.round(4).to_csv(OUT2 / "candle_regression_volatility.csv", index=False)
    _kco = [w for w in W5_ if w != "0-30m"]; print("\nLog forward range on today's range (coefficient per window; the trailing range level is controlled); 'late' = windows from 30 minutes on:")
    print(KC_RVOL[(KC_RVOL.variable == "range vs normal") & KC_RVOL.window.isin(_kco)].pivot(index="window", columns="split", values="beta").reindex(_kco).round(3).to_string())
    del A5, S5, KC_A5
# ---- what supports, and what survives beyond 30 minutes ----
NT = len(KC_EV) * max(len(KC_RES), 1); T_BON = float(_norm.ppf(1 - 0.025 / NT)); print(f"\nBonferroni over {NT} event x bar-size tests: |t| > {T_BON:.2f} on test; 'suggestive' = |t| > 2")
V = []
for tag, r in KC_RES.items():
    for ev in KC_EV:
        for ver in ("learned", "rulebook"):
            for thr_, lab in ((T_BON, "supported"), (2.0, "suggestive")):
                b = kc_beyond(r["D"], ev, ver, thr_); b["bar"] = tag; b["level"] = lab; V.append(b)
KC_V = pd.concat(V, ignore_index=True) if V else pd.DataFrame(); KC_V.to_csv(OUT2 / "candle_verdicts.csv", index=False)
KC_TAB = (KC_V[(KC_V.version == "learned") & (KC_V.level == "supported")].pivot_table(index=["bar", "event"], columns="world", values=["short_30m", "beyond_30m"], aggfunc="first") if len(KC_V) else pd.DataFrame())
print("\nLearned thresholds, level 'supported': inside 30 minutes / beyond 30 minutes, for the series (+1) and the negated series (-1):"); print(KC_TAB.to_string() if len(KC_TAB) else "nothing to show")
KC_SUG = KC_V[(KC_V.version == "learned") & (KC_V.level == "suggestive")]
# ---- charts ----
GREEN, RED = "#1baf7a", "#e34948"; _kcb0 = "5m" if "5m" in KC_RES else ("1D" if "1D" in KC_RES else None)
if _kcb0:
    fig, axs = plt.subplots(1, 4, figsize=(18, 4.2)); T_ = KC_THR[(KC_THR.bar == _kcb0) & (KC_THR.world == 1) & (KC_THR.q == 0.75)]; rb = {"uw": K_RULEBOOK["uw"], "brr": K_RULEBOOK["brr"], "rng": K_RULEBOOK["rng"], "vz": K_RULEBOOK["vz"]}
    for ax, (f, tt) in zip(axs, (("uw", "upper wick / range"), ("brr", "body / range"), ("rng", "log(range / normal range)"), ("vz", "log(volume / normal volume)"))):
        g = T_[T_.feature == f].set_index("regime").reindex(["low", "mid", "high"]); ax.bar(range(3), g.cut, color=SERIES[:3]); ax.axhline(rb[f], color=RED, ls="--", lw=1.2, label=f"rulebook {rb[f]:.2f}"); ax.set_xticks(range(3)); ax.set_xticklabels(["low vol", "mid vol", "high vol"]); ax.set_title(tt + "  (75th percentile, train)", fontsize=9); ax.legend(fontsize=7)
    plt.tight_layout(); savefig(fig, "18k_thresholds", "Learned candle thresholds by volatility regime against the rulebook values", "18k")
    fig, axs = plt.subplots(1, 3, figsize=(18, 4.8)); Dd = KC_RES[_kcb0]["D"]; wl = list(dict.fromkeys(Dd.window)); wl = sorted(wl, key=lambda x: (int(x.split("-")[1].rstrip("m")) if x.endswith("m") else 99, x))
    for ax, evs, ttl in ((axs[0], ["UWR_L", "BRK_L", "FAKE_L"], "direction events: effect by forward window (test, series +1 solid, negated -1 dashed)"), (axs[1], ["EXP", "CON"], "range events: excess forward range by window (test)")):
        for k, ev in enumerate(evs):
            for w, ls in ((1, "-"), (-1, "--")):
                g = Dd[(Dd.event == ev) & (Dd.version == "learned") & (Dd.world == w) & (Dd.split == "test") & (Dd.window != "0-30m")].set_index("window").reindex([x for x in wl if x != "0-30m"])
                ax.plot(range(len(g)), g["mean"] * KC_EV[ev][0] if KC_EV[ev][1] == "r" else g["mean"], ls, color=SERIES[k], marker="o", ms=4, label=f"{ev}" if w == 1 else None)
        ax.axhline(0, color="#c9c7c2"); ax.set_xticks(range(len([x for x in wl if x != "0-30m"]))); ax.set_xticklabels([x for x in wl if x != "0-30m"], fontsize=8); ax.set_title(ttl, fontsize=9); ax.legend(fontsize=7); ax.set_xlabel("forward window"); ax.axvline(3.5, color=RED, lw=.8, ls=":")
    if KC_RVOL is not None:
        for k, sp_ in enumerate(("train", "val", "test")):
            g = KC_RVOL[(KC_RVOL.variable == "range vs normal") & (KC_RVOL.split == sp_) & (KC_RVOL.window != "0-30m")].set_index("window").reindex([x for x in wl if x != "0-30m"]); axs[2].plot(range(len(g)), g.beta, marker="o", ms=4, color=SERIES[k], label=sp_)
        axs[2].axhline(0, color="#c9c7c2"); axs[2].axvline(3.5, color=RED, lw=.8, ls=":"); axs[2].set_xticks(range(len([x for x in wl if x != "0-30m"]))); axs[2].set_xticklabels([x for x in wl if x != "0-30m"], fontsize=8); axs[2].set_title("log forward range on today's range (volatility control kept): decay by window", fontsize=9); axs[2].legend(fontsize=7)
    plt.tight_layout(); savefig(fig, "18k_decay", "How long a candle's information lasts: forward windows up to 120 minutes (dotted line = 30 minutes)", "18k")
    fig, axs = plt.subplots(1, len(KC_RES), figsize=(5.4 * len(KC_RES), 4.8), squeeze=False); evs = [e for e in KC_EV if KC_EV[e][1] == "r"]
    for ax, (tag, r) in zip(axs[0], KC_RES.items()):
        Rr = r["R"][(r["R"].version == "learned") & (r["R"].regime == "all") & (r["R"].split == "test")]; ch = r["SEL"][r["SEL"].chosen]
        for k, w in enumerate((1, -1)):
            vals = []
            for ev in evs:
                c = ch[ch.event == ev]; 
                if not len(c): vals.append(np.nan); continue
                g = Rr[(Rr.event == ev) & (Rr.world == w) & (Rr.q == float(c.q.iloc[0])) & ((Rr.L == c.L.iloc[0]) if pd.notna(c.L.iloc[0]) else Rr.L.isna())]; vals.append(float(g.aligned_mean.iloc[0]) if len(g) else np.nan)
            ax.bar(np.arange(len(evs)) + (k - .5) * .38, vals, width=.38, color=(GREEN if w == 1 else RED), label=("series" if w == 1 else "negated series"))
        ax.axhline(0, color="#c9c7c2"); ax.set_xticks(range(len(evs))); ax.set_xticklabels(evs, rotation=25, fontsize=8); ax.set_title(f"{tag}: effect in the expected direction, test (30 min / next day)", fontsize=9); ax.legend(fontsize=7)
    plt.tight_layout(); savefig(fig, "18k_worlds", "Bullish structures versus their mirror image on the negated series", "18k")
# ---- summary ----
def _kcfirst(df, **kw):
    m = np.ones(len(df), bool)
    for k, v in kw.items(): m &= (df[k] == v).to_numpy()
    return df[m]
KC_SUMMARY = dict(bars=list(KC_RES), rows={k: v["rows"] for k, v in KC_RES.items()}, opening_price_stored=bool(KC_HAS_OPEN), bonferroni_t=T_BON, thresholds_q75_5m=(KC_THR[(KC_THR.bar == _kcb0) & (KC_THR.world == 1) & (KC_THR.q == 0.75)].round(3).to_dict("records") if _kcb0 else None),
                  chosen=KC_SELALL[KC_SELALL.chosen].round(3).to_dict("records"), verdicts_supported=(KC_V[(KC_V.version == "learned") & (KC_V.level == "supported")].to_dict("records") if len(KC_V) else []),
                  verdicts_suggestive=KC_SUG.to_dict("records"), drift_test=KC_DRIFT[KC_DRIFT.split == "test"].round(5).to_dict("records"))
_kcsup = KC_V[(KC_V.version == "learned") & (KC_V.level == "supported") & (KC_V.short_30m == True)] if len(KC_V) else pd.DataFrame(); _kcbey = KC_V[(KC_V.version == "learned") & (KC_V.level == "suggestive") & (KC_V.beyond_30m == True)] if len(KC_V) else pd.DataFrame()
_kcrb = KC_V[(KC_V.version == "rulebook") & (KC_V.level == "suggestive") & (KC_V.short_30m == True)] if len(KC_V) else pd.DataFrame()
def _kclab(df): return ", ".join(sorted(set("{} {} ({})".format(r.bar, r.event, "series" if r.world > 0 else "negated") for r in df.itertuples()))) or "none"
_kcdr = [(d["bar"], d["world"], round(d["mean_fwd"], 4)) for d in KC_SUMMARY["drift_test"] if d["world"] == 1]
KC_SUMMARY["reading"] = ("Candle anatomy studied on " + ", ".join(KC_RES) + " candles with every threshold learned per volatility regime (train quantiles; rulebook values kept as a comparison) and every structure also run on the negated series. "
    "Supported inside 30 minutes at |t| > {:.1f} on test with the learned thresholds: {}. ".format(T_BON, _kclab(_kcsup)) + "Still suggestive (|t| > 2, same sign in validation and test) beyond 30 minutes: {}. ".format(_kclab(_kcbey))
    + "Rulebook values, same suggestive test inside 30 minutes: {}. ".format(_kclab(_kcrb)) + "Drift of the series on test (mean next-window z; the negated series has the opposite sign by construction): {}.".format(_kcdr))
print("\n" + KC_SUMMARY["reading"])


## Step 18l — Threshold atlas: the market's own break points, regime by regime

**Why.** Step 18k compared three quantile levels (0.6 / 0.75 / 0.9) and three look-backs, chose one setting per bar size and event on validation, and kept the rulebook values as a comparison. That evaluates a nine-point grid, once, not per regime. The right question is the reverse: *where does the market itself change behaviour?* This step lets the data define the thresholds.

**Method (per bar size 5 / 15 / 30 minutes, per regime, per feature).**
- **Regimes:** the product's volatility regime (low / mid / high, as in 18k) and, separately, the market trend regime from Step 13f (bear / range / bull). Periods are the regime-balanced `train_R` / `val_R` of Step 13f; the test is chronological and read once.
- **Features:** range against the same-time-of-day median, body / range, upper and lower wick / range, the two rejection ratios (|high−open| / |high−close| and |open−low| / |close−low|, as logs) and volume against the same-time median. Every feature is a ratio, so it is comparable across products.
- **Outcomes:** the next-window signed return (the series and the negated series) and the next-window range, each in excess of its own mean in that regime and period; windows 0–30 minutes and 30–60 minutes, so the decay beyond 30 minutes is measured, not assumed.
- **Break search:** for each side (upper tail, lower tail) the tail start is searched over the 50th, 60th, 70th, 80th, 90th, 95th, 97th and 99th percentile (a solution on the last grid point, or on a point mass of the feature, is flagged as boundary or saturated) of the feature in the regime on `train_R`, and the one with the largest date-clustered |t| is kept. Its value (the edge) and the share of candles beyond it are the learned threshold.
- **Check:** the same edge is applied to `val_R` and the test. *Supported* = same sign in all three periods, |t| > 2 on validation and above the Bonferroni level on the test over all searches.
- **Comparison:** at the same searches the table gives the |t| of the fixed 75th percentile and of the rulebook value (uw 0.5, body 0.7, range ×1.5, volume ×1.5), where it exists, and the quantile at which the rulebook value actually sits in each regime.
- **Response curves:** 20-bin curves of the excess outcome against the feature quantile, per regime, train against test, for the 15-minute bars: the shape shows whether there is a break at all, or a smooth slope that no threshold would capture.

**Reading rules.** A break point is only a description if it is stable across periods; a learned quantile that moves a lot between regimes shows that one fixed value would be wrong; a break that is the same everywhere shows that a fixed quantile is fine. The range outcome mostly measures volatility clustering; the signed outcomes are where a directional rule would live. The series and negated-series rows are mirror images (an upper wick in one world is a lower wick in the other), so they are not independent evidence. Intraday bars only: the daily candle is not included here.

**Used later.** The learned break points and the nested-selection comparison with the rulebook values set the candle thresholds used by Step 18m and by any intraday rule that follows; the regime spread tells which thresholds must be regime-specific.

In [ ]:
# step 18l
KT_OK = bool(globals().get("RG_OK", False)) and all(k in globals() for k in ("kc_assemble", "kc_base", "kc_agg", "kc_prep5", "kc_windows_for", "KC_REG", "KC_PRODS", "SPL_R", "RG_IDX", "RG", "K_RULEBOOK", "bonferroni_t")) and globals().get("KC_RAW5") is not None
if not KT_OK:
    print("Step 18l needs Step 13f (regimes), Step 18k (candle anatomy) and the cached 5-minute bars: run them first; skipped")
else:
    # ---- 18l.1 the thresholds are not assumed: response curves and the break point the market itself shows, per regime ----
    KT_TAILS = (0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.97, 0.99); KT_FOLDS = 5; KT_NB = 20; KT_BARS = (5, 15, 30); KT_WIN = ("0-30m", "30-60m")
    KT_FEATS = dict(rng="range vs same-time median (log)", brr="body / range", uw="upper wick / range", lw="lower wick / range", rh="log |high-open| / |high-close|", rl="log |open-low| / |close-low|", vz="volume vs same-time median (log)")
    KT_RULE = dict(uw=("hi", K_RULEBOOK["uw"]), brr=("hi", K_RULEBOOK["brr"]), rng=("hi", K_RULEBOOK["rng"]), vz=("hi", K_RULEBOOK["vz"]))
    KT_SPLR = pd.Series(-1, index=RAW.index)
    for _ktk, _ktn in (("train", 0), ("val", 1), ("test", 2)): KT_SPLR.loc[RG_IDX[_ktk]] = _ktn
    def kt_assemble(bases, wins, short_bars, Ls):
        # as kc_assemble, but periods follow the regime-balanced split of Step 13f, and the calendar date is kept so that the market regimes can be attached
        parts = {1: [], -1: []}; sh = []
        for k, (p, b) in enumerate(bases.items()):
            dts = pd.Timestamp("2000-01-01") + pd.to_timedelta(b["tdi"].to_numpy(), unit="D")
            sp = KT_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); rg = KC_REG[p].reindex(dts).fillna(-1).to_numpy().astype(int); tr = RG["T"].reindex(dts).fillna(-1).to_numpy().astype(int); W = kc_wins(b, wins, short_bars, False)
            for s in (1, -1):
                f = kc_world(b, s, Ls); keep = (sp >= 0) & (rg >= 0) & (tr >= 0) & f["rng"].notna().to_numpy()
                d = {c: f[c].to_numpy()[keep].astype(np.float32) for c in KT_FEATS}
                for nm, (r, R) in W.items():
                    if nm in KT_WIN: d["r_" + nm] = (s * r[keep]).astype(np.float32); d["R_" + nm] = R[keep]
                parts[s].append(d)
            sh.append(dict(spl=sp[keep], vol=rg[keep], trend=tr[keep], day=b["tdi"].to_numpy()[keep]))
        A = {s: {c: np.concatenate([d[c] for d in parts[s]]) for c in parts[s][0]} for s in (1, -1)}; S = {c: np.concatenate([d[c] for d in sh]) for c in sh[0]}
        S["dayi"] = np.unique(S["day"], return_inverse=True)[1].astype(np.int64); return A, S
    def kt_group(y, day, g, nday):
        g = g & np.isfinite(y); n = int(g.sum())
        if n < 30: return n, np.nan, np.nan
        m = float(y[g].mean()); sres = np.bincount(day[g], weights=y[g] - m, minlength=nday); se = np.sqrt((sres ** 2).sum()) / n
        return n, m, (m / se if se > 0 else np.nan)
    def kt_excess(y, ratio):
        mu = np.nanmean(y) if np.isfinite(y).any() else np.nan
        return (y / mu - 1) if ratio else (y - mu)
    # unit check: the break-point search must find a planted threshold
    _ktr = np.random.default_rng(0); _ktx = _ktr.normal(size=40000); _kty = _ktr.normal(size=40000) + 0.5 * (_ktx > np.quantile(_ktx, 0.8)); _kty = _kty - _kty.mean(); _ktd = np.arange(40000) // 20; _ktn = int(_ktd.max()) + 1
    _ktt = {q: kt_group(_kty, _ktd, _ktx >= np.quantile(_ktx, q), _ktn)[2] for q in KT_TAILS}; assert max(_ktt, key=lambda q: abs(_ktt[q])) in (0.7, 0.8), "break-point search broken"; print("unit check passed: the search recovers a planted break at the 80th percentile")
    def kt_nested(X, Y, D, side, rule_edge, nday, K=KT_FOLDS, emb=None):
        # nested selection: the tail start is chosen on K-1 blocks of train_R + val_R (blocks purged by `emb` trading days around the held-out block) and applied to the held-out block;
        # the held-out results of the blocks are pooled. Learned, fixed 75th percentile and the rulebook value are scored on exactly the same held-out candles.
        emb = EMB if emb is None else emb; ok = np.isfinite(X) & np.isfinite(Y); X, Y, D = X[ok], Y[ok], D[ok]
        if len(X) < 4000: return None
        ed = np.quantile(np.unique(D), np.linspace(0, 1, K + 1)); fold = np.clip(np.searchsorted(ed, D, side="right") - 1, 0, K - 1); keep = {"learned": [], "fixed75": [], "rule": []}; qs = []
        for f in range(K):
            te = fold == f
            if te.sum() < 200: continue
            tr = (~te) & ((D < D[te].min() - emb) | (D > D[te].max() + emb))
            if tr.sum() < 1000: continue
            xt = X[tr]; cut = lambda q: np.quantile(xt, q if side == "upper" else 1 - q); inb = lambda x, e: (x >= e) if side == "upper" else (x <= e)
            tt = {q: kt_group(Y[tr], D[tr], inb(X[tr], cut(q)), nday)[2] for q in KT_TAILS}; okq = [q for q in KT_TAILS if np.isfinite(tt[q])]
            if not okq: continue
            q = max(okq, key=lambda z: abs(tt[z])); qs.append(q)
            for nm, e in (("learned", cut(q)), ("fixed75", cut(0.75)), ("rule", rule_edge)):
                if e is None: continue
                g = inb(X[te], e); keep[nm].append((Y[te][g], D[te][g]))
        out = {}
        for nm, v in keep.items():
            if not v: continue
            y = np.concatenate([a for a, _ in v]); d = np.concatenate([b for _, b in v]); n_, m_, t_ = kt_group(y, d, np.ones(len(y), bool), nday); out[nm] = (m_, t_)
        return dict(res=out, q_median=float(np.median(qs)) if qs else np.nan, q_range=float(np.ptp(qs)) if qs else np.nan)
    # unit check: nested selection on pure noise must not find a significant tail
    _ktr2 = np.random.default_rng(3); _kx = _ktr2.normal(size=60000); _ky = _ktr2.normal(size=60000); _kd = np.arange(60000) // 20; _kn = kt_nested(_kx, _ky, _kd, "upper", None, int(_kd.max()) + 1, emb=5)
    assert abs(_kn["res"]["learned"][1]) < 3.5 and abs(_kn["res"]["fixed75"][1]) < 3.5, "nested selection broken"; print("unit check passed: nested selection on noise finds nothing")
    KT_ROWS = []; KT_CURVE = []; KT_T0 = time.time()
    for BAR in KT_BARS:
        bases = {p: kc_base(kc_agg(kc_prep5(p), BAR), BAR) for p in KC_PRODS}; bases = {p: b for p, b in bases.items() if len(b) > 2000}
        if not bases: continue
        wins = kc_windows_for(BAR); KA, KS = kt_assemble(bases, {w: wins[w] for w in KT_WIN if w in wins}, 30 // BAR, K_LB); nday = int(KS["dayi"].max()) + 1
        for dim in ("vol", "trend"):
            names = ("low", "mid", "high") if dim == "vol" else RG_TN
            for k, rnm in enumerate(names):
                sub = {sp: np.where((KS["spl"] == sp) & (KS[dim] == k))[0] for sp in (0, 1, 2)}
                if min(len(v) for v in sub.values()) < 500: continue
                for w in KT_WIN:
                    if "r_" + w not in KA[1]: continue
                    outs = {"ret_series": (KA[1]["r_" + w], 1, False), "ret_negated": (KA[-1]["r_" + w], -1, False), "range": (KA[1]["R_" + w], 1, True)}
                    for on, (yy, world, ratio) in outs.items():
                        Y = {sp: kt_excess(yy[sub[sp]].astype(float), ratio) for sp in sub}
                        for f in KT_FEATS:
                            X = {sp: KA[world if on != "range" else 1][f][sub[sp]].astype(float) for sp in sub}; D = {sp: KS["dayi"][sub[sp]] for sp in sub}; xt = X[0][np.isfinite(X[0])]
                            if len(xt) < 500: continue
                            for side in ("upper", "lower"):
                                cut = lambda q: np.quantile(xt, q if side == "upper" else 1 - q); inb = lambda x, e: (x >= e) if side == "upper" else (x <= e)
                                tt = {q: kt_group(Y[0], D[0], inb(X[0], cut(q)), nday) for q in KT_TAILS}; ok = [q for q in KT_TAILS if np.isfinite(tt[q][2])]
                                if not ok: continue
                                qs = max(ok, key=lambda q: abs(tt[q][2])); e = cut(qs); res = {sp: kt_group(Y[sp], D[sp], inb(X[sp], e), nday) for sp in (0, 1, 2)}
                                r = dict(bar=f"{BAR}m", dim=dim, regime=rnm, feature=f, window=w, outcome=on, side=side, q_star=qs, edge=float(e), n_train=res[0][0], mean_train=res[0][1], t_train=res[0][2], n_val=res[1][0], mean_val=res[1][1], t_val=res[1][2], n_test=res[2][0], mean_test=res[2][1], t_test=res[2][2])
                                f75 = {sp: kt_group(Y[sp], D[sp], inb(X[sp], cut(0.75)), nday) for sp in (0, 1, 2)}; r.update(t_train_fixed75=f75[0][2], t_val_fixed75=f75[1][2], t_test_fixed75=f75[2][2], mean_test_fixed75=f75[2][1])
                                if f in KT_RULE and side == "upper":
                                    er = KT_RULE[f][1]; rr = {sp: kt_group(Y[sp], D[sp], X[sp] >= er, nday) for sp in (0, 1, 2)}; r.update(rule_edge=er, rule_quantile=float((xt < er).mean()), t_train_rule=rr[0][2], t_val_rule=rr[1][2], t_test_rule=rr[2][2], mean_test_rule=rr[2][1])
                                r["at_grid_edge"] = bool(qs == max(KT_TAILS)); r["saturated"] = bool(f in ("rh", "rl") and abs(e) >= 3.9)
                                Xp = np.concatenate([X[0], X[1]]); Yp = np.concatenate([Y[0], Y[1]]); Dp = np.concatenate([D[0], D[1]]); nr = kt_nested(Xp, Yp, Dp, side, (KT_RULE[f][1] if (f in KT_RULE and side == "upper") else None), nday)
                                if nr:
                                    for nm in ("learned", "fixed75", "rule"):
                                        if nm in nr["res"]: r["nested_mean_" + nm], r["nested_t_" + nm] = nr["res"][nm]
                                    r["nested_q_median"], r["nested_q_range"] = nr["q_median"], nr["q_range"]
                                KT_ROWS.append(r)
                            if on in ("range", "ret_series") and BAR in (15,) and w == "0-30m":
                                ed = np.quantile(xt, np.linspace(0, 1, KT_NB + 1)); ed[0], ed[-1] = -np.inf, np.inf
                                for sp in (0, 2):
                                    bi = np.clip(np.searchsorted(ed, X[sp], side="right") - 1, 0, KT_NB - 1)
                                    for b_ in range(KT_NB):
                                        n_, m_, t_ = kt_group(Y[sp], D[sp], bi == b_, nday); KT_CURVE.append(dict(bar=f"{BAR}m", dim=dim, regime=rnm, feature=f, outcome=on, split="train_R" if sp == 0 else "test", bin=b_, n=n_, mean=m_, t=t_))
        print(f"{BAR}-minute bars: {len(KS['spl']):,} candles, {time.time() - KT_T0:.0f}s elapsed")
    KT = pd.DataFrame(KT_ROWS); KT_CV = pd.DataFrame(KT_CURVE)
    print(f"break-point searches: {len(KT):,} (bar x regime dimension x regime x feature x window x outcome x side)")

In [ ]:
# step 18l
if KT_OK and len(KT):
    # ---- 18l.2 decisions: fixed on train_R, checked on val_R, read once on the test; learned break point versus the fixed grid and the rulebook ----
    KT["finite"] = KT[["t_train", "t_val", "t_test"]].notna().all(axis=1); NT_KT = int(KT.finite.sum()); T_KT = bonferroni_t(max(NT_KT, 1))
    KT["same_sign"] = KT.finite & (np.sign(KT.mean_train) == np.sign(KT.mean_val)) & (np.sign(KT.mean_val) == np.sign(KT.mean_test))
    KT["supported"] = KT.same_sign & (KT.t_val.abs() > 2) & (KT.t_test.abs() > T_KT)
    # de-duplicate mirror rows: the negated series is the series seen upside down, and upper wick / lower wick, rh / rl and tail sides of the same candles repeat each other. Same sample sizes and same absolute means = the same finding.
    KT["share_of_candles"] = 1 - KT.q_star
    KT["sig"] = KT.bar + "|" + KT.dim + "|" + KT.regime + "|" + KT.window + "|" + np.where(KT.outcome == "range", "R", "S") + "|" + KT.n_train.astype(str) + "|" + KT.n_test.astype(str) + "|" + KT.mean_train.abs().round(5).astype(str) + "|" + KT.mean_test.abs().round(5).astype(str)
    KT["_p"] = KT.outcome.map({"ret_series": 0, "range": 0, "ret_negated": 1}) + (KT.side == "lower") * 0.1; KT = KT.sort_values(["sig", "_p"]).reset_index(drop=True); KT["mirror"] = KT.duplicated("sig", keep="first"); KT_U = KT[~KT.mirror]
    KT.drop(columns=["_p"]).round(4).to_csv(OUT2 / "threshold_atlas_all.csv", index=False)
    print(f"Searches: {len(KT):,}; distinct after removing mirror rows: {len(KT_U):,}. Supported: {int(KT.supported.sum())} rows = {int(KT_U.supported.sum())} distinct findings.")
    print("\nDistinct supported findings by bar and regime (columns: regime dimension and level):")
    _cnt = KT_U[KT_U.supported].assign(col=lambda d: d.dim + ":" + d.regime).groupby(["bar", "col"]).size().unstack(fill_value=0); print(_cnt.to_string() if len(_cnt) else "none")
    print("\nDistinct supported findings by outcome, window and bar:"); _c2 = KT_U[KT_U.supported].groupby(["outcome", "window", "bar"]).size().unstack(fill_value=0); print(_c2.to_string() if len(_c2) else "none")
    print(f"searches whose chosen tail start sits at the edge of the grid ({max(KT_TAILS):g}): {int(KT_U.at_grid_edge.sum())} of {len(KT_U)}; saturated rejection-ratio edges: {int(KT_U.saturated.sum())}")
    if "nested_t_learned" in KT_U:
        _n = KT_U[KT_U[["nested_t_learned", "nested_t_fixed75"]].notna().all(axis=1)]
        print(f"\nNested selection (tail start chosen inside {KT_FOLDS} purged folds of train_R + val_R, scored on the held-out blocks), {len(_n)} distinct searches: mean |t| learned {_n.nested_t_learned.abs().mean():.2f}, fixed 75th percentile {_n.nested_t_fixed75.abs().mean():.2f}"
              + (f", rulebook {_n.nested_t_rule.abs().mean():.2f} (on the {int(_n.nested_t_rule.notna().sum())} searches that have a rule value)" if "nested_t_rule" in _n else ""))
        print(f"learned beats the fixed 75th percentile in {(_n.nested_t_learned.abs() > _n.nested_t_fixed75.abs()).mean():.0%} of searches; the learned tail start moves by a median of {_n.nested_q_range.median():.2f} between folds (stability of the break)")
    print(f"{NT_KT:,} break-point searches with all three periods; Bonferroni |t| > {T_KT:.2f} on test (and |t| > 2 on validation, same sign in train_R, val_R, test).")
    THR = KT_U[KT_U.supported].sort_values("t_test", key=abs, ascending=False)[["bar", "dim", "regime", "feature", "window", "outcome", "side", "q_star", "edge", "share_of_candles", "mean_train", "mean_val", "mean_test", "t_test"]]; THR.round(4).to_csv(OUT2 / "learned_thresholds.csv", index=False)
    print("\nThresholds the data supports (edge = value of the feature at the learned break; share = fraction of candles beyond it), strongest first:"); print(THR.head(30).round(3).to_string(index=False) if len(THR) else "none")
    # does the break point differ by regime? (spread of the learned quantile across regimes inside the same dimension, for results supported in at least two regimes)
    sp_ = KT_U[KT_U.supported].groupby(["bar", "dim", "feature", "window", "outcome", "side"]).agg(regimes=("regime", "nunique"), q_min=("q_star", "min"), q_max=("q_star", "max"), edge_min=("edge", "min"), edge_max=("edge", "max")).reset_index(); sp_ = sp_[sp_.regimes >= 2]
    KT_SPREAD = sp_; KT_SPREAD.round(3).to_csv(OUT2 / "threshold_regime_spread.csv", index=False)
    print("\nSame result supported in two or more regimes: how far the learned break moves between them:"); print(KT_SPREAD.round(3).to_string(index=False) if len(KT_SPREAD) else "none")
    # learned versus fixed grid versus rulebook (validation and test t, same events)
    rl_ = KT[KT.finite & KT.rule_edge.notna()] if "rule_edge" in KT else KT.iloc[0:0]
    if len(rl_):
        cmp_ = pd.DataFrame(dict(learned_val=rl_.t_val.abs(), fixed75_val=rl_.t_val_fixed75.abs(), rule_val=rl_.t_val_rule.abs(), learned_test=rl_.t_test.abs(), fixed75_test=rl_.t_test_fixed75.abs(), rule_test=rl_.t_test_rule.abs()))
        print(f"\nMean |t| over {len(cmp_)} comparable searches (learned break chosen on train_R; the benchmark rulebook value; the fixed 75th percentile):"); print(cmp_.mean().round(2).to_string())
        print(f"share where the learned break has the larger |t| on validation than the rulebook value: {(cmp_.learned_val > cmp_.rule_val).mean():.0%}; on test: {(cmp_.learned_test > cmp_.rule_test).mean():.0%}")
        print("rulebook value sits at this quantile of the feature (median over regimes):", rl_.groupby("feature").rule_quantile.median().round(2).to_dict())
    KT_BEYOND = KT_U[KT_U.supported].groupby("window").size().to_dict(); print("\nSupported inside 30 minutes vs beyond (30-60 minutes):", KT_BEYOND)
    # ---- figures ----
    _h = KT[(KT.bar == "15m") & (KT.outcome == "range") & (KT.window == "0-30m") & (KT.side == "upper")].assign(col=lambda d: d.dim + ":" + d.regime).pivot(index="feature", columns="col", values="q_star")
    _hs = KT[(KT.bar == "15m") & (KT.outcome == "range") & (KT.window == "0-30m") & (KT.side == "upper")].assign(col=lambda d: d.dim + ":" + d.regime).pivot(index="feature", columns="col", values="supported").reindex(index=_h.index, columns=_h.columns)
    if _h.size:
        fig, a = plt.subplots(figsize=(10, 4)); im = a.imshow(_h.values.astype(float), cmap="viridis", vmin=.5, vmax=.95, aspect="auto"); a.set_xticks(range(_h.shape[1])); a.set_xticklabels(_h.columns, rotation=45, ha="right"); a.set_yticks(range(_h.shape[0])); a.set_yticklabels(_h.index)
        for i in range(_h.shape[0]):
            for j in range(_h.shape[1]):
                if np.isfinite(_h.values[i, j]): a.text(j, i, f"{_h.values[i, j]:.2f}" + ("*" if _hs.values[i, j] == True else ""), ha="center", va="center", color="w", fontsize=8)
        fig.colorbar(im, label="learned break (quantile of the feature)"); a.set_title("15-minute bars, next-30-minute range: where the market's break lies, by regime (* = supported)"); fig.tight_layout(); fig.savefig(OUT2 / "18l_break_by_regime.png", dpi=110); plt.show()
    if len(KT_CV):
        for oc in ("range", "ret_series"):
            fig, axs = plt.subplots(1, 4, figsize=(17, 3.6), sharey=False)
            for a, f in zip(axs, ("rng", "uw", "brr", "vz")):
                for (rn, sp), g in KT_CV[(KT_CV.dim == "vol") & (KT_CV.feature == f) & (KT_CV.outcome == oc)].groupby(["regime", "split"]):
                    a.plot((g.bin + .5) / KT_NB, g["mean"], label=f"{rn}-vol {sp}", ls="-" if sp == "train_R" else "--", lw=1.2)
                a.set_title(KT_FEATS[f], fontsize=9); a.set_xlabel("quantile of the feature (own regime)"); a.grid(alpha=.3); a.axhline(0, color="k", lw=.6)
            axs[0].set_ylabel("excess next-30-minute " + ("range (ratio)" if oc == "range" else "return (z)")); axs[0].legend(fontsize=6); fig.suptitle(f"Response curves, 15-minute bars, {oc}: the market's own shape, solid = train_R, dashed = test"); fig.tight_layout(); fig.savefig(OUT2 / f"18l_response_{oc}.png", dpi=110); plt.show()
    KT_SUMMARY = dict(searches=NT_KT, distinct_searches=int(len(KT_U)), bonferroni_t=round(T_KT, 2), supported=int(KT_U.supported.sum()), supported_rows_with_mirrors=int(KT.supported.sum()), supported_by_outcome_window={f"{a}|{b}": int(v) for (a, b), v in KT_U[KT_U.supported].groupby(["outcome", "window"]).size().items()}, supported_by_bar=KT_U[KT_U.supported].groupby("bar").size().to_dict(),
                      nested=(dict(learned=round(float(_n.nested_t_learned.abs().mean()), 2), fixed75=round(float(_n.nested_t_fixed75.abs().mean()), 2), rule=(round(float(_n.nested_t_rule.abs().mean()), 2) if "nested_t_rule" in _n else None), q_range_median=round(float(_n.nested_q_range.median()), 2)) if "_n" in globals() and len(_n) else {}),
                      top=THR.head(15).round(3).to_dict("records"), regime_spread=KT_SPREAD.round(3).to_dict("records"),
                      learned_vs_rulebook=(dict(zip(cmp_.columns, cmp_.mean().round(2))) if len(rl_) else {}), rulebook_quantile=(rl_.groupby("feature").rule_quantile.median().round(2).to_dict() if len(rl_) else {}))
    KT_SUMMARY["reading"] = (f"Thresholds were searched, not assumed: for each regime the feature's break (50th-95th percentile tail, both sides) is chosen on train_R, checked on val_R and read once on the test; {KT_SUMMARY['supported']} distinct findings ({KT_SUMMARY['supported_rows_with_mirrors']} rows with mirrors) of {KT_SUMMARY['distinct_searches']:,} distinct searches pass (Bonferroni |t| > {KT_SUMMARY['bonferroni_t']}). "
                             + "Supported by outcome and window: " + str(KT_SUMMARY["supported_by_outcome_window"]) + ". By bar: " + str(KT_SUMMARY["supported_by_bar"]) + ". Learned versus rulebook mean |t| in-sample: " + str(KT_SUMMARY["learned_vs_rulebook"]) + "; with nested selection (honest): " + str(KT_SUMMARY["nested"]) + ".")
    print("\n" + KT_SUMMARY["reading"])

## Step 18m — Close location in the range: layered regimes, bounce versus edge, cost

**Why.** Step 18l found that the only directional effect inside 30 minutes is one effect seen through several saturated variables: a candle that closes at the extreme of its range is followed by a reversal. The rejection ratios `rh` and `rl` hit their cap (a close within the 2% smoothing of the extreme), so their "break" was a boundary solution. This step replaces them with a bounded variable and asks the questions the previous step could not.

**Variable.** Close location `CLV = (close − low) / (high − low)`, cut into **20 equal bins** of 0.05. The events are the top bin (closed at the high, expected reversal down) and the bottom bin (closed at the low, expected reversal up). The outcome is *reversal-aligned*: `y = −S × return`, with `S = +1` for the top bin and `−1` for the bottom bin, so `y > 0` means the market reversed. Bars: 5 and 15 minutes (the 12 focus products with cached 5-minute bars). The negated series is not run separately: close location is exactly mirrored by it (`CLV → 1 − CLV`).

**Entry and outcome.** The tradable outcome is the return from the **next bar's open** to the close 30 minutes after the event candle (6 bars at 5 minutes, 2 at 15 minutes). It is split into three parts: the gap from the event close to the next open (the part you cannot trade), the first bar after the open, and the later bars up to 30 minutes.

**Regime layers, slow to fast.**

| Layer | Adds | Cells |
|---|---|---|
| L1 | market trend (bear / range / bull, Step 13f) | 3 |
| L2 | the product's volatility regime (low / mid / high) | 9 |
| L3 | session phase: night session, first 30 minutes, morning, afternoon, last 30 minutes | 45 |
| L4 | sign of the overnight gap (|ONz| > 0.5 σ; day session only) | 135 |

**Layered test.** A child cell is *supported* only if it is positive in `train_R`, `val_R` and the test, **beats its parent** in all three periods (child mean above the mean of the rest of its parent), has a one-sided child-versus-parent *t* > 1.64 on the test and a test *t* above the Bonferroni level over all cells. **Thin cells** (fewer than 400 events in `train_R` + `val_R`) are never tested alone: their estimate is shrunk toward the parent mean with weight τ²/(τ²+SE²), where τ² is the between-cell variance (DerSimonian–Laird) among the siblings under the same parent. The table of out-of-sample error compares the cell alone, the shrunk estimate and the parent only: if shrinking helps on the test, finer layers carry more noise than information.

**Bounce versus edge.** (1) Where the 30-minute move happens: the gap to the next open, the first bar, the later bars. If most of it is in the gap, it is gone before you can trade. (2) Entry at the next open. (3) A round-trip cost grid (0.5 to 6 bp) and the **break-even round-trip cost**, which equals the mean gross gain. (4) The 20-bin response curve of the next-30-minute return on close location, per period and per session phase, so the shape, not only the extreme bins, is visible.

**Reading rules.** Events in consecutive candles overlap in time (the clustered *t* is by trading day); costs are applied once per event, so overlapping events cannot all be traded; the opening price must be stored in the 5-minute cache, otherwise the test says so and is not informative.

**Used later.** The supported cells, their break-even cost and the thin-cell shrinkage table decide whether a close-location reversal is carried into the intraday rules of the strategy library or dropped; the layered-test machinery (child must beat parent, shrink thin cells) is the template for any further regime-conditional study.

In [ ]:
# step 18m
KL_OK = bool(globals().get("RG_OK", False)) and all(k in globals() for k in ("kc_base", "kc_agg", "kc_prep5", "KC_REG", "KC_PRODS", "SPL_R", "RG_IDX", "RG", "bonferroni_t", "ONz", "RG_TN")) and globals().get("KC_RAW5") is not None
if not KL_OK:
    print("Step 18m needs Steps 13f (regimes), 18k (candle helpers) and the cached 5-minute bars: run them first; skipped")
else:
    # ---- 18m.1 close location in the range, in 20 bins, with regime layers from slow to fast ----
    # CLV = (close - low) / (high - low). A candle whose close is in the top bin (0.95-1) is "closed at the high", in the bottom bin "closed at the low".
    # The saturated rejection ratios of Step 18l are replaced by this bounded variable. Reversal-aligned outcome: y = -S * forward return, S = +1 (top bin) or -1 (bottom bin); y > 0 = reversal.
    KL_BARS = (5, 15); KL_NB = 20; KL_PH = ("night", "open30", "morning", "afternoon", "close30"); KL_GAP = ("gap down", "flat", "gap up"); KL_GAPZ = 0.5
    KL_COSTS = (0.5, 1.0, 2.0, 3.0, 4.0, 6.0)             # round-trip cost grid in bp (the notebook's own default is printed next to it)
    KL_SPLR = pd.Series(-1, index=RAW.index)
    for _klk, _kln in (("train", 0), ("val", 1), ("test", 2)): KL_SPLR.loc[RG_IDX[_klk]] = _kln
    def kl_phase(ts):
        m = (ts.hour * 60 + ts.minute).to_numpy(); ph = np.full(len(m), -1)
        ph[(m >= 1260) | (m <= 180)] = 0; ph[(m > 540) & (m <= 570)] = 1; ph[(m > 570) & (m <= 690)] = 2; ph[(m > 810) & (m <= 870)] = 3; ph[(m > 870) & (m <= 900)] = 4; return ph
    def kl_stats(y, spl, code, ncell, day, nday):
        n = np.zeros((3, ncell)); m = np.full((3, ncell), np.nan); t = np.full((3, ncell), np.nan)
        for s in range(3):
            sel = (spl == s) & (code >= 0) & np.isfinite(y); c = code[sel]; d = day[sel]; yy = y[sel].astype(float)
            nn = np.bincount(c, minlength=ncell).astype(float); mm = np.bincount(c, weights=yy, minlength=ncell) / np.maximum(nn, 1)
            sres = np.bincount(c * nday + d, weights=yy - mm[c], minlength=ncell * nday).reshape(ncell, nday); se = np.sqrt((sres ** 2).sum(1)) / np.maximum(nn, 1)
            n[s] = nn; m[s] = np.where(nn > 0, mm, np.nan); t[s] = np.where((nn >= 30) & (se > 0), mm / np.where(se > 0, se, 1), np.nan)
        return n, m, t
    def kl_diff(y, ch, day, nday):
        ok = np.isfinite(y); y, ch, day = y[ok], ch[ok], day[ok]; nc, nr = int(ch.sum()), int((~ch).sum())
        if nc < 30 or nr < 30: return np.nan, np.nan
        mc, mr = y[ch].mean(), y[~ch].mean(); w = np.where(ch, 1 / nc, -1 / nr); e = y - np.where(ch, mc, mr); se = np.sqrt((np.bincount(day, weights=w * e, minlength=nday) ** 2).sum())
        return float(mc - mr), float((mc - mr) / se) if se > 0 else np.nan
    # unit checks: bounded variable, bins, planted effect
    _kb = pd.DataFrame(dict(op=[1.0], hi=[2.0], lo=[0.5], cl=[2.0], volume=[1.0]), index=pd.to_datetime(["2020-01-02 09:05"])); _kbb = kc_base(_kb, 5)
    _clv = (_kbb["c"] - _kbb["l"]) / (_kbb["h"] - _kbb["l"]); assert abs(float(_clv.iloc[0]) - 1.0) < 1e-9 and int(np.clip(np.floor(1.0 * KL_NB), 0, KL_NB - 1)) == 19, "close location wrong"
    _r = np.random.default_rng(1); _n = 30000; _y = _r.normal(size=_n) + 0.3; _d = np.arange(_n) // 10; _s, _c = np.r_[np.zeros(_n // 2), np.ones(_n - _n // 2)].astype(int), (np.arange(_n) % 2)
    _nn, _mm, _tt = kl_stats(_y, _s, _c, 2, _d, int(_d.max()) + 1); assert abs(_mm[0, 0] - _y[(_s == 0) & (_c == 0)].mean()) < 1e-9 and _tt[0, 0] > 5, "cell statistics wrong"
    _dm, _dt = kl_diff(np.r_[_y[:2000], _y[2000:4000] + 0.5], np.r_[np.zeros(2000, bool), np.ones(2000, bool)], np.arange(4000) // 10, 400); assert 0.4 < _dm < 0.6 and _dt > 3, "difference test wrong"
    print("unit checks passed: close location, cell statistics, child-versus-parent difference")
    def kl_build(BAR):
        nb = 30 // BAR; cols = {k: [] for k in ("pid", "S", "bin", "spl", "vol", "trend", "phase", "gap", "tdi", "g", "m1", "mrest", "oe", "ce")}
        for k, p in enumerate(KC_PRODS):
            b = kc_base(kc_agg(kc_prep5(p), BAR), BAR)
            if len(b) < 2000: continue
            o, h, l, c = (b[x].to_numpy() for x in ("o", "h", "l", "c")); sc, sg, td, bad = b["sc"].to_numpy(), b["seg"].to_numpy(), b["tdi"].to_numpy(), b["bad"].to_numpy(); n = len(b); rg = h - l
            with np.errstate(all="ignore"): clv = (c - l) / rg
            ok = (rg > 1e-7) & (bad == 0) & np.isfinite(sc); bi = np.clip(np.floor(clv * KL_NB), 0, KL_NB - 1); ev = ok & ((bi == 0) | (bi == KL_NB - 1)); idx = np.where(ev & (np.arange(n) + nb < n))[0]
            if len(idx) == 0: continue
            same = (td[idx + nb] == td[idx]) & (sg[idx + nb] == sg[idx]); idx = idx[same]
            S = np.where(bi[idx] == KL_NB - 1, 1.0, -1.0); a = -S * 1e4
            dts = pd.Timestamp("2000-01-01") + pd.to_timedelta(td[idx], unit="D"); spl = KL_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); vo = KC_REG[p].reindex(dts).fillna(-1).to_numpy().astype(int); tr = RG["T"].reindex(dts).fillna(-1).to_numpy().astype(int)
            ph = kl_phase(b.index[idx]); oz = ONz[p].reindex(dts).to_numpy() if p in ONz else np.full(len(idx), np.nan); gp = np.where(~np.isfinite(oz), -1, np.where(oz > KL_GAPZ, 2, np.where(oz < -KL_GAPZ, 0, 1))); gp = np.where((ph >= 1), gp, -1)
            keep = (spl >= 0) & (vo >= 0) & (tr >= 0); i2 = idx[keep]
            cols["pid"].append(np.full(len(i2), k)); cols["S"].append(S[keep]); cols["bin"].append(bi[idx][keep]); cols["spl"].append(spl[keep]); cols["vol"].append(vo[keep]); cols["trend"].append(tr[keep]); cols["phase"].append(ph[keep]); cols["gap"].append(gp[keep]); cols["tdi"].append(td[i2])
            ak = a[keep]; cols["g"].append(ak * (o[i2 + 1] - c[i2])); cols["m1"].append(ak * (c[i2 + 1] - o[i2 + 1])); cols["mrest"].append(ak * (c[i2 + nb] - c[i2 + 1])); cols["oe"].append(ak * (c[i2 + nb] - o[i2 + 1])); cols["ce"].append(ak * (c[i2 + nb] - c[i2]))
        E = {k: np.concatenate(v) for k, v in cols.items() if len(v)}
        if E: E["dayi"] = np.unique(E["tdi"], return_inverse=True)[1].astype(np.int64)
        return E
    KL_E = {}
    for BAR in KL_BARS:
        KL_E[BAR] = kl_build(BAR)
        if KL_E[BAR]: print(f"{BAR}-minute bars: {len(KL_E[BAR]['S']):,} close-at-extreme candles in the 12 focus products (top or bottom of the 20 bins); open price stored: {bool(KC_HAS_OPEN)}")
    if not KC_HAS_OPEN: print("WARNING: the opening price is the previous close in this cache, so 'enter at the next open' cannot be separated from the close: the bounce test below is not informative")

In [ ]:
# step 18m
if KL_OK and all(KL_E.get(b) for b in KL_BARS):
    # ---- 18m.2 layered regime test: every child cell must beat its parent in all three periods; thin cells are shrunk toward the parent ----
    # Layers (slow to fast): L1 market trend -> L2 + product volatility -> L3 + session phase -> L4 + overnight-gap sign (day session only).
    KL_THIN = globals().get("KL_THIN", 400); KL_ROWS = []; KL_SHR = []
    for BAR in KL_BARS:
        E = KL_E[BAR]; nday = int(E["dayi"].max()) + 1; y = E["oe"]; c1 = E["trend"]; c2 = c1 * 3 + E["vol"]; c3 = np.where(E["phase"] >= 0, c2 * 5 + E["phase"], -1); c4 = np.where((c3 >= 0) & (E["gap"] >= 0), c3 * 3 + E["gap"], -1)
        spl = E["spl"]; allc = np.zeros(len(y), int)
        layers = [("L0 all", allc, 1, None), ("L1 trend", c1, 3, allc), ("L2 trend x volatility", c2, 9, c1), ("L3 + session phase", c3, 45, c2), ("L4 + overnight gap", c4, 135, c3)]
        par_of = {"L1 trend": lambda c: 0, "L2 trend x volatility": lambda c: c // 3, "L3 + session phase": lambda c: c // 5, "L4 + overnight gap": lambda c: c // 3}
        stat = {nm: kl_stats(y, spl, code, nc, E["dayi"], nday) for nm, code, nc, _ in layers}
        def label(nm, c):
            if nm == "L0 all": return "all"
            if nm == "L1 trend": return RG_TN[c]
            if nm == "L2 trend x volatility": return f"{RG_TN[c // 3]} / {('low', 'mid', 'high')[c % 3]}-vol"
            if nm == "L3 + session phase": return f"{label('L2 trend x volatility', c // 5)} / {KL_PH[c % 5]}"
            return f"{label('L3 + session phase', c // 3)} / {KL_GAP[c % 3]}"
        for li, (nm, code, nc, pcode) in enumerate(layers):
            if li == 0: continue
            pname = layers[li - 1][0]; pn, pm, pt = stat[pname]; cn, cm, ct = stat[nm]
            for c in range(nc):
                p = par_of[nm](c)
                if cn[:, c].sum() == 0: continue
                r = dict(bar=f"{BAR}m", layer=nm, cell=label(nm, c), n_train=int(cn[0, c]), n_val=int(cn[1, c]), n_test=int(cn[2, c]), m_train=cm[0, c], m_val=cm[1, c], m_test=cm[2, c], t_train=ct[0, c], t_val=ct[1, c], t_test=ct[2, c], parent=label(pname, p), parent_m_train=pm[0, p], parent_m_val=pm[1, p], parent_m_test=pm[2, p])
                dd = []
                for s in range(3):
                    sel = (spl == s) & np.isfinite(y) & ((pcode == p) & (code >= 0) if li > 1 else True); chd = (code == c)[sel]; dd.append(kl_diff(y[sel], chd, E["dayi"][sel], nday))
                r.update(diff_train=dd[0][0], diff_val=dd[1][0], diff_test=dd[2][0], tdiff_train=dd[0][1], tdiff_val=dd[1][1], tdiff_test=dd[2][1]); KL_ROWS.append(r)
        # shrinkage toward the parent on train_R + val_R pooled (DerSimonian-Laird between-cell variance), evaluated on the test
        for li, (nm, code, nc, pcode) in enumerate(layers):
            if li < 2: continue
            ncp, mcp, tcp = kl_stats(y, np.where(spl <= 1, 0, 2), code, nc, E["dayi"], nday)      # period 0 = train_R + val_R pooled, period 2 = test
            for p in np.unique(par_of[nm](np.arange(nc))):
                ch = [c for c in range(nc) if par_of[nm](c) == p and ncp[0, c] >= 30 and np.isfinite(mcp[0, c]) and np.isfinite(tcp[0, c]) and tcp[0, c] != 0]
                if len(ch) < 3: continue
                m_ = mcp[0, ch]; se = np.abs(m_ / tcp[0, ch]); w = 1 / se ** 2; mbar = (w * m_).sum() / w.sum(); Q = (w * (m_ - mbar) ** 2).sum(); tau2 = max(0.0, (Q - (len(ch) - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))
                Bw = tau2 / (tau2 + se ** 2); shr = mbar + Bw * (m_ - mbar)
                for j, c in enumerate(ch):
                    KL_SHR.append(dict(bar=f"{BAR}m", layer=nm, cell=label(nm, c), n_pool=int(ncp[0, c]), m_pool=m_[j], se=se[j], weight_on_cell=Bw[j], m_shrunk=shr[j], parent_mean=mbar, m_test=mcp[2, c], n_test=int(ncp[2, c]), thin=bool(ncp[0, c] < KL_THIN)))
    KL = pd.DataFrame(KL_ROWS); KS_ = pd.DataFrame(KL_SHR)
    KL["thin"] = (KL.n_train + KL.n_val) < KL_THIN; KL["fin"] = KL[["t_train", "t_val", "t_test"]].notna().all(axis=1) & ~KL.thin; NT_KL = int(KL.fin.sum()); T_KL = bonferroni_t(max(NT_KL, 1))
    KL["same_sign"] = KL.fin & (KL.m_train > 0) & (KL.m_val > 0) & (KL.m_test > 0)
    KL["beats_parent"] = KL.fin & (KL.diff_train > 0) & (KL.diff_val > 0) & (KL.diff_test > 0)
    KL["supported"] = KL.same_sign & KL.beats_parent & (KL.t_test > T_KL) & (KL.tdiff_test > 1.64)
    KL.round(4).to_csv(OUT2 / "layered_close_location.csv", index=False)
    print(f"Layered test of the reversal after a close at the extreme (outcome: bp from the next open to 30 minutes later, reversal positive); {NT_KL} cells with enough events, Bonferroni t > {T_KL:.2f} on test, child must beat its parent in train_R, val_R and test (one-sided t > 1.64 on test)")
    print("Cells with fewer than", KL_THIN, "events in train_R + val_R are shrunk toward their parent and never tested alone.")
    print("\nSupported cells by layer and bar:"); print(KL[KL.supported].groupby(["bar", "layer"]).size().unstack(fill_value=0).to_string() if KL.supported.any() else "none")
    print("\nStrongest supported cells:"); print(KL[KL.supported].sort_values("t_test", ascending=False).head(25)[["bar", "layer", "cell", "n_test", "m_train", "m_val", "m_test", "t_test", "parent", "diff_test", "tdiff_test"]].round(2).to_string(index=False) if KL.supported.any() else "none")
    print("\nParent layers: does a finer layer add anything? (L0 and L1 cells, mean bp and t)")
    print(KL[KL.layer.isin(["L1 trend", "L2 trend x volatility"])][["bar", "layer", "cell", "m_train", "m_val", "m_test", "t_test"]].round(2).to_string(index=False))
    if len(KS_):
        KS_.round(4).to_csv(OUT2 / "layered_shrinkage.csv", index=False)
        th_ = KS_[KS_.thin & (KS_.m_shrunk > 0) & (KS_.m_shrunk > KS_.parent_mean)].sort_values("m_shrunk", ascending=False).head(15)
        print(f"\nThin cells (< {KL_THIN} events) whose shrunk estimate is a reversal larger than their parent's: candidates to confirm when more data exist, not results:"); print(th_[["bar", "layer", "cell", "n_pool", "m_pool", "weight_on_cell", "m_shrunk", "parent_mean"]].round(2).to_string(index=False) if len(th_) else "none")
        ev_ = KS_[KS_.n_test >= 30].copy(); ev_["e_raw"] = (ev_.m_test - ev_.m_pool) ** 2; ev_["e_shr"] = (ev_.m_test - ev_.m_shrunk) ** 2; ev_["e_par"] = (ev_.m_test - ev_.parent_mean) ** 2
        KL_SHRSUM = ev_.groupby(["bar", "layer"]).apply(lambda g: pd.Series(dict(cells=len(g), mse_unshrunk=np.average(g.e_raw, weights=g.n_test), mse_shrunk=np.average(g.e_shr, weights=g.n_test), mse_parent_only=np.average(g.e_par, weights=g.n_test), mean_weight_on_cell=g.weight_on_cell.mean()))).reset_index()
        print("\nOut-of-sample error of each cell's estimate on the test (weighted by test events; lower is better): cell alone, shrunk toward its parent, parent only"); print(KL_SHRSUM.round(3).to_string(index=False))

In [ ]:
# step 18m
if KL_OK and all(KL_E.get(b) for b in KL_BARS):
    # ---- 18m.3 bounce or real edge? where the move happens, entry at the next open, and the cost it survives ----
    rows = []; nb_of = {5: 6, 15: 2}
    for BAR in KL_BARS:
        E = KL_E[BAR]; nday = int(E["dayi"].max()) + 1; groups = {"all events": np.zeros(len(E["S"]), int)}
        for nm, code, nc in (("session phase", np.where(E["phase"] >= 0, E["phase"], -1), 5), ("volatility", E["vol"], 3), ("trend", E["trend"], 3)): groups[nm] = (code, nc)
        for gname, spec in groups.items():
            code, nc = (spec if gname != "all events" else (spec, 1)); names = ("all",) if gname == "all events" else (KL_PH if gname == "session phase" else (("low", "mid", "high") if gname == "volatility" else RG_TN))
            res = {k: kl_stats(E[k], E["spl"], code, nc, E["dayi"], nday) for k in ("g", "m1", "mrest", "oe", "ce")}
            for c in range(nc):
                for s, sp in enumerate(("train_R", "val_R", "test")):
                    if res["oe"][0][s, c] < 30: continue
                    r = dict(bar=f"{BAR}m", group=gname, cell=names[c], split=sp, n=int(res["oe"][0][s, c]))
                    for k, lab in (("g", "gap_to_next_open_bp"), ("m1", "first_bar_from_open_bp"), ("mrest", "later_bars_bp"), ("oe", "from_next_open_30m_bp"), ("ce", "from_close_30m_bp")): r[lab] = res[k][1][s, c]; r[lab + "_t"] = res[k][2][s, c]
                    r["share_of_move_before_you_can_trade"] = r["gap_to_next_open_bp"] / r["from_close_30m_bp"] if abs(r["from_close_30m_bp"]) > 1e-9 else np.nan
                    for cst in KL_COSTS: r[f"net_{cst:g}bp"] = r["from_next_open_30m_bp"] - cst
                    r["break_even_round_trip_bp"] = r["from_next_open_30m_bp"]; rows.append(r)
    KL_B = pd.DataFrame(rows); KL_B.round(4).to_csv(OUT2 / "close_location_bounce_cost.csv", index=False)
    ov = KL_B[KL_B.group == "all events"]
    print("Where does the reversal after a close at the extreme happen? (reversal positive, bp; gap = close -> next open, first bar = next open -> its close, later = the rest of the 30 minutes)")
    print(ov[["bar", "split", "n", "gap_to_next_open_bp", "first_bar_from_open_bp", "later_bars_bp", "from_next_open_30m_bp", "from_next_open_30m_bp_t", "from_close_30m_bp", "share_of_move_before_you_can_trade"]].round(3).to_string(index=False))
    print(f"\nBreak-even round-trip cost (gross gain from entering at the next open and holding 30 minutes) against the cost grid {KL_COSTS} bp; the notebook's own default cost is {globals().get("DEFAULT_COST_BPS", float("nan")):g} bp per side:")
    print(KL_B[(KL_B.group != "trend")][["bar", "group", "cell", "split", "n", "break_even_round_trip_bp"] + [f"net_{c:g}bp" for c in KL_COSTS]].round(2).to_string(index=False))
    KL_BJ = KL_B[(KL_B.group == "all events") & (KL_B.split == "test")].set_index("bar")
    for BAR in KL_BARS:
        g = KL_B[(KL_B.group == "all events") & (KL_B.bar == f"{BAR}m") & (KL_B.split == "test")]
        if len(g): print(f"{BAR}m: gross {g.from_next_open_30m_bp.iloc[0]:.2f} bp from the next open on the test, {g.share_of_move_before_you_can_trade.iloc[0]:.0%} of the move from the close happens before the next open; break-even round-trip cost {g.break_even_round_trip_bp.iloc[0]:.2f} bp")
    # ---- 18m.4 the 20-bin response curve of the next-30-minute return on the close location, train_R / val_R / test, by session phase ----
    def kl_curve(BAR):
        nb = 30 // BAR; cc = {k: [] for k in ("bin", "spl", "phase", "tdi", "oe")}
        for p in KC_PRODS:
            b = kc_base(kc_agg(kc_prep5(p), BAR), BAR)
            if len(b) < 2000: continue
            o, h, l, c = (b[x].to_numpy() for x in ("o", "h", "l", "c")); sc, sg, td, bad = b["sc"].to_numpy(), b["seg"].to_numpy(), b["tdi"].to_numpy(), b["bad"].to_numpy(); n = len(b); rg = h - l
            with np.errstate(all="ignore"): clv = (c - l) / rg
            ok = (rg > 1e-7) & (bad == 0) & np.isfinite(sc) & (np.arange(n) + nb < n); idx = np.where(ok)[0]; idx = idx[(td[idx + nb] == td[idx]) & (sg[idx + nb] == sg[idx])]
            dts = pd.Timestamp("2000-01-01") + pd.to_timedelta(td[idx], unit="D"); spl = KL_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); keep = spl >= 0; i2 = idx[keep]
            cc["bin"].append(np.clip(np.floor(clv[i2] * KL_NB), 0, KL_NB - 1).astype(int)); cc["spl"].append(spl[keep]); cc["phase"].append(kl_phase(b.index[i2])); cc["tdi"].append(td[i2]); cc["oe"].append(((c[i2 + nb] - o[i2 + 1]) * 1e4).astype(np.float32))
        C = {k: np.concatenate(v) for k, v in cc.items()}; C["dayi"] = np.unique(C["tdi"], return_inverse=True)[1].astype(np.int64); return C
    KL_CV = []
    for BAR in KL_BARS:
        C = kl_curve(BAR); nday = int(C["dayi"].max()) + 1; mu = {s: np.nanmean(C["oe"][C["spl"] == s]) for s in range(3)}; y = C["oe"] - np.array([mu[s] for s in range(3)])[C["spl"]]       # excess over the period's mean (drift removed)
        for gname, code, nc in (("all", C["bin"], KL_NB), ("by phase", np.where(C["phase"] >= 0, C["phase"] * KL_NB + C["bin"], -1), 5 * KL_NB)):
            n_, m_, t_ = kl_stats(y, C["spl"], code, nc, C["dayi"], nday)
            for s, sp in enumerate(("train_R", "val_R", "test")):
                for c in range(nc): KL_CV.append(dict(bar=f"{BAR}m", group=gname, phase=("all" if gname == "all" else KL_PH[c // KL_NB]), bin=c % KL_NB, split=sp, n=int(n_[s, c]), mean_bp=m_[s, c], t=t_[s, c]))
    KL_CV = pd.DataFrame(KL_CV); KL_CV.round(4).to_csv(OUT2 / "close_location_curve.csv", index=False)
    fig, axs = plt.subplots(1, 2, figsize=(13, 4))
    for a, BAR in zip(axs, KL_BARS):
        for sp, col in (("train_R", "#1b6ca8"), ("val_R", "#2a9d8f"), ("test", "#c0392b")):
            g = KL_CV[(KL_CV.bar == f"{BAR}m") & (KL_CV.group == "all") & (KL_CV.split == sp)]; a.plot((g.bin + .5) / KL_NB, g.mean_bp, marker="o", ms=3, label=sp, color=col)
        a.axhline(0, color="k", lw=.7); a.set_title(f"{BAR}-minute bars: next 30 minutes from the next open, by close location"); a.set_xlabel("close location in the range (0 = low, 1 = high)"); a.set_ylabel("excess return, bp"); a.grid(alpha=.3); a.legend()
    fig.tight_layout(); fig.savefig(OUT2 / "18m_close_location_curve.png", dpi=110); plt.show()
    fig, axs = plt.subplots(1, 2, figsize=(13, 4))
    for a, BAR in zip(axs, KL_BARS):
        g = KL_B[(KL_B.bar == f"{BAR}m") & (KL_B.group == "session phase") & (KL_B.split == "test")]
        if len(g): x = np.arange(len(g)); a.bar(x - .27, g.gap_to_next_open_bp, .27, label="gap to next open"); a.bar(x, g.first_bar_from_open_bp, .27, label="first bar"); a.bar(x + .27, g.later_bars_bp, .27, label="later bars"); a.set_xticks(x); a.set_xticklabels(g.cell); a.axhline(0, color="k", lw=.7); a.legend(fontsize=7)
        a.set_title(f"{BAR}m, test: where the reversal happens (bp)"); a.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(OUT2 / "18m_bounce_decomposition.png", dpi=110); plt.show()
    KL_SUMMARY = dict(cells_tested=NT_KL, bonferroni_t=round(T_KL, 2), supported=int(KL.supported.sum()), supported_by_layer={f"{a}|{b}": int(v) for (a, b), v in KL[KL.supported].groupby(["bar", "layer"]).size().items()},
                      top=KL[KL.supported].sort_values("t_test", ascending=False).head(10)[["bar", "layer", "cell", "n_test", "m_train", "m_val", "m_test", "t_test"]].round(2).to_dict("records"),
                      overall_test=KL_B[(KL_B.group == "all events") & (KL_B.split == "test")][["bar", "n", "gap_to_next_open_bp", "first_bar_from_open_bp", "later_bars_bp", "from_next_open_30m_bp", "from_next_open_30m_bp_t", "share_of_move_before_you_can_trade", "break_even_round_trip_bp"]].round(3).to_dict("records"),
                      shrinkage=(KL_SHRSUM.round(3).to_dict("records") if "KL_SHRSUM" in globals() else []))
    _klo = KL_SUMMARY["overall_test"]
    KL_SUMMARY["reading"] = (f"Close at the extreme of a 5- or 15-minute candle, 20 equal bins of close location, regime layers trend > volatility > session phase > overnight-gap sign; {KL_SUMMARY['supported']} of {NT_KL} cells beat their parent in all three periods with t > {KL_SUMMARY['bonferroni_t']} on test. "
                             + "Overall on the test (bar: gross bp from the next open over 30 minutes, break-even round-trip cost, share of the move already gone at the next open): " + "; ".join(f"{r['bar']}: {r['from_next_open_30m_bp']:.2f} bp, {r['break_even_round_trip_bp']:.2f} bp, {r['share_of_move_before_you_can_trade']:.0%}" for r in _klo) + ".")
    print("\n" + KL_SUMMARY["reading"])

## Step 18n — Volume-weighted STARK / CAIRO / FLUX, tape features and open-interest-gated setups, at 5 and 15 minutes

**Why.** The audit of the five reference documents against this notebook found the same gap everywhere: the distribution-flow indicators (18g, 18h, 18i) exist only at daily level and without volume weights, nothing uses money, VWAP or open interest at the bar level, the candle setups of the documents (breakout, fakeout, pullback, spring, climax) are not OI-gated, and the lead-lag work is on returns, not on the indicators. This step closes those items. It is a *test*, not an adoption: every piece must hold in `train_R`, `val_R` and the chronological test, and every threshold is learned.

**Data fixes first (18n.1–18n.3).**

| Item in the notes | What the notebook does |
|---|---|
| CZCE counts money (or volume) twice before about 2010–2012 | The notes disagree with each other (money only, or money and volume; different symbol lists; "VWAP cancels it"). The step *measures* it: `money / (volume × close)` over the contract multiplier per product and year (a ratio near 2 = counted twice) and the year-on-year step of `volume / OI` (a step of log 2). The table is saved. No rule from the notes is applied; every money feature is divided by the same time-of-day median of the previous 20 days, so a 2× break distorts at most 20 days. |
| 9999 roll gaps | Returns are never taken across a contract change (the bar is masked), which is the same as using the raw contract for returns. OI change is masked on the roll bar (the OI of another contract), so the OI series is never spliced. |
| Session gaps | A return is never taken across a session break. Bar number within the trading day is stored. |
| VWAP, `price_vs_vwap` | `money / volume`, divided by the implied multiplier of the previous bars (cancels the contract multiplier and a persistent 2×); the gap to the close is in units of the bar's ex-ante range scale. |
| Net notional flow, volume pressure | `sign(close − open) × money` and `sign(close − open) × volume`, each relative to the same slot's previous-20-day median. |
| OI change, OI velocity, vol / OI | `ΔOI`, `ΔOI / volume` (clipped to ±1: the share of the traded lots that changed the open interest), `volume / OI` against its slot median; the four-state tape (long build, short cover, short build, long liquidation). |
| Parkinson diffusion | from high and low, 12-bar mean, against its own recent level. |

**Volume-weighted framework (18n.4).** Every return in a rolling window is weighted by its bar's relative volume (volume over the same slot's median, clipped to 0.05–10). Windows: 12 / 24 / 60 bars at 5 minutes (1 h, 2 h, 5 h), 12 / 24 / 40 bars at 15 minutes.

| Framework | Components built here |
|---|---|
| STARK | Wasserstein W1 between the fast and the slow volume-weighted distribution on a 7-point quantile grid, in units of the slow volatility, signed by the shift of the median and the 90th percentile; spike against its own recent mean; tail asymmetry from the 5 / 50 / 95 quantiles |
| CAIRO | volume-weighted mean, variance, skewness, excess kurtosis; derivatives of the mean and the skewness; a Fokker–Planck-style drift (change of the mean per unit change of the diffusion) |
| FLUX | 10 / 25 / 50 / 75 / 90 volume-weighted quantile velocities over 2 bars; alignment (all five layers the same way), layer cohesion; reconciliation of the median shift over 3 and 12 bars |

*Notes on the pasted formulas.* The product of three signs in the notes is +1 also when two layers fall, so it is not an alignment measure; here alignment means all layers have the same sign. A quantile of 12 points moves in steps, so one-bar velocities are mostly zero; velocities use a 2-bar change. The notes' fixed weights 0.3 / 0.3 / 0.4 and "volume above 2× its mean" are kept as comparisons only.

**Do the components carry any information (18n.6).** (a) Each component as a sign strategy over the next 30 minutes and the 30–60 minutes after, per period and volatility regime, with a Bonferroni bar. (b) A regime classifier: a four-component Gaussian mixture on daily volatility, 20-day trend, 5-day OI change and volume / OI, fitted on `train_R` days only, features shifted one day (known before the open); the clusters are named A (range), B (markup), C (distribution), D (markdown) from their own means. (c) A learned composite: ridge on the standardised signed components, pooled, by volatility regime, by hard mixture regime, and probability-weighted over the four mixture regimes; ridge strength is picked on `val_R`, the test is read once; the notes' fixed-weight composite is scored on the same bars.

**OI-gated setups (18n.7–18n.8).** Parents and what is added:

| Setup | Parent event | Gates added (each cut learned on train_R per volatility regime) | Thesis (outcome > 0) |
|---|---|---|---|
| A valid breakout | close above the prior L-bar high (L = 12, 36) | volume, OI up, close above VWAP, the 15 / 30 (30 / 60) minute candles agree | continues |
| B fakeout | the bar after a failed probe closes below the prior low | OI and volume of the probe bar, OI falling now, below VWAP | fade |
| C pullback, read 3 and 6 bars after a breakout | price back below the breakout close | **correction**: OI falls, volume fades, level holds; **reversal**: OI rises, volume stays high, level lost | correction resumes / reversal reverses |
| Wyckoff spring / climax | low breaks the prior L-bar low | volume, range, OI liquidation, close back inside | bounce (the negated series is the upthrust) |
| Doji after a run | top quartile of the trailing 6-bar move (train) | small body, volume, OI falls, small range | reverses |
| Fibonacci | pullback depth from the last confirmed 5-bar fractal high over the leg from the last fractal low (a pivot is known two bars after it) | none: response curve in 12 depth bins; the pasted levels 0.382 / 0.5 / 0.618 ± 0.05 against the rest of the 0.3–0.7 band | the swing resumes |

Every event is run on the series and on the negated series (a spring in the negated series is an upthrust). A gated variant is *supported* only if it is positive with *t* > 2 in `train_R` and `val_R`, has a day-clustered test *t* above the Bonferroni bar, **beats the rest of its parent** in validation and test, and has at least 100 events in each period. The outcome is the return of the next 30 minutes from the **close of the signal bar** (so it is not the 18m next-open entry: a gap to the next open is possible only at session breaks, but the first fill is still assumed at the close; the break-even cost reported next to it is therefore an upper bound). The 30–60-minute window is reported beside it.

**Lead-lag on the signals, weekly and yearly (18n.9).** Granger on the learned composite of each product: follower on its own two lags and on the leader's signal at 5, 10, 15, 30 minutes, day-clustered *t*, Bonferroni over all leader × follower × lag tests; transfer entropy (three equal-count states, a shuffled-leader null) on the survivors only. Daily features: gap fill ratio (share of the overnight gap retraced by the first 30 minutes) and log night / day volume. Weekly: skew of the daily returns, the week's close against its volume-weighted price, quarter-over-quarter OI growth, each as a cross-sectional sign strategy on the next week. Yearly: loadings on the equal-weight market factor and their year-to-year stability (a macro factor is not in the data).

**Reading rules.** Events in consecutive bars overlap in time: the *t* statistics cluster by trading day. The ridge and the mixture see only `train_R` days. The scale of every feature uses previous days only (checked by a truncation test). Everything above was exercised on synthetic data with planted effects before it was added; none of it has been run on the real data yet.

**Used later.** The supported rows, the break-even comparison and the composite table are carried into the findings report (Step 26b). Nothing here changes the strategy library until a variant is supported in all three periods and clears cost.

In [ ]:
# step 18n
_KN_SAVED = dict(globals())          # snapshot, so the loop variables of this step cannot overwrite a name that later steps read (it is restored at the end of the step)
KN_OK = bool(globals().get("RG_OK", False)) and all(k in globals() for k in ("kc_base", "kc_slot_med", "kc_world", "kc_wins", "kc_windows_for", "kc_prep5", "KC_REG", "KC_PRODS", "SPL_R", "RG_IDX", "RG", "bonferroni_t", "session_day", "MULT", "wide", "VOLU")) and globals().get("KC_RAW5") is not None
KN_HAS = KN_OK and {"money", "oi"} <= set(KC_RAW5.columns)
if not KN_OK:
    print("Step 18n needs Steps 13f (regimes), 18k (candle helpers) and the cached 5-minute bars: run them first; skipped")
else:
    if not KN_HAS: print("the cached 5-minute bars have no 'money' column (older Step 1): delete eda_cache/bars5m.parquet and re-run Step 1; the notional-flow and VWAP parts are skipped, OI parts run if 'oi' is there")
    # ---- 18n.0 settings. Nothing below assumes a threshold: every cut is learned on train_R, per volatility regime; the pasted numbers are reported next to the learned ones ----
    KN_BARS = (5, 15); KN_W = {5: (12, 24, 60), 15: (12, 24, 40)}          # fast / mid / slow windows in bars (5-minute: 1h / 2h / 5h; 15-minute: 3h / 6h / 10h)
    KN_U = np.array([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]); KN_QI = [1, 2, 3, 4, 5]          # quantile grid of the volume-weighted distribution; layers 10 / 25 / 50 / 75 / 90
    KN_TAILS = (0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.97, 0.99); KN_LBK = (12, 36); KN_DOC = dict(vz=float(np.log(2.0)), w=(0.3, 0.3, 0.4))      # pasted: volume > 2 x its mean; composite weights 0.3 / 0.3 / 0.4 (suggestions only)
    KN_SPLR = pd.Series(-1, index=RAW.index)
    for _k, _n in (("train", 0), ("val", 1), ("test", 2)): KN_SPLR.loc[RG_IDX[_k]] = _n
    KN_EDGE = ("mean_train", "mean_val", "mean_test")

    # ---- 18n.1 CZCE double count and the roll: measured from the data, product by product and year by year (daily panel, whole history) ----
    # implied multiplier = money / (volume x close); the ratio to the contract multiplier is 1 when money is clean and 2 when money is counted on both sides. Volume is checked through volume / OI (a doubling is a step of log 2).
    def kn_mult_check():
        rows = []; M, V, C_ = wide("money"), VOLU, wide("close")
        for p in V.columns:
            if p not in MULT: continue
            im = (M[p] / (V[p].where(V[p] > 0) * C_[p])) / MULT[p]; vo = None
            try: O_ = wide("oi")[p]; vo = np.log(V[p].where(V[p] > 0) / O_.where(O_ > 0))
            except Exception: vo = None
            for y, g in im.groupby(im.index.year):
                if g.notna().sum() < 100: continue
                r = dict(product=p, exchange=EXCH_NAME.get(EXCH.get(p, ""), EXCH.get(p, "")), year=int(y), money_ratio=float(g.median()), n_days=int(g.notna().sum()))
                if vo is not None: r["log_vol_oi"] = float(vo[vo.index.year == y].median())
                rows.append(r)
        return pd.DataFrame(rows)
    def kn_flag(df):
        # a product-year is flagged when the money ratio is close to 2 (or 0.5); a volume break is a year-on-year step of log 2 in volume / OI that is not seen on the other exchanges that year
        d = df.copy(); d["money_2x"] = d["money_ratio"].between(1.7, 2.35); d["money_half"] = d["money_ratio"].between(0.4, 0.58)
        if "log_vol_oi" in d:
            d["step"] = d.sort_values("year").groupby("product")["log_vol_oi"].diff(); med = d.groupby("year")["step"].transform("median"); d["vol_step_2x"] = (d["step"] - med).abs().sub(np.log(2)).abs() < 0.15
        else: d["vol_step_2x"] = False
        return d
    _kt = pd.DataFrame(dict(product=sum([[p] * 3 for p in "ABCDE"], []), year=[2008, 2009, 2010] * 5, money_ratio=[2.02, 1.98, 1.0] + [1.0, 1.01, 0.99] * 4, log_vol_oi=[0.0, 0.0, -np.log(2)] + [0.0] * 12)); _kt = kn_flag(_kt)
    assert _kt["money_2x"].tolist()[:6] == [True, True, False, False, False, False] and bool(_kt.loc[2, "vol_step_2x"]) and not _kt.loc[5, "vol_step_2x"], "CZCE detector wrong"
    KN_MC = kn_flag(kn_mult_check()); KN_MC.round(4).to_csv(OUT2 / "czce_money_ratio_by_year.csv", index=False)
    KN_BREAK = (KN_MC[KN_MC.money_2x].groupby(["exchange", "product"])["year"].agg(["min", "max", "count"]).reset_index()) if len(KN_MC) else pd.DataFrame()
    print(f"CZCE check on the daily panel ({KN_MC['product'].nunique()} products): money counted twice in {int(KN_MC.money_2x.sum())} product-years ({KN_BREAK['product'].nunique() if len(KN_BREAK) else 0} products), money halved in {int(KN_MC.money_half.sum())}, volume step of 2x in {int(KN_MC.vol_step_2x.sum())}")
    if len(KN_BREAK): print(KN_BREAK.to_string(index=False)); print("(exchanges: " + ", ".join(sorted(KN_BREAK.exchange.unique())) + ")")
    print("Every money-based feature below is scaled by the same time-of-day median of the previous 20 days, so a 2x break distorts at most 20 days and no rule from the notes is copied.")

    # ---- 18n.2 per-bar features from money, volume, OI and the candle. Point in time: every scale uses previous days only ----
    def kn_prep(p, KB):
        g = KC_RAW5[KC_RAW5["product"] == p].set_index("date").sort_index(); g = g[~g.index.duplicated()]
        d = pd.DataFrame(dict(op=g["op"] if "op" in g else g["cl"] * np.exp(-g["ret"].astype(float)), hi=g["hi"], lo=g["lo"], cl=g["cl"], volume=g["volume"])).astype(float)
        d["oi"] = g["oi"].astype(float) if "oi" in g else np.nan; d["money"] = g["money"].astype(float) if "money" in g else np.nan; d["rollb"] = g["rollb"].astype(float) if "rollb" in g else 0.0
        if KB != 5:
            r = d.resample(f"{KB}min", label="right", closed="right")
            d = pd.DataFrame(dict(op=r["op"].first(), hi=r["hi"].max(), lo=r["lo"].min(), cl=r["cl"].last(), volume=r["volume"].sum(min_count=1), oi=r["oi"].last(), money=r["money"].sum(min_count=1), rollb=r["rollb"].max())).dropna(subset=["cl"])
        return d
    def kn_sgn(x): return np.sign(x).astype(float)
    def kn_features(d, KB):
        # d: output of kn_prep. Returns base (kc_base columns) joined with the money / OI / VWAP / tape features
        b = kc_base(d[["op", "hi", "lo", "cl", "volume"]], KB); d = d.reindex(b.index); ts = b.index
        td = pd.DatetimeIndex(session_day(ts)); slot = (ts.hour * 60 + ts.minute).to_numpy(); win = max(60, 1200 * 5 // KB)
        o, c, h, l = (np.log(d[k].to_numpy(float)) for k in ("op", "cl", "hi", "lo")); vol = d["volume"].to_numpy(float); vol = np.where(vol > 0, vol, np.nan)
        out = pd.DataFrame(index=ts); seg = b["seg"].to_numpy(); newseg = np.r_[True, seg[1:] != seg[:-1]]
        rb = d["rollb"].to_numpy(float) > 0
        r = np.r_[np.nan, np.diff(c)]; r[newseg | rb] = np.nan; out["r"] = r                                  # clean return: never across a gap or a contract change
        sc = b["sc"].to_numpy(float); out["body"] = np.sign(c - o)
        # money / VWAP (only if the cache has money)
        if d["money"].notna().any():
            m = d["money"].to_numpy(float); m = np.where(m > 0, m, np.nan); mm = kc_slot_med(m, td, slot, 20, 8)
            imp = pd.Series(m / (vol * np.exp(c)), index=ts).rolling(win, min_periods=win // 4).median().shift(1).to_numpy()            # implied multiplier, previous bars only: cancels the contract multiplier and a persistent 2x
            vw = m / vol / imp; out["pvw"] = (c - np.log(vw)) / sc                                                                          # price versus VWAP in units of the ex-ante range scale
            out["nf"] = np.clip(out["body"].to_numpy() * m / mm, -10, 10)                                                                   # net notional flow = sign(close - open) x money, relative to the same slot's median money
        else: out["pvw"] = np.nan; out["nf"] = np.nan
        vm = b["vm"].to_numpy(float); out["vp"] = np.clip(out["body"].to_numpy() * vol / vm, -10, 10)                                       # volume pressure
        oi = d["oi"].to_numpy(float); oi = np.where(oi > 0, oi, np.nan); doi = np.r_[np.nan, np.diff(oi)]; doi[rb] = np.nan                                  # the roll bar is masked (its OI belongs to another contract): no spliced jump; the next bar compares two bars of the new contract
        out["oi"] = oi; out["doi"] = doi; oip = np.r_[np.nan, oi[:-1]]; out["dr"] = np.clip(doi / oip, -0.2, 0.2)
        voi = vol / oip; out["voiz"] = np.log(voi / kc_slot_med(voi, td, slot, 20, 8))                                                      # vol / OI churn, relative to the same slot
        out["ov"] = np.clip(doi / vol, -1, 1)                                                                                                # OI velocity: share of the traded lots that changed the open interest
        ad = kc_slot_med(np.abs(doi), td, slot, 20, 8); out["doiz"] = np.clip(doi / ad, -10, 10)
        q = np.zeros(len(b)); bd, sd_ = np.sign(c - o), np.sign(doi); q[(bd > 0) & (sd_ > 0)] = 1; q[(bd > 0) & (sd_ < 0)] = 2; q[(bd < 0) & (sd_ > 0)] = 3; q[(bd < 0) & (sd_ < 0)] = 4; out["q4"] = q   # 1 long build, 2 short cover, 3 short build, 4 long liquidation
        pk = (h - l) ** 2 / (4 * np.log(2)); out["pk"] = np.sqrt(pd.Series(pk, index=ts).rolling(12, min_periods=8).mean().to_numpy())             # Parkinson diffusion from high and low
        k = pd.Series(np.arange(len(b))).groupby(td.to_numpy()).cumcount().to_numpy() + 1; out["bar_idx"] = k                                  # bar number within the trading day
        return b.join(out)
    # point-in-time and mask checks on a tiny frame
    _ix = pd.to_datetime([f"2020-01-0{d} 09:{m:02d}" for d in (2, 3) for m in (5, 10, 15, 20)]); _kd = pd.DataFrame(dict(op=100.0, hi=101.0, lo=99.0, cl=[100, 100.5, 100.2, 100.4, 100, 100.3, 100.1, 100.2], volume=10.0, oi=[1000, 1010, 1005, 1020, 1030, 1040, 100, 110], money=10 * np.array([100, 100.5, 100.2, 100.4, 100, 100.3, 100.1, 100.2]) * 10.0, rollb=[0, 0, 0, 0, 0, 0, 1, 0]), index=_ix)
    _kf = kn_features(_kd, 5); assert np.isnan(_kf["doi"].iloc[6]) and abs(_kf["doi"].iloc[7] - 10) < 1e-9, "the roll bar must not carry the OI jump, the next bar is clean"; assert abs(_kf["doi"].iloc[1] - 10) < 1e-9 and _kf["q4"].iloc[1] == 1, "OI change or tape state wrong"
    print("unit checks passed: CZCE detector, OI masked at the roll, OI change and tape state")
    # ---- 18n.3 build the bar tables for every product ----
    def kn_build(KB):
        out = {}
        for p in KC_PRODS:
            d = kn_prep(p, KB)
            if len(d) < 3000: continue
            f = kn_features(d, KB); f["product"] = p; out[p] = f
        return out
    KN_T = {KB: kn_build(KB) for KB in KN_BARS}
    print({KB: {p: len(v) for p, v in t.items()} for KB, t in KN_T.items()})

In [ ]:
# ---- 18n.4 volume-weighted STARK / CAIRO / FLUX on the bars: the distribution of the last bars, each return weighted by its relative volume ----
if KN_OK and KN_T:
    from numpy.lib.stride_tricks import sliding_window_view as _swv
    KN_VLAG = 2                                                                            # quantile velocity = change over 2 bars (10 minutes at 5-minute bars): one-bar changes of a 12-point quantile are mostly zero
    def kn_vwq(r, w, W, U, chunk=40000):
        # volume-weighted quantiles of the last W returns ending at each bar (the bar itself included; bars with no clean return carry zero weight); NaN until 60% of the window is valid
        n = len(r); ok = np.isfinite(r) & np.isfinite(w) & (w > 0); rr = np.where(ok, r, 0.0).astype(np.float32); ww = np.where(ok, w, 0.0).astype(np.float32)
        rp = np.r_[np.zeros(W - 1, np.float32), rr]; wp = np.r_[np.zeros(W - 1, np.float32), ww]; R = _swv(rp, W); Wt = _swv(wp, W); out = np.full((n, len(U)), np.nan, np.float32)
        for s in range(0, n, chunk):
            a = np.array(R[s:s + chunk]); b = np.array(Wt[s:s + chunk]); o = np.argsort(a, axis=1); a = np.take_along_axis(a, o, 1); b = np.take_along_axis(b, o, 1)
            cw = np.cumsum(b, axis=1); tot = cw[:, -1]; nv = (b > 0).sum(1); ar = np.arange(len(a)); good = (nv >= 0.6 * W) & (tot > 0)
            for j, u in enumerate(U):
                idx = np.minimum((cw < (u * tot)[:, None] * (1 - 1e-6)).sum(1), W - 1); out[s:s + len(a), j] = np.where(good, a[ar, idx], np.nan)
        return out
    def kn_wmom(r, w, W):
        # volume-weighted rolling mean, variance, skewness, excess kurtosis over W bars (rolling sums of w, w r, w r^2, w r^3, w r^4)
        ok = np.isfinite(r) & np.isfinite(w) & (w > 0); rr = np.where(ok, r, 0.0); ww = np.where(ok, w, 0.0); mp = max(3, int(0.6 * W)); S = []
        for k in range(5): S.append(pd.Series(ww * rr ** k).rolling(W, min_periods=mp).sum().to_numpy())
        nv = pd.Series(ok.astype(float)).rolling(W, min_periods=mp).sum().to_numpy(); s0 = np.where(S[0] > 0, S[0], np.nan); m1 = S[1] / s0; m2 = S[2] / s0; m3 = S[3] / s0; m4 = S[4] / s0
        var = np.maximum(m2 - m1 ** 2, 0); mu3 = m3 - 3 * m1 * m2 + 2 * m1 ** 3; mu4 = m4 - 4 * m1 * m3 + 6 * m1 ** 2 * m2 - 3 * m1 ** 4
        with np.errstate(all="ignore"): sk = mu3 / var ** 1.5; ku = mu4 / var ** 2 - 3
        bad = (nv < mp) | (var <= 0); return [np.where(bad, np.nan, x) for x in (m1, var, sk, ku)]
    def kn_frame(f, KB):
        # f: kn_features table. Returns the STARK / CAIRO / FLUX / tape components, all point in time (a bar uses itself and earlier bars only)
        Wf, Wm, Ws = KN_W[KB]; r = f["r"].to_numpy(float)
        vol_rel = np.clip(f["vol"].to_numpy(float) / f["vm"].to_numpy(float), 0.05, 10.0)                       # relative volume = this bar's volume over its same-slot median
        Qf = kn_vwq(r, vol_rel, Wf, KN_U); Qs = kn_vwq(r, vol_rel, Ws, KN_U)
        mu_f, var_f, sk_f, ku_f = kn_wmom(r, vol_rel, Wf); mu_m, var_m, sk_m, ku_m = kn_wmom(r, vol_rel, Wm); mu_s, var_s, sk_s, ku_s = kn_wmom(r, vol_rel, Ws); sd_s = np.sqrt(var_s)
        out = pd.DataFrame(index=f.index)
        # STARK: transport between the fast and the slow volume-weighted distribution (W1 on the quantile grid), signed by the direction of the shift of the middle of the distribution
        w1 = np.nanmean(np.abs(Qf - Qs), axis=1); shift = np.nanmean(np.c_[Qf[:, 3] - Qs[:, 3], Qf[:, 5] - Qs[:, 5]], axis=1)
        w1n = w1 / sd_s; out["stark"] = w1n * np.sign(shift); out["w1n"] = w1n
        z = pd.Series(w1n); mz = z.rolling(120, min_periods=60).mean().shift(1); sz = z.rolling(120, min_periods=60).std().shift(1); out["w1_spike"] = ((z - mz) / sz).to_numpy()   # distribution shift against its own recent history (point in time)
        out["tasym"] = np.log((Qf[:, 6] - Qf[:, 3] + 0.02 * (Qf[:, 6] - Qf[:, 0]) + 1e-12) / (Qf[:, 3] - Qf[:, 0] + 0.02 * (Qf[:, 6] - Qf[:, 0]) + 1e-12))                       # tail asymmetry from the 5 / 50 / 95 quantiles
        # CAIRO: moment dynamics. Mean change in units of the slow volatility; skewness derivative on the 2-hour window (one-bar and three-bar differences)
        out["dmu"] = (mu_f - pd.Series(mu_f).shift(1).to_numpy()) / sd_s; out["dsk1"] = np.clip(sk_m - pd.Series(sk_m).shift(1).to_numpy(), -3, 3); out["dsk3"] = np.clip((sk_m - pd.Series(sk_m).shift(3).to_numpy()) / 3, -3, 3)
        out["dku"] = np.clip(ku_m - pd.Series(ku_m).shift(1).to_numpy(), -6, 6)
        dvar = np.abs(var_f - pd.Series(var_f).shift(1).to_numpy()); out["fp"] = np.clip(out["dmu"].to_numpy() * sd_s / (np.sqrt(dvar) + 0.1 * sd_s), -5, 5)       # Fokker-Planck style drift: change of the mean per unit change of the diffusion
        out["pkd"] = np.log(f["pk"].to_numpy(float) / pd.Series(f["pk"].to_numpy(float)).rolling(Ws, min_periods=Wf).median().shift(1).to_numpy())                                  # Parkinson diffusion against its recent level
        # FLUX: velocity of the 10 / 25 / 50 / 75 / 90 layers, alignment (all five same sign) and multi-time-frame reconciliation (shift of the median over 3 bars vs 12 bars)
        V = Qf[:, KN_QI] - pd.DataFrame(Qf[:, KN_QI]).shift(KN_VLAG).to_numpy(); sg = np.sign(V); out["flux"] = (np.all(sg > 0, axis=1).astype(float) - np.all(sg < 0, axis=1)); out["flux_c"] = np.nanmean(sg, axis=1)
        d3 = Qf[:, 3] - pd.Series(Qf[:, 3]).shift(3).to_numpy(); d12 = Qf[:, 3] - pd.Series(Qf[:, 3]).shift(12).to_numpy(); out["rec"] = np.sign(d3) * np.sign(d12); out["rec_dir"] = np.where(out["rec"] > 0, np.sign(d12), 0.0)
        # tape components, smoothed over 6 bars (30 minutes at 5-minute bars)
        for nm in ("nf", "vp"): out[nm + "6"] = f[nm].rolling(6, min_periods=3).mean().to_numpy()
        out["ovs"] = (f["body"] * f["ov"]).rolling(6, min_periods=3).mean().to_numpy(); out["pvw"] = f["pvw"].to_numpy(); out["dr6"] = f["dr"].rolling(6, min_periods=3).sum().to_numpy()
        out["doiz6"] = f["doiz"].rolling(6, min_periods=3).mean().to_numpy()
        return out
    # unit checks: the weighted quantile and the weighted moments against brute force
    _r0 = np.random.default_rng(3); _x = _r0.normal(size=400).astype(np.float32); _kw = _r0.uniform(0.2, 3, 400)
    _q = kn_vwq(_x, _kw, 30, KN_U); _i = 200; _o = np.argsort(_x[_i - 29:_i + 1]); _cw = np.cumsum(_kw[_i - 29:_i + 1][_o]); _bf = [_x[_i - 29:_i + 1][_o][np.searchsorted(_cw, u * _cw[-1] * (1 - 1e-6))] for u in KN_U]
    assert np.allclose(_q[_i], _bf, atol=1e-6), "weighted quantile wrong"
    _m = kn_wmom(_x.astype(float), _kw, 30); _xx = _x[_i - 29:_i + 1].astype(float); _ww = _kw[_i - 29:_i + 1]; _mu = np.average(_xx, weights=_ww); _v = np.average((_xx - _mu) ** 2, weights=_ww)
    assert abs(_m[0][_i] - _mu) < 1e-9 and abs(_m[1][_i] - _v) < 1e-9 and abs(_m[2][_i] - np.average((_xx - _mu) ** 3, weights=_ww) / _v ** 1.5) < 1e-6 and abs(_m[3][_i] - (np.average((_xx - _mu) ** 4, weights=_ww) / _v ** 2 - 3)) < 1e-6, "weighted moments wrong"
    _xs = _x.copy(); _xs[250:] = _r0.normal(size=150) * 3; assert np.allclose(kn_vwq(_xs[:251], _kw[:251], 30, KN_U)[:251], kn_vwq(_xs, _kw, 30, KN_U)[:251], equal_nan=True), "the weighted quantile must not look ahead"
    print("unit checks passed: volume-weighted quantiles and moments against brute force, no look-ahead")
    # causality: every component at a bar must be identical when all later bars are removed (the framework, the tape features and the scales)
    _kg = kn_prep(list(KN_T[5])[0], 5).iloc[-6000:]
    def _kn_pit(g):
        f = kn_features(g, 5); return pd.concat([f[["sc", "vm", "pvw", "nf", "vp", "doi", "dr", "voiz", "ov", "doiz", "q4", "pk"]], kn_frame(f, 5)], axis=1)
    causal_test("kn_framework_5m", _kn_pit, _kg, step="18n")
    # ---- 18n.5 assemble the panel: components + forward outcomes (labels use the future on purpose), split, regimes ----
    KN_WN = ("0-30m", "30-60m")
    def kn_panel(KB):
        wins = {k: v for k, v in kc_windows_for(KB).items() if k in KN_WN}; parts = []
        for k, (p, f) in enumerate(KN_T[KB].items()):
            fr = kn_frame(f, KB); W = kc_wins(f, wins, 30 // KB, False); n = len(f)
            dts = pd.Timestamp("2000-01-01") + pd.to_timedelta(f["tdi"].to_numpy(), unit="D"); sp = KN_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); vr = KC_REG[p].reindex(dts).fillna(-1).to_numpy().astype(int); tr = RG["T"].reindex(dts).fillna(-1).to_numpy().astype(int)
            keep = (sp >= 0) & (vr >= 0) & (tr >= 0) & f["sc"].notna().to_numpy()
            keep &= np.isfinite(W[KN_WN[0]][0]); unit = (f["sc"].to_numpy(float) * np.sqrt(30 // KB) * 1e4)          # bp per unit of the ex-ante scale for a window of 30 / KB bars
            d = fr.iloc[np.where(keep)[0]].copy(); d["y"] = W[KN_WN[0]][0][keep]; d["y2"] = W[KN_WN[1]][0][keep]; d["unit"] = unit[keep]; d["spl"] = sp[keep]; d["vol"] = vr[keep]; d["trend"] = tr[keep]; d["tdi"] = f["tdi"].to_numpy()[keep]; d["pid"] = k
            d["slot"] = (f.index.hour * 60 + f.index.minute).to_numpy()[keep]; d["bar_idx"] = f["bar_idx"].to_numpy()[keep]; d["td"] = pd.DatetimeIndex(dts)[keep]; parts.append(d)
        P = pd.concat(parts, ignore_index=True); P["dayi"] = np.unique(P["tdi"], return_inverse=True)[1].astype(np.int64); return P
    KN_P = {KB: kn_panel(KB) for KB in KN_BARS}
    print({KB: len(v) for KB, v in KN_P.items()}, "bars with a complete component set and a forward outcome")
    # shared helpers: day-clustered group statistics and the child-versus-parent difference
    KN_SPN = ("train_R", "val_R", "test"); KN_VN = ("low", "mid", "high")
    def kn_group(y, day, g, nday):
        g = g & np.isfinite(y); n = int(g.sum())
        if n < 50: return n, np.nan, np.nan
        m = float(y[g].mean()); sres = np.bincount(day[g], weights=y[g] - m, minlength=nday); se = np.sqrt((sres ** 2).sum()) / n
        return n, m, (m / se if se > 0 else np.nan)
    def kn_diff(y, ch, day, nday):
        ok = np.isfinite(y); y, ch, day = y[ok], ch[ok], day[ok]; nc, nr = int(ch.sum()), int((~ch).sum())
        if nc < 30 or nr < 30: return np.nan, np.nan
        mc, mr = y[ch].mean(), y[~ch].mean(); w = np.where(ch, 1 / nc, -1 / nr); e = y - np.where(ch, mc, mr); se = np.sqrt((np.bincount(day, weights=w * e, minlength=nday) ** 2).sum())
        return float(mc - mr), float((mc - mr) / se) if se > 0 else np.nan
    _r = np.random.default_rng(2); _y = _r.normal(size=20000) + 0.2; _d = np.arange(20000) // 10; _n, _m, _t = kn_group(_y, _d, np.ones(20000, bool), 2000); assert abs(_m - _y.mean()) < 1e-9 and _t > 5, "group statistic wrong"
    _dm, _dt = kn_diff(np.r_[_y[:2000], _y[2000:4000] + 0.5], np.r_[np.zeros(2000, bool), np.ones(2000, bool)], np.arange(4000) // 10, 400); assert 0.35 < _dm < 0.65 and _dt > 3, "difference wrong"

In [ ]:
# ---- 18n.6 do the framework components carry information at all? Component by component, then a learned composite, regime classifier and probability-weighted ensemble ----
if KN_OK and KN_T:
    KN_COMPS = {"stark": "STARK: W1 between the fast and slow volume-weighted distribution x direction of the shift", "dmu": "CAIRO: change of the weighted mean / slow volatility", "dsk1": "CAIRO: change of the weighted skewness (1 bar)", "dsk3": "CAIRO: change of the weighted skewness (3 bars)",
                "fp": "CAIRO: Fokker-Planck style drift (change of mean per unit change of diffusion)", "flux": "FLUX: all five quantile layers moving the same way", "flux_c": "FLUX: share of layers moving up minus down", "rec_dir": "FLUX: median shift over 3 and 12 bars agree (signed)",
                "tasym": "STARK: tail asymmetry from the 5 / 50 / 95 quantiles", "nf6": "net notional flow, sign(close-open) x money, 6-bar mean", "vp6": "volume pressure, sign(close-open) x volume, 6-bar mean", "ovs": "OI velocity: dOI / volume x body direction, 6-bar mean",
                "pvw": "close versus VWAP (in range scale)", "dr6": "relative OI change over 6 bars", "doiz6": "OI change vs same-slot median, 6-bar mean"}
    def kn_pnl_stat(score, P, mask, col="y"):
        # strategy: long when the score is positive, short when negative (no threshold); outcome in bp; day-clustered t. Also the pooled rank correlation of score and outcome.
        y = np.sign(score) * P[col].to_numpy(float) * P["unit"].to_numpy(float); nd = int(P["dayi"].max()) + 1
        n, m, t = kn_group(y, P["dayi"].to_numpy(), mask & np.isfinite(y) & (score != 0), nd)
        ok = mask & np.isfinite(score) & np.isfinite(P[col].to_numpy(float))
        ic = float(stats.spearmanr(score[ok], P[col].to_numpy(float)[ok])[0]) if ok.sum() > 100 else np.nan
        return n, m, t, ic
    # ---- (a) component by component: sign strategy per period and volatility regime, 0-30 and 30-60 minutes ----
    KN_CT = []
    for KB, P in KN_P.items():
        for cn in KN_COMPS:
            if cn == "fp": continue                       # fp = dmu x a positive factor: the same sign strategy as dmu (it differs only as a continuous input to the composite)
            x = P[cn].to_numpy(float) if cn in P else None
            if x is None or not np.isfinite(x).any(): continue
            for vname, vm_ in [("all", np.ones(len(P), bool))] + [(KN_VN[v], P["vol"].to_numpy() == v) for v in range(3)]:
                for col, win in (("y", "0-30m"), ("y2", "30-60m")):
                    row = dict(bar=f"{KB}m", component=cn, regime=vname, window=win)
                    for s, sn in enumerate(KN_SPN):
                        n, m, t, ic = kn_pnl_stat(np.sign(np.nan_to_num(x)), P, vm_ & (P["spl"].to_numpy() == s), col); row.update({f"n_{sn}": n, f"bp_{sn}": m, f"t_{sn}": t, f"ic_{sn}": ic})
                    KN_CT.append(row)
    KN_CT = pd.DataFrame(KN_CT); NT_CT = int((KN_CT.regime == "all").sum() * 4 / 2)           # tests: component x bar x (all + 3 regimes), on the 0-30 window
    T_CT = bonferroni_t(max(NT_CT, 1))
    KN_CT["supported"] = ((np.sign(KN_CT.t_train_R) == np.sign(KN_CT.t_val_R)) & (np.sign(KN_CT.t_val_R) == np.sign(KN_CT.t_test)) & (KN_CT.t_test.abs() > T_CT) & (KN_CT.t_val_R.abs() > 2) & (KN_CT.t_train_R.abs() > 2))
    KN_CT.round(4).to_csv(OUT2 / "framework_components.csv", index=False)
    print(f"Framework components, sign strategy over the next 30 minutes ({NT_CT} tests, Bonferroni t > {T_CT:.2f} on the test, t > 2 and the same sign in train_R and val_R):")
    _v = KN_CT[(KN_CT.regime == "all") & (KN_CT.window == "0-30m")].pivot_table(index="component", columns="bar", values=["t_train_R", "t_val_R", "t_test"]).round(2); print(_v.reindex(list(KN_COMPS)).to_string())
    print("supported:", KN_CT[KN_CT.supported & (KN_CT.window == "0-30m")][["bar", "component", "regime", "t_train_R", "t_val_R", "t_test", "bp_test"]].round(2).to_dict("records") or "none")
    # ---- (b) regime classifier: a Gaussian mixture on four daily features, fitted on train_R days only, features known before the open (shifted one day) ----
    from sklearn.mixture import GaussianMixture
    KN_K = 4; KN_RN = ("A accumulation / range", "KB markup", "C distribution", "D markdown")
    def kn_daily_feat(p):
        r = RAW[p]; v = r.abs().rolling(20, min_periods=15).mean(); tr = r.rolling(20, min_periods=15).sum() / (v * np.sqrt(20))
        try: oi = wide("oi")[p].where(lambda x: x > 0); oich = np.log(oi / oi.shift(5)); voi = np.log((VOLU[p].where(VOLU[p] > 0) / oi).rolling(5, min_periods=3).mean())
        except Exception: oich = pd.Series(0.0, index=r.index); voi = pd.Series(0.0, index=r.index)
        return pd.DataFrame(dict(vol=np.log(v), trend=tr, oi5=oich, voi=voi)).shift(1)
    def kn_label(means):
        # name the four clusters from their means (no assumption about which is which beyond the sign of the trend): KB = highest trend, D = lowest trend, A = lower volatility of the other two, C = the rest
        order = np.argsort(means[:, 1]); D_, B_ = int(order[0]), int(order[-1]); rest = [k for k in range(KN_K) if k not in (D_, B_)]; A_ = min(rest, key=lambda k: means[k, 0]); C_ = [k for k in rest if k != A_][0]
        return {A_: 0, B_: 1, C_: 2, D_: 3}
    KN_GM, KN_GP = {}, {}; _gm_rows = []
    for p in KC_PRODS:
        F = kn_daily_feat(p); tr_ix = F.index.intersection(RG_IDX["train"]); Xt = F.loc[tr_ix].dropna()
        if len(Xt) < 200: continue
        mu, sd = Xt.mean(), Xt.std().replace(0, 1); g = GaussianMixture(KN_K, covariance_type="full", n_init=3, random_state=0).fit(((Xt - mu) / sd).to_numpy()); lab = kn_label(g.means_)
        ok = F.dropna().index; pr = g.predict_proba(((F.loc[ok] - mu) / sd).to_numpy()); Pm = np.zeros((len(ok), KN_K));
        for k, a in lab.items(): Pm[:, a] = pr[:, k]
        KN_GP[p] = pd.DataFrame(Pm, index=ok, columns=range(KN_K)); KN_GM[p] = g
        for k, a in lab.items(): _gm_rows.append(dict(product=p, regime=KN_RN[a], weight=g.weights_[k], **{c: g.means_[k][j] * sd.iloc[j] + mu.iloc[j] for j, c in enumerate(F.columns)}))
    KN_GMT = pd.DataFrame(_gm_rows); KN_GMT.round(4).to_csv(OUT2 / "regime_mixture_clusters.csv", index=False)
    if len(KN_GMT): print("Regime mixture, mean daily features of each cluster (averaged over products):"); print(KN_GMT.groupby("regime")[["weight", "vol", "trend", "oi5", "voi"]].mean().round(3).to_string())
    def kn_probs(P, KB):
        names = list(KN_T[KB]); out = np.full((len(P), KN_K), np.nan)
        for k, p in enumerate(names):
            if p not in KN_GP: continue
            m = (P["pid"].to_numpy() == k); out[m] = KN_GP[p].reindex(P["td"][m]).to_numpy()
        return out
    # ---- (c) learned composite: ridge on the standardised signed components, pooled, by volatility regime, by hard mixture regime, probability-weighted ----
    KN_USE = [c for c in ("stark", "dmu", "dsk1", "fp", "flux", "flux_c", "rec_dir", "nf6", "vp6", "ovs", "pvw", "dr6") if c in KN_COMPS]
    def kn_ridge(X, y, w, lam):
        A = (X * w[:, None]).T @ X + lam * np.eye(X.shape[1]); return np.linalg.solve(A, (X * w[:, None]).T @ y)
    KN_CMP = []
    for KB, P in KN_P.items():
        Xr = P[KN_USE].to_numpy(float); trm = P["spl"].to_numpy() == 0; mu = np.nanmean(Xr[trm], 0); sd = np.nanstd(Xr[trm], 0); sd[sd == 0] = 1; X = np.clip(np.nan_to_num((Xr - mu) / sd), -5, 5); y = np.nan_to_num(P["y"].to_numpy(float)); yok = np.isfinite(P["y"].to_numpy(float))
        spl = P["spl"].to_numpy(); vol = P["vol"].to_numpy(); PR = kn_probs(P, KB); hasP = np.isfinite(PR).all(1); hard = np.where(hasP, np.nan_to_num(PR).argmax(1), -1); PRn = np.nan_to_num(PR)
        sc = {}; lam_best = {}
        def fitscore(group, probs=None):
            best = None
            for lam in (1e2, 1e3, 1e4, 1e5):
                s = np.zeros(len(P)); ok_all = True
                for k in range(group[1]):
                    if probs is None: m_tr = trm & yok & (group[0] == k); w_ = np.ones(len(P))
                    else: m_tr = trm & yok & hasP; w_ = probs[:, k]
                    if m_tr.sum() < 500: ok_all = False; break
                    b = kn_ridge(X[m_tr], y[m_tr], w_[m_tr], lam); sk = X @ b
                    s += sk * (np.ones(len(P)) if probs is None else probs[:, k]) * (1.0 if probs is not None else (group[0] == k))
                if not ok_all: continue
                vs = (spl == 1) & yok & (s != 0); ic = np.corrcoef(s[vs], y[vs])[0, 1] if vs.sum() > 200 else -9
                if best is None or ic > best[0]: best = (ic, lam, s)
            return best
        one = (np.zeros(len(P), int), 1); r1 = fitscore(one); rv = fitscore((vol, 3)); rh = fitscore((hard, KN_K)) if hasP.any() else None; rp = fitscore((hard, KN_K), PRn) if hasP.any() else None
        for nm, r in (("pooled ridge", r1), ("by volatility regime", rv), ("by mixture regime (hard)", rh), ("mixture probability-weighted", rp)):
            if r is not None: sc[nm] = r[2]; lam_best[nm] = r[1]
        pa = np.nan_to_num(P["w1n"].to_numpy(float)) if "w1n" in P else 0
        doc = 0.3 * np.sign(np.nan_to_num(P["dsk1"].to_numpy(float))) * (P["w1n"].to_numpy(float) / pd.Series(P["w1n"].to_numpy(float)).rolling(200, min_periods=50).max().to_numpy()) + 0.3 * np.sign(np.nan_to_num(P["vp6"].to_numpy(float))) * np.nan_to_num(P["dr6"].to_numpy(float)) + 0.4 * np.nan_to_num(P["flux"].to_numpy(float))
        sc["pasted weights 0.3 / 0.3 / 0.4"] = np.nan_to_num(doc)
        for nm, s in sc.items():
            row = dict(bar=f"{KB}m", model=nm, ridge_lambda=lam_best.get(nm, np.nan))
            for si, sn in enumerate(KN_SPN):
                for col, wn in (("y", "0-30m"), ("y2", "30-60m")):
                    n_, m_, t_, ic_ = kn_pnl_stat(s, P, spl == si, col); row.update({f"bp_{sn}_{wn}": m_, f"t_{sn}_{wn}": t_, f"ic_{sn}_{wn}": ic_});
                    if wn == "0-30m": row[f"n_{sn}"] = n_
            KN_CMP.append(row)
        if "mixture probability-weighted" in sc: P["score_prob"] = sc["mixture probability-weighted"]
        if "pooled ridge" in sc: P["score_pool"] = sc["pooled ridge"]
    KN_CMP = pd.DataFrame(KN_CMP); KN_CMP.round(4).to_csv(OUT2 / "framework_composite.csv", index=False)
    print("\nComposite signal over the next 30 minutes (sign strategy, bp per trade, day-clustered t; ridge fitted on train_R, ridge strength picked on val_R, test read once):")
    print(KN_CMP[["bar", "model", "bp_train_R_0-30m", "t_train_R_0-30m", "bp_val_R_0-30m", "t_val_R_0-30m", "bp_test_0-30m", "t_test_0-30m", "ic_test_0-30m", "t_test_30-60m"]].round(3).to_string(index=False))

In [ ]:
# ---- 18n.7 candle setups gated by open interest, volume and VWAP: breakout, fakeout, pullback (correction or reversal), spring / climax, doji after a run, Fibonacci retracement of the last fractal swing ----
if KN_OK and KN_T:
    KN_PH = ("night", "open30", "morning", "afternoon", "close30")
    def kn_phase(slot):
        m = np.asarray(slot); ph = np.full(len(m), 2); ph[(m >= 1260) | (m <= 180)] = 0; ph[(m > 540) & (m <= 570)] = 1; ph[(m > 810) & (m <= 870)] = 3; ph[(m > 870) & (m <= 900)] = 4; return ph
    def kn_pivots(hw, lw, tdi, cw, maxage=24):
        # 5-bar fractal swings, causal: a pivot at bar j needs bars j+1 and j+2, so it is known at bar j+2 and not before. State at bar i: last confirmed swing high LH after a swing low LL, same day.
        n = len(hw); H = pd.Series(hw); Lo = pd.Series(lw); ph = (H == H.rolling(5, center=True).max()).to_numpy(); pl = (Lo == Lo.rolling(5, center=True).min()).to_numpy()   # causal-ok: the centred window only finds the pivot; the pivot is used from bar j+2 on (confirmation lag), verified by the truncation unit check below
        ch = np.full(n, -1); cl = np.full(n, -1)
        for j in np.where(ph)[0]:
            if j + 2 < n: ch[j + 2] = j
        for j in np.where(pl)[0]:
            if j + 2 < n: cl[j + 2] = j
        lh = np.maximum.accumulate(ch); ll = np.maximum.accumulate(cl); ix = np.arange(n)
        ok = (lh >= 0) & (ll >= 0) & (lh > ll) & (ix - lh <= maxage) & (tdi[np.maximum(ll, 0)] == tdi) & (tdi[np.maximum(lh, 0)] == tdi)
        LH = np.where(ok, hw[np.maximum(lh, 0)], np.nan); LL = np.where(ok, lw[np.maximum(ll, 0)], np.nan); rng_ = LH - LL; rd = np.where(ok & (rng_ > 0), (LH - cw) / np.where(rng_ > 0, rng_, np.nan), np.nan)
        return rd, rng_, (ix - lh).astype(float)
    _hh = np.array([1, 2, 5, 3, 2, 1, 2, 3, 4.0, 3, 2, 1]); _ll = _hh - 1.0; _rd, _, _ = kn_pivots(_hh, _ll, np.zeros(12, int), _hh - 0.5)
    _rd2, _, _ = kn_pivots(_hh[:9], _ll[:9], np.zeros(9, int), (_hh - 0.5)[:9]); assert np.allclose(_rd[:9], _rd2, equal_nan=True), "swing state must not use bars after it"
    print("unit check passed: fractal swing state is point in time (confirmed two bars after the pivot)")
    def kn_mtf(p, KB, idx):
        # completed higher-frame candle bodies (sign of close - open), aligned to the bar index with a forward fill of bars that have already ended
        d5 = kn_prep(p, 5); out = {}
        for M in ((15, 30) if KB == 5 else (30, 60)):
            r = d5.resample(f"{M}min", label="right", closed="right"); g = pd.DataFrame(dict(op=r["op"].first(), cl=r["cl"].last())).dropna(); out[M] = np.sign(g["cl"] - g["op"]).reindex(idx, method="ffill").to_numpy()
        return out
    def kn_events(KB):
        wins = {k: v for k, v in kc_windows_for(KB).items() if k in KN_WN}; fam = {}
        def push(name, L, mask, f, s, side, extra, y, y2, unit, meta):
            ix = np.where(mask)[0]
            if not len(ix): return
            d = pd.DataFrame({k: v[ix] for k, v in extra.items()}); d["y"] = side * y[ix]; d["y2"] = side * y2[ix]; d["unit"] = unit[ix]
            for k, v in meta.items(): d[k] = v[ix]
            d["world"] = s; d["L"] = L; fam.setdefault(name, []).append(d)
        for pk_, p in enumerate(KN_T[KB]):
            f = KN_T[KB][p]; n = len(f); W = kc_wins(f, wins, 30 // KB, False); r1, r2 = W[KN_WN[0]][0].astype(float), W[KN_WN[1]][0].astype(float); unit = f["sc"].to_numpy(float) * np.sqrt(30 // KB) * 1e4
            dts = pd.Timestamp("2000-01-01") + pd.to_timedelta(f["tdi"].to_numpy(), unit="D"); sp = KN_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); vr = KC_REG[p].reindex(dts).fillna(-1).to_numpy().astype(int); tr = RG["T"].reindex(dts).fillna(-1).to_numpy().astype(int)
            ok0 = (sp >= 0) & (vr >= 0) & (tr >= 0) & np.isfinite(r1) & f["sc"].notna().to_numpy(); slot = (f.index.hour * 60 + f.index.minute).to_numpy(); ph = kn_phase(slot); tdi = f["tdi"].to_numpy()
            meta = dict(spl=sp, vol=vr, trend=tr, tdi=tdi, phase=ph, pid=np.full(n, pk_)); mt = kn_mtf(p, KB, f.index)
            for s in (1, -1):
                for L in KN_LBK:
                    Wd = kc_world(f, s, (L,)); g = lambda k: Wd[k].to_numpy(float)
                    if s == 1: hw, lw, cw = f["h"].to_numpy(float), f["l"].to_numpy(float), f["c"].to_numpy(float)
                    else: hw, lw, cw = -f["l"].to_numpy(float), -f["h"].to_numpy(float), -f["c"].to_numpy(float)
                    sc_ = f["sc"].to_numpy(float); bodyw = s * f["body"].to_numpy(float); doi = f["doi"].to_numpy(float); oi_ = f["oi"].to_numpy(float) if "oi" in f else np.full(n, np.nan)
                    base = dict(brr=g("brr"), rng=g("rng"), vz=g("vz"), uw=g("uw"), lw=g("lw"), tr6=g("tr6"), doiz=f["doiz"].to_numpy(float), dr=f["dr"].to_numpy(float), voiz=f["voiz"].to_numpy(float), ovs=bodyw * f["ov"].to_numpy(float),
                                pvw=s * f["pvw"].to_numpy(float), nf=s * f["nf"].to_numpy(float), vp=s * f["vp"].to_numpy(float), mtf=sum((s * m == bodyw).astype(float) * (bodyw != 0) for m in mt.values()))
                    # 1. breakout: close above the prior L-bar high; gates volume, open interest, VWAP, higher time frames
                    brk = g(f"brk{L}") == 1; push("BRK", L, brk & ok0, f, s, 1, base, r1 * s, r2 * s, unit, meta)
                    # 2. fakeout: the bar after a failed probe of the prior high closes below the prior low (fade). OI and volume of the probe bar are added
                    fk = g(f"fk{L}") == 1; prv = lambda a: np.r_[np.nan, a[:-1]]; ex = dict(base); ex.update(doiz_prev=prv(base["doiz"]), vz_prev=prv(base["vz"]), dr_prev=prv(base["dr"]), pvw_prev=prv(base["pvw"])); push("FAKE", L, fk & ok0, f, s, -1, ex, r1 * s, r2 * s, unit, meta)
                    # 3. pullback after a breakout, read k bars later: correction (OI and volume fade) or reversal (new positions against the breakout)
                    pm = pd.Series(hw).rolling(L, min_periods=L).max().shift(1).to_numpy(); cs = np.cumsum(np.nan_to_num(doi)); cv = np.cumsum(np.nan_to_num(f["vol"].to_numpy(float))); ix = np.arange(n); vol_ = f["vol"].to_numpy(float)
                    jb = np.maximum.accumulate(np.where(brk, ix, -1))
                    for k in (3, 6):
                        j = np.where(ix - k >= 0, ix - k, 0); isb = brk[j] & (tdi[j] == tdi) & (f["seg"].to_numpy()[j] == f["seg"].to_numpy()); pull = isb & (cw < cw[j])
                        ex = dict(base); ex.update(oi_since=(cs - cs[j]) / np.where(oi_[j] > 0, oi_[j], np.nan), vol_ratio=((cv - cv[j]) / k) / np.where(vol_[j] > 0, vol_[j], np.nan), depth=(cw[j] - cw) / sc_, held=(cw > pm[j]).astype(float), k=np.full(n, float(k)))
                        push("PULL", L, pull & ok0, f, s, 1, ex, r1 * s, r2 * s, unit, meta)
                    # 4. spring / selling climax: the low breaks the prior L-bar low (in the negated series: an upthrust / buying climax); liquidation (OI down) with the close back inside is the spring
                    pmin = pd.Series(lw).rolling(L, min_periods=L).min().shift(1).to_numpy(); newlow = lw < pmin; ex = dict(base); ex.update(back_in=(cw > pmin).astype(float), depth_lo=(pmin - lw) / sc_)
                    push("SPRING", L, newlow & ok0, f, s, 1, ex, r1 * s, r2 * s, unit, meta)
                    # 6. Fibonacci: pullback depth from the last confirmed fractal high, as a fraction of the leg from the last confirmed fractal low (resume in the leg's direction)
                    rd, leg, age = kn_pivots(hw, lw, tdi, cw); ex = dict(base); ex.update(rd=rd, leg=leg / sc_, age=age)
                    if L == KN_LBK[0]: push("FIB", 0, np.isfinite(rd) & (rd > 0) & (rd < 1.5) & ok0, f, s, 1, ex, r1 * s, r2 * s, unit, meta)
                # 5. doji after a run: the parent is a notable trailing move (top quartile of the training period), the candle is small-bodied; reversal
                Wd = kc_world(f, s, (KN_LBK[0],)); tr6 = Wd["tr6"].to_numpy(float); q75 = np.nanquantile(tr6[(sp == 0) & np.isfinite(tr6)], 0.75) if ((sp == 0) & np.isfinite(tr6)).sum() > 100 else np.nan
                base = dict(brr=Wd["brr"].to_numpy(float), rng=Wd["rng"].to_numpy(float), vz=Wd["vz"].to_numpy(float), uw=Wd["uw"].to_numpy(float), lw=Wd["lw"].to_numpy(float), tr6=tr6, doiz=f["doiz"].to_numpy(float), dr=f["dr"].to_numpy(float), pvw=s * f["pvw"].to_numpy(float))
                push("DOJI", 0, (tr6 >= q75) & ok0, f, s, -1, base, r1 * s, r2 * s, unit, meta)
        return {k: pd.concat(v, ignore_index=True) for k, v in fam.items()}
    KN_E = {KB: kn_events(KB) for KB in KN_BARS}
    for KB in KN_BARS:
        for k, v in KN_E[KB].items(): v["dayi"] = np.unique(v["tdi"], return_inverse=True)[1].astype(np.int64)
    print({f"{KB}m": {k: len(v) for k, v in d.items()} for KB, d in KN_E.items()}, "event rows (parents)")

In [ ]:
# ---- 18n.8 evaluate the setups: each gate's cut is learned on train_R per volatility regime (the pasted number is shown beside it); the gated setup must beat its own parent ----
if KN_OK and KN_T and KN_E:
    KN_COST = 2.0 * float(globals().get("DEFAULT_COST_BPS", 2.0));   # round trip: DEFAULT_COST_BPS is charged per side
    KN_RGS = [("all", None, None)] + [(f"vol {KN_VN[v]}", "vol", v) for v in range(3)] + [(f"trend {RG_TN[v]}", "trend", v) for v in range(3)]
    KN_SETS = {   # variant: (parent family, sign of the thesis outcome, gates). Gate kinds: hi / lo = learned tail of the feature; nat1 / nat0 / nat2 = natural split (feature == 1 / 0 / 2)
        "BRK_OI": ("BRK", 1, [("volume", "vz", "hi"), ("OI up", "doiz", "hi"), ("above VWAP", "pvw", "hi"), ("higher frames agree", "mtf", "nat2")]),
        "FAKE_OI": ("FAKE", 1, [("OI up on the probe", "doiz_prev", "hi"), ("volume on the probe", "vz_prev", "hi"), ("OI falls now", "doiz", "lo"), ("below VWAP now", "pvw", "lo")]),
        "PULL_CORR": ("PULL", 1, [("OI falls in the pullback", "oi_since", "lo"), ("volume fades", "vol_ratio", "lo"), ("breakout level holds", "held", "nat1")]),
        "PULL_REV": ("PULL", -1, [("OI rises in the pullback", "oi_since", "hi"), ("volume stays high", "vol_ratio", "hi"), ("breakout level lost", "held", "nat0")]),
        "SPRING_OI": ("SPRING", 1, [("volume", "vz", "hi"), ("range", "rng", "hi"), ("OI liquidation", "doiz", "lo"), ("closed back inside", "back_in", "nat1")]),
        "DOJI_RUN": ("DOJI", 1, [("small body", "brr", "lo"), ("volume", "vz", "hi"), ("OI falls", "doiz", "lo"), ("small range", "rng", "lo")]),
    }
    def kn_learn(df, feat, kind, yy):
        # cut per volatility regime from train_R rows of the parent: the tail start whose train t of the thesis (outcome > 0) is largest. A regime with fewer than 400 train rows borrows the pooled cut.
        # Validation and test play no part.
        x = df[feat].to_numpy(float); out = {}; qs = []; unit = df["unit"].to_numpy(float); day = df["dayi"].to_numpy(); nd = int(day.max()) + 1
        def best(tr):
            if tr.sum() < 400: return None
            b = None
            for q in KN_TAILS:
                cut = np.quantile(x[tr], q if kind == "hi" else 1 - q); g = tr & ((x >= cut) if kind == "hi" else (x <= cut)); n, m, t = kn_group(yy * unit, day, g, nd)
                if np.isfinite(t) and (b is None or t > b[0]): b = (t, q, cut)
            return b
        base = (df["spl"].to_numpy() == 0) & np.isfinite(x) & np.isfinite(yy); pooled = best(base)
        for v in range(3):
            b = best(base & (df["vol"].to_numpy() == v)) or pooled; out[v] = None if b is None else b[2]
            if b is not None: qs.append(b[1])
        return out, (float(np.median(qs)) if qs else np.nan)
    def kn_apply(df, feat, kind, cuts):
        x = df[feat].to_numpy(float)
        if kind.startswith("nat"): return np.isfinite(x) & (x == int(kind[3]))
        v = df["vol"].to_numpy(); c = np.array([np.nan if cuts.get(k) is None else cuts[k] for k in range(3)])[v]
        return np.isfinite(x) & np.isfinite(c) & ((x >= c) if kind == "hi" else (x <= c))
    KN_EV, KN_Q = [], []
    for KB in KN_BARS:
        for vname, (pf, sg, gates) in KN_SETS.items():
            D = KN_E[KB].get(pf)
            if D is None: continue
            for L in sorted(D["L"].unique()):
                d = D[D["L"] == L].reset_index(drop=True); yy = sg * d["y"].to_numpy(float); y2 = sg * d["y2"].to_numpy(float); unit = d["unit"].to_numpy(float); nd = int(d["dayi"].max()) + 1; day = d["dayi"].to_numpy(); spl = d["spl"].to_numpy()
                masks = [("parent", np.ones(len(d), bool))]; cum = np.ones(len(d), bool); alone = {}
                for gname, feat, kind in gates:
                    cuts, qm = kn_learn(d, feat, kind, yy); m_ = kn_apply(d, feat, kind, cuts) if feat in d else np.zeros(len(d), bool); alone[gname] = m_; cum = cum & m_; KN_Q.append(dict(variant=vname, bar=f"{KB}m", L=int(L), gate=gname, feature=feat, kind=kind, learned_tail=qm, cuts={k: (None if v_ is None else round(float(v_), 3)) for k, v_ in cuts.items()}))
                    masks.append(("+" + gname + " only", m_)); masks.append(("cumulative to " + gname, cum.copy()))
                for step, m_ in masks:
                    par = np.ones(len(d), bool)
                    for rname, rcol, rv in KN_RGS:
                        rm = (np.ones(len(d), bool) if rcol is None else (d[rcol].to_numpy() == rv)); row = dict(variant=vname, bar=f"{KB}m", L=int(L), step=step, regime=rname)
                        for si, sn in enumerate(KN_SPN):
                            sm = spl == si; ch = m_ & rm & sm; n_, mm, tt = kn_group(yy * unit, day, ch, nd); row.update({f"n_{sn}": n_, f"bp_{sn}": mm, f"t_{sn}": tt})
                            if step != "parent":
                                sel = rm & sm; dm, dt = kn_diff((yy * unit)[sel], (m_ & rm)[sel], day[sel], nd); row[f"diff_bp_{sn}"] = dm; row[f"diff_t_{sn}"] = dt
                        n_, mm, tt = kn_group(y2 * unit, day, m_ & rm & (spl == 2), nd); row.update(bp_test_30_60=mm, t_test_30_60=tt); KN_EV.append(row)
    KN_EV = pd.DataFrame(KN_EV); KN_QT = pd.DataFrame(KN_Q)
    NT_EV = int(((KN_EV.n_test >= 100)).sum()); T_EV = bonferroni_t(max(NT_EV, 1))
    c = KN_EV; sgn_ok = (c.t_train_R > 2) & (c.t_val_R > 2) & (c.t_test > T_EV) & (c.bp_test > 0)
    gate_ok = (c.step == "parent") | ((c.diff_t_val_R > 0) & (c.diff_t_test > 0) & (c.diff_bp_test > 0))
    KN_EV["supported"] = sgn_ok & gate_ok & (c.n_test >= 100) & (c.n_val_R >= 100) & (c.n_train_R >= 100); KN_EV["net_test_bp"] = KN_EV.bp_test - KN_COST
    KN_EV["clears_cost"] = KN_EV.supported & (KN_EV.net_test_bp > 0)
    KN_EV.round(4).to_csv(OUT2 / "setups_oi_gated.csv", index=False); KN_QT.to_csv(OUT2 / "setups_learned_cuts.csv", index=False)
    print(f"Setups gated by volume, open interest and VWAP: {len(KN_EV)} rows, {NT_EV} with at least 100 test events; Bonferroni t > {T_EV:.2f} on the test; thesis = outcome > 0 (breakout continues, failed probe is faded, correction resumes, reversal reverses, spring bounces, doji after a run reverses).")
    print("Parents and the gate ladder, regime 'all', test period (bp per trade over the next 30 minutes, day-clustered t, difference to the rest of the parent):")
    _sh = KN_EV[(KN_EV.regime == "all") & KN_EV.step.isin(["parent"] + [s for s in KN_EV.step.unique() if s.startswith("cumulative")])]
    print(_sh[["variant", "bar", "L", "step", "n_test", "bp_train_R", "bp_val_R", "bp_test", "t_test", "diff_t_test", "supported"]].round(2).to_string(index=False))
    print("Supported (positive in train_R, val_R and test, beats its parent, t above the bar):"); _sp = KN_EV[KN_EV.supported]; print(_sp[["variant", "bar", "L", "step", "regime", "n_test", "bp_test", "t_test", "diff_t_test", "bp_test_30_60", "clears_cost"]].round(2).sort_values("t_test", ascending=False).head(25).to_string(index=False) if len(_sp) else "none")
    print(f"learned tails (median over regimes) next to the pasted suggestion 'volume above 2x its mean' (log 2 = {KN_DOC['vz']:.2f} in the notebook's log-volume units):"); print(KN_QT[KN_QT.kind.isin(["hi", "lo"])].groupby(["variant", "gate"])["learned_tail"].median().round(2).to_string())
    # ---- Fibonacci: response curve of the next 30 minutes to the retracement depth; the pasted levels 0.382 / 0.5 / 0.618 against everything else in the same band ----
    KN_FIBR = []
    for KB in KN_BARS:
        D = KN_E[KB].get("FIB")
        if D is None: continue
        rd = D["rd"].to_numpy(float); yv = D["y"].to_numpy(float) * D["unit"].to_numpy(float); day = D["dayi"].to_numpy(); nd = int(day.max()) + 1; spl = D["spl"].to_numpy(); edges = np.linspace(0, 1.2, 13)
        mu = {s: np.nanmean(yv[spl == s]) for s in range(3)}; ye = yv - np.array([mu[s] for s in range(3)])[spl]
        for b in range(12):
            for si, sn in enumerate(KN_SPN):
                n_, m_, t_ = kn_group(ye, day, (rd >= edges[b]) & (rd < edges[b + 1]) & (spl == si), nd); KN_FIBR.append(dict(bar=f"{KB}m", kind="depth bin", cell=f"{edges[b]:.1f}-{edges[b+1]:.1f}", split=sn, n=n_, bp=m_, t=t_))
        near = np.zeros(len(D), bool)
        for lv in (0.382, 0.5, 0.618): near |= np.abs(rd - lv) <= 0.05
        band = (rd >= 0.3) & (rd <= 0.7)
        for si, sn in enumerate(KN_SPN):
            n_, m_, t_ = kn_group(ye, day, near & (spl == si), nd); KN_FIBR.append(dict(bar=f"{KB}m", kind="pasted levels (0.382 / 0.5 / 0.618 +- 0.05)", cell="levels", split=sn, n=n_, bp=m_, t=t_))
            sel = band & (spl == si); dm, dt = kn_diff(ye[sel], near[sel], day[sel], nd); KN_FIBR.append(dict(bar=f"{KB}m", kind="levels minus the rest of the 0.3-0.7 band", cell="difference", split=sn, n=int((near & band & (spl == si)).sum()), bp=dm, t=dt))
    KN_FIBR = pd.DataFrame(KN_FIBR); KN_FIBR.round(4).to_csv(OUT2 / "fibonacci_response.csv", index=False)
    print("\nFibonacci: excess next-30-minute return (bp) after a retracement of the last fractal swing; thesis = the swing resumes."); print(KN_FIBR[KN_FIBR.kind != "depth bin"].pivot_table(index=["bar", "kind"], columns="split", values="t").round(2).to_string())
    _fb = KN_FIBR[KN_FIBR.kind == "depth bin"]; KN_FIBSUP = _fb.pivot_table(index=["bar", "cell"], columns="split", values="t"); KN_FIBSUP = KN_FIBSUP[(np.sign(KN_FIBSUP.train_R) == np.sign(KN_FIBSUP.val_R)) & (np.sign(KN_FIBSUP.val_R) == np.sign(KN_FIBSUP.test)) & (KN_FIBSUP.test.abs() > bonferroni_t(24))]
    print("depth bins with the same sign in all three periods and |t| above the bar on the test:", KN_FIBSUP.round(2).to_dict("index") or "none")

In [ ]:
# ---- 18n.9 lead-lag on the framework signals (not on prices): Granger with a lag-decay matrix, then transfer entropy for the pairs that pass; daily / weekly / monthly features the notes ask for ----
if KN_OK and KN_T and "score_pool" in KN_P.get(5, {}):
    KN_LAGS = (1, 2, 3, 6)                                      # in 5-minute bars: 5, 10, 15 and 30 minutes
    P5 = KN_P[5]; names5 = list(KN_T[5]); KN_SIG = "score_pool"
    def kn_wide(col):
        # one column per product on a common (trading day, slot) grid; rows are bars
        d = P5[["pid", "tdi", "slot", col]].copy(); d["p"] = [names5[i] for i in d["pid"]]
        return d.pivot_table(index=["tdi", "slot"], columns="p", values=col, aggfunc="last").sort_index()
    def kn_granger(x, y, lags, spl, day, nday):
        # y_t on its own two lags and on x_{t-k}, one k at a time; day-clustered t of the x coefficient. x leads y when the coefficient is positive and stable across periods
        out = {}
        for k in lags:
            X = np.c_[np.ones(len(y)), np.r_[np.full(1, np.nan), y[:-1]], np.r_[np.full(2, np.nan), y[:-2]], np.r_[np.full(k, np.nan), x[:-k]]]; ok = np.isfinite(X).all(1) & np.isfinite(y)
            res = []
            for s in range(3):
                m = ok & (spl == s)
                if m.sum() < 500: res.append((np.nan, np.nan)); continue
                Xs, ys, ds = X[m], y[m], day[m]; XtX = Xs.T @ Xs; b = np.linalg.solve(XtX, Xs.T @ ys); e = ys - Xs @ b; sc = Xs * e[:, None]; G = np.stack([np.bincount(ds, weights=sc[:, j], minlength=nday) for j in range(4)], 1); Vi = np.linalg.inv(XtX); V = Vi @ (G.T @ G) @ Vi
                res.append((b[3], b[3] / np.sqrt(V[3, 3])))
            out[k] = res
        return out
    # unit check: a planted lead of 2 bars is found at lag 2 and not at lag 6
    _r = np.random.default_rng(5); _n = 30000; _x = _r.normal(size=_n); _y = 0.3 * np.r_[0, 0, _x[:-2]] + _r.normal(size=_n); _day = np.arange(_n) // 50; _sp = np.where(np.arange(_n) < _n // 2, 0, 1)
    _g = kn_granger(_x, _y, (1, 2, 6), _sp, _day, int(_day.max()) + 1); assert _g[2][0][1] > 8 and abs(_g[6][0][1]) < 3.5 and abs(_g[1][0][1]) < 3.5, "Granger lag detection wrong"
    def kn_te(x, y, k, bins=3):
        # transfer entropy x -> y at lag k, three equal-count states each, history of one: TE = sum p(y1, y0, x0) log [ p(y1 | y0, x0) / p(y1 | y0) ]
        ok = np.isfinite(x) & np.isfinite(y); xs = np.r_[np.full(k, np.nan), x[:-k]]; y0 = np.r_[np.nan, y[:-1]]; ok &= np.isfinite(xs) & np.isfinite(y0)
        if ok.sum() < 2000: return np.nan
        q = lambda v: np.digitize(v, np.quantile(v, np.linspace(0, 1, bins + 1)[1:-1])); a, b0, c = q(y[ok]), q(y0[ok]), q(xs[ok]); J = np.zeros((bins,) * 3); np.add.at(J, (a, b0, c), 1); J /= J.sum()
        p_y0x = J.sum(0); p_y1y0 = J.sum(2); p_y0 = J.sum((0, 2)); te = 0.0
        for i in range(bins):
            for j in range(bins):
                for l in range(bins):
                    if J[i, j, l] > 0: te += J[i, j, l] * np.log((J[i, j, l] / p_y0x[j, l]) / (p_y1y0[i, j] / p_y0[j]))
        return float(te)
    _tx = _r.normal(size=40000); _ty = np.r_[0, 0, 0, _tx[:-3]] + 0.8 * _r.normal(size=40000); assert kn_te(_tx, _ty, 3) > 5 * max(kn_te(_ty, _tx, 3), 1e-5), "transfer entropy direction wrong"
    print("unit checks passed: Granger recovers a planted 2-bar lead, transfer entropy has the right direction")
    SG = kn_wide(KN_SIG); SPLw = kn_wide("spl").fillna(-1).astype(int); DAYw = kn_wide("dayi").fillna(0).astype(int); SPLm = SPLw.mode(axis=1)[0].astype(int).to_numpy(); DAYm = DAYw.max(axis=1).to_numpy(); nday = int(DAYm.max()) + 1
    KN_GR = []
    for yv in SG.columns:
        for xv in SG.columns:
            if xv == yv: continue
            g = SG[[xv, yv]].dropna()
            if len(g) < 3000: continue
            ix = SG.index.get_indexer(g.index); out = kn_granger(g[xv].to_numpy(), g[yv].to_numpy(), KN_LAGS, SPLm[ix], DAYm[ix], nday)
            for k, res in out.items(): KN_GR.append(dict(leader=xv, follower=yv, lag_bars=k, lag_min=5 * k, **{f"{a}_{sn}": res[i][j] for i, sn in enumerate(KN_SPN) for j, a in enumerate(("beta", "t"))}))
    KN_GR = pd.DataFrame(KN_GR)
    if len(KN_GR):
        KN_GR["same_sign"] = (np.sign(KN_GR.t_train_R) == np.sign(KN_GR.t_val_R)) & (np.sign(KN_GR.t_val_R) == np.sign(KN_GR.t_test)); NT_GR = len(KN_GR); T_GR = bonferroni_t(NT_GR)
        KN_GR["supported"] = KN_GR.same_sign & (KN_GR.t_train_R.abs() > 2) & (KN_GR.t_val_R.abs() > 2) & (KN_GR.t_test.abs() > T_GR)
        top = KN_GR[KN_GR.supported].sort_values("t_test", key=abs, ascending=False)
        for r in top.head(10).itertuples():                                  # transfer entropy on the Granger survivors only, against a shuffled-lead null
            g = SG[[r.leader, r.follower]].dropna(); xx, yy_ = g[r.leader].to_numpy(), g[r.follower].to_numpy(); te = kn_te(xx, yy_, r.lag_bars); rng_ = np.random.default_rng(0); nul = [kn_te(rng_.permutation(xx), yy_, r.lag_bars) for _ in range(20)]
            KN_GR.loc[(KN_GR.leader == r.leader) & (KN_GR.follower == r.follower) & (KN_GR.lag_bars == r.lag_bars), ["te", "te_null_p95"]] = [te, float(np.nanquantile(nul, 0.95))]
        KN_GR.round(4).to_csv(OUT2 / "framework_leadlag.csv", index=False)
        KN_GRM = KN_GR.pivot_table(index="lag_min", values="t_test", aggfunc=lambda s: float(np.mean(np.abs(s)))); KN_GRS = KN_GR.groupby("lag_min")["supported"].sum()
        print(f"Granger on the learned framework signal ({KN_SIG}), {NT_GR} leader x follower x lag tests, Bonferroni t > {T_GR:.2f} on the test, t > 2 and the same sign in train_R and val_R:")
        print(pd.DataFrame({"mean |t| test": KN_GRM["t_test"], "supported pairs": KN_GRS}).round(2).to_string())
        if len(top): print(top.head(10)[["leader", "follower", "lag_min", "t_train_R", "t_val_R", "t_test"]].round(2).merge(KN_GR[["leader", "follower", "lag_min", "te", "te_null_p95"]].dropna(), on=["leader", "follower", "lag_min"], how="left").to_string(index=False))
    # ---- daily, weekly, monthly features that the notes list: gap fill, night / day volume ratio, weekly skew and volume profile, quarter-over-quarter OI growth, loadings on the market factor ----
    DA = []
    for p in KC_PRODS:
        f = KN_T[5][p]; td = pd.DatetimeIndex(session_day(f.index)); night = ((f.index.hour >= 20) | (f.index.hour < 4)); v = f["vol"]
        g = pd.DataFrame(dict(td=td, night=night, vol=v.to_numpy(), c=f["c"].to_numpy(), o=f["o"].to_numpy(), seg=f["seg"].to_numpy(), rb=f["r"].isna().to_numpy())); day = g[~g.night]
        if day.empty: continue
        day = day.assign(k=day.groupby("td").cumcount()); first = day.groupby("td").first(); lastc = day.groupby("td")["c"].last(); prev_close = lastc.shift(1); gap = first["o"] - prev_close
        c30 = day[day.k == 5].set_index("td")["c"].reindex(first.index).fillna(first["c"])
        fill = np.clip((c30 - first["o"]) / (-gap).replace(0, np.nan), -2, 2)          # share of the overnight gap retraced by the first 30 minutes (1 = fully filled, > 1 overshoot, < 0 extended)
        nv = g[g.night].groupby("td")["vol"].sum(); dv = day.groupby("td")["vol"].sum(); DA.append(pd.DataFrame(dict(product=p, gap=gap, gap_fill=fill, night_day_vol=np.log((nv.reindex(dv.index) + 1) / (dv + 1)))).reset_index())
    KN_DA = pd.concat(DA, ignore_index=True) if DA else pd.DataFrame()
    if len(KN_DA):
        print(f"Gap fill and night / day volume: {KN_DA['gap_fill'].notna().sum():,} product-days, median gap fill {KN_DA['gap_fill'].median():.2f}, median log night / day volume {KN_DA['night_day_vol'].median():.2f}")
    # weekly: skew of the week's daily returns and where the week's volume traded; monthly / quarterly: OI growth. Next-week return after demeaning each week across products (cross-section), split by train_R / val_R / test weeks
    wk = []
    for p in KC_PRODS:
        r = RAW[p].dropna(); v = VOLU[p].reindex(r.index); cl = np.exp(r.cumsum()); w = r.index.to_period("W")
        sk = r.groupby(w).apply(lambda z: z.skew() if len(z) >= 4 else np.nan); vp = (cl * v).groupby(w).sum() / v.groupby(w).sum()                 # volume-weighted price of the week
        ret = r.groupby(w).sum(); last = cl.groupby(w).last(); pos = np.log(last / vp)                                                           # where the week closed relative to where the volume traded
        try: oi = wide("oi")[p].reindex(r.index).where(lambda x: x > 0); q = np.log(oi.groupby(oi.index.to_period("Q")).last()).diff().shift(1)           # growth of the previous completed quarter: the quarter of the week is not finished yet
        except Exception: q = None
        d = pd.DataFrame(dict(product=p, wskew=sk, wpos=pos, wret=ret)); d["fwd_wk"] = d["wret"].shift(-1); d["wend"] = r.index.to_series().groupby(w).last().reindex(d.index)           # last trading day of the week: the week belongs to the period of that day
        if q is not None: d["qoq_oi"] = [q.get(x.asfreq("Q"), np.nan) for x in d.index]
        d.index.name = "week"; wk.append(d.reset_index())
    KN_WK = pd.concat(wk, ignore_index=True)
    for c in ("wskew", "wpos", "qoq_oi", "fwd_wk"):
        if c in KN_WK: KN_WK[c + "_x"] = KN_WK[c] - KN_WK.groupby("week")[c].transform("mean")
    sw = KN_SPLR.reindex(KN_WK["wend"]).to_numpy(); KN_WK["spl"] = np.where(np.isnan(sw), -1, sw).astype(int); KN_WK["wi"] = np.unique(KN_WK["week"].astype(str), return_inverse=True)[1]; nw = int(KN_WK["wi"].max()) + 1; rows = []
    for c, lab in (("wskew", "weekly skew of daily returns"), ("wpos", "week close vs volume-weighted price of the week"), ("qoq_oi", "quarter-over-quarter OI growth")):
        if c + "_x" not in KN_WK: continue
        for s, sn in enumerate(KN_SPN):
            m = (KN_WK.spl == s) & KN_WK[c + "_x"].notna() & KN_WK["fwd_wk_x"].notna(); st = np.sign(KN_WK.loc[m, c + "_x"]) * KN_WK.loc[m, "fwd_wk_x"] * 1e4; n_, mm, tt = kn_group(st.to_numpy(), KN_WK.loc[m, "wi"].to_numpy(), np.ones(int(m.sum()), bool), nw)
            rows.append(dict(feature=lab, split=sn, n=n_, bp=mm, t=tt))
    KN_WKT = pd.DataFrame(rows); KN_WKT.round(4).to_csv(OUT2 / "weekly_features.csv", index=False)
    print("\nWeekly and quarterly features, cross-sectional sign strategy on the next week (bp, week-clustered t); a negative mean = the feature reverses:"); print(KN_WKT.pivot_table(index="feature", columns="split", values="t").reindex(columns=list(KN_SPN)).round(2).to_string())
    # yearly loadings on the equal-weight market factor and whether last year's loading says anything about this year's return (macro-factor stand-in; the notes ask for macro factors we do not have)
    MKT_ = RAW.mean(axis=1); yl = {}
    for y, g in RAW.groupby(RAW.index.year):
        if len(g) > 120: yl[y] = g.apply(lambda s: s.cov(MKT_.loc[g.index]) / MKT_.loc[g.index].var())
    KN_YL = pd.DataFrame(yl).T; KN_YL.round(3).to_csv(OUT2 / "yearly_market_loadings.csv"); _stab = np.nanmean([KN_YL.loc[a].corr(KN_YL.loc[b]) for a, b in zip(KN_YL.index[:-1], KN_YL.index[1:])]) if len(KN_YL) > 1 else np.nan
    print(f"Yearly loading on the equal-weight market factor: {KN_YL.shape[0]} years x {KN_YL.shape[1]} products, year-to-year correlation of the loadings across products {_stab:.2f} (a macro factor is not in the data: this is the market factor only)")

In [ ]:
# ---- 18n.10 charts, summary for the findings report ----
KN_SUMMARY = None
if KN_OK and KN_T and "KN_EV" in globals():
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    _c = KN_CT[(KN_CT.regime == "all") & (KN_CT.window == "0-30m")]
    for KB, col in (("5m", "#1b6ca8"), ("15m", "#e67e22")):
        g = _c[_c.bar == KB].set_index("component").reindex([k for k in KN_COMPS if k != "fp"]); axs[0].barh(np.arange(len(g)) + (0.2 if KB == "15m" else -0.2), g["t_test"].to_numpy(), 0.4, label=KB, color=col)
    axs[0].set_yticks(range(len(g))); axs[0].set_yticklabels(g.index, fontsize=7); axs[0].axvline(T_CT, color="k", ls=":", lw=.8); axs[0].axvline(-T_CT, color="k", ls=":", lw=.8); axs[0].set_title("framework components: sign strategy, test t (next 30 min)"); axs[0].legend(fontsize=7); axs[0].grid(alpha=.3)
    _l = KN_EV[(KN_EV.regime == "all") & (KN_EV.variant == "BRK_OI") & (KN_EV.bar == "5m") & (KN_EV.L == KN_LBK[0]) & (KN_EV.step.str.startswith("cumulative") | (KN_EV.step == "parent"))]
    x = np.arange(len(_l));
    for k, (sn, col) in enumerate((("train_R", "#1b6ca8"), ("val_R", "#2a9d8f"), ("test", "#c0392b"))): axs[1].bar(x + (k - 1) * 0.27, _l[f"bp_{sn}"].to_numpy(), 0.27, label=sn, color=col)
    axs[1].set_xticks(x); axs[1].set_xticklabels([s.replace("cumulative to ", "+") for s in _l.step], rotation=35, ha="right", fontsize=7); axs[1].axhline(KN_COST, color="k", ls=":", lw=.8); axs[1].set_title("5m breakout: gate ladder, bp per trade (dotted = default round-trip cost)"); axs[1].legend(fontsize=7); axs[1].grid(alpha=.3)
    _m = KN_CMP[KN_CMP.bar == "5m"]; axs[2].barh(np.arange(len(_m)), _m["t_test_0-30m"].to_numpy(), color="#1b6ca8", label="0-30 min"); axs[2].barh(np.arange(len(_m)) + 0.35, _m["t_test_30-60m"].to_numpy(), 0.35, color="#e67e22", label="30-60 min"); axs[2].set_yticks(np.arange(len(_m)) + 0.17); axs[2].set_yticklabels(_m["model"], fontsize=7)
    axs[2].set_title("composite models, test t (5m): inside versus beyond 30 minutes"); axs[2].legend(fontsize=7); axs[2].grid(alpha=.3); fig.tight_layout(); fig.savefig(OUT2 / "18n_framework_and_setups.png", dpi=110); plt.show()
    _sup = KN_EV[KN_EV.supported & (KN_EV.step != "parent")]; _par = KN_EV[(KN_EV.regime == "all") & (KN_EV.step == "parent")]
    KN_SUMMARY = dict(
        czce=dict(product_years_money_2x=int(KN_MC.money_2x.sum()), product_years_money_half=int(KN_MC.money_half.sum()), product_years_volume_step=int(KN_MC.vol_step_2x.sum()),
                  exchanges_with_2x=sorted(KN_BREAK.exchange.unique().tolist()) if len(KN_BREAK) else [], products_with_2x=KN_BREAK.to_dict("records")[:20] if len(KN_BREAK) else []),
        components=dict(tests=NT_CT, bonferroni_t=round(T_CT, 2), supported=KN_CT[KN_CT.supported & (KN_CT.window == "0-30m")][["bar", "component", "regime", "t_train_R", "t_val_R", "t_test", "bp_test"]].round(2).to_dict("records")[:20]),
        composite=KN_CMP[["bar", "model", "bp_test_0-30m", "t_test_0-30m", "ic_test_0-30m", "t_test_30-60m"]].round(3).to_dict("records"),
        setups=dict(tests=NT_EV, bonferroni_t=round(T_EV, 2), parents=_par[["variant", "bar", "L", "n_test", "bp_test", "t_test", "supported"]].round(2).to_dict("records"),
                    gated_supported=_sup.sort_values("t_test", ascending=False).head(12)[["variant", "bar", "L", "step", "regime", "n_test", "bp_test", "t_test", "diff_t_test", "bp_test_30_60", "clears_cost"]].round(2).to_dict("records"), default_cost_bp=KN_COST),
        fibonacci=dict(supported_bins=KN_FIBSUP.round(2).reset_index().to_dict("records")[:10] if len(KN_FIBSUP) else []),
        leadlag=dict(tests=int(len(KN_GR)) if "KN_GR" in globals() else 0, supported=int(KN_GR.supported.sum()) if "KN_GR" in globals() and len(KN_GR) else 0),
        weekly=KN_WKT.round(3).to_dict("records") if "KN_WKT" in globals() else [])
    _co = KN_SUMMARY["components"]["supported"]; _cp = [r for r in KN_SUMMARY["composite"] if r["bar"] == "5m"]
    KN_SUMMARY["reading"] = (f"Volume-weighted STARK / CAIRO / FLUX at 5 and 15 minutes plus the tape features (net notional flow, volume pressure, OI velocity, price versus VWAP): {len(_co)} component-regime cells keep the same sign in train_R, val_R and test with t > {KN_SUMMARY['components']['bonferroni_t']} on the test; "
        + "composite over the next 30 minutes on the test (5m): " + "; ".join(f"{r['model']}: {r['bp_test_0-30m']:.2f} bp, t {r['t_test_0-30m']:.1f}, beyond 30 min t {r['t_test_30-60m']:.1f}" for r in _cp)
        + f". Open-interest-gated setups: {len(_sup)} gated variants beat their own parent in all three periods with t > {KN_SUMMARY['setups']['bonferroni_t']}; "
        + (f"{int(_sup.clears_cost.sum())} of them clear the default round-trip cost of {KN_COST:.1f} bp. " if len(_sup) else "")
        + f"CZCE: money counted twice in {KN_SUMMARY['czce']['product_years_money_2x']} product-years (exchanges: {', '.join(KN_SUMMARY['czce']['exchanges_with_2x']) or 'none'}); money-based features are scaled by the same slot's previous-20-day median, so no rule from the notes is applied. "
        + f"Granger on the framework signal: {KN_SUMMARY['leadlag']['supported']} of {KN_SUMMARY['leadlag']['tests']} leader x follower x lag tests survive. Fibonacci levels: {'depth bins with a stable sign: ' + str(len(KN_SUMMARY['fibonacci']['supported_bins'])) if KN_SUMMARY['fibonacci']['supported_bins'] else 'no depth bin keeps its sign across the periods'}.")
    print("\n" + KN_SUMMARY["reading"])
# restore every notebook name that this step rebound by accident (loop variables, temporaries); the KN_ / kn_ results are kept
for _k, _v in list(_KN_SAVED.items()):
    if _k in ("_KN_SAVED",) or _k.startswith(("KN_", "kn_")): continue
    if globals().get(_k) is not _v: globals()[_k] = _v

## Step 18o — Stage 2 plus breakout versus the full VCP, at daily, 60-minute and 30-minute frames; sweeps and volume nodes from the Taker note

**Question (as approved).** Does the full Volatility Contraction Pattern beat its own parent, "Stage 2 plus breakout", in train_R, val_R and test, in R multiples, at daily, 60-minute and 30-minute frames? The 5-minute breakout already faded (18n); trend-following is expected to survive in slower frames, so the frame is the thing being varied.

**Definitions (nothing numeric is assumed; every cut is learned on train_R per volatility regime).**

| Piece | Rule here | Pasted number (comparison only) |
|---|---|---|
| Stage 2 | close > EMA150 > EMA200 and EMA200 above its value 20 days earlier. EMA is used instead of SMA, as suggested. Both directions: the short side is the same rule on the negated log price. Intraday bars use the state of the previous trading day. | 150 / 200 averages, within 25 % of the 52-week high, 30 % above the low |
| Parent | Stage 2 and the first close above the prior L-bar high (L = 20 / 50 days; 16 / 32 hourly bars; 24 / 48 half-hour bars) | breakout of the base |
| Contraction | confirmed 5-bar fractal swings (known two bars after the pivot). A pullback is a swing high followed by a swing low; contractions are consecutive pullbacks whose depth shrinks | 2–6 contractions |
| VCP (child) | parent and two or more shrinking pullbacks, a close above the last contraction's high, and the base not older than 120 days (240 half-hours) | — |
| Further gates | 3+ pullbacks, tight and shallow last pullback, volume dry-up inside the last pullback, breakout volume, open interest up, nearness to the 52-week high, not extended above EMA150, stop distance (hurdle), range squeeze before, expansion bar, buy-side flow | volume 3×, TFI 0.65, hurdle 2.5 × cost |
| Stop and entry | stop below the last contraction low (parent: the last confirmed swing low, else the lowest low of 10 bars); entry at the close of the bar after the signal; risk floor of one bar's ex-ante volatility; a stop gapped through fills at the open | — |

**Outcomes.** Main test: R multiple with the structural stop, held for 10 days. Also reported: 5 and 20 days, 5 / 10 / 20-day returns without a stop, range expansion (log of the next 10 days' range over the previous 10), the stop moved to break-even after a 2 % or 3 % move, and a 3 or 5-day time stop for a trade that is not in profit. **Break-even cost** is the gross bp per round trip on the test; it is compared with the round-trip cost (2 × `DEFAULT_COST_BPS` = 6 bp: the cost constant is per side, which the 18n net columns had missed and which is now corrected). Net R subtracts cost divided by the stop distance, so a tighter stop makes a given cost larger in R, not smaller.

**Support rule.** A cell is supported when R is positive in train_R, val_R and test with t > 2, 2 and the Bonferroni bar on the test, and the child beats the rest of its parent in all three periods (difference of R positive, and t of the difference positive in val_R and test), with at least 60 events in each period.

**Taker note.** What bars can test is tested: the hurdle (stop distance learned against 2.5 × cost), squeeze and expansion, a bar-based trade-flow imbalance and VPIN proxy (signed volume from the bar's move, bulk classification), a liquidity sweep that reclaims the prior low with a flip of flow, and the penetration of a high-volume node from the previous 20 days' profile, with the exits compared. Depth decay, order and trade arrival rates, iceberg refresh, the empty book, marketable-limit/IOC orders and latency need order-book or tick data that is not in this data set and are not tested.

*Testing so far was on synthetic data only: the code runs, its unit checks and causality tests pass, but no result from real data has been seen.*

In [ ]:
# step 18o
_KO_SAVED = dict(globals())          # snapshot: the loop variables of this step must not overwrite a notebook name (restored at the end of the step)
KO_OK = bool(globals().get("KN_OK", False)) and bool(globals().get("KN_T")) and all(k in globals() for k in ("kn_learn", "kn_apply", "kn_group", "kn_diff", "KN_SPLR", "KN_SPN", "KN_VN", "KC_REG", "RG", "RG_TN", "KC_RAW5", "session_day", "bonferroni_t", "wide", "VOLU", "OIW", "ROLL", "RAW", "causal_test"))
if not KO_OK:
    print("Step 18o needs Step 18n (and the daily panel and cached 5-minute bars): run them first; skipped")
else:
    # ---- 18o.0 settings. No threshold is assumed: Stage 2 is the ordering of the averages (a definition); every numeric cut (depth, dry-up, volume, distance to the high, hurdle) is learned on train_R per volatility regime ----
    KO_FR = ("D", "60m", "30m"); KO_MIN = {"D": None, "60m": 60, "30m": 30}
    KO_L = {"D": (20, 50), "60m": (16, 32), "30m": (24, 48)}               # parent breakout: close above the prior L-bar high (20 / 50 days; about 3 / 6 days of 60-minute bars; 2 / 4 days of 30-minute bars)
    KO_AGE = {"D": 120, "60m": 120, "30m": 240}                              # a base older than this many bars is not a base
    KO_HDAYS = (5, 10, 20); KO_MAIN = 10                                    # horizons in days; the structural-stop outcome at 10 days is the one that is tested
    KO_BE = (0.02, 0.03); KO_TS = (3, 5)                                     # pasted: move the stop to break-even after a 2-3% move; exit after 3-5 days without profit (compared, not assumed)
    KO_RT = 2.0 * float(globals().get("DEFAULT_COST_BPS", 3.0))              # round trip: DEFAULT_COST_BPS is per side (this corrects the 18n net columns, which subtracted one side only)
    KO_MINN = 60; KO_TAILS = KN_TAILS
    KO_DOC = dict(ma=(150, 200), hurdle=2.5, tfi=0.65, vpin=0.7, hvn_vol=3.0, contractions=(2, 6), near_high=0.25, above_low=0.30)
    KO_RGS = [("all", None, None)] + [(f"vol {KN_VN[v]}", "vol", v) for v in range(3)] + [(f"trend {RG_TN[v]}", "trend", v) for v in range(3)]
    KO_T0 = pd.Timestamp("2000-01-01"); KO_NORM = stats.norm
    KO_REG = {p: (KC_REG[p] if p in KC_REG else kc_regime(p)) for p in RAW.columns}                # the same ex-ante volatility regime for the daily products that have no 5-minute bars
    KO_AD = RAW.fillna(0).cumsum()                                           # adjusted log level: depends on past returns only, so no later revision

    # ---- 18o.1 contraction state from confirmed fractal pivots (causal: a pivot at bar j is known at bar j+2) ----
    def ko_state(H, L, V, maxage):
        # H, L, V: x-space highs, lows (negated for shorts) and volume. For every bar: nc = number of consecutive pullbacks whose depth shrinks, last / first depth,
        # last contraction low Lx (the stop), breakout level Hb, age of the base, volume of the last pullback relative to the base (dry)
        n = len(H); Hs = pd.Series(H); Ls = pd.Series(L)
        ph = (Hs == Hs.rolling(5, center=True).max()).to_numpy(); pl = (Ls == Ls.rolling(5, center=True).min()).to_numpy()   # causal-ok: the centred window only finds the pivot; it is used from bar j+2 on (confirmation lag), checked by the truncation test below
        both = ph & pl; ph = ph & ~both; pl = pl & ~both
        ev = sorted([(j, 1) for j in np.where(ph)[0] if j + 2 < n] + [(j, 0) for j in np.where(pl)[0] if j + 2 < n]); cv = np.r_[0.0, np.cumsum(np.nan_to_num(V))]
        seq = []; upd = []; val = []
        for j, ty in ev:
            pr = H[j] if ty == 1 else L[j]
            if seq and seq[-1][0] == ty:
                if (ty == 1 and pr >= seq[-1][2]) or (ty == 0 and pr <= seq[-1][2]): seq[-1] = [ty, j, pr]
                else: continue
            else: seq.append([ty, j, pr])
            seq = seq[-14:]; pairs = [(seq[k], seq[k + 1]) for k in range(len(seq) - 1) if seq[k][0] == 1 and seq[k + 1][0] == 0]
            if not pairs: continue
            dep = [a[2] - b[2] for a, b in pairs]
            if dep[-1] <= 0: continue
            k = len(pairs) - 1; nc = 1
            while k > 0 and 0 < dep[k] < dep[k - 1]: nc += 1; k -= 1
            (hj, hp), (lj, lp) = (pairs[-1][0][1], pairs[-1][0][2]), (pairs[-1][1][1], pairs[-1][1][2]); jf = pairs[k][0][1]
            hb = max(hp, seq[-1][2]) if seq[-1][0] == 1 else hp
            sp_ = (cv[lj + 1] - cv[hj]) / (lj - hj + 1); ch_ = (cv[lj + 1] - cv[jf]) / (lj - jf + 1)
            upd.append(j + 2); val.append((nc, dep[-1], dep[k], lp, lj, hb, jf, (sp_ / ch_) if ch_ > 0 else np.nan))
        out = np.full((n, 8), np.nan)
        if upd:
            upd = np.array(upd); val = np.array(val, float); pos = np.searchsorted(upd, np.arange(n), side="right") - 1; ok = pos >= 0; out[ok] = val[pos[ok]]
        ix = np.arange(n); age = ix - out[:, 6]; good = np.isfinite(age) & (age <= maxage)
        res = pd.DataFrame(dict(nc=np.where(good, out[:, 0], 0.0), dlast=np.where(good, out[:, 1], np.nan), dfirst=np.where(good, out[:, 2], np.nan), Lx=np.where(good, out[:, 3], np.nan),
                                Hb=np.where(good, out[:, 5], np.nan), age=np.where(good, age, np.nan), dry=np.where(good, out[:, 7], np.nan)))
        lxi = np.where(good, out[:, 4], np.nan); res["lxage"] = ix - lxi
        return res
    # unit check on a hand-built path: pullbacks of depth 10, 6, 3 (three shrinking contractions); the state must show 3 only after the last low is confirmed, and must not change when later bars are removed
    _leg = lambda a, b: np.linspace(a, b, 9)[1:]
    _px = np.r_[100.0, _leg(100, 120), _leg(120, 110), _leg(110, 119), _leg(119, 113), _leg(113, 118), _leg(118, 115), _leg(115, 119), _leg(119, 125)]
    _hh = _px + 0.2; _ll = _px - 0.2; _vv = np.ones(len(_px)); _s = ko_state(_hh, _ll, _vv, 500)
    _l3 = int(np.argmin(np.where(np.arange(len(_px)) > 45, _px, 1e9)))              # the third low (115) sits at bar 56 or so
    assert _s["nc"].iloc[_l3 + 2] == 3 and abs(_s["Lx"].iloc[_l3 + 2] - 114.8) < 1e-6 and abs(_s["Hb"].iloc[_l3 + 2] - 118.2) < 1e-6, "contraction state wrong on the hand-built path"
    assert _s["nc"].iloc[_l3 + 1] <= 2, "the third contraction must not be known before its low is confirmed"
    _m = _l3 + 12; _s2 = ko_state(_hh[:_m], _ll[:_m], _vv[:_m], 500); assert np.allclose(_s.iloc[:_m].fillna(-9).to_numpy(), _s2.fillna(-9).to_numpy()), "state must not use bars after it"
    print("unit check passed: three shrinking pullbacks are counted once the last low is confirmed (two bars after it); the state is point in time")

    # ---- 18o.2 Stage 2 from the daily series (EMA 150 / 200 instead of SMA, as suggested), both directions through the negated series ----
    def ko_stage(dd, s):
        C = s * dd["cl"]; Hh = dd["hi"] if s == 1 else -dd["lo"]; Ll = dd["lo"] if s == 1 else -dd["hi"]
        e1 = C.ewm(span=KO_DOC["ma"][0], adjust=False, min_periods=KO_DOC["ma"][0]).mean(); e2 = C.ewm(span=KO_DOC["ma"][1], adjust=False, min_periods=KO_DOC["ma"][1]).mean()
        s2 = ((C > e1) & (e1 > e2) & (e2 > e2.shift(20))).astype(float).where(e2.shift(20).notna())
        return pd.DataFrame(dict(s2=s2, d52=Hh.rolling(252, min_periods=200).max() - C, up52=C - Ll.rolling(252, min_periods=200).min(), ext=C - e1, slope=e2 - e2.shift(20)))
    _d = pd.DataFrame(dict(cl=np.log(100 + np.cumsum(np.r_[np.zeros(300), np.ones(300)])), hi=0.0, lo=0.0)); _d["hi"] = _d["cl"] + 0.01; _d["lo"] = _d["cl"] - 0.01
    _a = ko_stage(_d.iloc[:450], 1); _b = ko_stage(_d, 1); assert np.allclose(_a.fillna(-9).to_numpy(), _b.iloc[:450].fillna(-9).to_numpy()) and _b["s2"].iloc[-1] == 1 and ko_stage(_d, -1)["s2"].iloc[-1] == 0, "stage 2 wrong or not point in time"
    print("unit check passed: Stage 2 is point in time and flips with the direction")

In [ ]:
# ---- 18o.3 frames: daily (from the daily panel) and 60 / 30 minutes (from the cached 5-minute bars), all on the back-adjusted log price ----
if KO_OK:
    def ko_base(p, FR):
        if FR == "D":
            cl = wide("close")[p].reindex(RAW.index).astype(float); off = KO_AD[p] - np.log(cl); hi = wide("high")[p].reindex(RAW.index).astype(float); lo = wide("low")[p].reindex(RAW.index).astype(float)
            try: op = wide("open")[p].reindex(RAW.index).astype(float)
            except Exception: op = None
            d = pd.DataFrame(dict(op=(np.log(op) + off) if op is not None else KO_AD[p].shift(1), hi=np.log(hi) + off, lo=np.log(lo) + off, cl=KO_AD[p], volume=VOLU[p].astype(float), oi=OIW[p].astype(float), rollb=ROLL[p].astype(float)))
            d = d.replace([np.inf, -np.inf], np.nan).dropna(subset=["hi", "lo", "cl", "op"]); d["slot"] = 0; d["tdi"] = (d.index - KO_T0).days.to_numpy(); return d
        g = KC_RAW5[KC_RAW5["product"] == p].set_index("date").sort_index(); g = g[~g.index.duplicated()]
        cl = g["cl"].astype(float); off = g["ret"].astype(float).fillna(0).cumsum() - np.log(cl); opn = g["op"].astype(float) if "op" in g else cl * np.exp(-g["ret"].astype(float))
        d = pd.DataFrame(dict(op=np.log(opn) + off, hi=np.log(g["hi"].astype(float)) + off, lo=np.log(g["lo"].astype(float)) + off, cl=np.log(cl) + off, volume=g["volume"].astype(float),
                              oi=g["oi"].astype(float) if "oi" in g else np.nan, rollb=g["rollb"].astype(float) if "rollb" in g else 0.0)).replace([np.inf, -np.inf], np.nan)
        r = d.resample(f"{KO_MIN[FR]}min", label="right", closed="right")
        d = pd.DataFrame(dict(op=r["op"].first(), hi=r["hi"].max(), lo=r["lo"].min(), cl=r["cl"].last(), volume=r["volume"].sum(min_count=1), oi=r["oi"].last(), rollb=r["rollb"].max())).dropna(subset=["cl", "hi", "lo", "op"])
        d["slot"] = (d.index.hour * 60 + d.index.minute).to_numpy(); d["tdi"] = (pd.DatetimeIndex(session_day(d.index)) - KO_T0).days.to_numpy(); return d
    KO_PR = {"D": [p for p in UNIV if p in RAW.columns], "60m": list(KC_PRODS), "30m": list(KC_PRODS)}
    KO_B = {FR: {} for FR in KO_FR}
    for FR in KO_FR:
        for p in KO_PR[FR]:
            try: d_ = ko_base(p, FR)
            except Exception as e_: print(f"  {FR} {p}: skipped ({type(e_).__name__}: {str(e_)[:60]})"); continue
            if len(d_) > 400: KO_B[FR][p] = d_
    KO_BPD = {FR: max(1, int(round(float(np.median([v.groupby("tdi").size().median() for v in KO_B[FR].values()]))))) if KO_B[FR] else 1 for FR in KO_FR}
    KO_S2 = {}
    for p, d_ in KO_B["D"].items():
        for s_ in (1, -1): KO_S2[(p, s_)] = ko_stage(d_, s_)
    print({FR: (len(KO_B[FR]), KO_BPD[FR]) for FR in KO_FR}, "(products, bars per trading day) per frame")

    # ---- 18o.4 per-bar features in the direction of the trade (x-space: negated log price for shorts) ----
    def ko_dir(d, s, FR):
        n = len(d); bpd = KO_BPD[FR]
        if s == 1: O, H, L, C = (d[k].to_numpy(float) for k in ("op", "hi", "lo", "cl"))
        else: O, H, L, C = -d["op"].to_numpy(float), -d["lo"].to_numpy(float), -d["hi"].to_numpy(float), -d["cl"].to_numpy(float)
        vol = pd.Series(d["volume"].to_numpy(float)); r = pd.Series(C).diff(); sc = r.rolling(60, min_periods=30).std().shift(1)
        med = vol.groupby(d["slot"].to_numpy()).transform(lambda x: x.rolling(20, min_periods=10).median().shift(1)); vz = np.log(vol.where(vol > 0) / med.where(med > 0))
        oi = pd.Series(d["oi"].to_numpy(float)); rb = d["rollb"].fillna(0).to_numpy() > 0; doi = (oi.diff() / oi.shift(1)).where(~(rb | np.r_[False, rb[:-1]])); doiz = doi / doi.rolling(60, min_periods=30).std().shift(1)
        z = (r / sc).to_numpy(); bf = pd.Series(2 * KO_NORM.cdf(np.nan_to_num(z, nan=0.0)) - 1).where(np.isfinite(z)); sv = (bf * vol.to_numpy()); v20 = vol.rolling(20, min_periods=10).sum()
        tfi = sv.rolling(20, min_periods=10).sum() / v20.where(v20 > 0); flip = ((bf > 0) & (bf.shift(1) < 0)).astype(float).where(bf.notna() & bf.shift(1).notna())
        rngb = pd.Series(H - L); sq = (pd.Series(H).rolling(20).max() - pd.Series(L).rolling(20).min()) / (rngb.rolling(100, min_periods=60).mean() * np.sqrt(20)); ex = r.rolling(5).std() / r.rolling(60, min_periods=30).std()
        f = pd.DataFrame(dict(O=O, H=H, L=L, C=C, sc=sc.to_numpy(), vz=vz.to_numpy(), doiz=doiz.to_numpy(), tfi=tfi.to_numpy(), flip=flip.to_numpy(), sq=sq.to_numpy(), exp=ex.to_numpy(), rollb=rb.astype(float), tdi=d["tdi"].to_numpy(), slot=d["slot"].to_numpy(), volume=d["volume"].to_numpy(float)), index=d.index)
        st = ko_state(H, L, d["volume"].to_numpy(float), KO_AGE[FR]); st.index = f.index; f = pd.concat([f, st], axis=1)
        f["tight"] = f["dlast"] / f["dfirst"]; return f
    def ko_pit(g, FR="30m"):
        f = ko_dir(g, 1, FR); return f[["sc", "vz", "doiz", "tfi", "flip", "sq", "exp", "nc", "dlast", "Lx", "Hb", "dry", "tight"]]
    if KO_B["30m"]:
        _p0 = list(KO_B["30m"])[0]; causal_test("ko_features_30m", ko_pit, KO_B["30m"][_p0].iloc[-4000:], step="18o")
    if KO_B["D"]:
        _p0 = list(KO_B["D"])[0]; causal_test("ko_features_D", lambda g: ko_pit(g, "D"), KO_B["D"][_p0].iloc[-2500:], step="18o")

    # ---- 18o.5 outcome simulation: entry at the close of the bar AFTER the signal (one bar delay), stop = last contraction low, optional break-even move and time stop ----
    def ko_sim(f, i, xs, risk, h_days, be=None, ts=None, FR="D"):
        # i: signal bars. Returns exit price (x-space) and bars held. Stop fills at the stop, or at the open if the bar opens beyond it.
        bpd = KO_BPD[FR]; hb = int(h_days * bpd); e = i + 1; H, L, C, O = (f[k].to_numpy(float) for k in ("H", "L", "C", "O")); out_x = np.full(len(i), np.nan); out_n = np.full(len(i), np.nan)
        for a in range(0, len(i), 4000):
            ee = e[a:a + 4000]; J = ee[:, None] + np.arange(1, hb + 1)[None, :]; Hm, Lm, Cm, Om = H[J], L[J], C[J], O[J]; xe = C[ee]; stop = np.repeat(xs[a:a + 4000, None], hb, axis=1)
            if be is not None:
                cm = np.maximum.accumulate(Hm, axis=1); prev = np.c_[np.full(len(ee), -np.inf), cm[:, :-1]]; stop = np.where(prev >= (xe + be)[:, None], np.maximum(stop, xe[:, None]), stop)
            hit = Lm <= stop; anyh = hit.any(1); first = np.where(anyh, hit.argmax(1), hb); fi = np.minimum(first, hb - 1); rw = np.arange(len(ee))
            fill = np.minimum(stop[rw, fi], Om[rw, fi]); ex = np.where(anyh, fill, Cm[:, -1]); nn = np.where(anyh, first + 1, hb).astype(float)
            if ts is not None:
                tb = int(ts * bpd) - 1; tx = (first > tb) & (Cm[:, tb] <= xe)
                ex = np.where(tx, Cm[:, tb], ex); nn = np.where(tx, tb + 1, nn)
            out_x[a:a + 4000] = ex; out_n[a:a + 4000] = nn
        return out_x, out_n
    KO_VAR = {"stop5": (5, None, None), "stop10": (10, None, None), "stop20": (20, None, None), "be2": (10, 0.02, None), "be3": (10, 0.03, None), "ts3": (10, None, 3), "ts5": (10, None, 5), "be2ts5": (10, 0.02, 5)}
    # unit check: a long entered at 100 with the stop at 98 that trades down to 97.5 is closed at 98 (-1 R), one that drifts up is closed at the horizon close, break-even turns a retrace into 0 R
    _n = 40; _fx = pd.DataFrame(dict(H=np.r_[np.full(10, 100.5), np.linspace(101, 110, 30)], L=np.r_[np.full(10, 99.5), np.linspace(100.5, 109.5, 30)], C=np.r_[np.full(10, 100.0), np.linspace(101, 110, 30)], O=np.r_[np.full(10, 100.0), np.linspace(101, 110, 30)]))
    _fx.loc[12, "L"] = 97.5; KO_BPD_SAVE = dict(KO_BPD); KO_BPD["D"] = 1
    _x1, _ = ko_sim(_fx, np.array([5]), np.array([98.0]), np.array([2.0]), 10, FR="D"); assert abs(_x1[0] - 98.0) < 1e-9, "stop must fill at the stop"
    _fy = _fx.copy(); _fy.loc[12, "L"] = 99.9; _x2, _ = ko_sim(_fy, np.array([5]), np.array([98.0]), np.array([2.0]), 10, FR="D"); assert abs(_x2[0] - _fy["C"].iloc[16]) < 1e-9, "no stop: exit at the horizon close"
    _fz = _fx.copy(); _fz.loc[12, "L"] = 99.9; _fz.loc[7, "H"] = 103.0; _fz.loc[8, "L"] = 99.0; _x3, _ = ko_sim(_fz, np.array([5]), np.array([98.0]), np.array([2.0]), 10, be=0.02, FR="D"); assert abs(_x3[0] - 100.0) < 1e-9, "break-even must exit at the entry"
    KO_BPD.update(KO_BPD_SAVE); print("unit check passed: stop fill, horizon exit and break-even exit")

In [ ]:
if KO_OK and KO_B:
    # ---- 18o.6 event tables: breakouts (with and without Stage 2, with the contraction state), liquidity sweeps that reclaim, high-volume-node penetrations ----
    def ko_hvn(f, nb=40, win=20, topq=0.8):
        # level crossed by this bar (x-space, upwards), from the volume profile of the previous 20 trading days only; NaN if none
        tdi = f["tdi"].to_numpy(); C = f["C"].to_numpy(float); V = np.nan_to_num(f["volume"].to_numpy(float)); n = len(f); lvl = np.full(n, np.nan)
        days = np.unique(tdi); st = np.searchsorted(tdi, days, "left"); en = np.searchsorted(tdi, days, "right")
        for k in range(win, len(days)):
            w0, w1 = st[k - win], st[k]; x = C[w0:w1]; ok = np.isfinite(x)
            if ok.sum() < 50: continue
            hist, ed = np.histogram(x[ok], bins=nb, weights=V[w0:w1][ok]); thr = np.quantile(hist, topq); lv = ((ed[:-1] + ed[1:]) / 2)[(hist >= thr) & (hist > 0)]
            if not len(lv): continue
            a, b = st[k] + 1, en[k]
            if b <= a: continue
            cp, cc = C[a - 1:b - 1], C[a:b]; cross = (cp[:, None] < lv[None, :]) & (cc[:, None] >= lv[None, :]); anyc = cross.any(1)
            lvl[a:b] = np.where(anyc, np.where(cross, lv[None, :], -np.inf).max(1), np.nan)
        return lvl
    # unit check: 25 days of flat trade at 10 build one heavy node at 10; a later bar that closes from 9.9 to 10.1 crosses it, and the level is known from earlier days only
    _px = np.r_[np.tile([9.9, 9.95, 10.0, 10.0], 25), [9.9, 10.1, 10.1, 10.1]]; _hv = pd.DataFrame(dict(tdi=np.repeat(np.arange(26), 4), C=_px, volume=1.0)); _hl = ko_hvn(_hv)
    assert np.isfinite(_hl[-3]) and abs(_hl[-3] - 10.0) < 0.1 and np.isfinite(_hl[-3:]).sum() == 1, "node crossing wrong"
    _hv2 = _hv.copy(); _hv2.loc[len(_hv2) - 1, "volume"] = 1e9; assert np.allclose(np.nan_to_num(ko_hvn(_hv2)[-4:-1]), np.nan_to_num(_hl[-4:-1])), "level must not use the current day"
    print("unit check passed: node level comes from previous days, crossing detected")
    def ko_range_ratio(H, L, w):
        n = len(H); fh = pd.Series(H[::-1]).rolling(w, min_periods=w).max().to_numpy()[::-1]; fl = pd.Series(L[::-1]).rolling(w, min_periods=w).min().to_numpy()[::-1]
        fwd_rng = np.r_[(fh - fl)[1:], np.nan]; bk = (pd.Series(H).rolling(w, min_periods=w).max() - pd.Series(L).rolling(w, min_periods=w).min()).to_numpy()
        with np.errstate(all="ignore"): return np.log(fwd_rng / bk)
    def ko_events(FR):
        bpd = KO_BPD[FR]; hmax = int(max(KO_HDAYS) * bpd); out = {"BRK": [], "SWEEP": [], "HVN": []}
        for pk_, (p, d) in enumerate(KO_B[FR].items()):
            n = len(d); dts = KO_T0 + pd.to_timedelta(d["tdi"].to_numpy(), unit="D"); sp = KN_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); vr = KO_REG[p].reindex(dts).fillna(-1).to_numpy().astype(int); tr = RG["T"].reindex(dts).fillna(-1).to_numpy().astype(int)
            for s in (1, -1):
                f = ko_dir(d, s, FR); H, L, C, O = (f[k].to_numpy(float) for k in ("H", "L", "C", "O")); sc = f["sc"].to_numpy(float); S2 = KO_S2.get((p, s))
                if S2 is None: continue
                if FR == "D": M = S2.reindex(f.index)
                else:
                    S_ = S2.shift(1); S_.index = (S_.index - KO_T0).days; M = S_[~S_.index.duplicated()].reindex(f["tdi"].to_numpy())
                s2 = M["s2"].to_numpy(float); d52 = M["d52"].to_numpy(float); up52 = M["up52"].to_numpy(float); ext = M["ext"].to_numpy(float); rb = f["rollb"].to_numpy(float) > 0
                ok0 = (sp >= 0) & (vr >= 0) & (tr >= 0) & np.isfinite(sc) & ~rb & ~np.r_[False, rb[:-1]] & (np.arange(n) + 2 + hmax < n) & (np.arange(n) >= 1)
                rr = {w: ko_range_ratio(H, L, w * bpd) for w in (10,)}; low10 = pd.Series(L).rolling(10, min_periods=1).min().to_numpy()
                def push(name, L_, mask, stop_fn, extra):
                    ix = np.where(mask & ok0)[0]
                    if not len(ix): return
                    xs = stop_fn(ix); e = ix + 1; xe = C[e]; valid = np.isfinite(xs) & (xe > xs) & np.isfinite(xe)
                    ix, xs, e, xe = ix[valid], xs[valid], e[valid], xe[valid]
                    if not len(ix): return
                    risk = np.maximum(xe - xs, sc[ix]); dd = pd.DataFrame({k: v[ix] for k, v in extra.items()}); dd["risk_bp"] = risk * 1e4
                    for vn, (hd, be, ts) in KO_VAR.items():
                        xx, nn = ko_sim(f, ix, xs, risk, hd, be, ts, FR); dd["R_" + vn] = (xx - xe) / risk; dd["B_" + vn] = (xx - xe) * 1e4; dd["N_" + vn] = nn
                    for hd in KO_HDAYS: dd[f"ret{hd}"] = (C[e + hd * bpd] - xe) * 1e4
                    dd["rx"] = rr[10][ix]; dd["spl"] = sp[ix]; dd["vol"] = vr[ix]; dd["trend"] = tr[ix]; dd["tdi"] = d["tdi"].to_numpy()[ix]; dd["pid"] = pk_; dd["dir"] = s; dd["L"] = L_; out[name].append(dd)
                base = dict(vz=f["vz"].to_numpy(float), doiz=f["doiz"].to_numpy(float), tfi=f["tfi"].to_numpy(float), flip=f["flip"].to_numpy(float), sq=f["sq"].to_numpy(float), exp=f["exp"].to_numpy(float), s2=s2, d52=d52, up52=up52, ext=ext)
                for L_ in KO_L[FR]:
                    pm = pd.Series(H).rolling(L_, min_periods=L_).max().shift(1).to_numpy(); brk = C > pm; first = brk & ~np.r_[False, brk[:-1]]
                    nc = f["nc"].to_numpy(float); Lx = f["Lx"].to_numpy(float); Hb = f["Hb"].to_numpy(float); vcp = (nc >= 2) & (C > Hb) & np.isfinite(Lx) & (Lx < C)
                    ex = dict(base); ex.update(vcp=vcp.astype(float), nc=nc, nc3=(nc >= 3).astype(float), tight=f["tight"].to_numpy(float), dry=f["dry"].to_numpy(float), dnorm=f["dlast"].to_numpy(float) / sc)
                    sw = np.where(np.isfinite(Lx) & (Lx < C), Lx, low10)
                    push("BRK", L_, first, lambda ix: sw[ix], ex)
                    pl_ = pd.Series(L).rolling(L_, min_periods=L_).min().shift(1).to_numpy(); sweep = (L < pl_) & (C > pl_); ex2 = dict(base); ex2.update(depth=(pl_ - L) / sc)
                    push("SWEEP", L_, sweep, lambda ix: L[ix], ex2)
                if FR != "D":
                    lv = ko_hvn(f); ex3 = dict(base); ex3.update(over=(C - lv) / sc)
                    push("HVN", 0, np.isfinite(lv), lambda ix: lv[ix] - sc[ix], ex3)
        res = {k: pd.concat(v, ignore_index=True) for k, v in out.items() if v}
        for k, v in res.items(): v["dayi"] = np.unique(v["tdi"], return_inverse=True)[1].astype(np.int64); v["unit"] = 1.0
        return res
    KO_E = {}
    for FR in KO_FR:
        if KO_B[FR]: t0_ = time.time(); KO_E[FR] = ko_events(FR); print(f"  {FR}: " + ", ".join(f"{k} {len(v)}" for k, v in KO_E[FR].items()) + f" event rows ({time.time() - t0_:.0f}s)")

In [ ]:
# ---- 18o.7 evaluation: parent versus child, each gate cut learned on train_R per volatility regime; the child must beat its parent in train_R, val_R and test ----
if KO_OK and KO_E:
    KO_SETS = {   # variant: (event family, parent restriction, ordered gates). kinds: hi / lo = learned tail, nat1 = the flag is 1
        "S2": ("BRK", None, [("Stage 2 (EMA150 > EMA200, rising)", "s2", "nat1")]),
        "VCP": ("BRK", "s2", [("VCP: 2+ shrinking pullbacks", "vcp", "nat1"), ("3+ pullbacks", "nc3", "nat1"), ("tight last pullback", "tight", "lo"), ("shallow last pullback", "dnorm", "lo"), ("volume dries up", "dry", "lo"),
                               ("breakout volume", "vz", "hi"), ("open interest up", "doiz", "hi"), ("near the 52-week high", "d52", "lo"), ("not extended above EMA150", "ext", "lo"), ("stop distance (hurdle)", "risk_bp", "hi"),
                               ("range squeeze before", "sq", "lo"), ("expansion bar", "exp", "hi"), ("buy-side flow", "tfi", "hi")]),
        "SWEEP": ("SWEEP", None, [("flow flips", "flip", "nat1"), ("volume", "vz", "hi"), ("open interest falls", "doiz", "lo"), ("buy-side flow", "tfi", "hi"), ("expansion bar", "exp", "hi"), ("range squeeze before", "sq", "lo"), ("stop distance (hurdle)", "risk_bp", "hi")]),
        "HVN": ("HVN", None, [("volume spike", "vz", "hi"), ("open interest up", "doiz", "hi"), ("buy-side flow", "tfi", "hi"), ("expansion bar", "exp", "hi"), ("range squeeze before", "sq", "lo"), ("stop distance (hurdle)", "risk_bp", "hi")]),
    }
    KO_MINL = 120      # smallest train_R sample for which a cut is learned (the 18n helper needs 400, too many for a rare setup such as a VCP); smaller regimes borrow the pooled cut, and below it the gate stays empty
    def ko_learn(df, feat, kind, yy):
        x = df[feat].to_numpy(float); out = {}; qs = []; unit = df["unit"].to_numpy(float); day = df["dayi"].to_numpy(); nd = int(day.max()) + 1
        def best(tr):
            if tr.sum() < KO_MINL: return None
            b = None
            for q in KO_TAILS:
                cut = np.quantile(x[tr], q if kind == "hi" else 1 - q); g = tr & ((x >= cut) if kind == "hi" else (x <= cut)); n, m, t = kn_group(yy * unit, day, g, nd)
                if n >= 30 and np.isfinite(t) and (b is None or t > b[0]): b = (t, q, cut)
            return b
        base = (df["spl"].to_numpy() == 0) & np.isfinite(x) & np.isfinite(yy); pooled = best(base)
        for v in range(3):
            b = best(base & (df["vol"].to_numpy() == v)) or pooled; out[v] = None if b is None else b[2]
            if b is not None: qs.append(b[1])
        return out, (float(np.median(qs)) if qs else np.nan)
    KO_EV, KO_Q, KO_X = [], [], []
    def ko_stats(R, B, N, rk, day, nd, m_, spl, rm_):
        row = {}
        for si, sn in enumerate(KN_SPN):
            ch = m_ & rm_ & (spl == si); n_, mm, tt = kn_group(R, day, ch, nd); row.update({f"n_{sn}": n_, f"r_{sn}": mm, f"t_{sn}": tt}); row[f"bp_{sn}"] = kn_group(B, day, ch, nd)[1]
        ch = m_ & rm_ & (spl == 2); _, row["netr_test"], row["t_netr_test"] = kn_group(R - KO_RT / rk, day, ch, nd); row["netbp_test"] = row["bp_test"] - KO_RT if np.isfinite(row["bp_test"]) else np.nan
        return row
    for FR, E_ in KO_E.items():
        for vname, (fam, pre, gates) in KO_SETS.items():
            D0 = E_.get(fam)
            if D0 is None: continue
            if vname == "HVN" and FR == "D": continue
            for L in sorted(D0["L"].unique()):
                d = D0[(D0["L"] == L) & ((D0[pre] == 1) if pre else True)].reset_index(drop=True)
                if len(d) < 400: continue
                R = d["R_stop10"].to_numpy(float); B = d["B_stop10"].to_numpy(float); rk = d["risk_bp"].to_numpy(float); day = d["dayi"].to_numpy(); nd = int(day.max()) + 1; spl = d["spl"].to_numpy()
                masks = [("parent", np.ones(len(d), bool))]; cum = np.ones(len(d), bool)
                for gname, feat, kind in gates:
                    if feat not in d: continue
                    cuts, qm = ko_learn(d, feat, kind, R); m_ = kn_apply(d, feat, kind, cuts); cum = cum & m_
                    KO_Q.append(dict(frame=FR, variant=vname, L=int(L), gate=gname, feature=feat, kind=kind, learned_tail=qm, cuts={k: (None if v_ is None else round(float(v_), 3)) for k, v_ in cuts.items()}))
                    masks.append(("+" + gname + " only", m_)); masks.append(("cumulative to " + gname, cum.copy()))
                for step, m_ in masks:
                    for rname, rcol, rv in KO_RGS:
                        rm_ = np.ones(len(d), bool) if rcol is None else (d[rcol].to_numpy() == rv); row = dict(frame=FR, variant=vname, L=int(L), step=step, regime=rname); row.update(ko_stats(R, B, None, rk, day, nd, m_, spl, rm_))
                        if step != "parent":
                            for si, sn in enumerate(KN_SPN):
                                sel = rm_ & (spl == si); dm, dt = kn_diff(R[sel], (m_ & rm_)[sel], day[sel], nd); row[f"diff_r_{sn}"] = dm; row[f"diff_t_{sn}"] = dt
                        KO_EV.append(row)
                        if rname == "all":
                            ch = m_ & (spl == 2); xr = dict(frame=FR, variant=vname, L=int(L), step=step, n_test=int(ch.sum()), win_test=float(np.nanmean(R[ch] > 0)) if ch.sum() else np.nan, hold_days=float(np.nanmean(d["N_stop10"].to_numpy(float)[ch]) / KO_BPD[FR]) if ch.sum() else np.nan)
                            for vn in KO_VAR:
                                if vn != "stop10": xr[f"r_{vn}"] = kn_group(d["R_" + vn].to_numpy(float), day, ch, nd)[1]
                            for hd in KO_HDAYS: _, xr[f"ret{hd}_bp"], xr[f"ret{hd}_t"] = kn_group(d[f"ret{hd}"].to_numpy(float), day, ch, nd)
                            _, xr["rx"], xr["rx_t"] = kn_group(d["rx"].to_numpy(float), day, ch, nd); KO_X.append(xr)
    KO_EV = pd.DataFrame(KO_EV); KO_Q = pd.DataFrame(KO_Q); KO_X = pd.DataFrame(KO_X)
    NT_KO = int((KO_EV.n_test >= KO_MINN).sum()); T_KO = bonferroni_t(max(NT_KO, 1)); c = KO_EV
    sgn_ok = (c.t_train_R > 2) & (c.t_val_R > 2) & (c.t_test > T_KO) & (c.r_test > 0)
    gate_ok = (c.step == "parent") | ((c.diff_r_train_R > 0) & (c.diff_r_val_R > 0) & (c.diff_r_test > 0) & (c.diff_t_val_R > 0) & (c.diff_t_test > 0))
    KO_EV["supported"] = sgn_ok & gate_ok & (c.n_test >= KO_MINN) & (c.n_val_R >= KO_MINN) & (c.n_train_R >= KO_MINN)
    KO_EV["be_cost_bp"] = KO_EV.bp_test; KO_EV["be_over_cost"] = KO_EV.bp_test / KO_RT; KO_EV["hurdle_train"] = KO_EV.bp_train_R / KO_RT
    KO_EV["beats_parent_all3"] = (KO_EV.step != "parent") & (KO_EV.diff_r_train_R > 0) & (KO_EV.diff_r_val_R > 0) & (KO_EV.diff_r_test > 0) & (KO_EV.n_test >= KO_MINN)
    KO_EV["clears_cost"] = KO_EV.supported & (KO_EV.netr_test > 0) & (KO_EV.netbp_test > 0)
    KO_EV.round(4).to_csv(OUT2 / "vcp_study.csv", index=False); KO_Q.to_csv(OUT2 / "vcp_learned_cuts.csv", index=False); KO_X.round(4).to_csv(OUT2 / "vcp_extras.csv", index=False)
    print(f"Stage 2 / VCP / sweep / node study: {len(KO_EV)} rows, {NT_KO} with at least {KO_MINN} test events; Bonferroni t > {T_KO:.2f}; outcome = R multiple (stop below the last contraction low, 10 days, entry one bar after the signal); round-trip cost {KO_RT:.1f} bp.")
    _sh = KO_EV[(KO_EV.regime == "all") & KO_EV.step.isin(["parent", "cumulative to VCP: 2+ shrinking pullbacks"])]
    print("Parent versus VCP (regime 'all'; R per trade; day-clustered t; diff = child minus the rest of the parent):")
    print(_sh[["frame", "variant", "L", "step", "n_test", "r_train_R", "r_val_R", "r_test", "t_test", "diff_r_train_R", "diff_r_val_R", "diff_r_test", "bp_test", "be_over_cost", "supported"]].round(3).to_string(index=False))
    print("Full ladders (regime 'all', cumulative steps):"); _lad = KO_EV[(KO_EV.regime == "all") & ((KO_EV.step == "parent") | KO_EV.step.str.startswith("cumulative"))]
    print(_lad[["frame", "variant", "L", "step", "n_test", "r_train_R", "r_val_R", "r_test", "t_test", "diff_t_test", "bp_test", "netr_test", "supported"]].round(3).to_string(index=False))
    print("Supported (positive in train_R, val_R and test, beats its parent in all three periods, t above the bar):"); _sp = KO_EV[KO_EV.supported & (KO_EV.step != "parent")]
    print(_sp[["frame", "variant", "L", "step", "regime", "n_test", "r_test", "t_test", "bp_test", "be_over_cost", "netr_test", "clears_cost"]].round(3).sort_values("t_test", ascending=False).head(30).to_string(index=False) if len(_sp) else "none")
    print("Parents that are positive themselves in all three periods:", KO_EV[(KO_EV.step == "parent") & KO_EV.supported][["frame", "variant", "L", "regime", "r_test", "t_test"]].round(3).to_dict("records") or "none")
    print(f"Extras on the test (regime 'all'): other exits, 5 / 10 / 20 day returns, range expansion:"); print(KO_X[KO_X.step.isin(["parent", "cumulative to VCP: 2+ shrinking pullbacks"])].round(3).to_string(index=False))
    print("Learned cuts (median over volatility regimes) next to the pasted suggestion (hurdle: 2.5 x round-trip cost = %.1f bp; volume spike 3x = log 3 = %.2f; TFI 0.65; 2-6 contractions):" % (KO_DOC["hurdle"] * KO_RT, np.log(KO_DOC["hvn_vol"])))
    print(KO_Q[KO_Q.kind.isin(["hi", "lo"])].groupby(["frame", "variant", "gate"])["learned_tail"].median().round(2).to_string())

In [ ]:
# ---- 18o.8 chart and summary for the findings report ----
KO_SUMMARY = None
if KO_OK and "KO_EV" in globals() and len(KO_EV):
    _a = KO_EV[(KO_EV.regime == "all") & (KO_EV.variant == "VCP") & ((KO_EV.step == "parent") | (KO_EV.step == "cumulative to VCP: 2+ shrinking pullbacks"))]
    fig, axs = plt.subplots(1, 3, figsize=(15, 4)); _lab = [f"{r.frame} L{r.L}\n{'VCP' if r.step != 'parent' else 'parent'}" for r in _a.itertuples()]; x = np.arange(len(_a))
    for k, (sn, col) in enumerate((("train_R", "#1b6ca8"), ("val_R", "#2a9d8f"), ("test", "#c0392b"))): axs[0].bar(x + (k - 1) * 0.27, _a[f"r_{sn}"].to_numpy(), 0.27, label=sn, color=col)
    axs[0].set_xticks(x); axs[0].set_xticklabels(_lab, fontsize=6); axs[0].axhline(0, color="k", lw=.8); axs[0].set_title("Stage 2 + breakout (parent) versus VCP: R per trade"); axs[0].legend(fontsize=7); axs[0].grid(alpha=.3)
    axs[1].bar(x, _a["bp_test"].to_numpy(), color="#1b6ca8"); axs[1].axhline(KO_RT, color="k", ls=":", lw=.8); axs[1].set_xticks(x); axs[1].set_xticklabels(_lab, fontsize=6); axs[1].set_title("break-even round-trip cost on the test (bp; dotted = default %.0f bp)" % KO_RT); axs[1].grid(alpha=.3)
    _l = KO_EV[(KO_EV.regime == "all") & (KO_EV.variant == "VCP") & (KO_EV.frame == KO_FR[0] if KO_FR[0] in set(KO_EV.frame) else True) & (KO_EV.step.str.startswith("cumulative") | (KO_EV.step == "parent"))]
    _l = _l[(_l.frame == _l.frame.iloc[0]) & (_l.L == _l.L.iloc[0])] if len(_l) else _l
    if len(_l):
        xx = np.arange(len(_l))
        for k, (sn, col) in enumerate((("train_R", "#1b6ca8"), ("val_R", "#2a9d8f"), ("test", "#c0392b"))): axs[2].bar(xx + (k - 1) * 0.27, _l[f"r_{sn}"].to_numpy(), 0.27, label=sn, color=col)
        axs[2].set_xticks(xx); axs[2].set_xticklabels([s_.replace("cumulative to ", "+")[:22] for s_ in _l.step], rotation=40, ha="right", fontsize=6); axs[2].axhline(0, color="k", lw=.8); axs[2].set_title(f"VCP gate ladder, {_l.frame.iloc[0]} L{_l.L.iloc[0]}: R per trade"); axs[2].legend(fontsize=7); axs[2].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(OUT2 / "18o_vcp_vs_parent.png", dpi=110); plt.show()
    _sup = KO_EV[KO_EV.supported & (KO_EV.step != "parent")]; _par = KO_EV[(KO_EV.regime == "all") & (KO_EV.step == "parent")]
    _vc = KO_EV[(KO_EV.regime == "all") & (KO_EV.step == "cumulative to VCP: 2+ shrinking pullbacks")]
    KO_SUMMARY = dict(tests=NT_KO, bonferroni_t=round(T_KO, 2), round_trip_cost_bp=KO_RT,
                      parents=_par[["frame", "variant", "L", "n_test", "r_train_R", "r_val_R", "r_test", "t_test", "bp_test", "be_over_cost", "supported"]].round(3).to_dict("records"),
                      vcp_vs_parent=_vc[["frame", "L", "n_test", "r_train_R", "r_val_R", "r_test", "t_test", "diff_r_train_R", "diff_r_val_R", "diff_r_test", "bp_test", "be_over_cost", "beats_parent_all3", "supported"]].round(3).to_dict("records"),
                      supported=_sup.sort_values("t_test", ascending=False).head(12)[["frame", "variant", "L", "step", "regime", "n_test", "r_test", "t_test", "bp_test", "be_over_cost", "netr_test", "clears_cost"]].round(3).to_dict("records"),
                      extras=KO_X[KO_X.step.isin(["parent", "cumulative to VCP: 2+ shrinking pullbacks"])].round(3).to_dict("records")[:24])
    _b3 = _vc[_vc.beats_parent_all3]
    KO_SUMMARY["reading"] = (f"Stage 2 (EMA150 > EMA200, rising) plus a breakout versus the full VCP (two or more shrinking pullbacks, stop below the last contraction low), daily / 60-minute / 30-minute: the VCP beats its parent in train_R, val_R and test in {len(_b3)} of {len(_vc)} frame x lookback cells (R per trade, regime 'all'); "
        + f"{len(_sup)} gated cells keep the same sign, beat their parent in all three periods and have t > {KO_SUMMARY['bonferroni_t']} on the test, {int(_sup.clears_cost.sum()) if len(_sup) else 0} of them clear the {KO_RT:.0f} bp round-trip cost. "
        + "Break-even cost (gross bp per trade on the test) of the VCP cells: " + "; ".join(f"{r.frame} L{r.L}: {r.bp_test:.1f} bp ({r.be_over_cost:.1f}x cost)" for r in _vc.itertuples()) + ". "
        + "Order-book items of the Taker note (depth decay, arrival rates, iceberg, empty book, IOC, latency) cannot be tested from bars.")
    print("\n" + KO_SUMMARY["reading"])
for _k, _v in list(_KO_SAVED.items()):
    if _k in ("_KO_SAVED",) or _k.startswith(("KO_", "ko_")): continue
    if globals().get(_k) is not _v: globals()[_k] = _v

## Step 18p — Incremental volume over its EMA: what it does to returns, open interest, breakouts and reversals

**Definition.** For every bar, incremental volume = volume ÷ EMA(volume over the previous *n* bars) − 1, where the EMA excludes the bar itself (so it is known at the bar's close). The reading used is the sum over the last **5 stamps** of the same frame, positive when volume has been running above its recent average and negative when below. The window is rebuilt at each frame: 5 bars of 5 minutes (25 min), 15 minutes (75 min), 30 minutes, 60 minutes, and 5 days. The EMA span *n* is searched from 2 to 13 on **train_R only**, by the size of the difference in outcome between positive and negative incremental volume; val_R must confirm the sign; the test is read once. The curve of t against *n* is printed, so a plateau can be told from a fitted peak, and a span at the edge of the grid is flagged.

**Two volume types.** *Raw* is the plain definition above. At intraday frames raw volume carries the time-of-day pattern (the opening bars always have the most), so "positive incremental volume" mostly means "near the open". *Adjusted* divides each bar by the median of the same slot over the previous 20 days before the EMA is taken. Only the adjusted series (and the daily series) are counted in the headline; raw intraday rows are shown to expose the artifact.

**Outcomes** (entry one bar after the signal; windows never cross a session break):

| Test | Outcome after positive minus negative incremental volume |
|---|---|
| cont1, cont5 | continuation of the 5-bar move over the next 1 and 5 bars (ex-ante volatility units) |
| vol | log of the next 5 bars' range over the last 5 bars' range |
| brk1, brk5, brkfail | breakout events (first close beyond the prior L-bar extreme, both directions): continuation, and the share that close back through the level within 5 bars |
| rev1, rev5, rev5_q95 | reversal after an extended move (5-bar move in the top 10 % or 5 % of the training period, per volatility regime) |

**Also reported at the chosen span:** the six regime layers; a tradeable reading (state of incremental volume and follow/fade picked on train_R, break-even against the round-trip cost of 2 × `DEFAULT_COST_BPS`); response by quintile of incremental volume; rank correlation with the open-interest change and with the size of the move; and the 2 × 2 × 2 tape (price up/down × OI up/down × volume up/down). Support rule: same sign as train_R in val_R with t > 2 and above the Bonferroni bar on the test, at least 100 events in each period.

*Testing so far was on synthetic data only: the code runs and its unit and causality checks pass; no result from real data has been seen.*

In [ ]:
# step 18p
_KP_SAVED = dict(globals())          # snapshot: loop variables of this step must not overwrite a notebook name (restored at the end of the step)
KP_OK = bool(globals().get("KO_OK", False)) and bool(globals().get("KO_B")) and all(k in globals() for k in ("kn_group", "kn_diff", "KO_REG", "KO_RGS", "KO_T0", "KN_SPLR", "KN_SPN", "KN_VN", "RG", "KC_RAW5", "KC_PRODS", "session_day", "bonferroni_t", "causal_test", "ko_base"))
if not KP_OK:
    print("Step 18p needs Step 18o (frames and helpers) and the cached 5-minute bars: run them first; skipped")
else:
    # ---- 18p.0 settings. Incremental volume of a bar = volume / EMA(volume over the previous n bars) - 1 (the EMA excludes the bar itself); the 5-stamp reading is the sum over the last 5 bars of the same frame ----
    KP_FR = ("5m", "15m", "30m", "60m", "D"); KP_MIN = {"5m": 5, "15m": 15, "30m": 30, "60m": 60, "D": None}
    KP_N = tuple(range(2, 14)); KP_W = 5; KP_H = (1, 5); KP_CLIP = 5.0                 # EMA span searched on train_R only, 2..13; window of 5 stamps; forward horizons in bars (after a one-bar delay)
    KP_BL = {"5m": 36, "15m": 24, "30m": 24, "60m": 16, "D": 20}                         # breakout: first close beyond the prior L-bar extreme
    KP_EXTQ = (0.90, 0.95)                                                               # reversal events: |5-bar move| above this quantile of the training period (per volatility regime)
    KP_MINN = 100; KP_VT = {"5m": ("raw", "adj"), "15m": ("raw", "adj"), "30m": ("raw", "adj"), "60m": ("raw", "adj"), "D": ("raw",)}
    KP_DOC = dict(ema=(13,), stamps=5)

    def kp_civ(V, n, W=KP_W):
        # incremental volume against the EMA of the previous n bars, summed over the last W bars; point in time (bar t uses volume up to t and the EMA up to t-1)
        s = pd.Series(V); ema = s.ewm(span=n, adjust=False, min_periods=n).mean().shift(1)
        iv = (s / ema.where(ema > 0) - 1.0).clip(-1.0, KP_CLIP); return iv.rolling(W, min_periods=W).sum().to_numpy()
    # unit check: hand-computed EMA and window sum
    _v = np.array([1.0, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1])
    _chk = kp_civ(_v, 2); _m = pd.Series(_v).ewm(span=2, adjust=False, min_periods=2).mean().shift(1).to_numpy(); _man = pd.Series(_v / _m - 1.0).rolling(5, min_periods=5).sum().to_numpy()
    assert np.allclose(_chk, _man, equal_nan=True) and np.isnan(_chk[:6]).all() and _chk[6] > 1.9 and _chk[7] > 1.0, "incremental volume wrong"
    _chk2 = kp_civ(_v[:9], 2); assert np.allclose(_chk2, _chk[:9], equal_nan=True), "incremental volume must not use later bars"
    print("unit check passed: incremental volume over the EMA is point in time; a volume spike lifts the 5-stamp sum for the next 5 bars")

    # ---- 18p.1 frames. 30m / 60m / D come from Step 18o; 5m / 15m are built the same way from the cached 5-minute bars ----
    def kp_base(p, FR):
        if FR in KO_B and p in KO_B[FR]: return KO_B[FR][p]
        if FR == "D": return None
        g = KC_RAW5[KC_RAW5["product"] == p].set_index("date").sort_index(); g = g[~g.index.duplicated()]
        cl = g["cl"].astype(float); off = g["ret"].astype(float).fillna(0).cumsum() - np.log(cl); opn = g["op"].astype(float) if "op" in g else cl * np.exp(-g["ret"].astype(float))
        d = pd.DataFrame(dict(op=np.log(opn) + off, hi=np.log(g["hi"].astype(float)) + off, lo=np.log(g["lo"].astype(float)) + off, cl=np.log(cl) + off, volume=g["volume"].astype(float),
                              oi=g["oi"].astype(float) if "oi" in g else np.nan, rollb=g["rollb"].astype(float) if "rollb" in g else 0.0)).replace([np.inf, -np.inf], np.nan)
        if KP_MIN[FR] != 5:
            r = d.resample(f"{KP_MIN[FR]}min", label="right", closed="right"); d = pd.DataFrame(dict(op=r["op"].first(), hi=r["hi"].max(), lo=r["lo"].min(), cl=r["cl"].last(), volume=r["volume"].sum(min_count=1), oi=r["oi"].last(), rollb=r["rollb"].max()))
        d = d.dropna(subset=["cl", "hi", "lo", "op"]); d["slot"] = (d.index.hour * 60 + d.index.minute).to_numpy(); d["tdi"] = (pd.DatetimeIndex(session_day(d.index)) - KO_T0).days.to_numpy(); return d
    KP_B = {FR: {} for FR in KP_FR}
    for FR in KP_FR:
        prods = KC_PRODS if FR != "D" else list(KO_B["D"])
        for p in prods:
            try: d_ = kp_base(p, FR)
            except Exception as e_: print(f"  {FR} {p}: skipped ({type(e_).__name__}: {str(e_)[:60]})"); continue
            if d_ is not None and len(d_) > 400: KP_B[FR][p] = d_
    print({FR: len(v) for FR, v in KP_B.items()}, "products per frame")

    def kp_vol(d, FR, vt):
        V = d["volume"].to_numpy(float)
        if vt == "raw" or FR == "D": return V
        med = pd.Series(V).groupby(d["slot"].to_numpy()).transform(lambda x: x.rolling(20, min_periods=10).median().shift(1)).to_numpy()      # seasonal adjustment: ratio to the same slot's median of the previous 20 days
        return np.where(med > 0, V / med, np.nan)
    def kp_pit(g, FR="30m"):
        out = {}
        for vt in ("raw", "adj"):
            Vx = kp_vol(g, FR, vt)
            for n in (2, 5, 13): out[f"{vt}{n}"] = kp_civ(Vx, n)
        return pd.DataFrame(out, index=g.index)
    _p0 = list(KP_B["30m"])[0] if KP_B["30m"] else None
    if _p0: causal_test("kp_incremental_volume_30m", kp_pit, KP_B["30m"][_p0].iloc[-4000:], step="18p")

In [ ]:
# ---- 18p.2 panel: per bar the 5-stamp state of price, open interest and breakouts, and the outcomes that follow (entry one bar after the signal) ----
if KP_OK:
    def kp_prod(d, FR, pk):
        n = len(d); C = d["cl"].to_numpy(float); H = d["hi"].to_numpy(float); L = d["lo"].to_numpy(float); OI = d["oi"].to_numpy(float); tdi = d["tdi"].to_numpy(); rb = d["rollb"].fillna(0).to_numpy() > 0
        intra = FR != "D"
        def same(k):          # bar t-k lies in the same trading day as bar t (daily bars: always)
            a = np.zeros(n, bool)
            if k < n: a[k:] = (tdi[k:] == tdi[:-k]) if intra else True
            return a
        def samef(k):         # bar t+k lies in the same trading day as bar t
            a = np.zeros(n, bool)
            if k < n: a[:-k] = (tdi[k:] == tdi[:-k]) if intra else True
            return a
        r1 = pd.Series(C).diff(); sc = r1.rolling(60, min_periods=30).std().shift(1).to_numpy()
        r5 = np.full(n, np.nan); r5[5:] = C[5:] - C[:-5]; v5 = same(5) & np.isfinite(r5) & np.isfinite(sc)
        oi5 = np.full(n, np.nan); oi5[5:] = (OI[5:] - OI[:-5]) / np.where(OI[:-5] > 0, OI[:-5], np.nan)
        rbw = pd.Series(rb.astype(float)).rolling(6, min_periods=1).max().to_numpy() > 0; oi5 = np.where(rbw, np.nan, oi5)
        y = {}
        for h in KP_H:
            ye = np.full(n, np.nan); ok = samef(1 + h)
            ix = np.where(ok)[0]; ix = ix[ix + 1 + h < n]; ye[ix] = C[ix + 1 + h] - C[ix + 1]
            y[h] = ye
        Hs = pd.Series(H); Ls = pd.Series(L)
        bmax = Hs.rolling(5, min_periods=5).max().to_numpy(); bmin = Ls.rolling(5, min_periods=5).min().to_numpy()
        fmax = Hs[::-1].rolling(5, min_periods=5).max()[::-1].shift(-1).to_numpy(); fmin = Ls[::-1].rolling(5, min_periods=5).min()[::-1].shift(-1).to_numpy()      # causal-ok: forward range label
        with np.errstate(all="ignore"): lrr = np.where(samef(5), np.log((fmax - fmin) / (bmax - bmin)), np.nan)
        BL = KP_BL[FR]; lu = Hs.rolling(BL, min_periods=BL).max().shift(1).to_numpy(); ld = Ls.rolling(BL, min_periods=BL).min().shift(1).to_numpy()
        up = C > lu; dn = C < ld; upf = up & ~np.r_[False, up[:-1]]; dnf = dn & ~np.r_[False, dn[:-1]]; bo = np.where(upf, 1, np.where(dnf, -1, 0)); blv = np.where(bo == 1, lu, ld)
        cmin = pd.Series(C)[::-1].rolling(5, min_periods=5).min()[::-1].shift(-1).to_numpy(); cmax = pd.Series(C)[::-1].rolling(5, min_periods=5).max()[::-1].shift(-1).to_numpy()      # causal-ok: forward label (does the breakout hold)
        fail = np.where(samef(5), np.where(bo == 1, (cmin < blv).astype(float), np.where(bo == -1, (cmax > blv).astype(float), np.nan)), np.nan)
        dts = KO_T0 + pd.to_timedelta(tdi, unit="D"); sp = KN_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int)
        return dict(n=n, r5=r5, sc=sc, v5=v5, oi5=oi5, y=y, lrr=lrr, bo=bo, fail=fail, tdi=tdi, sp=sp)
    KP_P = {}
    for FR in KP_FR:
        if not KP_B[FR]: continue
        parts = {k: [] for k in ("r5", "sc", "oi5", "y1", "y5", "lrr", "bo", "fail", "tdi", "spl", "vol", "trend", "pid")}; civs = {(vt, n): [] for vt in KP_VT[FR] for n in KP_N}
        for pk_, (p, d) in enumerate(KP_B[FR].items()):
            q = kp_prod(d, FR, p); dts = KO_T0 + pd.to_timedelta(q["tdi"], unit="D"); vr = KO_REG[p].reindex(dts).fillna(-1).to_numpy().astype(int); tr = RG["T"].reindex(dts).fillna(-1).to_numpy().astype(int)
            keep = q["v5"] & (q["sp"] >= 0) & (vr >= 0) & (tr >= 0)
            if keep.sum() < 300: continue
            for k, v in (("r5", q["r5"]), ("sc", q["sc"]), ("oi5", q["oi5"]), ("y1", q["y"][1]), ("y5", q["y"][5]), ("lrr", q["lrr"]), ("bo", q["bo"]), ("fail", q["fail"]), ("tdi", q["tdi"]), ("spl", q["sp"]), ("vol", vr), ("trend", tr)): parts[k].append(v[keep])
            parts["pid"].append(np.full(int(keep.sum()), pk_))
            for vt in KP_VT[FR]:
                Vx = kp_vol(d, FR, vt)
                for n in KP_N: civs[(vt, n)].append(kp_civ(Vx, n)[keep].astype(np.float32))
        if not parts["r5"]: continue
        P = {k: np.concatenate(v) for k, v in parts.items()}; P["civ"] = {k: np.concatenate(v) for k, v in civs.items()}; P["dayi"] = np.unique(P["tdi"], return_inverse=True)[1].astype(np.int64); P["nd"] = int(P["dayi"].max()) + 1
        sc_ = P["sc"]; P["z5"] = P["r5"] / (sc_ * np.sqrt(5)); P["sg"] = np.sign(P["r5"])
        for h in KP_H: P[f"yz{h}"] = P[f"y{h}"] / (sc_ * np.sqrt(h)); P[f"yb{h}"] = P[f"y{h}"] * 1e4
        az = np.abs(P["z5"]); P["ext"] = {}
        for qq in KP_EXTQ:
            thr = np.array([np.nanquantile(az[(P["spl"] == 0) & (P["vol"] == v) & np.isfinite(az)], qq) if ((P["spl"] == 0) & (P["vol"] == v)).sum() > 200 else np.nan for v in range(3)]); P["ext"][qq] = np.isfinite(az) & (az >= thr[P["vol"]])
        KP_P[FR] = P
    print({FR: len(P["r5"]) for FR, P in KP_P.items()}, "bars with a complete 5-stamp state, per frame")

In [ ]:
# ---- 18p.3 how does incremental volume (positive or negative, 5 stamps, EMA span 2..13) relate to what follows? Span chosen on train_R only; val_R confirms; the test is read once ----
if KP_OK and KP_P:
    KP_TESTS = {"cont1": "continuation of the 5-bar move, next bar", "cont5": "continuation of the 5-bar move, next 5 bars", "vol": "range of the next 5 bars over the range of the last 5 (log)", "brk1": "breakout continuation, next bar", "brk5": "breakout continuation, next 5 bars",
                "brkfail": "breakout fails (closes back through the level within 5 bars)", "rev1": "reversal after an extended move (top 10%), next bar", "rev5": "reversal after an extended move (top 10%), next 5 bars", "rev5_q95": "reversal after an extended move (top 5%), next 5 bars"}
    def kp_defs(P):
        sg = P["sg"]; allr = np.ones(len(sg), bool); bo = P["bo"]; br = bo != 0
        return {"cont1": (P["yz1"] * sg, P["yb1"] * sg, allr), "cont5": (P["yz5"] * sg, P["yb5"] * sg, allr), "vol": (P["lrr"], P["lrr"], allr), "brk1": (P["yz1"] * bo, P["yb1"] * bo, br), "brk5": (P["yz5"] * bo, P["yb5"] * bo, br),
                "brkfail": (P["fail"], P["fail"], br), "rev1": (-P["yz1"] * sg, -P["yb1"] * sg, P["ext"][KP_EXTQ[0]]), "rev5": (-P["yz5"] * sg, -P["yb5"] * sg, P["ext"][KP_EXTQ[0]]), "rev5_q95": (-P["yz5"] * sg, -P["yb5"] * sg, P["ext"][KP_EXTQ[1]])}
    def kp_diffs(Y, civ, rows, P, rm=None):
        out = []
        for si in range(3):
            sel = rows & (P["spl"] == si) & np.isfinite(Y) & np.isfinite(civ) & (civ != 0)
            if rm is not None: sel = sel & rm
            dm, dt = kn_diff(Y[sel], (civ > 0)[sel], P["dayi"][sel], P["nd"]); out.append((dm, dt, int(sel.sum())))
        return out
    KP_CURVE, KP_SEL, KP_DEF = [], [], {}
    for FR, P in KP_P.items():
        KP_DEF[FR] = kp_defs(P)
        for vt in KP_VT[FR]:
            for tname, (Y, Yb, rows) in KP_DEF[FR].items():
                rec = {}
                for n in KP_N:
                    r = kp_diffs(Y, P["civ"][(vt, n)], rows, P); rec[n] = r
                    for si, sn in enumerate(KN_SPN): KP_CURVE.append(dict(frame=FR, vtype=vt, test=tname, n=n, split=sn, diff=r[si][0], t=r[si][1], rows=r[si][2]))
                tt = np.array([rec[n][0][1] for n in KP_N], float)
                if not np.isfinite(tt).any(): continue
                k = int(np.nanargmax(np.abs(tt))); ns = KP_N[k]; sgn = np.sign(tt[k]); r = rec[ns]
                KP_SEL.append(dict(frame=FR, vtype=vt, test=tname, n_star=ns, sign=sgn, t_train=r[0][1], t_val=r[1][1], t_test=r[2][1], diff_train=r[0][0], diff_val=r[1][0], diff_test=r[2][0], n_test=r[2][2], n_val=r[1][2], n_train=r[0][2],
                                   at_edge=ns in (KP_N[0], KP_N[-1]), curve_peak_vs_median=float(np.nanmax(np.abs(tt)) - np.nanmedian(np.abs(tt)))))
    KP_CURVE = pd.DataFrame(KP_CURVE); KP_SEL = pd.DataFrame(KP_SEL)
    NT_KP = int((KP_SEL.n_test >= KP_MINN).sum()); T_KP = bonferroni_t(max(NT_KP, 1)); c = KP_SEL
    KP_SEL["supported"] = (c.n_train >= KP_MINN) & (c.n_val >= KP_MINN) & (c.n_test >= KP_MINN) & (c.t_train * c.sign > 2) & (c.t_val * c.sign > 2) & (c.t_test * c.sign > T_KP)
    KP_SEL["headline"] = (KP_SEL.vtype == "adj") | (KP_SEL.frame == "D")      # raw intraday volume is dominated by the time-of-day pattern (the opening bars always have the highest volume): listed, never counted
    KP_SEL.round(4).to_csv(OUT2 / "incremental_volume_selection.csv", index=False); KP_CURVE.round(4).to_csv(OUT2 / "incremental_volume_curve.csv", index=False)
    print(f"Incremental volume (volume / EMA of the previous n bars - 1, summed over 5 stamps): {len(KP_SEL)} frame x volume-type x outcome rules; span n = 2..13 picked on train_R by |t| of (positive minus negative incremental volume); {NT_KP} rules with enough test events, Bonferroni t > {T_KP:.2f} on the test.")
    print("diff = mean outcome after positive incremental volume minus after negative (units of the bar's ex-ante volatility; for 'vol' log range ratio; for 'brkfail' share of failures). Chosen span and t by period:")
    print(KP_SEL[["frame", "vtype", "test", "n_star", "at_edge", "t_train", "t_val", "t_test", "diff_test", "n_test", "supported"]].round(2).to_string(index=False))
    print("Supported (same sign as train_R in val_R with t > 2 and in the test above the bar; headline = seasonally adjusted volume or daily, raw intraday rows are the time-of-day artifact):"); print(KP_SEL[KP_SEL.supported][["frame", "vtype", "test", "n_star", "headline", "t_train", "t_val", "t_test", "diff_test"]].round(2).to_string(index=False) if KP_SEL.supported.any() else "none")
    # span sensitivity: how flat is the train t over n? (a sharp peak at one span is a fitted number; a plateau is a property)
    _c = KP_CURVE[(KP_CURVE.split == "train_R") & KP_CURVE.test.isin(["cont5", "vol", "brk5", "rev5"])].pivot_table(index=["frame", "vtype", "test"], columns="n", values="t"); print("train_R t by EMA span (cont5 / vol / brk5 / rev5):"); print(_c.round(1).to_string())

    # ---- 18p.4 at the chosen span: regimes, tradeable rule (state and direction picked on train_R), response by size of incremental volume, open interest ----
    KP_REG, KP_RULE, KP_BINS, KP_TAPE, KP_COR = [], [], [], [], []
    for r_ in KP_SEL.itertuples():
        FR, vt, tn, ns = r_.frame, r_.vtype, r_.test, r_.n_star; P = KP_P[FR]; Y, Yb, rows = KP_DEF[FR][tn]; civ = P["civ"][(vt, ns)]
        for rname, rcol, rv in KO_RGS:
            rm = None if rcol is None else (P[rcol] == rv); d3 = kp_diffs(Y, civ, rows, P, rm)
            KP_REG.append(dict(frame=FR, vtype=vt, test=tn, n_star=ns, regime=rname, **{f"diff_{sn}": d3[i][0] for i, sn in enumerate(KN_SPN)}, **{f"t_{sn}": d3[i][1] for i, sn in enumerate(KN_SPN)}, n_test=d3[2][2]))
        if tn in ("vol", "brkfail"): continue
        best = None
        for stn, scm in (("positive", civ > 0), ("negative", civ < 0)):
            for dn, dd in (("follow", 1.0), ("fade", -1.0)):
                n0, m0, t0 = kn_group(dd * Y, P["dayi"], rows & scm & (P["spl"] == 0), P["nd"])
                if np.isfinite(t0) and (best is None or t0 > best[0]): best = (t0, stn, dn, dd, scm)
        if best is None: continue
        _, stn, dn, dd, scm = best; row = dict(frame=FR, vtype=vt, test=tn, n_star=ns, state=stn, direction=dn)
        for si, sn in enumerate(KN_SPN):
            nn_, mm, tt = kn_group(dd * Y, P["dayi"], rows & scm & (P["spl"] == si), P["nd"]); row.update({f"n_{sn}": nn_, f"z_{sn}": mm, f"t_{sn}": tt}); row[f"bp_{sn}"] = kn_group(dd * Yb, P["dayi"], rows & scm & (P["spl"] == si), P["nd"])[1]
        KP_RULE.append(row)
    KP_REG = pd.DataFrame(KP_REG); KP_RULE = pd.DataFrame(KP_RULE); c = KP_RULE
    NT_RULE = int((c.n_test >= KP_MINN).sum()); T_RULE = bonferroni_t(max(NT_RULE, 1))
    KP_RULE["supported"] = (c.n_train_R >= KP_MINN) & (c.n_val_R >= KP_MINN) & (c.n_test >= KP_MINN) & (c.t_train_R > 2) & (c.t_val_R > 2) & (c.t_test > T_RULE) & (c.bp_test > 0)
    KP_RULE["net_test_bp"] = KP_RULE.bp_test - KO_RT; KP_RULE["clears_cost"] = KP_RULE.supported & (KP_RULE.net_test_bp > 0); KP_RULE["be_over_cost"] = KP_RULE.bp_test / KO_RT
    KP_REG.round(4).to_csv(OUT2 / "incremental_volume_regimes.csv", index=False); KP_RULE.round(4).to_csv(OUT2 / "incremental_volume_rules.csv", index=False)
    print(f"\nTradeable reading (state of incremental volume and follow / fade picked on train_R; {NT_RULE} rules, Bonferroni t > {T_RULE:.2f}; round-trip cost {KO_RT:.1f} bp):")
    print(KP_RULE[["frame", "vtype", "test", "n_star", "state", "direction", "n_test", "t_train_R", "t_val_R", "t_test", "bp_test", "be_over_cost", "supported", "clears_cost"]].round(2).to_string(index=False))
    _rg = KP_REG[KP_REG.regime != "all"].copy(); _rg["same3"] = (np.sign(_rg.diff_train_R) == np.sign(_rg.diff_val_R)) & (np.sign(_rg.diff_val_R) == np.sign(_rg.diff_test)) & (_rg.t_test.abs() > 2) & (_rg.n_test >= KP_MINN)
    print("Regime layers where the positive-minus-negative difference keeps its sign in all three periods with |t| > 2 on the test (count of 6 layers):"); print(_rg.groupby(["frame", "vtype", "test"])["same3"].sum().unstack("test").to_string())

In [ ]:
# ---- 18p.5 response by size of incremental volume, open interest and the tape; chart and summary ----
KP_SUMMARY = None
if KP_OK and KP_P and len(KP_SEL):
    KP_BINS, KP_TAPE, KP_COR = [], [], []
    _ref = KP_SEL[KP_SEL.test == "cont5"].set_index(["frame", "vtype"])["n_star"]
    for (FR, vt), ns in _ref.items():
        P = KP_P[FR]; civ = P["civ"][(vt, ns)]; Y = KP_DEF[FR]["cont5"][0]; lr = P["lrr"]; tr_ = (P["spl"] == 0) & np.isfinite(civ)
        edges = np.nanquantile(civ[tr_], np.linspace(0, 1, 6)); edges[0] = -np.inf; edges[-1] = np.inf
        for b in range(5):
            m = (civ >= edges[b]) & (civ < edges[b + 1])
            for si, sn in enumerate(KN_SPN):
                sel = m & (P["spl"] == si); _, mu, tu = kn_group(Y, P["dayi"], sel, P["nd"]); _, ml, tl = kn_group(lr, P["dayi"], sel, P["nd"])
                KP_BINS.append(dict(frame=FR, vtype=vt, n_star=ns, quintile=b + 1, edge_lo=edges[b], split=sn, cont5_z=mu, cont5_t=tu, next_range_lrr=ml, range_t=tl))
        oi = P["oi5"]
        for si, sn in enumerate(KN_SPN):
            s_ = (P["spl"] == si) & np.isfinite(civ) & np.isfinite(oi); s2_ = (P["spl"] == si) & np.isfinite(civ) & np.isfinite(P["z5"])
            KP_COR.append(dict(frame=FR, vtype=vt, n_star=ns, split=sn, rho_civ_oi_change=stats.spearmanr(civ[s_], oi[s_])[0] if s_.sum() > 50 else np.nan, rho_civ_abs_move=stats.spearmanr(civ[s2_], np.abs(P["z5"][s2_]))[0] if s2_.sum() > 50 else np.nan,
                               rho_civ_signed_oi_by_dir=stats.spearmanr(civ[s_], (oi * P["sg"])[s_])[0] if s_.sum() > 50 else np.nan))
            for pd_, pn in ((1, "price up"), (-1, "price down")):
                for od, on in ((1, "OI up"), (-1, "OI down")):
                    for cs, cn in ((1, "volume up"), (-1, "volume down")):
                        sel = (P["spl"] == si) & (P["sg"] == pd_) & (np.sign(oi) == od) & (np.sign(civ) == cs); n_, mu, tu = kn_group(Y, P["dayi"], sel, P["nd"]); KP_TAPE.append(dict(frame=FR, vtype=vt, n_star=ns, split=sn, price=pn, oi=on, volume=cn, n=n_, cont5_z=mu, t=tu))
    KP_BINS = pd.DataFrame(KP_BINS); KP_TAPE = pd.DataFrame(KP_TAPE); KP_COR = pd.DataFrame(KP_COR)
    KP_BINS.round(4).to_csv(OUT2 / "incremental_volume_bins.csv", index=False); KP_TAPE.round(4).to_csv(OUT2 / "incremental_volume_tape.csv", index=False); KP_COR.round(4).to_csv(OUT2 / "incremental_volume_correlations.csv", index=False)
    print("\nRank correlation of the 5-stamp incremental volume with the open-interest change over the same 5 stamps, with the size of the move, and with the OI change signed by the direction of the move (descriptive):")
    print(KP_COR.pivot_table(index=["frame", "vtype"], columns="split", values=["rho_civ_oi_change", "rho_civ_abs_move"]).round(3).to_string())
    print("\nContinuation of the 5-bar move (next 5 bars, ex-ante volatility units) by quintile of incremental volume (edges from train_R); test period:"); print(KP_BINS[KP_BINS.split == "test"].pivot_table(index=["frame", "vtype"], columns="quintile", values="cont5_z").round(3).to_string())
    print("Range of the next 5 bars over the last 5 (log) by quintile; test period:"); print(KP_BINS[KP_BINS.split == "test"].pivot_table(index=["frame", "vtype"], columns="quintile", values="next_range_lrr").round(3).to_string())
    _tp = KP_TAPE[KP_TAPE.split == "test"].pivot_table(index=["frame", "vtype", "price", "oi"], columns="volume", values="cont5_z"); print("Tape state x volume sign: continuation of the 5-bar move over the next 5 bars, test period:"); print(_tp.round(3).to_string())
    fig, axs = plt.subplots(1, 3, figsize=(15, 4)); _cur = KP_CURVE[(KP_CURVE.test == "cont5")]
    for FR in KP_P:
        vt = "adj" if FR != "D" else "raw"; g = _cur[(_cur.frame == FR) & (_cur.vtype == vt)]
        for sn, ax in (("train_R", axs[0]), ("val_R", axs[1])): ax.plot(g[g.split == sn].n, g[g.split == sn].t, marker="o", ms=3, label=FR)
    for ax, ttl in ((axs[0], "train_R"), (axs[1], "val_R")): ax.axhline(0, color="k", lw=.8); ax.set_title(f"continuation, positive minus negative incremental volume: t by EMA span ({ttl})"); ax.set_xlabel("EMA span n"); ax.grid(alpha=.3); ax.legend(fontsize=7)
    _hm = KP_SEL[KP_SEL.vtype == KP_SEL.vtype.map(lambda v: v)].copy(); _hm = _hm[(_hm.vtype == "adj") | (_hm.frame == "D")].pivot_table(index="test", columns="frame", values="t_test").reindex(columns=[f for f in KP_FR if f in KP_P]); im = axs[2].imshow(_hm.to_numpy(float), cmap="RdBu_r", vmin=-6, vmax=6, aspect="auto")
    axs[2].set_xticks(range(_hm.shape[1])); axs[2].set_xticklabels(_hm.columns, fontsize=7); axs[2].set_yticks(range(_hm.shape[0])); axs[2].set_yticklabels(_hm.index, fontsize=7); axs[2].set_title("test t at the chosen span (seasonally adjusted volume)")
    for i in range(_hm.shape[0]):
        for j in range(_hm.shape[1]):
            v_ = _hm.iloc[i, j]
            if np.isfinite(v_): axs[2].text(j, i, f"{v_:.1f}", ha="center", va="center", fontsize=6)
    fig.tight_layout(); fig.savefig(OUT2 / "18p_incremental_volume.png", dpi=110); plt.show()
    _sup = KP_SEL[KP_SEL.supported & KP_SEL.headline]; _art = KP_SEL[KP_SEL.supported & ~KP_SEL.headline]; _rs = KP_RULE[KP_RULE.supported & ((KP_RULE.vtype == "adj") | (KP_RULE.frame == "D"))]
    KP_SUMMARY = dict(rules=int(len(KP_SEL)), bonferroni_t=round(T_KP, 2), raw_intraday_artifacts=int(len(_art)), supported=_sup[["frame", "vtype", "test", "n_star", "t_train", "t_val", "t_test", "diff_test"]].round(3).to_dict("records"),
                      spans=KP_SEL[KP_SEL.test.isin(["cont5", "vol", "brk5", "rev5"])][["frame", "vtype", "test", "n_star", "at_edge", "t_train", "t_val", "t_test"]].round(2).to_dict("records"),
                      tradeable=_rs[["frame", "vtype", "test", "n_star", "state", "direction", "n_test", "t_test", "bp_test", "be_over_cost", "clears_cost"]].round(3).to_dict("records"),
                      correlations=KP_COR[KP_COR.split == "test"].round(3).to_dict("records"))
    _vol = _sup[_sup.test == "vol"]; _dir = _sup[_sup.test != "vol"]
    KP_SUMMARY["reading"] = (f"Incremental volume (volume over the EMA of the previous n bars, summed over 5 stamps, span 2-13 picked on train_R) at 5 / 15 / 30 / 60 minutes and daily, raw and seasonally adjusted: {len(_sup)} of {int(KP_SEL.headline.sum())} frame x outcome rules (seasonally adjusted volume, or daily) keep the sign of train_R in val_R (t > 2) and clear t > {KP_SUMMARY['bonferroni_t']} on the test; "
        + f"{len(_vol)} of them concern the size of the next move, {len(_dir)} concern direction (continuation, breakouts, reversals). Rules with a positive gross edge in all three periods: {len(_rs)}; {int(_rs.clears_cost.sum()) if len(_rs) else 0} clear the {KO_RT:.0f} bp round-trip cost. "
        + (f"{len(_art)} further rules on raw intraday volume are supported, which is the time-of-day pattern of volume (the opening bars always have the most volume), not information. " if len(_art) else "") + "Chosen spans at the edge of the 2-13 grid: " + (", ".join(f"{r.frame}/{r.vtype}/{r.test}" for r in KP_SEL[KP_SEL.at_edge].itertuples())[:240] or "none") + ".")
    print("\n" + KP_SUMMARY["reading"])
for _k, _v in list(_KP_SAVED.items()):
    if _k in ("_KP_SAVED",) or _k.startswith(("KP_", "kp_")): continue
    if globals().get(_k) is not _v: globals()[_k] = _v

## Step 18q — Per-instrument rules (breakout, climax reversal, liquidation reversal), the Return × ΔOI matrix, and what the bars before a spike look like

**Thresholds belong to the instrument.** Each rule family (BRK: first close beyond the prior L-bar extreme; REVA: climax and absorption; REVB: liquidation cascade) is gated by body-to-range, rejection wick, volume ratio, move rank and ΔOI. Every cut is run in three modes and compared: the pasted value, one pooled value, and a value learned **per instrument on train_R** (shrunk toward the pooled value with weight n/(n+400), so a thin instrument falls back to the pool). A final rule counts as supported only if it has the same sign in train_R, val_R and test, clears the Bonferroni bar on the test, and beats its parent family. A reproducibility table gives the rank correlation across instruments between the cut learned on train_R and on val_R: near zero means the per-instrument cut is noise, high means it is a property of the instrument.

**Pre-spike anatomy (nothing assumed).** A spike is a bar whose return, in ex-ante volatility units, is in the top 5 % or top 1 % of **that instrument's own training period**, in either direction. For the three bars before it: sign of the return, wick on the side the spike goes to and on the other side (share of range and multiple of the body), incremental volume (Step 18p span, seasonally adjusted), OI and ΔOI, the number of the last three bars in each of the four Return × ΔOI states, and range compression. These are compared with ordinary bars, and spikes that **continued** over the next 5 bars (one-bar delay) are compared with spikes that **reversed**. The share that continues is printed per period, so the data decides which is the rule.

**Return × ΔOI matrix.** Price up/OI up, price up/OI down, price down/OI up, price down/OI down, for any bar and for strong bars (top 20 % of |return| on train_R), with the next-bar and next-5-bar follow-through. Everything uses information up to the close of the bar; outcomes start one bar later.

In [ ]:
# step 18q
_KQ_SAVED = dict(globals())          # snapshot: loop variables of this step must not overwrite a notebook name (restored at the end of the step)
KQ_OK = bool(globals().get("KP_OK", False)) and all(k in globals() for k in ("KP_B", "kp_vol", "KO_REG", "KO_T0", "KN_SPLR", "KN_SPN", "RG", "kn_group", "kn_diff", "bonferroni_t", "causal_test", "KO_RT"))
if not KQ_OK:
    print("Step 18q needs Step 18p (frames) and the earlier candle steps: run them first; skipped")
else:
    from numpy.lib.stride_tricks import sliding_window_view as _swq
    # ---- 18q.0 settings. The pasted rules are the starting point and the comparison; every cut is learned on train_R, pooled over instruments and per instrument ----
    KQ_FR = ("5m", "15m", "30m", "60m", "D"); KQ_NF = 3; KQ_TAILS = (0.5, 0.6, 0.7, 0.8, 0.9, 0.95); KQ_MINL = 150; KQ_K = 400; KQ_MINN = 100          # forward bars (pasted: 3); tails tried; smallest own-training sample; shrinkage constant
    KQ_FAM = {    # family: (what the parent is, thesis sign of the forward move in the trade direction, gates (name, feature, kind, pasted cut))
        "BRK": ("first close beyond the prior L-bar extreme", 1.0, [("move rank in the last 50 bars", "retq", "hi", 0.80), ("body ratio", "brr", "hi", 0.70), ("no resistance wick", "wup", "lo", 0.10), ("volume", "vq", "hi", 1.5), ("open interest rises", "doi", "pos", 0.0), ("open-interest size", "doirk", "hi", None)]),
        "REVA": ("new L-bar extreme (top in the trade space)", -1.0, [("rejection wick", "wup", "hi", 0.60), ("small body", "brr", "lo", 0.30), ("climax volume", "vq", "hi", 2.0), ("open-interest spike", "doirk", "hi", 0.90)]),
        "REVB": ("bar against the prior 3-bar trend", 1.0, [("move rank in the last 50 bars", "retq", "hi", 0.95), ("body ratio", "brr", "hi", 0.70), ("no opposing wick", "wup", "lo", 0.15), ("panic volume", "vq", "hi", 2.5), ("open-interest plunge", "doirk", "lo", 0.10)])}
    KQ_FEATS = ("retq", "brr", "wup", "vr", "vq", "doi", "doirk")

    def kq_rank(a):
        # share of the previous 50 values below this bar's value (point in time)
        n = len(a); out = np.full(n, np.nan)
        if n > 51:
            w = _swq(a[:-1], 50); cur = a[50:]; out[50:] = np.where(np.isfinite(cur) & np.isfinite(w).all(1), (w < cur[:, None]).mean(1), np.nan)
        return out
    def kq_dir(d, s, FR):
        n = len(d); tdi = d["tdi"].to_numpy()
        if s == 1: O, H, L, C = (d[k].to_numpy(float) for k in ("op", "hi", "lo", "cl"))
        else: O, H, L, C = -d["op"].to_numpy(float), -d["lo"].to_numpy(float), -d["hi"].to_numpy(float), -d["cl"].to_numpy(float)
        V = d["volume"].to_numpy(float); OI = d["oi"].to_numpy(float); rb = d["rollb"].fillna(0).to_numpy() > 0; rng = H - L; ok = rng > 0
        brr = np.where(ok, np.abs(C - O) / np.where(ok, rng, 1), np.nan); wup = np.where(ok, (H - np.maximum(O, C)) / np.where(ok, rng, 1), np.nan); ret = C - O
        vr = V / pd.Series(V).rolling(20, min_periods=20).mean().to_numpy(); vq = vr if FR == "D" else kp_vol(d, FR, "adj")
        doi = np.full(n, np.nan); doi[1:] = (OI[1:] - OI[:-1]) / np.where(OI[:-1] > 0, OI[:-1], np.nan); doi = np.where(rb | np.r_[False, rb[:-1]], np.nan, doi)
        trend3 = np.full(n, np.nan); trend3[4:] = C[3:-1] - C[:-4]
        r1 = pd.Series(C).diff(); sc = r1.rolling(60, min_periods=30).std().shift(1).to_numpy()
        Hs = pd.Series(H); BL = KP_BL[FR]; lu = Hs.rolling(BL, min_periods=BL).max().shift(1).to_numpy(); up = C > lu; first = up & ~np.r_[False, up[:-1]]; newext = H > lu
        ye = np.full(n, np.nan); ix = np.arange(n - 1 - KQ_NF); same = (tdi[ix + 1 + KQ_NF] == tdi[ix]) if FR != "D" else np.ones(len(ix), bool); ye[ix[same]] = C[ix[same] + 1 + KQ_NF] - C[ix[same] + 1]
        f = dict(retq=kq_rank(ret), brr=brr, wup=wup, vr=vr, vq=vq, doi=doi, doirk=kq_rank(doi))
        par = dict(BRK=first, REVA=newext, REVB=(ret > 0) & (trend3 < 0))
        return f, par, ye, sc
    # unit check: the rank of a bar's move among the previous 50, and its point-in-time property
    _a = np.r_[np.linspace(-1, 1, 60), 5.0, -5.0]; _rk = kq_rank(_a); assert abs(_rk[60] - 1.0) < 1e-12 and abs(_rk[61] - 0.0) < 1e-12 and np.isnan(_rk[49]), "rank wrong"; _rk2 = kq_rank(_a[:61]); assert np.allclose(_rk2, _rk[:61], equal_nan=True), "rank must not use later bars"
    print("unit check passed: move rank among the previous 50 bars is point in time")
    def kq_pit(g, FR="30m"):
        f, par, ye, sc = kq_dir(g, 1, FR); out = pd.DataFrame({k: v for k, v in f.items()}, index=g.index); out["brk"] = par["BRK"].astype(float); out["revb"] = par["REVB"].astype(float); return out
    _p0 = list(KP_B["30m"])[0] if KP_B.get("30m") else None
    if _p0: causal_test("kq_features_30m", kq_pit, KP_B["30m"][_p0].iloc[-4000:], step="18q")

    # ---- 18q.1 event tables per frame and family: features at the signal bar, the forward move after a one-bar delay, split, regimes, instrument ----
    KQ_E = {}
    for FR in KQ_FR:
        if not KP_B.get(FR): continue
        rows = {k: [] for k in KQ_FAM}
        for pk_, (p, d) in enumerate(KP_B[FR].items()):
            dts = KO_T0 + pd.to_timedelta(d["tdi"].to_numpy(), unit="D"); sp = KN_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); vr_ = KO_REG[p].reindex(dts).fillna(-1).to_numpy().astype(int); tr_ = RG["T"].reindex(dts).fillna(-1).to_numpy().astype(int)
            for s in (1, -1):
                f, par, ye, sc = kq_dir(d, s, FR)
                for fam, (_, th, _g) in KQ_FAM.items():
                    m = par[fam] & (sp >= 0) & (vr_ >= 0) & (tr_ >= 0) & np.isfinite(ye) & np.isfinite(sc) & (sc > 0)
                    ix = np.where(m)[0]
                    if not len(ix): continue
                    dd = {k: f[k][ix].astype(np.float32) for k in KQ_FEATS}; dd["yz"] = (th * ye[ix] / (sc[ix] * np.sqrt(KQ_NF))).astype(np.float32); dd["yb"] = (th * ye[ix] * 1e4).astype(np.float32)
                    dd["spl"] = sp[ix]; dd["vol"] = vr_[ix]; dd["trend"] = tr_[ix]; dd["tdi"] = d["tdi"].to_numpy()[ix]; dd["pid"] = np.full(len(ix), pk_); dd["dir"] = np.full(len(ix), s); rows[fam].append(pd.DataFrame(dd))
        KQ_E[FR] = {}
        for fam, v in rows.items():
            if not v: continue
            D = pd.concat(v, ignore_index=True); D["dayi"] = np.unique(D["tdi"], return_inverse=True)[1].astype(np.int64); KQ_E[FR][fam] = D
    KQ_PROD = {FR: list(KP_B[FR]) for FR in KQ_E}
    print({FR: {k: len(v) for k, v in d.items()} for FR, d in KQ_E.items()}, "parent events per frame and family")

In [ ]:
# ---- 18q.2 thresholds: pasted, pooled over instruments (train_R), and per instrument (train_R, shrunk toward the pooled cut); every gate is added in order and must improve the training t ----
if KQ_OK and KQ_E:
    def kq_gate_mask(x, kind, cut):
        if cut is None: return np.ones(len(x), bool)
        return np.isfinite(x) & ((x > 0) if kind == "pos" else ((x >= cut) if kind == "hi" else (x <= cut)))
    def kq_learn(D, rows, gates, minl, shrink_to=None, w=0.0):
        # sequential choice of the cut of each gate on the rows in `rows` (train_R or val_R of one instrument or of all): the tail whose cumulative selection has the largest t of the thesis outcome
        yz = D["yz"].to_numpy(float); day = D["dayi"].to_numpy(); nd = int(day.max()) + 1; cum = np.ones(len(D), bool); cuts = []; own = []
        for gi, (name, feat, kind, pasted) in enumerate(gates):
            x = D[feat].to_numpy(float); fin = np.isfinite(x)
            if kind == "pos": cut = 0.0; own.append(0.0)
            else:
                best = None; base = rows & cum & fin
                for q in KQ_TAILS:
                    if base.sum() < minl: break
                    c_ = np.quantile(x[base], q if kind == "hi" else 1 - q); g = base & ((x >= c_) if kind == "hi" else (x <= c_)); n, m, t = kn_group(yz, day, g, nd)
                    if n >= minl // 3 and np.isfinite(t) and (best is None or t > best[0]): best = (t, c_)
                o = None if best is None else float(best[1]); own.append(o)
                if shrink_to is not None:
                    pc = shrink_to[gi]
                    cut = pc if o is None else (o if pc is None else float(w * o + (1 - w) * pc))
                else: cut = o
            cuts.append(cut); cum = cum & kq_gate_mask(x, kind, cut)
        return cuts, own
    def kq_chain(D, gates, cuts, pasted=False):
        cum = np.ones(len(D), bool); steps = [("parent", cum.copy())]
        for gi, (name, feat, kind, pst) in enumerate(gates):
            if pasted:
                if pst is None: continue
                f = "vr" if feat == "vq" else feat; cum = cum & kq_gate_mask(D[f].to_numpy(float), kind, pst)
            else: cum = cum & kq_gate_mask(D[feat].to_numpy(float), kind, cuts[gi])
            steps.append(("cumulative to " + name, cum.copy()))
        return steps
    KQ_EV, KQ_CUTS, KQ_PI = [], [], []
    def kq_rows(FR, fam, mode, step, m, D, parent_cum=None):
        yz = D["yz"].to_numpy(float); yb = D["yb"].to_numpy(float); day = D["dayi"].to_numpy(); nd = int(day.max()) + 1; spl = D["spl"].to_numpy(); row = dict(frame=FR, family=fam, mode=mode, step=step)
        for si, sn in enumerate(KN_SPN):
            ch = m & (spl == si); n, mu, t = kn_group(yz, day, ch, nd); row.update({f"n_{sn}": n, f"z_{sn}": mu, f"t_{sn}": t, f"bp_{sn}": kn_group(yb, day, ch, nd)[1], f"win_{sn}": float(np.mean(yb[ch & np.isfinite(yb)] > 0)) if (ch & np.isfinite(yb)).sum() else np.nan})
            if step != "parent":
                sel = spl == si; dm, dt = kn_diff(yz[sel], m[sel], day[sel], nd); row[f"diff_z_{sn}"] = dm; row[f"diff_t_{sn}"] = dt
        return row
    for FR, fams in KQ_E.items():
        for fam, D in fams.items():
            gates = KQ_FAM[fam][2]; spl = D["spl"].to_numpy(); pid = D["pid"].to_numpy(); pool_cuts, _ = kq_learn(D, spl == 0, gates, KQ_MINL)
            # per instrument: own cut from own train_R rows, shrunk toward the pooled cut with weight n / (n + K)
            prod_cum = np.zeros((len(D), len(gates)), bool); prod_cuts = {}
            for pk_ in np.unique(pid):
                ix = np.where(pid == pk_)[0]; Dp = D.iloc[ix].reset_index(drop=True); ntr = int((Dp["spl"] == 0).sum()); w = ntr / (ntr + KQ_K)
                cuts, own = kq_learn(Dp, (Dp["spl"] == 0).to_numpy(), gates, KQ_MINL, shrink_to=pool_cuts, w=w); prod_cuts[pk_] = (cuts, own, ntr, w)
                cum = np.ones(len(Dp), bool)
                for gi, (name, feat, kind, pst) in enumerate(gates): cum = cum & kq_gate_mask(Dp[feat].to_numpy(float), kind, cuts[gi]); prod_cum[ix, gi] = cum
                KQ_CUTS.append(dict(frame=FR, family=fam, product=KQ_PROD[FR][pk_], n_train_events=ntr, weight_own=round(w, 3), **{f"own_{g[0]}": o for g, o in zip(gates, own)}, **{f"used_{g[0]}": c for g, c in zip(gates, cuts)}, **{f"pooled_{g[0]}": c for g, c in zip(gates, pool_cuts)}))
            for mode, steps in (("parent", [("parent", np.ones(len(D), bool))]), ("pasted", kq_chain(D, gates, None, pasted=True)), ("pooled", kq_chain(D, gates, pool_cuts)), ("product", [("parent", np.ones(len(D), bool))] + [("cumulative to " + g[0], prod_cum[:, gi]) for gi, g in enumerate(gates)])):
                for step, m in steps:
                    if mode == "parent" or step != "parent": KQ_EV.append(kq_rows(FR, fam, mode, step, m, D))
            # per instrument result of its own final rule
            fin = prod_cum[:, -1]; yz = D["yz"].to_numpy(float); yb = D["yb"].to_numpy(float); day = D["dayi"].to_numpy(); nd = int(day.max()) + 1
            for pk_ in np.unique(pid):
                row = dict(frame=FR, family=fam, product=KQ_PROD[FR][pk_])
                for si, sn in enumerate(KN_SPN):
                    ch = fin & (pid == pk_) & (spl == si); n, mu, t = kn_group(yz, day, ch, nd); row.update({f"n_{sn}": n, f"bp_{sn}": kn_group(yb, day, ch, nd)[1], f"t_{sn}": t})
                KQ_PI.append(row)
    KQ_EV = pd.DataFrame(KQ_EV); KQ_CUTS = pd.DataFrame(KQ_CUTS); KQ_PI = pd.DataFrame(KQ_PI)
    _fin = KQ_EV[(KQ_EV["mode"] != "parent")]; _last = _fin.groupby(["frame", "family", "mode"])["step"].transform("last") == _fin["step"]; c = _fin
    NT_KQ = int(((c.n_test >= KQ_MINN) & _last).sum()); T_KQ = bonferroni_t(max(NT_KQ, 1))
    KQ_EV["final"] = False; KQ_EV.loc[_fin.index[_last], "final"] = True; c = KQ_EV
    KQ_EV["supported"] = c.final & (c["mode"] != "parent") & (c.n_train_R >= KQ_MINN) & (c.n_val_R >= KQ_MINN) & (c.n_test >= KQ_MINN) & (c.t_train_R > 2) & (c.t_val_R > 2) & (c.t_test > T_KQ) & (c.bp_test > 0) & (c.diff_t_val_R > 0) & (c.diff_t_test > 0)
    KQ_EV["clears_cost"] = KQ_EV.supported & (KQ_EV.bp_test > KO_RT); KQ_EV["be_over_cost"] = KQ_EV.bp_test / KO_RT
    KQ_EV.round(4).to_csv(OUT2 / "rules_by_instrument_modes.csv", index=False); KQ_CUTS.to_csv(OUT2 / "rules_learned_cuts_by_instrument.csv", index=False); KQ_PI.round(4).to_csv(OUT2 / "rules_by_instrument_results.csv", index=False)
    print(f"Pasted rules (breakout, climax-absorption reversal, liquidation-cascade reversal) with three threshold sets: pasted, pooled over instruments (train_R), per instrument (train_R, shrunk toward pooled). {NT_KQ} final rules with enough test events, Bonferroni t > {T_KQ:.2f}. Outcome: move over the next {KQ_NF} bars after a one-bar delay, in the thesis direction (breakout and cascade continue, absorption reverses).")
    _f = KQ_EV[KQ_EV.final | (KQ_EV["mode"] == "parent")]
    print(_f[["frame", "family", "mode", "n_train_R", "n_val_R", "n_test", "bp_train_R", "bp_val_R", "bp_test", "t_train_R", "t_val_R", "t_test", "win_test", "diff_t_test", "supported", "clears_cost"]].round(2).to_string(index=False))
    print("Final rules that are supported:"); print(_f[_f.supported][["frame", "family", "mode", "n_test", "bp_test", "t_test", "be_over_cost", "clears_cost"]].round(2).to_string(index=False) if _f.supported.any() else "none")
    # do the per-instrument cuts differ, and are the differences reproducible? the same procedure on val_R rows alone, cut by cut, rank-correlated across instruments with the train_R cut
    KQ_REP = []
    for FR, fams in KQ_E.items():
        for fam, D in fams.items():
            gates = KQ_FAM[fam][2]; pid = D["pid"].to_numpy(); tr_c, va_c = {}, {}
            for pk_ in np.unique(pid):
                Dp = D.iloc[np.where(pid == pk_)[0]].reset_index(drop=True)
                if (Dp["spl"] == 0).sum() >= KQ_MINL and (Dp["spl"] == 1).sum() >= KQ_MINL: tr_c[pk_] = kq_learn(Dp, (Dp["spl"] == 0).to_numpy(), gates, KQ_MINL)[1]; va_c[pk_] = kq_learn(Dp, (Dp["spl"] == 1).to_numpy(), gates, KQ_MINL)[1]
            for gi, g in enumerate(gates):
                if g[2] == "pos": continue
                pr = [k for k in tr_c if tr_c[k][gi] is not None and va_c[k][gi] is not None]
                if len(pr) >= 6:
                    a = np.array([tr_c[k][gi] for k in pr]); b = np.array([va_c[k][gi] for k in pr]); KQ_REP.append(dict(frame=FR, family=fam, gate=g[0], instruments=len(pr), spearman_train_vs_val=stats.spearmanr(a, b)[0], cv_across_instruments=float(np.std(a) / abs(np.mean(a))) if np.mean(a) != 0 else np.nan, pasted=g[3]))
    KQ_REP = pd.DataFrame(KQ_REP); KQ_REP.round(3).to_csv(OUT2 / "rules_cut_reproducibility.csv", index=False)
    print("\nAre per-instrument cuts a property of the instrument? Spearman correlation across instruments between the cut learned on train_R and the cut learned on val_R (near 0 = noise; high = reproducible), and the spread of the cut across instruments:")
    print(KQ_REP.round(2).to_string(index=False) if len(KQ_REP) else "too few instruments with enough events")
    _pi = KQ_PI.copy(); _pi["pos3"] = (_pi.bp_train_R > 0) & (_pi.bp_val_R > 0) & (_pi.bp_test > 0) & (_pi.n_test >= 30)
    print("\nInstruments whose own final rule has a positive mean in all three periods (count of instruments with at least 30 test events):"); print(_pi.groupby(["frame", "family"]).agg(instruments=("product", "count"), positive_all_three=("pos3", "sum")).to_string())

In [ ]:
# ---- 18q.3 anatomy of the bars just before a spike in return, and of what follows. Nothing is assumed: a spike is a bar whose return (in ex-ante volatility units) is in the top tail of ITS OWN instrument's training period; the last 3 bars before it are compared with all other bars, and spikes that continued are compared with spikes that reversed ----
if KQ_OK and KQ_E:
    KQ_SQ = (0.95, 0.99); KQ_BSUB = {"5m": 4, "15m": 2, "30m": 1, "60m": 1, "D": 1}; KQ_NREF = {}
    for FR in KQ_E:
        sel_ = KP_SEL[(KP_SEL.frame == FR) & (KP_SEL.test == "cont5") & (KP_SEL.vtype == ("raw" if FR == "D" else "adj"))] if "KP_SEL" in globals() else []
        KQ_NREF[FR] = int(sel_["n_star"].iloc[0]) if len(sel_) else 5
    def kq_lag(a, k):
        o = np.full(len(a), np.nan)
        if k < len(a): o[k:] = a[:-k]
        return o
    def kq_pre(d, s, FR, nref):
        n = len(d); tdi = d["tdi"].to_numpy()
        if s == 1: O, H, L, C = (d[k].to_numpy(float) for k in ("op", "hi", "lo", "cl"))
        else: O, H, L, C = -d["op"].to_numpy(float), -d["lo"].to_numpy(float), -d["hi"].to_numpy(float), -d["cl"].to_numpy(float)
        rng = H - L; ok = rng > 0; ret = C - O; body = np.abs(ret); sc = pd.Series(C).diff().rolling(60, min_periods=30).std().shift(1).to_numpy(); rz = ret / sc
        wfr = np.where(ok, (H - np.maximum(O, C)) / np.where(ok, rng, 1), np.nan); wbr = np.where(ok, (np.minimum(O, C) - L) / np.where(ok, rng, 1), np.nan); brr = np.where(ok, body / np.where(ok, rng, 1), np.nan)
        bp_ = body > 0; wfb = np.where(bp_, np.minimum((H - np.maximum(O, C)) / np.where(bp_, body, 1), 10.0), np.nan); wbb = np.where(bp_, np.minimum((np.minimum(O, C) - L) / np.where(bp_, body, 1), 10.0), np.nan)
        Vx = kp_vol(d, FR, "raw" if FR == "D" else "adj"); ema = pd.Series(Vx).ewm(span=nref, adjust=False, min_periods=nref).mean().shift(1).to_numpy(); iv = np.clip(Vx / np.where(ema > 0, ema, np.nan) - 1.0, -1.0, KP_CLIP)
        OI = d["oi"].to_numpy(float); rb = d["rollb"].fillna(0).to_numpy() > 0; doi = np.full(n, np.nan); doi[1:] = (OI[1:] - OI[:-1]) / np.where(OI[:-1] > 0, OI[:-1], np.nan); doi = np.where(rb | np.r_[False, rb[:-1]], np.nan, doi)
        f = {}
        for k in (1, 2, 3):
            f[f"r{k}"] = kq_lag(rz, k); f[f"wf{k}"] = kq_lag(wfr, k); f[f"wb{k}"] = kq_lag(wbr, k); f[f"wfb{k}"] = kq_lag(wfb, k); f[f"wbb{k}"] = kq_lag(wbb, k); f[f"brr{k}"] = kq_lag(brr, k); f[f"iv{k}"] = kq_lag(iv, k); f[f"doi{k}"] = kq_lag(doi, k) * 1e4
        f["tr3"] = np.r_[[np.nan] * 4, (C[3:-1] - C[:-4]) / (sc[4:] * np.sqrt(3))] if n > 4 else np.full(n, np.nan)
        f["nsame3"] = sum((kq_lag(rz, k) > 0).astype(float) for k in (1, 2, 3)); f["civ3"] = f["iv1"] + f["iv2"] + f["iv3"]; f["oi3"] = f["doi1"] + f["doi2"] + f["doi3"]
        for nm, cr, co in (("nm3", 1, 1), ("cs3", 1, -1), ("ns3", -1, 1), ("ll3", -1, -1)): f[nm] = sum(((np.sign(kq_lag(rz, k)) == cr) & (np.sign(kq_lag(doi, k)) == co)).astype(float) for k in (1, 2, 3)) + 0 * f["oi3"]
        rr_ = pd.Series(rng); f["sq3"] = (rr_.shift(1).rolling(3).mean() / rr_.shift(4).rolling(20, min_periods=15).mean()).to_numpy()
        f.update(wf0=wfr, wb0=wbr, wfb0=wfb, wbb0=wbb, brr0=brr, iv0=iv, doi0=doi * 1e4, rz0=rz)
        intra = FR != "D"; valid = np.ones(n, bool)
        if intra and n > 3: valid[3:] = tdi[3:] == tdi[:-3]; valid[:3] = False
        ye = {}
        for h in (1, 5):
            a = np.full(n, np.nan)
            if n > h + 1:
                ix = np.arange(n - 1 - h); same = (tdi[ix + 1 + h] == tdi[ix]) if intra else np.ones(len(ix), bool); a[ix[same]] = (C[ix[same] + 1 + h] - C[ix[same] + 1]) / (sc[ix[same]] * np.sqrt(h))
            ye[h] = a
        yb5 = np.full(n, np.nan)
        if n > 6:
            ix = np.arange(n - 6); same = (tdi[ix + 6] == tdi[ix]) if intra else np.ones(len(ix), bool); yb5[ix[same]] = (C[ix[same] + 6] - C[ix[same] + 1]) * 1e4
        return f, rz, sc, ye, yb5, valid
    # unit check: the lag features see only earlier bars (a spike on the last bar does not change its own pre-features)
    _dd = KP_B["30m"][list(KP_B["30m"])[0]].iloc[-1500:] if KP_B.get("30m") else None
    if _dd is not None:
        _f1 = kq_pre(_dd, 1, "30m", 5)[0]; _f2 = kq_pre(_dd.iloc[:-200], 1, "30m", 5)[0]
        for k in ("r1", "wf2", "wbb3", "iv1", "civ3", "doi2", "oi3", "nm3", "sq3", "tr3"): assert np.allclose(_f1[k][:len(_f2[k])], _f2[k], equal_nan=True), f"pre-spike feature {k} uses later bars"
        print("unit check passed: pre-spike features use only bars before the signal bar")
    KQ_PF = [f"{a}{k}" for k in (1, 2, 3) for a in ("r", "wf", "wb", "wfb", "wbb", "brr", "iv", "doi")] + ["tr3", "nsame3", "civ3", "oi3", "nm3", "cs3", "ns3", "ll3", "sq3"]; KQ_AT0 = ["wf0", "wb0", "wfb0", "wbb0", "brr0", "iv0", "doi0", "rz0"]
    KQ_X = {}
    for FR in KQ_E:
        parts = []
        for pk_, (p, d) in enumerate(KP_B[FR].items()):
            dts = KO_T0 + pd.to_timedelta(d["tdi"].to_numpy(), unit="D"); sp = KN_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); vr_ = KO_REG[p].reindex(dts).fillna(-1).to_numpy().astype(int); tr_ = RG["T"].reindex(dts).fillna(-1).to_numpy().astype(int)
            for s in (1, -1):
                f, rz, sc, ye, yb5, valid = kq_pre(d, s, FR, KQ_NREF[FR]); base = valid & (sp >= 0) & (vr_ >= 0) & (tr_ >= 0) & np.isfinite(rz) & np.isfinite(sc) & (sc > 0)
                trn = base & (sp == 0)
                if trn.sum() < 300: continue
                thr = {q: np.nanquantile(np.abs(rz[trn]), q) for q in KQ_SQ}; spk = {q: base & (rz >= thr[q]) for q in KQ_SQ}; ix = np.arange(len(d)); pick = base & (spk[KQ_SQ[0]] | (ix % KQ_BSUB[FR] == 0)); ii = np.where(pick)[0]
                dd = {k: f[k][ii].astype(np.float32) for k in KQ_PF + KQ_AT0}; dd["spk95"] = spk[0.95][ii]; dd["spk99"] = spk[0.99][ii]; dd["y1"] = ye[1][ii].astype(np.float32); dd["y5"] = ye[5][ii].astype(np.float32); dd["yb5"] = yb5[ii].astype(np.float32)
                dd["spl"] = sp[ii]; dd["vol"] = vr_[ii]; dd["trend"] = tr_[ii]; dd["tdi"] = d["tdi"].to_numpy()[ii]; dd["pid"] = np.full(len(ii), pk_); dd["dir"] = np.full(len(ii), s); dd["thr95"] = np.full(len(ii), thr[0.95], np.float32); parts.append(pd.DataFrame(dd))
        if parts:
            D = pd.concat(parts, ignore_index=True); D["dayi"] = np.unique(D["tdi"], return_inverse=True)[1].astype(np.int64); KQ_X[FR] = D
    print({FR: (int(D.spk95.sum()), int(D.spk99.sum()), len(D)) for FR, D in KQ_X.items()}, "spikes (top 5 %, top 1 %) and rows per frame")

    # T1: spike bars against all other bars, feature by feature; T2: spikes that continued against spikes that reversed
    KQ_PRE, KQ_CR = [], []
    for FR, D in KQ_X.items():
        day = D["dayi"].to_numpy(); nd = int(day.max()) + 1; spl = D["spl"].to_numpy(); y5 = D["y5"].to_numpy(float)
        for feat in KQ_PF + KQ_AT0:
            x = D[feat].to_numpy(float); fin = np.isfinite(x)
            for q, col in ((0.95, "spk95"), (0.99, "spk99")):
                ch = D[col].to_numpy(bool); row = dict(frame=FR, spike=f"top {int(round((1 - q) * 100))}%", feature=feat, lag=("at the spike" if feat in KQ_AT0 else "before"))
                for si, sn in enumerate(KN_SPN):
                    sel = fin & (spl == si); dm, dt = kn_diff(x[sel], ch[sel], day[sel], nd); sdb = np.nanstd(x[sel & ~ch]); row.update({f"d_{sn}": (dm / sdb) if sdb > 0 and np.isfinite(dm) else np.nan, f"t_{sn}": dt})
                    row[f"mean_spike_{sn}"] = float(np.nanmean(x[sel & ch])) if (sel & ch).any() else np.nan; row[f"mean_other_{sn}"] = float(np.nanmean(x[sel & ~ch])) if (sel & ~ch).any() else np.nan
                KQ_PRE.append(row)
            sp_ = D["spk95"].to_numpy(bool) & fin & np.isfinite(y5); row = dict(frame=FR, feature=feat, lag=("at the spike" if feat in KQ_AT0 else "before"))
            for si, sn in enumerate(KN_SPN):
                sel = sp_ & (spl == si); dm, dt = kn_diff(x[sel], (y5 > 0)[sel], day[sel], nd); sdb = np.nanstd(x[sel]); row.update({f"d_{sn}": (dm / sdb) if sdb > 0 and np.isfinite(dm) else np.nan, f"t_{sn}": dt, f"ic_{sn}": stats.spearmanr(x[sel], y5[sel])[0] if sel.sum() > 100 else np.nan})
            KQ_CR.append(row)
    KQ_PRE = pd.DataFrame(KQ_PRE); KQ_CR = pd.DataFrame(KQ_CR)
    NT_PRE = int(len(KQ_PRE)); T_PRE = bonferroni_t(NT_PRE); NT_CR = int(len(KQ_CR)); T_CR = bonferroni_t(NT_CR)
    for T_, tb, thr_ in ((KQ_PRE, "spk", T_PRE), (KQ_CR, "cr", T_CR)):
        T_["stable"] = (np.sign(T_.d_train_R) == np.sign(T_.d_val_R)) & (np.sign(T_.d_val_R) == np.sign(T_.d_test)) & (T_.t_train_R.abs() > 2) & (T_.t_val_R.abs() > 2) & (T_.t_test.abs() > thr_)
    KQ_PRE.round(4).to_csv(OUT2 / "prespike_vs_other_bars.csv", index=False); KQ_CR.round(4).to_csv(OUT2 / "prespike_continue_vs_reverse.csv", index=False)
    print(f"\nBars just before a spike (top 5 % / top 1 % of |return| in the instrument's own training period, in ex-ante volatility units, both directions in the spike's direction). d = (mean before spikes - mean before other bars) / std of other bars; day-clustered t; stable = same sign in train_R, val_R, test, |t| > 2 in the first two and above the Bonferroni bar ({T_PRE:.2f}) on the test.")
    print("Features are signed in the direction of the spike: wf = wick on the side the spike goes to, wb = wick on the other side (for an up spike wf is the upper wick, for a down spike the lower); wfb / wbb = the same wicks as a multiple of the body; r = return of the bar; iv = incremental volume of the bar over its EMA (span from Step 18p); doi = open-interest change in bp of OI; nm3 / cs3 / ns3 / ll3 = number of the last 3 bars that were (move with the spike and OI up / move with the spike and OI down / move against and OI up / move against and OI down).")
    _s = KQ_PRE[KQ_PRE.stable & (KQ_PRE.lag == "before")].copy(); _s["abs_d"] = _s.d_test.abs()
    print("Stable differences before the spike (strongest 30):"); print(_s.sort_values("abs_d", ascending=False).head(30)[["frame", "spike", "feature", "d_train_R", "d_val_R", "d_test", "t_test", "mean_spike_test", "mean_other_test"]].round(3).to_string(index=False) if len(_s) else "none")
    _c = KQ_CR[KQ_CR.stable].copy(); _c["abs_d"] = _c.d_test.abs()
    print(f"\nSpikes that continued (next 5 bars after a one-bar delay) minus spikes that reversed: stable features (Bonferroni {T_CR:.2f}; ic = rank correlation with the continuation size):"); print(_c.sort_values("abs_d", ascending=False).head(30)[["frame", "feature", "lag", "d_train_R", "d_val_R", "d_test", "t_test", "ic_test"]].round(3).to_string(index=False) if len(_c) else "none")
    print("\nShare of spikes (top 5 %) that continue over the next 5 bars, by period (data decides whether spikes continue or reverse):"); print(pd.DataFrame([dict(frame=FR, **{sn: float(np.nanmean(D.loc[D.spk95 & (D.spl == si) & np.isfinite(D.y5), "y5"] > 0)) for si, sn in enumerate(KN_SPN)}, mean_z_test=float(np.nanmean(D.loc[D.spk95 & (D.spl == 2), "y5"]))) for FR, D in KQ_X.items()]).round(3).to_string(index=False))

In [ ]:
# ---- 18q.4 Return x change in open interest, 2x2 on the signal bar itself, with its follow-through (next bar and next 5 bars after a one-bar delay), then summary and figure ----
if KQ_OK and KQ_X:
    KQ_M = []; KQ_STATE = {(1, 1): "price up, OI up (new longs)", (1, -1): "price up, OI down (short covering)", (-1, 1): "price down, OI up (new shorts)", (-1, -1): "price down, OI down (long liquidation)"}
    for FR, D in KQ_X.items():
        base = ~D["spk95"].to_numpy(bool); day = D["dayi"].to_numpy(); nd = int(day.max()) + 1; spl = D["spl"].to_numpy(); sr = np.sign(D["rz0"].to_numpy(float)); so = np.sign(D["doi0"].to_numpy(float)); rz = D["rz0"].to_numpy(float)
        thr = np.nanquantile(np.abs(rz[base & (spl == 0)]), 0.8) if (base & (spl == 0)).sum() > 200 else np.nan
        for (a, b), nm in KQ_STATE.items():
            for strength, sm in (("any", np.ones(len(D), bool)), ("strong", np.abs(rz) >= thr)):
                ch = (sr == a) & (so == b) & sm; row = dict(frame=FR, state=nm, size=strength)
                for h in ("y1", "y5"):
                    y = D[h].to_numpy(float)
                    for si, sn in enumerate(KN_SPN):
                        sel = base & (spl == si) & np.isfinite(y) & (sr != 0) & (so != 0); dm, dt = kn_diff(y[sel], ch[sel], day[sel], nd); row.update({f"{h}_{sn}": float(np.nanmean(y[sel & ch])) if (sel & ch).any() else np.nan, f"t{h}_{sn}": dt, f"n_{sn}": int((sel & ch).sum())})
                KQ_M.append(row)
    KQ_M = pd.DataFrame(KQ_M); T_M = bonferroni_t(len(KQ_M) * 2)
    KQ_M["stable"] = ((np.sign(KQ_M.y5_train_R) == np.sign(KQ_M.y5_val_R)) & (np.sign(KQ_M.y5_val_R) == np.sign(KQ_M.y5_test)) & (KQ_M.ty5_train_R.abs() > 2) & (KQ_M.ty5_val_R.abs() > 2) & (KQ_M.ty5_test.abs() > T_M))
    KQ_M.round(4).to_csv(OUT2 / "return_oi_matrix_followthrough.csv", index=False)
    print(f"Return x change in OI on the signal bar (both directions, in the direction of the bar's own move; y = move over the next bar / next 5 bars in volatility units, positive = the bar's direction carries on; t = against all other bars, day-clustered; stable = same sign in all three periods with Bonferroni {T_M:.2f} on the test):")
    print(KQ_M[KQ_M["size"] == "strong"][["frame", "state", "y1_train_R", "y1_val_R", "y1_test", "y5_train_R", "y5_val_R", "y5_test", "ty5_test", "stable"]].round(3).to_string(index=False))
    fig, axs = plt.subplots(1, 3, figsize=(15, 4)); _fr = list(KQ_X)
    _h = KQ_M[KQ_M["size"] == "strong"].pivot_table(index="state", columns="frame", values="y5_test").reindex(columns=_fr); axs[0].imshow(_h.to_numpy(float), cmap="RdBu", vmin=-0.3, vmax=0.3, aspect="auto"); axs[0].set_xticks(range(len(_fr))); axs[0].set_xticklabels(_fr); axs[0].set_yticks(range(len(_h))); axs[0].set_yticklabels([s[:22] for s in _h.index], fontsize=7); axs[0].set_title("next-5-bar follow-through after a strong bar (test)")
    _p = KQ_PRE[(KQ_PRE.spike == "top 5%") & (KQ_PRE.frame == "5m") & KQ_PRE.feature.isin(["wf1", "wb1", "r1", "iv1", "doi1", "sq3", "tr3"])]; axs[1].bar(_p.feature, _p.d_test); axs[1].set_title("5m, bar before a top-5% spike: standardised difference (test)"); axs[1].axhline(0, color="k", lw=.8)
    _c = KQ_CR[KQ_CR.frame == "5m"].sort_values("d_test", key=abs, ascending=False).head(8); axs[2].barh(_c.feature, _c.d_test); axs[2].set_title("5m: spikes that continued minus reversed (test)"); axs[2].axvline(0, color="k", lw=.8)
    fig.tight_layout(); fig.savefig(OUT2 / "18q_instrument_rules_prespike.png", dpi=110); plt.show()
    _ev = KQ_EV[KQ_EV.get("supported", False) == True] if "supported" in KQ_EV else KQ_EV.iloc[0:0]; _ps = KQ_PRE[KQ_PRE.stable & (KQ_PRE.lag == "before")]; _cs = KQ_CR[KQ_CR.stable]
    KQ_SUMMARY = dict(final_rules_supported=int(len(_ev)), supported_rules=_ev.round(3).to_dict("records") if len(_ev) else [], reproducibility=KQ_REP.round(3).to_dict("records"), instruments_positive_all_three=KQ_PI.round(3).to_dict("records") if len(KQ_PI) else [],
                      prespike_stable=int(len(_ps)), prespike_top=_ps.assign(a=_ps.d_test.abs()).sort_values("a", ascending=False).head(15)[["frame", "spike", "feature", "d_train_R", "d_val_R", "d_test", "t_test"]].round(3).to_dict("records"),
                      continue_vs_reverse_stable=int(len(_cs)), continue_vs_reverse=_cs.round(3).to_dict("records"), matrix_stable=KQ_M[KQ_M.stable][["frame", "state", "size", "y5_train_R", "y5_val_R", "y5_test", "ty5_test"]].round(3).to_dict("records"),
                      continuation_share=[dict(frame=FR, **{sn: round(float(np.nanmean(D.loc[D.spk95 & (D.spl == si) & np.isfinite(D.y5), "y5"] > 0)), 3) for si, sn in enumerate(KN_SPN)}) for FR, D in KQ_X.items()])
    KQ_SUMMARY["reading"] = (f"Per-instrument rules (breakout, climax-and-absorption reversal, liquidation-cascade reversal) with the pasted, pooled and per-instrument cuts: {len(_ev)} final rules are supported (same sign in train_R, val_R and test, above the Bonferroni bar, and better than their parent). "
        f"Pre-spike anatomy: spikes are the top 5 % / 1 % of |return| in the instrument's own training period; {len(_ps)} of {len(KQ_PRE[KQ_PRE.lag == 'before'])} pre-spike features differ from ordinary bars in the same direction in all three periods, and {len(_cs)} separate spikes that continued from spikes that reversed. "
        f"{len(KQ_M[KQ_M.stable])} of {len(KQ_M)} return x OI states have a stable follow-through. Share of spikes that continue over the next 5 bars is in continuation_share; the data, not a definition, decides.")
    print("\n" + KQ_SUMMARY["reading"])
for _k, _v in list(_KQ_SAVED.items()):
    if _k in ("_KQ_SAVED",) or _k.startswith(("KQ_", "kq_")): continue
    if globals().get(_k) is not _v: globals()[_k] = _v

## Step 18r — Direction by horizon: where does a signal's direction first pay more than the round-trip cost?

The earlier steps show that volume, range compression and open interest predict the **size** of the next move far better than its **direction**. This step asks, for every causal signal, how its direction behaves as the holding horizon grows from 5 minutes to 20 days, so the horizon is read from the data instead of guessed.

**Signals** (all known at the close of their bar): bar return, 5-bar trend, close location in the range, distance from the EMA(20), breakout beyond the prior L-bar extreme, OI-confirmed return, volume-confirmed direction, daily Stage 2 (long minus short), 60-day trend and the 20-day cross-sectional return rank (the last three are the previous day's values when mapped onto intraday bars).

**Outcome.** Entry one bar after the signal; the move from that entry over 5 minutes, 30 minutes, 1, 5 and 20 days (horizons shorter than one bar are skipped for that frame). Each product's own train_R mean move is removed first, so a drifting product does not look like a directional edge. The edge is the mean of sign(signal) × move in bp, with a day-block clustered t whose blocks are as long as the horizon, so overlapping moves are not counted twice. Two modes are shown: every signal bar, and the instrument's own top 20 % of |signal| (cut learned on train_R per instrument).

**Reading.** A cell is *stable* if its mean has the same sign in train_R, val_R and test, |t| > 2 in the first two, and passes the Bonferroni bar on the test. The sign is as measured; an inverted signal (negative on train_R and the same in val_R and test) is counted and flagged, since a signal that is reliably wrong is still a signal. *Edge over cost* is the oriented test edge divided by the round-trip cost (2 × the per-side cost). The first horizon at which a stable direction also beats cost is the answer to "which horizon"; the test period is read once.

In [ ]:
# step 18r
_KR_SAVED = dict(globals())          # snapshot: loop variables of this step must not overwrite a notebook name (restored at the end of the step)
KR_OK = bool(globals().get("KP_OK", False)) and all(k in globals() for k in ("KP_B", "kp_vol", "KP_BL", "KO_B", "ko_stage", "KO_AD", "KO_REG", "KO_T0", "KN_SPLR", "KN_SPN", "bonferroni_t", "causal_test", "KO_RT"))
if not KR_OK:
    print("Step 18r needs Step 18p (frames) and Step 18o (Stage 2): run them first; skipped")
else:
    # ---- 18r.0 settings. Question: at which horizon does the DIRECTION of a signal first pay more than the round-trip cost? Every signal is causal (known at the close of its bar), entry is one bar later, the outcome is the move from that entry over the horizon ----
    KR_FR = ("5m", "15m", "30m", "60m", "D"); KR_FM = {"5m": 5, "15m": 15, "30m": 30, "60m": 60, "D": None}
    KR_H = (("5m", ("m", 5)), ("30m", ("m", 30)), ("1d", ("d", 1)), ("5d", ("d", 5)), ("20d", ("d", 20)))          # horizon label, (unit, size)
    KR_SIG = ("r1", "tr5", "cl", "ext", "brk", "oir", "ivr", "s2", "m60", "xs20")
    KR_DOC = dict(r1="return of the bar over its ex-ante volatility (momentum if positive)", tr5="5-bar trend", cl="close location in the bar's range (-1 low .. +1 high)", ext="distance of the close from its EMA(20)",
                  brk="close beyond the prior L-bar extreme (+1 up, -1 down)", oir="bar return with the sign of the open-interest change (OI-confirmed move)", ivr="direction of the bar times its incremental volume over EMA(5)",
                  s2="daily Stage 2 long minus Stage 2 short, known the day before", m60="60-day trend over its volatility, known the day before", xs20="20-day return relative to the other products, known the day before")
    KR_TOP = 0.8; KR_MINN = 200; KR_IC_N = 100000
    KR_BPD = {}
    def kr_hbars(FR, unit, size):
        if unit == "m": return None if (KR_FM[FR] is None or size < KR_FM[FR] or size % KR_FM[FR]) else size // KR_FM[FR]
        return size if FR == "D" else int(KR_BPD[FR] * size)

    # ---- 18r.1 signals from the bar (no later information), plus daily slow signals mapped to intraday bars with a one-day lag ----
    def kr_sig(d, FR, ctx=None):
        n = len(d); O, H, L, C = (d[k].to_numpy(float) for k in ("op", "hi", "lo", "cl")); cs = pd.Series(C); sc = cs.diff().rolling(60, min_periods=30).std().shift(1).to_numpy(); rng = H - L; ok = rng > 0
        rz = (C - O) / sc; f = {"r1": rz, "tr5": (C - cs.shift(5).to_numpy()) / (sc * np.sqrt(5)), "cl": np.where(ok, 2 * (C - L) / np.where(ok, rng, 1) - 1, np.nan)}
        f["ext"] = (C - cs.ewm(span=20, adjust=False, min_periods=20).mean().to_numpy()) / (sc * np.sqrt(20)); Lb = KP_BL[FR]; hh = pd.Series(H).rolling(Lb).max().shift(1).to_numpy(); ll = pd.Series(L).rolling(Lb).min().shift(1).to_numpy(); f["brk"] = np.where(C > hh, 1.0, np.where(C < ll, -1.0, 0.0)); f["brk"] = np.where(np.isfinite(hh), f["brk"], np.nan)
        OI = d["oi"].to_numpy(float); rb = d["rollb"].fillna(0).to_numpy() > 0; doi = np.full(n, np.nan); doi[1:] = OI[1:] - OI[:-1]; doi = np.where(rb | np.r_[False, rb[:-1]], np.nan, doi); f["oir"] = np.where(np.isfinite(doi), rz * np.sign(doi), np.nan)
        Vx = kp_vol(d, FR, "raw" if FR == "D" else "adj"); ema = pd.Series(Vx).ewm(span=5, adjust=False, min_periods=5).mean().shift(1).to_numpy(); iv = np.clip(Vx / np.where(ema > 0, ema, np.nan) - 1.0, -1.0, 5.0); f["ivr"] = np.sign(rz) * iv
        for k in ("s2", "m60", "xs20"): f[k] = np.full(n, np.nan) if ctx is None else ctx[k]
        return f, C
    def kr_ctx(p, dd, idx_dates, FR):
        # daily slow signals; for intraday bars the value of the previous trading day (the daily close is not yet known during the day)
        s2 = (ko_stage(dd, 1)["s2"].fillna(0) - ko_stage(dd, -1)["s2"].fillna(0)); cl = dd["cl"]; sd = cl.diff().rolling(60, min_periods=40).std(); m60 = (cl - cl.shift(60)) / (sd * np.sqrt(60))
        r20 = KO_AD.diff(20); x20 = r20.sub(r20.mean(axis=1), axis=0).div(r20.std(axis=1), axis=0)[p].reindex(dd.index); S = pd.DataFrame(dict(s2=s2, m60=m60, xs20=x20))
        if FR != "D": S = S.shift(1)
        return {k: S[k].reindex(idx_dates, method="ffill").to_numpy(float) if FR != "D" else S[k].reindex(idx_dates).to_numpy(float) for k in S.columns}
    # unit checks: forward move alignment on a ramp, and the signals use only bars up to their own bar
    _C = np.arange(50.0); _h = 4; _i = np.arange(len(_C) - 1 - _h); assert np.allclose((_C[_i + 1 + _h] - _C[_i + 1]) * 1e4, _h * 1e4), "forward move misaligned"
    def kr_pit(g, FR="30m"):
        f, _ = kr_sig(g, FR); return pd.DataFrame({k: f[k] for k in KR_SIG if k not in ("s2", "m60", "xs20")}, index=g.index)
    if KP_B["30m"]:
        _p0 = list(KP_B["30m"])[0]; causal_test("kr_signals_30m", kr_pit, KP_B["30m"][_p0].iloc[-4000:], step="18r")
    print("unit check passed: forward move starts one bar after the signal bar")

    # ---- 18r.2 panel per frame: signals, forward moves in bp at each horizon, split, day ----
    KR_P = {}
    for FR in KR_FR:
        cols = {k: [] for k in KR_SIG}; fw = {lab: [] for lab, _ in KR_H}; meta = {k: [] for k in ("spl", "tdi", "pid")}; bp_ = []
        for pk_, (p, d) in enumerate(KP_B[FR].items()):
            if p not in KP_B["D"]: continue
            bpd = 1 if FR == "D" else int(max(1, round(float(d.groupby("tdi").size().median())))); KR_BPD[FR] = bpd; n = len(d)
            dts = KO_T0 + pd.to_timedelta(d["tdi"].to_numpy(), unit="D"); ctx = kr_ctx(p, KP_B["D"][p], pd.DatetimeIndex(dts) if FR != "D" else d.index, FR); f, C = kr_sig(d, FR, ctx)
            sp = KN_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); keep = (sp >= 0)
            for k in KR_SIG: cols[k].append(f[k][keep].astype(np.float32))
            for lab, (u, s) in KR_H:
                h = kr_hbars(FR, u, s); a = np.full(n, np.nan)
                if h is not None and n > h + 2: ix = np.arange(n - 1 - h); a[ix] = (C[ix + 1 + h] - C[ix + 1]) * 1e4
                fw[lab].append(a[keep].astype(np.float32))
            meta["spl"].append(sp[keep]); meta["tdi"].append(d["tdi"].to_numpy()[keep]); meta["pid"].append(np.full(int(keep.sum()), pk_))
        if not meta["spl"]: continue
        P = {k: np.concatenate(v) for k, v in cols.items()}; P.update({"y_" + k: np.concatenate(v) for k, v in fw.items()}); P.update({k: np.concatenate(v) for k, v in meta.items()}); KR_P[FR] = P
    print({FR: len(P["spl"]) for FR, P in KR_P.items()}, "rows per frame;", {FR: KR_BPD.get(FR) for FR in KR_FR}, "bars per day")

In [ ]:
# ---- 18r.3 edge of the signal's direction at each horizon: mean of sign(signal) x move (bp), day-block clustered t (blocks as long as the horizon, so overlapping moves are not counted twice), rank correlation, and the edge over the round-trip cost. Two modes: every signal bar, and the instrument's own top 20 % of |signal| (cut learned on train_R). The sign is read as measured and also as learned on train_R (an inverted signal is a signal) ----
if KR_OK and KR_P:
    from scipy.stats import spearmanr as _sp
    KR_RES = []; _rng = np.random.default_rng(18)
    for FR, P in KR_P.items():
        spl = P["spl"]; tdi = P["tdi"]; pid = P["pid"]
        for sg in KR_SIG:
            x = P[sg].astype(float); fin = np.isfinite(x) & (x != 0)
            if fin.sum() < 1000: continue
            thr = np.full(int(pid.max()) + 1, np.nan)
            for q_ in range(len(thr)):
                m_ = fin & (pid == q_) & (spl == 0)
                if m_.sum() >= 150: thr[q_] = np.quantile(np.abs(x[m_]), KR_TOP)
            top = fin & (np.abs(x) >= thr[pid])
            for lab, (u, s) in KR_H:
                y = P["y_" + lab].astype(float)
                dm_ = np.array([np.nanmean(y[(pid == q_) & (spl == 0)]) if ((pid == q_) & (spl == 0) & np.isfinite(y)).sum() > 50 else 0.0 for q_ in range(int(pid.max()) + 1)]); y = y - dm_[pid]          # drift removed with the product's own train_R mean move (no later information)
                if not np.isfinite(y).any(): continue
                bd = 1 if (u == "m" or s == 1) else s; blk = np.unique(tdi // bd, return_inverse=True)[1].astype(np.int64); nb = int(blk.max()) + 1
                for mode, msk in (("all", fin), ("top20", top)):
                    row = dict(frame=FR, signal=sg, mode=mode, horizon=lab)
                    for si, sn in enumerate(KN_SPN):
                        sel = msk & (spl == si) & np.isfinite(y)
                        e = np.sign(x[sel]) * y[sel]; n_ = int(sel.sum())
                        if n_ < KR_MINN: row.update({f"n_{sn}": n_, f"bp_{sn}": np.nan, f"t_{sn}": np.nan, f"ic_{sn}": np.nan}); continue
                        n_, m_, t_ = kn_group(e, blk[sel], np.ones(n_, bool), nb)
                        ic = np.nan
                        if mode == "all":
                            ii = np.where(sel)[0]
                            if len(ii) > KR_IC_N: ii = _rng.choice(ii, KR_IC_N, replace=False)
                            ic = float(_sp(x[ii], y[ii])[0])
                        row.update({f"n_{sn}": n_, f"bp_{sn}": m_, f"t_{sn}": t_, f"ic_{sn}": ic})
                    KR_RES.append(row)
    KR_RES = pd.DataFrame(KR_RES); NT_KR = int(KR_RES.t_test.notna().sum()); T_KR = bonferroni_t(max(NT_KR, 1))
    KR_RES["sign"] = np.sign(KR_RES.bp_train_R)
    for sn in KN_SPN: KR_RES[f"or_{sn}"] = KR_RES["sign"] * KR_RES[f"bp_{sn}"]
    KR_RES["same_sign"] = (np.sign(KR_RES.bp_train_R) == np.sign(KR_RES.bp_val_R)) & (np.sign(KR_RES.bp_val_R) == np.sign(KR_RES.bp_test))
    KR_RES["stable"] = KR_RES.same_sign & (KR_RES.t_train_R.abs() > 2) & (KR_RES.t_val_R.abs() > 2) & (KR_RES.t_test.abs() > T_KR)
    KR_RES["edge_over_cost"] = KR_RES.or_test / KO_RT; KR_RES["beats_cost"] = KR_RES.stable & (KR_RES.edge_over_cost > 1)
    KR_RES.round(4).to_csv(OUT2 / "direction_by_horizon.csv", index=False)
    HZ = [h for h, _ in KR_H]
    KR_FIRST = []
    for (fr, sg, md), g in KR_RES.groupby(["frame", "signal", "mode"]):
        g = g.set_index("horizon").reindex(HZ); st = g[g.stable == True]; bc = g[g.beats_cost == True]
        KR_FIRST.append(dict(frame=fr, signal=sg, mode=md, first_stable=(st.index[0] if len(st) else "none"), sign=("as measured" if (len(st) and st.sign.iloc[0] > 0) else ("inverted" if len(st) else "")), first_beats_cost=(bc.index[0] if len(bc) else "none"), best_edge_over_cost=float(g.edge_over_cost.where(g.stable == True).max()) if len(st) else np.nan))
    KR_FIRST = pd.DataFrame(KR_FIRST); KR_FIRST.round(3).to_csv(OUT2 / "direction_first_horizon.csv", index=False)
    print(f"Direction by horizon: {len(KR_RES)} rows, {NT_KR} with a test; stable = same sign in train_R, val_R and test, |t| > 2 in the first two, |t| > {T_KR:.2f} on the test (Bonferroni); edge over cost = oriented test edge in bp / round-trip cost {KO_RT:.1f} bp; a * marks a stable cell.")
    for FR in KR_P:
        for md in ("top20",):
            g = KR_RES[(KR_RES.frame == FR) & (KR_RES["mode"] == md)]
            if not len(g): continue
            tb = g.assign(v=g.edge_over_cost.map(lambda v: "" if not np.isfinite(v) else f"{v:.2f}") + np.where(g.stable, "*", "")).pivot(index="signal", columns="horizon", values="v").reindex(index=list(KR_SIG), columns=HZ)
            print(f"\n{FR}, {md} (instrument's own top 20 % of |signal|): oriented test edge over cost, by horizon"); print(tb.fillna("").to_string())
    print("\nFirst horizon at which a signal's direction is stable, and first at which it also beats cost:"); print(KR_FIRST[(KR_FIRST.first_stable != "none")].sort_values(["frame", "signal"]).round(2).to_string(index=False) if (KR_FIRST.first_stable != "none").any() else "none")
    fig, axs = plt.subplots(1, len(KR_P), figsize=(3.4 * len(KR_P), 3.6), squeeze=False)
    for ax, FR in zip(axs[0], KR_P):
        g = KR_RES[(KR_RES.frame == FR) & (KR_RES["mode"] == "top20")].pivot(index="signal", columns="horizon", values="edge_over_cost").reindex(index=list(KR_SIG), columns=HZ); ax.imshow(np.clip(g.to_numpy(float), -3, 3), cmap="RdBu", vmin=-3, vmax=3, aspect="auto"); ax.set_xticks(range(len(HZ))); ax.set_xticklabels(HZ, fontsize=7); ax.set_yticks(range(len(KR_SIG))); ax.set_yticklabels(KR_SIG, fontsize=7); ax.set_title(f"{FR}: edge / cost (top 20 %)", fontsize=8)
    fig.tight_layout(); fig.savefig(OUT2 / "18r_direction_by_horizon.png", dpi=110); plt.show()
    _st = KR_RES[KR_RES.stable].sort_values("edge_over_cost", ascending=False); _bc = KR_RES[KR_RES.beats_cost]
    KR_SUMMARY = dict(rows=int(len(KR_RES)), tests=NT_KR, bonferroni_t=round(T_KR, 2), stable=int(len(_st)), beats_cost=int(len(_bc)), round_trip_bp=round(float(KO_RT), 2), top_stable=_st.head(15)[["frame", "signal", "mode", "horizon", "sign", "bp_train_R", "bp_val_R", "bp_test", "t_test", "edge_over_cost"]].round(3).to_dict("records"),
                      beats=_bc.head(15)[["frame", "signal", "mode", "horizon", "sign", "bp_test", "t_test", "edge_over_cost"]].round(3).to_dict("records"), first=KR_FIRST[KR_FIRST.first_stable != "none"].round(3).to_dict("records"))
    _h_ = _st.groupby("horizon").size().reindex(HZ).fillna(0).astype(int).to_dict(); _hb = _bc.groupby("horizon").size().reindex(HZ).fillna(0).astype(int).to_dict()
    KR_SUMMARY["reading"] = (f"Direction by horizon: {len(KR_RES)} signal x frame x mode x horizon cells (10 causal signals, 5 frames, horizons 5 minutes to 20 days, moves measured from one bar after the signal, product drift (train_R mean move) removed, day-block clustered t so overlapping moves are not double counted); {len(_st)} are stable (same sign in train_R, val_R and test, Bonferroni {T_KR:.2f}) and {len(_bc)} of them also beat the {KO_RT:.0f} bp round-trip cost. Stable cells by horizon {_h_}; beating cost by horizon {_hb}. "
        + ("The shortest horizon at which any stable direction beats cost is " + (min(_bc.horizon, key=HZ.index) if len(_bc) else "none") + ". ") + "An inverted signal (negative train edge, same sign in all three periods) is counted and flagged.")
    print("\n" + KR_SUMMARY["reading"])
for _k, _v in list(_KR_SAVED.items()):
    if _k in ("_KR_SAVED",) or _k.startswith(("KR_", "kr_")): continue
    if globals().get(_k) is not _v: globals()[_k] = _v

## Step 18s — Sector atlas, regimes per time frame, and where momentum and reversion live (items 1-4 of the plan)

**What this step does.** (1) *Sector atlas*: the whole history of every sector in one figure per sector: vol-scaled cumulative return shaded by the sector's own trend regime, 60-day volatility with its train_R tercile cuts, and sector open interest and volume. (2) *Direction that fits each sector*: the next 5-day move of the sector (vol-scaled, drift removed) by the sector's own trend and volatility regime; the side (long or short) is learned on train_R and must hold in val_R and the test. (3) *Regimes per time frame*: for each frame (5, 15, 30, 60 minutes and daily) the volatility of that frame's own bars is cut into low / mid / high at the product's train_R terciles; the history before the test is cut into 60-day blocks, each block is named by the regime that dominates it, and within each regime the blocks are dealt out (every fourth to validation) so that every regime is in both train_F and val_F; the tail of a block that is followed by a block of the other split is purged. The test period is unchanged. (4) *Variance ratio and autocorrelation* (item 2 of the plan): for each sector and product, regime layer (volatility of the frame, market trend), frame and horizon (1-16 bars; 1-20 days), the slope b of the next q-bar move on the last q-bar move, both in volatility units, sampled without overlap and with a t clustered by calendar day. b > 0 is momentum, b < 0 is reversion, and 1 + b is the implied variance ratio at twice the horizon.

**Rule per cell.** The horizon and the sign are chosen on train_F only (strongest |t|), then checked at the same horizon and sign in val_F and the test. *Stable* = |t| > 2 in train_F and val_F, same sign in the test, and the test t above the Bonferroni bar over all rule cells. A regime rule counts as better than its parent only if it beats the same sector's unconditioned rule in train_F, val_F and test.

**Reading.** The tables answer "continue or reverse" per sector, regime and frame. They do not by themselves give a tradeable strategy: the slope is measured from the close of the signal bar, so entry delay and the round-trip cost still have to be paid (Step 18r shows the edge over cost). Carry is not in the panel.

In [ ]:
# step 18s
_KS_SAVED = dict(globals())          # snapshot: loop variables of this step must not overwrite a notebook name (restored at the end of the step)
KS_OK = bool(globals().get("KP_OK", False)) and all(k in globals() for k in ("KP_B", "KO_AD", "KO_T0", "KN_SPLR", "KN_SPN", "bonferroni_t", "causal_test", "RG", "SECTOR", "RAW", "VOLU", "OIW", "OUT2", "FIG_DIR"))
if not KS_OK:
    print("Step 18s needs Step 18p (frames), 13f (regimes) and Step 18n (splits): run them first; skipped")
else:
    # ---- 18s.1 sector atlas: the whole history of every sector, with its own trend and volatility regime (cut points learned on train_R, applied forward) ----
    KS_SECS = sorted({SECTOR.get(p, "Other") for p in RAW.columns}); KS_SOF = {p: SECTOR.get(p, "Other") for p in RAW.columns}
    _tr = (KN_SPLR == 0).reindex(RAW.index).fillna(False).to_numpy()
    def ks_cut(s, tr_mask):
        v = pd.Series(s).to_numpy(float)[tr_mask]; v = v[np.isfinite(v)]
        return (np.quantile(v, 1 / 3), np.quantile(v, 2 / 3)) if len(v) > 200 else (np.nan, np.nan)
    def ks_lev(s, cuts): s = pd.Series(s); return pd.Series(np.where(s.isna() | ~np.isfinite(cuts[0]), -1, (s > cuts[0]).astype(int) + (s > cuts[1]).astype(int)), index=s.index)
    KS_SR, KS_TS, KS_VS, KS_ROW = {}, {}, {}, []
    for sec in KS_SECS:
        ps = [p for p in RAW.columns if KS_SOF[p] == sec]
        Z = pd.DataFrame({p: (RAW[p] / RAW[p].ewm(halflife=30, min_periods=20).std().shift(1)).clip(-5, 5) for p in ps}); sr = Z.mean(axis=1).fillna(0.0); KS_SR[sec] = sr
        m = sr.rolling(60, min_periods=40).sum() / np.sqrt(60); eq = RAW[ps].mean(axis=1); v = eq.rolling(60, min_periods=40).std() * np.sqrt(252)
        KS_TS[sec] = ks_lev(m, ks_cut(m, _tr)); KS_VS[sec] = ks_lev(v, ks_cut(v, _tr))
        for sp_name, k in (("train_R", 0), ("val_R", 1), ("test", 2)):
            msk = (KN_SPLR.reindex(sr.index) == k).to_numpy(); x = sr.to_numpy()[msk]
            if len(x) < 100: continue
            ac1 = float(np.corrcoef(x[1:], x[:-1])[0, 1]); s5 = pd.Series(x).rolling(5).sum().dropna().to_numpy(); vr5 = float(np.var(s5[::5], ddof=1) / (5 * np.var(x, ddof=1))) if len(s5[::5]) > 20 else np.nan
            KS_ROW.append(dict(sector=sec, split=sp_name, n_products=len(ps), days=len(x), drift_sharpe=float(x.mean() / x.std() * np.sqrt(252)), ac1=ac1, vr5=vr5,
                               share_up=float(np.mean(KS_TS[sec].reindex(sr.index)[msk] == 2)), share_down=float(np.mean(KS_TS[sec].reindex(sr.index)[msk] == 0)), share_hivol=float(np.mean(KS_VS[sec].reindex(sr.index)[msk] == 2))))
    KS_ATLAS = pd.DataFrame(KS_ROW); KS_ATLAS.to_csv(OUT2 / "sector_atlas.csv", index=False)
    nS = len(KS_SECS); fig, axs = plt.subplots(nS, 3, figsize=(19, 2.7 * nS), squeeze=False)
    for r_, sec in enumerate(KS_SECS):
        ps = [p for p in RAW.columns if KS_SOF[p] == sec]; sr = KS_SR[sec]; cum = sr.cumsum(); ax = axs[r_, 0]; ax.plot(cum.index, cum.values, color="#1f3a5f", lw=1.0); lo, hi = cum.min(), cum.max()
        for lev, col in ((0, "#d9534f"), (2, "#2e8b57")): ax.fill_between(cum.index, lo, hi, where=(KS_TS[sec].reindex(cum.index) == lev).to_numpy(), color=col, alpha=0.13, step="pre", lw=0)
        for k in ("train", "val", "test"):
            ix = RG_IDX[k] if "RG_IDX" in globals() else None
            if ix is not None and len(ix): ax.axvline(ix[0], color="k", lw=0.6, ls=":")
        ax.set_title(f"{sec} ({len(ps)}): vol-scaled cumulative return; green = up-trend, red = down-trend regime", fontsize=8.5)
        ax = axs[r_, 1]; eq = RAW[ps].mean(axis=1); v = eq.rolling(60, min_periods=40).std() * np.sqrt(252); ax.plot(v.index, v.values, color="#b5651d", lw=0.9); c_ = ks_cut(v, _tr)
        for cc in c_:
            if np.isfinite(cc): ax.axhline(cc, color="grey", lw=0.7, ls="--")
        ax.set_title("60-day annualised volatility; dashes = train_R tercile cuts", fontsize=8.5)
        ax = axs[r_, 2]; oi = OIW[ps].sum(axis=1).rolling(20, min_periods=5).mean(); vo = VOLU[ps].sum(axis=1).rolling(20, min_periods=5).mean()
        for s_, lab, col in ((oi, "open interest", "#6a3d9a"), (vo, "volume", "#2a9d8f")):
            b_ = s_[_tr].median() if _tr.any() else s_.median(); ax.plot(s_.index, (s_ / b_).values, label=lab, color=col, lw=0.9)
        ax.set_yscale("log"); ax.legend(fontsize=7, loc="upper left"); ax.set_title("sector open interest and volume (20-day mean, x train_R median)", fontsize=8.5)
    fig.tight_layout(); fig.savefig(FIG_DIR / "18s_sector_atlas.png", dpi=110); plt.show()
    print("sector atlas (per split: risk-adjusted drift, lag-1 autocorrelation, 5-day variance ratio, share of days in each regime):")
    print(KS_ATLAS.round(3).to_string(index=False))

In [ ]:
# step 18s
# ---- 18s.2 per time frame: volatility regime of THAT frame, then every regime is allocated to train / validation / test ----
if KS_OK:
    KS_FR = ("5m", "15m", "30m", "60m", "D"); KS_BLK = 60; KS_PURGE = 20; KS_VAL_EVERY = 4
    def ks_vt(d, FR, bpd):
        C = d["cl"].to_numpy(float); r = np.r_[np.nan, np.diff(C)] * 1e4; span = 20 if FR == "D" else 10 * bpd
        return pd.Series(r).ewm(span=span, adjust=False, min_periods=max(20, span // 2)).std().to_numpy(), r          # volatility of the bar returns up to and including this bar
    def ks_vt_pit(g, FR="30m"): v, _ = ks_vt(g, FR, 10); return pd.DataFrame({"vt": v}, index=g.index)
    if KP_B["30m"]:
        _p0 = list(KP_B["30m"])[0]; causal_test("ks_regime_vol_30m", ks_vt_pit, KP_B["30m"][_p0].iloc[-4000:], step="18s")
    KS_RGB, KS_SPF, KS_AUX, KS_BPD, KS_ROWS = {}, {}, {}, {}, []
    _trend = RG["T"].shift(1)
    for FR in KS_FR:
        KS_RGB[FR], KS_SPF[FR], KS_AUX[FR] = {}, {}, {}
        for p, d in KP_B[FR].items():
            if p not in RAW.columns or len(d) < 1000: continue
            bpd = 1 if FR == "D" else int(max(1, round(float(d.groupby("tdi").size().median())))); KS_BPD[FR] = bpd; tdi = d["tdi"].to_numpy(); dts = pd.DatetimeIndex(KO_T0 + pd.to_timedelta(tdi, unit="D"))
            vt, r = ks_vt(d, FR, bpd); sp0 = KN_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); trn = (sp0 == 0) & np.isfinite(vt)
            if trn.sum() < 500: continue
            c1, c2 = np.quantile(vt[trn], 1 / 3), np.quantile(vt[trn], 2 / 3); g = np.where(np.isfinite(vt), (vt > c1).astype(int) + (vt > c2).astype(int), -1)
            blk = (tdi // KS_BLK).astype(int); spl = np.where(sp0 == 2, 2, -1); pre = np.isin(sp0, (0, 1)) & (g >= 0)
            # a block takes the regime that dominates it; within each regime the blocks are dealt out (every 4th goes to validation), so each regime is in both train_F and val_F
            if pre.any():
                bd = pd.DataFrame({"b": blk[pre], "g": g[pre]}); dom = bd.groupby("b")["g"].agg(lambda s: int(s.value_counts().idxmax()))
                bsplit = {}
                for gg in (0, 1, 2):
                    for j, b_ in enumerate(dom.index[dom.to_numpy() == gg]): bsplit[b_] = 1 if (j % KS_VAL_EVERY == KS_VAL_EVERY - 1) else 0
                bs = pd.Series(blk[pre]).map(bsplit).to_numpy(); spl[pre] = bs
                nxt = pd.Series(blk[pre]).map(lambda b: bsplit.get(b + 1, -9)).to_numpy(); lastd = (tdi[pre] % KS_BLK) >= (KS_BLK - KS_PURGE); sp_pre = spl[pre].copy(); sp_pre[lastd & (nxt != bs) & (nxt != -9)] = -1; spl[pre] = sp_pre      # purge the tail of a block that is followed by a block of the other split
            tr = _trend.reindex(dts, method="ffill").to_numpy(float); tr = np.where(np.isfinite(tr), tr, -1).astype(int)
            KS_RGB[FR][p] = g.astype(np.int8); KS_SPF[FR][p] = spl.astype(np.int8); KS_AUX[FR][p] = (r, tr.astype(np.int8), tdi.astype(np.int32))
            for gg in (0, 1, 2):
                for k, nm in ((0, "train_F"), (1, "val_F"), (2, "test")):
                    m = (g == gg) & (spl == k); KS_ROWS.append(dict(frame=FR, product=p, regime=gg, split=nm, bars=int(m.sum()), days=int(len(np.unique(tdi[m])))))
    KS_ALLOC = pd.DataFrame(KS_ROWS); KS_ALLOC.to_csv(OUT2 / "regime_allocation_by_frame.csv", index=False)
    KS_AG = KS_ALLOC.groupby(["frame", "regime", "split"])[["bars", "days"]].sum().reset_index(); KS_PIV = KS_AG.pivot_table(index=["frame", "regime"], columns="split", values="bars").reindex(columns=["train_F", "val_F", "test"])
    KS_PIV["val_share"] = KS_PIV["val_F"] / (KS_PIV["train_F"] + KS_PIV["val_F"]); print("bars per frame, volatility regime (0 low .. 2 high, cuts learned on train_R bars of each product) and split:"); print(KS_PIV.round(3).to_string())
    KS_SMALL = KS_PIV[(KS_PIV[["train_F", "val_F", "test"]].min(axis=1) < 500)]; print("regime x split cells with fewer than 500 bars:", len(KS_SMALL))
    fig, axs = plt.subplots(1, 2, figsize=(16, 4.4)); _w = KS_PIV.reset_index(); _lab = [f"{a}/{b}" for a, b in zip(_w["frame"], _w["regime"])]
    tot = _w[["train_F", "val_F", "test"]].sum(axis=1).to_numpy(); bot = np.zeros(len(_w))
    for nm, col in (("train_F", "#1f3a5f"), ("val_F", "#e0a030"), ("test", "#9b2226")): v_ = _w[nm].to_numpy() / np.where(tot > 0, tot, 1); axs[0].bar(_lab, v_, bottom=bot, label=nm, color=col); bot += v_
    axs[0].legend(fontsize=8); axs[0].tick_params(axis="x", rotation=60, labelsize=7); axs[0].set_title("share of bars in train_F / val_F / test, by frame and volatility regime (each regime is in every split)")
    _sh = KS_ALLOC[KS_ALLOC["split"].isin(["train_F", "val_F"])].groupby(["frame", "split"])["bars"].sum().unstack(); (_sh.div(_sh.sum(axis=1), axis=0)).reindex(list(KS_FR)).plot(kind="bar", stacked=True, ax=axs[1], color=["#1f3a5f", "#e0a030"]); axs[1].set_title("train_F versus val_F share of pre-test bars"); fig.tight_layout(); fig.savefig(FIG_DIR / "18s_regime_allocation.png", dpi=110); plt.show()

In [ ]:
# step 18s
# ---- 18s.3 where does momentum live and where does reversion live? past q-bar move (x) against the next q-bar move (y), both in volatility units, by sector / product, regime layer, frame and horizon ----
if KS_OK:
    KS_Q = {"D": (1, 2, 5, 10, 20)}; KS_Q.update({f: (1, 2, 4, 8, 16) for f in ("5m", "15m", "30m", "60m")}); KS_FMIN = {"5m": 5, "15m": 15, "30m": 30, "60m": 60, "D": None}
    KS_SN = ("train_F", "val_F", "test"); KS_MINROWS = 150; KS_DAYR = 1 << 15
    def ks_cells(ent, lev, spl, day, u, x2):
        # b = sum(x*y) / sum(x*x): the slope of the next move on the last move. b > 0 momentum, b < 0 reversion; 1 + b is the implied variance ratio at twice the horizon. t is clustered by calendar day (all products of a sector on one day share a cluster)
        m = (lev >= 0) & (spl >= 0) & (ent >= 0)
        if not m.any(): return pd.DataFrame(columns=["ent", "lev", "spl", "n", "b", "t"])
        key = (ent[m].astype(np.int64) * 4 + lev[m]) * 4 + spl[m]; u, x2, day = u[m], x2[m], day[m].astype(np.int64)
        code = key * KS_DAYR + day; uq, inv = np.unique(code, return_inverse=True); su = np.bincount(inv, weights=u); sx = np.bincount(inv, weights=x2); cn = np.bincount(inv)
        kq, kinv = np.unique(uq // KS_DAYR, return_inverse=True); SU = np.bincount(kinv, weights=su); SX = np.bincount(kinv, weights=sx); N = np.bincount(kinv, weights=cn); b = SU / np.where(SX > 0, SX, np.nan)
        c = su - b[kinv] * sx; ss = np.bincount(kinv, weights=c * c); t = b * SX / np.sqrt(np.where(ss > 0, ss, np.nan))
        return pd.DataFrame(dict(ent=(kq // 16).astype(int), lev=((kq // 4) % 4).astype(int), spl=(kq % 4).astype(int), n=N.astype(int), b=b, t=t))
    # unit check: a planted slope of 0.3 is recovered, and a null is not significant
    _r = np.random.default_rng(3); _x = _r.normal(size=60000); _y = 0.3 * _x + np.sqrt(1 - 0.09) * _r.normal(size=60000); _z = ks_cells(np.zeros(60000, int), np.full(60000, 3), np.zeros(60000, int), _r.integers(0, 3000, 60000), _x * _y, _x * _x)
    assert abs(_z["b"].iloc[0] - 0.3) < 0.02 and _z["t"].iloc[0] > 8, "slope not recovered"; _y0 = _r.normal(size=60000); _z0 = ks_cells(np.zeros(60000, int), np.full(60000, 3), np.zeros(60000, int), _r.integers(0, 3000, 60000), _x * _y0, _x * _x); assert abs(_z0["t"].iloc[0]) < 4, "null looks significant"
    print("unit check passed: planted slope 0.3 recovered, null not significant")
    KS_PIDS = {p: i for i, p in enumerate(sorted({p for FR in KS_FR for p in KS_RGB[FR]}))}; KS_SIDS = {s: i for i, s in enumerate(KS_SECS)}; KS_RES = []
    for FR in KS_FR:
        for q in KS_Q[FR]:
            acc = {k: [] for k in ("u", "x2", "day", "spl", "V", "T", "pid", "sid")}
            for p, g in KS_RGB[FR].items():
                r, tr, tdi = KS_AUX[FR][p]; spl = KS_SPF[FR][p]; n = len(r); sc = pd.Series(r).rolling(60, min_periods=30).std().shift(1).to_numpy()
                ok = np.isfinite(r) & np.isfinite(sc) & (sc > 0); zr = np.where(ok, r / np.where(sc > 0, sc, 1), 0.0); cz = np.cumsum(zr); co = np.cumsum(ok)
                if n < 4 * q + 10: continue
                i = np.arange(q, n - q, q); gd = ((co[i] - co[i - q]) == q) & ((co[i + q] - co[i]) == q); i = i[gd]
                x = (cz[i] - cz[i - q]) / np.sqrt(q); y = (cz[i + q] - cz[i]) / np.sqrt(q)
                acc["u"].append(x * y); acc["x2"].append(x * x); acc["day"].append(tdi[i]); acc["spl"].append(spl[i]); acc["V"].append(g[i]); acc["T"].append(tr[i]); acc["pid"].append(np.full(len(i), KS_PIDS[p])); acc["sid"].append(np.full(len(i), KS_SIDS[KS_SOF[p]]))
            if not acc["u"]: continue
            A = {k: np.concatenate(v) for k, v in acc.items()}
            for ename, ent in (("sector", A["sid"]), ("product", A["pid"])):
                for layer, lev in (("ALL", np.full(len(A["u"]), 3)), ("V", A["V"].astype(int)), ("T", A["T"].astype(int))):
                    z = ks_cells(ent, lev, A["spl"].astype(int), A["day"], A["u"], A["x2"])
                    if len(z): z["entity"] = ename; z["layer"] = layer; z["frame"] = FR; z["q"] = q; KS_RES.append(z)
    KS_ALLR = pd.concat(KS_RES, ignore_index=True) if KS_RES else pd.DataFrame(); KS_ALLR["name"] = np.where(KS_ALLR["entity"] == "sector", KS_ALLR["ent"].map({v: k for k, v in KS_SIDS.items()}), KS_ALLR["ent"].map({v: k for k, v in KS_PIDS.items()}))
    KS_ALLR["split"] = KS_ALLR["spl"].map(dict(enumerate(KS_SN))); KS_ALLR["level"] = KS_ALLR["lev"].map({0: "low/down", 1: "mid/flat", 2: "high/up", 3: "all"}); KS_ALLR["minutes"] = [(KS_FMIN[f] * q if KS_FMIN[f] else np.nan) for f, q in zip(KS_ALLR["frame"], KS_ALLR["q"])]
    W = KS_ALLR[KS_ALLR["n"] >= KS_MINROWS].pivot_table(index=["entity", "name", "frame", "layer", "level", "q"], columns="split", values=["b", "t", "n"]); W.columns = [f"{a}_{b}" for a, b in W.columns]; W = W.reset_index()
    for c in [f"{a}_{s}" for a in ("b", "t", "n") for s in KS_SN]:
        if c not in W: W[c] = np.nan
    W["vr_train"] = 1 + W["b_train_F"]; W["vr_val"] = 1 + W["b_val_F"]; W["vr_test"] = 1 + W["b_test"]
    # the rule of each cell: the horizon with the strongest train_F slope, its sign (momentum or reversion), then the same horizon and sign in val_F and test; no assumed sign
    Wt = W.dropna(subset=["t_train_F"]).copy(); Wt["abs_t"] = Wt["t_train_F"].abs(); sel = Wt.loc[Wt.groupby(["entity", "name", "frame", "layer", "level"])["abs_t"].idxmax()].copy()
    sel["rule"] = np.where(sel["b_train_F"] >= 0, "momentum", "reversion"); s_ = np.where(sel["b_train_F"] >= 0, 1.0, -1.0)
    for sp_ in KS_SN: sel[f"or_b_{sp_}"] = s_ * sel[f"b_{sp_}"]; sel[f"or_t_{sp_}"] = s_ * sel[f"t_{sp_}"]
    sel["same_sign"] = (sel["or_b_val_F"] > 0) & (sel["or_b_test"] > 0)
    # parent = the same entity, frame and horizon with no regime cut; a child must beat it in all three periods
    _pk = W[W["layer"] == "ALL"].set_index(["entity", "name", "frame", "q"]); 
    for sp_ in KS_SN:
        key = list(zip(sel["entity"], sel["name"], sel["frame"], sel["q"])); pb = _pk[f"b_{sp_}"].reindex(key).to_numpy(); sel[f"par_{sp_}"] = np.where(sel["rule"].to_numpy() == "momentum", 1.0, -1.0) * pb
    sel["beats_parent"] = np.where(sel["layer"] == "ALL", True, (sel["or_b_train_F"] > sel["par_train_F"]) & (sel["or_b_val_F"] > sel["par_val_F"]) & (sel["or_b_test"] > sel["par_test"]))
    nt_ = {e: int((sel["entity"] == e).sum()) for e in ("sector", "product")}; sel["bonf"] = sel["entity"].map(lambda e: bonferroni_t(max(nt_[e], 1)))
    sel["stable"] = (sel["or_t_train_F"] > 2) & (sel["or_t_val_F"] > 2) & (sel["or_b_test"] > 0) & (sel["or_t_test"] > sel["bonf"]); sel["stable_child"] = sel["stable"] & sel["beats_parent"]
    sel["minutes"] = [(KS_FMIN[f] * q if KS_FMIN[f] else np.nan) for f, q in zip(sel["frame"], sel["q"])]; sel["horizon"] = [(f"{int(m)}m" if np.isfinite(m) else f"{q}d") for m, q in zip(sel["minutes"], sel["q"])]
    KS_RULE = sel.drop(columns=["abs_t"]); KS_RULE.to_csv(OUT2 / "variance_ratio_rules.csv", index=False); W.to_csv(OUT2 / "variance_ratio_autocorr_all_cells.csv", index=False)
    SR_ = KS_RULE[KS_RULE["entity"] == "sector"]; PR_ = KS_RULE[KS_RULE["entity"] == "product"]
    print(f"\nsector rules: {len(SR_)} cells; stable {int(SR_['stable'].sum())} (momentum {int((SR_['stable'] & (SR_['rule']=='momentum')).sum())}, reversion {int((SR_['stable'] & (SR_['rule']=='reversion')).sum())}); stable and beating the unconditioned parent {int(SR_['stable_child'].sum())}")
    print(f"product rules: {len(PR_)} cells; stable {int(PR_['stable'].sum())} (momentum {int((PR_['stable'] & (PR_['rule']=='momentum')).sum())}, reversion {int((PR_['stable'] & (PR_['rule']=='reversion')).sum())}); stable and beating the parent {int(PR_['stable_child'].sum())}")
    _cols = ["name", "frame", "layer", "level", "horizon", "rule", "or_b_train_F", "or_b_val_F", "or_b_test", "or_t_test", "beats_parent"]
    print("\nstable sector rules (momentum or reversion that holds in train_F, val_F and test):"); print(SR_[SR_["stable"]].sort_values("or_t_test", ascending=False)[_cols].round(3).head(40).to_string(index=False) if SR_["stable"].any() else "none")
    print("\nsector x frame, no regime cut: sign of the slope per period at the strongest train horizon (momentum +, reversion -)")
    _a = SR_[SR_["layer"] == "ALL"].pivot_table(index="name", columns="frame", values="b_train_F").reindex(columns=list(KS_FR)); print(_a.round(3).to_string())
    # ---- 18s.4 direction that fits each sector: next-5-day move of the sector (vol-scaled, drift removed) by the sector's own trend and volatility regime; cuts learned on train_R ----
    KS_DROWS = []
    for sec in KS_SECS:
        sr = KS_SR[sec]; fwd_5d = sr.rolling(5).sum().shift(-5)   # causal-ok: days t+1..t+5, used only as the outcome
        sp = KN_SPLR.reindex(sr.index).fillna(-1).to_numpy().astype(int); pos = np.arange(len(sr)); mu = float(fwd_5d[(sp == 0)].mean())
        for layer, lv in (("ALL", pd.Series(3, index=sr.index)), ("trend", KS_TS[sec]), ("vol", KS_VS[sec])):
            for lev in ((3,) if layer == "ALL" else (0, 1, 2)):
                row = dict(sector=sec, layer=layer, level=lev)
                for sn, k in (("train_R", 0), ("val_R", 1), ("test", 2)):
                    m = (sp == k) & (lv.to_numpy() == lev) & (pos % 5 == 0) & fwd_5d.notna().to_numpy(); x = fwd_5d.to_numpy()[m] - mu; n = len(x)
                    row[f"n_{sn}"] = n; row[f"mean_{sn}"] = float(x.mean()) if n else np.nan; row[f"t_{sn}"] = float(x.mean() / (x.std(ddof=1) / np.sqrt(n))) if n > 20 and x.std(ddof=1) > 0 else np.nan
                KS_DROWS.append(row)
    KS_DIR = pd.DataFrame(KS_DROWS); KS_DIR["side"] = np.where(KS_DIR["mean_train_R"] >= 0, "long", "short"); sg = np.where(KS_DIR["mean_train_R"] >= 0, 1.0, -1.0)
    for sn in ("train_R", "val_R", "test"): KS_DIR[f"or_t_{sn}"] = sg * KS_DIR[f"t_{sn}"]; KS_DIR[f"or_mean_{sn}"] = sg * KS_DIR[f"mean_{sn}"]
    _nd = int((KS_DIR["layer"] != "ALL").sum()); KS_DIR["stable"] = (KS_DIR["or_t_train_R"] > 2) & (KS_DIR["or_t_val_R"] > 2) & (KS_DIR["or_mean_test"] > 0) & (KS_DIR["or_t_test"] > bonferroni_t(max(_nd, 1))) & (KS_DIR["layer"] != "ALL")
    KS_DIR["level_name"] = KS_DIR["level"].map({0: "low/down", 1: "mid/flat", 2: "high/up", 3: "all"}); KS_DIR.to_csv(OUT2 / "sector_direction_by_regime.csv", index=False)
    print(f"\nsector direction: {_nd} sector x regime cells; stable (side learned on train_R holds in val_R and test, drift removed): {int(KS_DIR['stable'].sum())}"); print(KS_DIR[KS_DIR["stable"]][["sector", "layer", "level_name", "side", "mean_train_R", "mean_val_R", "mean_test", "t_test"]].round(3).to_string(index=False) if KS_DIR["stable"].any() else "no sector x regime cell keeps its side in all three periods")
    # ---- figure and summary ----
    fig, axs = plt.subplots(1, 3, figsize=(20, 5.6)); hm = SR_[SR_["layer"] == "ALL"].copy(); hm["col"] = hm["frame"] + "/" + hm["horizon"]
    P_ = hm.pivot_table(index="name", columns="col", values="b_test"); P_ = P_.reindex(columns=[c for f in KS_FR for c in P_.columns if c.startswith(f + "/")])
    im = axs[0].imshow(P_.to_numpy(float), cmap="RdBu_r", vmin=-0.2, vmax=0.2, aspect="auto"); axs[0].set_xticks(range(P_.shape[1])); axs[0].set_xticklabels(P_.columns, rotation=75, fontsize=6.5); axs[0].set_yticks(range(P_.shape[0])); axs[0].set_yticklabels(P_.index, fontsize=8); axs[0].set_title("test slope at the strongest train horizon, no regime cut (red momentum, blue reversion)"); plt.colorbar(im, ax=axs[0], fraction=0.03)
    cnt = SR_.groupby(["frame", "rule"])["stable"].sum().unstack().reindex(list(KS_FR)).fillna(0); cnt.plot(kind="bar", ax=axs[1], color=["#2e8b57", "#d9534f"][:cnt.shape[1]][::-1] if cnt.shape[1] == 2 else None); axs[1].set_title("stable sector rules by frame (train_F, val_F and test agree)")
    lv_ = SR_.groupby(["layer", "level", "rule"])["stable"].sum().unstack().fillna(0); lv_.plot(kind="bar", ax=axs[2]); axs[2].set_title("stable sector rules by regime layer and level"); axs[2].tick_params(axis="x", labelsize=7); fig.tight_layout(); fig.savefig(FIG_DIR / "18s_variance_ratio_by_regime.png", dpi=110); plt.show()
    _top = {}
    for sec in KS_SECS:
        z = SR_[(SR_["name"] == sec) & SR_["stable"]].sort_values("or_t_test", ascending=False).head(3); _top[sec] = [f"{r.frame} {r.horizon} {r.layer}={r.level} {r.rule}" for r in z.itertuples()]
    _kst, _kch = int(SR_["stable"].sum()), int(SR_["stable_child"].sum()); _kpr = int(PR_["stable"].sum()); _kmo = int((SR_["stable"] & (SR_["rule"] == "momentum")).sum()); _kre = int((SR_["stable"] & (SR_["rule"] == "reversion")).sum())
    KS_SUMMARY = dict(sectors=KS_SECS, cells=dict(sector=int(len(SR_)), product=int(len(PR_)), direction=_nd), stable=dict(sector=_kst, momentum=_kmo, reversion=_kre, sector_beating_parent=_kch, product=_kpr, direction=int(KS_DIR["stable"].sum())),
        bonferroni=dict(sector=float(bonferroni_t(max(nt_["sector"], 1))), product=float(bonferroni_t(max(nt_["product"], 1)))), top_rules_by_sector=_top, regime_allocation_small_cells=int(len(KS_SMALL)),
        reading=(f"Momentum or reversion by sector, regime and horizon: {len(SR_)} sector rules (each cell learns its horizon and sign on train_F); {_kst} are stable (same sign in val_F and test, Bonferroni {bonferroni_t(max(nt_['sector'],1)):.2f} on the test): {_kmo} momentum, {_kre} reversion; {_kch} also beat the same sector's unconditioned rule. Products: {_kpr} of {len(PR_)}. "
                 f"Direction that fits each sector (next 5 days, drift removed, side learned on train_R): {int(KS_DIR['stable'].sum())} of {_nd} sector x regime cells hold in all three periods. Regimes are the volatility of each time frame (terciles of its own train_R bars), allocated to train_F / val_F in blocks so each regime is in both; the test period is unchanged."))
    for _k_, _v_ in _KS_SAVED.items():
        if not _k_.startswith("KS_") and _k_ in globals() and globals()[_k_] is not _v_ and _k_ not in ("_KS_SAVED",): globals()[_k_] = _v_

## Step 18t — Is the daily Stage 2 breakout robust? (item 8, moved ahead of the meta-labelling step)

Daily Stage 2 (close > EMA150 > EMA200, EMA200 rising) plus a 20- or 50-day breakout, with the structural stop and the 10-day horizon of Step 18o, is the only rule that was positive in train_R, val_R and the test. This step asks whether it survives the checks a single pooled t-statistic hides.

- **Overlap-robust t.** Every trade's net result (gross bp minus the round-trip cost) is clustered by 10-calendar-day block, so trades of different products that start together, and trades that overlap in time, count as one cluster, not as independent evidence.
- **Plain trend benchmark.** Stage 2 breakouts are compared with all breakouts and with breakouts that merely sit above the EMA(200); Stage 2 must add to the plain trend filter in all three periods.
- **Splits.** Long and short, sector, volatility regime, trend regime, calendar year and product.
- **Leave-out.** One sector removed, one product removed, and the best 5 % of trades of each period removed.
- **Portfolio.** Each trade's net result is spread over the days it was held and at most 10 positions share the capital; Sharpe, drawdown and a block-10 t are shown for Stage 2, the parent, the plain-trend breakouts and a time-series trend benchmark (sign of the 60-day return, 1/volatility, same cost on turnover).

**Verdict (printed per lookback).** Robust = same sign and t > 2 in train_R and val_R, test t above the Bonferroni bar, better than the plain trend filter in all three periods, two thirds of the sectors holding in all three periods, no single sector carrying it, and a positive portfolio Sharpe in all three periods. The test period is read once.

**Plan after this step (user order).** 18u meta-labelling around the Stage 2 direction at 5-20 days (volume and squeeze set size and the entry gate; must beat ungated Stage 2 in train_R, val_R and test); 18v learn the signs of the 18n components on train_R and compare with the pasted signs; then entry and timing tools for the daily signal: range expansion with rising incremental volume (item 4), passive entry (item 6), regime and calendar conditioning (item 7).

In [ ]:
# step 18t
_KB_SAVED = dict(globals())          # snapshot: loop variables of this step must not overwrite a notebook name (restored at the end of the step)
KB_OK = bool(globals().get("KO_OK", False)) and all(k in globals() for k in ("KO_E", "KO_B", "KO_RT", "KN_SPLR", "bonferroni_t", "SECTOR", "RAW", "KO_T0", "OUT2", "FIG_DIR")) and "D" in globals().get("KO_E", {}) and "BRK" in globals().get("KO_E", {}).get("D", {})
if not KB_OK:
    print("Step 18t needs Step 18o (daily Stage 2 breakout events): run it first; skipped")
else:
    # ---- 18t.0 the trades: every daily breakout of Step 18o (structural stop, 10-day horizon), with the Stage 2 flag, the plain-trend flag, product, sector and year. Net = gross minus the round-trip cost ----
    KB_BLK = 10; KB_CAP = 10; KB_MAIN = "stop10"; KB_PL = list(KO_B["D"]); KB_SN = ("train_R", "val_R", "test"); KB_MINN = 20
    E = KO_E["D"]["BRK"].copy().reset_index(drop=True); E = E[E["spl"] >= 0].reset_index(drop=True)
    E["product"] = [KB_PL[i] for i in E["pid"]]; E["sector"] = E["product"].map(lambda p: SECTOR.get(p, "Other")); E["net"] = E["B_" + KB_MAIN] - KO_RT; E["Rn"] = E["R_" + KB_MAIN] - KO_RT / E["risk_bp"]
    E["year"] = (KO_T0 + pd.to_timedelta(E["tdi"], unit="D")).dt.year; E["is_s2"] = E["s2"] == 1
    # plain trend benchmark: the same breakout, with the price above its EMA(200) (in the direction of the trade) and nothing else; Stage 2 must add to this
    t200 = np.zeros(len(E), bool)
    for (pid_, dr_), g_ in E.groupby(["pid", "dir"]):
        d_ = KO_B["D"][KB_PL[pid_]]; x_ = dr_ * d_["cl"].to_numpy(float); em_ = pd.Series(x_).ewm(span=200, adjust=False, min_periods=200).mean().to_numpy(); fl_ = pd.Series(x_ > em_, index=d_["tdi"].to_numpy()); fl_ = fl_[~fl_.index.duplicated()]
        t200[g_.index.to_numpy()] = fl_.reindex(g_["tdi"].to_numpy()).fillna(False).to_numpy(bool)
    E["t200"] = t200; blk_ = (E["tdi"].to_numpy() // KB_BLK).astype(int); KB_BC = np.unique(blk_, return_inverse=True)[1].astype(np.int64); KB_NB = int(KB_BC.max()) + 1
    def kb_group(y, m, minn=KB_MINN):
        # mean and t clustered by 10-calendar-day block: trades of all products that start in the same block share a cluster, so overlapping and simultaneous trades are not counted as independent
        m = m & np.isfinite(y); n = int(m.sum())
        if n < minn: return n, np.nan, np.nan
        mu = float(y[m].mean()); s = np.bincount(KB_BC[m], weights=y[m] - mu, minlength=KB_NB); se = np.sqrt((s ** 2).sum()) / n
        return n, mu, (mu / se if se > 0 else np.nan)
    def kb_diff(y, ch, m, minn=30):
        ok = m & np.isfinite(y); a, b = ok & ch, ok & ~ch
        if a.sum() < minn or b.sum() < minn: return np.nan, np.nan
        ma, mb = y[a].mean(), y[b].mean(); w = np.where(a, 1 / a.sum(), np.where(b, -1 / b.sum(), 0.0)); e = np.where(a, y - ma, np.where(b, y - mb, 0.0)); se = np.sqrt((np.bincount(KB_BC, weights=w * e, minlength=KB_NB) ** 2).sum())
        return float(ma - mb), float((ma - mb) / se) if se > 0 else np.nan
    # unit check: ten simultaneous trades in one block are ONE cluster: the clustered t is far below the naive t
    _r = np.random.default_rng(4); _yy = _r.normal(size=2000) * 1.0 + 0.1; _bb = np.arange(2000) // 10; _bs = np.bincount(_bb, weights=_yy - _yy.mean()); _t_cl = _yy.mean() / (np.sqrt((_bs ** 2).sum()) / 2000); _t_nv = _yy.mean() / (_yy.std(ddof=1) / np.sqrt(2000)); assert abs(_t_cl) <= abs(_t_nv) * 1.6 and abs(_t_nv) > 1.5, "cluster t check"
    print(f"{len(E)} daily breakout trades; Stage 2: {int(E['is_s2'].sum())}; plain trend (above EMA200): {int(E['t200'].sum())}; block of {KB_BLK} calendar days -> {KB_NB} clusters; cost {KO_RT:.0f} bp round trip")
    ycol = {"net": E["net"].to_numpy(float), "Rn": E["Rn"].to_numpy(float)}; SPLv = E["spl"].to_numpy(int); Lv = E["L"].to_numpy(int)
    def kb_row(mask, **lab):
        r = dict(lab)
        for k, nm in enumerate(KB_SN):
            m = mask & (SPLv == k); n, mu, t = kb_group(ycol["net"], m); r[f"n_{nm}"] = n; r[f"net_{nm}"] = mu; r[f"t_{nm}"] = t
            r[f"R_{nm}"] = kb_group(ycol["Rn"], m)[1]; r[f"win_{nm}"] = float((ycol["net"][m] > 0).mean()) if m.sum() >= KB_MINN else np.nan
        r["holds"] = bool(all(np.isfinite(r[f"net_{nm}"]) and r[f"net_{nm}"] > 0 for nm in KB_SN)); return r
    # ---- 18t.1 the main comparison: all breakouts, breakouts above EMA200, Stage 2 breakouts; and whether Stage 2 adds to the plain trend filter ----
    KB_ROWS = []
    for L_ in (20, 50):
        for nm, mk in (("parent: every breakout", np.ones(len(E), bool)), ("plain trend: above EMA200", E["t200"].to_numpy()), ("Stage 2", E["is_s2"].to_numpy())):
            KB_ROWS.append(kb_row(mk & (Lv == L_), L=L_, cohort=nm))
        r = KB_ROWS[-1]
        for k, nm_ in enumerate(KB_SN):
            m = E["t200"].to_numpy() & (Lv == L_) & (SPLv == k); r[f"vs_trend_{nm_}"], r[f"vs_trend_t_{nm_}"] = kb_diff(ycol["net"], E["is_s2"].to_numpy(), m)
    KB_MAIN_T = pd.DataFrame(KB_ROWS); KB_MAIN_T.to_csv(OUT2 / "stage2_robust_main.csv", index=False)
    print("\nnet bp per trade after the round-trip cost (t clustered by 10-day block), by cohort and period:"); print(KB_MAIN_T[["L", "cohort"] + [f"{a}_{s}" for s in KB_SN for a in ("n", "net", "t")] + ["holds"]].round(2).to_string(index=False))
    print("\nwhat Stage 2 adds to the plain trend filter (difference in net bp among breakouts above EMA200):"); print(KB_MAIN_T[KB_MAIN_T["cohort"] == "Stage 2"][["L"] + [f"{a}_{s}" for s in KB_SN for a in ("vs_trend", "vs_trend_t")]].round(2).to_string(index=False))

In [ ]:
# step 18t
if KB_OK:
    # ---- 18t.2 splits of the Stage 2 trades: long / short, sector, volatility regime, trend regime, calendar year, product ----
    S2m = E["is_s2"].to_numpy(); KB_CUT = []
    CUTS = (("direction", E["dir"].map({1: "long", -1: "short"}).to_numpy()), ("sector", E["sector"].to_numpy()), ("volatility regime", E["vol"].map({0: "low", 1: "mid", 2: "high"}).to_numpy()), ("trend regime", E["trend"].map({0: "bear", 1: "range", 2: "bull"}).to_numpy()), ("year", E["year"].astype(str).to_numpy()), ("product", E["product"].to_numpy()))
    for L_ in (20, 50):
        for cname, cv in CUTS:
            for lev in sorted(set(cv.tolist())):
                KB_CUT.append(kb_row(S2m & (Lv == L_) & (cv == lev), L=L_, cut=cname, level=lev))
    KB_CUTT = pd.DataFrame(KB_CUT); KB_CUTT.to_csv(OUT2 / "stage2_robust_cuts.csv", index=False)
    for cname in ("direction", "sector", "volatility regime", "trend regime", "year"):
        z = KB_CUTT[(KB_CUTT["cut"] == cname) & (KB_CUTT["L"] == 20)]; print(f"\nStage 2 (L20) net bp by {cname}:"); print(z[["level"] + [f"{a}_{s}" for s in KB_SN for a in ("n", "net", "t")] + ["holds"]].round(2).to_string(index=False))
    # leave one sector out, leave one product out, and the mean after dropping the best 5 % of trades of each period
    KB_LOO = []
    for L_ in (20, 50):
        base = S2m & (Lv == L_)
        for sec in sorted(E["sector"].unique()): KB_LOO.append(kb_row(base & (E["sector"].to_numpy() != sec), L=L_, drop=f"sector {sec}"))
        for p in KB_PL: KB_LOO.append(kb_row(base & (E["product"].to_numpy() != p), L=L_, drop=f"product {p}"))
        r = kb_row(base, L=L_, drop="best 5% of trades in each period")
        for k, nm in enumerate(KB_SN):
            m = base & (SPLv == k); y = np.sort(ycol["net"][m]); r[f"net_{nm}"] = float(y[:int(len(y) * 0.95)].mean()) if len(y) >= 40 else np.nan; r[f"t_{nm}"] = np.nan
        r["holds"] = bool(all(np.isfinite(r[f"net_{nm}"]) and r[f"net_{nm}"] > 0 for nm in KB_SN)); KB_LOO.append(r)
    KB_LOOT = pd.DataFrame(KB_LOO); KB_LOOT.to_csv(OUT2 / "stage2_robust_leave_out.csv", index=False)
    print("\nStage 2 (L20) with one sector / product removed: does any single piece carry the edge?"); z = KB_LOOT[KB_LOOT["L"] == 20]; print(z[["drop", "net_train_R", "net_val_R", "net_test", "holds"]].round(1).to_string(index=False))

    # ---- 18t.3 the portfolio: every trade's net result spread over the days it was held; at most KB_CAP positions share the capital (weight 1 / max(active, cap)); compared with the parent breakouts, the plain-trend breakouts and a time-series trend benchmark (sign of the 60-day return, 1 / volatility, gross 1, cost per side on turnover) ----
    KB_TA = {pid: KO_B["D"][p]["tdi"].to_numpy() for pid, p in enumerate(KB_PL)}; KB_G = np.unique(np.concatenate(list(KB_TA.values()))); KB_DATES = KO_T0 + pd.to_timedelta(KB_G, unit="D")
    def kb_spread(ev_pid, ev_tdi, ev_n, ev_net, TA, G, cap):
        pnl = np.zeros(len(G)); act = np.zeros(len(G)); items = []
        for pid_, td_, n_, net_ in zip(ev_pid, ev_tdi, ev_n, ev_net):
            ta = TA[pid_]; ix = int(np.searchsorted(ta, td_)); days = ta[ix + 2: ix + 2 + int(n_)]
            if len(days) == 0 or not np.isfinite(net_): continue
            gi = np.searchsorted(G, days); act[gi] += 1; items.append((gi, net_ / len(days)))
        w = 1.0 / np.maximum(act, cap)
        for gi, pd_ in items: pnl[gi] += pd_ * w[gi]
        return pnl, act
    # unit check: one trade of 4 days netting 40 bp pays 10 bp x 1/cap a day for 4 days, starting two bars after the signal
    _TA = {0: np.arange(100, 130)}; _G = np.arange(100, 130); _p, _a = kb_spread([0], [105], [4], [40.0], _TA, _G, 10); assert np.allclose(_p[7:11], 1.0) and _p[:7].sum() == 0 and _p[11:].sum() == 0 and _a.max() == 1, "portfolio spreading wrong"
    print("unit check passed: portfolio spreading")
    spG = KN_SPLR.reindex(KB_DATES).fillna(-1).to_numpy().astype(int); KB_BOOK = {}
    def kb_stats(pnl, act, name):
        out = {}
        for k, nm in enumerate(KB_SN):
            m = spG == k; x = pnl[m]
            if m.sum() < 100: continue
            bs = np.add.reduceat(x, np.arange(0, len(x), KB_BLK)); bs = bs[:len(x) // KB_BLK]; c = np.cumsum(x)
            out.update({f"sharpe_{nm}": float(x.mean() / x.std() * np.sqrt(252)) if x.std() > 0 else np.nan, f"t_{nm}": float(bs.mean() / (bs.std(ddof=1) / np.sqrt(len(bs)))) if len(bs) > 5 and bs.std(ddof=1) > 0 else np.nan, f"bp_year_{nm}": float(x.mean() * 252), f"maxdd_bp_{nm}": float((np.maximum.accumulate(c) - c).max()), f"active_{nm}": float(act[m].mean()) if act is not None else np.nan})
        out["book"] = name; return out
    for L_ in (20, 50):
        for nm, mk in (("parent: every breakout", np.ones(len(E), bool)), ("plain trend: above EMA200", E["t200"].to_numpy()), ("Stage 2", E["is_s2"].to_numpy())):
            ev = E[mk & (Lv == L_)]; pnl, act = kb_spread(ev["pid"].to_numpy(), ev["tdi"].to_numpy(), ev["N_" + KB_MAIN].to_numpy(), ev["net"].to_numpy(), KB_TA, KB_G, KB_CAP); KB_BOOK[(L_, nm)] = (pnl, act)
    sig_ = RAW.ewm(halflife=30, min_periods=20).std().shift(1); m60_ = RAW.rolling(60, min_periods=40).sum().shift(1); w_ = (np.sign(m60_) / sig_).where(np.isfinite(m60_ / sig_)); w_ = w_.div(w_.abs().sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    tb_ = (w_ * RAW.fillna(0)).sum(axis=1) * 1e4 - w_.diff().abs().sum(axis=1) * KO_RT / 2; KB_TB = tb_.reindex(KB_DATES).fillna(0.0).to_numpy(); KB_BOOK[(0, "time-series trend benchmark (60d sign)")] = (KB_TB, None)
    KB_PT = pd.DataFrame([dict(L=L_, **kb_stats(pnl, act, nm)) for (L_, nm), (pnl, act) in KB_BOOK.items()]); KB_PT.to_csv(OUT2 / "stage2_robust_portfolio.csv", index=False)
    print(f"\nportfolio of the trades (at most {KB_CAP} positions share the capital; net of the round-trip cost): annualised Sharpe, t on 10-day blocks, drawdown in bp of the capital per position slot:"); print(KB_PT[["L", "book"] + [f"{a}_{s}" for s in KB_SN for a in ("sharpe", "t", "maxdd_bp")]].round(2).to_string(index=False))

In [ ]:
# step 18t
if KB_OK:
    # ---- 18t.4 verdict per lookback, figure, summary ----
    KB_V = []; bf2 = bonferroni_t(2)
    for L_ in (20, 50):
        m = KB_MAIN_T[(KB_MAIN_T["L"] == L_) & (KB_MAIN_T["cohort"] == "Stage 2")].iloc[0]; c = KB_CUTT[KB_CUTT["L"] == L_]; lo = KB_LOOT[(KB_LOOT["L"] == L_) & KB_LOOT["drop"].str.startswith("sector")]
        sd = c[c["cut"] == "direction"].set_index("level"); sc_ = c[(c["cut"] == "sector") & (c[[f"n_{s}" for s in KB_SN]].min(axis=1) >= KB_MINN)]; bk = KB_PT[(KB_PT["L"] == L_) & (KB_PT["book"] == "Stage 2")].iloc[0]; tb = KB_PT[KB_PT["book"].str.startswith("time-series")].iloc[0]
        v = dict(L=L_, net_bp_train_R=m["net_train_R"], net_bp_val_R=m["net_val_R"], net_bp_test=m["net_test"], t_test=m["t_test"],
                 cond_net=bool(m["holds"] and m["t_train_R"] > 2 and m["t_val_R"] > 2 and m["t_test"] > bf2), cond_adds_to_trend=bool(all(m[f"vs_trend_{s}"] > 0 for s in KB_SN)),
                 long_holds=bool(sd["holds"].get("long", False)), short_holds=bool(sd["holds"].get("short", False)), sectors_holding=f"{int(sc_['holds'].sum())}/{len(sc_)}", cond_sectors=bool(len(sc_) > 0 and sc_["holds"].mean() >= 2 / 3),
                 cond_leave_one_sector_out=bool(len(lo) > 0 and lo["holds"].all()), cond_book=bool(all(bk[f"sharpe_{s}"] > 0 for s in KB_SN)), book_sharpe_test=bk["sharpe_test"], benchmark_sharpe_test=tb["sharpe_test"], book_beats_benchmark_test=bool(bk["sharpe_test"] > tb["sharpe_test"]))
        v["robust"] = bool(v["cond_net"] and v["cond_adds_to_trend"] and v["cond_sectors"] and v["cond_leave_one_sector_out"] and v["cond_book"]); KB_V.append(v)
    KB_VT = pd.DataFrame(KB_V); KB_VT.to_csv(OUT2 / "stage2_robust_verdict.csv", index=False); print("\nverdict:"); print(KB_VT.T.to_string())
    fig, axs = plt.subplots(2, 2, figsize=(17, 9)); ax = axs[0, 0]
    for (L_, nm), (pnl, act) in KB_BOOK.items():
        if L_ in (0, 20): ax.plot(KB_DATES, np.cumsum(pnl), label=nm if L_ else nm, lw=1.2 if nm == "Stage 2" else 0.9)
    for k in (1, 2):
        ix_ = np.where(spG == k)[0]
        if len(ix_): ax.axvline(KB_DATES[ix_[0]], color="k", ls=":", lw=0.7)
    ax.legend(fontsize=7); ax.set_title("cumulative net bp of capital: L20 books and the trend benchmark (dotted: val_R and test start)")
    ax = axs[0, 1]; z = KB_CUTT[(KB_CUTT["L"] == 20) & KB_CUTT["cut"].isin(["direction", "volatility regime", "trend regime"])]; lab = (z["cut"].str[:5] + ": " + z["level"]).tolist(); xs = np.arange(len(z))
    for j, (s, col) in enumerate(zip(KB_SN, ("#1f3a5f", "#e0a030", "#9b2226"))): ax.bar(xs + (j - 1) * 0.27, z[f"net_{s}"].to_numpy(float), 0.27, label=s, color=col)
    ax.set_xticks(xs); ax.set_xticklabels(lab, rotation=60, fontsize=7); ax.axhline(0, color="k", lw=0.6); ax.legend(fontsize=7); ax.set_title("Stage 2 L20: net bp per trade by direction and regime")
    ax = axs[1, 0]; z = KB_CUTT[(KB_CUTT["L"] == 20) & (KB_CUTT["cut"] == "sector")]; xs = np.arange(len(z))
    for j, (s, col) in enumerate(zip(KB_SN, ("#1f3a5f", "#e0a030", "#9b2226"))): ax.bar(xs + (j - 1) * 0.27, z[f"net_{s}"].to_numpy(float), 0.27, color=col)
    ax.set_xticks(xs); ax.set_xticklabels(z["level"], rotation=60, fontsize=7); ax.axhline(0, color="k", lw=0.6); ax.set_title("Stage 2 L20: net bp per trade by sector")
    ax = axs[1, 1]; z = KB_LOOT[(KB_LOOT["L"] == 20) & KB_LOOT["drop"].str.startswith("sector")]; xs = np.arange(len(z))
    for j, (s, col) in enumerate(zip(KB_SN, ("#1f3a5f", "#e0a030", "#9b2226"))): ax.bar(xs + (j - 1) * 0.27, z[f"net_{s}"].to_numpy(float), 0.27, color=col)
    ax.set_xticks(xs); ax.set_xticklabels(z["drop"].str.replace("sector ", ""), rotation=60, fontsize=7); ax.axhline(0, color="k", lw=0.6); ax.set_title("Stage 2 L20 with one sector removed")
    fig.tight_layout(); fig.savefig(FIG_DIR / "18t_stage2_robustness.png", dpi=110); plt.show()
    def _f(x): return f"{x:.0f}" if np.isfinite(x) else "n/a"
    _p = KB_VT.iloc[0]; _q = KB_VT.iloc[1]
    KB_SUMMARY = dict(verdict=KB_VT.to_dict("records"), trades=int(len(E)), stage2_trades=int(E["is_s2"].sum()), cluster_blocks=KB_NB, cost_bp=float(KO_RT), bonferroni=float(bf2),
        reading=(f"Daily Stage 2 breakout as a portfolio, {len(E)} breakout trades ({int(E['is_s2'].sum())} in Stage 2), net of the {KO_RT:.0f} bp round trip, t clustered by {KB_BLK}-day block so simultaneous and overlapping trades are not counted as independent. "
                 f"L20: net bp per trade {_f(_p['net_bp_train_R'])} / {_f(_p['net_bp_val_R'])} / {_f(_p['net_bp_test'])} (train_R / val_R / test), test t {_p['t_test']:.1f}; adds to the plain above-EMA200 trend filter in all three periods: {_p['cond_adds_to_trend']}; long holds: {_p['long_holds']}, short holds: {_p['short_holds']}; sectors holding in all three periods: {_p['sectors_holding']}; survives dropping any one sector: {_p['cond_leave_one_sector_out']}; portfolio Sharpe on the test {_p['book_sharpe_test']:.2f} against {_p['benchmark_sharpe_test']:.2f} for the 60-day time-series trend benchmark. Verdict L20 robust: {_p['robust']}. "
                 f"L50: net bp {_f(_q['net_bp_train_R'])} / {_f(_q['net_bp_val_R'])} / {_f(_q['net_bp_test'])}, test t {_q['t_test']:.1f}, robust: {_q['robust']}. Robust means all of: same sign and t>2 in train_R and val_R, test t above Bonferroni {bf2:.2f}, better than the plain trend filter in all three periods, two thirds of the sectors holding, no single sector carrying it, positive portfolio Sharpe in all three periods."))
    for _k_, _v_ in _KB_SAVED.items():
        if not _k_.startswith("KB_") and _k_ in globals() and globals()[_k_] is not _v_ and _k_ not in ("_KB_SAVED",): globals()[_k_] = _v_

## Step 18u — RSI-marked entries in bull and bear regimes (long in bull, short in bear)

**Idea.** Trade the bull regime on the long side and the bear regime on the short side, and use the RSI as the marker for the entry: long when the RSI crosses back **up** through a level inside a bull regime (a pullback that ends), short when it crosses back **down** through a level inside a bear regime (a rally that ends).

**RSI.** Wilder RSI with 14 periods on the close of each frame (daily, 60-minute, 30-minute). TA-Lib is used when it is installed (the notebook installs it from a wheel in an attached dataset, or from the internet when that is on); otherwise the same Wilder RSI is computed in numpy. A unit check compares the numpy version with a plain Wilder loop and, when present, with TA-Lib.

**Regime.** The 20-day mean of the daily RSI, known the day before for intraday bars. Three versions: *RSI learned* (terciles of that mean, cut points learned per product on train_R), *RSI pasted* (above 55 bull, below 45 bear) and *Stage 2* (long / short Stage 2 as bull / bear).

**Entry level.** Three modes (the optimum is also printed and saved for every product and every sector, see below): *pasted* (long on a cross up through 30, short on a cross down through 70), *pooled* (the level of 30-55 for longs, 70-45 for shorts with the best train_R net result, at least 200 events) and *product* (the same chosen per product, at least 40 train_R events, else the pooled level). The Spearman correlation between the train_R-chosen and the val_R-chosen level per product shows whether a level repeats.

**Outcome.** Entry at the close of the bar after the cross; the move over 5, 10, 20 days (daily frame), 1 and 5 days (60-minute), and one bar (the 30 minutes right after the cross), 1 and 5 days (30-minute frame), so the 30-minute decay is read directly. Net = direction × move − the round-trip cost. t is clustered by 10-calendar-day block.

**Reading.** The *parent* is the same regime entered on every bar, with the same exit horizon. A cell is *stable* if net is positive in all three periods, t > 2 in train_R and val_R and the test t is above the Bonferroni bar; it is *supported* if it also beats its parent in all three periods. The response curve shows net bp by RSI level, so the shape (not only the chosen level) can be read. The test period is read once.

**Optimal level per product and per sector.** For every product, frame, horizon, direction and regime source the RSI level (long: 30-55, short: 70-45) with the best train_R net result is chosen (at least 40 train_R events; the level chosen on val_R is shown next to it), and its events and mean net bp in train_R, val_R and test are reported with the same regime entered on every bar as the parent. The same is done pooled per sector (at least 100 train_R events). `rsi_optimal_level_by_product.csv` and `rsi_optimal_level_by_sector.csv` list them grouped by sector. The level is a train_R choice: read the val_R and test columns, not the train_R column, to judge it.

In [ ]:
# step 18u
_KU_SAVED = dict(globals())          # snapshot: loop variables of this step must not overwrite a notebook name (restored at the end of the step)
KU_OK = bool(globals().get("KP_OK", False)) and all(k in globals() for k in ("KP_B", "ko_stage", "KO_T0", "KN_SPLR", "bonferroni_t", "causal_test", "KO_RT", "SECTOR", "OUT2", "FIG_DIR"))
if not KU_OK:
    print("Step 18u needs Step 18p (frames) and Step 18o (Stage 2): run them first; skipped")
else:
    # ---- 18u.0 RSI (TA-Lib when it is installed, the same Wilder RSI written in numpy otherwise), bull / bear regime from the daily RSI, and RSI crosses as the marker for long and short entries ----
    try:
        import talib as _talib
    except Exception:
        _talib = None
    from scipy.signal import lfilter
    KU_N = 14; KU_FR = ("D", "60m", "30m"); KU_GL = (30, 35, 40, 45, 50, 55); KU_GS = (70, 65, 60, 55, 50, 45); KU_BLK = 10; KU_MINP = 200; KU_MINQ = 40; KU_PASTE = {"long": 30, "short": 70}; KU_PCUT = (45, 55); KU_SN = ("train_R", "val_R", "test")
    def ku_rsi_own(c, n=14):
        c = np.asarray(c, float); out = np.full(len(c), np.nan)
        if len(c) <= n + 1 or not np.isfinite(c).all(): return out
        d = np.diff(c); g = np.maximum(d, 0.0); l = np.maximum(-d, 0.0); a = 1.0 / n; sg, sl = g[:n].mean(), l[:n].mean()
        yg, _ = lfilter([a], [1, -(1 - a)], g[n:], zi=[(1 - a) * sg]); yl, _ = lfilter([a], [1, -(1 - a)], l[n:], zi=[(1 - a) * sl]); ag = np.r_[sg, yg]; al = np.r_[sl, yl]
        out[n:] = np.where(al > 0, 100.0 - 100.0 / (1.0 + ag / np.where(al > 0, al, 1.0)), 100.0); return out
    def ku_rsi(c, n=KU_N):
        c = pd.Series(np.asarray(c, float)).ffill().to_numpy(); k0 = int(np.argmax(np.isfinite(c))) if np.isfinite(c).any() else len(c); out = np.full(len(c), np.nan)
        if k0 >= len(c): return out
        out[k0:] = _talib.RSI(np.ascontiguousarray(c[k0:]), timeperiod=n) if _talib is not None else ku_rsi_own(c[k0:], n); return out
    # unit checks: Wilder RSI against a plain loop, bounds, a rising series = 100, and TA-Lib when present
    def _slow(c, n):
        o = np.full(len(c), np.nan); d = np.diff(c); ag = np.maximum(d[:n], 0).mean(); al = np.maximum(-d[:n], 0).mean(); o[n] = 100 - 100 / (1 + ag / al) if al > 0 else 100.0
        for i in range(n, len(d)):
            ag = (ag * (n - 1) + max(d[i], 0)) / n; al = (al * (n - 1) + max(-d[i], 0)) / n; o[i + 1] = 100 - 100 / (1 + ag / al) if al > 0 else 100.0
        return o
    _c = np.cumsum(np.random.default_rng(7).normal(size=400)) + 100; assert np.nanmax(np.abs(ku_rsi_own(_c) - _slow(_c, 14))) < 1e-8, "RSI differs from the plain Wilder loop"; assert np.nanmin(ku_rsi_own(_c)) >= 0 and np.nanmax(ku_rsi_own(_c)) <= 100 and abs(ku_rsi_own(np.arange(60.0))[-1] - 100) < 1e-9, "RSI bounds"
    if _talib is not None: assert np.nanmax(np.abs(_talib.RSI(_c, timeperiod=14) - ku_rsi_own(_c))) < 1e-6, "numpy RSI differs from TA-Lib"
    print("RSI backend:", "TA-Lib " + getattr(_talib, "__version__", "") if _talib is not None else "numpy Wilder RSI (TA-Lib not installed; identical to TA-Lib by the unit check)", "| unit checks passed")
    def ku_pit(g, FR="30m"):
        r = ku_rsi(g["cl"].to_numpy(float)); return pd.DataFrame({"rsi": r, "rsip": np.r_[np.nan, r[:-1]]}, index=g.index)
    if KP_B["30m"]:
        _p0 = list(KP_B["30m"])[0]; causal_test("ku_rsi_30m", ku_pit, KP_B["30m"][_p0].iloc[-4000:], step="18u")
    KU_BPD = {}
    for FR in KU_FR:
        if KP_B.get(FR): KU_BPD[FR] = 1 if FR == "D" else int(max(1, round(float(np.median([v.groupby("tdi").size().median() for v in KP_B[FR].values()])))))
    KU_HB = {"D": {"5d": 5, "10d": 10, "20d": 20}}; KU_HB.update({f: {"1d": KU_BPD[f], "5d": 5 * KU_BPD[f]} for f in ("60m", "30m") if f in KU_BPD})
    if "30m" in KU_BPD: KU_HB["30m"] = {"30m": 1, "1d": KU_BPD["30m"], "5d": 5 * KU_BPD["30m"]}          # 30m: the bar after the cross (inside 30 minutes) against one and five days (beyond)
    KU_MAINH = {"D": "10d", "60m": "1d", "30m": "1d"}
    # ---- 18u.1 panel per frame: RSI, regime (learned terciles of the 20-day mean daily RSI, cuts from train_R per product; and the pasted 45 / 55), Stage 2, forward moves in bp from the close after the cross ----
    KU_P = {}
    for FR in KU_BPD:
        cols = {k: [] for k in ("rsi", "rsip", "regL", "regP", "s2")}; fw = {h: [] for h in KU_HB[FR]}; meta = {k: [] for k in ("spl", "tdi", "pid")}; KU_PL = list(KP_B["D"])
        for p, d in KP_B[FR].items():
            if p not in KP_B["D"] or len(d) < 600: continue
            dd = KP_B["D"][p]; n = len(d); C = d["cl"].to_numpy(float); tdi = d["tdi"].to_numpy(); dts = pd.DatetimeIndex(KO_T0 + pd.to_timedelta(tdi, unit="D")); rs = ku_rsi(C)
            rd = ku_rsi(dd["cl"].to_numpy(float)); m20 = pd.Series(rd, index=dd.index).rolling(20, min_periods=15).mean(); trn = (KN_SPLR.reindex(dd.index).fillna(-1).to_numpy() == 0) & m20.notna().to_numpy()
            if trn.sum() < 250: continue
            c1, c2 = np.quantile(m20.to_numpy()[trn], [1 / 3, 2 / 3]); rl = pd.Series(np.where(m20.isna(), -1, (m20 > c1).astype(int) + (m20 > c2).astype(int)), index=dd.index); rp = pd.Series(np.where(m20.isna(), -1, np.where(m20 > KU_PCUT[1], 2, np.where(m20 < KU_PCUT[0], 0, 1))), index=dd.index)
            s2 = ko_stage(dd, 1)["s2"].fillna(0) - ko_stage(dd, -1)["s2"].fillna(0); S = pd.DataFrame(dict(regL=rl.astype(float), regP=rp.astype(float), s2=s2.astype(float)), index=dd.index)
            if FR != "D": S = S.shift(1); M = S.reindex(dts, method="ffill")
            else: M = S.reindex(d.index)
            sp = KN_SPLR.reindex(dts).fillna(-1).to_numpy().astype(int); rb = d["rollb"].fillna(0).to_numpy() > 0; keep = (sp >= 0)
            cols["rsi"].append(rs[keep].astype(np.float32)); cols["rsip"].append(np.r_[np.nan, rs[:-1]][keep].astype(np.float32))
            for k in ("regL", "regP"): cols[k].append(np.nan_to_num(M[k].to_numpy(float), nan=-1).astype(np.int8)[keep])
            cols["s2"].append(np.nan_to_num(M["s2"].to_numpy(float), nan=0).astype(np.int8)[keep])
            for lab, hb in KU_HB[FR].items():
                a = np.full(n, np.nan)
                if n > hb + 3: ix = np.arange(n - 2 - hb); a[ix] = (C[ix + 1 + hb] - C[ix + 1]) * 1e4          # entry at the close of the bar after the cross
                a[rb | np.r_[False, rb[:-1]]] = np.nan; fw[lab].append(a[keep].astype(np.float32))
            meta["spl"].append(sp[keep]); meta["tdi"].append(tdi[keep]); meta["pid"].append(np.full(int(keep.sum()), KU_PL.index(p)))
        if not meta["spl"]: continue
        P = {k: np.concatenate(v) for k, v in cols.items()}; P.update({"y_" + k: np.concatenate(v) for k, v in fw.items()}); P.update({k: np.concatenate(v) for k, v in meta.items()}); P["blk"] = np.unique(P["tdi"] // KU_BLK, return_inverse=True)[1].astype(np.int64); KU_P[FR] = P
    print({FR: len(P["spl"]) for FR, P in KU_P.items()}, "rows per frame;", KU_BPD, "bars per day")

In [ ]:
# step 18u
if KU_OK and KU_P:
    # ---- 18u.2 evaluation. Bull regime -> long entries when the RSI crosses back UP through a level; bear regime -> short entries when the RSI crosses back DOWN through a level. The level is pasted (30 / 70), learned pooled on train_R, or learned per product on train_R (pooled level when the product has fewer than 40 train_R events). Parent = every bar in the regime; the child (RSI cross) must beat it in all three periods ----
    KU_SRC = (("rsi_learned", "RSI regime, cuts learned on train_R (terciles of the 20-day mean daily RSI)"), ("rsi_pasted", "RSI regime, 20-day mean RSI above 55 / below 45"), ("s2", "Stage 2 long / short as the regime"))
    def ku_group(y, m, blk, nb, minn=30):
        m = m & np.isfinite(y); n = int(m.sum())
        if n < minn: return n, np.nan, np.nan
        mu = float(y[m].mean()); s = np.bincount(blk[m], weights=y[m] - mu, minlength=nb); se = np.sqrt((s ** 2).sum()) / n
        return n, mu, (mu / se if se > 0 else np.nan)
    def ku_diff(y, ch, m, blk, nb, minn=30):
        ok = m & np.isfinite(y); a, b = ok & ch, ok & ~ch
        if a.sum() < minn or b.sum() < minn: return np.nan, np.nan
        ma, mb = y[a].mean(), y[b].mean(); w = np.where(a, 1 / a.sum(), np.where(b, -1 / b.sum(), 0.0)); e = np.where(a, y - ma, np.where(b, y - mb, 0.0)); se = np.sqrt((np.bincount(blk, weights=w * e, minlength=nb) ** 2).sum())
        return float(ma - mb), float((ma - mb) / se) if se > 0 else np.nan
    # unit check: a cross through 40 is detected on the bar that crosses, not the one before
    _a = np.array([35., 38., 41., 45.]); _ap = np.r_[np.nan, _a[:-1]]; assert list(np.where((_ap < 40) & (_a >= 40))[0]) == [2], "cross detection"
    def ku_regime(P, src, dirn):
        if src == "rsi_learned": return P["regL"] == (2 if dirn > 0 else 0)
        if src == "rsi_pasted": return P["regP"] == (2 if dirn > 0 else 0)
        return P["s2"] == (1 if dirn > 0 else -1)
    def ku_cross(P, dirn, lev):
        return (P["rsip"] < lev) & (P["rsi"] >= lev) if dirn > 0 else (P["rsip"] > lev) & (P["rsi"] <= lev)
    def ku_levels(P, reg, y, dirn, grid, pid, npid, splitk, minq):
        # mean net bp by level, pooled and per product, in one split
        sp = P["spl"] == splitk; pooled = []; byp = []
        for lev in grid:
            m = reg & sp & ku_cross(P, dirn, lev) & np.isfinite(y); n = int(m.sum()); pooled.append((n, float(y[m].mean()) if n else np.nan))
            cnt = np.bincount(pid[m], minlength=npid); sm = np.bincount(pid[m], weights=y[m], minlength=npid); byp.append((cnt, np.where(cnt > 0, sm / np.maximum(cnt, 1), np.nan)))
        return pooled, byp
    KU_ROWS, KU_REPRO, KU_LEVELS = [], [], {}
    for FR, P in KU_P.items():
        nb = int(P["blk"].max()) + 1; pid = P["pid"].astype(np.int64); npid = int(pid.max()) + 1; SP = P["spl"]
        for hl in KU_HB[FR]:
            yg = P["y_" + hl].astype(float)
            for dirn, dname, grid in ((1, "long in bull", KU_GL), (-1, "short in bear", KU_GS)):
                yn = dirn * yg - KO_RT
                for src, srcdoc in KU_SRC:
                    reg = ku_regime(P, src, dirn); parent = reg & np.isfinite(yn)
                    if parent.sum() < 200: continue
                    pt, bt = ku_levels(P, reg, yn, dirn, grid, pid, npid, 0, KU_MINQ); pv, bv = ku_levels(P, reg, yn, dirn, grid, pid, npid, 1, KU_MINQ)
                    ok_ = [i for i, (n, mu) in enumerate(pt) if n >= KU_MINP and np.isfinite(mu)]; lev_pool = grid[max(ok_, key=lambda i: pt[i][1])] if ok_ else KU_PASTE["long" if dirn > 0 else "short"]
                    lev_prod = np.full(npid, float(lev_pool)); lev_val = np.full(npid, np.nan)
                    for pp in range(npid):
                        c_ = [(bt[i][1][pp], i) for i in range(len(grid)) if bt[i][0][pp] >= KU_MINQ and np.isfinite(bt[i][1][pp])]
                        if c_: lev_prod[pp] = grid[max(c_)[1]]
                        c2_ = [(bv[i][1][pp], i) for i in range(len(grid)) if bv[i][0][pp] >= KU_MINQ and np.isfinite(bv[i][1][pp])]
                        if c2_: lev_val[pp] = grid[max(c2_)[1]]
                    got = np.isfinite(lev_val) & np.array([any(bt[i][0][pp] >= KU_MINQ for i in range(len(grid))) for pp in range(npid)])
                    if got.sum() >= 8 and len(set(lev_prod[got])) > 1 and len(set(lev_val[got])) > 1: KU_REPRO.append(dict(frame=FR, h=hl, dir=dname, source=src, products=int(got.sum()), spearman_train_vs_val_level=float(stats.spearmanr(lev_prod[got], lev_val[got])[0])))
                    KU_LEVELS[(FR, hl, dname, src)] = (lev_pool, lev_prod)
                    for mode, levarr in (("pasted", np.full(npid, float(KU_PASTE["long" if dirn > 0 else "short"]))), ("pooled", np.full(npid, float(lev_pool))), ("product", lev_prod)):
                        L = levarr[pid]; ch = reg & ((P["rsip"] < L) & (P["rsi"] >= L) if dirn > 0 else (P["rsip"] > L) & (P["rsi"] <= L)) & np.isfinite(yn); r = dict(frame=FR, h=hl, direction=dname, source=src, mode=mode, level=(float(levarr[0]) if mode != "product" else np.nan))
                        for k, nm in enumerate(KU_SN):
                            m = SP == k; n, mu, t = ku_group(yn, ch & m, P["blk"], nb); n0, m0, t0 = ku_group(yn, parent & m, P["blk"], nb); d_, dt_ = ku_diff(yn, ch, parent & m, P["blk"], nb)
                            r.update({f"n_{nm}": n, f"net_{nm}": mu, f"t_{nm}": t, f"parent_net_{nm}": m0, f"parent_t_{nm}": t0, f"diff_{nm}": d_, f"diff_t_{nm}": dt_})
                        KU_ROWS.append(r)
    KU_T = pd.DataFrame(KU_ROWS); nb_ = max(len(KU_T), 1); bf_ = bonferroni_t(nb_)
    KU_T["same_sign"] = (KU_T[[f"net_{s}" for s in KU_SN]] > 0).all(axis=1); KU_T["beats_parent"] = (KU_T[[f"diff_{s}" for s in KU_SN]] > 0).all(axis=1)
    KU_T["stable"] = KU_T["same_sign"] & (KU_T["t_train_R"] > 2) & (KU_T["t_val_R"] > 2) & (KU_T["t_test"] > bf_); KU_T["supported"] = KU_T["stable"] & KU_T["beats_parent"]; KU_T["breakeven_x_cost_test"] = (KU_T["net_test"] + KO_RT) / KO_RT
    KU_T.to_csv(OUT2 / "rsi_regime_entries.csv", index=False); KU_REP = pd.DataFrame(KU_REPRO); KU_REP.to_csv(OUT2 / "rsi_level_reproducibility.csv", index=False)
    print(f"{len(KU_T)} cells (frame x horizon x direction x regime source x level mode); Bonferroni t on the test {bf_:.2f}; positive net in all three periods: {int(KU_T['same_sign'].sum())}; stable: {int(KU_T['stable'].sum())}; stable and better than the all-bars-in-regime parent: {int(KU_T['supported'].sum())}")
    _c = ["frame", "h", "direction", "source", "mode", "level", "net_train_R", "net_val_R", "net_test", "t_test", "parent_net_test", "diff_test", "beats_parent", "breakeven_x_cost_test"]
    print("\nstrongest cells by test t among those positive in all three periods:"); z = KU_T[KU_T["same_sign"]].sort_values("t_test", ascending=False); print(z[_c].head(25).round(2).to_string(index=False) if len(z) else "none")
    print("\nlearned RSI level: does the train_R choice repeat on val_R? (Spearman of the per-product level)"); print(KU_REP.round(2).to_string(index=False) if len(KU_REP) else "too few products with events in both periods")

    # ---- 18u.2b the optimal RSI level for every product and every sector (chosen on train_R only; val_R-best level shown beside it; val_R and test results at the train_R level) ----
    KU_PRODROWS, KU_SECROWS = [], []
    for FR, P in KU_P.items():
        nb = int(P["blk"].max()) + 1; pid = P["pid"].astype(np.int64); npid = int(pid.max()) + 1; SP = P["spl"]; secv = np.array([SECTOR.get(p, "Other") for p in KU_PL]); secid = pd.factorize(secv)[0]; sec_row = secid[pid]
        for hl in KU_HB[FR]:
            yg = P["y_" + hl].astype(float)
            for dirn, dname, grid in ((1, "long in bull", KU_GL), (-1, "short in bear", KU_GS)):
                yn = dirn * yg - KO_RT
                for src, _ in KU_SRC:
                    key = (FR, hl, dname, src)
                    if key not in KU_LEVELS: continue
                    reg = ku_regime(P, src, dirn); lev_pool, lev_prod = KU_LEVELS[key]; _, bv = ku_levels(P, reg, yn, dirn, grid, pid, npid, 1, KU_MINQ); _, bt = ku_levels(P, reg, yn, dirn, grid, pid, npid, 0, KU_MINQ)
                    L = lev_prod[pid]; ch = reg & ((P["rsip"] < L) & (P["rsi"] >= L) if dirn > 0 else (P["rsip"] > L) & (P["rsi"] <= L)) & np.isfinite(yn); par = reg & np.isfinite(yn)
                    cn = {k: np.bincount(pid[ch & (SP == k)], minlength=npid) for k in range(3)}; cs = {k: np.bincount(pid[ch & (SP == k)], weights=yn[ch & (SP == k)], minlength=npid) for k in range(3)}; pn = np.bincount(pid[par & (SP == 2)], minlength=npid); ps = np.bincount(pid[par & (SP == 2)], weights=yn[par & (SP == 2)], minlength=npid)
                    for pp in range(npid):
                        ntr = max((bt[i][0][pp] for i in range(len(grid))), default=0); lv_val = np.nan; cv = [(bv[i][1][pp], i) for i in range(len(grid)) if bv[i][0][pp] >= KU_MINQ and np.isfinite(bv[i][1][pp])]
                        if cv: lv_val = grid[max(cv)[1]]
                        if cn[0][pp] == 0 and cn[1][pp] == 0 and cn[2][pp] == 0: continue
                        KU_PRODROWS.append(dict(frame=FR, h=hl, direction=dname, source=src, sector=SECTOR.get(KU_PL[pp], "Other"), product=KU_PL[pp], level_train_R=float(lev_prod[pp]), level_chosen_on_val_R=lv_val, level_pooled=float(lev_pool), from_product=bool(ntr >= KU_MINQ),
                                                n_train_R=int(cn[0][pp]), net_train_R=float(cs[0][pp] / cn[0][pp]) if cn[0][pp] else np.nan, n_val_R=int(cn[1][pp]), net_val_R=float(cs[1][pp] / cn[1][pp]) if cn[1][pp] else np.nan, n_test=int(cn[2][pp]), net_test=float(cs[2][pp] / cn[2][pp]) if cn[2][pp] else np.nan, parent_net_test=float(ps[pp] / pn[pp]) if pn[pp] else np.nan))
                    for sid, sname in enumerate(pd.factorize(secv)[1]):
                        sm = (sec_row == sid); best = None
                        for lev in grid:
                            m = reg & sm & (SP == 0) & ku_cross(P, dirn, lev) & np.isfinite(yn)
                            if m.sum() >= 100 and (best is None or yn[m].mean() > best[1]): best = (lev, float(yn[m].mean()))
                        if best is None: continue
                        r = dict(frame=FR, h=hl, direction=dname, source=src, sector=sname, level_train_R=float(best[0]), products=int(len(set(pid[sm]))))
                        for k, nm in enumerate(KU_SN):
                            n, mu, t = ku_group(yn, reg & sm & (SP == k) & ku_cross(P, dirn, best[0]), P["blk"], nb); n0, m0, t0 = ku_group(yn, reg & sm & (SP == k), P["blk"], nb); r.update({f"n_{nm}": n, f"net_{nm}": mu, f"t_{nm}": t, f"parent_net_{nm}": m0})
                        KU_SECROWS.append(r)
    KU_OPTP = pd.DataFrame(KU_PRODROWS).sort_values(["frame", "h", "direction", "source", "sector", "product"]); KU_OPTS = pd.DataFrame(KU_SECROWS).sort_values(["frame", "h", "direction", "source", "sector"])
    KU_OPTP.to_csv(OUT2 / "rsi_optimal_level_by_product.csv", index=False); KU_OPTS.to_csv(OUT2 / "rsi_optimal_level_by_sector.csv", index=False)
    _fr = "D" if "D" in KU_P else list(KU_P)[0]; _h = KU_MAINH[_fr]
    for dname in ("long in bull", "short in bear"):
        z = KU_OPTS[(KU_OPTS["frame"] == _fr) & (KU_OPTS["h"] == _h) & (KU_OPTS["direction"] == dname) & (KU_OPTS["source"] == "rsi_learned")]; print(f"\noptimal RSI level per sector ({_fr}, hold {_h}, {dname}, RSI regime; level chosen on train_R):"); print(z[["sector", "products", "level_train_R", "n_train_R", "net_train_R", "n_val_R", "net_val_R", "n_test", "net_test", "t_test", "parent_net_test"]].round(1).to_string(index=False))
        z = KU_OPTP[(KU_OPTP["frame"] == _fr) & (KU_OPTP["h"] == _h) & (KU_OPTP["direction"] == dname) & (KU_OPTP["source"] == "rsi_learned")]; print(f"\noptimal RSI level per product, by sector ({_fr}, hold {_h}, {dname}):"); print(z[["sector", "product", "level_train_R", "level_chosen_on_val_R", "n_train_R", "net_train_R", "n_val_R", "net_val_R", "n_test", "net_test", "parent_net_test"]].round(1).to_string(index=False))
    print("\nall frames, horizons, regime sources: rsi_optimal_level_by_product.csv and rsi_optimal_level_by_sector.csv")

In [ ]:
# step 18u
if KU_OK and KU_P and len(KU_T):
    # ---- 18u.3 response curve (net bp by RSI cross level), figure, summary ----
    KU_CURVE = []
    for FR, P in KU_P.items():
        hl = KU_MAINH.get(FR, list(KU_HB[FR])[0]); yg = P["y_" + hl].astype(float); nb = int(P["blk"].max()) + 1
        for dirn, dname, grid in ((1, "long in bull", KU_GL), (-1, "short in bear", KU_GS)):
            yn = dirn * yg - KO_RT; reg = ku_regime(P, "rsi_learned", dirn) & np.isfinite(yn)
            for k, nm in enumerate(KU_SN):
                n0, m0, t0 = ku_group(yn, reg & (P["spl"] == k), P["blk"], nb); KU_CURVE.append(dict(frame=FR, h=hl, direction=dname, level="parent (every bar in regime)", split=nm, n=n0, net=m0, t=t0))
                for lev in grid: n, mu, t = ku_group(yn, reg & (P["spl"] == k) & ku_cross(P, dirn, lev), P["blk"], nb); KU_CURVE.append(dict(frame=FR, h=hl, direction=dname, level=str(lev), split=nm, n=n, net=mu, t=t))
    KU_CV = pd.DataFrame(KU_CURVE); KU_CV.to_csv(OUT2 / "rsi_response_curve.csv", index=False)
    fig, axs = plt.subplots(2, 3, figsize=(19, 9)); fr0 = "D" if "D" in KU_P else list(KU_P)[0]
    for j, (dname, grid) in enumerate((("long in bull", KU_GL), ("short in bear", KU_GS))):
        ax = axs[0, j]; z = KU_CV[(KU_CV["frame"] == fr0) & (KU_CV["direction"] == dname)]
        for nm, col in zip(KU_SN, ("#1f3a5f", "#e0a030", "#9b2226")):
            zz = z[(z["split"] == nm) & (z["level"] != "parent (every bar in regime)")]; ax.plot(zz["level"].astype(float), zz["net"], marker="o", color=col, label=nm); par = z[(z["split"] == nm) & (z["level"] == "parent (every bar in regime)")]["net"]
            if len(par): ax.axhline(float(par.iloc[0]), color=col, ls="--", lw=0.8)
        ax.axhline(0, color="k", lw=0.6); ax.set_xlabel(("RSI crosses UP through" if dname.startswith("long") else "RSI crosses DOWN through")); ax.set_ylabel("net bp per trade"); ax.legend(fontsize=7); ax.set_title(f"{fr0} {KU_MAINH[fr0]}: {dname}; dashed = every bar in the regime", fontsize=9)
    ax = axs[0, 2]; zz = KU_T[(KU_T["source"] == "rsi_learned") & (KU_T["mode"] == "pooled")]; lab = (zz["frame"] + "/" + zz["h"] + "\n" + zz["direction"].str.split().str[0]).tolist(); xs = np.arange(len(zz))
    ax.bar(xs - 0.2, zz["net_test"], 0.4, label="RSI cross (test)", color="#9b2226"); ax.bar(xs + 0.2, zz["parent_net_test"], 0.4, label="every bar in regime (test)", color="#999999"); ax.set_xticks(xs); ax.set_xticklabels(lab, fontsize=6.5, rotation=60); ax.axhline(0, color="k", lw=0.6); ax.legend(fontsize=7); ax.set_title("test net bp: RSI cross against its parent (RSI regime, pooled level)", fontsize=9)
    ax = axs[1, 0]; vals = [KU_LEVELS[k][1] for k in KU_LEVELS if k[0] == fr0 and k[1] == KU_MAINH[fr0] and k[3] == "rsi_learned" and k[2] == "long in bull"]
    if vals: ax.hist(vals[0], bins=np.arange(27.5, 60, 5), color="#2e8b57"); ax.axvline(KU_PASTE["long"], color="k", ls="--"); ax.set_title(f"long: per-product learned level (dashed = pasted {KU_PASTE['long']})", fontsize=9)
    ax = axs[1, 1]; vals = [KU_LEVELS[k][1] for k in KU_LEVELS if k[0] == fr0 and k[1] == KU_MAINH[fr0] and k[3] == "rsi_learned" and k[2] == "short in bear"]
    if vals: ax.hist(vals[0], bins=np.arange(42.5, 75, 5), color="#d9534f"); ax.axvline(KU_PASTE["short"], color="k", ls="--"); ax.set_title(f"short: per-product learned level (dashed = pasted {KU_PASTE['short']})", fontsize=9)
    ax = axs[1, 2]; z = KU_T.groupby(["frame", "source"])[["stable", "supported"]].sum().reset_index(); ax.bar(np.arange(len(z)) - 0.2, z["stable"], 0.4, label="stable"); ax.bar(np.arange(len(z)) + 0.2, z["supported"], 0.4, label="stable and beats parent"); ax.set_xticks(np.arange(len(z))); ax.set_xticklabels(z["frame"] + "\n" + z["source"], fontsize=7); ax.legend(fontsize=7); ax.set_title("cells passing, by frame and regime source", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "18u_rsi_regime_entries.png", dpi=110); plt.show()
    _s = KU_T[KU_T["supported"]]; _bs = _s.sort_values("t_test", ascending=False).head(3); _rep = float(KU_REP["spearman_train_vs_val_level"].median()) if len(KU_REP) else float("nan")
    _be = ", ".join(f"{r.frame} {r.h} {r.direction} ({r.source}, {r.mode}): {r.net_train_R:.0f} / {r.net_val_R:.0f} / {r.net_test:.0f} bp, {r.breakeven_x_cost_test:.1f}x cost" for r in _bs.itertuples())
    KU_SUMMARY = dict(backend=("TA-Lib" if _talib is not None else "numpy Wilder RSI"), cells=int(len(KU_T)), bonferroni=float(bf_), positive_all_periods=int(KU_T["same_sign"].sum()), stable=int(KU_T["stable"].sum()), supported=int(KU_T["supported"].sum()), level_reproducibility_median_spearman=_rep,
        reading=(f"RSI-marked entries in bull and bear regimes ({'TA-Lib' if _talib is not None else 'numpy Wilder'} RSI, 14 periods; regime from the 20-day mean of the daily RSI, cuts learned per product on train_R, or Stage 2): long when the RSI crosses back up through a level in a bull regime, short when it crosses back down in a bear regime; the level is pasted ({KU_PASTE['long']} / {KU_PASTE['short']}), learned pooled, or learned per product on train_R. {len(KU_T)} cells; {int(KU_T['same_sign'].sum())} are positive net of the {KO_RT:.0f} bp round trip in all three periods; "
                 f"{int(KU_T['stable'].sum())} are stable (t>2 in train_R and val_R, test t above Bonferroni {bf_:.2f}); {int(KU_T['supported'].sum())} of those also beat the same regime entered on every bar. " + (f"Best: {_be}. " if len(_bs) else "") + f"Per-product learned levels repeat from train_R to val_R with a median Spearman of {_rep:.2f}."))
    for _k_, _v_ in _KU_SAVED.items():
        if not _k_.startswith("KU_") and _k_ in globals() and globals()[_k_] is not _v_ and _k_ not in ("_KU_SAVED",): globals()[_k_] = _v_

## Step 18v — RSI inside the winning strategy: entry marker, exit and entry timing (daily Stage 2 breakout)

RSI is not traded on its own. It is used **inside the daily Stage 2 breakout** (long and short, 20- and 50-day lookback) in three places, each tested against the same trades without RSI. Everything is in trade-direction space: for a short the price is negated and the RSI is 100 − RSI, so one rule serves both sides. RSI is TA-Lib's (or the identical numpy Wilder RSI) on the daily close; every outcome is net of the round-trip cost, with the structural stop of Step 18o and a 10- or 20-day cap, and t is clustered by 10-day block.

1. **Entry marker** — the RSI in the trade direction at the signal bar. The cut (strong side: keep RSI at or above a quantile; calm side: keep RSI at or below it) is learned on train_R from the 20 / 40 / 60 / 80 % quantiles. The kept trades must beat all trades in train_R, val_R and test and must keep at least 70 % of the top-decile winners (the edge of a breakout system sits in its tail, so a marker that drops the tail is not a gain).
2. **Exit** — first of: the stop, an RSI target H (65-80), an RSI floor break F (cross down through 40-50) or the time cap. H, F are learned on train_R per cohort with the objective *net bp per day held* (capital tied up counts), and compared with stop + time only on the same trades; it improves only if the per-day result is better in all three periods and the test t of the paired difference is above Bonferroni.
3. **Entry timing** — wait up to 5 days for the RSI to cross back up through a level (40-60), then enter; unfilled signals earn nothing. Fill rate, what the unfilled trades would have earned (adverse selection) and the per-signal result against entering at once are shown; the level is learned on train_R.

The test is read once. Long and short, and the 20- and 50-day lookbacks, are separate cohorts, because 18t found the short side did not hold. The component-sign step moves to 18w.

In [ ]:
# step 18v
_KV_SAVED = dict(globals())          # snapshot: loop variables of this step must not overwrite a notebook name (restored at the end of the step)
KV_OK = bool(globals().get("KO_OK", False)) and all(k in globals() for k in ("KO_E", "KO_B", "KO_RT", "KN_SPLR", "bonferroni_t", "SECTOR", "KO_T0", "ku_rsi", "OUT2", "FIG_DIR")) and "D" in globals().get("KO_E", {}) and "BRK" in globals().get("KO_E", {}).get("D", {})
if not KV_OK:
    print("Step 18v needs Step 18o (Stage 2 breakout events) and Step 18u (RSI): run them first; skipped")
else:
    # ---- 18v.0 RSI INSIDE the winning strategy (daily Stage 2 breakout): as an entry marker (strength in the trade direction at the signal), as the exit, and as the entry timing (wait for a pullback). Everything is in trade-direction space: for a short the price is negated and the RSI is 100 - RSI, so one rule serves both sides. ----
    KV_MAXH = 20; KV_BLK = 10; KV_MAINT = 10; KV_SN = ("train_R", "val_R", "test"); KV_MINN = int(globals().get("KV_MINN", 150))
    KV_H = (None, 65, 70, 75, 80); KV_F = (None, 40, 45, 50); KV_T = (10, 20); KV_COH = (("S2 L20 long", 20, 1), ("S2 L20 short", 20, -1), ("S2 L50 long", 50, 1), ("S2 L50 short", 50, -1))
    EV = KO_E["D"]["BRK"].copy().reset_index(drop=True); EV = EV[(EV["spl"] >= 0) & (EV["s2"] == 1)].reset_index(drop=True); KV_PL = list(KO_B["D"]); EV["product"] = [KV_PL[i] for i in EV["pid"]]; EV["sector"] = EV["product"].map(lambda p: SECTOR.get(p, "Other"))
    n_ = len(EV); PC = np.full((n_, KV_MAXH + 1), np.nan); PR = np.full((n_, KV_MAXH + 1), np.nan); RXS = np.full(n_, np.nan)
    for (pid_, dr_), g_ in EV.groupby(["pid", "dir"]):
        d_ = KO_B["D"][KV_PL[pid_]]; cl_ = d_["cl"].to_numpy(float); ta_ = d_["tdi"].to_numpy(); rs_ = ku_rsi(cl_); cx = dr_ * cl_; rx = rs_ if dr_ > 0 else 100.0 - rs_
        ix = np.searchsorted(ta_, g_["tdi"].to_numpy()); e = ix + 1; ok = (e + KV_MAXH) < len(cl_); ii = g_.index.to_numpy()[ok]; ee = e[ok]; jj = np.arange(KV_MAXH + 1)[None, :]
        PC[ii] = (cx[ee[:, None] + jj] - cx[ee][:, None]) * 1e4; PR[ii] = rx[ee[:, None] + jj]; RXS[ii] = rx[ix[ok]]            # PC: price path from the entry close in bp, PR: RSI path from the entry close, RXS: RSI at the signal bar
    EV["rx_sig"] = RXS; EV["valid"] = np.isfinite(PC).all(axis=1) & np.isfinite(RXS) & np.isfinite(PR).all(axis=1); EV["blk"] = np.unique(EV["tdi"] // KV_BLK, return_inverse=True)[1].astype(np.int64); KV_NB = int(EV["blk"].max()) + 1
    def kv_group(y, m, minn=30):
        m = m & np.isfinite(y); n = int(m.sum())
        if n < minn: return n, np.nan, np.nan
        mu = float(y[m].mean()); s = np.bincount(EV["blk"].to_numpy()[m], weights=y[m] - mu, minlength=KV_NB); se = np.sqrt((s ** 2).sum()) / n
        return n, mu, (mu / se if se > 0 else np.nan)
    def kv_exit(Pc, Pr, risk, H, F, T):
        # first of: stop (close at or below -risk), RSI target (rsi >= H), RSI floor break (cross down through F), time T; returns gross bp and days held
        n = len(Pc); hit = Pc[:, :T + 1] <= -risk[:, None]
        if H is not None: hit = hit | (Pr[:, :T + 1] >= H)
        if F is not None: cr = np.zeros_like(hit); cr[:, 1:] = (Pr[:, :T][:, :] >= F) & (Pr[:, 1:T + 1] < F); hit = hit | cr
        hit[:, 0] = False; hit[:, T] = True; ex = hit.argmax(axis=1); return Pc[np.arange(n), ex], ex
    # unit checks: a path that never trips anything exits at T; the stop fires first on a drop; an RSI target at 70 fires on the bar it is reached; a floor cross needs a cross, not a level
    _pc = np.zeros((3, 21)); _pc[1, 3:] = -500.0; _pc[2, 5:] = 200.0; _pr = np.full((3, 21), 60.0); _pr[2, 5:] = 72.0; _g, _x = kv_exit(_pc, _pr, np.full(3, 300.0), 70, None, 10); assert list(_x) == [10, 3, 5] and _g[1] == -500.0 and _g[2] == 200.0, "exit logic"
    _pr2 = np.full((1, 21), 40.0); _g, _x = kv_exit(np.zeros((1, 21)), _pr2, np.full(1, 300.0), None, 45, 10); assert _x[0] == 10, "a level below the floor is not a cross"
    print(f"{int(EV['valid'].sum())} usable Stage 2 breakout trades of {len(EV)}; RSI backend: {'TA-Lib' if globals().get('_talib') is not None else 'numpy Wilder'}; unit checks passed")

In [ ]:
# step 18v
if KV_OK:
    # ---- 18v.1 RSI as the EXIT: first of stop, RSI target (H), RSI floor break (F), time (T). Rule learned on train_R per cohort (objective: net bp per day held, which counts capital tied up), then read in val_R and test; compared with the same trades under stop + time only ----
    SPv = EV["spl"].to_numpy(int); VAL = EV["valid"].to_numpy(bool); RISK = EV["risk_bp"].to_numpy(float); KV_R = []; KV_NET = {}
    for cname, L_, dr_ in KV_COH:
        cm = VAL & (EV["L"].to_numpy(int) == L_) & (EV["dir"].to_numpy(int) == dr_)
        if cm.sum() < 3 * KV_MINN: continue
        for T in KV_T:
            base = None; rows = []
            for H in KV_H:
                for F in KV_F:
                    gross = np.full(n_, np.nan); days = np.full(n_, np.nan); g_, x_ = kv_exit(PC[cm], PR[cm], RISK[cm], H, F, T); gross[cm] = g_; days[cm] = x_; net = gross - KO_RT; KV_NET[(cname, T, H, F)] = (net, days)
                    r = dict(cohort=cname, T=T, H=H if H else np.nan, F=F if F else np.nan, rule=f"RSI>={H}" if H else "no RSI target"); r["rule"] += f" | floor {F}" if F else " | no floor"
                    for k, nm in enumerate(KV_SN):
                        m = cm & (SPv == k); n, mu, t = kv_group(net, m); r[f"n_{nm}"] = n; r[f"net_{nm}"] = mu; r[f"t_{nm}"] = t; r[f"days_{nm}"] = float(np.nanmean(days[m])) if n else np.nan; r[f"bpday_{nm}"] = float(np.nansum(net[m]) / max(np.nansum(days[m] + 1), 1))
                    rows.append(r)
                    if H is None and F is None: base = (net, days)
            for r in rows:
                H, F = (None if np.isnan(r["H"]) else int(r["H"])), (None if np.isnan(r["F"]) else int(r["F"])); net, days = KV_NET[(cname, T, H, F)]
                for k, nm in enumerate(KV_SN):
                    m = cm & (SPv == k); d = net - base[0]; _, dm, dt = kv_group(d, m); r[f"diff_net_{nm}"] = dm; r[f"diff_t_{nm}"] = dt; r[f"diff_bpday_{nm}"] = r[f"bpday_{nm}"] - [x for x in rows if np.isnan(x["H"]) and np.isnan(x["F"])][0][f"bpday_{nm}"]
                r["learned"] = False
            ok = [r for r in rows if r["n_train_R"] >= KV_MINN and np.isfinite(r["bpday_train_R"])]
            if ok: max(ok, key=lambda r: r["bpday_train_R"])["learned"] = True
            KV_R += rows
    KV_X = pd.DataFrame(KV_R)
    if not KV_R: print("too few Stage 2 trades per cohort for the RSI exit, marker and pullback studies (needs 450 per cohort)")
    if KV_R:
        KV_X["baseline"] = KV_X["H"].isna() & KV_X["F"].isna(); KV_X.to_csv(OUT2 / "rsi_exit_grid.csv", index=False)
        KV_XL = KV_X[KV_X["learned"]].copy(); KV_XL["improves"] = (KV_XL[[f"diff_bpday_{s}" for s in KV_SN]] > 0).all(axis=1) & (KV_XL["diff_t_test"] > bonferroni_t(max(len(KV_XL), 1))) & ~KV_XL["baseline"]
        KV_XL.to_csv(OUT2 / "rsi_exit_learned.csv", index=False)
        print("exit rules learned on train_R (objective bp per day held) against stop + time only, same trades:"); print(KV_XL[["cohort", "T", "rule", "net_train_R", "net_val_R", "net_test", "days_test", "bpday_train_R", "bpday_val_R", "bpday_test", "diff_bpday_train_R", "diff_bpday_val_R", "diff_bpday_test", "diff_t_test", "improves"]].round(2).to_string(index=False))
        print("\nbaseline (stop + time only):"); print(KV_X[KV_X["baseline"]][["cohort", "T", "n_train_R", "net_train_R", "net_val_R", "net_test", "t_test", "days_test", "bpday_test"]].round(2).to_string(index=False))

        # ---- 18v.2 RSI as the ENTRY MARKER: RSI strength in the trade direction at the signal bar. Cut learned on train_R among quantiles (keep the strong side or the calm side), read in val_R and test; the kept trades must beat the dropped ones and the parent, and must keep the tail winners ----
        KV_F2 = []; RXq = EV["rx_sig"].to_numpy(float)
        for cname, L_, dr_ in KV_COH:
            cm = VAL & (EV["L"].to_numpy(int) == L_) & (EV["dir"].to_numpy(int) == dr_)
            if cm.sum() < 3 * KV_MINN: continue
            net = KV_NET[(cname, KV_MAINT, None, None)][0]; cuts = np.quantile(RXq[cm & (SPv == 0)], [0.2, 0.4, 0.6, 0.8]); cand = []
            for side in ("strong", "calm"):
                for q, c in zip((0.2, 0.4, 0.6, 0.8), cuts):
                    keep = cm & ((RXq >= c) if side == "strong" else (RXq <= c)); n0, mu0, _ = kv_group(net, keep & (SPv == 0)); cand.append((mu0 if n0 >= KV_MINN else -np.inf, side, q, c, keep))
            _, side, q, c, keep = max(cand, key=lambda x: x[0]); r = dict(cohort=cname, side=side, quantile=q, rsi_cut=float(c))
            for k, nm in enumerate(KV_SN):
                m = cm & (SPv == k); n, mu, t = kv_group(net, keep & m); n0, m0, t0 = kv_group(net, m); nd, md, td = kv_group(net, m & ~keep)
                hi = np.nanquantile(net[m], 0.9) if m.sum() > 20 else np.nan; share_tail = float(np.mean(keep[m & (net >= hi)])) if np.isfinite(hi) and (m & (net >= hi)).sum() > 0 else np.nan
                y = np.sort(net[keep & m]); trim = float(y[:int(len(y) * 0.95)].mean()) if len(y) >= 40 else np.nan
                r.update({f"n_{nm}": n, f"net_{nm}": mu, f"t_{nm}": t, f"parent_net_{nm}": m0, f"dropped_net_{nm}": md, f"kept_share_{nm}": float(keep[m].mean()) if m.sum() else np.nan, f"tail_kept_{nm}": share_tail, f"trim5_{nm}": trim})
            r["beats_parent"] = bool(all(np.isfinite(r[f"net_{s}"]) and r[f"net_{s}"] > r[f"parent_net_{s}"] for s in KV_SN)); r["tail_ok"] = bool(all(np.isfinite(r[f"tail_kept_{s}"]) and r[f"tail_kept_{s}"] >= 0.7 for s in KV_SN)); KV_F2.append(r)
        KV_FT = pd.DataFrame(KV_F2); KV_FT.to_csv(OUT2 / "rsi_entry_marker.csv", index=False)
        print("\nRSI strength at the signal as an entry marker (cut learned on train_R; kept = trades passing the marker; 10-day hold with stop):"); print(KV_FT[["cohort", "side", "rsi_cut", "net_train_R", "net_val_R", "net_test", "parent_net_train_R", "parent_net_val_R", "parent_net_test", "kept_share_test", "tail_kept_test", "beats_parent", "tail_ok"]].round(2).to_string(index=False))

In [ ]:
# step 18v
if KV_OK and len(KV_X):
    # ---- 18v.3 RSI as the ENTRY TIMING: after the signal, wait up to 5 days for a pullback (RSI crosses back up through a level) and enter then; the unfilled trades earn nothing. Fill rate, adverse selection (what the unfilled trades would have earned) and the per-signal result against entering at once ----
    KV_L = (40, 45, 50, 55, 60); KV_K = 5; KV_P = []
    for cname, L_, dr_ in KV_COH:
        cm = VAL & (EV["L"].to_numpy(int) == L_) & (EV["dir"].to_numpy(int) == dr_)
        if cm.sum() < 3 * KV_MINN: continue
        base = KV_NET[(cname, KV_MAINT, None, None)][0]; idx = np.where(cm)[0]; T = KV_MAINT
        for lev in KV_L:
            jj = np.full(n_, -1); a = (PR[idx, :KV_K] < lev) & (PR[idx, 1:KV_K + 1] >= lev); has = a.any(axis=1); jj[idx[has]] = a[has].argmax(axis=1) + 1
            new = np.full(n_, np.nan); f = idx[has]
            if len(f):
                j = jj[f]; M = PC[f[:, None], j[:, None] + np.arange(T + 1)[None, :]] - PC[f, j][:, None]; hit = M <= -RISK[f][:, None]; hit[:, 0] = False; hit[:, T] = True; ex = hit.argmax(axis=1); new[f] = M[np.arange(len(f)), ex] - KO_RT
            filled = np.isfinite(new); r = dict(cohort=cname, level=lev)
            for k, nm in enumerate(KV_SN):
                m = cm & (SPv == k); nf = int((m & filled).sum()); r[f"n_{nm}"] = int(m.sum()); r[f"fill_{nm}"] = nf / max(int(m.sum()), 1); _, mn, tn = kv_group(new, m & filled); r[f"net_new_{nm}"] = mn; r[f"t_new_{nm}"] = tn
                _, mb, _ = kv_group(base, m & filled); _, mu_, _ = kv_group(base, m & ~filled); _, mall, _ = kv_group(base, m); r[f"net_same_trades_now_{nm}"] = mb; r[f"net_unfilled_now_{nm}"] = mu_; r[f"expect_new_{nm}"] = float(np.nansum(np.where(filled & m, new, 0.0)) / max(m.sum(), 1)); r[f"expect_now_{nm}"] = mall
                _, dm, dt = kv_group(new - base, m & filled); r[f"diff_filled_{nm}"] = dm; r[f"diff_t_{nm}"] = dt
            KV_P.append(r)
    KV_PT = pd.DataFrame(KV_P); KV_PT["learned"] = False
    for cname in KV_PT["cohort"].unique():
        z = KV_PT[(KV_PT["cohort"] == cname) & (KV_PT[[f"n_{KV_SN[0]}"]].iloc[:, 0] > 0)]; ok = z[z["fill_train_R"] * z["n_train_R"] >= KV_MINN * 2 // 3]
        if len(ok): KV_PT.loc[ok["expect_new_train_R"].idxmax(), "learned"] = True
    KV_PT["adverse_selection"] = KV_PT["net_unfilled_now_test"] > KV_PT["net_same_trades_now_test"]; KV_PT["pullback_wins"] = (KV_PT[[f"expect_new_{s}" for s in KV_SN]].to_numpy() > KV_PT[[f"expect_now_{s}" for s in KV_SN]].to_numpy()).all(axis=1)
    KV_PT.to_csv(OUT2 / "rsi_pullback_entry.csv", index=False)
    print(f"\nwait up to {KV_K} days for the RSI to cross back up through a level, then enter (unfilled = no trade); learned level per cohort marked:"); print(KV_PT[KV_PT["learned"]][["cohort", "level", "fill_train_R", "fill_val_R", "fill_test", "expect_new_train_R", "expect_now_train_R", "expect_new_val_R", "expect_now_val_R", "expect_new_test", "expect_now_test", "net_unfilled_now_test", "net_same_trades_now_test", "adverse_selection", "pullback_wins"]].round(2).to_string(index=False))
    # ---- figure and summary ----
    fig, axs = plt.subplots(2, 3, figsize=(19, 9)); c0 = KV_COH[0][0]; cm0 = VAL & (EV["L"].to_numpy(int) == KV_COH[0][1]) & (EV["dir"].to_numpy(int) == KV_COH[0][2]); net0 = KV_NET[(c0, KV_MAINT, None, None)][0]
    ax = axs[0, 0]; edges = np.unique(np.quantile(RXq[cm0 & (SPv == 0)], np.linspace(0, 1, 6)))
    for k, (nm, col) in enumerate(zip(KV_SN, ("#1f3a5f", "#e0a030", "#9b2226"))):
        ys = [kv_group(net0, cm0 & (SPv == k) & (RXq >= lo) & (RXq <= hi + (1e-9 if i == len(edges) - 2 else -1e-9)), 20)[1] for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:]))]; ax.plot(np.arange(len(ys)), ys, marker="o", color=col, label=nm)
    ax.axhline(0, color="k", lw=0.6); ax.set_xticks(range(len(edges) - 1)); ax.set_xticklabels([f"{a:.0f}-{b:.0f}" for a, b in zip(edges[:-1], edges[1:])], fontsize=7); ax.legend(fontsize=7); ax.set_xlabel("RSI at the signal bar (trade direction), train_R quintiles"); ax.set_ylabel("net bp per trade"); ax.set_title(f"{c0}: result by RSI at the signal", fontsize=9)
    for j, nm in enumerate(("train_R", "test")):
        ax = axs[0, 1 + j]; z = KV_X[(KV_X["cohort"] == c0) & (KV_X["T"] == KV_MAINT)]; piv = z.pivot_table(index=z["F"].fillna(0), columns=z["H"].fillna(0), values=f"bpday_{nm}"); im = ax.imshow(piv.to_numpy(float), cmap="RdBu_r", aspect="auto", vmin=-np.nanmax(np.abs(piv.to_numpy(float))), vmax=np.nanmax(np.abs(piv.to_numpy(float))))
        ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels([("none" if v == 0 else int(v)) for v in piv.columns]); ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels([("none" if v == 0 else int(v)) for v in piv.index]); ax.set_xlabel("RSI target H"); ax.set_ylabel("RSI floor F"); ax.set_title(f"{c0}, 10-day cap: net bp per day held ({nm})", fontsize=9); plt.colorbar(im, ax=ax, fraction=0.04)
    ax = axs[1, 0]; z = KV_PT[KV_PT["cohort"] == c0]; ax.plot(z["level"], z["fill_test"], marker="o", label="fill rate (test)", color="#555555"); ax2 = ax.twinx(); ax2.plot(z["level"], z["expect_new_test"], marker="s", color="#9b2226", label="per signal, wait for pullback"); ax2.axhline(float(z["expect_now_test"].iloc[0]), color="#1f3a5f", ls="--", label="per signal, enter at once"); ax.set_xlabel("RSI level the pullback must cross back up through"); ax.set_ylabel("fill rate"); ax2.set_ylabel("net bp per signal (test)"); ax2.legend(fontsize=7, loc="lower left"); ax.set_title(f"{c0}: pullback entry", fontsize=9)
    ax = axs[1, 1]; z = KV_XL[~KV_XL["baseline"]]; xs = np.arange(len(z)); ax.bar(xs - 0.27, z["diff_bpday_train_R"], 0.27, label="train_R", color="#1f3a5f"); ax.bar(xs, z["diff_bpday_val_R"], 0.27, label="val_R", color="#e0a030"); ax.bar(xs + 0.27, z["diff_bpday_test"], 0.27, label="test", color="#9b2226"); ax.set_xticks(xs); ax.set_xticklabels(z["cohort"].str.replace("S2 ", "") + "\nT" + z["T"].astype(str), fontsize=6.5, rotation=60); ax.axhline(0, color="k", lw=0.6); ax.legend(fontsize=7); ax.set_title("learned RSI exit minus stop + time only (bp per day held)", fontsize=9)
    ax = axs[1, 2]; z = KV_FT; xs = np.arange(len(z)); ax.bar(xs - 0.2, z["net_test"], 0.4, label="kept by the RSI marker (test)", color="#9b2226"); ax.bar(xs + 0.2, z["parent_net_test"], 0.4, label="all trades (test)", color="#999999"); ax.set_xticks(xs); ax.set_xticklabels(z["cohort"].str.replace("S2 ", ""), fontsize=7, rotation=30); ax.axhline(0, color="k", lw=0.6); ax.legend(fontsize=7); ax.set_title("entry marker: kept trades against all trades, net bp (test)", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "18v_rsi_inside_stage2.png", dpi=110); plt.show()
    def _fx(x): return f"{x:.0f}" if np.isfinite(x) else "n/a"
    _imp = KV_XL[KV_XL["improves"]]; _mk = KV_FT[KV_FT["beats_parent"] & KV_FT["tail_ok"]]; _pb = KV_PT[KV_PT["learned"] & KV_PT["pullback_wins"]]; _b0 = KV_X[KV_X["baseline"] & (KV_X["T"] == KV_MAINT)]
    KV_SUMMARY = dict(cohorts=[c[0] for c in KV_COH], exit_improves=int(len(_imp)), exit_cells=int((~KV_XL["baseline"]).sum()), marker_beats_parent_and_keeps_tail=int(len(_mk)), marker_cells=int(len(KV_FT)), pullback_wins=int(len(_pb)), pullback_cells=int(KV_PT["learned"].sum()),
        baseline={r.cohort: [r.net_train_R, r.net_val_R, r.net_test] for r in _b0.itertuples()},
        reading=("RSI inside the daily Stage 2 breakout (trade-direction space; stop at the structural risk, 10 / 20-day cap): " + "; ".join(f"{r.cohort} baseline net bp {_fx(r.net_train_R)} / {_fx(r.net_val_R)} / {_fx(r.net_test)}" for r in _b0.itertuples()) + ". "
                 f"Exit (RSI target and floor learned on train_R, objective bp per day held): {len(_imp)} of {int((~KV_XL['baseline']).sum())} cohort x cap rules beat stop + time only in all three periods with a test t above Bonferroni. "
                 f"Entry marker (RSI at the signal, cut learned on train_R): {len(_mk)} of {len(KV_FT)} cohorts beat all trades in all three periods and keep at least 70% of the top-decile trades. "
                 f"Pullback entry (wait up to {KV_K} days, level learned on train_R): {len(_pb)} of {int(KV_PT['learned'].sum())} cohorts beat entering at once per signal in all three periods. Test read once."))
    for _k_, _v_ in _KV_SAVED.items():
        if not _k_.startswith("KV_") and _k_ in globals() and globals()[_k_] is not _v_ and _k_ not in ("_KV_SAVED",): globals()[_k_] = _v_

## Step 13g — Method fixes: demeaned, regime-aware, walk-forward re-tests of the state and field indicators

**Why.** The report found that in the test period every state had a positive mean (the test drift), which can make a "state" look like an edge when the whole market simply rose, and that several tests use overlapping windows or a single split. This step re-tests the earlier conditional results under stricter, shared rules, inside the Step 13f regimes. It adds no new indicator.

| Fix | What is done |
|---|---|
| Demean | Every label is also reported in excess of the **product's own mean in the same period and volatility regime** (`prod`) and in excess of **the day's cross-section** (`xs`). A result must survive both. |
| Non-overlapping labels | Next-5-day returns are sampled every fifth day inside each period, so observations do not share days. |
| Regime-aware allocation | Decisions are fixed on `train_R` and `val_R` (Step 13f), confirmed once on the chronological test; every result is also shown inside low / mid / high volatility. |
| Trailing-volatility control | The size-of-move test regresses the next absolute move on \|signal\| **with the trailing 20-day mean absolute move held fixed**; a result that only reflects "volatile days stay volatile" disappears. |
| Walk-forward | Seven calendar blocks with a 20-day embargo at the start of each; the cross-sectional IC per block, the share of positive blocks and the standard error across blocks replace a single 3-year test. |
| Bonferroni | Over all regime × state (or signal × test) cells. |

**What is re-tested.** (1) The eight volume × open-interest states of Step 18e (continuation or reversal calls): the table shows how many were confirmed on raw labels, after each demeaning, and which survive both. (2) ECHO (18i) and WAVE (18j): direction (cross-sectional rank IC against the day-demeaned label; own-product slope against the product-demeaned label) and size of move with and without the volatility control. The windows and parameters of those two studies were chosen in their own steps on the earlier chronological validation; they are **not** re-chosen here, which is a stated limitation.

**Reading.** A signal that is "supported" here has the same sign in `train_R`, `val_R` and the test and passes the Bonferroni bar after demeaning; the walk-forward table then shows whether that is stable block by block. Results that only appear on raw labels are drift, not edge. The size-of-move findings are the ones to carry into position sizing (Steps 22–23).

**Used later.** The demeaned verdicts on the volume / open-interest states and on ECHO and WAVE decide which indicators are kept (Step 26d ledger); a supported ECHO size-of-move effect is the candidate input for position sizing in Steps 22–23.

In [ ]:
# step 13g
RDG_OK = bool(globals().get("RG_OK", False)) and all(k in globals() for k in ("RG", "RG_IDX", "Z", "OV5", "OV_NAMES", "OV_DIR"))
if not RDG_OK:
    print("Step 13g needs Step 13f (regimes) and Step 18e (volume x open-interest states): run them first; skipped")
else:
    # ---- 13g.1 shared helpers: regime-cell demeaning, date-clustered mean, non-overlapping labels ----
    RD_SP = {"train_R": RG_IDX["train"], "val_R": RG_IDX["val"], "test": RG_IDX["test"]}; RD_REGS = [("all", None)] + [(RG_VN[k] + "-vol", k) for k in range(3)]
    RD_TV = Z.abs().rolling(20, min_periods=15).mean()                                        # trailing size of moves, known at t
    RD_LAB = {"fwd5": Z.rolling(5).sum().shift(-5), "fwdabs1": Z.abs().shift(-1)}
    def rd_cm(y, dates, minn=30):
        y = np.asarray(y, float); n = len(y)
        if n < minn: return np.nan, np.nan
        m = y.mean(); s = pd.Series(y - m).groupby(np.asarray(dates)).sum(); se = np.sqrt((s ** 2).sum()) / n
        return float(m), float(m / se) if se > 0 else np.nan
    _t = np.r_[1.0, 1.0, 3.0, 3.0]; _m, _tt = rd_cm(_t, [0, 0, 1, 1], minn=2); assert abs(_m - 2.0) < 1e-12 and abs(_tt - 2.0 / (np.sqrt(((1 - 2) * 2) ** 2 + ((3 - 2) * 2) ** 2) / 4)) < 1e-9, "clustered t broken"; print("unit check passed: date-clustered mean and t")
    RD_SPLIT = pd.Series("", index=RAW.index)
    for _k, _ix in RD_SP.items(): RD_SPLIT.loc[_ix] = _k
    RD_KEEP = pd.Series(False, index=RAW.index)
    for _k, _ix in RD_SP.items(): RD_KEEP.loc[_ix[::5]] = True                                  # non-overlapping 5-day labels inside each period
    def rd_long(panels):
        d = pd.DataFrame({k: v.stack().dropna() for k, v in {**panels, **{"y_" + a: b for a, b in RD_LAB.items()}, "tv": RD_TV}.items()}).dropna(subset=["y_fwd5"])
        dt = d.index.get_level_values(0); d["split"] = RD_SPLIT.reindex(dt).to_numpy(); d["keep"] = RD_KEEP.reindex(dt).to_numpy(); d["V"] = RG["V"].reindex(dt).to_numpy()
        d = d[(d.split != "") & d.keep & d.V.notna()].copy(); d.index.names = ["date", "product"]
        g = d.groupby([d.index.get_level_values("product"), "split", "V"])["y_fwd5"].transform("mean"); d["y_prod"] = d["y_fwd5"] - g                 # excess over the product's mean in the same period and regime
        d["y_xs"] = d["y_fwd5"] - d.groupby(level="date")["y_fwd5"].transform("mean")                                                               # excess over the day's cross-section
        return d
    # ---- 13g.2 the nine volume x open-interest states of Step 18e, re-tested on demeaned labels inside each volatility regime ----
    RD_ST = rd_long(dict(state=OV5["state"], hv=OV5["hv"]))
    rows = []
    for rn, rk in RD_REGS:
        for (stt, hv), g0 in RD_ST.groupby(["state", "hv"]):
            for sp in RD_SP:
                g = g0[(g0.split == sp) & ((g0.V == rk) if rk is not None else True)]
                r = dict(regime=rn, state=OV_NAMES[int(stt)], volume="high" if hv else "low", split=sp, n=len(g))
                for lab in ("y_fwd5", "y_prod", "y_xs"): r[lab[2:] + "_mean"], r[lab[2:] + "_t"] = rd_cm(g[lab], g.index.get_level_values("date"))
                rows.append(r)
    RD_STAB = pd.DataFrame(rows); RD_STAB.round(4).to_csv(OUT2 / "regime_states_demeaned.csv", index=False)
    W_ = RD_STAB.pivot_table(index=["regime", "state", "volume"], columns="split", values=["fwd5_mean", "fwd5_t", "prod_mean", "prod_t", "xs_mean", "xs_t"])
    out = []; NT_ST = len(W_)
    RD_TB = bonferroni_t(NT_ST)
    for ix_, r in W_.iterrows():
        stt = [k for k, v in OV_NAMES.items() if v == ix_[1]][0]; sg = lambda v: np.sign(v) * OV_DIR[stt]; res = dict(regime=ix_[0], state=ix_[1], volume=ix_[2])
        for mode in ("fwd5", "prod", "xs"):
            a, b, c = sg(r[(mode + "_mean", "train_R")]), sg(r[(mode + "_mean", "val_R")]), sg(r[(mode + "_mean", "test")]); t = r[(mode + "_t", "test")]
            call = "continuation" if (a > 0 and b > 0) else "reversal" if (a < 0 and b < 0) else "no edge"
            res[mode + "_call"] = call; res[mode + "_test_t"] = t; res[mode + "_confirmed"] = False
            if call == "reversal": res[mode + "_confirmed"] = bool(c < 0 and abs(t) > RD_TB)
            if call == "continuation": res[mode + "_confirmed"] = bool(c > 0 and abs(t) > RD_TB)
        res["survives_both_demeanings"] = res["prod_confirmed"] and res["xs_confirmed"]; out.append(res)
    RD_STDEC = pd.DataFrame(out); RD_STDEC.round(3).to_csv(OUT2 / "regime_states_decisions.csv", index=False)
    print(f"Volume x open-interest states, next-5-day z, non-overlapping, {NT_ST} regime x state cells, Bonferroni |t| > {RD_TB:.2f}.")
    print("raw = as in Step 18e; prod = minus the product's mean in the same period and regime; xs = minus the day's cross-section. 'call' is fixed on train_R and val_R, confirmed on test.")
    print(f"confirmed on raw labels: {int(RD_STDEC.fwd5_confirmed.sum())} | on product-demeaned: {int(RD_STDEC.prod_confirmed.sum())} | on cross-sectionally demeaned: {int(RD_STDEC.xs_confirmed.sum())} | survive both demeanings: {int(RD_STDEC.survives_both_demeanings.sum())}")
    print(RD_STDEC[RD_STDEC.fwd5_confirmed | RD_STDEC.survives_both_demeanings][["regime", "state", "volume", "fwd5_call", "fwd5_test_t", "prod_call", "prod_test_t", "xs_call", "xs_test_t", "survives_both_demeanings"]].round(2).to_string(index=False) if (RD_STDEC.fwd5_confirmed | RD_STDEC.survives_both_demeanings).any() else "no cell confirmed on any label")
    RD_DRIFT = RD_ST.groupby("split")[["y_fwd5"]].mean().rename(columns={"y_fwd5": "mean_next5_z_all_rows"}); print("\nDrift in the label itself (mean next-5-day z over all rows, per period):"); print(RD_DRIFT.round(4).to_string())

In [ ]:
# step 13g
if RDG_OK:
    # ---- 13g.3 ECHO and WAVE by regime on demeaned labels; size-of-move tests with a trailing-volatility control ----
    import statsmodels.api as sm
    from scipy import stats as _rdst
    RD_SIG = {k: v for k, v in (("ECHO", globals().get("EC_ENS")), ("WAVE", globals().get("WV_ENS"))) if v is not None}
    print("signals available:", list(RD_SIG) or "none (Steps 18i / 18j not run)")
    def rd_ic(d, col, lab):
        # mean over days of the cross-sectional rank correlation of the signal with the label, and a t over days
        v = []
        for _, g in d[[col, lab]].replace([np.inf, -np.inf], np.nan).dropna().groupby(level="date"):
            if len(g) >= 8 and g[col].nunique() > 3: v.append(_rdst.spearmanr(g[col], g[lab])[0])
        v = np.asarray([x for x in v if np.isfinite(x)])
        return (float(v.mean()), float(v.mean() / (v.std(ddof=1) / np.sqrt(len(v))))) if len(v) >= 30 else (np.nan, np.nan)
    def rd_slope(d, x, y, control=None):
        cols = [x] + ([control] if control else []) + [y]; dd = d[cols].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
        if len(dd) < 200 or dd[x].std() == 0: return np.nan, np.nan
        X = dd[[x] + ([control] if control else [])]; X = sm.add_constant((X - X.mean()) / X.std().replace(0, np.nan)).dropna(axis=1)
        r = sm.OLS(dd[y], X).fit(cov_type="cluster", cov_kwds=dict(groups=pd.factorize(dd.index.get_level_values("date"))[0])); return float(r.params[x]), float(r.tvalues[x])
    rows = []; RD_LONG = {}
    for nm, S in RD_SIG.items():
        d = rd_long(dict(sig=S)); d["asig"] = d["sig"].abs(); d["ltv"] = np.log(d["tv"].clip(lower=1e-3)); RD_LONG[nm] = d
        for rn, rk in RD_REGS:
            for sp in RD_SP:
                g = d[(d.split == sp) & ((d.V == rk) if rk is not None else True)]
                r = dict(signal=nm, regime=rn, split=sp, n=len(g))
                r["ic_xs"], r["ic_xs_t"] = rd_ic(g, "sig", "y_xs")                                         # direction, cross-section (day-demeaned label)
                r["slope_prod"], r["slope_prod_t"] = rd_slope(g, "sig", "y_prod")                           # direction, own-product (product-demeaned label)
                r["abs_slope"], r["abs_slope_t"] = rd_slope(g, "asig", "y_fwdabs1")                         # size of next move on |signal|, no control
                r["abs_slope_ctl"], r["abs_slope_ctl_t"] = rd_slope(g, "asig", "y_fwdabs1", control="ltv")     # with the trailing size of moves held fixed
                rows.append(r)
    RD_CONT = pd.DataFrame(rows)
    if len(RD_CONT):
        RD_CONT.round(4).to_csv(OUT2 / "regime_signals_demeaned.csv", index=False)
        NT_C = len(RD_CONT) // 3; RD_TC = bonferroni_t(NT_C * 4); res = []
        for (nm, rn), g in RD_CONT.groupby(["signal", "regime"]):
            g = g.set_index("split")
            for c in ("ic_xs", "slope_prod", "abs_slope", "abs_slope_ctl"):
                a, b, e = (np.sign(g.loc[k, c]) for k in ("train_R", "val_R", "test")); t = g.loc["test", c + "_t"]
                res.append(dict(signal=nm, regime=rn, test=c, train_R=g.loc["train_R", c], val_R=g.loc["val_R", c], test_value=g.loc["test", c], test_t=t, same_sign=bool(a == b == e and a != 0), supported=bool(a == b == e and a != 0 and abs(t) > RD_TC)))
        RD_CDEC = pd.DataFrame(res); RD_CDEC.round(4).to_csv(OUT2 / "regime_signals_decisions.csv", index=False)
        print(f"\nECHO / WAVE: direction (xs IC, own-product slope) and size of move (|signal| on next |z|, without and with the trailing-volatility control); Bonferroni |t| > {RD_TC:.2f} on test; supported = same sign in train_R, val_R, test:")
        print(RD_CDEC[RD_CDEC.regime == "all"].round(3).to_string(index=False))
        print("\nSupported in any regime:"); print(RD_CDEC[RD_CDEC.supported].round(3).to_string(index=False) if RD_CDEC.supported.any() else "none")
        _lost = RD_CDEC[RD_CDEC.test == "abs_slope"].merge(RD_CDEC[RD_CDEC.test == "abs_slope_ctl"], on=["signal", "regime"], suffixes=("", "_c")); _lost = _lost[_lost.supported & ~_lost.supported_c]
        print(f"size-of-move results that vanish once trailing volatility is held fixed: {len(_lost)}")
    # ---- 13g.4 purged walk-forward blocks: does the sign hold block by block, not only in one chronological test? ----
    WF_K, WF_EMB = 7, EMB; _wfd = RAW.index; _wfe = np.linspace(0, len(_wfd), WF_K + 1).astype(int)
    # the regime-balanced split assigns days to train/val/test; a walk-forward fold needs its own calendar blocks, so rebuild the long frame on the full calendar
    def rd_long_all(panels):
        D = pd.DataFrame({k: v.stack().dropna() for k, v in {**panels, "y_fwd5": RD_LAB["fwd5"], "y_fwdabs1": RD_LAB["fwdabs1"]}.items()}).dropna(); D.index.names = ["date", "product"]; return D
    rows = []
    for nm, S in RD_SIG.items():
        D = rd_long_all(dict(sig=S))
        for k in range(WF_K):
            lo, hi = _wfe[k] + (WF_EMB if k > 0 else 0), _wfe[k + 1]; ix = _wfd[lo:hi][::5]; g = D[D.index.get_level_values("date").isin(ix)].copy()
            g["y_xs"] = g["y_fwd5"] - g.groupby(level="date")["y_fwd5"].transform("mean"); ic, t = rd_ic(g, "sig", "y_xs")
            rows.append(dict(signal=nm, fold=k, start=_wfd[lo].date(), end=_wfd[hi - 1].date(), n=len(g), ic_xs=ic, t=t))
    RD_WF = pd.DataFrame(rows)
    if len(RD_WF):
        RD_WF.round(4).to_csv(OUT2 / "walk_forward_blocks.csv", index=False)
        RD_WFS = RD_WF.groupby("signal").apply(lambda g: pd.Series(dict(folds=int(g.ic_xs.notna().sum()), mean_ic=g.ic_xs.mean(), se_across_folds=g.ic_xs.std(ddof=1) / np.sqrt(max(g.ic_xs.notna().sum(), 1)), share_positive=(g.ic_xs > 0).mean(), worst=g.ic_xs.min()))).reset_index()
        print(f"\nPurged walk-forward blocks ({WF_K} calendar blocks, {WF_EMB}-day embargo at the start of each block, non-overlapping 5-day labels): cross-sectional IC per block"); print(RD_WF.round(3).to_string(index=False)); print(RD_WFS.round(3).to_string(index=False))
    RDG_SUMMARY = dict(state_cells=int(len(RD_STDEC)), confirmed_raw=int(RD_STDEC.fwd5_confirmed.sum()), confirmed_product_demeaned=int(RD_STDEC.prod_confirmed.sum()), confirmed_xs_demeaned=int(RD_STDEC.xs_confirmed.sum()),
                       survive_both=RD_STDEC[RD_STDEC.survives_both_demeanings][["regime", "state", "volume", "prod_call", "prod_test_t", "xs_test_t"]].round(3).to_dict("records"),
                       signal_supported=(RD_CDEC[RD_CDEC.supported & (RD_CDEC.test != "abs_slope")].round(3).to_dict("records") if len(RD_SIG) and "RD_CDEC" in globals() else []), walk_forward=(RD_WFS.round(3).to_dict("records") if len(RD_WF) else []),
                       drift_of_label=RD_DRIFT.round(4).to_dict()["mean_next5_z_all_rows"])
    RDG_SUMMARY["reading"] = (f"Of {RDG_SUMMARY['state_cells']} regime x state cells, {RDG_SUMMARY['confirmed_raw']} were confirmed on raw labels, {RDG_SUMMARY['confirmed_product_demeaned']} after removing the product's mean and {RDG_SUMMARY['confirmed_xs_demeaned']} after removing the day's cross-section; "
                              + f"{len(RDG_SUMMARY['survive_both'])} survive both. ECHO / WAVE supported results: {len(RDG_SUMMARY['signal_supported'])} (direction tests and size-of-move tests with the trailing-volatility control; the uncontrolled size-of-move rows are printed but not counted). Labels are non-overlapping 5-day z returns; decisions on train_R and val_R, test read once.")
    print("\n" + RDG_SUMMARY["reading"])

## Step 19 — Multi-time-frame bull / bear ranks (UpR / DnR)

For every product and every time frame (`TFS` = 5m, 15m, 30m, 60m, 1D, 1W), indicators from five families plus breadth are mapped to a signed score $s_i \in [-1, 1]$:

| Family | Indicators (window = `RANK_WIN[tf]` bars) |
|---|---|
| Trend | price vs slow EMA (vol-scaled), slow slope t-stat |
| Momentum | fast vol-scaled return, RSI-style up/down ratio, relative strength (cross-sectional rank) |
| Volume | volume z × direction, OI change × direction |
| Volatility | low / falling realised vol = bullish (rule) |
| Mean reversion | oversold vs the fast mean = bullish (rule) |
| Breadth | share of the market and of the sector in an uptrend |

$$\text{UpR} = 100\,\frac{\sum_i w_i \max(s_i,0)}{\sum_i w_i}, \qquad \text{DnR} = -100\,\frac{\sum_i w_i \max(-s_i,0)}{\sum_i w_i}$$

UpR and DnR are separate (a product can have some bullish and some bearish evidence), and Net = UpR + DnR.

**Two variants:**
* `rule`: fixed indicator directions and `FAMILY_W` weights, the classic technical-ranking approach.
* `learned`: per time frame, each indicator's direction and weight come from its **train** IC against the forward return over `RANK_H[tf]` bars, so an indicator that works the other way at a frequency (for example momentum reverting at 5m) is flipped.

**Multi-time-frame composite:**
* The latest bar of each frame at the daily close is weighted by `TF_WEIGHTS`, with emphasis on 60m and above.
* **MTF rule:** BULL when UpR ≥ `RANK_THRESH` on every frame in `MTF_RULE_TFS` (the "60m and above at ≥ 50" rule); BEAR when DnR ≤ −threshold on all of them.

**Validation, on each frame's own validation and test periods:**
* IC of Net against the forward return.
* Forward return by Net decile.
* Hit rate of bull / bear calls.
* Rank persistence and transition matrix.
* The latest **scorecard**, like the one in the screenshot.

**Used later.** UpR / DnR ranks feed the rank strategies A1–A5 and the meta-labelling strategy E (Step 23) and the board (Step 26).

In [ ]:
# step 19
def rank_indicators(F, tf):
    nf, ns = RANK_WIN[tf]; R = F["ret"]; cols = R.columns
    LPt = R.fillna(0).cumsum().where(R.notna().cumsum() > 0)
    sd = R.ewm(halflife=ns, min_periods=nf).std()
    th = lambda x, s=1.5: np.tanh(x / s)
    I, FAMI = {}, {}
    def add(name, fam, val):
        I[name] = val.replace([np.inf, -np.inf], np.nan).astype("float32"); FAMI[name] = fam
    add("trend_ema", "trend", th((LPt - LPt.ewm(span=ns, min_periods=nf).mean()) / (sd * np.sqrt(ns))))
    add("trend_slope", "trend", th(rolling_tstat(LPt, ns), 3))
    mom = R.rolling(nf, min_periods=max(nf // 2, 2)).sum()
    add("mom_fast", "momentum", th(mom / (sd * np.sqrt(nf))))
    up = R.clip(lower=0).rolling(14, min_periods=7).sum(); dn = (-R.clip(upper=0)).rolling(14, min_periods=7).sum()
    add("mom_rsi", "momentum", (up - dn) / (up + dn))
    add("rel_strength", "momentum", (mom / sd).rank(axis=1, pct=True) * 2 - 1)
    lv_ = np.log1p(F["volume"].reindex_like(R)); vz = (lv_ - lv_.rolling(ns, min_periods=nf).mean()) / lv_.rolling(ns, min_periods=nf).std()
    add("volume_conf", "volume", th(vz) * np.sign(mom))
    oi_ = F["oi"].reindex_like(R); doi = np.log(oi_.where(oi_ > 0)).diff(nf)
    add("oi_conf", "volume", np.sign(doi) * np.sign(mom))
    lr_ = np.log(F["rv2"].reindex_like(R).where(lambda x: x > 0).rolling(nf, min_periods=max(nf // 2, 2)).mean())
    add("volatility", "volatility", -th((lr_ - lr_.rolling(ns * 4, min_periods=ns).mean()) / lr_.rolling(ns * 4, min_periods=ns).std()))
    add("mean_rev", "mean_reversion", -th((LPt - LPt.rolling(nf, min_periods=max(nf // 2, 2)).mean()) / (sd * np.sqrt(nf))))
    te = (I["trend_ema"] > 0).astype(float).where(I["trend_ema"].notna())
    add("breadth_mkt", "breadth", pd.DataFrame(np.repeat((te.mean(axis=1) * 2 - 1).to_numpy()[:, None], len(cols), 1), index=R.index, columns=cols))
    add("breadth_sector", "breadth", te.T.groupby(SEC_MAP.reindex(cols)).transform("mean").T * 2 - 1)
    fwd = (R / sd.shift(1)).rolling(RANK_H[tf], min_periods=RANK_H[tf]).sum().shift(-RANK_H[tf]) / np.sqrt(RANK_H[tf])
    return I, FAMI, fwd

def updn(I, w, o):
    num_u = num_d = den = 0
    for k, v in I.items():
        if w.get(k, 0) <= 0:
            continue
        s = v * o.get(k, 1.0); a = v.notna()
        num_u = num_u + w[k] * s.clip(lower=0).fillna(0); num_d = num_d + w[k] * (-s).clip(lower=0).fillna(0); den = den + w[k] * a
    den = den.where(den > 0)
    return (100 * num_u / den).astype("float32"), (-100 * num_d / den).astype("float32")

# point-in-time audit of the rank indicators (daily frame): cut the data, recompute, compare
_F = tf_frames("1D"); _cut = SPL["train"][-1]
_I1, _, _ = rank_indicators(_F, "1D"); _I2, _, _ = rank_indicators({k: v.loc[:_cut] for k, v in _F.items()}, "1D")
_d = max(float((_I2[k].astype(float) - _I1[k].loc[:_cut].astype(float)).abs().max().max()) for k in _I2)
assert _d < 1e-4, f"rank indicators are not causal (max diff {_d})"
print(f"point-in-time audit passed for the {len(_I2)} rank indicators (1D): max |diff| after cutting at {_cut.date()} = {_d:.1e}"); del _F, _I1, _I2
causal_test("Step 19 rank indicators (1D)", lambda F_: rank_indicators(F_, "1D")[0], tf_frames("1D"), step="19")
if len(load_p5()):
    _F5 = tf_frames("5m"); _ix5 = next(iter(_F5.values())).index
    causal_test("Step 19 rank indicators (5m)", lambda F_: rank_indicators(F_, "5m")[0], _F5, cuts=[_ix5[int(len(_ix5) * q)] for q in (0.5, 0.8)], step="19"); del _F5
RANKS, RANK_EVAL, RANK_W, LATEST = {}, [], {}, {}
t0 = time.time()
for tf in TFS:
    F_ = tf_frames(tf)
    if F_ is None:
        print(f"{tf}: no data — skipped"); continue
    I, FAMI, fwd = rank_indicators(F_, tf); spl = split_index(F_["ret"].index, emb=max(EMB, RANK_H[tf])); del F_
    nfam = pd.Series(FAMI).value_counts()
    w_rule = {k: FAMILY_W[f] / nfam[f] for k, f in FAMI.items()}; o_rule = {k: 1.0 for k in I}
    w_l, o_l = {}, {}
    for k, v in I.items():
        ic = xs_ic_fast(v.loc[spl["train"]], fwd.loc[spl["train"]], min_n=5) if FAMI[k] != "breadth" else pd.Series(dtype=float)
        if len(ic) > 30:
            m_, t_ = ic.mean(), ic_t(ic, RANK_H[tf])
        else:
            m_ = ts_ic_pooled(v.loc[spl["train"]], fwd.loc[spl["train"]]); t_ = m_ * np.sqrt(len(spl["train"]) / RANK_H[tf]) if np.isfinite(m_) else np.nan
        o_l[k] = float(np.sign(m_)) if np.isfinite(m_) and m_ != 0 else 1.0
        w_l[k] = abs(m_) if np.isfinite(t_) and abs(t_) > 1 else 0.2 * abs(m_) if np.isfinite(m_) else 0.0
    if sum(w_l.values()) == 0:
        w_l, o_l = w_rule, o_rule
    RANK_W[tf] = pd.DataFrame({"family": FAMI, "rule_w": w_rule, "learned_w": w_l, "learned_dir": o_l})
    for variant, (w, o) in {"rule": (w_rule, o_rule), "learned": (w_l, o_l)}.items():
        U, Dn = updn(I, w, o); N_ = U + Dn
        RANKS[(tf, variant)] = (U, Dn)
        for sp in ("train", "val", "test"):
            idx = spl[sp]; ic = xs_ic_fast(N_.loc[idx], fwd.loc[idx], min_n=5)
            f_ = fwd.loc[idx].stack(); u_ = U.loc[idx].stack().reindex(f_.index); d_ = Dn.loc[idx].stack().reindex(f_.index)
            bull, bear = u_ >= RANK_THRESH[tf], d_ <= -RANK_THRESH[tf]
            RANK_EVAL.append(dict(tf=tf, variant=variant, split=sp, ic=ic.mean(), ic_t=ic_t(ic, RANK_H[tf]),
                                  bull_n=int(bull.sum()), bull_fwd=f_[bull].mean(), bull_hit=(f_[bull] > 0).mean(),
                                  bear_n=int(bear.sum()), bear_fwd=f_[bear].mean(), bear_hit=(f_[bear] < 0).mean(), all_fwd=f_.mean()))
        if variant == RANK_VARIANT:
            lastrow = N_.notna().any(axis=1)
            LATEST[tf] = pd.DataFrame({"UpR": U[lastrow].iloc[-1], "DnR": Dn[lastrow].iloc[-1]}); LATEST[tf].attrs["time"] = U[lastrow].index[-1]
    del I, fwd; gc.collect()
    print(f"{tf}: done ({time.time()-t0:.0f}s)")
REVAL = pd.DataFrame(RANK_EVAL); REVAL.to_csv(OUT2 / "rank_validation.csv", index=False)
print("\nRank validation (IC of Net vs forward return; bull/bear forward vol-scaled return and hit rate):")
print(REVAL.pivot_table(index=["tf", "variant"], columns="split", values=["ic", "ic_t", "bull_fwd", "bear_fwd"]).round(3).to_string())
print("\nLearned indicator directions (−1 = works opposite to the rule) and weights:")
print(pd.concat({tf: w.learned_dir for tf, w in RANK_W.items()}, axis=1).to_string())

In [ ]:
# step 19
# daily alignment, composite, MTF rule
def daily_align(df, tf):
    if tf == "1D":
        out = df.reindex(RAW.index)
    elif tf == "1W":
        out = df.reindex(RAW.index, method="ffill")
    else:
        out = df.groupby(to_trading_day(df.index)).last().reindex(RAW.index)
    return out.reindex(columns=RAW.columns)

RD = {tf: tuple(daily_align(x, tf) for x in RANKS[(tf, RANK_VARIANT)]) for tf in TFS if (tf, RANK_VARIANT) in RANKS}
RD_RULE = {tf: tuple(daily_align(x, tf) for x in RANKS[(tf, "rule")]) for tf in TFS if (tf, "rule") in RANKS}
def composite(Rd):
    num_u = num_d = den = 0
    for tf, (U, Dn) in Rd.items():
        a = U.notna(); w = TF_WEIGHTS[tf]
        num_u = num_u + w * U.fillna(0); num_d = num_d + w * Dn.fillna(0); den = den + w * a
    den = den.where(den > 0)
    return num_u / den, num_d / den
def mtf_state(Rd):
    tfs = [t for t in MTF_RULE_TFS if t in Rd] or [t for t in ("1D", "1W") if t in Rd]
    nb = sum((Rd[t][0] >= RANK_THRESH[t]).astype(int) for t in tfs); ns_ = sum((Rd[t][1] <= -RANK_THRESH[t]).astype(int) for t in tfs)
    na = sum(Rd[t][0].notna().astype(int) for t in tfs)
    st = pd.DataFrame(0, index=RAW.index, columns=RAW.columns)
    st[(nb == na) & (na >= 2)] = 1; st[(ns_ == na) & (na >= 2)] = -1
    return st.where(na >= 2)
COMP_UP, COMP_DN = composite(RD); COMP_NET = COMP_UP + COMP_DN
MTF_STATE = mtf_state(RD); MTF_STATE_RULE = mtf_state(RD_RULE)
pd.concat({"up": COMP_UP, "dn": COMP_DN}, axis=1).to_parquet(OUT2 / "rank_composite_daily.parquet")

mrows = []
for h in (1, 5, 10):
    fw = FWD[h] if h in FWD else RAW.rolling(h, min_periods=h).sum().shift(-h) / (SD_D * np.sqrt(h))
    for sp in ("train", "val", "test"):
        idx = RAW.index.intersection(SPL[sp]); f_ = fw.loc[idx].stack(); st_ = MTF_STATE.loc[idx].stack().reindex(f_.index)
        ic = xs_ic_fast(COMP_NET.loc[idx], fw.loc[idx])
        for lab, m in [("BULL", st_ == 1), ("BEAR", st_ == -1), ("all", st_.notna())]:
            v = f_[m]
            mrows.append(dict(h=h, split=sp, state=lab, n=len(v), mean_fwd=v.mean(), hit=(v > 0).mean() if lab != "BEAR" else (v < 0).mean(),
                              t=v.mean() / v.std() * np.sqrt(len(v) / h) if len(v) > 5 else np.nan, comp_ic=ic.mean()))
MTFE = pd.DataFrame(mrows); MTFE.to_csv(OUT2 / "mtf_rule_validation.csv", index=False)
print(f"MTF rule on {[t for t in MTF_RULE_TFS if t in RD]} (threshold {RANK_THRESH}): forward vol-scaled returns after signals")
print(MTFE.pivot_table(index=["state", "h"], columns="split", values=["n", "mean_fwd", "hit", "t"]).round(3).to_string())

# rank dynamics: persistence and transitions
pers = {lag: np.nanmean([COMP_NET[c].autocorr(lag) for c in COMP_NET if COMP_NET[c].notna().sum() > 50]) for lag in (1, 5, 20)}
st = MTF_STATE.stack(); prev = MTF_STATE.shift(1).stack().reindex(st.index)
TRANS = pd.crosstab(prev, st, normalize="index").rename(index={-1: "BEAR", 0: "NEUTRAL", 1: "BULL"}, columns={-1: "BEAR", 0: "NEUTRAL", 1: "BULL"})
dur = {lab: 1 / (1 - TRANS.loc[lab, lab]) if lab in TRANS.index and lab in TRANS.columns and TRANS.loc[lab, lab] < 1 else np.nan for lab in ["BULL", "BEAR"]}
print(f"\ncomposite Net persistence (autocorrelation): {dict((k, round(v, 3)) for k, v in pers.items())}")
print("MTF state transition matrix (daily):\n", TRANS.round(3).to_string(), "\nexpected duration (days):", {k: round(v, 1) for k, v in dur.items()})

In [ ]:
# step 19
# the scorecard: latest bar of every time frame
SC = pd.concat({tf: L for tf, L in LATEST.items()}, axis=1)
SC[("MTF", "UpR")] = COMP_UP.ffill().iloc[-1]; SC[("MTF", "DnR")] = COMP_DN.ffill().iloc[-1]
SC[("MTF", "Net")] = SC[("MTF", "UpR")] + SC[("MTF", "DnR")]
SC[("MTF", "state")] = MTF_STATE.ffill().iloc[-1].map({1: "BULL", -1: "BEAR", 0: "NEUTRAL"})
SC = SC.sort_values(("MTF", "Net"), ascending=False); SC.insert(0, ("", "sector"), SC.index.map(SEC_MAP))
SC.to_csv(OUT2 / "rank_scorecard_latest.csv")
print("Latest bar per frame:", {tf: str(L.attrs.get("time")) for tf, L in LATEST.items()})
print(SC.round(1).head(25).to_string())

def rank_panel(ax, L, title, n=20):
    d_ = L.dropna().sort_values("UpR", ascending=False).head(n)
    ax.imshow(np.zeros((len(d_), 2)), cmap="Greys", vmin=0, vmax=1, aspect="auto")
    for i, (p, r) in enumerate(d_.iterrows()):
        ax.add_patch(plt.Rectangle((-0.5, i - 0.5), 1, 1, color=plt.cm.Blues(0.15 + 0.75 * r.UpR / 100)))
        ax.add_patch(plt.Rectangle((0.5, i - 0.5), 1, 1, color=plt.cm.Reds(0.15 + 0.75 * -r.DnR / 100)))
        ax.text(0, i, f"{r.UpR:.0f}", ha="center", va="center", fontsize=7, color="white" if r.UpR > 55 else INK)
        ax.text(1, i, f"{r.DnR:.0f}", ha="center", va="center", fontsize=7, color="white" if r.DnR < -55 else INK)
    ax.set_yticks(range(len(d_))); ax.set_yticklabels(d_.index, fontsize=7); ax.set_xticks([0, 1]); ax.set_xticklabels(["UpR", "DnR"], fontsize=7)
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(len(d_) - 0.5, -0.5); ax.grid(False); ax.set_title(title, fontsize=9)
frames = [tf for tf in TFS if tf in LATEST] + ["MTF"]
fig, axs = plt.subplots(1, len(frames), figsize=(2.3 * len(frames), 6.5))
for ax, tf in zip(np.atleast_1d(axs), frames):
    L = LATEST[tf] if tf != "MTF" else pd.DataFrame({"UpR": SC[("MTF", "UpR")], "DnR": SC[("MTF", "DnR")]})
    rank_panel(ax, L, f"{tf} — bull rank")
fig.suptitle(f"Multi-time-frame bull rank — latest ({RANK_VARIANT} weights)", fontweight="bold", color=INK)
plt.tight_layout()
savefig(fig, "19_mtf_scorecard", "Multi-time-frame bull rank — latest", "19")

fig, axs = plt.subplots(1, 3, figsize=(18, 4.5))
ax = axs[0]
r_ = REVAL[(REVAL.split == "test")].pivot(index="tf", columns="variant", values="ic").reindex([t for t in TFS if t in LATEST])
x = np.arange(len(r_))
for k, v in enumerate(["rule", "learned"]):
    ax.bar(x + (k - 0.5) * 0.35, r_[v], width=0.35, color=[MUTE, SERIES[0]][k], label=v)
ax.set_xticks(x); ax.set_xticklabels(r_.index); ax.axhline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=8)
ax.set_ylabel("test IC of Net vs forward return"); ax.set_title("Does the rank predict, per time frame? (test)")
ax = axs[1]; d5 = MTFE[(MTFE.h == 5) & (MTFE.state != "all")]
for k, stt in enumerate(["BULL", "BEAR"]):
    v = d5[d5.state == stt].set_index("split").reindex(["train", "val", "test"])
    ax.bar(np.arange(3) + (k - 0.5) * 0.35, v.mean_fwd, width=0.35, color=[SERIES[0], SERIES[1]][k], label=stt)
ax.set_xticks(range(3)); ax.set_xticklabels(["train", "val", "test"]); ax.axhline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=8)
ax.set_ylabel("mean 5-day forward vol-scaled return"); ax.set_title(f"MTF rule ({'/'.join(t for t in MTF_RULE_TFS if t in RD)} ≥ threshold)")
ax = axs[2]
last = COMP_NET.ffill().iloc[-120:]; order_ = last.iloc[-1].sort_values(ascending=False).index
heat(ax, last[order_].T.iloc[:, ::5].set_axis([d.strftime("%m-%d") for d in last.index[::5]], axis=1), cmap=DIV, center=0, vmax=100, label="composite Net")
ax.set_title("Composite rank history (last 120 days)")
plt.tight_layout()
savefig(fig, "19_rank_validation", "Rank validation and history", "19")

---
# Part E — Regimes (Step 20)
## Step 20 — Regimes, structural breaks and rule changes

**Regime definitions.** Every one uses information available at the time, so all of them can be used for trading:

| Regime | Definition |
|---|---|
| Volatility | median across products of σₜ / σ̄ (σ = Step 11's validation-selected model, a one-step-ahead forecast); LOW / MID / HIGH by train terciles |
| Markov switching | 2-state switching-mean/variance model on the equal-weight market return, fitted on train. **Filtered** (not smoothed) probabilities → CALM / TURBULENT |
| Breadth | share of products above their 50-day EMA: BULL > `BREADTH_BULL`, BEAR < `BREADTH_BEAR`, otherwise NEUTRAL |

**Breaks and events:**
* **Data-driven variance breaks** per product: ICSS binary segmentation with the kurtosis-robust κ₁ statistic.
* **Rule-change event studies** for `RULE_EVENTS` and each product's night-session start (detected in Step 1). These compare 120 days before and after: volatility, volume, autocorrelation, and whether a statistical break falls near the event.

**What changes across regimes:**
* Factor IC and factor *ranking* by regime.
* MTF rank skill by regime.
* Granger link strength by regime.
* Regime-specific factor weights, used by strategy D in Step 23.
* The impact on every strategy, also in Step 23.

**Used later.** Regime labels feed the regime-switching strategy D (Step 23), the P&L attribution, and the regime behaviour of factors and ranks in Steps 21–22.

In [ ]:
# step 20
MKT = RAW[UNIV].mean(axis=1)
rel_sig = SIGMA[UNIV_V].div(SIGMA[UNIV_V].loc[SIGMA.index.intersection(SPL["train"])].mean())
MVOL = rel_sig.median(axis=1).reindex(RAW.index)
q1_, q2_ = MVOL.loc[idx_tr].quantile(VOL_REGIME_Q[0]), MVOL.loc[idx_tr].quantile(VOL_REGIME_Q[1])
VOL_STATE = pd.Series(np.select([MVOL <= q1_, MVOL >= q2_], ["LOW", "HIGH"], "MID"), index=RAW.index).where(MVOL.notna())
y_ = (MKT * 100).dropna(); ytr_ = y_.loc[y_.index <= SPL["train"][-1]]
try:
    ms_res = MarkovRegression(ytr_.values, k_regimes=2, trend="c", switching_variance=True).fit(disp=False)
    hi_ = int(np.argmax([ms_res.params[4], ms_res.params[5]]))
    filt = MarkovRegression(y_.values, k_regimes=2, trend="c", switching_variance=True).filter(ms_res.params)
    P_HIGH = pd.Series(np.asarray(filt.filtered_marginal_probabilities)[:, hi_], index=y_.index).reindex(RAW.index)
    print("Markov switching (train):", dict(zip(["p00", "p10", "mu0", "mu1", "var0", "var1"], np.round(ms_res.params, 4))))
except Exception as e:
    print("Markov switching failed:", e); P_HIGH = pd.Series(np.nan, index=RAW.index)
MS_STATE = pd.Series(np.where(P_HIGH > 0.5, "TURBULENT", "CALM"), index=RAW.index).where(P_HIGH.notna())
BREADTH = (B["breadth_mkt"].iloc[:, 0] + 0.5)
BR_STATE = pd.Series(np.select([BREADTH > BREADTH_BULL, BREADTH < BREADTH_BEAR], ["BULL", "BEAR"], "NEUTRAL"), index=RAW.index).where(BREADTH.notna())
REG = pd.DataFrame({"vol": VOL_STATE, "ms": MS_STATE, "breadth": BR_STATE})
REG["combo"] = REG.vol + "/" + REG.breadth
REG.to_csv(OUT2 / "regimes_daily.csv")
for col in ("vol", "ms", "breadth"):
    s_ = REG[col].dropna(); tm = pd.crosstab(s_.shift(1), s_, normalize="index")
    dur = {k: round(1 / (1 - tm.loc[k, k]), 1) if k in tm.columns and tm.loc[k, k] < 1 else np.inf for k in tm.index}
    print(f"\n{col} regime — share of days: {s_.value_counts(normalize=True).round(3).to_dict()} | expected duration (days): {dur}")
print(f"\ncurrent regime ({RAW.index[-1].date()}): {REG.ffill().iloc[-1].to_dict()}")

In [ ]:
# step 20
# structural breaks (ICSS) and rule-change event studies
brk_rows = []
for s in UNIV:
    r = RAW[s].dropna()
    for pos in icss_breaks(r.values):
        if pos < len(r):
            brk_rows.append(dict(product=s, sector=SEC_MAP[s], date=r.index[pos],
                                 vol_before=r.iloc[max(0, pos - 60):pos].std() * np.sqrt(ANN), vol_after=r.iloc[pos:pos + 60].std() * np.sqrt(ANN)))
BRK = pd.DataFrame(brk_rows); BRK.to_csv(OUT2 / "variance_breaks.csv", index=False)
events = [(pd.Timestamp(d), [p for p in prods if p in RAW.columns], desc) for d, prods, desc in RULE_EVENTS]
if "night_since" in quality_raw:
    for r in quality_raw.dropna(subset=["night_since"]).itertuples():
        if r.product in RAW.columns and pd.Timestamp(r.night_since) > RAW[r.product].first_valid_index() + pd.Timedelta(days=60):
            events.append((pd.Timestamp(r.night_since), [r.product], "night session introduced"))
events = [e for e in events if e[1] and RAW.index[0] + pd.Timedelta(days=EVENT_WIN) < e[0] < RAW.index[-1] - pd.Timedelta(days=EVENT_WIN)]
es_rows = []
for d, prods, desc in events:
    for p in prods:
        r = RAW[p].dropna(); idx = r.index; k = idx.searchsorted(d)
        pre, post = idx[max(0, k - EVENT_GAP - EVENT_WIN):max(0, k - EVENT_GAP)], idx[k + EVENT_GAP:k + EVENT_GAP + EVENT_WIN]
        if len(pre) < 40 or len(post) < 40:
            continue
        lrv_ = np.log(RV[p].where(RV[p] > 0)); lvo = np.log1p(VOLU[p])
        t_rv = stats.ttest_ind(lrv_.loc[post].dropna(), lrv_.loc[pre].dropna(), equal_var=False)
        t_vo = stats.ttest_ind(lvo.loc[post].dropna(), lvo.loc[pre].dropna(), equal_var=False)
        near = BRK[(BRK["product"] == p) & ((BRK.date - d).abs() <= pd.Timedelta(days=60))] if len(BRK) else []
        es_rows.append(dict(date=d.date(), product=p, event=desc, vol_change_pct=(np.exp(lrv_.loc[post].mean() - lrv_.loc[pre].mean()) - 1) * 100,
                            vol_t=t_rv.statistic, volume_change_pct=(np.exp(lvo.loc[post].mean() - lvo.loc[pre].mean()) - 1) * 100, volume_t=t_vo.statistic,
                            ac1_pre=r.loc[pre].autocorr(1), ac1_post=r.loc[post].autocorr(1), break_within_60d=len(near) > 0))
EVSTUDY = pd.DataFrame(es_rows)
if len(EVSTUDY):
    EVSTUDY.to_csv(OUT2 / "rule_change_event_study.csv", index=False)
    print("Rule-change / session event study (120 days after vs before):\n", EVSTUDY.round(3).to_string(index=False))
print(f"\nICSS variance breaks: {len(BRK)} across {BRK['product'].nunique() if len(BRK) else 0} products")

In [ ]:
# step 20
# what changes across regimes: factors, ranks, Granger links
topf = [f for f in FSCORE.index if f in ICSER][:12]
rf_rows = []
for f in topf:
    ic = ICSER[f]
    for var in ("vol", "ms", "breadth"):
        lab = REG[var].reindex(ic.index)
        for sp, idx in (("train", idx_tr), ("test", idx_te)):
            g = ic.loc[ic.index.intersection(idx)].groupby(lab.loc[ic.index.intersection(idx)])
            for state, v in g:
                rf_rows.append(dict(feature=f, regime=var, state=state, split=sp, mean_ic=v.mean(), t=ic_t(v, 5), n=len(v)))
REGF = pd.DataFrame(rf_rows); REGF.to_csv(OUT2 / "factor_ic_by_regime.csv", index=False)
icn = xs_ic_fast(COMP_NET, FWD[5])
rk_rows = [dict(regime=var, state=st_, split=sp, rank_ic=v.mean(), t=ic_t(v, 5), n=len(v))
           for var in ("vol", "ms", "breadth") for sp, idx in (("train", idx_tr), ("val", idx_va), ("test", idx_te))
           for st_, v in icn.loc[icn.index.intersection(idx)].groupby(REG[var].reindex(icn.index.intersection(idx)))]
REGR = pd.DataFrame(rk_rows)
print("Composite MTF rank IC (5-day) by regime:\n", REGR.pivot_table(index=["regime", "state"], columns="split", values="rank_ic").round(4).to_string())
def pr2_masked(Zf, c, e, p, mask):
    y, Xr, Xu = design(Zf, c, e, p, CONDITION_ON_MARKET); m = np.isfinite(y) & np.isfinite(Xu).all(1) & mask
    if m.sum() < 60:
        return np.nan
    rss_r, _ = ols(Xr[m], y[m]); rss_u, _ = ols(Xu[m], y[m])
    return (rss_r - rss_u) / rss_r
gl_rows = []
if "1D" in GRES:
    Zf = Z[UNIV]
    for c, e in GRES["1D"]["CAND"][["cause", "effect"]].values:
        row = {"link": f"{c}→{e}"}
        for st_ in ("LOW", "MID", "HIGH"):
            row[f"pr2_{st_}"] = pr2_masked(Zf, c, e, P_LAG_TF["1D"], (REG.vol.reindex(Zf.index) == st_).to_numpy())
        gl_rows.append(row)
GLREG = pd.DataFrame(gl_rows)
if len(GLREG):
    print("\nDaily Granger link strength (partial R²) by volatility regime:\n", GLREG.round(4).to_string(index=False))
# regime-specific factor weights (train) for strategy D
REG_W = {}
for st_ in ("LOW", "MID", "HIGH"):
    w = {}
    for f in [f for f in SEL_F if f in ICSER]:
        ic = ICSER[f]; m = (REG.vol.reindex(ic.index) == st_) & ic.index.isin(idx_tr)
        if m.sum() > 30:
            w[f] = ic[m].mean()
    if w:
        REG_W[st_] = w
if len(REGF):
    piv = REGF[(REGF.regime == "vol") & (REGF.split == "train")].pivot(index="feature", columns="state", values="mean_ic")
    ranks_by = piv.rank(ascending=False)
    print("\nFactor rank by volatility regime (1 = best IC, train):\n", ranks_by.round(0).to_string())
    if piv.shape[1] > 1:
        print("Spearman between regime rankings:\n", piv.corr(method="spearman").round(2).to_string())

In [ ]:
# step 20
fig, axs = plt.subplots(2, 2, figsize=(17, 8.5), gridspec_kw={"height_ratios": [1, 1.1]})
ax = axs[0, 0]
cm_ = (MKT.fillna(0).cumsum() * 100)
ax.plot(cm_.index, cm_.values, color=INK, lw=1.2, label="equal-weight market (cum. log %)")
hv = (REG.vol == "HIGH").to_numpy()
ax.fill_between(cm_.index, cm_.min(), cm_.max(), where=hv, color=SERIES[1], alpha=0.12, linewidth=0, label="HIGH vol regime")
ax2 = ax.twinx(); ax2.plot(P_HIGH.index, P_HIGH.values, color=SERIES[0], lw=0.7, alpha=0.7); ax2.set_ylim(0, 1); ax2.set_ylabel("P(turbulent)", color=SERIES[0]); ax2.grid(False)
for sp in ("val", "test"):
    ax.axvline(SPL[sp][0], color=INK2, ls="--", lw=1)
ax.legend(frameon=False, fontsize=7, loc="upper left"); ax.set_title("Regimes over time (shaded = HIGH vol; blue = Markov P(turbulent))")
ax = axs[0, 1]
if len(BRK):
    prods_ = sorted(BRK["product"].unique(), key=lambda p: (SEC_MAP[p], p)); ypos = {p: i for i, p in enumerate(prods_)}
    up_ = BRK.vol_after > BRK.vol_before
    ax.scatter(BRK.date, BRK["product"].map(ypos), c=np.where(up_, SERIES[1], SERIES[0]), s=14)
    ax.set_yticks(range(len(prods_))); ax.set_yticklabels(prods_, fontsize=6)
    for d, prods, desc in events:
        ax.axvline(d, color=MUTE, lw=0.8)
    ax.scatter([], [], color=SERIES[1], label="vol up"); ax.scatter([], [], color=SERIES[0], label="vol down"); ax.legend(frameon=False, fontsize=7)
ax.set_title("Variance breaks (ICSS) with rule-change / session events (grey lines)")
ax = axs[1, 0]
if len(REGF):
    pv = REGF[(REGF.split == "train")].pivot_table(index="feature", columns=["regime", "state"], values="mean_ic")
    heat(ax, pv, cmap=DIV, center=0, label="mean 5-day IC (train)")
    ax.set_title("Factor IC by regime (does the factor's sign or strength change?)")
ax = axs[1, 1]
rr = REGR[REGR.regime == "vol"].pivot(index="state", columns="split", values="rank_ic").reindex(["LOW", "MID", "HIGH"])
x = np.arange(len(rr))
for k, sp in enumerate(["train", "val", "test"]):
    if sp in rr:
        ax.bar(x + (k - 1) * 0.27, rr[sp], width=0.27, color=[MUTE, "#86b6ef", SERIES[0]][k], label=sp)
ax.set_xticks(x); ax.set_xticklabels(rr.index); ax.axhline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=8)
ax.set_title("Composite MTF rank IC by volatility regime")
plt.tight_layout()
savefig(fig, "20_regimes", "Regimes, breaks, regime-dependent factors", "20")

---
# Part F — Dynamics and monitoring (Steps 21–22)
Steps 18–20 say *which* factors, ranks and relationships have worked. Two further questions decide whether they can be traded:

| Step | Question |
|---|---|
| 21 | Do factors, sectors and factor × sector cells show **momentum**, **mean reversion** or **rotation** in their own returns? Which factor works in which sector, and does that map persist? Do crowding and dispersion tell when rotation pays? |
| 22 | When does an edge **stop working**? A drift-and-decay monitor for every factor, factor × sector cell, rotation strategy, and Granger / pair / fundamental relationship, with a rank-drift measure and a test of whether switching off decaying edges helps out of sample. |

## Step 21 — Factor, sector and factor × sector dynamics: momentum, mean reversion, rotation

**Objects.** Three kinds of return series, all built without look-ahead from the Step 18 factor library:
* **Factor**: for each family × level (e.g. *momentum, within-sector*), the daily return of the long-short portfolio that trades that factor (orientation fixed on train, volatility-targeted, before costs).
* **Sector**: the equal-risk return of each sector, relative to the market.
* **Factor × sector cell**: the return of a family's own-history (TS) signal traded only inside one sector. It can be negative, which says the factor works the other way in that sector.

**Three behaviours**, tested on each series on train, then validation and test:
1. **Time-series momentum / reversal.** Rank IC of the trailing $L$-day return with the next $h$ days (grid `DYN_LOOKBACKS` × `DYN_HOLDS`); best configuration chosen on train only. Plus the **variance ratio** $VR(q)$ (Lo–MacKinlay, heteroskedasticity-robust): above 1 is momentum, below 1 is reversal; and the half-life of the cumulative return's deviation from its 250-day mean.
2. **Rotation** (cross-sectional): every $h$ days rank the objects by trailing return; hold the top $k$ (momentum) or the bottom $k$ (reversal). For sectors the strategy is long the top $k$ and short the bottom $k$. For factors and cells it is the top $k$ against holding all of them.
3. **Mean reversion of the cell**: the z-score of the cumulative return against its own 250-day history; stretched below its mean should recover.

**Honest accounting.** Many series × configurations means some will look good by chance. Two safeguards: the configuration is chosen on train only, and the table reports the **replication rate** (the share of train-significant series that keep the sign in validation and test), to be compared with the 50% chance level.

**Strategies produced** (added to the library in Step 23): F0 equal-weight factors (benchmark), F1 factor rotation, F2 sector rotation, F3 factor × sector rotation, F4 factor timing (hold a factor only while its trailing return has the chosen sign), F5 factor × sector mean reversion.

**Used later.** The chosen rotation / mean-reversion configurations feed strategies F0–F5 (Step 23). The state variables feed the board tilt. The dynamics series feed the monitor (Step 22).

In [ ]:
# step 21
# ---- 21.0 execution toolkit (shared by Steps 21-25) ----
PRODS = list(RAW.columns); RS = SIMPLE.fillna(0.0)
tick = pd.Series({p: TICK.get(p, np.nan) for p in PRODS})
COST = (FEE_BPS * 1e-4 + SLIP_TICKS * tick / CLOSE).fillna(DEFAULT_COST_BPS * 1e-4)
ROLLN = ROLL.astype(float)          # roll cost charged on the roll day itself (the switch is only known when it happens)
TAU = TARGET_VOL / np.sqrt(ANN)
CAP = pd.Series({p: min(LEV_CAP, 1 / MARGIN[p]) if p in MARGIN else LEV_CAP for p in PRODS})
WF_START = SPL["val"][0]
REFITS = [WF_START] + [d for d in RAW.index.to_series().groupby(RAW.index.year).first() if d > WF_START]
sec_n = SEC_MAP.value_counts()

def to_weights(S):
    S = S.reindex(index=RAW.index, columns=PRODS).fillna(0.0)
    n_act = (S != 0).sum(axis=1).replace(0, np.nan)
    W = S.div(SD_D).mul(TAU).div(np.sqrt(n_act), axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return W.clip(lower=-CAP, upper=CAP, axis=1)

def backtest(W, delay, cost_fn=None, lock=None):
    # cost_fn(turn, Wd) -> daily cost (Step 23a execution model); lock: bool frame, True = limit-locked day (no trading, position held)
    Wd = W.shift(delay).fillna(0.0)
    if lock is not None:
        Wd = Wd.mask(lock.reindex(index=Wd.index, columns=Wd.columns).fillna(False).astype(bool)).ffill().fillna(0.0)
    gross = (Wd.shift(1).fillna(0.0) * RS).sum(axis=1)
    turn = Wd.diff().abs().fillna(Wd.abs())
    cost = (turn * COST).sum(axis=1) + (2 * Wd.abs() * ROLLN * COST).sum(axis=1) if cost_fn is None else cost_fn(turn, Wd)
    return pd.DataFrame({"gross": gross, "net": gross - cost.shift(1).fillna(0.0), "turn": turn.sum(axis=1)})

def period_mask(index, k):
    d0 = REFITS[k]; d1 = REFITS[k + 1] if k + 1 < len(REFITS) else index[-1] + pd.Timedelta(days=1)
    return (index >= d0) & (index < d1)

def q_ls(X, q=0.2):
    ok = X.notna().sum(axis=1) >= MIN_XS; pr = X.rank(axis=1, pct=True)
    S = (pr > 1 - q).astype(float) - (pr <= q).astype(float); S.loc[~ok] = 0.0
    return S

def ws_ls(X):
    S = pd.DataFrame(0.0, index=X.index, columns=X.columns)
    for sec, n_ in sec_n.items():
        cols = [c for c in X.columns if SEC_MAP[c] == sec]
        if n_ < MIN_SEC_WS or len(cols) < MIN_SEC_WS:
            continue
        sub = X[cols]; ok = (sub.notna().sum(axis=1) >= MIN_SEC_WS).to_numpy()[:, None]
        S[cols] = (sub.eq(sub.max(axis=1), axis=0) & ok).astype(float) - (sub.eq(sub.min(axis=1), axis=0) & ok).astype(float)
    return S

def sec_ls(X, k=2):
    secm = X.T.groupby(SEC_MAP.reindex(X.columns)).mean().T; n_ = secm.notna().sum(axis=1)
    rk = secm.rank(axis=1, ascending=False)
    Ss = rk.le(k).astype(float) - rk.sub(n_, axis=0).gt(-k).astype(float); Ss.loc[n_ < 2 * k + 1] = 0.0
    S = Ss.reindex(columns=SEC_MAP.reindex(X.columns).values).set_axis(X.columns, axis=1)
    return S.where(X.notna(), 0.0)

In [ ]:
# step 21
# ---- 21.1 factor, sector and factor x sector return series (no look-ahead: signal at close t, return of day t+1) ----
shp = lambda x: (x.dropna().mean() / x.dropna().std() * np.sqrt(ANN)) if x.dropna().size > 20 and x.dropna().std() > 0 else np.nan
IDX = {sp: RAW.index.intersection(SPL[sp]) for sp in ("train", "val", "test")}
gross_ret = lambda W: backtest(W.astype(float), 0)["gross"]

FSIG, FW = {}, {}
for fam in sorted(FSCORE.family.unique()):
    for lev in ("XS", "WS", "SEC", "TS"):
        fl = FSCORE[(FSCORE.family == fam) & (FSCORE.level == lev)].index
        if not len(fl):
            continue
        X_ = sum(FEATS[f].astype(float) * FSCORE.loc[f, "orient"] for f in fl) / len(fl)
        S_ = {"XS": q_ls, "WS": ws_ls, "SEC": sec_ls}.get(lev, lambda x: np.sign(x).fillna(0.0))(X_)
        FSIG[(fam, lev)] = S_; FW[(fam, lev)] = to_weights(S_).astype("float32")
FR = pd.DataFrame({f"{fam}|{lev}": gross_ret(W) for (fam, lev), W in FW.items()})
FR = FR.loc[:, (FR.notna().sum() > 300) & (FR.std() > 0)]

sec_prods = {s: [c for c in PRODS if SEC_MAP[c] == s] for s in sorted(set(SEC_MAP[PRODS]))}
SECS = [s for s, v in sec_prods.items() if len(v) >= 2]
FCELL_W = {}
for fam in sorted(FSCORE.family.unique()):
    if (fam, "TS") not in FSIG:
        continue
    for sec in SECS:
        S_ = FSIG[(fam, "TS")].copy(); S_.loc[:, [c for c in PRODS if c not in sec_prods[sec]]] = 0.0
        FCELL_W[(fam, sec)] = to_weights(S_).astype("float32")
FS = pd.DataFrame({f"{fam}|{sec}": gross_ret(W) for (fam, sec), W in FCELL_W.items()})
FS = FS.loc[:, (FS.notna().sum() > 300) & (FS.std() > 0)]
SECZ = Z[UNIV].T.groupby(SEC_MAP.reindex(UNIV)).mean().T
SECZ = SECZ[[s for s in SECS if s in SECZ.columns]]
SECREL = SECZ.sub(Z[UNIV].mean(axis=1), axis=0)             # sector return relative to the market, in sigma units
print(f"{FR.shape[1]} factor series | {SECREL.shape[1]} sectors | {FS.shape[1]} factor x sector cells | sectors used: {list(SECREL.columns)}")
print("train Sharpe of the factor portfolios (gross):")
print(pd.DataFrame({sp: FR.loc[IDX[sp]].apply(shp) for sp in IDX}).round(2).sort_values("train", ascending=False).to_string())

In [ ]:
# step 21
# ---- 21.2 time-series dynamics: momentum / reversal IC, variance ratios, half-lives ----
def tic(past, fut, idx, h):
    d = pd.concat([past.loc[idx], fut.loc[idx]], axis=1).dropna()
    if len(d) < 60:
        return np.nan, np.nan
    c = d.iloc[:, 0].corr(d.iloc[:, 1], method="spearman"); n_eff = max(len(d) / h, 5)
    return c, c * np.sqrt(n_eff - 2) / np.sqrt(max(1 - c * c, 1e-9))

def vr_test(r, q):
    # Lo-MacKinlay variance ratio with heteroskedasticity-robust z
    r = np.asarray(pd.Series(r).dropna(), float); n = len(r)
    if n < 3 * q + 50:
        return np.nan, np.nan
    mu = r.mean(); d = r - mu; s1 = (d ** 2).sum() / (n - 1)
    rq = pd.Series(r).rolling(q).sum().dropna().to_numpy(); m = q * (n - q + 1) * (1 - q / n)
    sq = ((rq - q * mu) ** 2).sum() / m; vr = sq / s1
    d2 = d ** 2; den = d2.sum() ** 2
    theta = sum((2 * (q - j) / q) ** 2 * (n * (d2[j:] * d2[:-j]).sum() / den) for j in range(1, q))
    return vr, (vr - 1) / np.sqrt(theta / n)

def ou_halflife(r, idx):
    c = r.fillna(0).cumsum(); dev = (c - c.rolling(DYN_Z_WIN, min_periods=100).mean()).loc[idx].dropna()
    if len(dev) < 100:
        return np.nan
    b = np.polyfit(dev.shift(1).dropna().values, dev.diff().dropna().values, 1)[0]
    return -np.log(2) / np.log(1 + b) if -1 < b < 0 else np.inf

def dyn_table(RET, cls):
    rows = []
    for c in RET.columns:
        r = RET[c]
        for L in DYN_LOOKBACKS:
            past = r.rolling(L, min_periods=L).sum()
            for h in DYN_HOLDS:
                fut = r.rolling(h, min_periods=h).sum().shift(-h); row = dict(cls=cls, series=c, L=L, h=h)
                for sp in IDX:
                    row[f"ic_{sp}"], row[f"t_{sp}"] = tic(past, fut, IDX[sp], h)
                rows.append(row)
    T = pd.DataFrame(rows)
    best = T.loc[T.groupby("series").t_train.apply(lambda s: s.abs().idxmax() if s.notna().any() else s.index[0])].set_index("series")
    for q in VR_Q:
        for sp in IDX:
            vz = {c: vr_test(RET[c].loc[IDX[sp]], q) for c in RET.columns}
            best[f"vr{q}_{sp}"] = pd.Series({c: v[0] for c, v in vz.items()}); best[f"vrz{q}_{sp}"] = pd.Series({c: v[1] for c, v in vz.items()})
    best["halflife_train"] = pd.Series({c: ou_halflife(RET[c], IDX["train"]) for c in RET.columns})
    best["behaviour"] = np.select([(best.t_train > 2), (best.t_train < -2)], ["momentum", "reversal"], "none")
    nz = best.behaviour != "none"
    best["val_ok"] = nz & (np.sign(best.ic_val) == np.sign(best.ic_train)) & (best.t_val.abs() > 1)
    best["test_ok"] = best.val_ok & (np.sign(best.ic_test) == np.sign(best.ic_train))
    best["test_sign_same"] = nz & (np.sign(best.ic_test) == np.sign(best.ic_train))
    return best

DYN = pd.concat([dyn_table(FR, "factor"), dyn_table(SECREL, "sector"), dyn_table(FS, "factor_sector")])
DYN.to_csv(OUT2 / "dynamics_time_series.csv")
summ = []
for cls, d in DYN.groupby("cls"):
    nz = d[d.behaviour != "none"]
    summ.append(dict(cls=cls, series=len(d), momentum_train=(d.behaviour == "momentum").sum(), reversal_train=(d.behaviour == "reversal").sum(),
                     val_ok=int(d.val_ok.sum()), test_ok=int(d.test_ok.sum()),
                     replication_val_test=nz.test_sign_same.mean() if len(nz) else np.nan,
                     binom_p=stats.binomtest(int(nz.test_sign_same.sum()), len(nz), 0.5, alternative="greater").pvalue if len(nz) else np.nan,
                     mean_VR20_test=d.vr20_test.mean()))
DYN_SUM = pd.DataFrame(summ).set_index("cls"); DYN_SUM.to_csv(OUT2 / "dynamics_summary.csv")
print("Time-series momentum / reversal of the objects' own returns (best (L, h) chosen on train):")
print(DYN_SUM.round(3).to_string())
print("\nBest-supported series (train |t|, kept sign in validation and test):")
print(DYN[DYN.test_ok][["cls", "behaviour", "L", "h", "ic_train", "t_train", "ic_val", "ic_test", "vr20_test", "halflife_train"]].round(3).sort_values("ic_train", key=abs, ascending=False).head(15).to_string())

In [ ]:
# step 21
# ---- 21.3 rotation: rank the objects by trailing return, hold the winners (momentum) or the losers (reversal) ----
def alloc(RET, L, h, mode, k, kind):
    # allocation matrix (rows = decision date t, known at close t). kind "ls": long top-k / short bottom-k; "long": long top-k only
    past = RET.rolling(L, min_periods=L).sum(); sc = past if mode == "mom" else -past
    rk = sc.rank(axis=1, ascending=False); n = sc.notna().sum(axis=1)
    if kind == "ls":
        A = (rk <= k).astype(float) / k - (rk > n.values[:, None] - k).astype(float) / k
        A[n < 2 * k + 1] = 0.0
    else:
        A = (rk <= k).astype(float) / k; A[n < k + 1] = 0.0
    A = A.copy(); A.iloc[np.flatnonzero((np.arange(len(A)) % h) != 0)] = np.nan          # rebalance every h days, hold in between
    return A.ffill().fillna(0.0)

def rot_pnl(RET, L, h, mode, k, kind):
    A = alloc(RET, L, h, mode, k, kind)
    pnl = (A.shift(1) * RET.fillna(0.0)).sum(axis=1)
    if kind != "ls":                                   # relative to holding every object with equal weight
        pnl = pnl - RET.mean(axis=1).fillna(0.0) * (A.sum(axis=1).shift(1) > 0)
    return pnl

ROT_ROWS, CHOSEN = [], {}
for cls, RET, kind in (("factor", FR, "long"), ("sector", SECREL, "ls"), ("factor_sector", FS, "long")):
    k = DYN_TOPK[cls]
    for mode in ("mom", "rev"):
        for L in DYN_LOOKBACKS:
            for h in DYN_HOLDS:
                p_ = rot_pnl(RET, L, h, mode, k, kind)
                ROT_ROWS.append(dict(cls=cls, mode=mode, L=L, h=h, k=k, **{f"sharpe_{sp}": shp(p_.loc[IDX[sp]]) for sp in IDX}))
ROT = pd.DataFrame(ROT_ROWS)
for cls, d in ROT.groupby("cls"):
    b = d.loc[d.sharpe_train.idxmax()]; CHOSEN[cls] = dict(mode=b["mode"], L=int(b.L), h=int(b.h), k=int(b.k))
ROT.to_csv(OUT2 / "dynamics_rotation.csv", index=False)
print("Rotation (active return vs holding all objects; sectors long-short). Configuration chosen on TRAIN Sharpe, then frozen:")
print(pd.DataFrame({c: {**v, **{f"sharpe_{sp}": ROT[(ROT.cls == c) & (ROT["mode"] == v["mode"]) & (ROT.L == v["L"]) & (ROT.h == v["h"])][f"sharpe_{sp}"].iloc[0] for sp in IDX}}
                    for c, v in CHOSEN.items()}).T.round(2).to_string())
print("\nMean Sharpe over the whole grid (all 16 configurations, i.e. no selection), by mode:")
print(ROT.groupby(["cls", "mode"])[["sharpe_train", "sharpe_val", "sharpe_test"]].mean().round(2).to_string())
ROT_PNL = {c: rot_pnl({"factor": FR, "sector": SECREL, "factor_sector": FS}[c], v["L"], v["h"], v["mode"], v["k"], "ls" if c == "sector" else "long") for c, v in CHOSEN.items()}

In [ ]:
# step 21
# ---- 21.4 factor x sector map: which factor works where, and does the map persist? ----
FSM = {sp: FS.loc[IDX[sp]].apply(shp) for sp in IDX}
def to_map(s):
    m = pd.DataFrame({"fam": [i.split("|")[0] for i in s.index], "sec": [i.split("|")[1] for i in s.index], "v": s.values})
    return m.pivot(index="fam", columns="sec", values="v")
FSMAP = {sp: to_map(FSM[sp]) for sp in IDX}
a_, b_ = FSM["train"], FSM["test"]; ok_ = a_.notna() & b_.notna()
rho_tt = stats.spearmanr(a_[ok_], b_[ok_])[0] if ok_.sum() > 5 else np.nan
same = (np.sign(a_[ok_]) == np.sign(b_[ok_])).mean()
strong = a_[ok_][a_[ok_].abs() > 0.5].index
same_strong = (np.sign(a_[strong]) == np.sign(b_[strong])).mean() if len(strong) else np.nan
top = a_[ok_].sort_values(ascending=False).head(8).index; bot = a_[ok_].sort_values().head(8).index
print(f"factor x sector cells: {ok_.sum()} | Spearman(train map, test map) = {rho_tt:.2f} | same sign train/test: {same:.0%} (chance 50%) | "
      f"among cells with |train Sharpe| > 0.5: {same_strong:.0%} ({len(strong)} cells)")
print(f"top-8 train cells: test Sharpe {b_[top].mean():.2f} | bottom-8 train cells: test Sharpe {b_[bot].mean():.2f} | all cells: {b_[ok_].mean():.2f}")
print("\nTrain Sharpe map (rows = factor family, columns = sector):\n", FSMAP["train"].round(1).to_string())
# an economic read-out: where is each family best (train) and what did it do in test?
best_sec = pd.DataFrame({fam: {"best_sector_train": FSMAP["train"].loc[fam].idxmax(), "train": FSMAP["train"].loc[fam].max(),
                               "test_same_cell": FSMAP["test"].loc[fam, FSMAP["train"].loc[fam].idxmax()]} for fam in FSMAP["train"].index}).T
print("\nBest sector per family (train) and the same cell in test:\n", best_sec.round(2).to_string())
FSMAP_TBL = pd.concat({sp: m.stack() for sp, m in FSMAP.items()}, axis=1); FSMAP_TBL.to_csv(OUT2 / "factor_sector_map.csv")

In [ ]:
# step 21
# ---- 21.5 state variables: does crowding or dispersion tell when rotation pays? ----
def avg_corr(X, win=STATE_WIN):
    m = X.mean(axis=1); n = X.notna().sum(axis=1)
    vm = m.rolling(win, min_periods=win // 2).var(); vi = X.rolling(win, min_periods=win // 2).var().mean(axis=1)
    return ((n * vm - vi) / ((n - 1) * vi)).clip(-1, 1)
STATE = pd.DataFrame({"avg_corr_products": avg_corr(Z[UNIV]),
                      "sector_dispersion": SECREL.std(axis=1).rolling(STATE_WIN, min_periods=STATE_WIN // 2).mean(),
                      "factor_crowding": avg_corr(FR)})
rows = []
for var in STATE.columns:
    s_ = STATE[var].shift(1)                                # known before the P&L day
    q1_, q2_ = s_.loc[IDX["train"]].quantile(1 / 3), s_.loc[IDX["train"]].quantile(2 / 3)
    terc = pd.Series(np.select([s_ <= q1_, s_ >= q2_], ["low", "high"], "mid"), index=s_.index).where(s_.notna())
    for cls, p_ in ROT_PNL.items():
        for tc in ("low", "mid", "high"):
            m = terc == tc
            rows.append(dict(state=var, cls=cls, tercile=tc, sharpe_train=shp(p_.loc[IDX["train"]][m.reindex(IDX["train"]).fillna(False).to_numpy()]),
                             sharpe_val_test=shp(p_.loc[IDX["val"].append(IDX["test"])][m.reindex(IDX["val"].append(IDX["test"])).fillna(False).to_numpy()])))
STATE_TBL = pd.DataFrame(rows); STATE_TBL.to_csv(OUT2 / "dynamics_state_variables.csv", index=False)
print("Frozen rotation strategies: Sharpe by tercile of each state variable (state known the day before):")
print(STATE_TBL.pivot_table(index=["state", "cls"], columns="tercile", values=["sharpe_train", "sharpe_val_test"]).round(2).to_string())
print("\nstate now:", STATE.iloc[-1].round(2).to_dict())

In [ ]:
# step 21
# ---- 21.6 rotation / timing / mean-reversion strategies for the library (Step 23) ----
def combine(Wd, A):
    # weight-space allocation: W_t = sum_j A[t, j] * W_j[t]
    out = np.zeros((len(RAW.index), len(PRODS)), dtype="float64")
    for j, key in enumerate(Wd):
        a = A.iloc[:, j].to_numpy()
        if np.any(a != 0):
            out += a[:, None] * Wd[key].reindex(RAW.index).fillna(0.0).to_numpy()
    return pd.DataFrame(out, index=RAW.index, columns=PRODS).clip(lower=-CAP, upper=CAP, axis=1)

NEW_STRATS = {}
fkeys = [tuple(c.split("|")) for c in FR.columns]; Wf = {k: FW[k] for k in fkeys}
ckeys = [tuple(c.split("|")) for c in FS.columns]; Wc = {k: FCELL_W[k] for k in ckeys}
n_f, n_c = len(fkeys), len(ckeys)
NEW_STRATS["F0 Factors equal-weight (benchmark)"] = dict(family="F Dynamics: benchmark", W=combine(Wf, pd.DataFrame(1.0 / n_f, index=RAW.index, columns=FR.columns)))
cf = CHOSEN["factor"]
NEW_STRATS[f"F1 Factor rotation ({cf['mode']} L{cf['L']} h{cf['h']} top{cf['k']})"] = dict(
    family="F Dynamics: factor", W=combine(Wf, alloc(FR, cf["L"], cf["h"], cf["mode"], cf["k"], "long")))
cs_ = CHOSEN["sector"]; A_s = alloc(SECREL, cs_["L"], cs_["h"], cs_["mode"], cs_["k"], "ls")
S_sec = pd.DataFrame(0.0, index=RAW.index, columns=PRODS)
for sec in A_s.columns:
    S_sec[sec_prods[sec]] = np.repeat(np.sign(A_s[sec].to_numpy())[:, None], len(sec_prods[sec]), axis=1)
NEW_STRATS[f"F2 Sector rotation ({cs_['mode']} L{cs_['L']} h{cs_['h']} ±{cs_['k']})"] = dict(family="F Dynamics: sector", W=to_weights(S_sec))
cc = CHOSEN["factor_sector"]
NEW_STRATS[f"F3 Factor x sector rotation ({cc['mode']} L{cc['L']} h{cc['h']} top{cc['k']})"] = dict(
    family="F Dynamics: factor x sector", W=combine(Wc, alloc(FS, cc["L"], cc["h"], cc["mode"], cc["k"], "long")))
pastF = FR.rolling(cf["L"], min_periods=cf["L"]).sum(); gate = ((pastF > 0) if cf["mode"] == "mom" else (pastF < 0)).astype(float) / n_f
NEW_STRATS[f"F4 Factor timing ({cf['mode']} L{cf['L']})"] = dict(family="F Dynamics: factor timing", W=combine(Wf, gate.where(pastF.notna(), 0.0)))
CS = FS.fillna(0).cumsum(); zc = ((CS - CS.rolling(DYN_Z_WIN, min_periods=100).mean()) / CS.rolling(DYN_Z_WIN, min_periods=100).std()).clip(-3, 3)
NEW_STRATS["F5 Factor x sector mean reversion"] = dict(family="F Dynamics: factor x sector", W=combine(Wc, (-zc / 3 / n_c).fillna(0.0)))
for k_, v in NEW_STRATS.items():
    g_ = gross_ret(v["W"])
    print(f"{k_:62s} gross Sharpe  train {shp(g_.loc[IDX['train']]):5.2f} | val {shp(g_.loc[IDX['val']]):5.2f} | test {shp(g_.loc[IDX['test']]):5.2f}")

In [ ]:
# step 21
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
ax = axs[0]; cl = ["factor", "sector", "factor_sector"]; x = np.arange(len(cl))
mom = [DYN_SUM.loc[c, "momentum_train"] for c in cl]; rev = [DYN_SUM.loc[c, "reversal_train"] for c in cl]; tot = [DYN_SUM.loc[c, "series"] for c in cl]
ok = [DYN_SUM.loc[c, "test_ok"] for c in cl]
ax.bar(x - 0.27, np.array(mom) / tot, width=0.26, color=SERIES[0], label="momentum (train |t|>2)")
ax.bar(x, np.array(rev) / tot, width=0.26, color=SERIES[1], label="reversal (train |t|>2)")
ax.bar(x + 0.27, np.array(ok) / tot, width=0.26, color=SERIES[2], label="keeps sign in validation and test")
ax.set_xticks(x); ax.set_xticklabels(cl); ax.set_ylabel("share of series"); ax.legend(frameon=False, fontsize=7)
ax.set_title("Own-return momentum / reversal by object class")
ax = axs[1]
for k, (cls, p_) in enumerate(ROT_PNL.items()):
    ax.plot(p_.cumsum().index, p_.cumsum().values / max(p_.loc[IDX["train"]].std(), 1e-9), color=SERIES[k], lw=1.3, label=f"{cls}: {CHOSEN[cls]['mode']} L{CHOSEN[cls]['L']} h{CHOSEN[cls]['h']}")
for sp in ("val", "test"):
    ax.axvline(SPL[sp][0], color=INK2, ls="--", lw=1)
ax.legend(frameon=False, fontsize=7); ax.set_ylabel("cumulative active return (train σ units)"); ax.set_title("Rotation (config chosen on train; dashed = validation, test)")
ax = axs[2]
heat(ax, FSMAP["test"].reindex(index=FSMAP["train"].index, columns=FSMAP["train"].columns), cmap=DIV, center=0, vmax=3, label="test Sharpe", annot=True, fmt="{:.1f}")
ax.set_title(f"Factor × sector map, TEST (rows ordered as train; train–test ρ = {rho_tt:.2f})")
plt.tight_layout()
savefig(fig, "21_dynamics", "Factor, sector and factor × sector dynamics", "21")
fig, axs = plt.subplots(1, 3, figsize=(18, 4.6))
ax = axs[0]
heat(ax, FSMAP["train"], cmap=DIV, center=0, vmax=3, label="train Sharpe", annot=True, fmt="{:.1f}"); ax.set_title("Factor × sector map, TRAIN")
ax = axs[1]
ax.scatter(FSM["train"], FSM["test"], s=14, color=SERIES[0]); lim = max(np.nanmax(np.abs(FSM["train"])), np.nanmax(np.abs(FSM["test"])), 1)
ax.plot([-lim, lim], [-lim, lim], color=INK2, ls="--", lw=1); ax.axhline(0, color=MUTE, lw=0.8); ax.axvline(0, color=MUTE, lw=0.8)
ax.set_xlabel("train Sharpe of the cell"); ax.set_ylabel("test Sharpe of the same cell"); ax.set_title(f"Does the map persist? same sign {same:.0%}")
ax = axs[2]; sp_ = STATE_TBL.pivot_table(index=["state", "cls"], columns="tercile", values="sharpe_val_test").reindex(columns=["low", "mid", "high"])
heat(ax, sp_.set_axis([f"{a[:12]} | {b[:8]}" for a, b in sp_.index]), cmap=DIV, center=0, vmax=2, label="Sharpe (val+test)", annot=True, fmt="{:.1f}")
ax.set_title("Rotation Sharpe by tercile of the state variable")
plt.tight_layout()
savefig(fig, "21_factor_sector_state", "Factor × sector map and state variables", "21")

## Step 22 — Drift and decay monitor: when does an edge stop working?

Every edge is a daily **edge series** $x_t$ (the gross return of the factor, cell, rotation strategy, or relationship), signed so that its **train mean is positive**. The monitor then asks, using data up to $t$ only, whether the edge is still there:

| Piece | Definition |
|---|---|
| train baseline | $\mu_0$ and $\sigma_0$ of $x_t$ on train; the edge counts if the train Sharpe is at least `MON_MIN_TRAIN_SHARPE` |
| rolling skill | Sharpe and t-stat over `MON_WIN` days |
| **CUSUM (Page)** | the log-likelihood-ratio form $S_t=\max\{0,\;S_{t-1}+\mu_0(\mu_0/2-x_t/\sigma_0)\}$: evidence that the mean has fallen from $\mu_0$ to 0. The alarm level $\ln(\text{ARL}_0)$ (Lorden) makes false alarms rarer than one per `MON_ARL0` days *whatever the size of the edge*. |
| status | **WORKING**; **WEAKENING** ($S_t>\tfrac12\ln$ ARL₀ or rolling t < −1.5); **BROKEN** ($S_t>\ln$ ARL₀ or rolling t < −2.5); **EMERGING (+/−)** (no train edge, rolling t beyond ±3); **NO EDGE** |
| **rank drift** | the Spearman correlation between today's ranking of the factors (or cells) by rolling Sharpe and the ranking `RANK_DRIFT_LAG` days earlier. Low means the ranking is unstable, so rotation has nothing stable to exploit. |
| gating test | does trading an edge only while its status on the *previous* day is WORKING or EMERGING(+) improve validation and test Sharpe? This is the monitor's real value. |

**Detection is slow by nature.** A Sharpe-1 edge has a daily mean of only about 6% of its daily standard deviation. With one false alarm allowed per `MON_ARL0` days, a CUSUM needs roughly $2\ln(\text{ARL}_0)/\mu_0^2$ days of *zero* return to alarm: years for Sharpe 1, months for Sharpe 3. The fast rolling t-test catches only large, sudden failures. The tables report the actual alarm dates, so the delay is visible, and the gating test says whether the monitor earns its keep.

**Objects monitored:** `factor:` portfolios, `cell:` factor × sector, `rot:` the Step 21 rotation strategies, `gr:` frozen daily Granger links, `pair:` cointegrated pairs, `chain:` fundamental chains, `rank:` the composite MTF rank.

**Used later.** Each edge's status feeds the gated books G1 / G2 (Step 23) and the health adjustment of the board (Step 26).

In [ ]:
# step 22
EDGE = {}
for c in FR.columns: EDGE[f"factor:{c}"] = FR[c]
for c in FS.columns: EDGE[f"cell:{c}"] = FS[c]
for k_, v in NEW_STRATS.items():
    if not k_.startswith("F0"):
        EDGE[f"rot:{k_.split(' (')[0]}"] = gross_ret(v["W"])
# frozen daily Granger links: edge = sign(frozen predictor at t-1) x standardised effect return at t
if "1D" in GRES:
    p_ = P_LAG_TF["1D"]
    for mdl in GRES["1D"]["frozen"]["models"]:
        c, e = mdl["cause"], mdl["effect"]; b = np.array(mdl["beta_arx"][-p_:])
        S = np.sign(sum(b[i] * Z[c].shift(i) for i in range(p_)))
        EDGE[f"gr:{c}→{e}"] = (S.shift(1) * Z[e]) * 0.01                       # sigma units -> ~1% daily risk, for a comparable scale
# cointegrated pairs and fundamental chains: edge = -z(t-1) (clipped) x spread change at t
def spread_edge(a, legs, beta):
    s = LP[a] - LP[legs] @ np.array(beta); zz = (s - s.rolling(PAIR_Z_WIN).mean()) / s.rolling(PAIR_Z_WIN).std()
    return (-zz.shift(1).clip(-2, 2) * s.diff())
for x in COINT[COINT.valid].itertuples():
    EDGE[f"pair:{x.a}~{x.b}"] = spread_edge(x.a, x.b.split("+"), x.beta)
for x in FUND[FUND.valid].itertuples():
    EDGE[f"chain:{x.chain[:34]}"] = spread_edge(x.dep, list(x.drivers), x.beta)
EDGE["rank:MTF composite long-short"] = gross_ret(to_weights(q_ls(COMP_NET)))
EDGE = pd.DataFrame(EDGE).reindex(RAW.index)
EDGE = EDGE.loc[:, EDGE.loc[IDX["train"]].notna().sum() > 200]

def cusum_run(xs, mu0, k):
    C = np.zeros(len(xs)); c = 0.0
    for i, v in enumerate(xs):
        c = max(0.0, c + (mu0 - k) - v) if np.isfinite(v) else c
        C[i] = c
    return C

MON, SIGN0, BASE = {}, {}, {}
for name in EDGE.columns:
    r = EDGE[name]; tr_ = r.loc[IDX["train"]].dropna()
    s0 = 1.0 if tr_.mean() >= 0 else -1.0; x = s0 * r; sd0 = tr_.std()
    if not np.isfinite(sd0) or sd0 == 0:
        continue
    mu0 = tr_.mean() * s0 / sd0; base_sh = mu0 * np.sqrt(ANN); xs = (x / sd0)
    roll_m, roll_s = xs.rolling(MON_WIN, min_periods=MON_WIN // 2).mean(), xs.rolling(MON_WIN, min_periods=MON_WIN // 2).std()
    roll_t = roll_m / roll_s * np.sqrt(MON_WIN); roll_sh = roll_m / roll_s * np.sqrt(ANN)
    has_edge = base_sh >= MON_MIN_TRAIN_SHARPE
    C = pd.Series(mu0 * cusum_run(xs.to_numpy(), mu0, mu0 / 2) if has_edge else np.zeros(len(xs)), index=xs.index)     # log-likelihood-ratio scale
    h_ = np.log(MON_ARL0)
    if has_edge:
        st = np.select([(C > h_) | (roll_t < -2.5), (C > h_ / 2) | (roll_t < -1.5)], ["BROKEN", "WEAKENING"], "WORKING")
    else:
        st = np.select([roll_t > 3, roll_t < -3], ["EMERGING(+)", "EMERGING(-)"], "NO EDGE")
    st = pd.Series(st, index=xs.index).where(roll_t.notna() | (C > 0), "NO DATA").where(xs.notna().cumsum() > MON_WIN // 2, "NO DATA")
    MON[name] = pd.DataFrame({"x": x, "status": st, "cusum": C, "roll_sharpe": roll_sh, "roll_t": roll_t})
    SIGN0[name] = s0; BASE[name] = base_sh
MON_STATUS = pd.DataFrame({k: v.status for k, v in MON.items()})
OKST = ["WORKING", "EMERGING(+)"]
GATE = MON_STATUS.isin(OKST)

# latest status table + alarm dates
rows = []
for name, m in MON.items():
    st = m.status; broken = (st == "BROKEN")
    runs = broken & ~broken.shift(1, fill_value=False); alarms = list(st.index[runs])
    after = [d for d in alarms if d > SPL["train"][-1]]
    rows.append(dict(object=name, kind=name.split(":")[0], train_sharpe=BASE[name], sign_train=SIGN0[name],
                     sharpe_val=shp(m.x.loc[IDX["val"]]), sharpe_test=shp(m.x.loc[IDX["test"]]),
                     roll_sharpe_now=m.roll_sharpe.dropna().iloc[-1] if m.roll_sharpe.notna().any() else np.nan, roll_t_now=m.roll_t.dropna().iloc[-1] if m.roll_t.notna().any() else np.nan,
                     cusum_now=m.cusum.iloc[-1], status_now=st.iloc[-1], share_working=(st.loc[IDX["val"].append(IDX["test"])] == "WORKING").mean(),
                     first_alarm_after_train=after[0].date() if after else None, n_alarms=len(alarms),
                     days_in_status=int((st != st.iloc[-1]).iloc[::-1].cumsum().eq(0).sum())))
MON_NOW = pd.DataFrame(rows).set_index("object").sort_values(["status_now", "train_sharpe"], ascending=[True, False]); MON_NOW.to_csv(OUT2 / "monitor_latest.csv")
print(f"{len(MON)} edge series monitored | with a train edge (Sharpe >= {MON_MIN_TRAIN_SHARPE}): {(MON_NOW.train_sharpe >= MON_MIN_TRAIN_SHARPE).sum()}")
print("\nStatus now, by kind:\n", pd.crosstab(MON_NOW.kind, MON_NOW.status_now).to_string())
print("\nEdges with an alarm after train, and what they did out of sample:")
print(MON_NOW[MON_NOW.first_alarm_after_train.notna()][["train_sharpe", "sharpe_val", "sharpe_test", "first_alarm_after_train", "status_now", "roll_sharpe_now"]].round(2).head(25).to_string())

In [ ]:
# step 22
# gating test: trade an edge only while its status on the PREVIOUS day is WORKING or EMERGING(+)
g_rows = []
for name, m in MON.items():
    if BASE[name] < MON_MIN_TRAIN_SHARPE:
        continue
    ok_prev = GATE[name].shift(1, fill_value=False); xg = m.x.where(ok_prev, 0.0)
    for sp in ("val", "test"):
        g_rows.append(dict(object=name, kind=name.split(":")[0], split=sp, always_on=shp(m.x.loc[IDX[sp]]), gated=shp(xg.loc[IDX[sp]]),
                           days_on=ok_prev.loc[IDX[sp]].mean(), mean_always=m.x.loc[IDX[sp]].mean(), mean_gated=xg.loc[IDX[sp]].mean()))
GT = pd.DataFrame(g_rows)
if len(GT):
    GT["improves"] = GT.gated > GT.always_on
    GT.to_csv(OUT2 / "monitor_gating_test.csv", index=False)
    s_ = GT.groupby(["kind", "split"]).agg(n=("object", "size"), always_on=("always_on", "mean"), gated=("gated", "mean"), improves=("improves", "mean"), days_on=("days_on", "mean"))
    print("Gating test (mean Sharpe over objects that had a train edge):\n", s_.round(2).to_string())
    for sp in ("val", "test"):
        d = GT[GT.split == sp]; print(f"{sp}: gated beats always-on for {d.improves.mean():.0%} of {len(d)} edges (sign test p = "
                                      f"{stats.binomtest(int(d.improves.sum()), len(d), 0.5, alternative='greater').pvalue:.3f}); edges that decayed (always-on Sharpe < 0): "
                                      f"{(d.always_on < 0).sum()}, of which gating helped {(d[d.always_on < 0].improves).sum()}")

# rank drift: how stable is the ranking of the factors / cells?
def rank_drift(RET, win=60, lag=RANK_DRIFT_LAG):
    rs = RET.rolling(win, min_periods=win // 2).mean() / RET.rolling(win, min_periods=win // 2).std()
    rk = rs.rank(axis=1)
    cor = pd.Series(np.nan, index=rk.index)
    A = rk.to_numpy(); B_ = rk.shift(lag).to_numpy()
    for i in range(len(A)):
        a, b = A[i], B_[i]; m = np.isfinite(a) & np.isfinite(b)
        if m.sum() >= 5 and np.std(a[m]) > 0 and np.std(b[m]) > 0:
            cor.iloc[i] = np.corrcoef(a[m], b[m])[0, 1]
    return cor, rk
RD_F, RK_F = rank_drift(FR); RD_C, RK_C = rank_drift(FS)
RANK_DRIFT = pd.DataFrame({"factor": RD_F, "factor_sector": RD_C}); RANK_DRIFT.to_csv(OUT2 / "monitor_rank_drift.csv")
print(f"\nRank stability (Spearman of today's rolling-Sharpe ranking vs {RANK_DRIFT_LAG} days earlier): "
      + ", ".join(f"{c}: train {RANK_DRIFT[c].loc[IDX['train']].mean():.2f} | val {RANK_DRIFT[c].loc[IDX['val']].mean():.2f} | test {RANK_DRIFT[c].loc[IDX['test']].mean():.2f}" for c in RANK_DRIFT))

# monitor-gated books for the strategy library
def gated_book(prefix, Wd, RET):
    keys = [tuple(c.split("|")) for c in RET.columns]; cols = [f"{prefix}:{c}" for c in RET.columns]
    ok = GATE.reindex(columns=cols).fillna(False).astype(float)
    sgn = pd.Series([SIGN0.get(c, 1.0) for c in cols], index=cols)
    ok = ok * (pd.Series([BASE.get(c, 0) for c in cols], index=cols) >= MON_MIN_TRAIN_SHARPE).astype(float)
    n = ok.sum(axis=1).replace(0, np.nan)
    A = (ok.mul(sgn, axis=1).div(n, axis=0)).fillna(0.0)
    return combine({k: Wd[k] for k in keys}, A.set_axis(RET.columns, axis=1))
NEW_STRATS["G1 Monitor-gated factors"] = dict(family="G Monitor-gated", W=gated_book("factor", Wf, FR))
NEW_STRATS["G2 Monitor-gated factor x sector cells"] = dict(family="G Monitor-gated", W=gated_book("cell", Wc, FS))
for k_ in ("G1 Monitor-gated factors", "G2 Monitor-gated factor x sector cells"):
    g_ = gross_ret(NEW_STRATS[k_]["W"])
    print(f"{k_:42s} gross Sharpe  train {shp(g_.loc[IDX['train']]):5.2f} | val {shp(g_.loc[IDX['val']]):5.2f} | test {shp(g_.loc[IDX['test']]):5.2f}")

In [ ]:
# step 22
fig, axs = plt.subplots(1, 3, figsize=(18, 6), gridspec_kw={"width_ratios": [1.5, 1, 1]})
ax = axs[0]
show = MON_NOW[MON_NOW.train_sharpe >= MON_MIN_TRAIN_SHARPE].sort_values("train_sharpe", ascending=False).head(34).index
code_ = {"WORKING": 0, "WEAKENING": 1, "BROKEN": 2, "EMERGING(+)": 3, "EMERGING(-)": 4, "NO EDGE": 5, "NO DATA": np.nan}
M = MON_STATUS[show].iloc[::10].apply(lambda col: col.map(lambda v: code_.get(v, np.nan))).T
cm = matplotlib.colors.ListedColormap([SERIES[2], SERIES[3], SERIES[1], SERIES[0], "#86b6ef", MUTE])
ax.imshow(M.values.astype(float), aspect="auto", cmap=cm, vmin=-0.5, vmax=5.5, interpolation="nearest",
          extent=[0, M.shape[1], M.shape[0], 0]); ax.grid(False)
ax.set_yticks(np.arange(len(show)) + 0.5); ax.set_yticklabels([s[:34] for s in show], fontsize=6)
xt = np.linspace(0, M.shape[1] - 1, 7).astype(int); ax.set_xticks(xt + 0.5); ax.set_xticklabels([MON_STATUS.index[::10][i].strftime("%Y-%m") for i in xt], fontsize=7)
ax.set_title("Status over time (green working · amber weakening · red broken · blue emerging)")
for lab, c in (("working", SERIES[2]), ("weakening", SERIES[3]), ("broken", SERIES[1])):
    ax.scatter([], [], color=c, marker="s", label=lab)
ax.legend(frameon=False, fontsize=7, loc="lower left")
ax = axs[1]
RANK_DRIFT.rolling(60, min_periods=20).mean().plot(ax=ax, lw=1.2, color=SERIES[:2]); ax.axhline(0, color=INK2, lw=1)
for sp in ("val", "test"):
    ax.axvline(SPL[sp][0], color=INK2, ls="--", lw=1)
ax.set_title(f"Rank stability ({RANK_DRIFT_LAG}-day lag)"); ax.set_ylabel("Spearman of rankings"); ax.legend(frameon=False, fontsize=8)
ax = axs[2]
if len(GT):
    d = GT[GT.split == "test"]
    for k, kd in enumerate(sorted(d.kind.unique())):
        dd = d[d.kind == kd]; ax.scatter(dd.always_on, dd.gated, s=16, color=SERIES[k % 8], label=kd)
    lim = max(d[["always_on", "gated"]].abs().max().max(), 1)
    ax.plot([-lim, lim], [-lim, lim], color=INK2, ls="--", lw=1); ax.axhline(0, color=MUTE, lw=0.8); ax.axvline(0, color=MUTE, lw=0.8)
    ax.set_xlabel("test Sharpe, always on"); ax.set_ylabel("test Sharpe, monitor-gated"); ax.legend(frameon=False, fontsize=6)
    ax.set_title("Does the monitor help? (above the line = yes)")
plt.tight_layout()
savefig(fig, "22_monitor", "Drift and decay monitor", "22")
nm = MON_NOW[MON_NOW.first_alarm_after_train.notna()].train_sharpe.sort_values(ascending=False).head(4).index.tolist() or MON_NOW.head(2).index.tolist()
if nm:
    fig, axs = plt.subplots(1, len(nm), figsize=(4.6 * len(nm), 3.8), squeeze=False)
    for ax, name in zip(axs[0], nm):
        m = MON[name]; ax.plot(m.x.fillna(0).cumsum().index, m.x.fillna(0).cumsum().values / max(m.x.loc[IDX["train"]].std(), 1e-9), color=SERIES[0], lw=1.0)
        ax2 = ax.twinx(); ax2.plot(m.cusum.index, m.cusum.values, color=SERIES[1], lw=0.9); ax2.axhline(np.log(MON_ARL0), color=SERIES[1], ls="--", lw=0.8); ax2.grid(False)
        for sp in ("val", "test"):
            ax.axvline(SPL[sp][0], color=INK2, ls="--", lw=0.8)
        ax.set_title(name[:40], fontsize=8); ax.tick_params(labelsize=7)
    plt.tight_layout()
    savefig(fig, "22_monitor_examples", "Cumulative edge and CUSUM for edges that raised an alarm", "22")

---
# Part G — Strategies, portfolio and risk (Steps 23–25)

## Step 23a — Execution model: what a trade really costs

The strategy library (Step 23) so far pays a **flat** cost: fee + `SLIP_TICKS` ticks per side. A researcher checks that number before trusting any Sharpe ratio, because every high-turnover edge lives or dies on it. The data are 1-minute OHLCV bars of the **main contract only**: there are **no quotes, no order book and no queue positions**. So the model is built from what bars can reveal, and says where it can only give bounds.

| Piece | Estimated from | Used for |
|---|---|---|
| **Half-spread**, by product and time of day | Corwin–Schultz (2012) high–low estimator on 5-min bars (Roll's serial-covariance estimator when high / low are not cached), floored at half a tick and capped at one tick for liquid products (`EXEC_HS_CAP_TICKS` ticks for the least liquid third): bar-based estimates mostly measure volatility, so they are an upper bound | cost per side |
| **Market impact** | square-root law: cost per unit traded = `EXEC_Y` · σ_day · √(traded value / ADV), with ADV the lagged 20-day median traded value | cost grows with book size (`EXEC_AUM`) |
| **Passive vs aggressive** | a limit order resting at the previous close: *touch* fill (optimistic: queue ignored) and *through* fill (price traded one tick beyond: queue safely consumed); mark-out to the bar close after the fill | bounds on the fill rate and on adverse selection |
| **Limit-up / limit-down** | days that close at the exchange limit with a narrow range | no trading on a locked day: the position is held |
| **Time of day** | the same estimators by session bucket (night, open, morning, afternoon, close) | cost of intraday links (Step 23, C4) |

Everything is estimated on data **before the validation period** whenever the 5-min history allows it (`EXEC_EST_END`), so the backtest never prices trades with spreads it could not have known.

**What this cannot give.** True queue position and L1 spread need quote data. Time-of-day spreads from bars are noisy for the liquid products (the true spread is one tick, so the estimate is floored there). The honest reading is a *range*: flat cost, model cost, and the AUM at which an edge disappears (Step 23b).

**Used later.** The cost model re-costs the library in Step 23b and the intraday links in Step 23. Half-spreads and traded values also feed the capacity estimate of Step 24.

In [ ]:
# step 23a
# ---- 23a.1 data: 5-min high / low / close (cached by Step 1) and the estimation window ----
EXEC_EST_END = SPL["val"][0]                                   # estimate on bars before the validation period when there are enough of them
BARS = pd.read_parquet(CACHE / "bars5m.parquet") if (CACHE / "bars5m.parquet").exists() else pd.DataFrame()
HAS_HL = len(BARS) > 0 and {"hi", "lo", "cl"} <= set(BARS.columns)
if len(BARS):
    BARS = BARS[~BARS["product"].isin(EXCLUDE)]
    W5 = {c: BARS.pivot(index="date", columns="product", values=c).sort_index().astype(float) for c in (["ret", "volume"] + (["hi", "lo", "cl"] if HAS_HL else []))}
    n_pre = int((W5["ret"].index < EXEC_EST_END).sum())
    EXEC_SPAN = "before validation" if n_pre >= 5000 else "all bars (too few before validation)"
    if n_pre >= 5000:
        W5 = {k: v.loc[v.index < EXEC_EST_END] for k, v in W5.items()}
    del BARS
else:
    W5, EXEC_SPAN = {}, "no 5-min bars"
print(f"5-min bars for the execution model: {len(W5.get('ret', []))} bars ({EXEC_SPAN}); high/low/close cached: {HAS_HL}")
if W5 and not HAS_HL:
    print("  the cache was written by an older Step 1 (no high / low / close): spreads use Roll's estimator and passive fills are skipped; delete eda_cache/bars5m.parquet and re-run Step 1 to add them")

def tod_bucket(idx):
    m = (idx.hour * 60 + idx.minute).to_numpy(); lab = np.full(len(idx), "other", dtype=object)
    for nm, lo, hi in TOD_BUCKETS:
        lab[(((m >= lo) & (m < hi)) | ((m + 1440 >= lo) & (m + 1440 < hi))) & (lab == "other")] = nm
    return lab
BUCKETS = [b[0] for b in TOD_BUCKETS]
price_ref = CLOSE.iloc[-250:].median()                           # recent price level, for tick -> basis points
TICK_BP = (pd.Series({p: TICK.get(p, np.nan) for p in PRODS}) / price_ref.reindex(PRODS) * 1e4)

In [ ]:
# step 23a
# ---- 23a.2 half-spread by product and time of day ----
def corwin_schultz(H, L, consecutive):
    # Corwin & Schultz (2012): two-bar high-low estimator; bars that are not consecutive (session gaps, roll bars) are dropped
    a = np.log(H / L) ** 2; beta = a + a.shift(1)
    g = np.log(np.maximum(H, H.shift(1)) / np.minimum(L, L.shift(1))) ** 2
    k = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k - np.sqrt(g / k)
    S = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    return S.where(consecutive, np.nan).clip(lower=0)

HS_TOD = pd.DataFrame(np.nan, index=PRODS, columns=BUCKETS)          # half-spread (fraction of price) by product x bucket
CS_ALL = pd.Series(np.nan, index=PRODS); ROLL_ALL = pd.Series(np.nan, index=PRODS); VSHARE = pd.DataFrame(np.nan, index=PRODS, columns=BUCKETS)
if W5:
    idx5 = W5["ret"].index; bk = pd.Series(tod_bucket(idx5), index=idx5)
    cons = pd.Series(idx5, index=idx5).diff() <= pd.Timedelta(minutes=6)            # previous bar is the previous 5 minutes
    vol5 = W5["volume"]
    for p in PRODS:
        if p not in W5["ret"].columns:
            continue
        r = W5["ret"][p]; ok = cons & r.notna() & r.shift(1).notna()
        rr = r.where(ok)
        cv = rr.groupby(bk).apply(lambda x: x.cov(x.shift(1)) if x.notna().sum() > 200 else np.nan)       # bucket-level serial covariance, for Roll
        roll_b = 2 * np.sqrt(np.maximum(-cv, 0))
        ROLL_ALL[p] = 2 * np.sqrt(max(-rr.cov(rr.shift(1)), 0)) if rr.notna().sum() > 200 else np.nan
        if HAS_HL:
            H, L = W5["hi"][p], W5["lo"][p]; okh = (H > 0) & (L > 0) & (np.log(H / L).abs() < 0.1)
            S = corwin_schultz(H.where(okh), L.where(okh), cons)
            S_b = S.groupby(bk).mean(); CS_ALL[p] = S.mean()
            est = S_b.reindex(BUCKETS)
        else:
            est = roll_b.reindex(BUCKETS)
        vs = vol5[p].groupby(bk).sum().reindex(BUCKETS); VSHARE.loc[p] = (vs / vs.sum()).to_numpy()
        HS_TOD.loc[p] = (est / 2).to_numpy()
# half-spread used by the cost model: floored at half a tick, capped at EXEC_HS_CAP_TICKS ticks; products without bars fall back to one tick (the flat model)
tick_frac = (TICK_BP / 1e4).reindex(PRODS)
floor_ = (tick_frac / 2)
# a high-low estimator on 5-min bars mostly measures volatility, so it overstates the spread of liquid contracts that quote one tick wide: those are capped at one tick,
# the least liquid third (by traded value) at EXEC_HS_CAP_TICKS ticks. The estimate is an upper bound; the tick floor is the lower bound.
adv_med = wide("money").reindex(columns=PRODS).iloc[-250:].median()
illiquid = adv_med <= adv_med.quantile(1 / 3)
cap_ = tick_frac * pd.Series(np.where(illiquid, EXEC_HS_CAP_TICKS, 1.0), index=PRODS)
HS_TOD = HS_TOD.apply(lambda c: np.minimum(np.maximum(c, floor_), cap_))
HS_FALLBACK = tick_frac.fillna(DEFAULT_COST_BPS * 1e-4 / 2)
HS = (HS_TOD * VSHARE.fillna(1 / len(BUCKETS))).sum(axis=1, min_count=1) / (VSHARE.fillna(1 / len(BUCKETS)).where(HS_TOD.notna()).sum(axis=1))
HS = HS.fillna(HS_FALLBACK)                                       # volume-weighted across the day
FLAT_SLIP = SLIP_TICKS * tick_frac.fillna(DEFAULT_COST_BPS * 1e-4)
print(f"half-spread (bp, volume-weighted over the day): median {HS.median()*1e4:.2f} | flat model slippage (1 tick) median {FLAT_SLIP.median()*1e4:.2f} | "
      f"products where the estimate exceeds one tick: {(HS > tick_frac.fillna(np.inf) * 1.05).sum()} of {len(HS)}")
def tod_cost(prod, index):
    # per-side cost of a bar-level trade in `prod` at the timestamps `index`: fee + the half-spread of that time-of-day bucket
    hs = HS_TOD.loc[prod] if prod in HS_TOD.index else pd.Series(np.nan, index=BUCKETS)
    hb = pd.Series(tod_bucket(pd.DatetimeIndex(index))).map(hs).fillna(HS.get(prod, np.nanmedian(HS.values))).to_numpy()
    return FEE_BPS * 1e-4 + hb

In [ ]:
# step 23a
# ---- 23a.3 passive vs aggressive: fill-rate bounds and adverse selection (bars cannot see the queue) ----
PASSIVE = pd.DataFrame(index=PRODS, columns=["fill_touch", "fill_through", "markout_bp", "adverse_over_spread"], dtype=float)
PASSIVE_TOD = pd.DataFrame(np.nan, index=PRODS, columns=BUCKETS)
if W5 and HAS_HL:
    for p in PRODS:
        if p not in W5["hi"].columns:
            continue
        H, L, C = W5["hi"][p], W5["lo"][p], W5["cl"][p]; c0 = C.shift(1); tk = TICK.get(p, np.nan)
        ok = cons & c0.notna() & H.notna() & L.notna() & (np.log(H / L).abs() < 0.1)
        if not np.isfinite(tk) or ok.sum() < 500:
            continue
        buy_t, sell_t = (L <= c0), (H >= c0)                               # touched the resting price
        buy_x, sell_x = (L <= c0 - tk), (H >= c0 + tk)                     # traded through it by a full tick
        mo_buy = (C - c0) / c0 * 1e4; mo_sell = (c0 - C) / c0 * 1e4        # price after the fill, in favour of the passive side
        okb = ok & (np.abs(mo_buy) < 500)
        ft = ((buy_t & okb).sum() + (sell_t & okb).sum()) / (2 * okb.sum()); fx = ((buy_x & okb).sum() + (sell_x & okb).sum()) / (2 * okb.sum())
        # the limit price is c0 -/+ 0: a "through" buy is filled at c0 and then the bar closes at C: mark-out vs the fill price
        mk = pd.concat([mo_buy[buy_x & okb], mo_sell[sell_x & okb]]).mean()
        PASSIVE.loc[p] = [ft, fx, mk, (-mk) / (2 * HS[p] * 1e4) if mk < 0 else 0.0]
        PASSIVE_TOD.loc[p] = (buy_x & okb).groupby(bk).mean().reindex(BUCKETS).to_numpy()
    print(f"passive limit order at the last close (all products; touch = optimistic about the queue, through = conservative): "
          f"fill rate median {PASSIVE.fill_touch.median():.2f} (touch) / {PASSIVE.fill_through.median():.2f} (through) | mark-out after a through-fill, median {PASSIVE.markout_bp.median():.2f} bp "
          f"(negative = adverse selection) | adverse selection above the full spread in {(PASSIVE.adverse_over_spread > 1).sum()} of {PASSIVE.fill_through.notna().sum()} products")
else:
    print("passive-fill bounds skipped: needs 5-min high / low / close")

In [ ]:
# step 23a
# ---- 23a.4 impact (square-root law) and the cost function; limit-locked days ----
MONEYW = wide("money").reindex(index=RAW.index, columns=PRODS)
ADV = MONEYW.rolling(EXEC_ADV_WIN, min_periods=5).median().shift(1)
ADV = ADV.fillna(MONEYW.expanding(min_periods=5).median().shift(1)).fillna(MONEYW.loc[:SPL["train"][-1]].median()).replace(0, np.nan)   # warm-up: fixed training median, never a backfill
SIG = SD_D.reindex(index=RAW.index, columns=PRODS)
def make_cost_fn(aum, Y=None, hs=None):
    # cost of a day's trades for a book of `aum` CNY: fee + half-spread (linear) + Y * sigma * sqrt(participation) (impact), on every product traded; rolls pay fee + half-spread both ways
    Y = EXEC_Y if Y is None else Y; hs = HS if hs is None else hs
    lin = (FEE_BPS * 1e-4 + hs.reindex(PRODS)).to_numpy()[None, :]
    def fn(turn, Wd):
        part = (turn * aum / ADV).clip(lower=0)
        imp = (Y * SIG * turn * np.sqrt(part)).fillna(0.0)
        return (turn * lin + imp + 2 * Wd.abs() * ROLLN * lin).sum(axis=1)
    return fn

OPEN_, HIGH_, LOW_ = (wide(c).reindex(index=RAW.index, columns=PRODS) for c in ("open", "high", "low"))
LIM = pd.Series({p: LIMIT.get(p, 10) / 100 for p in PRODS})
rng_ = (HIGH_ - LOW_) / CLOSE.reindex(index=RAW.index, columns=PRODS)
LOCK = ((SIMPLE.abs() >= LOCK_FRAC * LIM) & (rng_ <= LOCK_RANGE * LIM)).reindex(index=RAW.index, columns=PRODS).fillna(False)
print(f"limit-locked product-days: {int(LOCK.sum().sum())} ({LOCK.values.mean()*100:.2f}% of all) | products with most locked days: " +
      ", ".join(f"{p} {int(n)}" for p, n in LOCK.sum().sort_values(ascending=False).head(5).items()))

# impact scale check: Amihud illiquidity should line up with the sqrt-law cost per unit traded (sigma / sqrt(ADV))
AMIHUD = (SIMPLE.abs() / MONEYW.replace(0, np.nan)).median() * 1e9
SQRTL = (SIG / np.sqrt(ADV)).median()
am_rho = pd.concat([AMIHUD, SQRTL], axis=1).dropna().corr(method="spearman").iloc[0, 1]
IMP1 = {p: EXEC_Y * SIG[p].median() * np.sqrt(0.01) * 1e4 for p in PRODS}                       # impact in bp when trading 1% of ADV
EXEC = pd.DataFrame({"tick_bp": TICK_BP, "cs_bp": CS_ALL * 1e4, "roll_bp": ROLL_ALL * 1e4, "half_spread_bp": HS * 1e4, "flat_slippage_bp": FLAT_SLIP * 1e4,
                     "adv_cny_m": ADV.iloc[-60:].median() / 1e6, "daily_vol_pct": SIG.median() * 100, "impact_bp_at_1pct_adv": pd.Series(IMP1),
                     "fill_touch": PASSIVE.fill_touch, "fill_through": PASSIVE.fill_through, "markout_bp": PASSIVE.markout_bp,
                     "locked_days_pct": LOCK.mean() * 100, "sector": SEC_MAP.reindex(PRODS)})
EXEC = EXEC.assign(model_vs_flat=(FEE_BPS + EXEC.half_spread_bp) / (FEE_BPS + EXEC.flat_slippage_bp)).sort_values("adv_cny_m", ascending=False)
EXEC.round(3).to_csv(OUT2 / "execution_model.csv")
print(f"Amihud illiquidity vs sqrt-law cost per unit traded: Spearman rho = {am_rho:.2f} across products (a sign that ADV and volatility are the right scale)")
print("\nMost / least liquid products (CNY m traded per day, half-spread bp vs flat 1-tick bp, impact bp at 1% of ADV):")
print(pd.concat([EXEC.head(6), EXEC.tail(6)])[["adv_cny_m", "half_spread_bp", "flat_slippage_bp", "impact_bp_at_1pct_adv", "fill_touch", "fill_through", "markout_bp", "locked_days_pct"]].round(2).to_string())

In [ ]:
# step 23a
fig, axs = plt.subplots(1, 3, figsize=(18, 5.4), gridspec_kw={"width_ratios": [1.1, 1, 1]})
ax = axs[0]
M = (HS_TOD * 1e4).dropna(how="all").reindex(EXEC.index[:40]).dropna(how="all")
heat(ax, M, cmap=SEQ, label="half-spread, bp", annot=False)
ax.set_title("Half-spread by time of day (liquid products first)"); ax.tick_params(axis="y", labelsize=6)
ax = axs[1]
e_ = EXEC.dropna(subset=["half_spread_bp"])
ax.scatter(e_.flat_slippage_bp, e_.half_spread_bp, s=14, color=SERIES[0]); lim_ = max(e_.flat_slippage_bp.max(), e_.half_spread_bp.max()) * 1.05
ax.plot([0, lim_], [0, lim_], color=INK2, lw=1, ls="--"); ax.set_xlabel("flat model slippage (1 tick, bp)"); ax.set_ylabel("estimated half-spread (bp)")
ax.set_title("Is one tick the right slippage? (above the line = flat model is too cheap)")
ax = axs[2]
if PASSIVE.fill_through.notna().any():
    ax.scatter(PASSIVE.fill_through, PASSIVE.markout_bp, s=14, color=SERIES[1]); ax.axhline(0, color=INK2, lw=1)
    ax.set_xlabel("fill rate, price traded through (conservative)"); ax.set_ylabel("mark-out to bar close, bp"); ax.set_title("Passive orders: fill rate vs adverse selection")
else:
    ax.text(0.5, 0.5, "needs 5-min high / low / close", ha="center", transform=ax.transAxes)
plt.tight_layout()
savefig(fig, "23a_execution_model", "Execution model: spread, passive fills, impact", "23")

## Step 23 — Strategy library: from opportunity ranks to positions

| Code | Strategy | Built from | Type |
|---|---|---|---|
| A1 / A2 | **MTF rule**: long BULL, short BEAR (rule / learned ranks) | Step 19 | individual |
| A3 | Composite rank, long top quintile / short bottom quintile | Step 19 | cross-section |
| A4 | **Sector rotation**: long the 2 best sectors by rank, short the 2 worst | Step 19 | sector |
| A5 | **Within-sector pairs**: long the best-ranked, short the worst-ranked member of each sector | Step 19 | synthetic spread |
| B | **Factor portfolios** for each family × level (XS quintiles, WS within-sector, SEC rotation, TS outright), plus the validated multi-factor | Step 18 | factor |
| C1 | Daily Granger links (betas refit yearly) | Steps 14, 17 | relationship |
| C2 | Cointegrated pairs and spreads (hedge ratio refit yearly) | Steps 15, 17 | relationship |
| C3 | **EVT event trades**: after a tail event, hold `EVT_HOLD` days in the train-estimated continuation or reversal direction for that sector | Step 13 | tail |
| C4 | Intraday Granger links at their own frequency (bar-level, with costs) | Step 14 | intraday |
| D | **Regime-switching multi-factor**: factor weights depend on the current (ex-ante) volatility regime | Step 20 | regime |
| C5 | **Fundamental chains**: trade the chain spread back to its fundamental value (hedge ratio refit yearly) | Step 16 | economic spread |
| F0–F5 | **Dynamics**: factor equal-weight benchmark, factor rotation, sector rotation, factor × sector rotation, factor timing, factor × sector mean reversion (configurations chosen on train) | Step 21 | factor / sector rotation |
| G1 / G2 | **Monitor-gated books**: factors and factor × sector cells traded only while the drift monitor says they work | Step 22 | decay control |
| E | **Meta-labelled rank**: the composite rank gives the side; a random forest (purged CV) predicts whether the triple barrier is hit in that direction; positions are sized by AFML bet sizing | Steps 18–20 | ML |

**Execution realism:**
* Costs per side are fee + `SLIP_TICKS` ticks, plus a round trip on contract rolls (the flat model used for selection). Step 23a estimates a spread and impact model; **Step 23b re-costs the whole library with it** and runs CPCV.
* Delay 0 and 1 bar; selection uses `MAIN_DELAY`.
* Ex-ante vol targeting, with leverage capped at min(`LEV_CAP`, 1/exchange margin).
* Profit and loss uses simple returns.

**Overfitting control:**
* Strategies are selected on validation only.
* The **Deflated Sharpe** hurdle accounts for the number of strategies tried.
* The **Probability of Backtest Overfitting** (PBO, CSCV) is reported.
* Performance is shown **per regime**, together with a **regime-switching book** (for each volatility regime, the strategies that worked in that regime on validation) against a static book and an HRP-weighted book.

**Used later.** The backtests (`BT`, `NETS`) feed the execution re-costing and CPCV (Step 23b), the portfolio (Step 24) and the board (Step 26).

In [ ]:
# step 23
# execution toolkit (PRODS, COST, CAP, to_weights, backtest, q_ls, ws_ls, sec_ls ...) is defined in Step 21.0
STRATS = {}
def add(name, family, W):
    STRATS[name] = dict(family=family, W=W)

# A: opportunity ranks
add("A1 MTF rule (rule ranks)", "A Rank: individual", to_weights(MTF_STATE_RULE.fillna(0)))
add("A2 MTF rule (learned ranks)", "A Rank: individual", to_weights(MTF_STATE.fillna(0)))
add("A3 Rank XS long-short", "A Rank: cross-section", to_weights(q_ls(COMP_NET)))
add("A4 Rank sector rotation", "A Rank: sector", to_weights(sec_ls(COMP_NET)))
add("A5 Rank within-sector pairs", "A Rank: within sector", to_weights(ws_ls(COMP_NET)))
# B: factor portfolios, orientation fixed on train (signals and weights are built once, in Step 21.1)
for (fam, lev), W_ in FW.items():
    add(f"B {fam} {lev}", f"B Factor: {lev}", W_)
add("B multi-factor (validated)", "B Factor: multi", to_weights(q_ls(FACTOR_SCORE)))
if COMBO_BEST_SCORE is not None:                                          # Step 18b: best group of 2+ families confirmed on validation
    add("B family combo (best on validation)", "B Factor: combo", to_weights(q_ls(COMBO_BEST_SCORE) if COMBO_BEST[0] == "XS" else ws_ls(COMBO_BEST_SCORE)))
print(f"A/B strategies built: {len(STRATS)}")

In [ ]:
# step 23
t0 = time.time()
# C1: daily Granger links, betas refit yearly
if "1D" in GRES:
    for mdl in GRES["1D"]["frozen"]["models"]:
        c, e = mdl["cause"], mdl["effect"]; S = pd.DataFrame(0.0, index=RAW.index, columns=PRODS); p_ = P_LAG_TF["1D"]
        for k, d0 in enumerate(REFITS):
            y, Xr, Xu = design(Z.loc[Z.index < d0, UNIV], c, e, p_, CONDITION_ON_MARKET); m = np.isfinite(y) & np.isfinite(Xu).all(axis=1)
            _, b = ols(Xu[m], y[m]); term = sum(b[-p_:][i] * Z[c].shift(i) for i in range(p_))
            msk = period_mask(RAW.index, k); S.loc[msk, e] = np.sign(term[msk]).fillna(0.0)
        add(f"C1 Granger {c}→{e}", "C Relationship: Granger", to_weights(S))
# C2 / C5: spread strategies (cointegrated pairs, fundamental chains): hedge ratio refit yearly on past data only
def spread_strategy(yy, xs):
    W = pd.DataFrame(0.0, index=RAW.index, columns=PRODS)
    for k, d0 in enumerate(REFITS):
        hist = LP.loc[LP.index < d0, [yy] + xs].dropna().tail(PAIR_LOOKBACK)
        if len(hist) < 120:
            continue
        _, b = ols(np.c_[np.ones(len(hist)), hist[xs].values], hist[yy].values)
        spread = LP[yy] - LP[xs] @ b[1:]
        zsp = (spread - spread.rolling(PAIR_Z_WIN).mean()) / spread.rolling(PAIR_Z_WIN).std(); dsp = spread.diff().rolling(PAIR_Z_WIN).std()
        pos = 0.0
        for t in np.flatnonzero(period_mask(RAW.index, k)):
            zt = zsp.iloc[t]
            if not np.isfinite(zt):
                pos = 0.0
            elif pos == 0 and abs(zt) > PAIR_Z_IN:
                pos = -np.sign(zt)
            elif pos != 0 and abs(zt) < PAIR_Z_OUT:
                pos = 0.0
            kk = TAU / dsp.iloc[t] if np.isfinite(dsp.iloc[t]) and dsp.iloc[t] > 0 else 0.0
            W.iat[t, PRODS.index(yy)] = pos * kk
            for xj, bj in zip(xs, b[1:]):
                W.iat[t, PRODS.index(xj)] = -pos * kk * bj
    return W.clip(lower=-CAP, upper=CAP, axis=1)
for x in COINT[COINT.valid].itertuples():
    add(f"C2 Pair {x.a}~{x.b}", "C Relationship: pairs", spread_strategy(x.a, x.b.split("+")))
if len(FUND):
    for x in FUND[FUND.valid].itertuples():
        add(f"C5 Fundamental {x.dep}~{'+'.join(x.drivers)}", "C Relationship: fundamental", spread_strategy(x.dep, list(x.drivers)))
# C3: EVT event trades
S_ev = pd.DataFrame(0.0, index=RAW.index, columns=PRODS)
for c in UNIV_V:
    sec = SEC_MAP[c]
    S_ev[c] = (E_D[c] == 1) * EVT_ORIENT.get((sec, 1), 0.0) - (E_D[c] == -1) * EVT_ORIENT.get((sec, -1), 0.0)
S_ev = S_ev.replace(0, np.nan).ffill(limit=EVT_HOLD - 1).fillna(0.0)
add("C3 EVT tail-event trades", "C Tail events", to_weights(S_ev))
# D: regime-switching multi-factor
if REG_W:
    XR = pd.DataFrame(np.nan, index=RAW.index, columns=PRODS)
    for st_, w in REG_W.items():
        comp = sum(FEATS[f].rank(axis=1, pct=True).sub(0.5) * wv for f, wv in w.items()) / sum(abs(v) for v in w.values())
        msk = (VOL_STATE == st_).to_numpy(); XR.loc[msk] = comp.loc[msk]
    add("D Regime-switching multi-factor", "D Regime", to_weights(q_ls(XR)))
print(f"C/D strategies built in {time.time()-t0:.0f}s")

In [ ]:
# step 23
# E: meta-labelled composite rank (López de Prado): primary side from the rank, secondary RF decides size
t0 = time.time()
side = np.sign(COMP_NET).where(COMP_NET.abs() >= 20).reindex(columns=PRODS)
agree = sum(np.sign(RD[tf][0] + RD[tf][1]).reindex(columns=PRODS).eq(side.reindex(columns=PRODS)).astype(float) for tf in RD) / len(RD)
bc = lambda s: pd.DataFrame(np.repeat(s.reindex(RAW.index).to_numpy()[:, None], len(PRODS), 1), index=RAW.index, columns=PRODS)
META_F = {"abs_net": COMP_NET.abs() / 100, "tf_agree": agree, "mvol": bc(MVOL), "p_turb": bc(P_HIGH), "breadth": bc(BREADTH),
          "factor_x_side": (FACTOR_SCORE.rank(axis=1, pct=True) - 0.5) * side, "vol_contract": B["vol_contract"].astype(float),
          "mom20_x_side": B["mom_20"].astype(float) * side}
mf = list(META_F)
ev_side = side.to_numpy()[EV.t0.values, EV["product"].map(colpos).values]
okm = np.isfinite(ev_side)
Xm = np.column_stack([META_F[k].reindex(columns=PRODS).to_numpy()[EV.t0.values, EV["product"].map(colpos).values] for k in mf])[okm]
Xm = np.nan_to_num(Xm); ym = ((EV.label.values[okm] * ev_side[okm]) > 0).astype(int); per = EV.period.values[okm]
trm = per == "train"
meta_ok = trm.sum() > 200 and len(set(ym[trm])) == 2
if meta_ok:
    folds_m = purged_kfold(EV.t0.values[okm][trm], EV.t1.values[okm][trm])
    aucs = []
    for a_, b_ in folds_m:
        if len(set(ym[trm][a_])) == 2 and len(set(ym[trm][b_])) == 2:
            mm = RandomForestClassifier(n_estimators=200, min_samples_leaf=50, max_features=2, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
            mm.fit(Xm[trm][a_], ym[trm][a_]); aucs.append(roc_auc_score(ym[trm][b_], mm.predict_proba(Xm[trm][b_])[:, 1]))
    META_RF = RandomForestClassifier(n_estimators=200, min_samples_leaf=50, max_features=2, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED).fit(Xm[trm], ym[trm])
    va_ = per == "val"
    print(f"meta-label: primary hit rate (train) {ym[trm].mean():.3f} | purged CV AUC {np.mean(aucs):.3f} | "
          f"val AUC {roc_auc_score(ym[va_], META_RF.predict_proba(Xm[va_])[:, 1]) if va_.sum() > 50 and len(set(ym[va_])) == 2 else np.nan:.3f}")
    live = (RAW.index >= WF_START)
    rows_i, cols_i = np.nonzero(np.isfinite(side.to_numpy()) & live[:, None])
    Xd = np.nan_to_num(np.column_stack([META_F[k].reindex(columns=PRODS).to_numpy()[rows_i, cols_i] for k in mf]))
    pr = META_RF.predict_proba(Xd)[:, 1]; size = np.where(pr > 0.5, bet_size(pr), 0.0)
    S_meta = np.zeros(side.shape); S_meta[rows_i, cols_i] = side.to_numpy()[rows_i, cols_i] * size
    add("E Meta-labelled rank", "E Meta-labelling", to_weights(pd.DataFrame(S_meta, index=RAW.index, columns=PRODS)))
print(f"meta-labelling in {time.time()-t0:.0f}s | total daily strategies: {len(STRATS)}")

for name, v in NEW_STRATS.items():          # F: factor / sector / factor-sector dynamics (Step 21);  G: monitor-gated books (Step 22)
    add(name, v["family"], v["W"])
print(f"+ {len(NEW_STRATS)} strategies from Steps 21-22 | total {len(STRATS)}")
BT = {name: {d: backtest(v["W"], d) for d in DELAYS} for name, v in STRATS.items()}

In [ ]:
# step 23
rows = []
for name, by_d in BT.items():
    for d, df in by_d.items():
        for sp in ("val", "test"):
            idx = df.index.intersection(SPL[sp])
            rows.append(dict(strategy=name, family=STRATS[name]["family"], delay=d, split=sp,
                             **perf(df.net.reindex(idx), df.gross.reindex(idx), df.turn.reindex(idx))))
BTS = pd.DataFrame(rows)
v1 = BTS[(BTS.split == "val") & (BTS.delay == MAIN_DELAY)].set_index("strategy")
t1 = BTS[(BTS.split == "test") & (BTS.delay == MAIN_DELAY)].set_index("strategy")
t0d = BTS[(BTS.split == "test") & (BTS.delay == 0)].set_index("strategy")
SELECTED = v1.index[(v1.sharpe > 0) & (v1.t > 1.0)].tolist()
N_TRIALS = len(v1); sr_star = dsr_threshold((v1.sharpe / np.sqrt(ANN)).values)
t1 = t1.assign(selected=t1.index.isin(SELECTED), sharpe_val=v1.sharpe, sharpe_test_gross_d0=t0d.sharpe_gross, sharpe_test_delay0=t0d.sharpe,
               dsr=[psr(r.sharpe / np.sqrt(ANN), r.n, r.skew, r.kurt + 3, sr_star) if np.isfinite(r.sharpe) else np.nan for r in t1.itertuples()])
OOS = RAW.index[RAW.index >= WF_START]
NETS = pd.DataFrame({k: BT[k][MAIN_DELAY].net.reindex(OOS) for k in BT})
PBO, PBO_LOGITS = pbo_cscv(NETS.fillna(0).values) if NETS.shape[1] > 2 else (np.nan, np.array([]))
# per-regime Sharpe (validation + test)
def sh(x):
    x = x.dropna(); return x.mean() / x.std() * np.sqrt(ANN) if len(x) > 20 and x.std() > 0 else np.nan
REGL = REG.shift(1)          # regime known at the start of the P&L day: Markov / breadth labels at t contain day t's return
REGP = {f"{var}:{st_}": NETS.loc[REGL[var].reindex(OOS) == st_].apply(sh) for var in ("vol", "ms", "breadth") for st_ in sorted(REGL[var].dropna().unique())}
REGP = pd.DataFrame(REGP)
# books: static (selected on val), HRP (val covariance), regime-switching (per vol state, strategies good in that state on val)
VALI, TESTI = RAW.index.intersection(SPL["val"]), RAW.index.intersection(SPL["test"])
BOOKS = {}
if SELECTED:
    BOOKS["static (val-selected, equal weight)"] = NETS[SELECTED].mean(axis=1)
    if len(SELECTED) > 2:
        w_h = hrp_weights(NETS.loc[VALI, SELECTED].cov().values + np.eye(len(SELECTED)) * 1e-10)
        BOOKS["HRP (val covariance)"] = NETS[SELECTED] @ w_h
pick = {}
for st_ in ("LOW", "MID", "HIGH"):
    m = REGL.vol.reindex(VALI) == st_
    s_ = NETS.loc[VALI][m.to_numpy()].apply(sh).dropna().sort_values(ascending=False)
    pick[st_] = s_[s_ > 0].index[:5].tolist()
if any(pick.values()):
    stt = REGL.vol.reindex(OOS)
    BOOKS["regime-switching (vol state)"] = pd.Series([NETS.loc[d, pick.get(s_, [])].mean() if pick.get(s_) else 0.0 for d, s_ in stt.items()], index=OOS)
BOOK_PERF = pd.DataFrame({k: perf(v.reindex(TESTI)) for k, v in BOOKS.items()}).T.reindex(columns=["n", "ann_ret", "ann_vol", "sharpe", "sortino", "max_dd", "calmar", "rachev", "hit", "t", "skew", "kurt", "psr0"])
cols = ["family", "sharpe_val", "sharpe", "sharpe_gross", "sharpe_test_delay0", "sharpe_test_gross_d0", "ann_ret", "max_dd", "sortino",
        "rachev", "hit", "turnover", "cost_drag", "t", "psr0", "dsr", "selected"]
BT_SUMMARY = t1[cols].rename(columns={"sharpe": "sharpe_test", "sharpe_gross": "sharpe_test_gross"}).sort_values("sharpe_val", ascending=False)
BT_SUMMARY.to_csv(OUT2 / "backtest_summary.csv"); REGP.to_csv(OUT2 / "strategy_sharpe_by_regime.csv")
print(f"{N_TRIALS} daily strategies | selected on validation: {len(SELECTED)} | DSR hurdle = {sr_star*np.sqrt(ANN):.2f} | "
      f"PBO (CSCV, S={PBO_S}) = {PBO:.2f}  (share of splits where the in-sample best is below the median out of sample)")
print(BT_SUMMARY.round(3).to_string())
print("\nBooks on TEST:\n", BOOK_PERF[["sharpe", "ann_ret", "max_dd", "rachev", "psr0"]].round(3).to_string() if len(BOOK_PERF) else "no book")
print("\nStrategies picked per volatility regime (validation):", pick)

In [ ]:
# step 23
# C4: intraday Granger links at their own frequency (bar-level backtest)
ib_rows, IB_CURVES = [], {}
for tf in [t for t in GRES if t not in ("1D", "1W")]:
    models = GRES[tf]["frozen"]["models"][:5]
    if not models:
        continue
    D = TFDATA[tf]; Zs = D["Z"]; F_ = tf_frames(tf); Rb = np.expm1(F_["ret"].reindex(index=Zs.index)); del F_
    bpy = Zs.notna().any(axis=1).groupby(to_trading_day(Zs.index)).sum().median() * ANN
    for mdl in models:
        c, e = mdl["cause"], mdl["effect"]; p_ = GRES[tf]["p"]; b = np.array(mdl["beta_arx"][-p_:])
        S = np.sign(sum(b[i] * Zs[c].shift(i) for i in range(p_))).fillna(0.0)
        cst = float(COST[e].median()) if e in COST else DEFAULT_COST_BPS * 1e-4
        for d in DELAYS:
            pos = S.shift(1 + d).fillna(0.0); gross = pos * Rb[e].fillna(0.0)
            net = gross - S.shift(d).diff().abs().fillna(0.0).shift(1).fillna(0.0) * cst
            net_t = gross - S.shift(d).diff().abs().fillna(0.0).shift(1).fillna(0.0) * pd.Series(tod_cost(e, Zs.index), index=Zs.index)          # fee + half-spread of the bar's time-of-day bucket (Step 23a)
            for sp in ("val", "test"):
                g_, n_, nt_ = gross.loc[D["SPL"][sp]], net.loc[D["SPL"][sp]], net_t.loc[D["SPL"][sp]]
                ib_rows.append(dict(tf=tf, link=f"{c}→{e}", delay=d, split=sp, sharpe_gross=g_.mean() / g_.std() * np.sqrt(bpy) if g_.std() > 0 else np.nan,
                                    sharpe_net=n_.mean() / n_.std() * np.sqrt(bpy) if n_.std() > 0 else np.nan,
                                    sharpe_net_tod=nt_.mean() / nt_.std() * np.sqrt(bpy) if nt_.std() > 0 else np.nan,
                                    gross_bp_per_bar=g_.mean() * 1e4, trades_per_day=S.diff().abs().loc[D["SPL"][sp]].mean() / 2 * bpy / ANN, cost_bp=cst * 1e4))
            if d == MAIN_DELAY:
                IB_CURVES[f"[{tf}] {c}→{e}"] = (net.loc[D["SPL"]["test"]].cumsum(), gross.loc[D["SPL"]["test"]].cumsum())
INTRA_BT = pd.DataFrame(ib_rows)
if len(INTRA_BT):
    INTRA_BT.to_csv(OUT2 / "intraday_granger_backtest.csv", index=False)
    print("Intraday Granger links traded at their own frequency (test; sharpe_net: fee + 1 tick per side; sharpe_net_tod: fee + estimated half-spread of the bar's time-of-day bucket, Step 23a):")
    print(INTRA_BT[INTRA_BT.split == "test"].round(3).to_string(index=False))

In [ ]:
# step 23
fig, ax = plt.subplots(figsize=(13, max(4, 0.2 * len(BT_SUMMARY))))
b_ = BT_SUMMARY.iloc[::-1]; yy = np.arange(len(b_))
ax.barh(yy + 0.2, b_.sharpe_val, height=0.38, color=MUTE, label="validation")
ax.barh(yy - 0.2, b_.sharpe_test, height=0.38, color=np.where(b_.selected, SERIES[0], "#86b6ef"), label="test (dark = selected on validation)")
ax.set_yticks(yy); ax.set_yticklabels(b_.index, fontsize=6); ax.axvline(0, color=INK2, lw=1)
ax.axvline(sr_star * np.sqrt(ANN), color=SERIES[1], lw=1, ls="--", label="DSR hurdle")
ax.set_xlabel(f"net Sharpe, delay {MAIN_DELAY}"); ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.set_title(f"Strategy library: validation vs test (PBO = {PBO:.2f})")
savefig(fig, "23_strategy_sharpe", "Strategy library Sharpe ratios", "23")

fig, axs = plt.subplots(1, 3, figsize=(18, 5), gridspec_kw={"width_ratios": [1.3, 1.1, 0.8]})
ax = axs[0]
for k, (nm, v) in enumerate(BOOKS.items()):
    ax.plot(OOS, (1 + v.reindex(OOS).fillna(0)).cumprod(), color=SERIES[k % 8], lw=1.6, label=nm)
for k, nm in enumerate(BT_SUMMARY.head(4).index):
    ax.plot(OOS, (1 + NETS[nm].fillna(0)).cumprod(), color=SERIES[(k + 3) % 8], lw=0.9, alpha=0.7, label=nm)
ax.axvline(SPL["test"][0], color=INK2, ls="--", lw=1); ax.set_yscale("log"); ax.legend(frameon=False, fontsize=6)
ax.set_title("Books and top strategies (walk-forward, net); dashed = test start")
ax = axs[1]
top_ = BT_SUMMARY.head(20).index
heat(ax, REGP.loc[top_].clip(-4, 4), cmap=DIV, center=0, vmax=4, label="Sharpe in regime (val+test)")
ax.set_title("Strategy performance by regime")
ax = axs[2]
if len(PBO_LOGITS):
    ax.hist(PBO_LOGITS, bins=30, color=SERIES[0], edgecolor="#fcfcfb"); ax.axvline(0, color=SERIES[1], lw=1.5)
ax.set_xlabel("logit of out-of-sample rank of the in-sample best"); ax.set_title(f"PBO = {PBO:.2f} (mass left of 0)")
plt.tight_layout()
savefig(fig, "23_books_regimes_pbo", "Books, regime dependence, PBO", "23")
if IB_CURVES:
    fig, ax = plt.subplots(figsize=(12, 3.8))
    for k, (nm, (n_, g_)) in enumerate(list(IB_CURVES.items())[:6]):
        ax.plot(n_.index, n_.values * 100, color=SERIES[k % 8], lw=1.3, label=f"{nm} net")
        ax.plot(g_.index, g_.values * 100, color=SERIES[k % 8], lw=0.8, ls="--")
    ax.axhline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=7); ax.set_ylabel("cumulative return (%), dashed = gross")
    ax.set_title("Intraday Granger links on test: gross vs net of costs")
    savefig(fig, "23_intraday_granger", "Intraday Granger strategies", "23")

## Step 23b — Execution-adjusted library, and CPCV: a distribution of Sharpe instead of one number

**1. Re-cost the library.** Every strategy is back-tested again with the Step 23a model (half-spread by product, square-root impact at book sizes `EXEC_AUM`, limit-locked days not tradable). Two questions follow: *which strategies selected under flat costs fail under realistic costs?* and *at what AUM does each edge disappear?*

**2. Combinatorial purged cross-validation (López de Prado, AFML ch. 12).** The library has one out-of-sample test number per strategy, which depends on one particular split. CPCV cuts the walk-forward return history into `CPCV_N` groups, uses every combination of `CPCV_K` groups as the test set and the others for **selection**, with `CPCV_PURGE` days purged on each side of every test group. The selection rule is then evaluated on the test groups only. Because each group is a test group in $\binom{N-1}{K-1}$ different splits, the test pieces can be re-assembled into that many **different full-history paths**, each a complete out-of-sample track record of the *selection procedure*. The spread of the paths' Sharpe ratios is the honest uncertainty of "pick strategies on past performance".

Library members are already walk-forward with parameters frozen before the period (Step 23), so what CPCV re-samples is the **choice among strategies**. A path-level Sharpe is therefore a statement about the procedure, not about a single strategy.

Rules compared: `top1` (best train Sharpe), `topM` (equal weight of the `CPCV_TOPM` best), `all` (equal weight of everything: no selection, the baseline the rules must beat). Run on flat-cost and on execution-model returns. (`all` involves no selection, so all its paths are the same series: its spread is zero by construction; it is the baseline.)

**Used later.** Strategies that survive the execution model and the CPCV check enter the portfolio of Step 24, and the break-even book size caps the capacity quoted on the board.

In [ ]:
# step 23b
# ---- 23b.1 the library under the execution model ----
NETS_X, LOCKONLY = {}, {}
for aum in EXEC_AUM:
    fn = make_cost_fn(aum)
    NETS_X[aum] = pd.DataFrame({k: backtest(v["W"], MAIN_DELAY, cost_fn=fn, lock=LOCK).net.reindex(OOS) for k, v in STRATS.items()})
LOCKONLY = pd.DataFrame({k: backtest(v["W"], MAIN_DELAY, lock=LOCK).net.reindex(OOS) for k, v in STRATS.items()})

def sh_split(R, sp):
    x = R.loc[R.index.intersection(SPL[sp])]
    return x.apply(lambda c: c.mean() / c.std() * np.sqrt(ANN) if c.std() > 0 else np.nan)
ex_rows = []
for k in STRATS:
    r = dict(strategy=k, family=STRATS[k]["family"], flat_val=sh_split(NETS[[k]], "val").iloc[0], flat_test=sh_split(NETS[[k]], "test").iloc[0],
             locked_test=sh_split(LOCKONLY[[k]], "test").iloc[0])
    for aum in EXEC_AUM:
        r[f"val_{aum/1e6:.0f}m"] = sh_split(NETS_X[aum][[k]], "val").iloc[0]; r[f"test_{aum/1e6:.0f}m"] = sh_split(NETS_X[aum][[k]], "test").iloc[0]
    ex_rows.append(r)
EXL = pd.DataFrame(ex_rows).set_index("strategy")
# break-even AUM: where the test Sharpe (execution model) crosses zero, interpolated in log AUM
def breakeven(row):
    xs = np.log(EXEC_AUM); ys = np.array([row[f"test_{a/1e6:.0f}m"] for a in EXEC_AUM], float)
    if not np.isfinite(ys).all():
        return np.nan
    if ys[0] <= 0:
        return 0.0
    if ys[-1] > 0:
        return np.inf
    j = np.flatnonzero(ys > 0)[-1]; return float(np.exp(xs[j] + (0 - ys[j]) * (xs[j + 1] - xs[j]) / (ys[j + 1] - ys[j])))
EXL["breakeven_aum_cny_m"] = EXL.apply(breakeven, axis=1) / 1e6
mm = f"{EXEC_AUM_MAIN/1e6:.0f}m"
EXL["selected_flat"] = EXL.index.isin(SELECTED)
# selection redone on the execution-model validation Sharpe: who drops out?
vx = NETS_X[EXEC_AUM_MAIN].loc[NETS_X[EXEC_AUM_MAIN].index.intersection(SPL["val"])]
t_val = vx.mean() / vx.std() * np.sqrt(vx.notna().sum())                 # same screen as Step 23: mean > 0 and t > 1 on validation
EXL["selected_exec"] = EXL.index.isin(t_val.index[(vx.mean() > 0) & (t_val > 1.0)])
EXL.round(3).to_csv(OUT2 / "strategies_execution_adjusted.csv")
drop = EXL[EXL.selected_flat & ~EXL.selected_exec].index.tolist(); add_ = EXL[~EXL.selected_flat & EXL.selected_exec].index.tolist()
print(f"Execution-adjusted library (book size {EXEC_AUM_MAIN/1e6:.0f}m CNY; limit-locked days not tradable)")
print(f"selected on validation under flat costs: {int(EXL.selected_flat.sum())} | under the execution model: {int(EXL.selected_exec.sum())} | dropped: {len(drop)} {drop[:6]} | added: {len(add_)}")
show = ["family", "flat_test", "locked_test"] + [f"test_{a/1e6:.0f}m" for a in EXEC_AUM] + ["breakeven_aum_cny_m", "selected_flat", "selected_exec"]
print(EXL[EXL.selected_flat | EXL.selected_exec].sort_values("flat_test", ascending=False)[show].round(2).to_string())
print(f"\nmedian test Sharpe across the library: flat {EXL.flat_test.median():.2f} | locked days only {EXL.locked_test.median():.2f} | " +
      " | ".join(f"{a/1e6:.0f}m {EXL[f'test_{a/1e6:.0f}m'].median():.2f}" for a in EXEC_AUM))

In [ ]:
# step 23b
# ---- 23b.2 CPCV of the selection procedure ----
def cpcv(M, N=CPCV_N, k=CPCV_K, purge=CPCV_PURGE, m_top=CPCV_TOPM):
    X = M.fillna(0.0).to_numpy(); T, S = X.shape; groups = np.array_split(np.arange(T), N)
    combos = list(itertools.combinations(range(N), k)); seg = {r: {g: [] for g in range(N)} for r in ("top1", "topM", "all")}
    srows = []
    for sid, cb in enumerate(combos):
        test = np.concatenate([groups[g] for g in cb]); banned = np.zeros(T, bool); banned[test] = True
        for g in cb:
            a, b = groups[g][0], groups[g][-1]; banned[max(0, a - purge):a] = True; banned[b + 1:b + 1 + purge] = True
        tr = np.flatnonzero(~banned)
        mu, sd = X[tr].mean(0), X[tr].std(0); act = (X[tr] != 0).sum(0) >= 30
        sh = np.where((sd > 1e-12) & act, mu / np.where(sd > 1e-12, sd, 1) * np.sqrt(ANN), -np.inf)
        order = np.argsort(-sh); good = order[np.isfinite(sh[order]) & (sh[order] > 0)]
        picks = {"top1": good[:1], "topM": good[:m_top], "all": np.flatnonzero(act)}
        for r, pk in picks.items():
            book = X[:, pk].mean(1) if len(pk) else np.zeros(T)
            for g in cb:
                seg[r][g].append(book[groups[g]])
        te_sh = np.where(X[test].std(0) > 1e-12, X[test].mean(0) / np.where(X[test].std(0) > 1e-12, X[test].std(0), 1) * np.sqrt(ANN), np.nan)
        if len(good):
            b1 = good[0]; rk = (np.nansum(te_sh < te_sh[b1]) / max(np.isfinite(te_sh).sum() - 1, 1))
            srows.append(dict(split=sid, groups=cb, train_sharpe_pick=sh[b1], test_sharpe_pick=te_sh[b1], test_rank_pct=rk))
    phi = len(seg["top1"][0]); paths = {}
    for r in seg:
        P_ = []
        for j in range(phi):
            x = np.concatenate([seg[r][g][j] for g in range(N)]); P_.append(x.mean() / x.std() * np.sqrt(ANN) if x.std() > 0 else np.nan)
        paths[r] = np.array(P_)
    return paths, pd.DataFrame(srows), len(combos), phi

CPCV_RES, CPCV_SPLITS, CPCV_ROWS = {}, {}, []
for label, M in (("flat costs", NETS), (f"execution model {EXEC_AUM_MAIN/1e6:.0f}m", NETS_X[EXEC_AUM_MAIN])):
    paths, spl_, ncomb, phi = cpcv(M); CPCV_RES[label] = paths; CPCV_SPLITS[label] = spl_
    pbo_c = float((spl_.test_rank_pct < 0.5).mean()) if len(spl_) else np.nan
    for r, p_ in paths.items():
        CPCV_ROWS.append(dict(returns=label, rule=r, paths=phi, mean_sharpe=np.nanmean(p_), sd=np.nanstd(p_), p5=np.nanpercentile(p_, 5), median=np.nanmedian(p_),
                              p95=np.nanpercentile(p_, 95), share_negative=float(np.nanmean(p_ < 0)), pbo_top1=pbo_c if r == "top1" else np.nan))
CPCV_SUM = pd.DataFrame(CPCV_ROWS)
CPCV_SUM.round(3).to_csv(OUT2 / "cpcv_summary.csv", index=False)
print(f"CPCV: {CPCV_N} groups of ~{len(NETS)//CPCV_N} days, {CPCV_K} test groups per split → {ncomb} splits and {phi} out-of-sample paths; {CPCV_PURGE} days purged each side of every test group")
print(CPCV_SUM.round(2).to_string(index=False))
one = {"top1": "best single strategy", "topM": f"top {CPCV_TOPM} equal weight", "all": "no selection"}
b_ = CPCV_SUM[CPCV_SUM.returns == "flat costs"].set_index("rule")
print(f"\nSelecting on past Sharpe (flat costs): mean path Sharpe top-{CPCV_TOPM} {b_.loc['topM','mean_sharpe']:.2f} vs no selection {b_.loc['all','mean_sharpe']:.2f}; "
      f"{b_.loc['topM','share_negative']*100:.0f}% of top-{CPCV_TOPM} paths lose money. Single-split test Sharpe of the same idea: see Step 23 (one number).")

In [ ]:
# step 23b
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
ax = axs[0]
labs = list(CPCV_RES); pos = 0; ticks, tl = [], []
for li, lab in enumerate(labs):
    for ri, r in enumerate(("top1", "topM", "all")):
        v = CPCV_RES[lab][r]; v = v[np.isfinite(v)]
        if len(v):
            bp_ = ax.boxplot([v], positions=[pos], widths=0.6, patch_artist=True, showfliers=True)
            for b in bp_["boxes"]:
                b.set_facecolor([SERIES[0], SERIES[1], MUTE][ri]); b.set_alpha(0.85)
        ticks.append(pos); tl.append(f"{r}\n{lab[:14]}"); pos += 1
    pos += 0.8
ax.set_xticks(ticks); ax.set_xticklabels(tl, fontsize=6); ax.axhline(0, color=INK2, lw=1); ax.set_ylabel("Sharpe of one out-of-sample path")
ax.set_title(f"CPCV: {phi} paths per rule")
ax = axs[1]
s_ = CPCV_SPLITS[labs[0]]
if len(s_):
    ax.scatter(s_.train_sharpe_pick, s_.test_sharpe_pick, s=18, color=SERIES[0]); ax.axhline(0, color=INK2, lw=1)
    lim_ = max(s_.train_sharpe_pick.max(), 1); ax.plot([0, lim_], [0, lim_], color=INK2, ls="--", lw=1)
    ax.set_xlabel("train Sharpe of the pick"); ax.set_ylabel("test Sharpe of the pick"); ax.set_title(f"Selection decay across {len(s_)} splits (below the line = overfit)")
ax = axs[2]
for k, aum in enumerate(EXEC_AUM):
    v_ = np.sort(EXL[f"test_{aum/1e6:.0f}m"].dropna().values)[::-1]; ax.plot(np.arange(len(v_)), v_, color=SERIES[k], lw=1.3, label=f"{aum/1e6:.0f}m CNY")
v_ = np.sort(EXL.flat_test.dropna().values)[::-1]; ax.plot(np.arange(len(v_)), v_, color=INK2, ls="--", lw=1, label="flat cost"); ax.axhline(0, color=INK2, lw=1)
ax.legend(frameon=False, fontsize=7); ax.set_xlabel("strategies, sorted"); ax.set_ylabel("test Sharpe"); ax.set_title("Library test Sharpe by book size (execution model)")
plt.tight_layout()
savefig(fig, "23b_cpcv_execution", "CPCV paths and execution-adjusted library", "23")

## Step 23c — Overnight strategy: enter before the close, exit after the opening move

**Idea.** Hold a position only through the night and the *initial move* of the next day: enter shortly **before** the day-session close, exit `k` minutes **after** the open. The edge, if any, is the overnight return (Step 13d) harvested before the opening noise and before the intraday part (which the study found weaker). Two ways to choose the side:

* **OV1 — per-product bias.** The side of each product (long, short or flat) is fixed on **train** by the sign of its mean overnight-to-`k` return when $|t|>$ `OVN_T_MIN`.
* **OV3 — pent-up pressure (cross-section).** Each evening the products are ranked on their intraday move up to the entry time; the top and bottom quintiles are traded in the direction that had the better rank IC on **train** (continuation or reversal).
* **OV2 — long everything overnight** (benchmark, not selectable).

**Grid.** Entry `m` ∈ {0, 5, 15, 30} minutes before the close × exit `k` ∈ {1, 5, 15, 30, 60} minutes after the open. Return of a variant: $\;r^{\text{pre}\,m}_{t-1}+r^{\text{night}}_t+r^{\text{open}\,k}_t$ — **all clean 1-minute returns, no roll inside**. The variant is chosen on **validation**; test is touched once. The number of variants tried is used in the deflated Sharpe ratio.

**Realism.** Sizing: $\text{TARGET\_VOL}$ over the overnight-return volatility of the last `OVN_VOL_WIN` days (known at the entry), capped at `LEV_CAP`. Cost: **both legs every day**: fee + the half-spread of the *close* bucket at entry and of the *open* bucket at exit (Step 23a estimates), times `OVN_COST_MULT`. If the product is limit-locked at the open (|night + first bar| ≥ `LOCK_FRAC` × limit) the exit is impossible and the position is held to the close of that day. A position held overnight is closed the next day, not the same day, so the exchange's "close-today" fee does not apply; the flat fee is used.

**Used later.** The chosen variants are added to the strategy library (family `H Overnight`) and enter the portfolio of Step 24 if they pass the same validation screen as every other strategy. Step 26c reports their cost break-even and the sector that carries them. Not included in the CPCV of Step 23b (it runs before this step, and the overnight return is not a close-to-close return).

In [ ]:
# step 23c
# ---- 23c.1 grid of overnight variants ----
if OVN_OK:
    idx_tr, idx_va, idx_te = IDX["train"], IDX["val"], IDX["test"]
    LIMv = pd.Series({p: LIMIT.get(p, 10) / 100 for p in PRODS})
    cs_entry = pd.Series({p: FEE_BPS * 1e-4 + (HS_TOD.loc[p, "close"] if p in HS_TOD.index and np.isfinite(HS_TOD.loc[p, "close"]) else HS.get(p, np.nanmedian(HS.values))) for p in PRODS})
    cs_exit = pd.Series({p: FEE_BPS * 1e-4 + (HS_TOD.loc[p, "open"] if p in HS_TOD.index and np.isfinite(HS_TOD.loc[p, "open"]) else HS.get(p, np.nanmedian(HS.values))) for p in PRODS})
    CSIDE = (cs_entry + cs_exit) * OVN_COST_MULT
    def ovn_ret(m, k):
        pre = OV[f"ret_pre{m}"].shift(1) if m else 0.0
        night = OV["ret_night"].fillna(0)
        R = pre + night + OV[f"ret_open{k}"]
        lock = (night + OV["ret_open1"]).abs() >= LOCK_FRAC * LIMv
        R = R.where(~lock, pre + night + OV["ret_day"])
        return R.where(RAW.notna() & OV[f"ret_open{k}"].notna()).reindex(columns=PRODS)
    def pos_from_side(S, R):
        sg = R.rolling(OVN_VOL_WIN, min_periods=40).std()
        act = ((S != 0) & sg.notna()).sum(axis=1).replace(0, np.nan)
        return (S * (TAU / sg)).div(np.sqrt(act), axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(lower=-CAP, upper=CAP, axis=1)
    def ovn_pnl(pos, R):
        p1 = pos.shift(1).fillna(0.0); gross = (p1 * np.expm1(R).fillna(0.0)).sum(axis=1); cost = (p1.abs() * CSIDE).sum(axis=1)
        return pd.DataFrame({"gross": gross, "net": gross - cost, "turn": 2 * p1.abs().sum(axis=1)})
    def ovn_variant(kind, m, k):
        R = ovn_ret(m, k)
        if kind == "OV2":
            S = pd.DataFrame(1.0, index=R.index, columns=PRODS)
        elif kind == "OV1":
            tr_ = R.loc[idx_tr]; t_ = tr_.mean() / (tr_.std() / np.sqrt(tr_.count())); side = np.sign(tr_.mean()).where(t_.abs() > OVN_T_MIN, 0.0).fillna(0.0)
            S = pd.DataFrame(np.tile(side.to_numpy(), (len(R), 1)), index=R.index, columns=PRODS)
        else:
            sx = ((OV["ret_day_ex_open"] - (OV[f"ret_pre{m}"] if m else 0.0)) / SIGd).reindex(columns=PRODS)        # intraday move to the entry time, vol units
            rk = sx.rank(axis=1, pct=True) - 0.5; ic = xs_ic_fast(sx.loc[idx_tr], R.shift(-1).loc[idx_tr], min_n=5)
            dsg = 1.0 if ic.mean() >= 0 else -1.0
            S = dsg * (rk.where(rk >= 0.3, 0.0).gt(0).astype(float) - rk.where(rk <= -0.3, 0.0).lt(0).astype(float)).where(sx.notna(), 0.0)
        pos = pos_from_side(S, R); out = ovn_pnl(pos, R); return out, pos, R
    GRID = []; OVN_RES = {}; t0 = time.time()
    for kind in ("OV1", "OV2", "OV3"):
        for m in OVN_M:
            for k in OVN_K:
                out, pos, R = ovn_variant(kind, m, k); OVN_RES[(kind, m, k)] = (out, pos)
                r = dict(kind=kind, m=m, k=k)
                for sp, ix in IDX.items():
                    pf = perf(out["net"].reindex(ix), out["gross"].reindex(ix)); r[f"sharpe_{sp}"] = pf.get("sharpe"); r[f"sharpe_gross_{sp}"] = pf.get("sharpe_gross"); r[f"t_{sp}"] = pf.get("t")
                GRID.append(r)
    OVN_GRID = pd.DataFrame(GRID); OVN_GRID.round(4).to_csv(OUT2 / "overnight_strategy_grid.csv", index=False)
    print(f"{len(OVN_GRID)} overnight variants in {time.time()-t0:.0f}s")
    print("\nValidation / test net Sharpe by entry (rows m = minutes before the close) and exit (columns k = minutes after the open):")
    for kind in ("OV1", "OV3", "OV2"):
        g_ = OVN_GRID[OVN_GRID.kind == kind]
        print(f"\n{kind}  validation:\n", g_.pivot(index="m", columns="k", values="sharpe_val").round(2).to_string(), f"\n{kind}  test:\n", g_.pivot(index="m", columns="k", values="sharpe_test").round(2).to_string())

In [ ]:
# step 23c
# ---- 23c.2 selection on validation, one look at test, honest accounting ----
if OVN_OK:
    SEL = OVN_GRID[OVN_GRID.kind.isin(["OV1", "OV3"]) & (OVN_GRID.sharpe_train > 0) & (OVN_GRID.sharpe_val > 0)]
    N_TR_OVN = int(OVN_GRID.kind.isin(["OV1", "OV3"]).sum())
    sr_star_o = dsr_threshold((OVN_GRID[OVN_GRID.kind.isin(["OV1", "OV3"])].sharpe_val.dropna() / np.sqrt(ANN)).values)
    OVN_PICK = {}
    for kind in ("OV1", "OV3"):
        g_ = SEL[SEL.kind == kind]
        if len(g_): OVN_PICK[kind] = g_.sort_values("sharpe_val", ascending=False).iloc[0]
    rows = []
    for kind, r in OVN_PICK.items():
        out, pos = OVN_RES[(kind, int(r.m), int(r.k))]; nt = out["net"].reindex(idx_te).dropna()
        p_ = perf(nt, out["gross"].reindex(idx_te), out["turn"].reindex(idx_te)); trades = pos.shift(1).abs().reindex(idx_te); nt_ = float((trades > 0).sum().sum())
        gross_bp = float(out["gross"].reindex(idx_te).sum() / max(trades.sum().sum(), 1e-9) * 1e4); cost_bp = float(CSIDE.mean() * 1e4)
        rows.append(dict(strategy=kind, entry_min_before_close=int(r.m), exit_min_after_open=int(r.k), sharpe_train=r.sharpe_train, sharpe_val=r.sharpe_val, sharpe_test=r.sharpe_test, sharpe_gross_test=r.sharpe_gross_test, t_test=r.t_test,
                         psr0_test=p_.get("psr0"), dsr_test=psr(r.sharpe_test / np.sqrt(ANN), p_.get("n", 0), p_.get("skew", 0), p_.get("kurt", 0) + 3, sr_star_o) if np.isfinite(r.sharpe_test) else np.nan,
                         ann_ret_test=p_.get("ann_ret"), ann_vol_test=p_.get("ann_vol"), max_dd_test=p_.get("max_dd"), hit_test=p_.get("hit"), gross_bp_per_position_day=gross_bp, round_trip_cost_bp=cost_bp,
                         break_even_cost_x=gross_bp / cost_bp if cost_bp > 0 else np.nan))
    OVN_SEL = pd.DataFrame(rows); OVN_SEL.round(4).to_csv(OUT2 / "overnight_strategy_selected.csv", index=False)
    if len(OVN_SEL):
        print(f"Selected on validation (train Sharpe > 0 and validation Sharpe > 0), {N_TR_OVN} variants tried (deflated Sharpe uses them):")
        print(OVN_SEL.round(3).T.to_string())
    else:
        print("no overnight variant has a positive Sharpe on train AND validation: nothing is added to the library (the grid above is the evidence)")
    # overfitting check: do validation winners stay winners?
    G_ = OVN_GRID[OVN_GRID.kind.isin(["OV1", "OV3"])].dropna(subset=["sharpe_val", "sharpe_test"])
    OVN_RANKCORR = float(stats.spearmanr(G_.sharpe_val, G_.sharpe_test)[0]) if len(G_) > 5 else np.nan
    print(f"\nrank correlation of validation and test Sharpe across all {len(G_)} variants: {OVN_RANKCORR:.2f} (near 0 = the choice on validation carries no information about test)")

In [ ]:
# step 23c
# ---- 23c.3 where the edge lives, cost stress, library registration ----
if OVN_OK and len(OVN_SEL):
    OVN_BY_SEC, OVN_STRESS = {}, []
    for kind, r in OVN_PICK.items():
        m, k = int(r.m), int(r.k); out, pos = OVN_RES[(kind, m, k)]; R = ovn_ret(m, k)
        contrib = (pos.shift(1).fillna(0.0) * np.expm1(R).fillna(0.0) - pos.shift(1).fillna(0.0).abs() * CSIDE).loc[idx_te]; OVN_BY_SEC[kind] = contrib.T.groupby(SECN.reindex(PRODS)).sum().T.sum() / contrib.shape[0] * ANN
        for cm in COST_STRESS:
            net_c = out["gross"] - (out["gross"] - out["net"]) * cm
            OVN_STRESS.append(dict(strategy=kind, cost_multiple=cm, sharpe_val=perf(net_c.reindex(idx_va)).get("sharpe"), sharpe_test=perf(net_c.reindex(idx_te)).get("sharpe")))
    OVN_STRESS = pd.DataFrame(OVN_STRESS); OVN_STRESS.to_csv(OUT2 / "overnight_cost_stress.csv", index=False)
    print("Cost stress (net Sharpe):\n", OVN_STRESS.pivot_table(index="cost_multiple", columns="strategy", values=["sharpe_val", "sharpe_test"]).round(2).to_string())
    print("\nAnnualised net P&L by sector on test (share of capital at the vol target):\n", pd.DataFrame(OVN_BY_SEC).round(4).to_string())
    # exit-time profile: where in the opening move is the return earned (mean overnight-to-k return, bp, by period)
    prof = {}
    for sp, ix in IDX.items():
        prof[sp] = pd.Series({k: float(np.nanmean(ovn_ret(int(OVN_SEL.entry_min_before_close.iloc[0]), k).loc[ix].mean()) * 1e4) for k in OVN_K})
    OVN_PROFILE = pd.DataFrame(prof); print("\nmean return from the entry to k minutes after the open, bp (all products, unsigned):\n", OVN_PROFILE.round(2).T.to_string())
    # register in the library
    OVN_LIB = {}
    for kind, r in OVN_PICK.items():
        nm = f"H{1 if kind == 'OV1' else 2} Overnight {'per-product bias' if kind == 'OV1' else 'pent-up pressure'} (-{int(r.m)}m → +{int(r.k)}m)"
        out, pos = OVN_RES[(kind, int(r.m), int(r.k))]
        STRATS[nm] = dict(family="H Overnight", W=pos.reindex(RAW.index).fillna(0.0))
        BT[nm] = {d: out.reindex(RAW.index).fillna(0.0) for d in DELAYS}
        NETS[nm] = out["net"].reindex(OOS)
        v_ = perf(out["net"].reindex(idx_va))
        if v_.get("sharpe", 0) > 0 and v_.get("t", 0) > 1.0 and nm not in SELECTED: SELECTED.append(nm)
        OVN_LIB[nm] = dict(val_sharpe=v_.get("sharpe"), selected=nm in SELECTED)
    print("\nadded to the strategy library:", OVN_LIB)
    OVN_STRAT_SUMMARY = dict(selected=OVN_SEL.round(3).to_dict("records"), rank_corr=OVN_RANKCORR, library=OVN_LIB)
else:
    OVN_BY_SEC, OVN_STRESS = {}, pd.DataFrame(columns=["strategy", "cost_multiple", "sharpe_val", "sharpe_test"])
    OVN_STRAT_SUMMARY = dict(selected=[], rank_corr=globals().get("OVN_RANKCORR"), library={})

In [ ]:
# step 23c
if OVN_OK:
    fig, axs = plt.subplots(2, 3, figsize=(18, 9))
    for j, kind in enumerate(("OV1", "OV3")):
        ax = axs[0, j]; g_ = OVN_GRID[OVN_GRID.kind == kind]; heat(ax, g_.pivot(index="m", columns="k", values="sharpe_val"), cmap=DIV, center=0, label="validation net Sharpe", annot=True, fmt="{:.2f}")
        ax.set_xlabel("exit: minutes after the open"); ax.set_ylabel("entry: minutes before the close"); ax.set_title(f"{kind}: validation Sharpe (rows m, columns k)")
    ax = axs[0, 2]; G2 = OVN_GRID[OVN_GRID.kind.isin(["OV1", "OV3"])]; ax.scatter(G2.sharpe_val, G2.sharpe_test, s=22, color=np.where(G2.kind == "OV1", SERIES[0], SERIES[1]), alpha=0.8)
    ax.axhline(0, color=INK2, lw=1); ax.axvline(0, color=INK2, lw=1); ax.set_xlabel("validation Sharpe"); ax.set_ylabel("test Sharpe"); ax.set_title(f"Do validation winners hold? rank corr {OVN_RANKCORR:.2f} (blue OV1, orange OV3)")
    ax = axs[1, 0]
    for k_, (kind, r) in enumerate(OVN_PICK.items()):
        out, pos = OVN_RES[(kind, int(r.m), int(r.k))]; ax.plot(OOS, (1 + out["net"].reindex(OOS).fillna(0)).cumprod(), color=SERIES[k_], lw=1.4, label=f"{kind} (-{int(r.m)}m → +{int(r.k)}m)")
    bench = OVN_RES[("OV2", 0, 30)][0]["net"].reindex(OOS).fillna(0); ax.plot(OOS, (1 + bench).cumprod(), color=MUTE, lw=1, label="OV2 long everything (benchmark)")
    ax.axvline(SPL["test"][0], color=INK2, ls="--", lw=1); ax.set_yscale("log"); ax.legend(frameon=False, fontsize=8); ax.set_title("Overnight strategies, net, validation then test")
    ax = axs[1, 1]
    if OVN_BY_SEC:
        d_ = pd.DataFrame(OVN_BY_SEC); d_.plot.barh(ax=ax, color=[SERIES[0], SERIES[1]][:d_.shape[1]], width=0.8); ax.axvline(0, color=INK2, lw=1); ax.set_xlabel("annualised net P&L, test"); ax.set_title("Which sectors carry the overnight edge?"); ax.legend(frameon=False, fontsize=8)
    ax = axs[1, 2]
    for k_, (kind, g__) in enumerate(OVN_STRESS.groupby("strategy") if len(OVN_STRESS) else []): ax.plot(g__.cost_multiple, g__.sharpe_test, marker="o", color=SERIES[k_], label=f"{kind} test"); ax.plot(g__.cost_multiple, g__.sharpe_val, marker="o", ls="--", color=SERIES[k_], alpha=0.5, label=f"{kind} validation")
    ax.axhline(0, color=INK2, lw=1); ax.set_xlabel("cost multiple (1 = fee + estimated half-spreads, both legs)"); ax.set_ylabel("net Sharpe"); ax.legend(frameon=False, fontsize=7); ax.set_title("Cost stress: both legs, every day")
    plt.tight_layout()
    savefig(fig, "23c_overnight_strategy", "Overnight strategy: grid, selection, sectors, cost", "23")

## Step 23d — Opening reversal, gated by the volatility regime

**Idea.** Step 13e measures whether the first minutes after the open give back part of the overnight gap, and whether that happens more in high-volatility regimes. This step turns the test into a **same-day** trade: observe the gap when the first day bar closes, trade against it, and exit $k$ minutes after the open. It uses the machinery of Step 23c (clean 1-minute returns, the same costs, limit-lock rule, vol target and deflated Sharpe) and registers in the same strategy family.

**Formula.** For product $i$ and day $t$, with $z^{ON}_t$ the overnight gap in vol units (known when the first bar closes) and $S_t\in\{LOW,MID,HIGH\}$ the ex-ante regime (known the evening before):
$$\text{side}_{i,t}=-\,\text{sign}\big(z^{ON}_{i,t}\big)\;\mathbf 1\big[\,|z^{ON}_{i,t}|>\theta\,\big]\;\mathbf 1\big[S_{i,t}\in G\big]\;\mathbf 1[\text{not limit-locked at the open}],\qquad R^{(k)}_{i,t}=\sum_{b=2}^{k}r_{i,t,b}$$
$$w_{i,t}=\text{side}_{i,t}\,\frac{\tau}{\hat\sigma_{R,i,t-1}\sqrt{n_t}},\qquad \text{net}_t=\sum_i w_{i,t}\big(e^{R^{(k)}_{i,t}}-1\big)-\sum_i |w_{i,t}|\cdot 2\,(\text{fee}+\text{half-spread}_{open})$$
$\hat\sigma_R$ is the trailing standard deviation of $R^{(k)}$ up to $t-1$; $n_t$ is the number of active products. The position is opened and closed on the same day, so nothing is carried overnight.

**Grid.** gate $G\in\{$all, not LOW, HIGH$\}$ × $\theta\in\{0,0.5,1,1.5\}$ × exit $k\in\{5,15,30,60\}$ × way $\in\{$reversal, continuation$\}$ (the continuation rows are the control: if both ways win, the result is not about reversal). The variant is chosen on **validation** among those with a positive Sharpe on train and validation; test is touched once; every variant tried enters the deflated Sharpe ratio. An **ablation** then turns the regime gate on and off for the chosen $(\theta,k)$, and the test P&L is split by regime.

**Used later.** A selected variant joins the strategy library (family `H Overnight`) and is judged like every other strategy in Steps 24 to 26. As for 23c, it is not in the CPCV of Step 23b, and its position matrix is a same-day position, not a close-to-close one.

In [ ]:
# step 23d
# ---- 23d.1 grid of opening-reversal variants, regime-gated ----
if OVN_OK:
    idx_tr, idx_va, idx_te = IDX["train"], IDX["val"], IDX["test"]
    VREGP = VREG.reindex(index=RAW.index, columns=PRODS); GAP = ONz.reindex(columns=PRODS)
    LOCKD = ((OV["ret_night"].fillna(0) + OV["ret_open1"]).abs() >= LOCK_FRAC * LIMv).reindex(columns=PRODS)
    CS_OR = (2 * cs_exit * OVN_COST_MULT).reindex(PRODS)                                       # entry and exit both in the opening bucket
    GATE_M = {"all": pd.DataFrame(True, index=RAW.index, columns=PRODS), "not LOW": VREGP >= 0, "HIGH": VREGP == 1}
    def or_ret(k):
        return (OV[f"ret_open{k}"] - OV["ret_day_open"]).where(RAW.notna() & OV[f"ret_open{k}"].notna() & OV["ret_day_open"].notna()).reindex(columns=PRODS)
    def or_positions(GAP_, REGP_, LOCK_, R, gate, theta, way):
        # side is decided when the first bar closes (gap, lock) from the regime of the evening before; size uses the trailing vol of R up to t-1
        sgn = -1.0 if way == "reversal" else 1.0
        gm = {"all": pd.DataFrame(True, index=GAP_.index, columns=GAP_.columns), "not LOW": REGP_ >= 0, "HIGH": REGP_ == 1}[gate]
        S = (sgn * np.sign(GAP_)).where((GAP_.abs() > theta) & gm & ~LOCK_, 0.0).fillna(0.0)
        sg = R.rolling(OVN_VOL_WIN, min_periods=40).std().shift(1)
        act = ((S != 0) & sg.notna()).sum(axis=1).replace(0, np.nan)
        return (S * (TAU / sg)).div(np.sqrt(act), axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(lower=-CAP, upper=CAP, axis=1)
    causal_test("Step 23d opening-reversal positions", or_positions, GAP, VREGP, LOCKD, or_ret(30), "HIGH", 0.5, "reversal", step="23d")
    def or_variant(gate, theta, k, way):
        R = or_ret(k); pos = or_positions(GAP, VREGP, LOCKD, R, gate, theta, way)
        gross = (pos * np.expm1(R).fillna(0.0)).sum(axis=1); cost = (pos.abs().where(R.notna(), 0.0) * CS_OR).sum(axis=1)
        return pd.DataFrame({"gross": gross, "net": gross - cost, "turn": 2 * pos.abs().sum(axis=1)}), pos, R
    GRID_OR, OR_RES, t0 = [], {}, time.time()
    for gate in GATE_M:
        for theta in OR_THETA:
            for k in OR_K:
                for way in ("reversal", "continuation"):
                    out, pos, R = or_variant(gate, theta, k, way); OR_RES[(gate, theta, k, way)] = (out, pos)
                    r = dict(gate=gate, theta=theta, k=k, way=way, trades=float((pos != 0).sum().sum()))
                    for sp, ix in IDX.items():
                        pf = perf(out["net"].reindex(ix), out["gross"].reindex(ix)); r[f"sharpe_{sp}"] = pf.get("sharpe"); r[f"sharpe_gross_{sp}"] = pf.get("sharpe_gross"); r[f"t_{sp}"] = pf.get("t")
                    GRID_OR.append(r)
    OR_GRID = pd.DataFrame(GRID_OR); OR_GRID.round(4).to_csv(OUT2 / "opening_reversal_grid.csv", index=False)
    print(f"{len(OR_GRID)} opening-reversal variants in {time.time()-t0:.0f}s. Net Sharpe, reversal rows, by gate (rows) and exit minute (columns), theta = 0.5, validation | test:")
    for gate in GATE_M:
        g_ = OR_GRID[(OR_GRID.way == "reversal") & (OR_GRID.theta == 0.5) & (OR_GRID.gate == gate)].set_index("k")
        print(f"{gate:8s}", "val ", g_.sharpe_val.round(2).to_dict(), "| test", g_.sharpe_test.round(2).to_dict())
    print("continuation control (same grid):", OR_GRID[(OR_GRID.way == "continuation") & (OR_GRID.theta == 0.5)].groupby("gate")[["sharpe_val", "sharpe_test"]].mean().round(2).to_dict("index"))

In [ ]:
# step 23d
# ---- 23d.2 selection on validation, one look at test, ablation of the regime gate, P&L by regime ----
if OVN_OK:
    SEL_OR = OR_GRID[(OR_GRID.sharpe_train > 0) & (OR_GRID.sharpe_val > 0)]
    sr_star_or = dsr_threshold((OR_GRID.sharpe_val.dropna() / np.sqrt(ANN)).values); OR_PICK = None; OR_SEL = pd.DataFrame(); OR_ABL = pd.DataFrame(); OR_ATTR = {}
    if len(SEL_OR):
        OR_PICK = SEL_OR.sort_values("sharpe_val", ascending=False).iloc[0]; key = (OR_PICK.gate, OR_PICK.theta, int(OR_PICK.k), OR_PICK.way); out, pos = OR_RES[key]; nt = out["net"].reindex(idx_te).dropna()
        p_ = perf(nt, out["gross"].reindex(idx_te), out["turn"].reindex(idx_te)); tr_ = pos.abs().reindex(idx_te); gross_bp = float(out['gross'].reindex(idx_te).sum() / max(float(tr_.sum().sum()), 1e-9) * 1e4)
        cost_bp = float(CS_OR.mean() * 1e4)
        OR_SEL = pd.DataFrame([dict(gate=OR_PICK.gate, theta=OR_PICK.theta, exit_min_after_open=int(OR_PICK.k), way=OR_PICK.way, sharpe_train=OR_PICK.sharpe_train, sharpe_val=OR_PICK.sharpe_val, sharpe_test=OR_PICK.sharpe_test, sharpe_gross_test=OR_PICK.sharpe_gross_test, t_test=OR_PICK.t_test,
                                    psr0_test=p_.get("psr0"), dsr_test=psr(OR_PICK.sharpe_test / np.sqrt(ANN), p_.get("n", 0), p_.get("skew", 0), p_.get("kurt", 0) + 3, sr_star_or) if np.isfinite(OR_PICK.sharpe_test) else np.nan,
                                    ann_ret_test=p_.get("ann_ret"), ann_vol_test=p_.get("ann_vol"), max_dd_test=p_.get("max_dd"), hit_test=p_.get("hit"), gross_bp_per_unit_position=gross_bp, round_trip_cost_bp=cost_bp, break_even_cost_x=gross_bp / cost_bp if cost_bp > 0 else np.nan, variants_tried=len(OR_GRID))])
        OR_SEL.round(4).to_csv(OUT2 / "opening_reversal_selected.csv", index=False)
        print(f"Selected on validation (train and validation Sharpe > 0): {len(SEL_OR)} of {len(OR_GRID)} variants qualify; {len(OR_GRID)} tried (deflated Sharpe uses them):\n", OR_SEL.round(3).T.to_string())
        abl = []
        for gate in GATE_M:
            o_, _ = OR_RES[(gate, OR_PICK.theta, int(OR_PICK.k), OR_PICK.way)]
            abl.append(dict(gate=gate, **{f"sharpe_{sp}": perf(o_["net"].reindex(ix)).get("sharpe") for sp, ix in IDX.items()}, sharpe_gross_test=perf(o_["net"].reindex(idx_te), o_["gross"].reindex(idx_te)).get("sharpe_gross")))
        OR_ABL = pd.DataFrame(abl); OR_ABL.round(3).to_csv(OUT2 / "opening_reversal_ablation.csv", index=False)
        print("\nAblation: the same (theta, exit, way) with the regime gate on and off, net Sharpe:\n", OR_ABL.round(2).to_string(index=False))
        R_ = or_ret(int(OR_PICK.k)); contrib = pos * np.expm1(R_).fillna(0.0) - pos.abs().where(R_.notna(), 0.0) * CS_OR
        for k_, rn in REGN.items():
            OR_ATTR[rn] = dict(ann_pnl_pct=float(contrib.where(VREGP == k_).sum(axis=1).loc[idx_te].sum() / max(len(idx_te), 1) * ANN * 100), trade_share=float(((pos != 0) & (VREGP == k_)).loc[idx_te].sum().sum() / max((pos != 0).loc[idx_te].sum().sum(), 1)))
        print("\nTest P&L (annualised, % of capital at the vol target) and share of trades by regime:\n", pd.DataFrame(OR_ATTR).round(3).to_string())
        OR_STRESS = []
        for cm in COST_STRESS:
            net_c = out["gross"] - (out["gross"] - out["net"]) * cm; OR_STRESS.append(dict(cost_multiple=cm, sharpe_val=perf(net_c.reindex(idx_va)).get("sharpe"), sharpe_test=perf(net_c.reindex(idx_te)).get("sharpe")))
        OR_STRESS = pd.DataFrame(OR_STRESS); OR_STRESS.to_csv(OUT2 / "opening_reversal_cost_stress.csv", index=False); print("\nCost stress (net Sharpe):\n", OR_STRESS.round(2).to_string(index=False))
    else:
        print("no opening-reversal variant has a positive Sharpe on train AND validation: nothing is added to the library (the grid above is the evidence)")
    G_ = OR_GRID.dropna(subset=["sharpe_val", "sharpe_test"]); OR_RANKCORR = float(stats.spearmanr(G_.sharpe_val, G_.sharpe_test)[0]) if len(G_) > 5 else np.nan
    print(f"\nrank correlation of validation and test Sharpe across all {len(G_)} variants: {OR_RANKCORR:.2f}")
    OR_LIB = {}
    if OR_PICK is not None:
        nm = f"H3 Opening reversal ({OR_PICK.gate}, |gap|>{OR_PICK.theta:g}, +{int(OR_PICK.k)}m, {OR_PICK.way})"; out, pos = OR_RES[key]
        STRATS[nm] = dict(family="H Overnight", W=pos.reindex(RAW.index).fillna(0.0)); BT[nm] = {d: out.reindex(RAW.index).fillna(0.0) for d in DELAYS}; NETS[nm] = out["net"].reindex(OOS)
        v_ = perf(out["net"].reindex(idx_va))
        if v_.get("sharpe", 0) > 0 and v_.get("t", 0) > 1.0 and nm not in SELECTED: SELECTED.append(nm)
        OR_LIB[nm] = dict(val_sharpe=v_.get("sharpe"), selected=nm in SELECTED); print("\nadded to the strategy library:", OR_LIB)
    _g = lambda gate, c: float(OR_ABL.set_index("gate").loc[gate, c]) if len(OR_ABL) else np.nan
    _rd = ("No opening-reversal variant has a positive Sharpe on both train and validation: nothing added." if OR_PICK is None else
           f"Opening reversal ({OR_PICK.gate}, |gap| > {OR_PICK.theta:g} vol units, exit {int(OR_PICK.k)} min after the open, {OR_PICK.way}): Sharpe train {OR_PICK.sharpe_train:+.2f}, validation {OR_PICK.sharpe_val:+.2f}, test {OR_PICK.sharpe_test:+.2f} net "
           f"(gross {OR_PICK.sharpe_gross_test:+.2f}); regime gate ablation on test: all {_g('all', 'sharpe_test'):+.2f}, not LOW {_g('not LOW', 'sharpe_test'):+.2f}, HIGH {_g('HIGH', 'sharpe_test'):+.2f}. Validation-to-test rank correlation across {len(G_)} variants {OR_RANKCORR:+.2f}.")
    print("\n" + _rd)
    OR_STRAT_SUMMARY = dict(selected=OR_SEL.round(3).to_dict("records"), ablation=OR_ABL.round(3).to_dict("records"), attribution=OR_ATTR, rank_corr=OR_RANKCORR, library=OR_LIB, reading=_rd)
else:
    OR_STRAT_SUMMARY = dict(selected=[], library={}, reading="skipped: the cached daily panel has no opening-move columns")

In [ ]:
# step 23d
if OVN_OK:
    fig, axs = plt.subplots(2, 2, figsize=(15, 9))
    ax = axs[0, 0]
    for gate, c_ in zip(GATE_M, (MUTE, SERIES[0], SERIES[1])):
        for way, mk in (("reversal", "o"), ("continuation", "x")):
            g_ = OR_GRID[(OR_GRID.gate == gate) & (OR_GRID.way == way)]; ax.scatter(g_.sharpe_val, g_.sharpe_test, color=c_, marker=mk, s=26, label=f"{gate} {way}")
    ax.axhline(0, color=INK2, lw=1); ax.axvline(0, color=INK2, lw=1); ax.set_xlabel("validation net Sharpe"); ax.set_ylabel("test net Sharpe"); ax.legend(frameon=False, fontsize=7, ncol=2); ax.set_title("Every variant: validation against test (reversal o, continuation x)")
    ax = axs[0, 1]
    if OR_PICK is not None:
        o_, _ = OR_RES[key]; cum = o_["net"].fillna(0).cumsum() * 100; ax.plot(cum.index, cum.values, color=SERIES[0], lw=1.3, label="net"); ax.plot(cum.index, (o_["gross"].fillna(0).cumsum() * 100).values, color=SERIES[1], lw=1, ls="--", label="gross")
        for sp in ("val", "test"): ax.axvline(SPL[sp][0], color=INK2, ls=":", lw=1)
        ax.legend(frameon=False, fontsize=8); ax.set_ylabel("cumulative return, % of capital"); ax.set_title(f"Chosen variant: {OR_PICK.gate}, |gap| > {OR_PICK.theta:g}, +{int(OR_PICK.k)} min")
    else:
        ax.text(0.5, 0.5, "no variant selected", ha="center", va="center", transform=ax.transAxes); ax.set_title("Chosen variant")
    ax = axs[1, 0]
    if len(OR_ABL):
        x_ = np.arange(len(OR_ABL))
        for j, (sp, c_) in enumerate(zip(("train", "val", "test"), (MUTE, SERIES[0], SERIES[1]))): ax.bar(x_ + (j - 1) * 0.27, OR_ABL[f"sharpe_{sp}"].values, 0.27, color=c_, label=sp)
        ax.set_xticks(x_); ax.set_xticklabels(OR_ABL.gate); ax.axhline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=8)
    ax.set_ylabel("net Sharpe"); ax.set_title("Ablation: does the regime gate help? (same theta, exit, way)")
    ax = axs[1, 1]
    if OR_ATTR:
        a_ = pd.DataFrame(OR_ATTR).T.reindex(["LOW", "MID", "HIGH"]); ax.bar(np.arange(3) - 0.2, a_.ann_pnl_pct.values, 0.4, color=SERIES[0], label="annualised test P&L, %"); ax2 = ax.twinx(); ax2.bar(np.arange(3) + 0.2, a_.trade_share.values, 0.4, color=SERIES[3], label="share of trades"); ax2.set_ylim(0, 1); ax2.grid(False)
        ax.set_xticks(range(3)); ax.set_xticklabels(a_.index); ax.axhline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=8, loc="upper left"); ax2.legend(frameon=False, fontsize=8, loc="upper right")
    ax.set_title("Test P&L and trades by volatility regime")
    plt.tight_layout(); savefig(fig, "23d_opening_reversal", "Opening reversal gated by the volatility regime", "23")

## Step 23e — Late-day high-volatility runs: how long they last, and entering at their start

**Question.** Step 23c enters a few minutes before the close. If volatility is already high **before** the close, and such episodes usually last more than an hour, then entering at the **start** of the episode (rather than just before the close) would also collect the pre-close move, at the cost of holding through more noise. This step measures the episodes first, and only then tests the entry rule.

**Late-day volatility (ex-ante baseline).** The last `PRE_BLK` five-minute blocks of the day session are indexed from the close: block $j=1$ is the last 5 trading minutes, $j=2$ the 5 before, and so on. For product $i$, day $t$:
$$\rho_{j,t}=\frac{RV^{(5)}_{j,t}}{\operatorname{median}\big(RV^{(5)}_{j,t-1},\dots,RV^{(5)}_{j,t-W}\big)},\qquad H_{j,t}=\mathbf 1[\rho_{j,t}\ge x]$$
$W=60$ days. The baseline is the same block of the previous days, so the usual intraday pattern (the close is always busier) does not count as "high".

**Duration.** The run that ends at the close has length (in blocks, 5 trading minutes each)
$$L_t=\max\{n:\;H_{1,t}=H_{2,t}=\dots=H_{n,t}=1\},\qquad L_t=0\text{ if the last block is not high.}$$
$P(L\ge n\mid L\ge1)$ is the **survival curve**: the chance that a high-volatility episode seen at the close had already lasted $n$ blocks. The expected duration used by the rule is the trailing mean $\widehat{D}_{i,t}=\overline{5L}$ over earlier days with $L\ge1$ (expanding, at least 30 episodes). Episodes that end before the close, and runs cut off by the edge of the window, are counted separately. "Trading minutes" skip the lunch break, so a window of 24 blocks can reach into the morning session.

**Entry rule (causal).** A high-volatility run is *confirmed* after $C$ consecutive high blocks. If the earliest confirmed run starts at block $j_0$, the position is opened when block $j_0-C+1$ closes, which leaves $n=j_0-C$ blocks to the close. The rule applies only when $\widehat D_{i,t-1}\ge D_{\min}=60$ minutes (the user's condition); otherwise, or when no run is confirmed, the entry is the usual 5 minutes before the close ($n=1$). The trade is then the 23c trade:
$$R_{i,t}=\underbrace{\textstyle\sum_{b=1}^{n}r^{(5)}_{b,t-1}}_{\text{pre-close move from the entry}}+r^{night}_t+r^{open\,k}_t,\qquad \text{cost}=\text{fee}+\text{half-spread}_{\text{afternoon (early) or close (fallback)}}+\text{fee}+\text{half-spread}_{open}$$
The side is chosen as in 23c: **OV1** per-product bias from train, **OV3** pent-up pressure (intraday move up to the entry time). Controls: fixed entries $n\in\{0,1,3,6,12,24\}$ blocks before the close, which show whether an earlier entry adds anything at all.

**Tests.** (1) Duration statistics by period, regime and sector. (2) Fixed-entry profile: mean gross return against entry time. (3) The early-entry rule against the best fixed entry, both chosen on validation, test touched once, every variant tried counted in the deflated Sharpe ratio. "Higher gain" is supported only if the rule beats the best fixed entry on **test net**, and its gross gain per trade is higher.

**Needs** the five-minute pre-close block columns from Step 1 (`pre_rv5_*`, `pre_ret5_*`); a cache without them is rebuilt once, automatically.

In [ ]:
# step 23e
# ---- 23e.1 late-day high-volatility runs: frequency and duration ----
VPRE_OK = OVN_OK and all(f"pre_{a}5_{j}" in dp.columns for a in ("rv", "ret") for j in (1, PRE_BLK))
if not VPRE_OK:
    print("the cached daily panel has no pre-close block columns: set FORCE_REBUILD = True in Step 0 and re-run Step 1 (Step 23e is skipped)")
    HVE_SUMMARY = dict(available=False, selected=[], library={}, reading="skipped: no pre-close block columns in the cache")
else:
    J_ = PRE_BLK
    RVB = {j: wide(f"pre_rv5_{j}").reindex(index=RAW.index, columns=PRODS) for j in range(1, J_ + 1)}
    RTB = {j: wide(f"pre_ret5_{j}").reindex(index=RAW.index, columns=PRODS) for j in range(1, J_ + 1)}
    def hv_ratio(X):
        return X / X.rolling(HV_WIN, min_periods=30).median().shift(1)                      # same block, previous HV_WIN days
    causal_test("Step 23e late-day volatility ratio (block 1)", hv_ratio, RVB[1], step="23e")
    RHO = {j: hv_ratio(RVB[j]) for j in range(1, J_ + 1)}
    def hv_flags(x): return {j: (RHO[j] >= x) & RHO[j].notna() for j in range(1, J_ + 1)}
    def run_len(HI):
        alive = HI[1].copy(); L = alive.astype(int)
        for j in range(2, J_ + 1):
            alive = alive & HI[j]; L = L + alive.astype(int)
        return L
    X0 = HV_X[len(HV_X) // 2]; HI0 = hv_flags(X0); L0 = run_len(HI0)
    SPLIT_OF = pd.Series("", index=RAW.index); [SPLIT_OF.__setitem__(IDX[sp], sp) for sp in IDX]
    # episodes inside the window (time ascending: block J first)
    Hs = np.stack([HI0[j].to_numpy() for j in range(J_, 0, -1)], axis=0); cur = np.zeros(Hs.shape[1:], int); lc = np.zeros(Hs.shape[1:], bool); recs = []
    for s_ in range(J_):
        h = Hs[s_]; ended = (~h) & (cur > 0)
        if ended.any():
            ti, pi = np.nonzero(ended); recs.append(pd.DataFrame(dict(t=ti, p=pi, blocks=cur[ended], left=lc[ended], at_close=False)))
        lc = np.where((cur == 0) & h, s_ == 0, lc); cur = np.where(h, cur + 1, 0)
    ti, pi = np.nonzero(cur > 0)
    if len(ti): recs.append(pd.DataFrame(dict(t=ti, p=pi, blocks=cur[cur > 0], left=lc[cur > 0], at_close=True)))
    EP = pd.concat(recs, ignore_index=True); EP["minutes"] = 5 * EP.blocks; EP["split"] = SPLIT_OF.to_numpy()[EP.t]; EP["sector"] = SEC_MAP.reindex(PRODS).to_numpy()[EP.p]
    RGd = VREG.reindex(index=RAW.index, columns=PRODS).to_numpy(); EP["regime"] = pd.Series(RGd[EP.t, EP.p]).map(REGN).to_numpy(); EP = EP[EP.split != ""]
    rows = []
    for sp in ("train", "val", "test"):
        ix = IDX[sp]; l_ = L0.loc[ix].stack(); n_all = len(l_); e_ = EP[EP.split == sp]; c_ = e_[e_.at_close]; w_ = e_[(~e_.at_close) & (~e_.left)]
        rows.append(dict(split=sp, share_days_high_at_close=float((l_ >= 1).sum() / max(n_all, 1)), episodes_at_close=len(c_), mean_min_at_close=float(c_.minutes.mean()), median_min_at_close=float(c_.minutes.median()),
                         share_ge_60min=float((c_.minutes >= 60).mean()), share_cut_by_window=float((c_.blocks >= J_).mean()), ended_before_close=len(w_), mean_min_ended_before_close=float(w_.minutes.mean())))
    HVE_DUR = pd.DataFrame(rows); HVE_DUR.round(3).to_csv(OUT2 / "late_day_vol_duration.csv", index=False)
    print(f"Late-day high-volatility runs (a 5-minute block is high when its variance is >= {X0}x its own median over the previous {HV_WIN} days; trading minutes, window {5*J_} min):")
    print(HVE_DUR.set_index("split").round(3).to_string())
    ec = EP[EP.at_close & (EP.split != "")]
    byreg = ec.groupby(["split", "regime"]).minutes.agg(["count", "mean", "median"]).round(1); bysec = ec[ec.split == "test"].groupby("sector").minutes.agg(["count", "mean", "median"]).round(1)
    print("\nDuration of the run that ends at the close, by volatility regime:\n", byreg.to_string()); print("\nSame, test period, by sector:\n", bysec.to_string())
    HV_SURV = pd.DataFrame({sp: [float((L0.loc[IDX[sp]].stack() >= n).sum() / max((L0.loc[IDX[sp]].stack() >= 1).sum(), 1)) for n in range(1, J_ + 1)] for sp in IDX}, index=[5 * n for n in range(1, J_ + 1)])
    HV_SURV.index.name = "minutes"; HV_SURV.round(3).to_csv(OUT2 / "late_day_vol_survival.csv")
    print("\nSurvival: share of close-time episodes that had lasted at least this many trading minutes:\n", HV_SURV.loc[[5, 15, 30, 60, 90, 120]].round(3).T.to_string())
    BYREG_HVE = byreg.reset_index(); BYREG_HVE.to_csv(OUT2 / "late_day_vol_duration_by_regime.csv", index=False)

In [ ]:
# step 23e
# ---- 23e.2 entry blocks, fixed-entry profile, and the early-entry grid ----
if VPRE_OK:
    CUM = {0: pd.DataFrame(0.0, index=RAW.index, columns=PRODS)}
    for n_ in range(1, J_ + 1): CUM[n_] = CUM[n_ - 1] + RTB[n_]                       # sum of the last n 5-minute blocks; NaN if any block is missing
    def cum_at(NB):
        out = pd.DataFrame(np.nan, index=NB.index, columns=NB.columns)
        for n_ in range(0, J_ + 1): out = out.where(NB != n_, CUM[n_])
        return out
    def entry_blocks(HI, C):
        # earliest confirmed run (C consecutive high blocks); returns blocks left to the close after confirmation, NaN if none
        ix_, cl_ = HI[1].index, HI[1].columns
        nb_ = pd.DataFrame(np.nan, index=ix_, columns=cl_); found = pd.DataFrame(False, index=ix_, columns=cl_)
        for j0 in range(J_, C - 1, -1):
            ok = HI[j0].copy()
            for c in range(1, C): ok = ok & HI[j0 - c]
            take = ok & ~found; nb_ = nb_.where(~take, float(j0 - C)); found = found | take
        return nb_
    def dur_gate(L, dmin):
        ed = (5.0 * L.where(L >= 1)).expanding(min_periods=30).mean().shift(1)                  # trailing mean duration of close-time episodes
        return (ed >= dmin) if dmin > 0 else pd.DataFrame(True, index=L.index, columns=L.columns)
    causal_test("Step 23e entry blocks", entry_blocks, HI0, HV_CONFIRM[-1], step="23e")
    LIMv = pd.Series({p: LIMIT.get(p, 10) / 100 for p in PRODS}); HS_AFT = pd.Series({p: HS_TOD.loc[p, "afternoon"] if ("afternoon" in HS_TOD.columns and p in HS_TOD.index and np.isfinite(HS_TOD.loc[p, "afternoon"])) else HS.get(p, np.nanmedian(HS.values)) for p in PRODS})
    CS_EARLY = FEE_BPS * 1e-4 + HS_AFT; CS_CLOSE = cs_entry; CS_OPEN = cs_exit
    FB = 1                                                                                    # fallback entry: 5 minutes before the close (as in 23c, m = 5)
    def hve_ret(NB, k):
        n_ = NB.fillna(FB); pre = cum_at(n_).shift(1); night = OV["ret_night"].fillna(0)
        R = pre + night + OV[f"ret_open{k}"]; lock = (night + OV["ret_open1"]).abs() >= LOCK_FRAC * LIMv
        R = R.where(~lock, pre + night + OV["ret_day"]); return R.where(RAW.notna() & OV[f"ret_open{k}"].notna()).reindex(columns=PRODS), n_
    def hve_variant(kind, NB, k):
        R, n_ = hve_ret(NB, k)
        if kind == "OV1":
            tr_ = R.loc[idx_tr]; t_ = tr_.mean() / (tr_.std() / np.sqrt(tr_.count())); side = np.sign(tr_.mean()).where(t_.abs() > OVN_T_MIN, 0.0).fillna(0.0)
            S = pd.DataFrame(np.tile(side.to_numpy(), (len(R), 1)), index=R.index, columns=PRODS)
        else:
            sx = ((OV["ret_day_ex_open"] - cum_at(n_)) / SIGd).reindex(columns=PRODS); rk = sx.rank(axis=1, pct=True) - 0.5; ic = xs_ic_fast(sx.loc[idx_tr], R.shift(-1).loc[idx_tr], min_n=5)       # causal-ok: train-period IC picks the direction
            dsg = 1.0 if ic.mean() >= 0 else -1.0; S = dsg * (rk.where(rk >= 0.3, 0.0).gt(0).astype(float) - rk.where(rk <= -0.3, 0.0).lt(0).astype(float)).where(sx.notna(), 0.0)
        pos = pos_from_side(S, R); p1 = pos.shift(1).fillna(0.0)
        cs_in = pd.DataFrame(np.where(n_.shift(1) > FB, CS_EARLY.reindex(PRODS).to_numpy()[None, :], CS_CLOSE.reindex(PRODS).to_numpy()[None, :]), index=R.index, columns=PRODS)
        gross = (p1 * np.expm1(R).fillna(0.0)).sum(axis=1); cost = (p1.abs() * (cs_in + CS_OPEN.reindex(PRODS).to_numpy()[None, :]) * OVN_COST_MULT).sum(axis=1)
        return pd.DataFrame({"gross": gross, "net": gross - cost, "turn": 2 * p1.abs().sum(axis=1)}), pos, R, n_
    # ---- fixed-entry profile and controls ----
    FIXED = {}; HVE_GR = []; t0 = time.time()
    for nfix in HV_FIXED:
        NBf = pd.DataFrame(float(nfix), index=RAW.index, columns=PRODS)
        for kind in ("OV1", "OV3"):
            for k in HV_K:
                out, pos, R, n_ = hve_variant(kind, NBf, k); FIXED[(kind, nfix, k)] = (out, pos)
                r = dict(rule="fixed", kind=kind, x=np.nan, C=np.nan, gate="-", n_fixed=nfix, k=k, early_share=float(0))
                for sp, ix in IDX.items():
                    pf = perf(out["net"].reindex(ix), out["gross"].reindex(ix)); r[f"sharpe_{sp}"] = pf.get("sharpe"); r[f"sharpe_gross_{sp}"] = pf.get("sharpe_gross"); r[f"t_{sp}"] = pf.get("t")
                HVE_GR.append(r)
    # ---- early-entry variants ----
    EARLY = {}; ENT = {}
    for x in HV_X:
        HIx = hv_flags(x); Lx = run_len(HIx)
        for C_ in HV_CONFIRM:
            NBx = entry_blocks(HIx, C_)
            for gname, dmin in (("none", 0), (f"D>={HV_DUR_MIN}", HV_DUR_MIN)):
                NBg = NBx.where(dur_gate(Lx, dmin))
                for kind in ("OV1", "OV3"):
                    for k in HV_K:
                        out, pos, R, n_ = hve_variant(kind, NBg, k); EARLY[(kind, x, C_, gname, k)] = (out, pos); ENT[(x, C_, gname)] = n_
                        r = dict(rule="early", kind=kind, x=x, C=C_, gate=gname, n_fixed=np.nan, k=k, early_share=float((n_ > FB).where(RAW.notna()).stack().mean()))
                        for sp, ix in IDX.items():
                            pf = perf(out["net"].reindex(ix), out["gross"].reindex(ix)); r[f"sharpe_{sp}"] = pf.get("sharpe"); r[f"sharpe_gross_{sp}"] = pf.get("sharpe_gross"); r[f"t_{sp}"] = pf.get("t")
                        HVE_GR.append(r)
    HVE_GRID = pd.DataFrame(HVE_GR); HVE_GRID.round(4).to_csv(OUT2 / "late_day_entry_grid.csv", index=False)
    print(f"{len(HVE_GRID)} variants in {time.time()-t0:.0f}s ({(HVE_GRID.rule=='early').sum()} early-entry, {(HVE_GRID.rule=='fixed').sum()} fixed-entry controls)")
    print("\nFixed entry n blocks (5 min each) before the close, exit k=15 min after the open: net Sharpe by period")
    for kind in ("OV1", "OV3"):
        g_ = HVE_GRID[(HVE_GRID.rule == "fixed") & (HVE_GRID.kind == kind) & (HVE_GRID.k == HV_K[len(HV_K) // 2])].set_index("n_fixed")[["sharpe_train", "sharpe_val", "sharpe_test", "sharpe_gross_test"]]
        print(kind, "\n", g_.round(2).to_string())

In [ ]:
# step 23e
# ---- 23e.3 selection on validation, one look at test, early entry against the best fixed entry ----
if VPRE_OK:
    EG = HVE_GRID[HVE_GRID.rule == "early"]; FG = HVE_GRID[HVE_GRID.rule == "fixed"]
    SELE = EG[(EG.sharpe_train > 0) & (EG.sharpe_val > 0)]; HVE_PICK = None; HVE_FIX_PICK = None
    sr_star_h = dsr_threshold((EG.sharpe_val.dropna() / np.sqrt(ANN)).values)
    SELF = FG[(FG.sharpe_train > 0) & (FG.sharpe_val > 0)]
    rows = []
    def sel_row(lbl, r, res):
        out, pos = res; nt = out["net"].reindex(idx_te).dropna(); p_ = perf(nt, out["gross"].reindex(idx_te), out["turn"].reindex(idx_te)); tr_ = pos.shift(1).abs().reindex(idx_te)
        gbp = float(out["gross"].reindex(idx_te).sum() / max(float(tr_.sum().sum()), 1e-9) * 1e4)
        return dict(role=lbl, rule=r.rule, kind=r.kind, x=r.x, C=r.C, gate=r.gate, n_fixed=r.n_fixed, exit_min_after_open=int(r.k), early_share=r.early_share, sharpe_train=r.sharpe_train, sharpe_val=r.sharpe_val, sharpe_test=r.sharpe_test,
                    sharpe_gross_test=r.sharpe_gross_test, t_test=r.t_test, psr0_test=p_.get("psr0"), dsr_test=psr(r.sharpe_test / np.sqrt(ANN), p_.get("n", 0), p_.get("skew", 0), p_.get("kurt", 0) + 3, sr_star_h) if np.isfinite(r.sharpe_test) else np.nan,
                    ann_ret_test=p_.get("ann_ret"), max_dd_test=p_.get("max_dd"), gross_bp_per_unit_position=gbp)
    if len(SELE):
        HVE_PICK = SELE.sort_values("sharpe_val", ascending=False).iloc[0]; kE = (HVE_PICK.kind, HVE_PICK.x, HVE_PICK.C, HVE_PICK.gate, int(HVE_PICK.k)); rows.append(sel_row("early-entry rule (chosen on validation)", HVE_PICK, EARLY[kE]))
    if len(SELF):
        HVE_FIX_PICK = SELF.sort_values("sharpe_val", ascending=False).iloc[0]; kF = (HVE_FIX_PICK.kind, int(HVE_FIX_PICK.n_fixed), int(HVE_FIX_PICK.k)); rows.append(sel_row("best fixed entry (chosen on validation)", HVE_FIX_PICK, FIXED[kF]))
    HVE_SEL = pd.DataFrame(rows); HVE_SEL.round(4).to_csv(OUT2 / "late_day_entry_selected.csv", index=False)
    if len(HVE_SEL):
        print(f"{len(SELE)} of {len(EG)} early-entry variants have a positive Sharpe on train and validation; {len(EG)} tried (deflated Sharpe uses them).\n", HVE_SEL.set_index("role").round(3).T.to_string())
    else: print("no variant has a positive Sharpe on train and validation: nothing is added (the grid is the evidence)")
    # does the earlier entry add gain?  paired view of the grid: gross and net Sharpe, early minus fixed n=1 (the 23c-style entry), same side and exit
    cmp_ = []
    for (kind, x, C_, g, k), (o_, _) in EARLY.items():
        base = FIXED.get((kind, 1, k))
        if base is None: continue
        r = dict(kind=kind, x=x, C=C_, gate=g, k=k)
        for sp, ix in IDX.items():
            r[f"d_net_{sp}"] = perf(o_["net"].reindex(ix)).get("sharpe") - perf(base[0]["net"].reindex(ix)).get("sharpe"); r[f"d_gross_{sp}"] = perf(o_["gross"].reindex(ix)).get("sharpe") - perf(base[0]["gross"].reindex(ix)).get("sharpe")
        cmp_.append(r)
    HVE_CMP = pd.DataFrame(cmp_); HVE_CMP.round(3).to_csv(OUT2 / "late_day_entry_vs_fixed1.csv", index=False)
    if len(HVE_CMP):
        print("\nEarly entry minus the 5-minutes-before-close entry (same side and exit), mean over variants, net / gross Sharpe difference:\n",
              HVE_CMP.groupby("gate")[["d_net_val", "d_net_test", "d_gross_val", "d_gross_test"]].mean().round(3).to_string(), "\nshare of variants where the early entry has the higher test net Sharpe:", round(float((HVE_CMP.d_net_test > 0).mean()), 2))
    G_ = EG.dropna(subset=["sharpe_val", "sharpe_test"]); HVE_RANK = float(stats.spearmanr(G_.sharpe_val, G_.sharpe_test)[0]) if len(G_) > 5 else np.nan
    print(f"rank correlation of validation and test Sharpe across the {len(G_)} early-entry variants: {HVE_RANK:.2f}")
    HVE_LIB = {}
    if HVE_PICK is not None:
        out, pos = EARLY[kE]; nm = f"H4 Overnight early entry at high-vol start ({HVE_PICK.kind}, x{HVE_PICK.x:g}, C{int(HVE_PICK.C)}, {HVE_PICK.gate}, +{int(HVE_PICK.k)}m)"
        STRATS[nm] = dict(family="H Overnight", W=pos.reindex(RAW.index).fillna(0.0)); BT[nm] = {d: out.reindex(RAW.index).fillna(0.0) for d in DELAYS}; NETS[nm] = out["net"].reindex(OOS); v_ = perf(out["net"].reindex(idx_va))
        if v_.get("sharpe", 0) > 0 and v_.get("t", 0) > 1.0 and nm not in SELECTED: SELECTED.append(nm)
        HVE_LIB[nm] = dict(val_sharpe=v_.get("sharpe"), selected=nm in SELECTED); print("\nadded to the strategy library:", HVE_LIB)
    r0 = HVE_DUR.set_index("split").loc["test"]
    sup = bool(HVE_PICK is not None and HVE_FIX_PICK is not None and HVE_PICK.sharpe_test > HVE_FIX_PICK.sharpe_test and HVE_PICK.sharpe_test > 0 and HVE_PICK.sharpe_gross_test > HVE_FIX_PICK.sharpe_gross_test)
    _g = lambda lbl, c: float(HVE_SEL.set_index("role").loc[lbl, c]) if len(HVE_SEL) and lbl in HVE_SEL.role.values else np.nan
    _rd = (f"High volatility (block variance >= {X0}x its own recent median) is present in the last 5 minutes on {100*r0.share_days_high_at_close:.0f}% of product-days (test); when it is, the run lasts {r0.mean_min_at_close:.0f} trading minutes on average "
           f"(median {r0.median_min_at_close:.0f}); {100*r0.share_ge_60min:.0f}% of runs last 60 minutes or more ({100*r0.share_cut_by_window:.0f}% are cut off by the {5*J_}-minute window). "
           + ("No early-entry variant had a positive Sharpe on train and validation." if HVE_PICK is None else
              f"Entering at the start of the run (chosen on validation: {HVE_PICK.kind}, {HVE_PICK.gate}) has net Sharpe train {HVE_PICK.sharpe_train:+.2f}, validation {HVE_PICK.sharpe_val:+.2f}, test {HVE_PICK.sharpe_test:+.2f} against the best fixed entry "
              f"{_g('best fixed entry (chosen on validation)', 'sharpe_test'):+.2f} on test. Higher gain from the earlier entry: {'SUPPORTED' if sup else 'NOT supported'} (needs a higher test net and gross Sharpe than the best fixed entry)."))
    print("\n" + _rd)
    HVE_SUMMARY = dict(available=True, supported=sup, mean_min_at_close_test=float(r0.mean_min_at_close), share_ge_60min_test=float(r0.share_ge_60min), selected=HVE_SEL.round(3).to_dict("records"), rank_corr=HVE_RANK, library=HVE_LIB, reading=_rd)

In [ ]:
# step 23e
if VPRE_OK:
    fig, axs = plt.subplots(2, 3, figsize=(18, 9)); cs3 = [MUTE, SERIES[0], SERIES[1]]
    ax = axs[0, 0]
    for sp, c_ in zip(("train", "val", "test"), cs3): ax.plot(HV_SURV.index, HV_SURV[sp], color=c_, lw=1.6, label=sp)
    ax.axvline(60, color=INK2, ls=":", lw=1); ax.set_xlabel("trading minutes the run had lasted at the close"); ax.set_ylabel("share of close-time episodes"); ax.legend(frameon=False, fontsize=8); ax.set_title("How long do late-day high-volatility runs last? (survival)")
    ax = axs[0, 1]; ec_ = EP[EP.at_close]
    for sp, c_ in zip(("train", "val", "test"), cs3): ax.hist(ec_[ec_.split == sp].minutes, bins=np.arange(2.5, 5 * J_ + 7.5, 5), weights=np.ones((ec_.split == sp).sum()) / max((ec_.split == sp).sum(), 1), histtype="step", color=c_, lw=1.5, label=sp)
    ax.axvline(60, color=INK2, ls=":", lw=1); ax.set_xlabel("minutes (last bar = window edge)"); ax.set_ylabel("share"); ax.legend(frameon=False, fontsize=8); ax.set_title("Duration of the run that ends at the close")
    ax = axs[0, 2]; k_ = HV_K[len(HV_K) // 2]
    for kind, mk_ in (("OV1", "o-"), ("OV3", "s--")):
        g_ = HVE_GRID[(HVE_GRID.rule == "fixed") & (HVE_GRID.kind == kind) & (HVE_GRID.k == k_)].set_index("n_fixed")
        for sp, c_ in (("val", SERIES[0]), ("test", SERIES[1])): ax.plot(g_.index * 5, g_[f"sharpe_{sp}"], mk_, color=c_, label=f"{kind} {sp}")
    ax.axhline(0, color=INK2, lw=1); ax.set_xlabel("entry, minutes before the close (fixed)"); ax.set_ylabel("net Sharpe"); ax.legend(frameon=False, fontsize=7); ax.set_title("Does an earlier fixed entry pay?")
    ax = axs[1, 0]
    for rule, mk_, c_ in (("early", "o", SERIES[0]), ("fixed", "x", SERIES[3])):
        g_ = HVE_GRID[HVE_GRID.rule == rule]; ax.scatter(g_.sharpe_val, g_.sharpe_test, marker=mk_, color=c_, s=24, label=rule)
    ax.axhline(0, color=INK2, lw=1); ax.axvline(0, color=INK2, lw=1); ax.set_xlabel("validation net Sharpe"); ax.set_ylabel("test net Sharpe"); ax.legend(frameon=False, fontsize=8); ax.set_title("Every variant: validation against test")
    ax = axs[1, 1]
    for lbl, key_, c_ in (("early-entry rule", "E", SERIES[0]), ("best fixed entry", "F", SERIES[1])):
        res = EARLY.get(kE) if (key_ == "E" and HVE_PICK is not None) else (FIXED.get(kF) if (key_ == "F" and HVE_FIX_PICK is not None) else None)
        if res is not None: cm_ = res[0]["net"].fillna(0).cumsum() * 100; ax.plot(cm_.index, cm_.values, color=c_, lw=1.3, label=lbl)
    for sp in ("val", "test"): ax.axvline(SPL[sp][0], color=INK2, ls=":", lw=1)
    ax.legend(frameon=False, fontsize=8); ax.set_ylabel("cumulative net return, % of capital"); ax.set_title("Chosen early-entry rule against the best fixed entry")
    ax = axs[1, 2]
    if len(HVE_CMP):
        d_ = HVE_CMP.groupby("gate")[["d_net_val", "d_net_test", "d_gross_val", "d_gross_test"]].mean(); d_.plot.bar(ax=ax, color=[SERIES[0], SERIES[1], MUTE, SERIES[3]], width=0.8); ax.axhline(0, color=INK2, lw=1); ax.tick_params(axis="x", rotation=0)
        ax.legend(frameon=False, fontsize=7); ax.set_ylabel("Sharpe difference (early minus 5-min entry)")
    ax.set_title("Early entry minus the 5-minute entry, mean over variants")
    plt.tight_layout(); savefig(fig, "23e_late_day_vol_entry", "Late-day high-volatility runs and the early-entry rule", "23")

## Step 24 — Portfolio construction: volatility targeting, risk parity, HRP

Step 23 gives a library of strategy return streams. This step turns the validated ones into **one book**, still walk-forward and net of costs:

**1. Strategy-level volatility targeting.** Each strategy is scaled by $\min(\text{cap},\;\sigma^\*/\hat\sigma_{t-1})$, where $\hat\sigma$ is an EWMA of its own returns known the day before. Strategies then contribute comparable risk, and risk is not concentrated where volatility happens to be high.

**2. Allocation across strategies**, re-estimated every `PORT_REBAL` days from the previous `PORT_COV_WIN` days only (Ledoit–Wolf shrunk covariance):

| Scheme | Weights |
|---|---|
| EW | $1/N$ |
| IV | $\propto 1/\sigma_i$ (inverse volatility) |
| **ERC / risk parity** | every strategy contributes the same risk: $w_i(\Sigma w)_i = \text{const}$ |
| **HRP** | López de Prado's hierarchical risk parity: cluster by correlation, split risk down the tree. No matrix inversion. |
| MV | long-only minimum variance (the classical optimiser, for contrast) |

**3. Portfolio-level volatility targeting** (a volatility-managed book): the combined return is scaled to `PORT_TARGET_VOL` with leverage capped at `PORT_LEV_CAP`.

**4. What the book is made of:** risk contribution by strategy and by sector, net and gross exposure by sector.

**5. Can it be run, and at what cost?** *Capacity*: the AUM at which positions reach `PARTICIPATION` of the 20-day median traded value. *Cost stress*: the book's Sharpe if costs are 0.5×, 1×, 2×, 3× the assumption, and the break-even multiple.

**Used later.** The book weights give exposure and risk by sector, capacity and cost stress, which become the sizing notes of the board.

In [ ]:
# step 24
from sklearn.covariance import LedoitWolf
CAND = [s for s in SELECTED if s in NETS.columns]
if len(CAND) < 3:
    CAND = BT_SUMMARY.sort_values("sharpe_val", ascending=False).index[:6].tolist()
    print("fewer than 3 strategies passed the validation screen: using the 6 best on validation, for illustration only")
CAND = list(dict.fromkeys(CAND))[:25]
RETN = NETS[CAND].fillna(0.0)
GRS = pd.DataFrame({k: BT[k][MAIN_DELAY].gross.reindex(OOS) for k in CAND}).fillna(0.0)
ewv = lambda r: r.pow(2).ewm(halflife=PORT_VOL_HL, min_periods=20).mean().pow(0.5).shift(1)        # ex-ante: returns up to t-1
TGT_D = PORT_TARGET_VOL / np.sqrt(ANN)
LEV = (TGT_D / ewv(RETN)).clip(upper=PORT_LEV_CAP).fillna(0.0)                                      # strategy-level vol target
VT = RETN * LEV
print(f"{len(CAND)} strategies enter the book (selected on validation); median leverage after strategy-level vol targeting: {LEV.loc[LEV.sum(axis=1) > 0].median().median():.2f}")

def w_iv(C):
    v = 1 / np.sqrt(np.maximum(np.diag(C), 1e-14)); return v / v.sum()
def w_erc(C, iters=300):
    n = len(C); w = w_iv(C)
    for _ in range(iters):
        rc = w * (C @ w); w = w * np.sqrt(rc.mean() / np.maximum(rc, 1e-16)); w /= w.sum()
    return w
def w_hrp(C):
    return hrp_weights(C + np.eye(len(C)) * 1e-12)
def w_mv(C):
    n = len(C); r = optimize.minimize(lambda w: w @ C @ w, np.ones(n) / n, jac=lambda w: 2 * C @ w, bounds=[(0, 1)] * n,
                                      constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], method="SLSQP")
    return r.x if r.success else w_iv(C)
SCHEMES = {"EW": lambda C: np.ones(len(C)) / len(C), "IV": w_iv, "ERC (risk parity)": w_erc, "HRP": w_hrp, "MV (long-only)": w_mv}
reb = list(range(PORT_MIN_OBS, len(VT), PORT_REBAL))
WP, PRET = {}, {}
for nm, fn in SCHEMES.items():
    W = pd.DataFrame(np.nan, index=VT.index, columns=VT.columns)
    for i in reb:
        X = VT.iloc[max(0, i - PORT_COV_WIN + 1): i + 1].to_numpy()
        try:
            W.iloc[i] = fn(LedoitWolf().fit(X).covariance_)
        except Exception:
            W.iloc[i] = 1 / VT.shape[1]
    WP[nm] = W.ffill().fillna(1 / VT.shape[1])
    PRET[nm] = (WP[nm].shift(1) * VT).sum(axis=1)
PRET["EW of raw strategies (no vol target)"] = RETN.mean(axis=1)
# portfolio-level vol target
PV = {nm: p * (TGT_D / ewv(p)).clip(upper=PORT_LEV_CAP).fillna(0.0) for nm, p in PRET.items()}
cv = lambda r: r[r <= r.quantile(0.05)].mean()
rows = []
for variant, D in (("strategy-level target", PRET), ("+ portfolio-level target", PV)):
    for nm, p in D.items():
        for sp, idx in (("val", VALI), ("test", TESTI)):
            q = p.reindex(idx).dropna(); pf = perf(q)
            rows.append(dict(variant=variant, scheme=nm, split=sp, sharpe=pf.get("sharpe"), ann_vol=pf.get("ann_vol"), max_dd=pf.get("max_dd"),
                             cvar5=cv(q), skew=pf.get("skew"), psr0=pf.get("psr0")))
PORT = pd.DataFrame(rows)
for nm, W in WP.items():
    sub = PORT.scheme == nm
    PORT.loc[sub, "turnover_per_rebal"] = W.diff().abs().sum(axis=1).loc[W.index[reb]].mean()
    PORT.loc[sub, "eff_N"] = (1 / (W.loc[TESTI] ** 2).sum(axis=1)).mean(); PORT.loc[sub, "max_w"] = W.loc[TESTI].max(axis=1).mean()
PORT.to_csv(OUT2 / "portfolio_schemes.csv", index=False)
print("Allocation schemes, walk-forward, net of costs:")
print(PORT.pivot_table(index=["variant", "scheme"], columns="split", values=["sharpe", "ann_vol", "max_dd", "cvar5"]).round(3).to_string())
print("\nweights: turnover per rebalance / effective number of strategies / largest weight (test)")
print(PORT[(PORT.variant == "strategy-level target") & (PORT.split == "test")].set_index("scheme")[["turnover_per_rebal", "eff_N", "max_w"]].round(2).to_string())
BEST = PORT[(PORT.variant == "+ portfolio-level target") & (PORT.split == "val") & PORT.scheme.isin(WP)].sort_values("sharpe", ascending=False).scheme.iloc[0]
print(f"\nscheme with the best validation Sharpe (used below): {BEST}")

In [ ]:
# step 24
# what the book is made of: risk contribution by strategy, exposure and risk by sector (scheme chosen on validation)
Wb = WP[BEST]; lev_b = LEV
last_i = len(VT) - 1
Xl = VT.iloc[max(0, last_i - PORT_COV_WIN + 1): last_i + 1].to_numpy(); Cl = LedoitWolf().fit(Xl).covariance_
RC = {}
for nm in ("EW", "ERC (risk parity)", "HRP"):
    w = WP[nm].iloc[-1].to_numpy(); rc = w * (Cl @ w); RC[nm] = pd.Series(rc / rc.sum(), index=VT.columns)
RCD = pd.DataFrame(RC); RCD.to_csv(OUT2 / "portfolio_risk_contribution.csv")
print("Risk contribution by strategy (latest estimate):\n", RCD.round(3).sort_values("EW", ascending=False).head(12).to_string())
# product-level book weights: sum_i a_i * lev_i * W_i (weights decided at t, applied to the return of t+1)
BW = np.zeros((len(OOS), len(PRODS)))
for j, nm in enumerate(CAND):
    a = (Wb[nm] * lev_b[nm]).to_numpy()[:, None]
    BW += a * STRATS[nm]["W"].reindex(OOS).fillna(0.0).shift(MAIN_DELAY).fillna(0.0).to_numpy()
BW = pd.DataFrame(BW, index=OOS, columns=PRODS)
sec_series = SEC_MAP.reindex(PRODS)
net_sec = BW.T.groupby(sec_series).sum().T; gross_sec = BW.abs().T.groupby(sec_series).sum().T
vol_p = COV_FN(RS.reindex(OOS).iloc[-PORT_COV_WIN:].fillna(0.0).to_numpy()); wl = BW.iloc[-1].to_numpy()          # covariance estimator chosen on validation in Step 13b
rc_p = wl * (vol_p @ wl); rc_sec = pd.Series(rc_p, index=PRODS).groupby(sec_series).sum()
rc_sec = rc_sec / rc_sec.abs().sum()
print(f"\nbook (before portfolio-level scaling) on {OOS[-1].date()}: gross exposure {BW.abs().sum(axis=1).iloc[-1]:.2f}x NAV, net {BW.sum(axis=1).iloc[-1]:+.2f}x")
print("sector risk contribution (share of book variance):\n", rc_sec.sort_values(ascending=False).round(3).head(8).to_string())

# capacity: AUM at which a position reaches PARTICIPATION of the 20-day median traded value
MONEY = wide("money").reindex(index=RAW.index, columns=PRODS)
ADV = MONEY.rolling(20, min_periods=10).median().shift(1).reindex(OOS)
aum_i = (PARTICIPATION * ADV / BW.abs().replace(0, np.nan))
def wq(row, wts, q=0.10):
    m = row.notna() & (wts > 0)
    if m.sum() == 0:
        return np.nan
    o = row[m].sort_values(); cw = (wts[m].reindex(o.index).cumsum() / wts[m].sum())
    return o[cw >= q].iloc[0]
cap_t = pd.Series({d: wq(aum_i.loc[d], BW.loc[d].abs()) for d in TESTI})
bind = aum_i.loc[TESTI].dropna(how="all").idxmin(axis=1).value_counts().head(3).to_dict()
CAPACITY = dict(median=float(cap_t.median()), p10=float(cap_t.quantile(0.1)), binding=bind)
print(f"\ncapacity (book scaled to the strategy-level target, positions <= {PARTICIPATION:.0%} of 20-day median traded value): "
      f"90% of exposure fits at AUM ≈ CNY {cap_t.median()/1e8:.1f} bn (median day), {cap_t.quantile(0.1)/1e8:.1f} bn (10th percentile day); tightest products: {bind}")

# cost stress on the chosen scheme
cs_rows = []
costs = (GRS - RETN)                                     # gross minus net = costs paid
for m_ in COST_STRESS:
    Vm = (GRS - m_ * costs) * LEV
    pm = (Wb.shift(1) * Vm).sum(axis=1); pm = pm * (TGT_D / ewv(pm)).clip(upper=PORT_LEV_CAP).fillna(0.0)
    cs_rows.append(dict(cost_multiple=m_, **{f"sharpe_{sp}": perf(pm.reindex(idx)).get("sharpe") for sp, idx in (("val", VALI), ("test", TESTI))}))
CSTRESS = pd.DataFrame(cs_rows); CSTRESS.to_csv(OUT2 / "portfolio_cost_stress.csv", index=False)
print("\nCost stress (book Sharpe):\n", CSTRESS.round(2).to_string(index=False))
be = np.interp(0, CSTRESS.sharpe_test.values[::-1], CSTRESS.cost_multiple.values[::-1]) if CSTRESS.sharpe_test.notna().all() and CSTRESS.sharpe_test.iloc[-1] < 0 < CSTRESS.sharpe_test.iloc[0] else np.nan
print(f"break-even cost multiple on test: {be:.1f}x" if np.isfinite(be) else "break-even cost multiple: not reached inside the tested range")
PORT_BOOKS = {f"{nm} + vol target": PV[nm] for nm in WP}

In [ ]:
# step 24
fig, axs = plt.subplots(1, 3, figsize=(18, 4.9))
ax = axs[0]
for k, nm in enumerate(list(WP) + ["EW of raw strategies (no vol target)"]):
    ax.plot(OOS, (1 + PV[nm].reindex(OOS).fillna(0)).cumprod(), color=SERIES[k % 8], lw=1.3, label=nm)
for sp in ("test",):
    ax.axvline(SPL[sp][0], color=INK2, ls="--", lw=1)
ax.set_yscale("log"); ax.legend(frameon=False, fontsize=6); ax.set_title("Allocation schemes with portfolio-level vol target (net, walk-forward)")
ax = axs[1]
Wb.rolling(5, min_periods=1).mean().iloc[::5].plot.area(ax=ax, stacked=True, linewidth=0, legend=False, color=[SERIES[i % 8] for i in range(Wb.shape[1])])
ax.set_ylim(0, 1); ax.set_title(f"Strategy weights over time — {BEST}"); ax.set_xlabel("")
ax = axs[2]
x = np.arange(len(RCD)); wdt = 0.27
for k, nm in enumerate(RCD.columns):
    ax.bar(x + (k - 1) * wdt, RCD[nm].values, width=wdt, color=SERIES[k], label=nm)
ax.set_xticks(x); ax.set_xticklabels([s[:16] for s in RCD.index], rotation=80, ha="right", fontsize=6); ax.axhline(1 / len(RCD), color=INK2, ls="--", lw=1)
ax.legend(frameon=False, fontsize=7); ax.set_ylabel("share of book variance"); ax.set_title("Risk contribution by strategy")
plt.tight_layout()
savefig(fig, "24_portfolio_schemes", "Portfolio construction: schemes, weights, risk contributions", "24")
fig, axs = plt.subplots(1, 3, figsize=(18, 4.4))
ax = axs[0]; net_sec.iloc[::5].plot(ax=ax, lw=0.9, color=[SERIES[i % 8] for i in range(net_sec.shape[1])], legend=False)
ax.axhline(0, color=INK2, lw=1); ax.set_title("Net exposure by sector (× NAV)")
ax = axs[1]; rs_ = rc_sec.sort_values()
ax.barh(rs_.index, rs_.values, color=np.where(rs_.values > 0, SERIES[0], SERIES[1]), height=0.6); ax.axvline(0, color=INK2, lw=1)
ax.set_title("Sector share of book risk (latest)")
ax = axs[2]
ax.plot(CSTRESS.cost_multiple, CSTRESS.sharpe_val, marker="o", color=MUTE, label="validation"); ax.plot(CSTRESS.cost_multiple, CSTRESS.sharpe_test, marker="o", color=SERIES[0], label="test")
ax.axhline(0, color=INK2, lw=1); ax.set_xlabel("cost multiple (1 = assumed fee + slippage)"); ax.set_ylabel("book Sharpe"); ax.legend(frameon=False, fontsize=8)
ax.set_title("Cost stress")
plt.tight_layout()
savefig(fig, "24_portfolio_exposure", "Exposure, sector risk, cost stress", "24")

## Step 24b — Efficient frontier and the copula of the portfolio

**1. Efficient frontier of the strategy library.** The question is not only "which book" (Step 24) but *where it sits on the opportunity set*. Using only **validation-period** returns of the strategies that passed the screen: expected returns are shrunk to the cross-sectional mean (weight `FRONT_SHRINK`, because sample means are the least reliable input in portfolio theory) and the covariance is Ledoit–Wolf. The long-only frontier is traced from the minimum-variance portfolio to the maximum-return one; the **tangency (max-Sharpe)** and minimum-variance portfolios are marked. Every frontier portfolio keeps its validation weights and is then **evaluated on test**: the distance between the in-sample frontier and where those same portfolios land out of sample is the honest cost of estimation error. The Step 24 books and the individual strategies are shown on the same axes.

**2. The copula of the portfolio.** Strategy returns are not jointly normal. A Student-$t$ (and Gaussian) copula is fitted on validation to the strategies of the book, with empirical marginals; $10^5$ joint days are simulated; the book's **VaR and ES (95 %, 99 %)** from the copula are compared with the historical and the Gaussian-copula values, and with what happened on test (exceedance rate against the nominal one). Pairwise tail dependence between strategies shows which of them fail together.

**Used later.** The frontier tells which strategies carry the book's expected return and which only diversify (summary, Step 26c); the copula VaR / ES sets the book's tail-risk limits next to the Step 25 risk model.

In [ ]:
# step 24b
# ---- 24b.1 efficient frontier on validation, evaluated on test ----
FS = [s for s in CAND if s in NETS.columns]
RV_ = NETS.loc[VALI, FS].fillna(0.0); RT_ = NETS.loc[TESTI, FS].fillna(0.0)
mu_s = RV_.mean() * ANN; lw_ = LedoitWolf().fit(RV_.to_numpy()); C_ = lw_.covariance_ * ANN
mu_sh = FRONT_SHRINK * mu_s.mean() + (1 - FRONT_SHRINK) * mu_s.to_numpy()
n_ = len(FS); bnds = [(0, FRONT_MAXW)] * n_; cons = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
def _opt(obj, jac, w0=None):
    r = optimize.minimize(obj, np.ones(n_) / n_ if w0 is None else w0, jac=jac, bounds=bnds, constraints=cons, method="SLSQP", options={"maxiter": 400})
    return r.x if r.success else None
w_mv = _opt(lambda w: w @ C_ @ w, lambda w: 2 * C_ @ w)
w_mv = np.ones(n_) / n_ if w_mv is None else w_mv
r_lo, r_hi = float(w_mv @ mu_sh), float(mu_sh[np.argsort(-mu_sh)[:max(1, int(np.ceil(1 / FRONT_MAXW)))]].mean())
FW, FRONT_ROWS = [], []
for tr_ in np.linspace(r_lo, max(r_hi, r_lo * 1.0001), FRONT_N):
    c2 = cons + [{"type": "eq", "fun": lambda w, tr_=tr_: w @ mu_sh - tr_}]
    r = optimize.minimize(lambda w: w @ C_ @ w, w_mv, jac=lambda w: 2 * C_ @ w, bounds=bnds, constraints=c2, method="SLSQP", options={"maxiter": 400})
    if r.success: FW.append(r.x)
FW = np.array(FW) if len(FW) else w_mv[None, :]
fr_ret_v = FW @ mu_sh; fr_vol_v = np.sqrt(np.einsum("ij,jk,ik->i", FW, C_, FW))
sr_ = fr_ret_v / fr_vol_v; w_tan = FW[int(np.argmax(sr_))]
def realised(w, R):
    p = R.to_numpy() @ w; return float(np.mean(p) * ANN), float(np.std(p, ddof=1) * np.sqrt(ANN))
FRONT = pd.DataFrame({"expected_ret_val": fr_ret_v, "vol_val": fr_vol_v, "sharpe_val_est": sr_})
FRONT["realised_ret_val"], FRONT["realised_vol_val"] = zip(*[realised(w, RV_) for w in FW]); FRONT["ret_test"], FRONT["vol_test"] = zip(*[realised(w, RT_) for w in FW])
FRONT["sharpe_test"] = FRONT.ret_test / FRONT.vol_test; FRONT["eff_N"] = 1 / (FW ** 2).sum(axis=1)
FRONT.round(4).to_csv(OUT2 / "efficient_frontier.csv", index=False)
named = {"min variance": w_mv, "tangency (max Sharpe, shrunk μ)": w_tan, "equal weight": np.ones(n_) / n_}
w_iv_ = 1 / np.sqrt(np.diag(C_)); named["inverse volatility"] = w_iv_ / w_iv_.sum()
named["HRP"] = hrp_weights(C_ + np.eye(n_) * 1e-12); named["ERC (risk parity)"] = w_erc(C_)
FRONT_PTS = pd.DataFrame({nm: dict(zip(("ret_val", "vol_val"), realised(w, RV_)), **dict(zip(("ret_test", "vol_test"), realised(w, RT_))), eff_N=1 / (w ** 2).sum(), max_w=w.max()) for nm, w in named.items()}).T
FRONT_PTS["sharpe_val"] = FRONT_PTS.ret_val / FRONT_PTS.vol_val; FRONT_PTS["sharpe_test"] = FRONT_PTS.ret_test / FRONT_PTS.vol_test
FRONT_PTS.round(3).to_csv(OUT2 / "efficient_frontier_named.csv")
SINGLE = pd.DataFrame({"ret_val": RV_.mean() * ANN, "vol_val": RV_.std() * np.sqrt(ANN), "ret_test": RT_.mean() * ANN, "vol_test": RT_.std() * np.sqrt(ANN)})
print(f"{n_} strategies | long-only, max weight {FRONT_MAXW:.0%}, expected returns shrunk {FRONT_SHRINK:.0%} to the cross-sectional mean | {len(FW)} frontier points")
print("\nNamed portfolios (weights fixed on VALIDATION, performance on both periods; annualised):\n", FRONT_PTS[["ret_val", "vol_val", "sharpe_val", "ret_test", "vol_test", "sharpe_test", "eff_N", "max_w"]].round(3).to_string())
print(f"\nweights of the tangency portfolio (validation): " + ", ".join(f"{FS[i]}: {w_tan[i]:.0%}" for i in np.argsort(-w_tan)[:6] if w_tan[i] > 0.01))
print(f"out-of-sample frontier quality: mean Sharpe of the frontier points {FRONT.sharpe_val_est.mean():.2f} estimated vs {FRONT.sharpe_test.mean():.2f} realised on test | tangency {FRONT_PTS.loc['tangency (max Sharpe, shrunk μ)','sharpe_val']:.2f} → {FRONT_PTS.loc['tangency (max Sharpe, shrunk μ)','sharpe_test']:.2f}")
FRONT_SUMMARY = dict(n=n_, tangency_val=float(FRONT_PTS.loc['tangency (max Sharpe, shrunk μ)', 'sharpe_val']), tangency_test=float(FRONT_PTS.loc['tangency (max Sharpe, shrunk μ)', 'sharpe_test']),
                     minvar_test=float(FRONT_PTS.loc['min variance', 'sharpe_test']), ew_test=float(FRONT_PTS.loc['equal weight', 'sharpe_test']), erc_test=float(FRONT_PTS.loc['ERC (risk parity)', 'sharpe_test']),
                     tangency_weights={FS[i]: float(w_tan[i]) for i in np.argsort(-w_tan)[:5] if w_tan[i] > 0.01})

In [ ]:
# step 24b
# ---- 24b.2 copula of the portfolio: dependence between strategies, copula VaR / ES vs historical, backtest on test ----
Rb = (WP[BEST].shift(1) * VT).sum(axis=1)                                   # the book chosen in Step 24 (strategy-level target)
SB = [s for s in VT.columns]
V_ = VT.loc[VALI, SB].fillna(0.0); T_ = VT.loc[TESTI, SB].fillna(0.0); wb = WP[BEST].loc[VALI].mean().to_numpy()
Uv = np.column_stack([pseudo_obs(V_[c].to_numpy()) for c in SB])
Sv = stats.norm.ppf(_clip(Uv)); Rv = np.corrcoef(Sv.T); Rv = 0.9 * Rv + 0.1 * np.eye(len(Rv))
prof_p = {}
for nu in _NU_GRID:
    X_ = stats.t.ppf(_clip(Uv), nu); prof_p[nu] = float(np.mean(multivariate_t(loc=np.zeros(len(Rv)), shape=Rv, df=nu).logpdf(X_) - stats.t.logpdf(X_, nu).sum(axis=1)))
NU_P = max(prof_p, key=prof_p.get)
# pairwise tail dependence between strategies (lower, validation + test pooled) and Kendall tau
VT_all = VT.loc[VALI.union(TESTI), SB].fillna(0.0); Ua = np.column_stack([pseudo_obs(VT_all[c].to_numpy()) for c in SB])
LAM = np.array([[tail_emp(Ua[:, i], Ua[:, j], 0.10)[0] if i != j else np.nan for j in range(len(SB))] for i in range(len(SB))])
TAU_S = pd.DataFrame(np.array([[stats.kendalltau(Ua[:, i], Ua[:, j])[0] if i != j else 1.0 for j in range(len(SB))] for i in range(len(SB))]), index=SB, columns=SB)
rho_p = TAU_S.to_numpy()[np.triu_indices(len(SB), 1)]; lam_p = LAM[np.triu_indices(len(SB), 1)]
print(f"{len(SB)} strategies in the book | Student-t copula fitted on validation: ν = {NU_P} (profile: " + ", ".join(f"{k}:{v:.1f}" for k, v in prof_p.items()) + f")")
print(f"strategy pairs: median Kendall τ {np.median(rho_p):.2f}, median lower-tail dependence {np.nanmedian(lam_p):.2f} (independence 0.10); pairs with λ_L > 0.3: {int((lam_p > 0.3).sum())} of {len(lam_p)}")
# marginals: empirical (validation); joint simulation
rngp = np.random.default_rng(SEED + 1); NSP = PORT_COPULA_SIM; Lp = np.linalg.cholesky(Rv)
Xg_ = rngp.standard_normal((NSP, len(SB))) @ Lp.T; Ug_ = stats.norm.cdf(Xg_)
Wc_ = rngp.chisquare(NU_P, NSP) / NU_P; Xt_ = (rngp.standard_normal((NSP, len(SB))) @ Lp.T) / np.sqrt(Wc_)[:, None]; Ut_ = stats.t.cdf(Xt_, NU_P)
Vs = np.sort(V_.to_numpy(), axis=0); n_v = len(Vs)
def to_ret(U):
    idx = np.clip((U * n_v).astype(int), 0, n_v - 1); return np.take_along_axis(Vs, idx, axis=0)
pg = to_ret(Ug_) @ wb; pt = to_ret(Ut_) @ wb; ph = V_.to_numpy() @ wb
def var_es(x, a):
    q = np.quantile(x, 1 - a); return -q, -x[x <= q].mean()
rows = []
for a in (0.95, 0.99):
    for nm, x in (("historical (validation)", ph), ("Gaussian copula", pg), ("Student-t copula", pt)):
        v_, e_ = var_es(x, a); exc = float(np.mean(T_.to_numpy() @ wb < -v_))
        nT = len(T_); xx = int(round(exc * nT)); pp = 1 - a
        kp = float(stats.chi2.sf(-2 * (xx * np.log(pp) + (nT - xx) * np.log(1 - pp) - xx * np.log(xx / nT) - (nT - xx) * np.log(1 - xx / nT)), 1)) if 0 < xx < nT else np.nan
        rows.append(dict(level=a, model=nm, VaR=v_, ES=e_, test_exceedance=exc, nominal=pp, kupiec_p=kp))
PCOP = pd.DataFrame(rows); PCOP.round(5).to_csv(OUT2 / "portfolio_copula_risk.csv", index=False)
print("\nBook risk, daily, % of NAV (weights = mean Step 24 weights on validation):\n", PCOP.assign(VaR=lambda d: d.VaR * 100, ES=lambda d: d.ES * 100).round(3).to_string(index=False))
PORT_COPULA_SUMMARY = dict(nu=NU_P, median_tau=float(np.median(rho_p)), median_lowtail=float(np.nanmedian(lam_p)), es99_hist=float(PCOP[(PCOP.level == .99) & PCOP.model.str.startswith("hist")].ES.iloc[0]),
                           es99_t=float(PCOP[(PCOP.level == .99) & PCOP.model.str.startswith("Student")].ES.iloc[0]), es99_gauss=float(PCOP[(PCOP.level == .99) & PCOP.model.str.startswith("Gauss")].ES.iloc[0]))

In [ ]:
# step 24b
fig, axs = plt.subplots(2, 3, figsize=(18, 9))
ax = axs[0, 0]
ax.plot(FRONT.realised_vol_val * 100, FRONT.realised_ret_val * 100, color=SERIES[0], lw=2, label="frontier, validation (weights fitted here)")
ax.plot(FRONT.vol_test * 100, FRONT.ret_test * 100, color=SERIES[1], lw=2, label="same weights, test")
ax.scatter(SINGLE.vol_val * 100, SINGLE.ret_val * 100, s=14, color=MUTE, label="single strategies (validation)")
ax.scatter(SINGLE.vol_test * 100, SINGLE.ret_test * 100, s=14, color=SERIES[3], label="single strategies (test)")
for k, (nm, r_) in enumerate(FRONT_PTS.iterrows()):
    ax.scatter(r_.vol_test * 100, r_.ret_test * 100, marker="D", s=40, color=SERIES[(k + 2) % 8], edgecolor=INK, lw=0.5, zorder=5); ax.annotate(nm.split(" (")[0], (r_.vol_test * 100, r_.ret_test * 100), fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("annualised volatility, %"); ax.set_ylabel("annualised return, %"); ax.legend(frameon=False, fontsize=7); ax.set_title("Efficient frontier: fitted on validation, realised on test (diamonds = named portfolios, test)")
ax = axs[0, 1]; ax.stackplot(np.arange(len(FW)), FW.T, labels=FS, colors=[SERIES[i % 8] for i in range(len(FS))]); ax.set_xlabel("frontier point (minimum variance → maximum return)"); ax.set_ylabel("weight"); ax.set_ylim(0, 1); ax.set_title("Frontier weights")
ax = axs[0, 2]; x_ = np.arange(len(FRONT_PTS)); ax.bar(x_ - 0.2, FRONT_PTS.sharpe_val, 0.4, color=MUTE, label="validation"); ax.bar(x_ + 0.2, FRONT_PTS.sharpe_test, 0.4, color=SERIES[0], label="test")
ax.set_xticks(x_); ax.set_xticklabels([s.split(" (")[0] for s in FRONT_PTS.index], rotation=35, ha="right", fontsize=8); ax.axhline(0, color=INK2, lw=1); ax.legend(frameon=False, fontsize=8); ax.set_title("Named portfolios: Sharpe")
ax = axs[1, 0]; heat(ax, TAU_S, cmap=DIV, center=0, label="Kendall τ between strategies", annot=False); ax.set_title("Dependence between strategies (rank, validation + test)")
ax = axs[1, 1]; ax.hist(lam_p[np.isfinite(lam_p)], bins=18, color=SERIES[0]); ax.axvline(0.10, color=INK2, ls="--"); ax.set_xlabel("lower-tail dependence λ_L (10%)"); ax.set_title(f"Do strategies fail together? (independence = 0.10) — copula ν = {NU_P}")
ax = axs[1, 2]; pv_ = PCOP.pivot(index="model", columns="level", values="ES") * 100; pv_ = pv_.reindex(["historical (validation)", "Gaussian copula", "Student-t copula"])
x_ = np.arange(len(pv_)); ax.bar(x_ - 0.2, pv_[0.95], 0.4, color=MUTE, label="ES 95%"); ax.bar(x_ + 0.2, pv_[0.99], 0.4, color=SERIES[1], label="ES 99%")
ax.set_xticks(x_); ax.set_xticklabels(["historical", "Gaussian\ncopula", "Student-t\ncopula"]); ax.set_ylabel("% of NAV per day"); ax.legend(frameon=False, fontsize=8); ax.set_title("Book expected shortfall by dependence model")
plt.tight_layout()
savefig(fig, "24b_frontier_copula", "Efficient frontier and portfolio copula", "24")

## Step 24c — Selection audit and keep / drop ledger

In the frontier figure of Step 24b, some single strategies lie **above** the "same weights, test" curve. Two questions follow: *why were they not chosen*, and *would choosing them have been sound?*

**1. Selection audit.** For every strategy: Sharpe in validation, in test (flat costs) and in test under the execution model (Step 23b); its weight at the tangency portfolio; and whether it sits above the test curve. Three numbers say how much to trust a dot above the curve:
- the **rank correlation** between validation Sharpe and test Sharpe across all strategies (does the selection rule carry information?);
- the **average test Sharpe of the best-$k$ on validation (what the book could know) against the best-$k$ on test (hindsight)**;
- the **Sharpe that luck alone produces**: with $N$ strategies and no edge, the best of them has a test Sharpe of about $\mathbb{E}[\max_N Z]/\sqrt{\text{years}}$. A dot above the curve that is inside that band is not evidence.

**2. Ledger.** Every strategy receives a decision and a one-line reason:

| Decision | Rule |
|---|---|
| **KEEP** | validation ≥ `FOLLOW_MIN_VAL`, test ≥ `FOLLOW_MIN_TEST`, test after execution costs ≥ `FOLLOW_MIN_EXEC` |
| **KEEP-SMALL** | passed the validation screen and is still positive after execution costs, below the KEEP thresholds: small risk budget |
| **RE-TEST** | strong on test and after costs but failed validation: cannot be adopted on test evidence; trade only after it passes on new data |
| **DROP** | the reason is stated: costs consume the edge, edge did not hold on test, no edge anywhere, too thin |

**Used later.** The ledger is the strategy list of the findings (26b). It uses test Sharpe, so it is an **audit**: the portfolios of Steps 24d–24e are built from validation information only.

In [ ]:
# step 24c
# ---- 24c.1 the dots above the "same weights, test" curve: why were they not in the book, or not weighted more? ----
from scipy.stats import spearmanr
mm = f"{EXEC_AUM_MAIN/1e6:.0f}m"
LED = BT_SUMMARY.join(EXL[[f"val_{mm}", f"test_{mm}", "breakeven_aum_cny_m", "locked_test"]], how="left").rename(columns={f"val_{mm}": "val_exec", f"test_{mm}": "test_exec"})
LED["ret_val"], LED["vol_val"] = NETS.loc[VALI].mean().reindex(LED.index) * ANN, NETS.loc[VALI].std().reindex(LED.index) * np.sqrt(ANN)
LED["ret_test"], LED["vol_test"] = NETS.loc[TESTI].mean().reindex(LED.index) * ANN, NETS.loc[TESTI].std().reindex(LED.index) * np.sqrt(ANN)
_xv = NETS.loc[VALI]; LED["t_val"] = (_xv.mean() / _xv.std() * np.sqrt(_xv.notna().sum())).reindex(LED.index)
LED["w_tangency"] = pd.Series(w_tan, index=FS).reindex(LED.index).fillna(0.0)               # weight the max-Sharpe portfolio gave it (validation)
LED["w_frontier_mean"] = pd.Series(FW.mean(axis=0), index=FS).reindex(LED.index).fillna(0.0)  # average weight along the whole frontier
_fx = FRONT.sort_values("vol_test"); LED["curve_ret_at_vol"] = np.interp(LED.vol_test.fillna(0), _fx.vol_test, _fx.ret_test)
LED["above_test_curve"] = LED.ret_test > LED.curve_ret_at_vol                                 # sits above the orange curve of the figure
LED["in_frontier_set"] = LED.index.isin(FS)

def why_not(r):
    if r.in_frontier_set:
        if r.w_tangency >= 0.05:
            return f"in the book: tangency weight {r.w_tangency:.0%}"
        return f"in the frontier set but weight {r.w_tangency:.0%}: weights follow validation mean and covariance, and its validation Sharpe ({r.sharpe_val:+.2f}) ranked below the others"
    if r.sharpe_val <= 0:
        return f"failed the validation screen: validation Sharpe {r.sharpe_val:+.2f} ≤ 0, no edge was visible before the test period"
    return f"failed the validation screen: validation t = {r.t_val:.1f} ≤ 1"
LED["why_not_in_book"] = LED.apply(why_not, axis=1)

# how much of "above the curve" is skill and how much is picking the best of N afterwards
N_STRAT = len(LED); yrs_t = len(TESTI) / ANN
emax = (1 - np.euler_gamma) * stats.norm.ppf(1 - 1 / N_STRAT) + np.euler_gamma * stats.norm.ppf(1 - 1 / (N_STRAT * np.e))      # expected maximum of N standard normals
luck_max = emax / np.sqrt(yrs_t)                                                              # s.d. of an annual Sharpe estimate over the test years ≈ 1/sqrt(years)
rho_flat = spearmanr(LED.sharpe_val, LED.sharpe_test, nan_policy="omit")[0]; rho_exec = spearmanr(LED.val_exec, LED.test_exec, nan_policy="omit")[0]
k_ = len(FS)
by_val = LED.sort_values("sharpe_val", ascending=False).head(k_); by_test = LED.sort_values("sharpe_test", ascending=False).head(k_)
AUDIT = dict(n=N_STRAT, rho_flat=float(rho_flat), rho_exec=float(rho_exec), test_years=float(yrs_t), luck_max=float(luck_max), best_test=float(LED.sharpe_test.max()),
             k=k_, book_mean_test=float(by_val.sharpe_test.mean()), hindsight_mean_test=float(by_test.sharpe_test.mean()), all_mean_test=float(LED.sharpe_test.mean()),
             above=LED.index[LED.above_test_curve & LED.in_frontier_set].tolist(),
             above_alive=LED.index[LED.above_test_curve & LED.in_frontier_set & (LED.test_exec > 0)].tolist())
print(f"{N_STRAT} strategies tested; {k_} entered the frontier set (" + ("selected by the validation screen" if len(SELECTED) >= 3 else "fewer than 3 passed the screen, so the 6 best on validation were used") + f"). Test period = {yrs_t:.1f} years.\n")
print(f"1. Does validation rank predict test rank?  Spearman ρ = {rho_flat:+.2f} with flat costs, {rho_exec:+.2f} with the execution model ({mm} CNY). "
      f"The selection rule is informative; it is not decisive.")
print(f"2. The {k_} best on VALIDATION (the book) averaged a test Sharpe of {AUDIT['book_mean_test']:+.2f}; the {k_} best on TEST (hindsight, unusable) {AUDIT['hindsight_mean_test']:+.2f}; "
      f"a random pick {AUDIT['all_mean_test']:+.2f}. The gap between the last two is what picking the winners afterwards adds.")
print(f"3. With {N_STRAT} strategies and NO edge at all, the best test Sharpe would be about {luck_max:.2f} by chance alone; the best observed is {AUDIT['best_test']:.2f}"
      f" ({'within' if AUDIT['best_test'] <= 1.25 * luck_max else 'above'} what luck produces). A dot above the curve is not evidence by itself.")
show = LED[LED.in_frontier_set].copy()
print(f"\nThe {k_} strategies in the frontier set (the dots of the figure), test period:\n" +
      show[["sharpe_val", "ret_test", "vol_test", "sharpe_test", "test_exec", "cost_drag", "w_tangency", "above_test_curve"]].round(3).sort_values("sharpe_test", ascending=False).to_string())
print(f"\nDots above the orange curve: {AUDIT['above']}")
print(f"  … of which still profitable after the execution model at {mm} CNY: {AUDIT['above_alive'] or 'none'}")
print("\nBest strategies on test that were NOT in the frontier set, and why:")
print(LED[~LED.in_frontier_set].sort_values("sharpe_test", ascending=False).head(6)[["sharpe_val", "sharpe_test", "test_exec", "dsr", "why_not_in_book"]].round(3).to_string())

In [ ]:
# step 24c
# ---- 24c.2 keep / drop ledger: every strategy, its Sharpe in each period, the decision and the reason ----
mm = f"{EXEC_AUM_MAIN/1e6:.0f}m"
def decide(r):
    v, t, x, g = r.sharpe_val, r.sharpe_test, r.test_exec, r.sharpe_test_gross
    xf = x if np.isfinite(x) else t
    if v >= FOLLOW_MIN_VAL and t >= FOLLOW_MIN_TEST and xf >= FOLLOW_MIN_EXEC:
        return "KEEP", f"passes all three: validation {v:+.2f}, test {t:+.2f}, test after execution costs {xf:+.2f}"
    if bool(r.selected) and t > 0 and xf > 0:
        return "KEEP-SMALL", f"selected on validation ({v:+.2f}) and still positive after execution costs ({xf:+.2f}), but below the FOLLOW thresholds: small risk budget, re-test on new data"
    if (not bool(r.selected)) and t >= FOLLOW_MIN_TEST and xf >= FOLLOW_MIN_EXEC:
        return "RE-TEST", f"good on test ({t:+.2f}) and after costs ({xf:+.2f}) but it failed the validation screen ({v:+.2f}): cannot be adopted on test evidence alone; trade it only after it passes on data not yet seen"
    why = []
    if t > 0 and xf <= 0:
        why.append(f"costs consume the edge: Sharpe {t:+.2f} → {xf:+.2f} at {mm} CNY (cost drag {r.cost_drag*100:.1f}%/yr, gross {g:+.2f})")
    elif bool(r.selected) and t <= 0:
        why.append(f"selected on validation ({v:+.2f}) but negative on test ({t:+.2f}): the validation edge did not hold")
    elif v <= 0 and t <= 0:
        why.append(f"no edge in validation ({v:+.2f}) or test ({t:+.2f})")
    elif t > 0:
        why.append(f"edge too thin after costs ({xf:+.2f} < {FOLLOW_MIN_EXEC}) and not significant (DSR {r.dsr:.2f})")
    else:
        why.append(f"negative on test ({t:+.2f})")
    if r.max_dd < -0.35: why.append(f"deep drawdown ({r.max_dd:.0%})")
    return "DROP", "; ".join(why)
_dr = LED.apply(decide, axis=1, result_type="expand"); LED["decision"], LED["reason"] = _dr[0], _dr[1]
LEDGER = LED[["family", "sharpe_val", "sharpe_test", "test_exec", "sharpe_test_gross", "cost_drag", "turnover", "max_dd", "dsr", "breakeven_aum_cny_m", "decision", "reason"]].copy()
_o = {"KEEP": 0, "KEEP-SMALL": 1, "RE-TEST": 2, "DROP": 3}
LEDGER = LEDGER.iloc[np.lexsort((-LEDGER.test_exec.fillna(-9).to_numpy(), LEDGER.decision.map(_o).to_numpy()))]
LEDGER.round(3).to_csv(OUT2 / "strategy_ledger.csv")
cnt = LEDGER.decision.value_counts().to_dict()
print(f"Ledger of {len(LEDGER)} strategies: " + ", ".join(f"{k} {cnt.get(k, 0)}" for k in _o) + f"   (thresholds: validation ≥ {FOLLOW_MIN_VAL}, test ≥ {FOLLOW_MIN_TEST}, test after execution costs ≥ {FOLLOW_MIN_EXEC})")
print("sharpe_test_exec = test Sharpe under the execution model at " + mm + " CNY; breakeven_aum in CNY m\n")
pd.set_option("display.max_colwidth", 150)
print(LEDGER[LEDGER.decision != "DROP"][["sharpe_val", "sharpe_test", "test_exec", "dsr", "decision", "reason"]].round(2).to_string())
print("\nDropped, by reason:")
_rs = LEDGER[LEDGER.decision == "DROP"].reason.str.replace(r"; deep drawdown.*", "", regex=True).str.split(":").str[0].str.replace(r"[-+]?\d+\.\d+", "x", regex=True).value_counts()
print(_rs.to_string())
print("\nDropped (strongest on test first):")
print(LEDGER[LEDGER.decision == "DROP"].sort_values("sharpe_test", ascending=False)[["sharpe_val", "sharpe_test", "test_exec", "reason"]].round(2).head(15).to_string())
fig, axs = plt.subplots(1, 3, figsize=(18, 5.2))
ax = axs[0]
for dcs, col in (("DROP", MUTE), ("RE-TEST", SERIES[2]), ("KEEP-SMALL", SERIES[1]), ("KEEP", SERIES[0])):
    m_ = LED.decision == dcs
    ax.scatter(LED.sharpe_val[m_], LED.sharpe_test[m_], s=26, color=col, label=f"{dcs} ({int(m_.sum())})", edgecolor=INK, lw=0.3)
for nm in LED.index[LED.in_frontier_set | (LED.sharpe_test > 0.9)]:
    ax.annotate(nm.split(" (")[0][:20], (LED.sharpe_val[nm], LED.sharpe_test[nm]), fontsize=6, xytext=(3, 3), textcoords="offset points")
ax.axhline(0, color=INK2, lw=0.8); ax.axvline(0, color=INK2, lw=0.8); ax.set_xlabel("validation Sharpe"); ax.set_ylabel("test Sharpe (flat costs)")
ax.set_title(f"Validation vs test: ρ = {AUDIT['rho_flat']:+.2f}"); ax.legend(frameon=False, fontsize=7)
ax = axs[1]
d_ = LED.sort_values("sharpe_test_gross", ascending=False).head(22)
yy = np.arange(len(d_)); ax.barh(yy + 0.2, d_.sharpe_test_gross, 0.4, color=MUTE, label="gross (no costs)"); ax.barh(yy - 0.2, d_.sharpe_test, 0.4, color=SERIES[1], label="flat costs")
ax.scatter(d_.test_exec, yy - 0.2, color=INK, s=14, zorder=5, label=f"execution model {mm}"); ax.set_yticks(yy); ax.set_yticklabels([s[:26] for s in d_.index], fontsize=6); ax.invert_yaxis()
ax.axvline(0, color=INK2, lw=0.8); ax.legend(frameon=False, fontsize=7); ax.set_title("What costs do to the best gross Sharpes (test)")
ax = axs[2]
f_ = LED[LED.in_frontier_set]; ax.scatter(f_.w_tangency * 100, f_.sharpe_test, s=60, color=SERIES[0], edgecolor=INK)
for nm, r_ in f_.iterrows(): ax.annotate(nm.split(" (")[0][:20], (r_.w_tangency * 100, r_.sharpe_test), fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.axhline(0, color=INK2, lw=0.8); ax.set_xlabel("weight in the tangency portfolio, % (fixed on validation)"); ax.set_ylabel("test Sharpe"); ax.set_title("Weight given vs what happened")
plt.tight_layout(); savefig(fig, "24c_selection_audit", "Selection audit and keep/drop ledger", "24")

## Step 24d — Portfolios by investor type: risk appetite × minimum lock-in

Two things define an investor: **how much risk they accept** and **how long the money cannot be withdrawn**. Each pair gets its own book, built only from information available before the test period, on returns **after execution costs**.

| Risk appetite | Target volatility | Drawdown budget | Loss probability limit | Eligible strategies (validation, after costs) | Weights |
|---|---|---|---|---|---|
| Conservative | 3 % | 6 % | 15 % | Sharpe ≥ 0.5, t ≥ 1.5 | HRP |
| Moderate | 6 % | 12 % | 25 % | Sharpe ≥ 0.3, t ≥ 1.0 | ERC (risk parity) |
| Growth | 10 % | 20 % | 35 % | Sharpe > 0, t ≥ 0.5 | inverse volatility |
| Aggressive | 15 % | 30 % | 45 % | Sharpe > 0 | capped, shrunk mean-variance |

**Lock-in** (3 months, 1 year, 3 years) sets the horizon of the **admission test**. If the Sharpe of the book on validation, shrunk by half (the mean is the least reliable input), is the truth, the chance of a loss over the lock-in window is $\Phi(-\widehat{SR}\sqrt{L/252})$. The book is held only if that chance is below the investor's limit; otherwise the profile stays in **cash**. A longer lock-in also allows slower rebalancing, which lowers turnover cost.

**Reading the table.** *Admitted* = the admission test passed on validation. The test columns are the audit: realised return, volatility, drawdown, the share of lock-in windows with a loss, and whether the mandate (volatility at most 30 % above target, drawdown inside the budget, loss share inside the limit) was met.

**Used later.** The grid is the investor-facing result of the study (findings, 26b). Cells that fall back to cash say that no strategy of the library earns a place at that risk and horizon.

In [ ]:
# step 24d
# ---- 24d.0 tools shared by Steps 24d-24f: a walk-forward book of strategies, built from any return frame ----
INV_RISK = {                       # risk appetite: target volatility, drawdown budget, largest tolerated probability of a loss over the lock-in window, eligibility screen on validation, weighting scheme
    "Conservative": dict(vol=0.03, dd=0.06, loss_p=0.15, min_sh=0.5, min_t=1.5, scheme="HRP", lev=2.0),
    "Moderate":     dict(vol=0.06, dd=0.12, loss_p=0.25, min_sh=0.3, min_t=1.0, scheme="ERC (risk parity)", lev=3.0),
    "Growth":       dict(vol=0.10, dd=0.20, loss_p=0.35, min_sh=0.0, min_t=0.5, scheme="IV", lev=3.0),
    "Aggressive":   dict(vol=0.15, dd=0.30, loss_p=0.45, min_sh=0.0, min_t=0.0, scheme="MV-cap (shrunk)", lev=4.0)}
INV_LOCK = {"3 months": 63, "1 year": 252, "3 years": 756}          # minimum lock-in, trading days
INV_MAXN = 10                                                       # most strategies in one book
SHRINK_SR = 0.5                                                     # validation Sharpe is shrunk by this much before it is believed

def w_mvcap(C, mu, cap=0.4):
    n = len(C); m = FRONT_SHRINK * mu.mean() + (1 - FRONT_SHRINK) * mu
    f = lambda w: -(w @ m) / np.sqrt(max(w @ C @ w, 1e-16))
    r = optimize.minimize(f, np.ones(n) / n, bounds=[(0, max(cap, 1.0 / n))] * n, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], method="SLSQP", options={"maxiter": 200})
    return r.x if r.success else w_iv(C)
SCHEME_FN = {"EW": lambda C, mu: np.ones(len(C)) / len(C), "IV": lambda C, mu: w_iv(C), "ERC (risk parity)": lambda C, mu: w_erc(C),
             "HRP": lambda C, mu: w_hrp(C) if len(C) > 2 else w_iv(C), "MV-cap (shrunk)": w_mvcap}

def walk_book(R, pool, scheme, reb=None, tgt=None, lev_cap=None, win=None):
    # R: daily net returns of the strategies. Weights at t use returns up to t only; strategy-level and portfolio-level volatility targets use the EWMA up to t-1.
    reb = PORT_REBAL if reb is None else reb; tgt = PORT_TARGET_VOL if tgt is None else tgt; lev_cap = PORT_LEV_CAP if lev_cap is None else lev_cap; win = PORT_COV_WIN if win is None else win
    idx = R.index
    if len(pool) == 0:
        return pd.Series(0.0, index=idx), pd.DataFrame(0.0, index=idx, columns=[]), pd.DataFrame(0.0, index=idx, columns=[])
    Rp = R[pool].fillna(0.0); lev = (TGT_D / ewv(Rp)).clip(upper=lev_cap).fillna(0.0); V = Rp * lev
    W = pd.DataFrame(np.nan, index=idx, columns=pool)
    for i in range(PORT_MIN_OBS, len(V), reb):
        X = V.iloc[max(0, i - win + 1): i + 1]
        try:
            W.iloc[i] = SCHEME_FN[scheme](LedoitWolf().fit(X.to_numpy()).covariance_, X.mean().to_numpy() * ANN)
        except Exception:
            W.iloc[i] = 1.0 / len(pool)
    W = W.ffill().fillna(1.0 / len(pool)); p = (W.shift(1) * V).sum(axis=1)
    g = (tgt / np.sqrt(ANN) / ewv(p)).clip(upper=lev_cap).fillna(0.0)
    return p * g, W, lev

def window_stats(r, L=None):
    # annualised return / vol / Sharpe, maximum drawdown, 95% expected shortfall of a day, and (if L) the loss probability and worst return over L-day windows
    r = pd.Series(r).dropna()
    if len(r) < 30 or r.std() == 0:
        return dict(ann_ret=np.nan, ann_vol=np.nan, sharpe=np.nan, max_dd=np.nan, es95=np.nan, p_loss=np.nan, worst_L=np.nan)
    eq = (1 + r).cumprod(); out = dict(ann_ret=r.mean() * ANN, ann_vol=r.std() * np.sqrt(ANN), sharpe=r.mean() / r.std() * np.sqrt(ANN),
                                      max_dd=float((eq / eq.cummax() - 1).min()), es95=float(-r[r <= r.quantile(0.05)].mean()), p_loss=np.nan, worst_L=np.nan)
    if L and len(eq) > L + 20:
        w_ = (eq / eq.shift(L) - 1).dropna(); out["p_loss"] = float((w_ < 0).mean()); out["worst_L"] = float(w_.min())
    return out

def pool_rule(R, min_sh, min_t, upto, maxn=INV_MAXN, start=None):
    # strategies eligible under a rule, judged on validation-period returns before `upto` (never on test)
    Xv = R.loc[(R.index >= (VALI[0] if start is None else start)) & (R.index < upto)]; sh_ = Xv.mean() / Xv.std() * np.sqrt(ANN); t_ = sh_ / np.sqrt(ANN) * np.sqrt(Xv.notna().sum())
    ok = (sh_ > 0) & (sh_ >= min_sh) & (t_ >= min_t)
    return sh_[ok].sort_values(ascending=False).index[:maxn].tolist()
def pre_test_pool(R, risk, upto, maxn=INV_MAXN):
    c = INV_RISK[risk]; return pool_rule(R, c["min_sh"], c["min_t"], upto, maxn)
print(f"tools ready: {len(SCHEME_FN)} weighting schemes, risk profiles {list(INV_RISK)}, lock-ins {list(INV_LOCK)}")

In [ ]:
# step 24d
# ---- 24d.1 the investor grid: risk appetite x minimum lock-in, built on returns AFTER execution costs ----
mm = f"{EXEC_AUM_MAIN/1e6:.0f}m"
RX = NETS_X[EXEC_AUM_MAIN].copy()
PRE_END = TESTI[0]
rows, BOOK_INV, POOL_INV = [], {}, {}
for rk, c in INV_RISK.items():
    pool = pre_test_pool(RX, rk, PRE_END); POOL_INV[rk] = pool
    for lk, L in INV_LOCK.items():
        reb = int(np.clip(L // 12, PORT_REBAL, 126))                 # a longer lock-in tolerates slower rebalancing, hence lower turnover cost
        adm, sr_hat, p_loss = False, np.nan, np.nan
        if pool:
            b0, _, _ = walk_book(RX, pool, c["scheme"], reb=reb, tgt=c["vol"], lev_cap=c["lev"])
            sr_v = window_stats(b0.loc[VALI]).get("sharpe", np.nan); sr_hat = SHRINK_SR * sr_v if np.isfinite(sr_v) else np.nan
            p_loss = float(stats.norm.cdf(-sr_hat * np.sqrt(L / ANN))) if np.isfinite(sr_hat) else np.nan     # chance of a loss over the lock-in if the shrunk Sharpe is the truth
            adm = bool(np.isfinite(p_loss) and sr_hat > 0 and p_loss <= c["loss_p"])
        if adm:
            book, Wb_, _ = walk_book(RX, pool, c["scheme"], reb=reb, tgt=c["vol"], lev_cap=c["lev"])
        else:
            book, Wb_ = pd.Series(0.0, index=RX.index), pd.DataFrame(0.0, index=RX.index, columns=pool)
        BOOK_INV[(rk, lk)] = book
        r = dict(risk=rk, lock_in=lk, n_strategies=len(pool), scheme=c["scheme"], admitted=adm, shrunk_sharpe_val=sr_hat, p_loss_if_true=p_loss, loss_limit=c["loss_p"], target_vol=c["vol"], dd_budget=c["dd"])
        for sp, idx in (("val", VALI), ("test", TESTI)):
            s_ = window_stats(book.loc[idx], L) if adm else dict(ann_ret=0.0, ann_vol=0.0, sharpe=np.nan, max_dd=0.0, es95=0.0, p_loss=0.0, worst_L=0.0)
            r.update({f"{k}_{sp}": v for k, v in s_.items()})
        r["vol_ok_test"] = (r["ann_vol_test"] <= 1.3 * c["vol"]) if adm else True
        r["dd_ok_test"] = (r["max_dd_test"] >= -c["dd"]) if adm else True
        r["loss_ok_test"] = (r["p_loss_test"] <= c["loss_p"]) if adm and np.isfinite(r["p_loss_test"]) else True
        r["mandate_met_test"] = bool(r["vol_ok_test"] and r["dd_ok_test"] and r["loss_ok_test"]) if adm else np.nan
        r["status"] = "cash" if not adm else ("held, mandate met" if r["mandate_met_test"] else "held, mandate breached")
        r["profitable_test"] = bool(adm and r["ann_ret_test"] > 0)
        rows.append(r)
INV = pd.DataFrame(rows); INV.round(4).to_csv(OUT2 / "investor_profiles.csv", index=False)
print(f"Investor profiles built on returns after the execution model ({mm} CNY). Eligibility, admission and weights use data before the test period only; the test column is the audit.\n")
print("How a profile is built: (1) risk appetite sets the target volatility, drawdown budget, loss-probability limit, which strategies are eligible (validation Sharpe and t) and how they are weighted;")
print("(2) the lock-in sets the horizon of the admission test: the book is held only if the chance of a loss over the lock-in, from the SHRUNK validation Sharpe, is below the limit; otherwise the profile stays in cash.\n")
show = INV[["risk", "lock_in", "n_strategies", "scheme", "admitted", "shrunk_sharpe_val", "p_loss_if_true", "loss_limit", "sharpe_val", "sharpe_test", "ann_ret_test", "ann_vol_test", "max_dd_test", "p_loss_test", "worst_L_test", "status"]]
print(show.round(3).to_string(index=False))
print("\ntest Sharpe (rows: risk appetite, columns: lock-in; blank = cash):")
print(INV.pivot(index="risk", columns="lock_in", values="sharpe_test").reindex(index=list(INV_RISK), columns=list(INV_LOCK)).round(2).to_string())
print("\nstatus on test (mandate = volatility at most 30% above target, drawdown inside the budget, loss share inside the limit; NaN loss share = test shorter than the lock-in):")
print(INV.pivot(index="risk", columns="lock_in", values="status").reindex(index=list(INV_RISK), columns=list(INV_LOCK)).to_string())
print("\nstrategies each risk appetite may hold (eligible on validation):")
for rk, p_ in POOL_INV.items(): print(f"  {rk}: {p_ if p_ else 'none'}")
fig, axs = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, col, ttl, cm in ((axs[0], "sharpe_test", "Test Sharpe", DIV), (axs[1], "max_dd_test", "Test maximum drawdown", SEQ), (axs[2], "p_loss_test", "Share of lock-in windows with a loss (test)", SEQ)):
    M = INV.assign(**{col: INV[col].where(INV.admitted)}).pivot(index="risk", columns="lock_in", values=col).reindex(index=list(INV_RISK), columns=list(INV_LOCK)).astype(float)       # cash cells stay blank
    heat(ax, M, cmap=cm, center=0 if col == "sharpe_test" else None, label=ttl, annot=True, fmt="{:.2f}"); ax.set_title(ttl)
plt.tight_layout(); savefig(fig, "24d_investor_profiles", "Investor profiles: risk appetite x lock-in", "24")
INVESTOR_SUMMARY = dict(admitted=int(INV.admitted.sum()), cells=len(INV), profitable=int(INV.profitable_test.sum()), mandate=int((INV.status == "held, mandate met").sum()))

## Step 24e — Profitable portfolios: cost-aware library and a recipe search chosen without the test

**1. Turnover is the first lever.** The median strategy has a gross Sharpe far above its net Sharpe: costs, not signals, take the edge. A **no-trade band** holds the previous position unless the target has moved by more than $\tau$ times the strategy's typical gross exposure (L1 distance; uses only information up to $t-1$). One global $\tau$ from the grid `BAND_GRID` is chosen on validation.

**2. A fixed grid of recipes.** Library (plain or banded) × eligibility rule (strict, moderate, loose, from validation Sharpe and t) × weighting (EW, IV, ERC, HRP, capped shrunk MV). Selection discipline:
- eligibility is judged on the **first half of validation**, the recipe is scored on the **second half**, and the winner is the best on that second half;
- the chosen recipe is then rebuilt with all pre-test data and evaluated on test; **every** recipe is reported next to it, with a deflated Sharpe for the number of recipes tried;
- diversification is measured by the number of effective bets (entropy of risk shares along principal components), the effective number of weights and the diversification ratio.

**Used later.** The chosen recipe enters `PORT_BOOKS` (risk model, Step 25) and the findings. Whether the search finds anything profitable is a result, not an assumption: if the field is centred at zero, the report says so.

In [ ]:
# step 24e
# ---- 24e.1 a cost-aware library: no-trade band. The median strategy loses most of its gross Sharpe to costs (gross ≫ net), so turnover is the first lever ----
BAND_GRID = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]       # trade only when the target has moved by more than tau x the strategy's typical gross exposure (L1 distance)
def band_weights(W, tau):
    if tau <= 0:
        return W
    A = np.nan_to_num(W.to_numpy(float)); out = np.zeros_like(A)
    g = W.abs().sum(axis=1).rolling(60, min_periods=10).mean().shift(1).fillna(0.0).to_numpy()     # exposure known up to t-1
    prev = np.zeros(A.shape[1])
    for t in range(len(A)):
        if np.abs(A[t] - prev).sum() > tau * g[t]:
            prev = A[t].copy()
        out[t] = prev
    return pd.DataFrame(out, index=W.index, columns=W.columns)
cost_x = make_cost_fn(EXEC_AUM_MAIN)
t0 = time.time(); BAND_NETS, BAND_ROWS, BAND_W = {}, [], {}
for tau in BAND_GRID:
    R_ = {}; Wt_ = {}
    for nm in RX.columns:                                                          # close-to-close strategies of Step 23 (the overnight sleeves of 23c are costed there)
        Wb_ = band_weights(STRATS[nm]["W"], tau); Wt_[nm] = Wb_
        R_[nm] = RX[nm] if tau == 0 else backtest(Wb_, MAIN_DELAY, cost_fn=cost_x, lock=LOCK).net.reindex(OOS)          # tau = 0 is the library already computed in Step 23b
    BAND_NETS[tau] = pd.DataFrame(R_); BAND_W[tau] = Wt_
    Rv, Rt = BAND_NETS[tau].loc[VALI], BAND_NETS[tau].loc[TESTI]; shv = Rv.mean() / Rv.std() * np.sqrt(ANN); sht = Rt.mean() / Rt.std() * np.sqrt(ANN)
    BAND_ROWS.append(dict(tau=tau, val_median=shv.median(), test_median=sht.median(), val_pos=int((shv > 0.3).sum()), test_pos=int((sht > 0.3).sum()), test_pos_any=int((sht > 0).sum()),
                          turnover=np.nanmedian([(Wt_[n].diff().abs().sum(axis=1).mean() * ANN) for n in Wt_])))
BAND = pd.DataFrame(BAND_ROWS).set_index("tau")
TAU_STAR = float(BAND.val_median.idxmax())                                  # ONE global parameter, chosen on validation only
RXB = BAND_NETS[TAU_STAR].copy()
print(f"no-trade band: {len(BAND_GRID)} values x {RX.shape[1]} strategies in {time.time()-t0:.0f}s; all returns under the execution model at {EXEC_AUM_MAIN/1e6:.0f}m CNY")
print(BAND.round(3).to_string())
print(f"\nband chosen on VALIDATION (highest median Sharpe): tau = {TAU_STAR}. Test: median Sharpe {BAND.loc[0.0, 'test_median']:+.2f} → {BAND.loc[TAU_STAR, 'test_median']:+.2f}; "
      f"strategies with test Sharpe > 0.3: {int(BAND.loc[0.0, 'test_pos'])} → {int(BAND.loc[TAU_STAR, 'test_pos'])} of {RX.shape[1]}")
LED["test_exec_band"] = (RXB.loc[TESTI].mean() / RXB.loc[TESTI].std() * np.sqrt(ANN)).reindex(LED.index); LED["val_exec_band"] = (RXB.loc[VALI].mean() / RXB.loc[VALI].std() * np.sqrt(ANN)).reindex(LED.index)
if TAU_STAR == max(BAND_GRID): print("  (the chosen band is the widest of the grid: widen BAND_GRID if turnover is still the main cost)")
imp = (LED.test_exec_band - LED.test_exec).dropna()
print(f"change in test Sharpe per strategy (band vs none): median {imp.median():+.2f}, improved {int((imp > 0).sum())} of {len(imp)}")
print("\nBand effect on the strategies that were in the book:")
print(LED[LED.in_frontier_set][["val_exec", "val_exec_band", "test_exec", "test_exec_band"]].round(2).to_string())

In [ ]:
# step 24e
# ---- 24e.2 search for profitable portfolios: a fixed grid of recipes, chosen WITHOUT looking at test ----
# Recipe = library (plain / banded) x eligibility rule x weighting scheme. The recipe is chosen on the second half of validation (eligibility from the first half only),
# then rebuilt with all pre-test data and evaluated on test. All recipes are reported, so the choice can be judged against the field (and deflated for the number of trials).
LIBS = {"plain": RX, f"band {TAU_STAR}": RXB}
RULES = {"strict (Sharpe ≥ 0.5, t ≥ 1.5)": (0.5, 1.5), "moderate (≥ 0.2, t ≥ 1)": (0.2, 1.0), "loose (> 0)": (0.0, 0.0)}
SCHEMES_R = ["EW", "IV", "ERC (risk parity)", "HRP", "MV-cap (shrunk)"]
VA, VB = VALI[: len(VALI) // 2], VALI[len(VALI) // 2:]            # selection windows inside validation
def eff_bets(W_, V_):
    # effective number of independent bets (Meucci): entropy of the risk shares along the principal components of the strategy returns, at the average weights
    w = W_.loc[TESTI].mean().to_numpy(); C = np.cov(V_.loc[TESTI].to_numpy().T) if V_.shape[1] > 1 else np.array([[V_.loc[TESTI].iloc[:, 0].var()]]); C = np.atleast_2d(C)
    lam, E = np.linalg.eigh(C); pk = (E.T @ w) ** 2 * np.maximum(lam, 0); pk = pk / pk.sum() if pk.sum() > 0 else np.ones(len(pk)) / len(pk)
    sd = np.sqrt(np.diag(C)); return float(np.exp(-np.sum(pk[pk > 0] * np.log(pk[pk > 0])))), float((w * sd).sum() / np.sqrt(max(w @ C @ w, 1e-18)))
rec_rows, REC_BOOK = [], {}
t0 = time.time()
for ln, Rl in LIBS.items():
    for rn, (ms, mt) in RULES.items():
        pool_A = pool_rule(Rl, ms, mt, VB[0]); pool_F = pool_rule(Rl, ms, mt, PRE_END)
        for sc in SCHEMES_R:
            row = dict(library=ln, rule=rn, scheme=sc, n_pool_A=len(pool_A), n_pool=len(pool_F))
            if len(pool_A) >= 2:
                bA, _, _ = walk_book(Rl, pool_A, sc); row["sharpe_B"] = window_stats(bA.loc[VB])["sharpe"]
            else:
                row["sharpe_B"] = np.nan
            if len(pool_F) >= 2:
                bF, WF, levF = walk_book(Rl, pool_F, sc); REC_BOOK[(ln, rn, sc)] = (bF, WF, levF, pool_F)
                vs_, ts_ = window_stats(bF.loc[VALI]), window_stats(bF.loc[TESTI]); VF = Rl[pool_F].fillna(0.0) * levF
                nb, dr = eff_bets(WF, VF)
                row.update(sharpe_val=vs_["sharpe"], sharpe_test=ts_["sharpe"], ret_test=ts_["ann_ret"], vol_test=ts_["ann_vol"], dd_test=ts_["max_dd"], es95_test=ts_["es95"],
                           eff_N=float((1 / (WF.loc[TESTI] ** 2).sum(axis=1)).mean()), eff_bets=nb, div_ratio=dr, turnover_per_rebal=float(WF.diff().abs().sum(axis=1).loc[WF.index[PORT_MIN_OBS::PORT_REBAL]].mean()))
            rec_rows.append(row)
REC = pd.DataFrame(rec_rows); REC.round(4).to_csv(OUT2 / "portfolio_recipes.csv", index=False)
valid = REC[REC.sharpe_B.notna() & REC.sharpe_test.notna()].copy()
if len(valid):
    sr_star_r = dsr_threshold((valid.sharpe_val / np.sqrt(ANN)).values)
    # deflated Sharpe of each recipe on test, for the number of recipes tried
    def _dsr(r):
        b = REC_BOOK[(r.library, r.rule, r.scheme)][0].loc[TESTI]; b = b[b != 0]
        return psr(r.sharpe_test / np.sqrt(ANN), len(b), b.skew(), b.kurt() + 3, sr_star_r) if len(b) > 30 and np.isfinite(r.sharpe_test) else np.nan
    valid["dsr_test"] = valid.apply(_dsr, axis=1)
    valid["profitable"] = (valid.sharpe_test > 0) & (valid.sharpe_B > 0)
    CHOSEN = valid.sort_values("sharpe_B", ascending=False).iloc[0]
    HIND = valid.sort_values("sharpe_test", ascending=False).iloc[0]
    print(f"{len(REC)} recipes ({len(valid)} with enough eligible strategies) in {time.time()-t0:.0f}s | DSR hurdle for {len(valid)} trials: annualised Sharpe {sr_star_r*np.sqrt(ANN):.2f}")
    print(f"test Sharpe > 0 in {int((valid.sharpe_test > 0).sum())} of {len(valid)} recipes; > 0.5 in {int((valid.sharpe_test > 0.5).sum())}; median {valid.sharpe_test.median():+.2f}")
    cols = ["library", "rule", "scheme", "n_pool", "sharpe_B", "sharpe_val", "sharpe_test", "ret_test", "vol_test", "dd_test", "eff_N", "eff_bets", "div_ratio", "dsr_test"]
    print("\nTop 10 by Sharpe on the selection window (second half of validation):")
    print(valid.sort_values("sharpe_B", ascending=False)[cols].head(10).round(2).to_string(index=False))
    print(f"\nCHOSEN without test: {CHOSEN.library} | {CHOSEN.rule} | {CHOSEN.scheme} → test Sharpe {CHOSEN.sharpe_test:+.2f}, return {CHOSEN.ret_test:+.1%}, vol {CHOSEN.vol_test:.1%}, max drawdown {CHOSEN.dd_test:.1%}, DSR {CHOSEN.dsr_test:.2f}")
    print(f"best on test (hindsight, not usable): {HIND.library} | {HIND.rule} | {HIND.scheme} → {HIND.sharpe_test:+.2f}")
    print("\nMedian test Sharpe by library / scheme (is the gain from costs, from weights, or from nothing?):")
    print(valid.pivot_table(index="scheme", columns="library", values="sharpe_test", aggfunc="median").round(2).to_string())
    PROFIT = valid[valid.profitable].sort_values("sharpe_test", ascending=False)
    print(f"\nProfitable on test AND on the selection window ({len(PROFIT)}):")
    print(PROFIT[cols].head(10).round(2).to_string(index=False) if len(PROFIT) else "none")
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    ax = axs[0]; ax.scatter(valid.sharpe_B, valid.sharpe_test, s=24, color=SERIES[0], edgecolor=INK, lw=0.3)
    ax.scatter([CHOSEN.sharpe_B], [CHOSEN.sharpe_test], s=90, color=SERIES[1], edgecolor=INK, label="chosen without test", zorder=5); ax.axhline(0, color=INK2, lw=0.8); ax.axvline(0, color=INK2, lw=0.8)
    ax.set_xlabel("Sharpe on the selection window (second half of validation)"); ax.set_ylabel("test Sharpe"); ax.legend(frameon=False, fontsize=8); ax.set_title("Does the selection window predict test?")
    ax = axs[1]; g_ = valid.groupby("scheme").sharpe_test.apply(list); ax.boxplot([g_[s] for s in g_.index]); ax.set_xticks(range(1, len(g_) + 1)); ax.set_xticklabels([s.split(" (")[0] for s in g_.index]); ax.axhline(0, color=INK2, lw=0.8); ax.set_ylabel("test Sharpe"); ax.set_title("Weighting scheme")
    ax = axs[2]; ax.scatter(valid.eff_bets, valid.sharpe_test, s=24, color=SERIES[2], edgecolor=INK, lw=0.3); ax.axhline(0, color=INK2, lw=0.8); ax.set_xlabel("effective number of bets (test)"); ax.set_ylabel("test Sharpe"); ax.set_title("Diversification vs outcome")
    plt.tight_layout(); savefig(fig, "24e_recipe_search", "Profitable portfolio search", "24")
else:
    CHOSEN = HIND = None; PROFIT = pd.DataFrame()
    print("no recipe has two or more eligible strategies: nothing to search; see 24e.1 for the cost lever")

## Step 24f — Netting, test-phase behaviour and what more can be done

**1. Netting.** The book trades the **sum** of its sleeves' positions, so buys in one sleeve cancel sells in another before costs. Cost of the netted book against the cost if each sleeve traded alone (execution model, test period).

**2. How the strategies behaved in the test phase.** Average correlation and effective independent bets in validation against test; where the Sharpe goes (gross → flat costs → execution model); sign changes caused by costs; the chosen book's beta to the market and the share of profit earned in its best ten days. The test period has been seen, so these are **hypotheses for new data**, never inputs to a rule.

**3. Levers.** A table of what could still raise return per unit of risk and diversification: what is already measured in this run, what is implemented, and how each hypothesis could be tested without peeking.

**Used later.** The levers table is the research to-do list of the findings (26b); the netted cost saving is a sizing note for the book.

In [ ]:
# step 24f
# ---- 24f.1 netting: the book trades the SUM of the sleeves' positions, so offsetting trades cancel before costs are paid ----
def netting_saving(key):
    bF, WF, levF, pool_F = REC_BOOK[key]
    src = BAND_W[TAU_STAR] if key[0].startswith("band") else {n: STRATS[n]["W"] for n in pool_F}
    A = (WF.shift(1) * levF).fillna(0.0)                                   # allocation x strategy-level leverage (decided the day before)
    parts = {n: src[n].reindex(RX.index).fillna(0.0).mul(A[n], axis=0) for n in pool_F}
    BWt = sum(parts.values()); b_net = backtest(BWt, MAIN_DELAY, cost_fn=cost_x, lock=LOCK)
    c_net = (b_net.gross - b_net.net); c_sep = sum((lambda b: b.gross - b.net)(backtest(p_, MAIN_DELAY, cost_fn=cost_x, lock=LOCK)) for p_ in parts.values())
    g_ = b_net.gross; r_net, r_sep = g_ - c_net, g_ - c_sep
    sh = lambda x: x.loc[TESTI].mean() / x.loc[TESTI].std() * np.sqrt(ANN)
    return dict(cost_separate=float(c_sep.loc[TESTI].mean() * ANN), cost_netted=float(c_net.loc[TESTI].mean() * ANN), saving=float(1 - c_net.loc[TESTI].sum() / c_sep.loc[TESTI].sum()),
                sharpe_separate=float(sh(r_sep)), sharpe_netted=float(sh(r_net)), gross_exposure=float(BWt.abs().sum(axis=1).loc[TESTI].mean()))
NETTING = None
if CHOSEN is not None:
    try:
        NETTING = netting_saving((CHOSEN.library, CHOSEN.rule, CHOSEN.scheme))
        print(f"book of the chosen recipe ({CHOSEN.library} | {CHOSEN.rule} | {CHOSEN.scheme}) on test, before the portfolio-level scaling:")
        print(f"  costs if every sleeve trades alone {NETTING['cost_separate']*100:.2f}%/yr → one netted book {NETTING['cost_netted']*100:.2f}%/yr  (saving {NETTING['saving']:.0%}); "
              f"Sharpe {NETTING['sharpe_separate']:+.2f} → {NETTING['sharpe_netted']:+.2f}; mean gross exposure {NETTING['gross_exposure']:.2f}x NAV")
    except Exception as e:
        print("netting skipped:", repr(e))
else:
    print("netting skipped: no recipe was chosen")

# ---- 24f.2 how did the strategies behave in the test phase? (descriptive: the test period is already seen, so these are hypotheses for NEW data, never inputs to a rule) ----
pool_d = pool_rule(RX, 0.0, 0.0, PRE_END, maxn=30)
if len(pool_d) < 5:                                                                           # too few eligible strategies to look at diversification: use the 12 best on validation, whatever their sign
    _v = RX.loc[VALI]; pool_d = (_v.mean() / _v.std()).sort_values(ascending=False).index[:12].tolist(); POOL_NOTE = "the 12 best on validation (fewer than 5 had a positive Sharpe after costs)"
else:
    POOL_NOTE = "strategies with a positive validation Sharpe under execution costs"
cv_, ct_ = RX.loc[VALI, pool_d].corr(), RX.loc[TESTI, pool_d].corr(); tri = np.triu_indices(len(pool_d), 1)
def _enb(C):
    lam = np.maximum(np.linalg.eigvalsh(C.fillna(0).to_numpy()), 1e-12); p = lam / lam.sum(); return float(np.exp(-np.sum(p * np.log(p))))
bk = (REC_BOOK[(CHOSEN.library, CHOSEN.rule, CHOSEN.scheme)][0] if CHOSEN is not None else BOOK_INV.get(("Growth", "1 year"), pd.Series(0.0, index=RX.index))).loc[TESTI]
mkt = RS.reindex(TESTI).mean(axis=1); ok_ = bk.notna() & mkt.notna()
beta_m = float(np.polyfit(mkt[ok_], bk[ok_], 1)[0]) if ok_.sum() > 30 and bk.std() > 0 else np.nan; r2_m = float(np.corrcoef(mkt[ok_], bk[ok_])[0, 1] ** 2) if ok_.sum() > 30 and bk.std() > 0 else np.nan
top10 = float(bk.sort_values(ascending=False).head(10).sum() / bk.sum()) if bk.sum() > 0 else np.nan
BEHAV = dict(n_pool=len(pool_d), corr_val=float(cv_.to_numpy()[tri].mean()), corr_test=float(ct_.to_numpy()[tri].mean()), enb_val=_enb(cv_), enb_test=_enb(ct_), beta_to_market=beta_m, r2_to_market=r2_m, top10_share=top10,
             median_gross=float(LED.sharpe_test_gross.median()), median_flat=float(LED.sharpe_test.median()), median_exec=float(LED.test_exec.median()),
             sign_flips=int(((LED.sharpe_test_gross > 0) & (LED.test_exec <= 0)).sum()))
print(f"\nTest-phase behaviour ({len(pool_d)} strategies: {POOL_NOTE}):")
print(f"  average pairwise correlation: validation {BEHAV['corr_val']:+.2f} → test {BEHAV['corr_test']:+.2f}; effective independent bets (eigenvalue entropy): {BEHAV['enb_val']:.1f} → {BEHAV['enb_test']:.1f} of {len(pool_d)}")
print(f"  median test Sharpe across all {len(LED)} strategies: gross {BEHAV['median_gross']:+.2f} → flat costs {BEHAV['median_flat']:+.2f} → execution model {BEHAV['median_exec']:+.2f}; "
      f"{BEHAV['sign_flips']} strategies are profitable gross and unprofitable after execution costs")
print(f"  chosen book on test: beta to the equal-weight market {beta_m:+.2f} (R² {r2_m:.2f}); " + (f"best 10 days = {top10:.0%} of total profit" if np.isfinite(top10) else "the book did not make a profit on test") if np.isfinite(beta_m) else "  chosen book: not enough data for beta")
sz = EXL[[c for c in EXL.columns if c.startswith("test_") and c.endswith("m")]].median()
print("  median test Sharpe by book size (CNY m): " + ", ".join(f"{k.replace('test_','')}: {v:+.2f}" for k, v in sz.items()))

# ---- 24f.3 levers: what could still raise return per unit of risk and diversification; what is measured here and what must wait for new data ----
LEV_ROWS = [
 ("Cut turnover: no-trade band", f"test Sharpe median {BAND.loc[0.0,'test_median']:+.2f} → {BAND.loc[TAU_STAR,'test_median']:+.2f} at band {TAU_STAR}; turnover {BAND.loc[0.0,'turnover']:.0f} → {BAND.loc[TAU_STAR,'turnover']:.0f}×/yr", "implemented (24e.1)", "one parameter chosen on validation"),
 ("Trade one netted book, not separate sleeves", (f"costs {NETTING['cost_separate']*100:.2f}% → {NETTING['cost_netted']*100:.2f}%/yr ({NETTING['saving']:.0%} saved)" if NETTING else "not measured"), "implemented (24f.1)", "pure accounting identity, no fitting"),
 ("Hold fewer, cheaper strategies", f"median Sharpe falls from gross {BEHAV['median_gross']:+.2f} to {BEHAV['median_exec']:+.2f}; {BEHAV['sign_flips']} strategies change sign", "implemented (24c ledger)", "rank by Sharpe after execution costs on validation"),
 ("More independent bets", f"effective bets {BEHAV['enb_val']:.1f} → {BEHAV['enb_test']:.1f} of {BEHAV['n_pool']}; correlation {BEHAV['corr_val']:+.2f} → {BEHAV['corr_test']:+.2f}", "partly (24e.2 eff_bets)", "add sleeves with low correlation to the book: overnight (T+1), term-structure carry slope, Step 16b chains once valid"),
 ("Small size where capacity binds", "test Sharpe by book size above; capacity in Step 24", "measured (23b)", "size each sleeve at the smaller of its target and 5% of ADV"),
 ("Evaluate rule-based sleeves on train + validation", "sleeves with no fitted parameters could use the whole pre-test history as evidence (3x the sample)", "hypothesis", "needs a rule-by-rule list of what was fitted; re-run on a new period"),
 ("Strategy-level regime gating", "see Step 22 / G1, G2 and strategy_sharpe_by_regime.csv", "tested (22): not adopted", "only if the gate is fixed before a new period"),
]
LEVERS = pd.DataFrame(LEV_ROWS, columns=["lever", "evidence in this run", "status", "how to test it without peeking"]); LEVERS.to_csv(OUT2 / "improvement_levers.csv", index=False)
pd.set_option("display.max_colwidth", 120); print("\nWhat more can be done:\n" + LEVERS.to_string(index=False))
fig, axs = plt.subplots(1, 3, figsize=(18, 4.8))
ax = axs[0]; im = heat(ax, cv_, cmap=DIV, center=0, label="correlation, validation", annot=False); ax.set_title("Strategy correlation: validation")
ax = axs[1]; heat(ax, ct_, cmap=DIV, center=0, label="correlation, test", annot=False); ax.set_title(f"Strategy correlation: test (effective bets {BEHAV['enb_val']:.1f} → {BEHAV['enb_test']:.1f})")
ax = axs[2]; xs_ = ["gross", "flat costs", f"execution {EXEC_AUM_MAIN/1e6:.0f}m"]; vs_ = [BEHAV["median_gross"], BEHAV["median_flat"], BEHAV["median_exec"]]
ax.bar(xs_, vs_, color=[MUTE, SERIES[1], SERIES[0]]); ax.axhline(0, color=INK2, lw=0.8); ax.set_ylabel("median test Sharpe"); ax.set_title("Where the Sharpe goes")
plt.tight_layout(); savefig(fig, "24f_levers", "Test-phase behaviour and levers", "24")
if CHOSEN is not None:
    PORT_BOOKS["chosen recipe (24e): " + f"{CHOSEN.scheme.split(' (')[0]}"] = REC_BOOK[(CHOSEN.library, CHOSEN.rule, CHOSEN.scheme)][0]

## Step 25 — Risk model: VaR, ES and the Rachev ratio, with backtests

Daily risk of a **long** position (loss = −return, in %). All parameters are fitted on train. The method choice uses validation. Test is the backtest.

| Method | VaR / ES from |
|---|---|
| `HS-250` | rolling 250-day historical simulation (unconditional) |
| `RiskMetrics` | EWMA σ × normal quantile |
| `GARCH-n` | GARCH(1,1) σ × normal quantile |
| `BestVol-normal` | validation-selected σ (Step 11) × normal quantile |
| `BestVol-winner` | that σ × quantile of the Step 12 winning distribution |
| `FHS` | that σ × empirical quantile of train residuals (filtered historical simulation) |
| `EVT` | that σ × GPD tail quantile (McNeil–Frey conditional EVT, Step 13) |
| `Winner→FHS` | winner where it passed the **validation** KS test, FHS where it failed. This is the "FHS where the winner failed out of sample" rule, decided on validation so test is never peeked at |

**Backtests on test (and validation):**
* **Kupiec POF** (right number of exceedances).
* **Christoffersen** independence and conditional coverage (no clustering of exceedances).
* **Basel traffic light** (99% exceptions per 250 days: green ≤ 4, yellow 5–9, red ≥ 10).
* **Acerbi–Szekely Z₂** for ES at 97.5%: 0 is ideal; > 0.7 means a yellow-zone ES underestimate, > 1.8 red.
* **McNeil–Frey**: exceedance residuals should average zero.

**Rachev ratio** RR = E[gain | gain in top 5%] / E[loss | loss in bottom 5%]. It compares the reward tail with the risk tail (RR > 1 means fatter upside). It is computed empirically on train and test, and implied by each series' winning distribution.

**Used later.** The chosen VaR / ES model gives the position-risk limits quoted with every board entry.

In [ ]:
# step 25
METHODS = ["HS-250", "RiskMetrics", "GARCH-n", "BestVol-normal", "BestVol-winner", "FHS", "EVT", "Winner→FHS"]
LEVELS = sorted(set(VAR_LEVELS) | {ES_LEVEL})
risk_rows, VARS = [], {}
t0 = time.time()
for s in UNIV_V:
    r = RAW[s].dropna() * 100; idx_eval = r.index.intersection(SPL["val"].append(SPL["test"]))
    if len(idx_eval) < 50:
        continue
    sig_b = SIGMA[s].reindex(r.index); mu_b = MU[s]
    sig_ewma = np.sqrt(VOLFIT[s]["EWMA"]["h"]).reindex(r.index)
    gn = VOLFIT[s].get("GARCH-n", VOLFIT[s]["EWMA"]); sig_gn = np.sqrt(gn["h"]).reindex(r.index); mu_gn = gn["mu"]
    ztr = RES["train"][s].dropna().to_numpy(); f_w = WINS.loc[s, "winner"]; par_w = PARAMS[(s, f_w)]
    use_winner = WINS.loc[s, "ks_val_p"] > 0.05
    rv_arr = r.to_numpy(); pos_eval = np.flatnonzero(r.index.isin(idx_eval))
    for a in LEVELS:
        za = stats.norm.ppf(a); es_n = stats.norm.pdf(za) / (1 - a)
        q_f = np.quantile(-ztr, a); es_f = np.mean(-ztr[-ztr >= q_f])
        v_w, e_w = dist_tail(f_w, par_w, a)
        u_, xi, beta, nn, nu = EVT_PAR[(s, "loss")]; v_e, e_e = gpd_var_es(u_, xi, beta, nn, nu, a) if a > EVT_Q else (q_f, es_f)
        hs_v = np.full(len(r), np.nan); hs_e = np.full(len(r), np.nan); hs_s = np.full(len(r), np.nan)
        for t in pos_eval:
            w_ = -rv_arr[max(0, t - HS_WIN):t]
            if len(w_) >= 100:
                hs_v[t] = np.quantile(w_, a); hs_e[t] = w_[w_ >= hs_v[t]].mean(); hs_s[t] = w_.std()
        V = {"HS-250": (hs_v, hs_e, hs_s),
             "RiskMetrics": (sig_ewma * za, sig_ewma * es_n, sig_ewma),
             "GARCH-n": (-mu_gn + sig_gn * za, -mu_gn + sig_gn * es_n, sig_gn),
             "BestVol-normal": (-mu_b + sig_b * za, -mu_b + sig_b * es_n, sig_b),
             "BestVol-winner": (-mu_b + sig_b * v_w, -mu_b + sig_b * e_w, sig_b),
             "FHS": (-mu_b + sig_b * q_f, -mu_b + sig_b * es_f, sig_b),
             "EVT": (-mu_b + sig_b * v_e, -mu_b + sig_b * e_e, sig_b)}
        V["Winner→FHS"] = V["BestVol-winner"] if use_winner else V["FHS"]
        for mth, (vv, ee, ss) in V.items():
            vv, ee, ss = (pd.Series(np.asarray(x_, float), r.index) for x_ in (vv, ee, ss))
            VARS[(s, mth, a)] = vv
            for sp in ("val", "test"):
                idx = r.index.intersection(SPL[sp]); L = -r.reindex(idx); v_ = vv.reindex(idx); e_ = ee.reindex(idx)
                ok = v_.notna() & e_.notna(); L, v_, e_, s_ = L[ok], v_[ok], e_[ok], ss.reindex(idx)[ok]
                if len(L) < 50:
                    continue
                I = (L > v_).to_numpy(); n_exc = int(I.sum()); ind_p, cc_p = christoffersen(I, 1 - a)
                row = dict(series=s, method=mth, split=sp, level=a, n=len(L), n_exc=n_exc, exc_rate=I.mean(),
                           kupiec_p=kupiec(I, 1 - a), ind_p=ind_p, cc_p=cc_p)
                if a == 0.99:
                    row["basel"] = basel_zone(n_exc, len(L))
                if a == ES_LEVEL:
                    row["Z2"] = np.sum(L.values * I / e_.values) / (len(L) * (1 - a)) - 1
                    ex = ((L - e_) / s_)[I]
                    row["mf_p"] = stats.ttest_1samp(ex, 0, alternative="greater").pvalue if len(ex) >= 5 else np.nan
                risk_rows.append(row)
RISK = pd.DataFrame(risk_rows); RISK.to_csv(OUT2 / "var_backtest.csv", index=False)
print(f"VaR/ES backtests in {time.time()-t0:.0f}s")

te = RISK[RISK.split == "test"]
summary_risk = pd.DataFrame({
    "exc_rate_95_med": te[te.level == 0.95].groupby("method").exc_rate.median(),
    "kupiec95_pass": te[te.level == 0.95].groupby("method").kupiec_p.apply(lambda v: (v > 0.05).mean()),
    "cc95_pass": te[te.level == 0.95].groupby("method").cc_p.apply(lambda v: (v > 0.05).mean()),
    "exc_rate_99_med": te[te.level == 0.99].groupby("method").exc_rate.median(),
    "kupiec99_pass": te[te.level == 0.99].groupby("method").kupiec_p.apply(lambda v: (v > 0.05).mean()),
    "cc99_pass": te[te.level == 0.99].groupby("method").cc_p.apply(lambda v: (v > 0.05).mean()),
    "basel_green": te[te.level == 0.99].groupby("method").basel.apply(lambda v: (v == "green").mean()),
    "Z2_med": te[te.level == ES_LEVEL].groupby("method").Z2.median(),
    "Z2_ok": te[te.level == ES_LEVEL].groupby("method").Z2.apply(lambda v: (v < 0.7).mean()),
    "mf_pass": te[te.level == ES_LEVEL].groupby("method").mf_p.apply(lambda v: (v > 0.05).mean()),
}).reindex(METHODS)
print("\nTest-period backtest summary (shares = fraction of series passing; nominal exceedance 5% and 1%):")
print(summary_risk.round(3).to_string())

rach = []
for s in UNIV_V:
    f_w = WINS.loc[s, "winner"]; par_w = PARAMS[(s, f_w)]
    rach.append(dict(series=s, rachev_train=rachev(RAW.loc[SPL["train"], s]), rachev_test=rachev(RAW.loc[SPL["test"], s]),
                     rachev_model=dist_gain_tail_mean(f_w, par_w, RACHEV_A) / dist_tail(f_w, par_w, 1 - RACHEV_A)[1],
                     rachev_resid_test=rachev(RES["test"][s])))
RACH = pd.DataFrame(rach).set_index("series"); RACH.to_csv(OUT2 / "rachev.csv")
print("\nRachev ratio (5%/5%): median train", round(RACH.rachev_train.median(), 3), "| test", round(RACH.rachev_test.median(), 3),
      "| model-implied", round(RACH.rachev_model.median(), 3))
print(f"corr(model-implied, test) = {RACH[['rachev_model', 'rachev_resid_test']].corr().iloc[0, 1]:.2f}; "
      f"corr(train, test) = {RACH[['rachev_train', 'rachev_test']].corr().iloc[0, 1]:.2f}")

In [ ]:
# step 25
s0 = "RB" if "RB" in UNIV_V else UNIV_V[0]
idx = RAW[s0].dropna().index.intersection(SPL["test"])
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(idx, RAW.loc[idx, s0] * 100, color=MUTE, width=1.0, label="daily return (%)")
for mth, col in [("HS-250", SERIES[3]), ("RiskMetrics", SERIES[1]), ("EVT", SERIES[0]), ("Winner→FHS", SERIES[2])]:
    v = VARS.get((s0, mth, 0.99))
    if v is not None:
        ax.plot(idx, -v.reindex(idx), color=col, lw=1.2, label=f"−VaR99 {mth}")
v = VARS[(s0, "EVT", 0.99)].reindex(idx); exc = RAW.loc[idx, s0] * 100 < -v
ax.scatter(idx[exc], RAW.loc[idx, s0][exc] * 100, color=SERIES[7], s=18, zorder=3, label="EVT exceedance")
ax.legend(frameon=False, fontsize=7, ncol=6); ax.set_ylabel("%"); ax.set_title(f"{s0}: 99% VaR on the test period")
savefig(fig, "25_var_band", "VaR band, showcase series", "25")

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, a in zip(axs[:2], [0.95, 0.99]):
    d_ = te[te.level == a]
    for k, mth in enumerate(METHODS):
        v = d_[d_.method == mth].exc_rate * 100
        ax.scatter(v, np.full(len(v), k) + rng.uniform(-0.15, 0.15, len(v)), s=10, color=SERIES[0], alpha=0.6)
        ax.scatter([v.median()], [k], s=60, color=INK, marker="|")
    ax.axvline((1 - a) * 100, color=SERIES[1], lw=1.2, ls="--")
    ax.set_yticks(range(len(METHODS))); ax.set_yticklabels(METHODS); ax.invert_yaxis()
    ax.set_xlabel("test exceedance rate (%)"); ax.set_title(f"{int(a*100)}% VaR: exceedances (dashed = nominal)")
heat(axs[2], summary_risk[["kupiec95_pass", "cc95_pass", "kupiec99_pass", "cc99_pass", "basel_green", "Z2_ok", "mf_pass"]],
     vmin=0, vmax=1, label="share of series passing", annot=True)
axs[2].set_title("Backtest pass rates (test)")
plt.tight_layout()
savefig(fig, "25_var_backtests", "VaR/ES backtests", "25")

fig, axs = plt.subplots(1, 2, figsize=(12, 4.3))
for ax, (xc, yc, xl) in zip(axs, [("rachev_train", "rachev_test", "train (empirical)"), ("rachev_model", "rachev_resid_test", "implied by winning distribution")]):
    ax.scatter(RACH[xc], RACH[yc], color=SERIES[0], s=28, edgecolors="#fcfcfb", linewidths=1)
    for s_, r_ in RACH.iterrows():
        ax.annotate(s_, (r_[xc], r_[yc]), fontsize=6, color=INK2, xytext=(2, 2), textcoords="offset points")
    lo_, hi_ = np.nanmin(RACH[[xc, yc]].values) * 0.95, np.nanmax(RACH[[xc, yc]].values) * 1.05
    ax.plot([lo_, hi_], [lo_, hi_], color=INK2, lw=1, ls="--"); ax.axhline(1, color=MUTE, lw=1); ax.axvline(1, color=MUTE, lw=1)
    ax.set_xlabel(f"Rachev ratio, {xl}"); ax.set_ylabel("Rachev ratio, test")
axs[0].set_title("Is the tail asymmetry persistent?"); axs[1].set_title("Model-implied vs realised (residuals)")
plt.tight_layout()
savefig(fig, "25_rachev", "Rachev ratios", "25")

---
# Part H — The opportunity board (Step 26)
## Step 26 — The live opportunity board and the report

For every product on the latest date, everything above is combined into one board:

| Column | From |
|---|---|
| UpR / DnR per time frame, composite, MTF state | Step 19 |
| factor score (percentile of the validated multi-factor) | Step 18 |
| relationship bull / bear score (Granger, pairs, fundamental chains, sector lead, spillover signals live now), health-adjusted | Steps 16–17, 22 |
| dynamics tilt (factor / sector / factor × sector rotation, monitor-gated books) | Steps 21–22 |
| tail risk: EVT 99% VaR tomorrow (%) | Steps 11, 13 |
| regime | Step 20 |

$$\text{Opportunity} = 100 \times \big(0.45\,\tfrac{\text{Net}}{100} + 0.20\,(2\,\text{factor pct} - 1) + 0.20\,\tanh(\text{rel bull} - \text{rel bear}) + 0.15\,\text{dyn tilt}\big)$$

The relationship term counts a relationship at full weight only while the Step 22 monitor says it works (weakening edges count half, broken edges count zero). The **dynamics tilt** is the average sign of today's positions in the factor-rotation, sector-rotation, factor × sector rotation and monitor-gated books (Steps 21–22).

* **BULL** if > 20, **BEAR** if < −20.
* The **vehicle** suggests how to express the view:
  * *outright* when the MTF state agrees;
  * *within-sector pair* when a sector peer has the opposite signal;
  * *sector* when the whole sector leans the same way;
  * *relationship* when the signal comes mainly from Step 17.
* The **strategy mix** for the current regime is taken from Step 23's per-regime picks.

**Used later.** This is the final output: the board, `research_summary.json` and `report.html`.

In [ ]:
# step 26
last = RAW.index[-1]
BOARD = pd.DataFrame(index=PRODS)
BOARD["sector"] = SEC_MAP
for tf in TFS:
    if tf in LATEST:
        BOARD[f"{tf}_Net"] = (LATEST[tf].UpR + LATEST[tf].DnR).reindex(PRODS)
BOARD["UpR"], BOARD["DnR"] = COMP_UP.ffill().iloc[-1], COMP_DN.ffill().iloc[-1]
BOARD["Net"] = BOARD.UpR + BOARD.DnR
BOARD["MTF_state"] = MTF_STATE.ffill().iloc[-1].map({1: "BULL", -1: "BEAR", 0: "NEUTRAL"})
BOARD["factor_pct"] = FACTOR_NOW.reindex(PRODS)
# relationships are scaled by their health (Step 22): BROKEN edges count 0, WEAKENING edges count half
_h = REL_OPP.mkey.map(MON_NOW.status_now).map({"BROKEN": 0.0, "WEAKENING": 0.5}).fillna(1.0) if len(REL_OPP) else pd.Series(dtype=float)
REL_H = REL_OPP.assign(opp=REL_OPP.opp * _h) if len(REL_OPP) else REL_OPP
if len(REL_H):
    REL_H = REL_H[~REL_H.via.str.contains(r"granger_(?:5m|15m)")]
BOARD["rel_bull"] = (REL_H[REL_H.side == "BULL"].groupby("product").opp.sum() if len(REL_H) else pd.Series(dtype=float)).reindex(PRODS).fillna(0) / 100
BOARD["rel_bear"] = (REL_H[REL_H.side == "BEAR"].groupby("product").opp.sum() if len(REL_H) else pd.Series(dtype=float)).reindex(PRODS).fillna(0) / 100
# dynamics tilt (Steps 21-22): average sign of today's positions in the rotation books and the monitor-gated books
TILT = {p_: np.sign(NEW_STRATS[k]["W"].iloc[-1]) for p_ in ("F1", "F2", "F3", "G1", "G2") for k in NEW_STRATS if k.startswith(p_)}
BOARD["dyn_tilt"] = pd.DataFrame(TILT).mean(axis=1).reindex(PRODS).fillna(0)
BOARD["vol_state"] = VOLSTATE_NOW.state.reindex(PRODS); BOARD["vol_ratio"] = VOLSTATE_NOW.ratio.reindex(PRODS)        # Step 11c: expansion / contraction
if "carry_roll|XS" in FEATS:
    BOARD["carry_pct"] = (FEATS["carry_roll|XS"].ffill().iloc[-1] + 0.5).reindex(PRODS)
u_l = {s: EVT_PAR.get((s, "loss")) for s in UNIV_V}
BOARD["EVT_VaR99_%"] = pd.Series({s: -MU[s] + SIGMA[s].dropna().iloc[-1] * gpd_var_es(*u_l[s], 0.99)[0] for s in UNIV_V if u_l[s]})
BOARD["opportunity"] = 100 * (0.45 * BOARD.Net.fillna(0) / 100 + 0.20 * (2 * BOARD.factor_pct.fillna(0.5) - 1)
                              + 0.20 * np.tanh(BOARD.rel_bull - BOARD.rel_bear) + 0.15 * BOARD.dyn_tilt)
BOARD["side"] = np.select([BOARD.opportunity > 20, BOARD.opportunity < -20], ["BULL", "BEAR"], "—")
sec_mean = BOARD.groupby("sector").opportunity.mean()
def vehicle(p, r):
    if r.side == "—":
        return ""
    peers = BOARD[(BOARD.sector == r.sector) & (BOARD.index != p)]
    opp_peer = peers[(np.sign(peers.opportunity) == -np.sign(r.opportunity)) & (peers.opportunity.abs() > 20)]
    if len(opp_peer):
        return f"pair vs {opp_peer.opportunity.abs().idxmax()}"
    if abs(sec_mean.get(r.sector, 0)) > 20 and np.sign(sec_mean[r.sector]) == np.sign(r.opportunity):
        return "sector"
    if abs(np.tanh(r.rel_bull - r.rel_bear)) > abs(r.Net) / 100:
        return "relationship"
    return "outright" if r.MTF_state in ("BULL", "BEAR") else "outright (MTF not aligned)"
BOARD["vehicle"] = [vehicle(p, r) for p, r in BOARD.iterrows()]
BOARD = BOARD.sort_values("opportunity", ascending=False); BOARD.to_csv(OUT2 / "opportunity_board_latest.csv")
REGIME_NOW = REG.ffill().iloc[-1].to_dict()
MIX_NOW = pick.get(REGIME_NOW.get("vol"), SELECTED)
print(f"Opportunity board for {last.date()} | regime now: {REGIME_NOW} | strategy mix for this regime: {MIX_NOW}")
print(BOARD.round(2).to_string())
fig, axs = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={"width_ratios": [1.6, 1]})
ax = axs[0]; bb = pd.concat([BOARD.head(12), BOARD.tail(12)]).drop_duplicates()
ax.barh(bb.index[::-1], bb.opportunity[::-1], color=np.where(bb.opportunity[::-1] > 0, SERIES[0], SERIES[1]), height=0.65)
for i, (p, r) in enumerate(bb[::-1].iterrows()):
    ax.text(r.opportunity + (2 if r.opportunity >= 0 else -2), i, r.vehicle, va="center", ha="left" if r.opportunity >= 0 else "right", fontsize=6, color=INK2)
ax.axvline(0, color=INK2, lw=1); ax.axvline(20, color=MUTE, ls="--", lw=1); ax.axvline(-20, color=MUTE, ls="--", lw=1)
lim_ = max(bb.opportunity.abs().max(), 25) * 1.9; ax.set_xlim(-lim_, lim_)
ax.tick_params(axis="y", labelsize=7); ax.set_xlabel("opportunity (−100 bear … +100 bull)")
ax.set_title(f"Opportunity board — {last.date()} (labels: suggested vehicle)")
ax = axs[1]; sm = sec_mean.sort_values()
ax.barh(sm.index, sm.values, color=np.where(sm.values > 0, SERIES[0], SERIES[1]), height=0.6); ax.axvline(0, color=INK2, lw=1)
ax.set_title("Sector board (mean opportunity)")
plt.tight_layout()
savefig(fig, "26_board", "Opportunity board", "26")

In [ ]:
# step 26
def img(path):
    return base64.b64encode(Path(path).read_bytes()).decode()
def safe(fn, default=None):
    try:
        return fn()
    except Exception:
        return default
summary = {
    "as_of": str(last.date()), "regime_now": REGIME_NOW, "strategy_mix_now": MIX_NOW,
    "splits": {k: split_info[k] for k in SPL}, "universe": len(UNIV),
    "volatility": safe(lambda: {"robust_model_all_regimes_and_timeframes": VOL_UNIVERSAL, "best_by_timeframe": VOL_TF_BEST.to_dict(), "best_per_series_robust": BEST_VOL.value_counts().to_dict(),
                                "walk_forward_vs_static_test_qlike": WFS["test"].round(4).to_dict() if "test" in WFS else None,
                                "expansion_now": VOLSTATE_NOW[VOLSTATE_NOW.state == "EXPANSION"].index.tolist(), "squeeze_now": VOLSTATE_NOW[VOLSTATE_NOW.state == "SQUEEZE"].index.tolist()}),
    "multivariate": safe(lambda: {**STRUCT, "fama_macbeth_same_sign": FMT.index[FMT.same_sign_all].tolist(), "sector_links_replicating": int(SECG.replicates.sum()) if len(SECG) else 0,
                                  "tail_spillover_windows_replicating": TSW.groupby("window").replicates.sum().to_dict() if len(TSW) else {}}),
    "fundamental_16b": safe(lambda: {"status": FUND2.status.value_counts().to_dict(), "cases": FUND2.case.value_counts().to_dict(), "pooled": FUND_POOL.round(3).to_dict(orient="records")}),
    "copulas": safe(lambda: COPULA_SUMMARY), "frontier": safe(lambda: {**FRONT_SUMMARY, "portfolio_copula": PORT_COPULA_SUMMARY}),
    "overnight": safe(lambda: {"t1": OVN_SUMMARY, "strategy": OVN_STRAT_SUMMARY}),
    "overnight_regime": safe(lambda: {"har_on": HARON_SUMMARY, "regime": REGON_SUMMARY, "opening_reversal": OR_STRAT_SUMMARY, "high_vol_entry": HVE_SUMMARY}),
    "residual_diagnostics": safe(lambda: DIAG11E_SUMMARY),
    "price_volume_geometry": safe(lambda: PV_SUMMARY),
    "price_volume_topology": safe(lambda: TD_SUMMARY),
    "volume_oi_price": safe(lambda: OV_SUMMARY),
    "trend_phases_oi": safe(lambda: OF_SUMMARY),
    "quantile_flow": safe(lambda: QG_SUMMARY),
    "flow_indicator_overlap": safe(lambda: PW_SUMMARY),
    "echo_field_momentum": safe(lambda: EC_SUMMARY),
    "candle_anatomy": safe(lambda: KC_SUMMARY),
    "regimes_overnight": safe(lambda: RGS_SUMMARY),
    "demeaned_retest": safe(lambda: RDG_SUMMARY),
    "threshold_atlas": safe(lambda: KT_SUMMARY),
    "layered_close_location": safe(lambda: KL_SUMMARY),
    "framework_setups": safe(lambda: KN_SUMMARY),
    "stage2_vcp": safe(lambda: KO_SUMMARY),
    "incremental_volume": safe(lambda: KP_SUMMARY),
    "instrument_rules": safe(lambda: KQ_SUMMARY),
    "direction_horizon": safe(lambda: KR_SUMMARY),
    "sector_regimes": safe(lambda: KS_SUMMARY),
    "stage2_robustness": safe(lambda: KB_SUMMARY),
    "rsi_regime_entries": safe(lambda: KU_SUMMARY),
    "rsi_inside_stage2": safe(lambda: KV_SUMMARY),
    "data_quality": safe(lambda: DQ_LEDGER[["issue", "applied", "rows_affected", "products"]].to_dict(orient="records")),
    "factor_groups": safe(lambda: {"groups_tested": len(GR), "agree_out_of_sample": int(GR.agree.sum()), "best": None if COMBO_BEST is None else {"level": COMBO_BEST[0], "families": list(COMBO_BEST[1])}}),
    "granger_all_links": safe(lambda: REL14.verdict.value_counts().to_dict() if len(REL14) else {}),
    "granger_by_tf": safe(lambda: GSUM[["train_sig", "candidates", "replicated"]].to_dict(orient="index")),
    "relationships": safe(lambda: {"n": len(REL), "by_type": REL.type.value_counts().to_dict(), "top5": REL.head(5)[["type", "a", "b", "score"]].round(1).to_dict(orient="records")}),
    "factors": safe(lambda: {"purged_cv_auc": float(np.mean(cv_auc)), "oos_auc": oos_auc, "top5": FSCORE.head(5).index.tolist()}),
    "ranks_test_ic": safe(lambda: {f"{t}|{v}": round(float(x), 4) for (t, v), x in REVAL[REVAL.split == "test"].set_index(["tf", "variant"]).ic.items()}),
    "backtest": safe(lambda: {"n": N_TRIALS, "selected": SELECTED, "pbo": PBO, "dsr_hurdle": float(sr_star * np.sqrt(ANN)),
                              "books_test_sharpe": BOOK_PERF.sharpe.round(3).to_dict()}),
    "fundamental": safe(lambda: {"chains": len(FUND), "valid": int(FUND.valid.sum()), "test_ok": int(FUND.test_ok.sum()), "signal_test_ok": int(FUND.signal_test_ok.sum())}),
    "dynamics": safe(lambda: {"summary": DYN_SUM[["series", "momentum_train", "reversal_train", "test_ok", "replication_val_test"]].round(3).to_dict(orient="index"),
                              "chosen_rotation": CHOSEN, "factor_sector_map_train_test_rho": round(float(rho_tt), 3)}),
    "monitor": safe(lambda: {"edges": len(MON_NOW), "status_now": MON_NOW.status_now.value_counts().to_dict()}),
    "portfolio": safe(lambda: {"best_scheme_by_val": BEST, "test_sharpe": PORT[(PORT.variant == "+ portfolio-level target") & (PORT.split == "test")].set_index("scheme").sharpe.round(3).to_dict(),
                               "capacity_cny_bn": {k: (round(v / 1e8, 2) if isinstance(v, float) else v) for k, v in CAPACITY.items()}}),
    "execution": safe(lambda: {"half_spread_bp_median": round(float(EXEC.half_spread_bp.median()), 2), "flat_slippage_bp_median": round(float(EXEC.flat_slippage_bp.median()), 2),
                               "locked_days_pct": round(float(LOCK.values.mean() * 100), 3), "dropped_by_exec_model": drop, "median_test_sharpe": {"flat": round(float(EXL.flat_test.median()), 3),
                               **{f"{a/1e6:.0f}m": round(float(EXL[f"test_{a/1e6:.0f}m"].median()), 3) for a in EXEC_AUM}}}),
    "cpcv": safe(lambda: CPCV_SUM.round(3).to_dict("records")),
    "carry_slope_events": safe(lambda: int(SLOPE.notna().sum().sum()) if SLOPE is not None else 0),
    "top_bull": BOARD[BOARD.side == "BULL"].index[:10].tolist(), "top_bear": BOARD[BOARD.side == "BEAR"].index[::-1][:10].tolist(),
}
json.dump(summary, open(OUT2 / "research_summary.json", "w"), indent=2, default=str)
parts = {"13c": "Step 13c — Copulas", "13d": "Step 13d — Overnight and T+1", "23c": "Step 23c — Overnight strategy", "24b": "Step 24b — Efficient frontier and portfolio copula", "24c": "Step 24c — Selection audit and keep / drop ledger", "24d": "Step 24d — Investor profiles", "24e": "Step 24e — Profitable portfolio search", "24f": "Step 24f — Netting, behaviour and levers", "9": "Step 9 — Data, splits, stationarity, fractional differentiation", "10": "Step 10 — Mean models", "11": "Step 11 — Volatility models, robustness, expansion / contraction",
         "12": "Step 12 — Distributions", "13": "Step 13 — Extreme value theory, beyond-days links, multivariate structure", "14": "Step 14 — Multi-time-frame Granger and survival",
         "15": "Step 15 — Cointegration, correlation, tail dependence", "17": "Step 17 — Relationship library", "18": "Step 18 — Factor library",
         "19": "Step 19 — Multi-time-frame bull / bear ranks", "20": "Step 20 — Regimes", "23": "Step 23 — Execution model, strategies, backtests, CPCV",
         "16": "Step 16 — Fundamental-linked relationships", "21": "Step 21 — Factor, sector and factor × sector dynamics", "22": "Step 22 — Drift and decay monitor", "24": "Step 24 — Portfolio construction", "25": "Step 25 — Risk model", "26": "Step 26 — Opportunity board"}
def board_html(df):
    def cell(v, col):
        if isinstance(v, (float, np.floating)) and np.isfinite(v) and (col.endswith("Net") or col in ("UpR", "DnR", "opportunity")):
            a = min(abs(v) / 100, 1); c = f"rgba(42,120,214,{a:.2f})" if v > 0 else f"rgba(235,104,52,{a:.2f})"
            return f"<td style='background:{c}'>{v:.0f}</td>"
        return f"<td>{v if not isinstance(v, float) else round(v, 2)}</td>"
    head = "<tr><th>product</th>" + "".join(f"<th>{c}</th>" for c in df.columns) + "</tr>"
    body = "".join("<tr><td><b>" + str(i) + "</b></td>" + "".join(cell(v, c) for c, v in r.items()) + "</tr>" for i, r in df.iterrows())
    return f"<table class='t'>{head}{body}</table>"
tables = {"16": [("Fundamental chains", safe(lambda: FUND.drop(columns=["drivers"], errors="ignore"))), ("Chains now", safe(lambda: FUND_NOW))],
          "21": [("Dynamics summary", safe(lambda: DYN_SUM)), ("Rotation grid", safe(lambda: ROT))], "22": [("Monitor — latest status", safe(lambda: MON_NOW)), ("Gating test", safe(lambda: GT))],
          "24": [("Allocation schemes", safe(lambda: PORT)), ("Cost stress", safe(lambda: CSTRESS))],
          "14": [("Granger by time frame", safe(lambda: GSUM)), ("Survival", safe(lambda: SURV))],
          "17": [("Relationship library (top 40)", safe(lambda: REL.drop(columns=["legs"], errors="ignore").head(40))), ("Opportunities now", safe(lambda: REL_OPP))],
          "18": [("Factor scorecard (top 30)", safe(lambda: FSCORE.head(30)))], "19": [("Rank validation", safe(lambda: REVAL))],
          "20": [("Rule-change event study", safe(lambda: EVSTUDY))], "23": [("Backtest summary", safe(lambda: BT_SUMMARY)), ("Books (test)", safe(lambda: BOOK_PERF)), ("Execution model by product", safe(lambda: EXEC)),
                 ("Library under the execution model", safe(lambda: EXL[EXL.selected_flat | EXL.selected_exec])), ("CPCV", safe(lambda: CPCV_SUM))],
          "25": [("VaR / ES backtest pass rates", safe(lambda: summary_risk))]}
css = ("body{font-family:system-ui,sans-serif;max-width:1200px;margin:24px auto;padding:0 16px;color:#0b0b0b;background:#f9f9f7}"
       "h2{margin-top:36px;border-bottom:1px solid #e4e3df;padding-bottom:4px}img{max-width:100%;background:#fcfcfb;border:1px solid #e4e3df;border-radius:6px;margin:8px 0}"
       ".t{border-collapse:collapse;font-size:11px;display:block;overflow-x:auto}.t td,.t th{padding:3px 7px;border-bottom:1px solid #eee;text-align:right}"
       "pre{background:#fcfcfb;border:1px solid #e4e3df;padding:10px;font-size:11px;white-space:pre-wrap}")
html = [f"<html><head><meta charset='utf-8'><title>MFT China research report</title><style>{css}</style></head><body>",
        f"<h1>MFT China futures — research report ({last.date()})</h1>", "<h2>Opportunity board</h2>", board_html(BOARD.drop(columns=["sector"]).round(2)),
        f"<pre>{json.dumps(summary, indent=2, default=str)}</pre>"]
for p in list(parts) + sorted({pp for pp, _, _ in FIGS} - set(parts), key=lambda x: int(x) if x.isdigit() else 99):
    html.append(f"<h2>{parts.get(p, 'Step ' + p)}</h2>")
    for pp, title, path in FIGS:
        if pp == p:
            html.append(f"<h3>{title}</h3><img src='data:image/png;base64,{img(path)}'>")
    for title, df in tables.get(p, []):
        if df is not None and len(df):
            html.append(f"<h3>{title}</h3>{df.head(60).round(4).to_html(classes='t', border=0)}")
html.append("</body></html>")
(OUT2 / "report.html").write_text("\n".join(html))
print(json.dumps(summary, indent=2, default=str)[:3000]); print("\nreport:", OUT2 / "report.html")

### How to read the study, in research order
1. **Data and univariate facts (Parts A–B).** Clean returns and splits. Prices are I(1) and returns are stationary (d\* < 1 keeps memory). Volatility clusters, often asymmetrically. Residuals are fat-tailed (EVT ξ > 0) and extremes cluster. These facts set every downstream choice: vol scaling, fat-tailed risk, and EVT event rules.
2. **Relationships (Part C).** A statistical lead–lag only counts if it survives FDR on train, validation and the blind test; its **survival** gives the holding horizon and whether it is alive now. Fundamental chains (Step 16) start from economics, so there are few hypotheses and each has a reason. The library (Step 17) ranks all of them.
3. **Factors and ranks (Part D).** Factor IC by family × level (including carry) shows *where* the opportunity is. MTF ranks turn it into a per-product bull / bear score at every frequency.
4. **Regimes (Part E).** The overnight-versus-day split, the night-session term in the volatility forecast and the opening reversal are read by volatility regime too (Steps 11d, 13e, 23d); Step 18c reads price and volume together as a path in phase space (signed area, efficiency, quadrants, a product geometry map) Step 18h adds the Wasserstein flow indicator and measures how much the three flow indicators overlap; Step 18g tracks how the quantiles of the return distribution migrate (Probability Mass Flow Tracker, Quantile Flow Field; windows chosen on validation, ensemble shown); Step 18f follows trend episodes through before / during / after phases with volume and OI (rises and falls separately, five cheat-sheet rules tested); Step 18e adds open interest (new longs / short covering / new shorts / long liquidation by volume) and Step 18d reads their topology (persistent loops and fragmentation, a Mapper graph of market states and the flows between them); Step 11e checks that the final volatility model leaves no clustering in the standardised residuals before Steps 12 and 13 use them, and Step 23e measures how long late-day high-volatility runs last and whether entering at their start beats entering just before the close. If a factor's IC changes sign or rank across regimes, use regime-specific weights. If rule changes coincide with breaks, treat pre- and post-change data separately. Steps 18i (ECHO field momentum), (the WAVE classifier of Step 18j was retired, see Step 26d) and 18k (candle anatomy: range, wicks, rejection, volume, volatility with regime-specific learned thresholds, windows inside and beyond 30 minutes, and the negated-series mirror test) add the short-horizon view.
5. **Dynamics and monitoring (Part F).** Do factors, sectors and factor × sector cells trend or revert, and does rotation pay (Step 21)? When does an edge stop working, and does switching it off help out of sample (Step 22)?
   Step 13f looks at regimes first: it defines volatility / trend / co-movement regimes from earlier data only, allocates train and validation by regime blocks (purged, test chronological), and studies the overnight return by sector (EquityIndex separately) and positive versus negative overnight days.
   Step 13g re-tests the state, ECHO and WAVE results on demeaned, non-overlapping labels inside the regimes, with a trailing-volatility control and walk-forward blocks: read it before trusting any directional claim from 18e–18j.
   Step 18l replaces the assumed candle thresholds of 18k by break points the data itself shows in each regime; read its comparison with the rulebook values before using any threshold.
   Step 18m studies the one directional candle effect that survived (a reversal after a close at the extreme of the range) through regime layers, the bounce test and break-even cost.
   Step 18n tests the volume-weighted STARK / CAIRO / FLUX components, the money / OI tape features and the open-interest-gated breakout, fakeout, pullback, spring, doji and Fibonacci setups at 5 and 15 minutes, with a learned composite and a Granger / transfer-entropy lead-lag on the signals.
   Step 18o compares "Stage 2 plus breakout" with the full volatility-contraction pattern at daily, 60-minute and 30-minute frames in R multiples (stop below the last contraction low), with break-even cost, and tests the bar-testable parts of the Taker note (hurdle, squeeze, flow imbalance, sweep, volume node).
   Step 18p measures incremental volume over its EMA (span 2-13, 5-stamp window) at 5 / 15 / 30 / 60 minutes and daily against returns, ranges, open interest, breakouts and reversals.
   Step 18q learns the breakout / reversal thresholds per instrument, tabulates Return x change in OI, and profiles the last 2-3 bars before a spike and before a continuation or a reversal.
   Step 18r tabulates, for ten causal signals, the edge of their direction at 5 minutes to 20 days against the round-trip cost, to find the horizon at which direction first pays.
   Step 18s draws every sector, finds the direction and regime that fit it, splits each time frame by its own volatility regime across train / validation / test, and measures momentum versus reversion by sector, regime and horizon.
   Step 18t tests the daily Stage 2 breakout as a portfolio: block-clustered t, long / short, sector, regime, year, leave-one-out and a plain trend benchmark.
   Step 18u marks long entries (RSI crosses up in a bull regime) and short entries (RSI crosses down in a bear regime) with a learned RSI level and compares them with entering the same regime on every bar.
   Step 18v uses the RSI inside the daily Stage 2 breakout as an entry marker, an exit and an entry timing, each tested against the same trades without RSI.
6. **Strategies, portfolio and risk (Part G).** Deploy only what was selected on validation, clears the DSR hurdle and survives delay 1 net of costs. Check PBO. Combine with risk parity or HRP under a volatility target, check capacity and cost stress (Step 24), and size by the risk model whose VaR / ES pass their backtests (Step 25). Step 24c explains why some dots lie above the frontier and gives every strategy a keep / drop decision with its reason; 24d builds a book per investor type (risk appetite × lock-in); 24e searches portfolio recipes on validation only, with a no-trade band for costs; 24f measures netting and lists what else can be done.
7. **Board (Step 26).** Today's long (bull) and short (bear) candidates, with the suggested vehicle, relationship health and the strategy mix for the current regime.

## Step 26a — Causality audit (static scan + every truncation test)

The final check before the board is trusted, in two layers:

1. **Truncation tests.** Every `causal_test` run in this notebook (Steps 0c, 1, 11c, 18, 18b, 19), with the cut dates, the outputs compared, the largest difference and the first timestamp that changed.
2. **Static scan.** The source of every executed cell is searched for forbidden operations. `LABEL` = target construction (allowed: a target looks ahead by definition and is never an input); `ALLOWED` = marked `# causal-ok:` with its reason (descriptive EDA, same-date cross-section, cost accounting); `REVIEW` = must be fixed or justified.

**Used later.** A `REVIEW` hit or a failed truncation test is listed in the issues appendix of the report deck (Step 26c); with `CAUSAL_STRICT = True` a failed test already stopped the run where it happened.

In [ ]:
# step 26a
CAUSAL = pd.DataFrame(CAUSAL_LOG)
CAUSAL.to_csv(OUT2 / "causality_truncation_tests.csv", index=False)
real = CAUSAL[~CAUSAL.test.str.startswith(("negative control", "positive control"))]
summ = real.groupby(["step", "test"]).agg(cuts=("cut", "nunique"), outputs=("output", "nunique"), passed=("passed", "all"), max_abs_diff=("max_abs_diff", "max")).reset_index()
print(f"truncation tests: {len(summ)} builders, {len(real)} comparisons, failed: {int((~summ.passed).sum())}")
print(summ.to_string(index=False))
try:
    _src = {f"cell {i}": s for i, s in enumerate(get_ipython().user_ns.get("In", [])) if s}
except Exception:
    _src = {}
SCAN = static_causality_scan(_src)
SCAN.to_csv(OUT2 / "causality_static_scan.csv", index=False)
print(f"\nstatic scan of {len(_src)} executed cells: " + (", ".join(f"{k} {v}" for k, v in SCAN.verdict.value_counts().items()) if len(SCAN) else "no hit"))
rev = SCAN[SCAN.verdict == "REVIEW"].drop_duplicates("code")
print("REVIEW (fix or justify):\n" + (rev[["cell", "issue", "code"]].to_string(index=False) if len(rev) else "none"))

## Step 26b — Findings report: what worked, what did not, which strategies to follow

Every earlier step prints its own evidence. This step **reads the saved results of all of them** and writes the conclusions in one place, as a researcher would for a decision meeting. Nothing new is estimated here; each sentence is generated from a result table, so the report cannot drift from the numbers.

**Outputs.** `research_findings.md` and `research_findings.html` (full text with tables), and the same sections as **slides at the front of the PDF deck** (Step 26c). Each section ends with the **verdict rule** used, so a reader can disagree with the threshold instead of the conclusion.

**How a method or strategy is judged.** Only out-of-sample evidence counts, in this order: *validation* (used to choose) then *test* (looked at once). A result is called **worked** if it has the same sign in validation and test and clears the stated threshold; **did not work** if it was significant on train and failed to replicate; **inconclusive** if the sample is too short to say.

| Verdict for a strategy | Rule (settings `FOLLOW_*`) |
|---|---|
| **FOLLOW** | validation Sharpe ≥ `FOLLOW_MIN_VAL`, test Sharpe ≥ `FOLLOW_MIN_TEST`, and the test Sharpe after the execution model at 100 m CNY ≥ `FOLLOW_MIN_EXEC` |
| **WATCH** | positive on validation and test, but fails one of the above |
| **DROP** | negative on test, or only positive before costs |

**Used later.** This is the last analysis step. The markdown and HTML files are the hand-off for review; the slides are the executive summary of the PDF deck.

In [ ]:
# step 26b
import textwrap, json as _json
FINDINGS = []                         # sections: dict(part, title, bullets, table, rule)
def rd(name):
    p = OUT2 / name
    try: return pd.read_csv(p) if p.exists() else None
    except Exception: return None
def sec(part, title, bullets, table=None, rule=""):
    FINDINGS.append(dict(part=part, title=title, bullets=[b for b in bullets if b], table=table, rule=rule))
def pct(x, d=0): return f"{100 * x:.{d}f}%"
def f2(x, d=2): return "n/a" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{d}f}"
J = _json.loads((OUT2 / "research_summary.json").read_text()) if (OUT2 / "research_summary.json").exists() else {}

# ---------------- data ----------------
try:
    led = DQ_LEDGER if "DQ_LEDGER" in globals() else rd("data_quality_treatment.csv")
    sec("A", "Data and what was done about its problems",
        [f"{J.get('universe', len(UNIV))} products with enough history; period {SPL['train'][0].date()} → {RAW.index[-1].date()}; one calendar split for every frequency: train to {SPL['train'][-1].date()}, validation {SPL['val'][0].date()} → {SPL['val'][-1].date()}, test from {SPL['test'][0].date()} (embargo {EMB} days).",
         "Every data-quality flag has a rule (Step 2f): errors removed, real events kept and clipped, no look-ahead. " + "; ".join(f"{r.issue}: {r.applied}" for r in led.itertuples() if not str(r.issue).startswith("…")) + ".",
         (lambda rl: f"Causality: {len(rl)} truncation comparisons on the real builders, {sum(1 for r in rl if r['passed'] is False)} failed (negative and positive controls excluded).")([r for r in CAUSAL_LOG if not str(r['test']).startswith(('negative control', 'positive control'))]) if "CAUSAL_LOG" in globals() else ""],
        led[["issue", "applied", "rows_affected", "products"]], "every flag of Step 2 appears; look-ahead audits must all pass")
except Exception as e: print("findings: data section skipped:", repr(e))

# ---------------- Part B ----------------
try:
    vol = J.get("volatility", {}) or {}; wf = vol.get("walk_forward_vs_static_test_qlike") or {}
    mv = J.get("multivariate", {}) or {}; cop = J.get("copulas", {}) or {}; ov = (J.get("overnight", {}) or {})
    b = []
    sd = rd("stationarity.csv"); fd = rd("fracdiff.csv")
    if sd is not None: b.append(f"Prices are non-stationary and returns are stationary in {pct((sd[sd.variable.str.contains('ret', case=False)].verdict.str.contains('stationary', case=False) & ~sd[sd.variable.str.contains('ret', case=False)].verdict.str.contains('non', case=False)).mean()) if (sd.variable.str.contains('ret', case=False)).any() else 'most'} of the series tested (ADF / KPSS)" + (f"; the fractional-differencing order d* has median {fd.d_star.median():.1f}" if fd is not None else "") + ".")
    ms_ = rd("model_scores.csv")
    if ms_ is not None:
        t_ = ms_[ms_.split == "test"].groupby("model").r2.median().sort_values(ascending=False); b.append(f"Linear mean models barely predict returns: best median out-of-sample R² on test is {t_.iloc[0]:.4f} ({t_.index[0]}); " + ("none beats zero." if t_.iloc[0] <= 0 else "tiny but positive.") + " Predictability must come from relationships between series, factors and ranks, not from a series' own past.")
    if vol: b.append(f"Volatility: the model that survives every regime and time frame is {vol.get('robust_model_all_regimes_and_timeframes')}; by time frame {vol.get('best_by_timeframe')}. Walk-forward re-selection scored {wf.get('wf')} on test blocks against {wf.get('static_val')} for the single validation winner (lower QLIKE is better).")
    dw = rd("dist_winners.csv")
    if dw is not None: b.append(f"Return distribution: winning family counts {dw.winner.value_counts().head(4).to_dict()}; mean out-of-sample log-likelihood gain over the normal {dw.oos_ll_gain_vs_norm.mean():.3f} per observation.")
    gp = rd("evt_gpd.csv")
    if gp is not None: b.append(f"Tails: median GPD shape ξ = {gp.xi.median():.2f} (heavy if > 0.1). Normal-based risk numbers understate tail losses.")
    sec("B", "Univariate facts: what the series look like", b, None, "stationarity from ADF and KPSS agreeing; volatility model chosen by worst-regime rank")
    ev = rd("evt_event_study.csv")
    b = []
    if ev is not None:
        e1 = ev[(ev.split != "train")].groupby(["event", "h"]).continuation.mean().unstack(0).round(3)
        b.append("Tail events: after a daily shock, continuation (positive = same direction) in validation + test, mean by horizon (days): " + "; ".join(f"h={h}: " + ", ".join(f"{c} {v:+.2f}" for c, v in r.items()) for h, r in e1.iterrows()))
    sl = rd("sector_lead_lag.csv"); tsw = rd("evt_tail_spillover_windows.csv"); llh = rd("lead_lag_long_horizon.csv")
    b.append(f"Beyond a few days: sector Granger links replicating in validation and test: {mv.get('sector_links_replicating', 'n/a')}; tail-spillover pairs replicating by window (days) {mv.get('tail_spillover_windows_replicating')}.")
    if llh is not None:
        ok = llh[llh.split == "test"].merge(llh[llh.split == "train"], on=["h", "driver"], suffixes=("_te", "_tr")); ok = ok[np.sign(ok.ic_te) == np.sign(ok.ic_tr)]
        b.append("Long-horizon lead-lag with the same sign in train and test: " + (", ".join(f"{r.driver} @ {r.h}d" for r in ok.itertuples()) if len(ok) else "none") + ".")
    sec("B", "Beyond a few days (extremes, spillover, long-horizon links)", b, None, "same sign in train, validation and test and |t| > 2 in train")
    b = [f"One market factor explains {pct(mv.get('pc1_share', np.nan))} of daily co-movement; {mv.get('n_pc_above_noise')} component(s) are above the random-matrix noise edge." if mv else "",
         f"Best covariance estimator on validation: {mv.get('cov_best')}; idiosyncratic share of variance after market and sector (median) {pct(mv.get('idio_median', np.nan))}.",
         f"Fama–MacBeth characteristics with the same sign in all three periods: {mv.get('fama_macbeth_same_sign') or 'none'}."]
    if cop:
        b += [f"Copulas ({cop.get('pairs')} pairs): the best family on validation is {cop.get('best_family_val')} and on test {cop.get('best_family_test')}; lower minus upper tail dependence averages {cop.get('lower_minus_upper_test', np.nan):+.3f} on test (persistence of the asymmetry train→test: {cop.get('asym_persistence', np.nan):.2f}).",
              f"Universe copula: Student-t with ν = {cop.get('nu')} beats Gaussian by {cop.get('ll_gain_val', np.nan):.2f} (validation) and {cop.get('ll_gain_test', np.nan):.2f} (test) log-likelihood units per day; the model closest to the observed frequency of joint crashes is the {cop.get('closest_joint_tail')}."]
    sec("B", "Multivariate structure and copulas", b, rd("copula_joint_extremes.csv"), "t-copula preferred when it wins out of sample on both validation and test")
    # overnight / T+1
    b = []
    od = rd("overnight_decomposition.csv"); t1 = rd("overnight_t1_comparison.csv")
    if od is not None:
        a_ = od[od.sector == "ALL"].set_index("split"); b.append("Overnight vs intraday, all products (bp/day): " + "; ".join(f"{sp}: ON {a_.loc[sp, 'ON_bp']:+.1f} (t {a_.loc[sp, 'ON_t']:+.1f}) vs ID {a_.loc[sp, 'ID_bp']:+.1f} (t {a_.loc[sp, 'ID_t']:+.1f})" for sp in a_.index))
    t1s = (ov.get("t1") or {})
    if t1s:
        b.append(f"T+1 test: the pent-up-pressure coefficient (yesterday's intraday move → tonight's overnight return) has a stable sign across the three periods for EquityIndex futures: {t1s.get('equity_pent_up_consistent')}, and for all other products: {t1s.get('other_pent_up_consistent')}. Gap fade in EquityIndex stable: {t1s.get('equity_gap_fade_consistent')}.")
        eq_, ot_ = t1s.get('equity_pent_up_consistent'), t1s.get('other_pent_up_consistent')
        b.append("Reading: " + ("the pattern is stable in EquityIndex but not elsewhere, which is what a T+1 channel predicts." if eq_ and not ot_ else "the pattern is stable in the other products too, so it is not specific to equity-index futures (no T+1 evidence)." if eq_ and ot_ else "no stable pent-up-pressure pattern in EquityIndex: the data do not support a T+1 effect in futures.") + " Futures themselves trade T+0; any T+1 effect can only reach them through the equity-index link.")
    sec("B", "Overnight return puzzle and the T+1 effect", b, t1, "stable sign in train, validation and test; EquityIndex compared with the rest")
except Exception as e: print("findings: Part B skipped:", repr(e))
try:
    rdg = J.get("residual_diagnostics") or {}
    if rdg.get("reading"):
        sec("B", "Volatility model adequacy: residual diagnostics after the final model", [rdg["reading"], ("Not adequate out of sample (tail and VaR estimates for these are less reliable): " + ", ".join(rdg["not_adequate"])) if rdg.get("not_adequate") else "All series are adequate out of sample."], rd("std_resid_diagnostics.csv"), "adequate = Ljung-Box on squared standardised residuals and ARCH-LM both p > 0.05 on validation + test with frozen parameters")
except Exception as e: print("findings: residual-diagnostics section skipped:", repr(e))
try:
    b = []; ro = J.get("overnight_regime", {}) or {}
    for k_ in ("regime", "har_on", "opening_reversal", "high_vol_entry"):
        if (ro.get(k_) or {}).get("reading"): b.append(ro[k_]["reading"])
    if b: sec("B", "Overnight and day returns by volatility regime, night-session variance in HAR, opening reversal", b, rd("regime_hypotheses.csv"), "supported = predicted sign in train, validation and test and |t| > 2 on test; the strategy is judged like every other one")
except Exception as e: print("findings: overnight-regime section skipped:", repr(e))
try:
    pv_ = J.get("price_volume_geometry") or {}
    if pv_.get("reading"):
        sec("C", "Price-volume geometry: signed area, efficiency, quadrants and the product map", [pv_["reading"], "Geometry map clusters (training profile): " + "; ".join(f"{k}: {', '.join(v)}" for k, v in (pv_.get("map", {}).get("clusters") or {}).items()) + ".", "Confirmed out of sample: " + (", ".join(pv_["confirmed_features"]) if pv_.get("confirmed_features") else "none")], rd("pv_geometry_ic.csv"), "confirmed = BH-adjusted on train, same sign in validation and test, |t| > 2 on test; features overlap over 20 days, so NW lags are used")
except Exception as e: print("findings: price-volume section skipped:", repr(e))
try:
    td_ = J.get("price_volume_topology") or {}
    if td_.get("reading"):
        sec("C", "Price-volume topology: loops, fragmentation and the Mapper state graph", [td_["reading"], "Mapper flows (from node, to node, days): " + str((td_.get("mapper") or {}).get("top_flows"))], rd("topology_market_vol.csv"), "controls_ok = same sign in train, validation and test, |t| > 2 on test, with trailing volatility as control; the loop test compares real windows with windows whose volume was shuffled")
except Exception as e: print("findings: price-volume topology section skipped:", repr(e))
try:
    ov_ = J.get("volume_oi_price") or {}
    if ov_.get("reading"):
        sec("C", "Volume and open interest together: what they do to price", [ov_["reading"], "Product consistency: " + "; ".join(f"{r['state']}: sign agrees train/test in {r['sign_agree_train_test']:.0%} of products" for r in (ov_.get("consistency") or []))], rd("oi_volume_decisions.csv"), "calls are made on train and validation only; confirmed = same sign and |t| > 2 on the test period")
except Exception as e: print("findings: volume-OI section skipped:", repr(e))
try:
    tp_ = J.get("trend_phases_oi") or {}
    if tp_.get("reading"):
        sec("C", "Trend phases with volume and open interest: before, during and after, rises and falls separately", [tp_["reading"], "Expected signs come from the pasted rule set (skill volume-oi-momentum-phases), fixed before looking at the data; a rule or phase signature is supported only if the sign holds in train, validation and test and |t| > 2 on test."], rd("trend_rule_tests.csv"), "supported = expected sign in all three periods and |t| > 2 on test")
except Exception as e: print("findings: trend-phase section skipped:", repr(e))
try:
    qg_ = J.get("quantile_flow") or {}
    if qg_.get("reading"):
        sec("C", "Quantile flow: Probability Mass Flow Tracker and Quantile Flow Field", [qg_["reading"], "Six (fast, slow) window pairs were tried; the pair is chosen on validation only (worst-regime IC, plateau rule) and an equal-weight ensemble is shown beside it; Bonferroni |t| > 2.64 for the chosen pair."], rd("quantile_flow_selection.csv"), "confirmed = same sign of IC in train, validation and test and the stated |t| threshold on test")
except Exception as e: print("findings: quantile-flow section skipped:", repr(e))
try:
    pw_ = J.get("flow_indicator_overlap") or {}
    if pw_.get("reading"):
        sec("C", "Are the three flow indicators independent evidence?", [pw_["reading"], "Pairwise cross-sectional correlations (test): " + "; ".join(f"{k}: {v:+.2f}" for k, v in (pw_.get("corr_test") or {}).items())], rd("flow_indicators_combination.csv"), "gain = IC of the equal-weight combination / mean single IC: 1 = same information, 1.73 = three independent signals")
except Exception as e: print("findings: flow-overlap section skipped:", repr(e))
try:
    ec_ = J.get("echo_field_momentum") or {}
    if ec_.get("reading"):
        sec("C", "ECHO field momentum (Step 18i)", [ec_["reading"]], rd("echo_hypotheses.csv"), "spans chosen on validation only; supported = expected sign in validation and test and |t_test| above 2")
except Exception as e: print("findings: echo section skipped:", repr(e))
try:
    kc_ = J.get("candle_anatomy") or {}
    if kc_.get("reading"):
        sec("C", "Candle anatomy: range, wicks, rejection, volume, volatility (Step 18k)", [kc_["reading"]], rd("candle_event_selection.csv"), "thresholds learned per volatility regime on train; configuration chosen on validation; every structure also run on the negated series; effects in excess of the unconditional mean")
except Exception as e: print("findings: candle section skipped:", repr(e))

try:
    rg_ = J.get("regimes_overnight") or {}
    if rg_.get("reading"):
        sec("C", "Regimes first: regime-balanced split, overnight by sector, positive vs negative overnight days (Step 13f)", [rg_["reading"]], rd("overnight_by_regime_cells.csv"), "regimes defined from earlier data only; train/validation are purged interleaved blocks per regime, test chronological and read once; Bonferroni over all cells; supported = same sign in all three periods")
except Exception as e: print("findings: regimes section skipped:", repr(e))

try:
    rd_ = J.get("demeaned_retest") or {}
    if rd_.get("reading"):
        sec("C", "Method fixes: demeaned, regime-aware, walk-forward re-tests (Step 13g)", [rd_["reading"]], rd("regime_states_decisions.csv"), "labels in excess of product and cross-section means, non-overlapping 5-day windows, trailing-volatility control on size-of-move tests, seven purged walk-forward blocks")
except Exception as e: print("findings: demeaned re-test section skipped:", repr(e))

try:
    kt_ = J.get("threshold_atlas") or {}
    if kt_.get("reading"):
        sec("C", "Threshold atlas: the market's own break points by regime (Step 18l)", [kt_["reading"]], rd("learned_thresholds.csv"), "break point searched per regime on train_R, checked on val_R, test read once; Bonferroni over all searches; compared with the fixed 75th percentile and the rulebook value")
except Exception as e: print("findings: threshold atlas section skipped:", repr(e))

try:
    kl_ = J.get("layered_close_location") or {}
    if kl_.get("reading"):
        sec("C", "Close location in the range: layered regimes, bounce versus edge, cost (Step 18m)", [kl_["reading"]], rd("layered_close_location.csv"), "20 equal close-location bins; child cell must beat its parent in train_R, val_R and test; thin cells shrunk toward the parent; entry at the next open; break-even round-trip cost reported")
except Exception as e: print("findings: layered close-location section skipped:", repr(e))

try:
    kn_ = J.get("framework_setups") or {}
    if kn_.get("reading"):
        sec("C", "Volume-weighted STARK / CAIRO / FLUX, tape features and open-interest-gated setups (Step 18n)", [kn_["reading"]], rd("setups_oi_gated.csv"), "5 and 15 minute bars; every cut learned on train_R per volatility regime; a gated setup must beat its own parent in train_R, val_R and test; outcome from the close of the signal bar; composite ridge picked on val_R")
except Exception as e: print("findings: framework section skipped:", repr(e))

try:
    ko_ = J.get("stage2_vcp") or {}
    if ko_.get("reading"):
        sec("C", "Stage 2 plus breakout versus the full VCP, sweeps and volume nodes (Step 18o)", [ko_["reading"]], rd("vcp_study.csv"), "daily, 60 and 30 minute frames; every cut learned on train_R per volatility regime; the VCP must beat its parent in train_R, val_R and test; R multiples with the stop below the last contraction low; round-trip cost 2 x the per-side default")
except Exception as e: print("findings: stage 2 section skipped:", repr(e))

try:
    kp_ = J.get("incremental_volume") or {}
    if kp_.get("reading"):
        sec("C", "Incremental volume over its EMA: returns, open interest, breakouts, reversals (Step 18p)", [kp_["reading"]], rd("incremental_volume_selection.csv"), "5 / 15 / 30 / 60 minute and daily frames; EMA span 2-13 chosen on train_R only; 5-stamp window; seasonally adjusted volume is the headline; val_R must confirm, test read once")
except Exception as e: print("findings: incremental volume section skipped:", repr(e))

try:
    kq_ = J.get("instrument_rules") or {}
    if kq_.get("reading"):
        sec("C", "Per-instrument rules, Return x OI matrix and the bars before a spike (Step 18q)", [kq_["reading"]], rd("rules_by_instrument_results.csv"), "cuts learned per instrument on train_R (shrunk to the pooled cut); spikes defined from each instrument's own training tail; val_R must confirm, test read once")
except Exception as e: print("findings: instrument rules section skipped:", repr(e))

try:
    kr_ = J.get("direction_horizon") or {}
    if kr_.get("reading"):
        sec("C", "Direction by horizon: where a signal's direction first pays more than the cost (Step 18r)", [kr_["reading"]], rd("direction_by_horizon.csv"), "10 causal signals, horizons 5 minutes to 20 days, drift removed, block-clustered t; sign as measured or inverted; val_R must confirm, test read once")
except Exception as e: print("findings: direction by horizon section skipped:", repr(e))

try:
    ks_ = J.get("sector_regimes") or {}
    if ks_.get("reading"):
        sec("C", "Sectors: direction, momentum versus reversion by regime and horizon (Step 18s)", [ks_["reading"]], rd("variance_ratio_rules.csv"), "slope of the next q-bar move on the last, clustered by day; horizon and sign learned on train_F, confirmed in val_F, test read once; volatility regimes per time frame")
except Exception as e: print("findings: sector regimes section skipped:", repr(e))

try:
    kb_ = J.get("stage2_robustness") or {}
    if kb_.get("reading"):
        sec("C", "Daily Stage 2 breakout: robustness as a portfolio (Step 18t)", [kb_["reading"]], rd("stage2_robust_main.csv"), "net of cost, t clustered by 10-day block, versus the plain trend filter; long / short, sector, regime, year, leave-one-out; test read once")
except Exception as e: print("findings: stage 2 robustness section skipped:", repr(e))

try:
    ku_ = J.get("rsi_regime_entries") or {}
    if ku_.get("reading"):
        sec("C", "RSI-marked entries in bull and bear regimes (Step 18u)", [ku_["reading"]], rd("rsi_regime_entries.csv"), "long on RSI cross-up in a bull regime, short on cross-down in a bear regime; level pasted, pooled or per product (train_R); child must beat the same regime entered on every bar; test read once")
except Exception as e: print("findings: rsi regime entries section skipped:", repr(e))

try:
    kv_ = J.get("rsi_inside_stage2") or {}
    if kv_.get("reading"):
        sec("C", "RSI inside the Stage 2 breakout: entry marker, exit, entry timing (Step 18v)", [kv_["reading"]], rd("rsi_exit_learned.csv"), "RSI target / floor learned on train_R (bp per day held); marker must beat all trades and keep the tail; pullback entry per signal; test read once")
except Exception as e: print("findings: rsi inside stage 2 section skipped:", repr(e))

# ---------------- Part C-D ----------------
try:
    b = []; ga = rd("granger_all_links.csv")
    if ga is not None: b.append(f"Lead-lag links (Granger): {ga.verdict.value_counts().to_dict()} — REPLICATED means significant on train, confirmed on validation and on test; BLIND ONLY links found only by the sealed test are treated as leads to re-check, not as edges.")
    gt = rd("granger_by_timeframe.csv")
    if gt is not None: b.append("By time frame, candidates that replicated: " + ", ".join(f"{r.tf}: {r.replicated}/{r.candidates}" for r in gt.itertuples()) + ".")
    co = rd("cointegration.csv"); fc = rd("fundamental_chains.csv")
    if co is not None: b.append(f"Cointegrated pairs valid on train and validation: {int(co.valid.sum())} of {len(co)}; still mean-reverting on test: {int(co.test_ok.sum())}.")
    if fc is not None: b.append(f"Fundamental chains (steel, crush, chemicals ...): {int(fc.valid.sum())} valid of {len(fc)} tested; confirmed on test {int(fc.test_ok.sum())}; trading signal positive on test in {int(fc.signal_test_ok.sum())}.")
    rl = rd("relationship_library.csv")
    if rl is not None: b.append("Relationship library: " + ", ".join(f"{k} {v}" for k, v in rl.type.value_counts().items()) + ". Top by score: " + "; ".join(f"{r.type} {r.a}-{r.b}" for r in rl.head(5).itertuples()) + ".")
    f16 = rd("fundamental_chains_16b.csv"); pl16 = rd("fundamental_pooled_16b.csv"); sk16 = rd("fundamental_skip_reasons.csv")
    if f16 is not None:
        b.append("Fundamental chains with a rolling hedge ratio (16b): " + ", ".join(f"{k} {v}" for k, v in f16.status.value_counts().items()) + (f"; Step 16 had skipped {int((sk16.step16 == 'skipped').sum())} chains, of which {int(((sk16.step16 == 'skipped') & sk16.fixable).sum())} can now be evaluated (see fundamental_skip_reasons.csv)." if sk16 is not None else "."))
    if pl16 is not None and len(pl16):
        r_ = pl16[pl16.period == "validation + test"]
        b.append("Pooled pull-back signal across chains (validation + test): " + "; ".join(f"{r.group}: slope {r.slope:+.3f}, t {r.t:+.1f} ({int(r.chains)} chains)" for r in r_.itertuples()) + ". Chains with three years or less of signal are WATCH only and carry a reduced size.")
    sec("C", "Relationships between products", b, ga.head(12) if ga is not None else None, "REPLICATED = same sign and q < 0.05 on train, validation, test")
    b = []; fs = rd("factor_scorecard.csv")
    if fs is not None:
        w = fs[(fs.val_confirm == 1) & (fs.test_t5 > 1.5)].sort_values("test_t5", ascending=False)
        b.append(f"Factor features tested: {len(fs)}; confirmed on validation: {int((fs.val_confirm == 1).sum())}; also |t|>1.5 on test with the train sign: {len(w)}. Best on test: " + ", ".join(f"{r.feature} (t {r.test_t5:.1f})" for r in w.head(5).itertuples()) + ".")
        fam = fs.groupby("family").agg(n=("feature", "size"), val_ok=("val_confirm", "sum"), test_t5=("test_t5", "median")).round(2); b.append("By family (median test t): " + ", ".join(f"{i} {r.test_t5:+.1f}" for i, r in fam.iterrows()) + ".")
    fg = J.get("factor_groups", {}) or {}
    if fg: b.append(f"Factor groups of 1–4 families: {fg.get('groups_tested')} tested, {fg.get('agree_out_of_sample')} agree in train, validation and test (about {max(1, round(0.03 * (fg.get('groups_tested') or 0)))} by luck); best validated group {fg.get('best')}.")
    rk = rd("rank_validation.csv")
    if rk is not None:
        tt = rk[rk.split == "test"]; b.append("Rank indicators, test rank IC by time frame: " + ", ".join(f"{r.tf}/{r.variant} {r.ic:+.3f}" for r in tt.dropna(subset=["ic"]).head(8).itertuples()))
    sec("D", "Factors and opportunity ranks", b, fs.sort_values("test_t5", ascending=False).head(10)[["feature", "family", "level", "ic5", "val_t5", "test_t5"]] if fs is not None else None, "confirmed = same sign on validation, |t| > 1.5 on test")
except Exception as e: print("findings: Part C-D skipped:", repr(e))

# ---------------- dynamics / monitor ----------------
try:
    dy = J.get("dynamics", {}).get("summary", {}) if J.get("dynamics") else {}; mo = J.get("monitor", {}) or {}
    b = [f"Momentum vs reversal inside {k.replace('_', ' ')} series: {v.get('momentum_train')} momentum and {v.get('reversal_train')} reversal on train, {v.get('test_ok')} confirmed on test." for k, v in dy.items() if v.get("series")]
    if mo: b.append(f"Drift monitor on {mo.get('edges')} edges today: {mo.get('status_now')}.")
    gt_ = rd("monitor_gating_test.csv")
    if gt_ is not None: b.append(f"Gating an edge by the monitor improves test Sharpe in {pct((gt_[gt_.split == 'test'].improves.astype(bool)).mean())} of edges (a monitor that does not help is not worth running).")
    sec("F", "Dynamics and monitoring", b, None, "rotation or reversal counts as found only if it is confirmed on test")
except Exception as e: print("findings: dynamics skipped:", repr(e))

# ---------------- strategies ----------------
try:
    bt = rd("backtest_summary.csv"); ex = rd("strategies_execution_adjusted.csv"); rg = rd("strategy_sharpe_by_regime.csv"); cp = rd("cpcv_summary.csv")
    T = bt.set_index("strategy")[["family", "sharpe_val", "sharpe_test", "dsr", "selected"]]
    if ex is not None: T = T.join(ex.set_index("strategy")[["test_100m", "breakeven_aum_cny_m"]])
    else: T["test_100m"] = np.nan; T["breakeven_aum_cny_m"] = np.nan
    ovl = (ov.get("strategy") or {}).get("selected") or []
    for r in ovl:                                               # the overnight strategies are not in the close-to-close library tables
        nm = f"overnight {r['strategy']} (-{r['entry_min_before_close']}m → +{r['exit_min_after_open']}m)"
        T.loc[nm, ["family", "sharpe_val", "sharpe_test", "dsr", "selected", "test_100m", "breakeven_aum_cny_m"]] = ["H Overnight", r["sharpe_val"], r["sharpe_test"], r.get("dsr_test"), True, r["sharpe_test"], np.nan]
    orl = (((J.get("overnight_regime") or {}).get("opening_reversal") or {}).get("selected")) or []
    for r in orl:                                               # the same-day opening-reversal strategy (Step 23d) is not in the close-to-close tables either
        nm = f"opening reversal ({r['gate']}, |gap|>{r['theta']:g}, +{r['exit_min_after_open']}m, {r['way']})"
        T.loc[nm, ["family", "sharpe_val", "sharpe_test", "dsr", "selected", "test_100m", "breakeven_aum_cny_m"]] = ["H Overnight", r["sharpe_val"], r["sharpe_test"], r.get("dsr_test"), True, r["sharpe_test"], np.nan]
    for r in (((J.get("overnight_regime") or {}).get("high_vol_entry") or {}).get("selected")) or []:
        if str(r.get("role", "")).startswith("early"):
            nm = f"early entry at high-vol start ({r['kind']}, x{r['x']:g}, {r['gate']}, +{r['exit_min_after_open']}m)"
            T.loc[nm, ["family", "sharpe_val", "sharpe_test", "dsr", "selected", "test_100m", "breakeven_aum_cny_m"]] = ["H Overnight", r["sharpe_val"], r["sharpe_test"], r.get("dsr_test"), True, r["sharpe_test"], np.nan]
    T["test_100m"] = T["test_100m"].fillna(T["sharpe_test"] * 0 + np.nan)
    def verdict(r):
        if r.sharpe_test <= 0 or (np.isfinite(r.test_100m) and r.test_100m <= 0 and r.sharpe_test < 1): return "DROP"
        if r.sharpe_val >= FOLLOW_MIN_VAL and r.sharpe_test >= FOLLOW_MIN_TEST and (not np.isfinite(r.test_100m) or r.test_100m >= FOLLOW_MIN_EXEC): return "FOLLOW"
        return "WATCH" if r.sharpe_val > 0 and r.sharpe_test > 0 else "DROP"
    T["verdict"] = T.apply(verdict, axis=1); T = T.sort_values(["verdict", "sharpe_test"], ascending=[True, False])
    if rg is not None: T = T.join(rg.set_index("Unnamed: 0").apply(lambda r: pd.Series({"best_regime": r.idxmax() if r.notna().any() else "", "worst_regime": r.idxmin() if r.notna().any() else ""}), axis=1))
    T.round(3).to_csv(OUT2 / "strategy_verdicts.csv"); STRAT_VERDICTS = T
    fol = T[T.verdict == "FOLLOW"]; wat = T[T.verdict == "WATCH"]
    fam = T.groupby("family").agg(n=("sharpe_test", "size"), positive_test=("sharpe_test", lambda s: int((s > 0).sum())), median_test_sharpe=("sharpe_test", "median")).round(2).sort_values("median_test_sharpe", ascending=False)
    b = [f"{len(T)} strategies tested; {int(T.selected.astype(bool).sum())} passed the validation screen; verdicts: FOLLOW {len(fol)}, WATCH {len(wat)}, DROP {int((T.verdict == 'DROP').sum())}.",
         "FOLLOW: " + (", ".join(f"{i} (val {r.sharpe_val:.1f}, test {r.sharpe_test:.1f})" for i, r in fol.iterrows()) if len(fol) else "none passes all three thresholds — see WATCH."),
         "WATCH: " + (", ".join(f"{i} (test {r.sharpe_test:.1f})" for i, r in wat.head(6).iterrows()) if len(wat) else "none") + ".",
         "Method families by median test Sharpe: " + ", ".join(f"{i} {r.median_test_sharpe:+.2f} ({r.positive_test}/{r.n} positive)" for i, r in fam.head(6).iterrows()) + "."]
    bk = J.get("backtest", {}) or {}
    b.append(f"Selection risk: probability of backtest overfitting {f2(bk.get('pbo'))}; deflated-Sharpe hurdle {f2(bk.get('dsr_hurdle'))} (annualised) for {bk.get('n')} trials. A strategy with test Sharpe below the hurdle may be luck.")
    if cp is not None: r_ = cp[(cp.returns == "flat costs") & (cp.rule == "topM")]; b.append("CPCV: choosing the top-M strategies on past performance gives " + (f"mean path Sharpe {r_.mean_sharpe.iloc[0]:.2f} (5th pct {r_.p5.iloc[0]:.2f}, negative paths {pct(r_.share_negative.iloc[0])})" if len(r_) else "n/a") + " — the procedure matters more than any single pick.")
    ex_ = J.get("execution", {}) or {}
    b.append(f"Execution: median half-spread {ex_.get('half_spread_bp_median')} bp vs flat slippage {ex_.get('flat_slippage_bp_median')} bp; dropped once realistic costs apply: {ex_.get('dropped_by_exec_model')}.")
    if ovl: b.append("Overnight (enter before the close, exit after the opening move): " + "; ".join(f"{r['strategy']} −{r['entry_min_before_close']}m → +{r['exit_min_after_open']}m: val {r['sharpe_val']:.2f}, test {r['sharpe_test']:.2f}, cost break-even {r.get('break_even_cost_x', np.nan):.1f}×" for r in ovl) + f"; rank correlation of validation and test Sharpe across the grid {f2((ov.get('strategy') or {}).get('rank_corr'))}.")
    elif ov: b.append("Overnight strategy: no variant had a positive Sharpe on train and validation, nothing was added to the library.")
    sec("G", "Strategies: what to follow", b, T.reset_index().rename(columns={"index": "strategy"})[["strategy", "family", "sharpe_val", "sharpe_test", "test_100m", "dsr", "verdict"]].head(18), "FOLLOW needs validation, test and execution-adjusted Sharpe above the thresholds")
except Exception as e: print("findings: strategies skipped:", repr(e))

# ---------------- portfolio ----------------
try:
    po = J.get("portfolio", {}) or {}; fr = J.get("frontier", {}) or {}; pc_ = fr.get("portfolio_copula") or {}
    b = [f"Allocation across strategies (test Sharpe): {po.get('test_sharpe')}; scheme chosen on validation: {po.get('best_scheme_by_val')}.",
         f"Capacity at {PARTICIPATION:.0%} participation: median CNY {po.get('capacity_cny_bn', {}).get('median')} bn, 10th percentile {po.get('capacity_cny_bn', {}).get('p10')} bn; tightest products {po.get('capacity_cny_bn', {}).get('binding')}."]
    if fr: b.append(f"Efficient frontier ({fr.get('n')} strategies): tangency portfolio Sharpe {fr.get('tangency_val', np.nan):.2f} on validation → {fr.get('tangency_test', np.nan):.2f} on test; min-variance {fr.get('minvar_test', np.nan):.2f}, equal weight {fr.get('ew_test', np.nan):.2f}, risk parity {fr.get('erc_test', np.nan):.2f} on test. Weights: {', '.join(f'{k} {100*v:.0f}%' for k, v in (fr.get('tangency_weights') or {}).items())}. The gap between the two numbers is the price of estimation error.")
    if pc_: b.append(f"Portfolio copula: strategy returns have t-copula ν = {pc_.get('nu')}; median Kendall τ {pc_.get('median_tau', np.nan):.2f}, median lower-tail dependence {pc_.get('median_lowtail', np.nan):.2f} (0.10 = independent). 99% daily ES: historical {100 * pc_.get('es99_hist', np.nan):.2f}%, Gaussian copula {100 * pc_.get('es99_gauss', np.nan):.2f}%, Student-t copula {100 * pc_.get('es99_t', np.nan):.2f}% of NAV.")
    vb = rd("var_backtest.csv")
    if vb is not None:
        t_ = vb[(vb.split == "test") & (vb.level == vb.level.max())].groupby("method").agg(exc=("exc_rate", "mean"), kupiec=("kupiec_p", "median")).sort_values("kupiec", ascending=False); b.append(f"Risk model: on test, the VaR method with the best median Kupiec p-value is {t_.index[0]} (exceedance {pct(t_.exc.iloc[0], 1)}, nominal {pct(1 - vb.level.max(), 1)}).")
    sec("G", "Portfolio, frontier and tail risk", b, rd("efficient_frontier_named.csv"), "frontier weights are fixed on validation and judged on test")
except Exception as e: print("findings: portfolio skipped:", repr(e))

# ---------------- today ----------------
try:
    rn = J.get("regime_now", {}) or {}
    b = [f"As of {J.get('as_of')}: volatility regime {rn.get('vol')}, Markov state {rn.get('ms')}, breadth {rn.get('breadth')}.",
         f"Strategy mix suggested by the current regime: {J.get('strategy_mix_now')}.",
         f"Strongest bull candidates: {J.get('top_bull')}; strongest bear candidates: {J.get('top_bear')}.",
         f"Volatility monitor: expanding {vol.get('expansion_now') or 'none'}, squeezed {vol.get('squeeze_now') or 'none'}."]
    sec("H", "The board today", b, rd("opportunity_board_latest.csv").head(10) if rd("opportunity_board_latest.csv") is not None else None, "from the latest ex-ante ranks, regime and volatility state")
except Exception as e: print("findings: board skipped:", repr(e))

# ---------------- cautions (automatic) ----------------
try:
    caut = []
    fails = [r for r in RUNLOG if not r["ok"]] if "RUNLOG" in globals() else []
    if fails: caut.append(f"{len(fails)} cell(s) raised an error during this run: " + ", ".join(f"cell {r['cell']} (step {r['step']})" for r in fails[:6]) + " — results of those steps are missing.")
    caut += [f"Multiple testing: Step 14 alone tests hundreds of pairs per time frame; only links that replicate on validation and test are called edges, and the deflated Sharpe hurdle above applies to strategies.",
             f"The test period is {len(TESTI)} trading days (≈ {len(TESTI) / 250:.1f} years): a Sharpe estimated on it has a standard error of about {np.sqrt(1 / (len(TESTI) / 250)):.1f}. Differences below that are noise.",
             "Costs use bar-based half-spreads (an upper bound) and a square-root impact law; there is no order-book data, so capacity numbers are approximate.",
             "Intraday frames start at INTRADAY_START (Step 0); a later start means a shorter training period for intraday results."]
    sec("H", "Cautions", caut, None, "")
except Exception as e: print("findings: cautions skipped:", repr(e))

# ---------------- executive summary (first) ----------------
try:
    top = [f"Strategies to follow: {', '.join(T[T.verdict == 'FOLLOW'].index[:6]) or 'none clears all thresholds'}; to watch: {', '.join(T[T.verdict == 'WATCH'].index[:4]) or 'none'}.",
           f"Relationships that replicated out of sample: Granger {(J.get('granger_all_links') or {}).get('REPLICATED', 0)}, fundamental chains {(J.get('fundamental') or {}).get('test_ok', 'n/a')} of {(J.get('fundamental') or {}).get('chains', 'n/a')}.",
           f"Volatility model that works in every regime and time frame: {(J.get('volatility') or {}).get('robust_model_all_regimes_and_timeframes')}; covariance estimator: {(J.get('multivariate') or {}).get('cov_best')}.",
           f"Dependence: {'Student-t copula (ν = ' + str(cop.get('nu')) + ') beats Gaussian out of sample' if cop and cop.get('ll_gain_test', 0) > 0 else 'Gaussian copula is adequate'}; strategies fail together with lower-tail dependence {f2(pc_.get('median_lowtail') if 'pc_' in globals() and pc_ else None)}.",
           "Overnight / T+1: see the dedicated section; the verdict depends on the EquityIndex versus other-products comparison.",
           f"Regime today: {(J.get('regime_now') or {}).get('combo')}."]
    FINDINGS.insert(0, dict(part="ALL", title="Executive summary", bullets=top, table=None, rule=""))
except Exception as e: print("findings: executive summary skipped:", repr(e))

# ---------------- Part G (portfolio): selection audit, ledger, investor profiles, recipes, levers ----------------
try:
    if "LEDGER" in globals() and "AUDIT" in globals():
        cnt_ = LEDGER.decision.value_counts().to_dict()
        b = [f"Ledger of {len(LEDGER)} strategies: " + ", ".join(f"{k} {cnt_.get(k, 0)}" for k in ("KEEP", "KEEP-SMALL", "RE-TEST", "DROP")) + ".",
             f"Rank correlation between validation and test Sharpe across strategies: {AUDIT['rho_flat']:+.2f} with flat costs, {AUDIT['rho_exec']:+.2f} under the execution model. With {AUDIT['n']} strategies and no edge at all, the best test Sharpe would be about {AUDIT['luck_max']:.2f} by luck; the best observed is {AUDIT['best_test']:.2f}.",
             f"The {AUDIT['k']} best on validation averaged a test Sharpe of {AUDIT['book_mean_test']:+.2f}; the {AUDIT['k']} best on test (hindsight) {AUDIT['hindsight_mean_test']:+.2f}.",
             f"Dots above the test curve in the frontier figure: {', '.join(AUDIT['above']) or 'none'}; still profitable after execution costs: {', '.join(AUDIT['above_alive']) or 'none'}."]
        tb = LEDGER.rename_axis("strategy").reset_index()[["strategy", "sharpe_val", "sharpe_test", "test_exec", "decision", "reason"]].copy(); tb["reason"] = tb["reason"].str.replace(r" \(cost drag.*$", "", regex=True).map(lambda x: x if len(x) <= 90 else x[:89] + "…")
        sec("G", "Selection audit and keep / drop ledger", b, tb.head(18), "decisions use test Sharpe: an audit, never an input to portfolio construction")
    if "INV" in globals():
        adm_ = INV[INV.admitted]; b = [f"{int(INV.admitted.sum())} of {len(INV)} investor profiles (risk appetite x lock-in) pass the admission test on validation; the others stay in cash. Of the admitted, {int(adm_.profitable_test.sum())} were profitable on test and {int((adm_.status == 'held, mandate met').sum())} met their mandate (volatility at most 30% above target, drawdown, loss share)."]
        if len(adm_): bb = adm_.sort_values("sharpe_test", ascending=False).iloc[0]; b.append(f"Best on test: {bb.risk} / {bb.lock_in}, Sharpe {bb.sharpe_test:+.2f}, return {bb.ann_ret_test:+.1%}, volatility {bb.ann_vol_test:.1%}, drawdown {bb.max_dd_test:.1%}.")
        sec("G", "Portfolios by investor type", b, INV[["risk", "lock_in", "admitted", "sharpe_val", "sharpe_test", "ann_ret_test", "max_dd_test", "status"]].round(3), "pool, admission and weights from validation only; test is the audit")
    if "REC" in globals() and CHOSEN is not None:
        b = [f"{len(valid)} recipes tried; test Sharpe above zero in {int((valid.sharpe_test > 0).sum())}, median {valid.sharpe_test.median():+.2f}.",
             f"Chosen without the test: {CHOSEN.library} | {CHOSEN.rule} | {CHOSEN.scheme}: test Sharpe {CHOSEN.sharpe_test:+.2f}, return {CHOSEN.ret_test:+.1%}, volatility {CHOSEN.vol_test:.1%}, drawdown {CHOSEN.dd_test:.1%}, deflated-Sharpe probability {CHOSEN.dsr_test:.2f}; effective bets {CHOSEN.eff_bets:.1f}.",
             f"No-trade band chosen on validation: tau = {TAU_STAR}; median test Sharpe {BAND.loc[0.0, 'test_median']:+.2f} → {BAND.loc[TAU_STAR, 'test_median']:+.2f}." + (f" Netting the sleeves' trades saves {NETTING['saving']:.0%} of costs." if NETTING else "")]
        sec("G", "Profitable portfolio search and levers", b, LEVERS[["lever", "status", "evidence in this run"]].assign(**{"evidence in this run": LEVERS["evidence in this run"].str[:80]}), "recipe chosen on the second half of validation; all recipes reported")
except Exception as e: print("findings: Part G portfolio section skipped:", repr(e))

# ---------------- write md / html ----------------
md_ = ["# MFT China futures — findings\n", f"_as of {J.get('as_of')} | generated by Step 26b from the saved result tables_\n"]; html_ = ["<html><head><meta charset='utf-8'><style>body{font-family:sans-serif;max-width:1100px;margin:2em auto}table{border-collapse:collapse;font-size:12px}td,th{border:1px solid #ddd;padding:3px 6px}</style></head><body><h1>MFT China futures — findings</h1>"]
for s_ in FINDINGS:
    md_ += [f"\n## {s_['title']}  (part {s_['part']})\n"] + [f"- {x}" for x in s_["bullets"]]
    html_ += [f"<h2>{s_['title']} <small>(part {s_['part']})</small></h2><ul>" + "".join(f"<li>{x}</li>" for x in s_["bullets"]) + "</ul>"]
    if s_["table"] is not None and len(s_["table"]):
        md_ += ["", s_["table"].round(3).to_markdown(index=False) if hasattr(s_["table"], "to_markdown") and "tabulate" in sys.modules or True else ""]
        html_ += [s_["table"].round(3).to_html(index=False, border=0)]
    if s_["rule"]: md_ += [f"\n_Rule: {s_['rule']}_"]; html_ += [f"<p><i>Rule: {s_['rule']}</i></p>"]
try:
    (OUT2 / "research_findings.md").write_text("\n".join(md_)); (OUT2 / "research_findings.html").write_text("\n".join(html_) + "</body></html>")
except Exception as e:
    (OUT2 / "research_findings.md").write_text("\n".join(m for m in md_ if isinstance(m, str))); print("markdown tables skipped:", repr(e))
print(f"findings written: {len(FINDINGS)} sections →", OUT2 / "research_findings.md")
for s_ in FINDINGS[:1] + [x for x in FINDINGS if x["title"].startswith("Strategies")]:
    print("\n" + s_["title"].upper()); [print(" -", textwrap.shorten(x, 300)) for x in s_["bullets"]]

## Step 26d — Ledger: what was used, what was tested and failed, and why it is not carried forward

This ledger is the housekeeping record for the whole notebook. Evidence is the last full run (report 3) and the re-tests of Steps 13f, 13g, 18l and 18m. "Kept" means the method or result is used by a later step or by the findings. "Not carried forward" means it was tested, did not hold out of sample (same sign in train, validation and test, and above the Bonferroni bar), or is redundant, and the reason is given. Nothing below was removed unless no other cell depends on it (checked with an automatic definition/use scan); where a retired study is still needed as plumbing, the code stays and only its claim is withdrawn.

**Kept and working (methods, maths and statistics)**

| Area | What is kept | Why it is relevant |
|---|---|---|
| Data and checks (Steps 1–3, 2f) | ingestion, quality flags, one treatment rule, return distributions | every later step reads them; flags decide the universe |
| Splits and causality (9, 26a) | chronological train / validation / test, embargo, truncation tests, regime-balanced split SPL_R (13f) | the test is read once; look-ahead is checked mechanically |
| Volatility models (11, 11b, 11d, 11e) | GARCH-family selection by regime and time frame, HAR with the overnight variance, residual diagnostics | the volatility forecast is the one result that is stable out of sample |
| Distributions and tails (12, 13) | standardised-residual fits, EVT / GPD tails | used by risk (25) and tail trades |
| Dependence (13b, 13c, 15) | Student-t copula (beats Gaussian out of sample), cointegration, tail dependence | used by portfolio (24) and the risk model |
| Overnight (13d, 13e, 13f) | broad overnight premium, sector split, positive vs negative overnight days, regime atlas | a real, wide effect; the regime layers are the base of every later test |
| Method fixes (13g) | demeaning by product × period × regime and by day cross-section, trailing-volatility control, non-overlapping labels, walk-forward blocks | the tests that remove false positives; every directional state claim is re-judged with them |
| Candle anatomy and thresholds (18k, 18l) | range, body, wick, rejection features; learned per-regime thresholds, extended grid to the 99th percentile, nested selection, mirror de-duplication | the range break at about the 80th percentile is the one stable threshold |
| Close location (18m) | 20-bin close location, layered regimes with shrinkage, bounce vs real edge, break-even cost | the only directional intraday effect, tested for cost |
| Framework, tape and OI-gated setups (18n) | volume-weighted STARK / CAIRO / FLUX at 5 and 15 minutes, notional flow, VWAP, OI velocity, mixture regimes with probability-weighted composite, OI-gated breakout / fakeout / pullback / spring / doji / Fibonacci, Granger and transfer entropy on the signals | closes the gaps found in the audit of the reference documents; kept only where it holds in all three periods |
| Stage 2 / VCP, sweeps, volume nodes (18o) | Stage 2 (EMA150/200) plus breakout versus the full VCP at daily, 60 and 30 minutes, R multiples with the stop below the last contraction low, break-even and time-stop variants, bar-testable Taker items (hurdle, squeeze, flow imbalance, sweep, volume node) | the child must beat its parent in all three periods; order-book items of the Taker note are not testable here |
| Incremental volume over EMA (18p) | volume ÷ EMA of the previous n bars − 1, summed over 5 stamps, n = 2…13 chosen on train_R, at 5 / 15 / 30 / 60 minutes and daily; effect on returns, ranges, OI, breakouts, reversals | raw intraday volume is a time-of-day artifact; only seasonally adjusted and daily rows are counted |
| Per-instrument rules and pre-spike anatomy (18q) | breakout / climax reversal / liquidation reversal with pasted, pooled and per-instrument cuts learned on train_R; Return × ΔOI matrix; last 2-3 bars before spikes that continue or reverse | tested on synthetic data only; spike = own-instrument top tail, not a pasted definition |
| Direction by horizon (18r) | rank correlation and edge of the direction of 10 causal signals at 5m, 30m, 1d, 5d, 20d against the round-trip cost; first horizon where a stable direction beats cost | drift removed per product (train_R); sign as measured or inverted; block-clustered t |
| Sectors and regimes (18s) | atlas per sector; direction that fits each sector; volatility regimes per time frame dealt across train_F / val_F; momentum or reversion (variance-ratio slope) by sector, regime, frame, horizon | horizon and sign learned on train_F; regime must beat its unconditioned parent; test read once |
| Stage 2 robustness (18t) | daily Stage 2 breakout net of cost as a portfolio; versus plain trend filter; long / short, sector, regime, year, leave-one-out | t clustered by 10-day block; test read once |
| RSI-marked entries (18u) | long on RSI cross-up in a bull regime, short on cross-down in a bear regime; regime from RSI or Stage 2; level pasted, pooled or per product | level learned on train_R; child must beat the same regime on every bar; test read once |
| RSI inside Stage 2 (18v) | RSI strength at the signal as entry marker; RSI target / floor exit; pullback entry; daily Stage 2 long and short | learned on train_R; same trades without RSI as parent; tail winners must be kept; test read once |
| ECHO (18i) | field momentum for size of move (with the trailing-volatility control) | predicts the size of the next move, not its direction |
| Strategies, execution, portfolio, risk (21–25) | cost model, CPCV, volatility targeting, risk parity, HRP, VaR / ES backtests | the honest end of the pipeline |

**Tested and not carried forward (with the reason)**

| Study | Result | Reason it is not relevant |
|---|---|---|
| WAVE template classifier (18j) | test IC 0.000 (t = 0.0); 0 of 32 regime × state cells survive demeaning | no directional or size information beyond what ECHO already gives; **code removed** because nothing else depends on it |
| Direction calls of ECHO (18i) | test IC −0.007 (t = −0.8); direction not confirmed | direction fails; only the size-of-move use is kept (13g) |
| Open-interest / volume state calls (18e) | confirmed on raw labels, 0 after demeaning | the drift in the sample, not the state; kept only as input to 13g |
| Price–volume geometry (18c) | 5 of 33 pairs confirmed, mean efficiency 0.12 | weak; the plumbing (PVG) is used by 18d–18j, so the code stays, the claim is withdrawn |
| Topology: loops, Mapper graph (18d) | 9% of real windows beat the 95th percentile of the null (chance is 5%) | close to chance; code stays because its residual feeds 18g–18j |
| Three flow indicators (18g, 18h) | effective number of independent signals 1.98 of 3; correlations 0.42–0.62 | largely one signal; code stays for the redundancy check |
| Directional state results of 13g | 0 survive both demeaning checks | drift, not state |
| Tangency portfolio (24b) | Sharpe 2.59 validation → 0.11 test | overfitted to validation; equal weight and risk parity are the references |
| Opening reversal (23d) | Sharpe +1.80 train, +1.28 val, +0.21 test net | decays; kept on WATCH, not followed |
| Candle effects beyond 30 minutes (18k, 18l, 18m) | none at 30m bars, none past 30 minutes | the effect decays inside 30 minutes; do not hold longer |
| Thresholds taken from a rulebook (18l) | learned thresholds beat the rulebook by only about 6% on test | the rulebook is acceptable; learned thresholds are reported, not required |

**Why some retired code stays.** A study can fail and still define a variable that a later step uses (for example the topology residual `td_resid` and the geometry output `PVG` feed Steps 18g–18i). Removing it would break those steps, so the code stays and the result is marked "not carried forward" here. The dependency scan is `unresolved()` in the build script; it lists every name that would become undefined if a cell range were dropped.

## Step 26c — Report deck: every plot of every step, plus the issues log

One landscape slide per figure, in research order, grouped by part, with a divider slide per part, a contents slide, and an **issues appendix** (failed cells with traceback, warnings, output lines that mention a failure / skip / empty result). Each figure slide carries the step's *"used later"* note, so a reader sees what each result feeds.

* `build_report()` writes `report_cumulative.pdf` (everything run so far) and `report_part_<X>.pdf` for each part that has figures, into `research_out/report/`.
* It can be called **at any point** (after any part, or after a failure in a later cell) to get the report of what exists so far; the issues appendix then tells you what to fix.
* Figures that steps do not save themselves (the EDA of Steps 2–8) are captured automatically when they are shown (Step 0b recorder).

**Used later.** This is a read-out for review and debugging: the failed-cell list is the first thing to read after a run, the flagged lines show silent skips that never raised an error.

In [ ]:
# step 26c
import textwrap
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle

STEP_USE = {"0": "Every later step reads these settings and the file registry. Sector and contract metadata drive the sector-level work of Steps 13b, 18, 21 and 24.",
    "1": "The daily panel feeds Steps 2–26. The 5-minute bars (return, realised variance, volume, OI, high / low / close) feed Steps 11b, 13, 14, 19 and 23a. The roll gap and the contract codes feed the carry factors of Step 18.",
    "2": "Quality flags decide which products and periods enter the universe (Step 9) and which bars are dropped (session opens, roll days, constant series).",
    "3": "The return distributions justify the fat-tailed models of Steps 11–13 and the use of log returns in the relationship tests. Simple returns are used for P&L in Steps 21–25.",
    "4": "Volatility, volume and OI profiles set the volatility-scaling window (Step 9), the time-of-day buckets (Steps 13 and 23a) and the liquidity ranks (Steps 17 and 24).",
    "5": "Correlation clusters and lead–lag candidates seed the relationship search of Steps 14–15 and the sector structure of Step 13b.",
    "6": "Exploratory feature ICs decide which feature families are built properly in Step 18.",
    "7": "These hypotheses become the factor families of Step 18 and the strategy templates of Step 23.",
    "8": "The hand-off table lists the facts that Parts B–H test formally, with train / validation / test discipline.",
    "9": "`SPL` (one calendar for all frequencies) and the stationary variable (return or fractionally differenced price) are used by every step from 10 to 26. `UNIV` defines the universe.",
    "10": "The mean-model results say how much linear own-series predictability exists (little), which is why Step 14 tests links *between* series and Step 18 predicts through cross-sectional ranks.",
    "11": "`BEST_VOL` (made robust in Step 11b) standardises the residuals used by Steps 12–13 and the VaR / ES models of Step 25. The conditional variances also give the volatility scaling of the risk tests.",
    "16": "Valid chains join the relationship library (Step 17), the strategy library as C5 (Step 23) and the monitor (Step 22).",
    "23b": "Strategies that survive the execution model and the CPCV check enter the portfolio of Step 24, and the break-even book size caps the capacity quoted on the board.",
    "12": "The winning distribution gives the innovations for VaR / ES (Step 25) and for the tail studies of Step 13.",
    "13": "GPD tails give VaR / ES in Step 25. The tail events of 13.3 feed strategy C3 (Step 23) and the tail-spillover links of the relationship library (Step 17).",
    "14": "Replicated links go to the relationship library (Step 17), to strategies C1 / C4 (Step 23) and to the monitor (Step 22). Survival gives the holding period.",
    "15": "Cointegrated pairs and tail dependence go to the relationship library (Step 17) and to strategy C2 (Step 23). The sector lead–lag feeds Step 21.",
    "17": "The scored library feeds the opportunity board (Step 26) and the monitor (Step 22).",
    "18": "Orientations and scores feed the ranks of Step 19, the dynamics of Step 21, the B strategies of Step 23, and the family groups of Step 18b.",
    "19": "UpR / DnR ranks feed the rank strategies A1–A5 and the meta-labelling strategy E (Step 23) and the board (Step 26).",
    "20": "Regime labels feed the regime-switching strategy D (Step 23), the P&L attribution, and the regime behaviour of factors and ranks in Steps 21–22.",
    "21": "The chosen rotation / mean-reversion configurations feed strategies F0–F5 (Step 23). The state variables feed the board tilt. The dynamics series feed the monitor (Step 22).",
    "22": "Each edge's status feeds the gated books G1 / G2 (Step 23) and the health adjustment of the board (Step 26).",
    "23a": "The cost model re-costs the library in Step 23b and the intraday links in Step 23. Half-spreads and traded values also feed the capacity estimate of Step 24.",
    "23": "The backtests (`BT`, `NETS`) feed the execution re-costing and CPCV (Step 23b), the portfolio (Step 24) and the board (Step 26).",
    "24": "The book weights give exposure and risk by sector, capacity and cost stress, which become the sizing notes of the board.",
    "25": "The chosen VaR / ES model gives the position-risk limits quoted with every board entry.",
    "16b": "The rolling-hedge chains, their status (VALID, WATCH, WAIT, SKIP) and size caps feed the findings report; `fundamental_skip_reasons.csv` says why each chain was skipped in Step 16.",
    "2f": "The treated daily panel replaces the raw one for every later step; the ledger `data_quality_treatment.csv` is quoted in the findings report (Step 26b).",
    "13c": "Tail dependence and the t-copula choice feed the joint-extreme risk of Step 24b and the portfolio ES quoted in the findings.",
    "13d": "The overnight / intraday split and the T+1 comparison motivate the overnight strategies of Step 23c.",
    "23c": "H1 / H2 enter the strategy library (outside CPCV, because their P&L is not close-to-close) and are judged by Step 24 and the findings verdicts.",
    "24b": "Frontier weights are frozen on validation and judged on test; the portfolio copula gives the ES quoted in Step 26b.",
    "24c": "The ledger is the strategy list of the findings (26b); it uses test Sharpe, so it audits and never feeds portfolio construction.",
    "24d": "The investor grid is the investor-facing result of the findings (26b); cells in cash say that nothing in the library earns a place at that risk and horizon.",
    "24e": "The chosen recipe enters PORT_BOOKS and the findings; the no-trade band is the cost lever for any later strategy work.",
    "24f": "The levers table is the research to-do list of the findings; the netting saving is a sizing note for the book.",
    "26b": "The findings feed the executive slides of the report deck (Step 26c).",
    "26": "This is the final output: the board, `research_summary.json` and `report.html`."}
REPORT_DIR = OUT2 / "report"; REPORT_DIR.mkdir(exist_ok=True)
REPORT_PARTS = [("A", "Data and EDA", 0, 8), ("B", "Univariate and multivariate properties", 9, 13), ("C", "Relationships between products", 14, 17),
                ("D", "Factors and opportunity ranks", 18, 19), ("E", "Regimes", 20, 20), ("F", "Dynamics and monitoring", 21, 22),
                ("G", "Strategies, portfolio and risk", 23, 25), ("H", "The opportunity board", 26, 26), ("I", "Future steps", 27, 999)]
_W, _H = 13.333, 7.5
_INK, _INK2, _MUTE, _ACC = "#0b0b0b", "#52514e", "#b9b8b2", "#2a78d6"

def _num(label):
    m = re.match(r"(\d+)", str(label)); return int(m.group(1)) if m else 999
def _part_of(label):
    n = _num(label)
    for code, name, lo, hi in REPORT_PARTS:
        if lo <= n <= hi: return code
    return "I"
def _use_note(label):
    return STEP_USE.get(str(label)) or STEP_USE.get(re.sub(r"[a-z]+$", "", str(label)), "")

def _collect():
    items = [(str(p), t, Path(pth), None) for p, t, pth in FIGS if Path(pth).exists()] + [(str(s), t, Path(pth), c) for s, t, pth, c in AUTOFIGS if Path(pth).exists()]
    items = [(s, t, p, c, k) for k, (s, t, p, c) in enumerate(items)]
    items.sort(key=lambda r: (_num(r[0]), re.sub(r"^\d+", "", r[0]), r[4]))
    return items

def _slide():
    fig = plt.figure(figsize=(_W, _H), facecolor="white"); return fig
def _band(fig, left, right=""):
    fig.add_artist(Rectangle((0, 0.93), 1, 0.07, transform=fig.transFigure, color="#0b0b0b"))
    fig.text(0.02, 0.965, left, color="white", fontsize=13, fontweight="bold", va="center")
    if right: fig.text(0.98, 0.965, right[:95], color="#d9d8d3", fontsize=10.5, va="center", ha="right")
def _foot(fig, left, page):
    fig.text(0.02, 0.012, left[:170], fontsize=7.5, color=_INK2, va="bottom"); fig.text(0.98, 0.012, f"{page}", fontsize=8, color=_INK2, ha="right", va="bottom")

def _fails(codes=None):
    return [r for r in RUNLOG if not r["ok"] and (codes is None or _part_of(r["step"]) in codes)]

def _finding_slides(fd):
    out = [dict(kind="findings", title=fd["title"], part=fd["part"], bullets=fd["bullets"], table=None, rule=fd["rule"])]
    if fd.get("table") is not None and len(fd["table"]):
        out.append(dict(kind="findings", title=fd["title"] + " (table)", part=fd["part"], bullets=[], table=fd["table"].head(14), rule=fd["rule"]))
    return out

def _slides_for(codes, with_appendix):
    items = [i for i in _collect() if _part_of(i[0]) in codes]
    S = []
    if len(codes) > 1:
        for fd in (globals().get("FINDINGS") or []):
            if fd["part"] == "ALL": S += _finding_slides(fd)
    for code, name, lo, hi in REPORT_PARTS:
        if code not in codes: continue
        mine = [i for i in items if _part_of(i[0]) == code]
        nf = len(_fails({code}))
        if not mine and not nf and not any(fd["part"] == code for fd in (globals().get("FINDINGS") or [])): continue
        S.append(dict(kind="divider", code=code, name=name, steps=sorted({i[0] for i in mine}, key=lambda x: (_num(x), x)), nfig=len(mine), nfail=nf))
        for fd in (globals().get("FINDINGS") or []):
            if fd["part"] == code: S += _finding_slides(fd)
        for s, t, p, c, k in mine: S.append(dict(kind="fig", code=code, step=s, title=t, path=p, cell=c))
        if not with_appendix:
            S += _issue_slides({code})
    if with_appendix: S += _issue_slides(None)
    return S

def _issue_slides(codes):
    S = []; fails = _fails(codes)
    flags = [f for f in FLAGS if codes is None or _part_of(f[1]) in codes]
    seen, uf = set(), []
    for f in flags:
        if f[2] not in seen: seen.add(f[2]); uf.append(f)
    ttl = "Issues" + ("" if codes is None else f" — part {sorted(codes)[0]}")
    S.append(dict(kind="health", title=ttl, fails=fails, nflags=len(uf), nwarn=len(WARNLOG) if codes is None else None, ncells=len(RUNLOG)))
    if codes is None and globals().get("CAUSAL_LOG"):
        C_ = pd.DataFrame(CAUSAL_LOG); C_ = C_[~C_.test.str.startswith(("negative control", "positive control"))]
        g_ = C_.groupby(["step", "test"]).agg(passed=("passed", "all"), cuts=("cut", "nunique"), outputs=("output", "nunique"), diff=("max_abs_diff", "max")).reset_index()
        sc_ = globals().get("SCAN"); rv_ = sc_[sc_.verdict == "REVIEW"].drop_duplicates("code") if isinstance(sc_, pd.DataFrame) and len(sc_) else pd.DataFrame()
        S.append(dict(kind="causal", title="Causality guardrails", rows=g_.to_dict("records"), review=rv_.to_dict("records") if len(rv_) else [], scan=(sc_.verdict.value_counts().to_dict() if isinstance(sc_, pd.DataFrame) and len(sc_) else {})))
    for r in fails[:15]: S.append(dict(kind="trace", title=ttl, rec=r))
    if codes is None and WARNLOG:
        W = sorted(WARNLOG.items(), key=lambda kv: -kv[1])
        for i in range(0, min(len(W), 60), 18): S.append(dict(kind="warn", title=ttl, rows=W[i:i + 18]))
    for i in range(0, min(len(uf), 300), 17): S.append(dict(kind="flags", title=ttl, rows=uf[i:i + 17]))
    return S

def _draw(pdf, sl, page, total, toc):
    fig = _slide(); k = sl["kind"]
    if k == "title":
        fig.add_artist(Rectangle((0, 0), 1, 1, transform=fig.transFigure, color="#0b0b0b"))
        fig.text(0.06, 0.62, "MFT China futures", color="white", fontsize=40, fontweight="bold")
        fig.text(0.06, 0.52, sl["subtitle"], color="#d9d8d3", fontsize=18)
        fig.text(0.06, 0.38, sl["line"], color="#b9b8b2", fontsize=12); fig.text(0.06, 0.33, sl["line2"], color="#b9b8b2", fontsize=12)
    elif k == "toc":
        _band(fig, "Contents", "slide number in brackets")
        y = 0.86
        for code, name, steps, pg, nfig, nfail in toc:
            fig.text(0.05, y, f"Part {code} — {name}", fontsize=14, fontweight="bold", color=_INK)
            fig.text(0.95, y, f"[{pg}]", fontsize=14, ha="right", color=_ACC)
            fig.text(0.07, y - 0.035, f"{nfig} figures | steps: {', '.join(steps) if steps else 'none'}" + (f" | {nfail} failed cell(s)" if nfail else ""), fontsize=9.5, color=_INK2)
            y -= 0.085
    elif k == "divider":
        fig.add_artist(Rectangle((0, 0), 1, 1, transform=fig.transFigure, color="#f4f8fd"))
        fig.text(0.06, 0.62, f"Part {sl['code']}", fontsize=22, color=_ACC, fontweight="bold")
        fig.text(0.06, 0.52, sl["name"], fontsize=34, fontweight="bold", color=_INK)
        fig.text(0.06, 0.40, f"{sl['nfig']} figures | steps {', '.join(sl['steps']) if sl['steps'] else '-'}", fontsize=13, color=_INK2)
        fig.text(0.06, 0.34, "failed cells in this part: " + (str(sl["nfail"]) if sl["nfail"] else "none"), fontsize=13, color="#c4471a" if sl["nfail"] else _INK2)
    elif k == "fig":
        _band(fig, f"Part {sl['code']}  ·  Step {sl['step']}", sl["title"])
        ax = fig.add_axes([0.02, 0.15, 0.96, 0.77]); ax.axis("off")
        ax.imshow(plt.imread(sl["path"]), aspect="equal"); ax.set_anchor("C")
        note = _use_note(sl["step"])
        if note: fig.text(0.02, 0.105, "\n".join(textwrap.wrap("Used later: " + note.replace("`", ""), 190)[:3]), fontsize=8.5, color=_INK2, va="top")
        _foot(fig, f"{sl['path'].name}" + (f"  |  notebook cell {sl['cell']}" if sl.get("cell") else ""), page)
    elif k == "health":
        _band(fig, sl["title"], "run health")
        nf = len(sl["fails"]); fig.text(0.04, 0.86, f"{sl['ncells']} cells recorded | {nf} failed | {sl['nflags']} distinct flagged output lines" + (f" | {sl['nwarn']} warning kinds" if sl["nwarn"] is not None else ""), fontsize=13, color=_INK)
        if not nf: fig.text(0.04, 0.78, "No cell raised an error.", fontsize=13, color="#1baf7a")
        y = 0.78
        for r in sl["fails"][:12]:
            fig.text(0.04, y, f"cell {r['cell']}  step {r['step']}", fontsize=10, fontweight="bold", color="#c4471a")
            fig.text(0.20, y, textwrap.shorten(r["first"], 70), fontsize=9, color=_INK2, family="monospace")
            fig.text(0.04, y - 0.027, "\n".join(textwrap.wrap(r["error"], 170)[:2]), fontsize=8.5, color=_INK, va="top"); y -= 0.062
    elif k == "trace":
        r = sl["rec"]; _band(fig, f"{sl['title']} · cell {r['cell']} (step {r['step']})", "traceback")
        fig.text(0.03, 0.89, r["first"][:150], fontsize=9, family="monospace", color=_INK2)
        fig.text(0.03, 0.85, "\n".join(sum([textwrap.wrap(l, 175) or [""] for l in r["traceback"].splitlines()[-26:]], [])[-34:]), fontsize=7.6, family="monospace", va="top", color=_INK)
    elif k == "causal":
        _band(fig, sl["title"], "truncation tests (fn(data cut at T) == fn(full data) for t <= T) and static scan")
        y = 0.88
        for r in sl["rows"][:14]:
            ok_ = bool(r["passed"]); fig.text(0.03, y, "PASS" if ok_ else "FAIL", fontsize=10, fontweight="bold", color="#1baf7a" if ok_ else "#c4471a")
            fig.text(0.09, y, f"step {r['step']}: {r['test']}"[:110], fontsize=9.5, color=_INK); fig.text(0.97, y, f"{r['cuts']} cuts × {r['outputs']} outputs | max diff {r['diff']:.1e}", fontsize=8.5, color=_INK2, ha="right"); y -= 0.045
        y -= 0.02; fig.text(0.03, y, "static scan of executed cells: " + (", ".join(f"{k} {v}" for k, v in sl["scan"].items()) or "no hit"), fontsize=10.5, fontweight="bold", color=_INK); y -= 0.045
        for r in sl["review"][:6]:
            fig.text(0.03, y, f"REVIEW {r['cell']}: {r['issue']} | {r['code']}"[:200], fontsize=8.5, family="monospace", color="#c4471a"); y -= 0.04
    elif k == "findings":
        _band(fig, ("Findings · " + sl["title"])[:90], "part " + sl["part"])
        y = 0.88
        for bl in sl["bullets"]:
            lines = textwrap.wrap(bl, 150) or [""]
            if y - 0.03 * len(lines) < 0.1: break
            fig.text(0.03, y, "•", fontsize=12, color=_ACC); fig.text(0.05, y, "\n".join(lines), fontsize=11, color=_INK, va="top"); y -= 0.034 * len(lines) + 0.022
        if sl["table"] is not None:
            tb = sl["table"].copy()
            for c_ in tb.columns:
                if tb[c_].dtype.kind == "f": tb[c_] = tb[c_].round(2)
            tb = tb.astype(str).apply(lambda col: col.str.slice(0, 38))
            ax = fig.add_axes([0.02, 0.1, 0.96, 0.78]); ax.axis("off")
            t_ = ax.table(cellText=tb.values, colLabels=list(tb.columns), loc="upper center", cellLoc="left"); t_.auto_set_font_size(False); t_.set_fontsize(8); t_.scale(1, 1.35)
        if sl["rule"]: fig.text(0.03, 0.06, "Rule: " + sl["rule"][:200], fontsize=9, style="italic", color=_INK2)
    elif k == "warn":
        _band(fig, sl["title"], "python warnings (first occurrence printed once, repeats counted)")
        y = 0.88
        for (cat, msg), n in sl["rows"]:
            fig.text(0.03, y, f"{n}x", fontsize=9.5, fontweight="bold", color=_ACC); fig.text(0.08, y, f"{cat}: {msg}"[:190], fontsize=8.5, family="monospace", color=_INK); y -= 0.048
    elif k == "flags":
        _band(fig, sl["title"], "output lines that mention a failure, skip or empty result (these do not raise errors)")
        y = 0.89
        for cell, step, line in sl["rows"]:
            fig.text(0.03, y, f"cell {cell} · step {step}", fontsize=8, color=_ACC, fontweight="bold"); fig.text(0.17, y, line[:165], fontsize=8, family="monospace", color=_INK); y -= 0.049
    if k not in ("title", "fig"): _foot(fig, "MFT China futures research report", page)
    pdf.savefig(fig); plt.close(fig)

def _write(path, slides, subtitle):
    toc = []
    pg = 3
    # page numbers: title = 1, contents = 2, then slides in order
    pages = {}
    for i, sl in enumerate(slides): pages[i] = 3 + i
    for i, sl in enumerate(slides):
        if sl["kind"] == "divider": toc.append((sl["code"], sl["name"], sl["steps"], pages[i], sl["nfig"], sl["nfail"]))
    first = RUNLOG[0]["cell"] if RUNLOG else 0
    head = [dict(kind="title", subtitle=subtitle, line=f"{len(_collect())} figures | {len(_fails())} failed cells | {len(RUNLOG)} cells recorded",
                 line2=f"built {pd.Timestamp.now():%Y-%m-%d %H:%M} | data {RAW.index[0].date() if 'RAW' in globals() else '?'} to {RAW.index[-1].date() if 'RAW' in globals() else '?'} | calendar splits train/val/test from Step 9"), dict(kind="toc")]
    allp = head + slides; total = len(allp)
    with PdfPages(path) as pdf:
        for i, sl in enumerate(allp): _draw(pdf, sl, i + 1, total, toc)
    return total

def build_report(only=None):
    out = {}
    cum = _slides_for({c for c, *_ in REPORT_PARTS}, True)
    n = _write(REPORT_DIR / "report_cumulative.pdf", cum, "Research report — all parts run so far"); out["report_cumulative.pdf"] = n
    for code, name, lo, hi in REPORT_PARTS:
        if only and code not in only: continue
        sl = _slides_for({code}, False)
        if sl: out[f"report_part_{code}.pdf"] = _write(REPORT_DIR / f"report_part_{code}.pdf", sl, f"Part {code} — {name}")
    for f, n in out.items(): print(f"{f}: {n} slides, {(REPORT_DIR / f).stat().st_size / 1e6:.1f} MB")
    print("folder:", REPORT_DIR)
    return out

REPORT_FILES = build_report()
fails_ = _fails()
print(f"\n{len(fails_)} failed cell(s)" + ("" if not fails_ else ": " + ", ".join(f"cell {r['cell']} (step {r['step']})" for r in fails_)))

In [ ]:
# step 27 — REPORT.MD: run this one cell to build and download report.md (all steps, all printed values and tables)
import base64, html as _html
from IPython.display import HTML, FileLink, display as _disp
_STEP_NAMES = globals().get("STEP_NAMES", {})
def _fence(t): return "```text\n" + t.rstrip() + "\n```\n"
L = ["# MFT China futures — full run report", "", f"_built {pd.Timestamp.now():%Y-%m-%d %H:%M}; data {RAW.index[0].date()} → {RAW.index[-1].date()}; splits: train to {SPL['train'][-1].date()}, validation {SPL['val'][0].date()} → {SPL['val'][-1].date()}, test from {SPL['test'][0].date()}_", ""]
# 1. findings (Step 26b)
fp = OUT2 / "research_findings.md"
L += ["## 0. Findings (what worked, what did not, what to follow)", "", fp.read_text() if fp.exists() else "_Step 26b has not been run._", ""]
# 2. run health
fails_ = [r for r in RUNLOG if not r["ok"]]
L += ["## 1. Run health", "", f"- cells recorded: {len(RUNLOG)}; failed: {len(fails_)}"] + [f"  - cell {r['cell']} (step {r['step']}): {r['error']}" for r in fails_]
L += [f"- flagged output lines: {len({f[2] for f in FLAGS})}"] + [f"  - cell {c} (step {s}): {t}" for c, s, t in list({f[2]: f for f in FLAGS}.values())[:80]] + [""]
# 3. every cell, in order, grouped by step
L += ["## 2. Output of every cell, step by step", ""]
def _snum(x): 
    m = re.match(r"(\d+)([a-z]?)", str(x)); return (int(m.group(1)), m.group(2)) if m else (999, "")
by_step = {}
for cell in sorted(CELLSRC):
    st, src = CELLSRC[cell]; by_step.setdefault(st, []).append(cell)
for st in sorted(by_step, key=_snum):
    L += [f"### Step {st}", ""]
    figs = [p for s_, t_, p in FIGS if str(s_) == st] if "FIGS" in globals() else []
    for cell in by_step[st]:
        src = CELLSRC[cell][1]; out = CELLOUT.get(cell, [])
        if not out and "savefig" not in src: continue
        title = next((l.strip("# ").strip() for l in src.splitlines()[1:6] if l.strip().startswith("# ---") or l.strip().startswith("# ")), src.strip().splitlines()[0][:90])
        L += [f"**cell {cell}** — {title}", ""]
        for kind, txt in out:
            L.append(_fence(txt))
        L.append("")
    if figs: L += ["Figures: " + ", ".join(f"`{Path(p).name}`" for p in figs), ""]
# 4. result tables
L += ["## 3. Result tables (first 40 rows of every CSV in research_out)", ""]
for p in sorted(OUT2.glob("*.csv")):
    try:
        d = pd.read_csv(p); L += [f"### {p.name}  ({len(d)} rows × {d.shape[1]} columns)", "", _fence(d.head(40).round(4).to_string(index=False, max_colwidth=40))]
    except Exception as e: L += [f"### {p.name}", f"_unreadable: {e}_", ""]
# 5. write + offer download
txt = "\n".join(L); REPORT_MD = OUT2 / "report.md"; REPORT_MD.write_text(txt, encoding="utf-8")
print(f"report.md written: {REPORT_MD}  ({len(txt) / 1e6:.2f} MB, {len(by_step)} steps, {sum(len(v) for v in by_step.values())} cells)")
b64 = base64.b64encode(txt.encode("utf-8")).decode()
_disp(HTML(f'<a download="report.md" href="data:text/markdown;base64,{b64}" style="font-size:16px;font-weight:bold">⬇ download report.md</a>'))
_disp(FileLink(str(REPORT_MD)))

## Step 27b — Report delta: only what changed since the previous run

`report_delta.md` and `report_delta.pdf` list only the differences against the previous `report.md`: findings sections that are new, removed or changed (changed lines only), cell outputs that are new, removed or changed, figures added or removed, and result tables whose rows changed. Cell numbers, timestamps and paths are ignored so that moving a cell does not look like a change.

**To use it:** add the previous run's `report.md` as an input dataset (any folder under `/kaggle/input`), or set `PREV_REPORT = "path/to/report.md"` before this cell. The newest `report*.md` in the input folders is taken as the previous version. Files are written to `research_out/report/`. If no previous report is found the cell says so and does nothing.

**Used later.** This is a read-out for review; no later step reads it.

In [ ]:
# step 27b — REPORT DELTA: report_delta.md and report_delta.pdf hold only what changed against the previous report.md
import re, difflib, textwrap, html as _h
from pathlib import Path as _P

# Where the previous report lives. Add the previous run's report.md as an input dataset (any folder under /kaggle/input), or set PREV_REPORT to a path.
PREV_REPORT = globals().get("PREV_REPORT", None)

def _find_prev(cur_path=None):
    if PREV_REPORT and _P(PREV_REPORT).exists(): return _P(PREV_REPORT)
    roots = [_P("/kaggle/input"), _P("/mnt/user-data/attachments"), _P("prev_report")]
    c = []
    for r in roots:
        if r.exists(): c += [p for p in r.rglob("*.md") if "report" in p.name.lower() and "delta" not in p.name.lower() and (cur_path is None or p.resolve() != _P(cur_path).resolve())]
    return max(c, key=lambda p: p.stat().st_mtime) if c else None

_VOLATILE = re.compile(r"(_built .*?_|\d+\.\d+ ?s\b|elapsed|runtime|seconds|/kaggle/\S+|/home/\S+|/tmp/\S+|\bcell \d+\b|report\.md written.*)", re.I)
def _norm(line): return _VOLATILE.sub("", line).strip()

def _parse(txt):
    # -> dict: sections {title: [lines]} for findings; steps {(step, title, k): [lines]}; tables {name: [lines]}
    sec = {}; steps = {}; tabs = {}; part = "head"; cur = None; seen = {}
    step = None; in_fence = False
    for ln in txt.splitlines():
        if ln.startswith("## 0."): part = "find"; cur = None; continue
        if ln.startswith("## 1."): part = "health"; cur = ("health",); sec.setdefault("Run health", []); cur = sec["Run health"]; continue
        if ln.startswith("## 2."): part = "steps"; cur = None; continue
        if ln.startswith("## 3."): part = "tabs"; cur = None; continue
        if part == "find":
            if ln.startswith("## "): cur = sec.setdefault(ln[3:].strip(), [])
            elif cur is not None and ln.strip(): cur.append(ln)
        elif part == "health":
            if ln.strip(): sec["Run health"].append(ln)
        elif part == "steps":
            if ln.startswith("### Step"): step = ln[4:].strip(); cur = None
            elif ln.startswith("**cell "):
                m = re.match(r"\*\*cell (\d+)\*\* — (.*)", ln); title = (m.group(2) if m else ln)[:90].strip()
                k = seen.get((step, title), 0); seen[(step, title)] = k + 1
                cur = steps.setdefault((step, title, k), [])
            elif ln.startswith("Figures:") and step: steps.setdefault((step, "Figures", 0), []).extend([f.strip(" `") for f in ln[8:].split(",")])
            elif cur is not None and not ln.startswith("```") and ln.strip(): cur.append(ln)
        elif part == "tabs":
            if ln.startswith("### "): cur = tabs.setdefault(ln[4:].strip(), [])
            elif cur is not None and not ln.startswith("```") and ln.strip(): cur.append(ln)
    return sec, steps, tabs

def _diff(a, b, maxn=60):
    a_ = [_norm(x) for x in a]; b_ = [_norm(x) for x in b]
    rem = [x for x in a if _norm(x) not in set(b_) and _norm(x)]; add = [x for x in b if _norm(x) not in set(a_) and _norm(x)]
    return rem[:maxn], add[:maxn], len(rem), len(add)

def build_delta(prev_path, cur_text, out_dir, cur_label="current", prev_label=None):
    prev_txt = _P(prev_path).read_text(encoding="utf-8", errors="replace"); prev_label = prev_label or _P(prev_path).name
    ps, pst, pt = _parse(prev_txt); cs, cst, ct = _parse(cur_text)
    L = [f"# Report delta — only what changed", "", f"_previous: `{prev_label}` → current: `{cur_label}`_", ""]
    S = {"findings_added": 0, "findings_removed": 0, "findings_changed": 0, "steps_new": 0, "steps_gone": 0, "steps_changed": 0, "tables_new": 0, "tables_gone": 0, "tables_changed": 0}
    body = []
    # 1. findings
    fb = []
    for t in [t for t in cs if t not in ps]: S["findings_added"] += 1; fb += [f"### NEW section: {t}", ""] + cs[t][:40] + [""]
    for t in [t for t in ps if t not in cs]: S["findings_removed"] += 1; fb += [f"### REMOVED section: {t}", ""]
    for t in [t for t in cs if t in ps]:
        rem, add, nr, na = _diff(ps[t], cs[t])
        if rem or add: S["findings_changed"] += 1; fb += [f"### CHANGED: {t}  (−{nr} / +{na} lines)", ""] + [f"- ~~{x.strip()[:300]}~~" for x in rem[:25]] + [f"+ {x.strip()[:300]}" for x in add[:25]] + [""]
    # 2. step outputs
    sb = []
    newk = [k for k in cst if k not in pst]; gone = [k for k in pst if k not in cst]
    S["steps_new"] = len(newk); S["steps_gone"] = len(gone)
    for k in newk:
        if k[1] == "Figures": continue
        sb += [f"**NEW** {k[0]} — {k[1]}", ""] + ["    " + x for x in cst[k][:12]] + (["    …"] if len(cst[k]) > 12 else []) + [""]
    for k in gone:
        if k[1] == "Figures": continue
        sb += [f"**REMOVED** {k[0]} — {k[1]}", ""]
    for k in [k for k in cst if k in pst]:
        rem, add, nr, na = _diff(pst[k], cst[k], 30)
        if k[1] == "Figures":
            if rem or add: sb += [f"**FIGURES** {k[0]}: " + (f"added {', '.join(add)}" if add else "") + (f"; removed {', '.join(rem)}" if rem else ""), ""]
        elif rem or add:
            S["steps_changed"] += 1; sb += [f"**CHANGED** {k[0]} — {k[1]}  (−{nr} / +{na} lines)", ""] + [f"    − {x.strip()[:220]}" for x in rem[:15]] + [f"    + {x.strip()[:220]}" for x in add[:15]] + [""]
    # 3. tables
    tb = []
    for n in [n for n in ct if n not in pt]: S["tables_new"] += 1; tb += [f"**NEW table** {n}", ""]
    for n in [n for n in pt if n not in ct]: S["tables_gone"] += 1; tb += [f"**REMOVED table** {n}", ""]
    for n in [n for n in ct if n in pt]:
        rem, add, nr, na = _diff(pt[n], ct[n], 12)
        if rem or add: S["tables_changed"] += 1; tb += [f"**CHANGED table** {n}  (−{nr} / +{na} rows)", ""] + [f"    − {x.strip()[:200]}" for x in rem[:6]] + [f"    + {x.strip()[:200]}" for x in add[:6]] + [""]
    L += ["## Summary", "", "| | new | removed | changed |", "|---|---|---|---|",
          f"| findings sections | {S['findings_added']} | {S['findings_removed']} | {S['findings_changed']} |",
          f"| cell outputs | {S['steps_new']} | {S['steps_gone']} | {S['steps_changed']} |",
          f"| result tables | {S['tables_new']} | {S['tables_gone']} | {S['tables_changed']} |", ""]
    L += ["## 1. Findings that changed", ""] + (fb or ["_no change_", ""]) + ["## 2. Cell outputs that changed", ""] + (sb or ["_no change_", ""]) + ["## 3. Result tables that changed", ""] + (tb or ["_no change_", ""])
    txt = "\n".join(L); out_dir = _P(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "report_delta.md").write_text(txt, encoding="utf-8")
    # PDF: text pages only (changes only)
    import matplotlib; from matplotlib.backends.backend_pdf import PdfPages; import matplotlib.pyplot as _plt
    lines = []
    for ln in txt.splitlines():
        ln = re.sub(r"[*`#]|~~", "", ln)
        lines += textwrap.wrap(ln, 120, subsequent_indent="      ") or [""]
    per = 62
    with PdfPages(out_dir / "report_delta.pdf") as pdf:
        for p in range(0, max(1, len(lines)), per):
            fig = _plt.figure(figsize=(11.69, 8.27)); fig.text(0.04, 0.97, "Report delta — only what changed", fontsize=11, weight="bold", va="top")
            fig.text(0.04, 0.94, "\n".join(lines[p:p + per]), fontsize=7.2, family="monospace", va="top"); pdf.savefig(fig); _plt.close(fig)
    return S, txt

if "REPORT_MD" in globals() and _P(REPORT_MD).exists():
    _prev = _find_prev(REPORT_MD)
    if _prev is None:
        print("report delta skipped: no previous report.md found. Add the previous run's report.md as an input dataset (or set PREV_REPORT = 'path') and re-run this cell.")
    else:
        # the previous report is read from the input folder; the report just written is the current one
        _S, _t = build_delta(_prev, _P(REPORT_MD).read_text(encoding="utf-8"), OUT2 / "report", cur_label=_P(REPORT_MD).name, prev_label=_prev.name)
        print("report delta vs", _prev, "->", OUT2 / "report" / "report_delta.md", "and report_delta.pdf"); print(_S)
else:
    print("report delta skipped: run the report.md cell (Step 27) first.")

---
# Part I — Future steps (27+)
**Conventions:**
* Configure in Step 0b.
* Read caches from `OUT_DIR` (Part A) and `OUT2` (Parts B–H).
* Fit on `SPL["train"]`, select on `SPL["val"]`, touch `SPL["test"]` once; embargo at least the longest forward horizon (`EMB`).
* Reuse the toolbox: `granger_discover/blind/score`, `purged_kfold`, `triple_barrier`, `pbo_cscv`, `hrp_weights`, `to_weights`, `backtest`, `perf`, `alloc`, `combine`, `MON` (monitor), `shp`.
* Save figures with `savefig(fig, name, title, "31")` and re-run Step 26.

| Step | Idea | Needs |
|---|---|---|
| 27 | **Queue position and L1 spread**: replace the bar-based bounds of Step 23a with a quote-based model (fill probability by queue rank, quoted spread, dynamic slippage by intraday volatility) | tick / order-book data |
| 28 | **Full term-structure carry**: roll yield and curvature over the whole curve (Step 18 uses the main-contract roll slope only) | non-main contract files |
| 29 | **CPCV with re-fitting** for the model-based strategies (meta-labelling, regime weights): Step 23b re-samples the choice among frozen strategies, not their parameters | compute only |
| 30 | Inventory, warehouse-receipt and basis data as fundamentals for the chains in Step 16 | external data |

In [ ]:
# step 26c
# ---------------- Step 27 template ----------------
# OUT3 = OUT2 / "step27"; OUT3.mkdir(exist_ok=True)
# ... fit on SPL["train"], select on SPL["val"], evaluate once on SPL["test"] ...
# fig, ax = plt.subplots(); ...; savefig(fig, "27_example", "Step 27 example", "27")

## Save the cache\n\nRe-run the cell below to refresh `eda_cache.zip` with everything the later steps cached. Download it from the Output tab and upload it as a Dataset.

In [ ]:
# step 28
# Pack the cache into ONE zip (Kaggle: after Save Version, Output tab > New Dataset creates a dataset from it with no download).
# Attach that Dataset to later runs: the notebook copies it in and Step 1 loads it (no rebuild). Re-run this cell at the end of the notebook to refresh the zip with the later-step files.
import zipfile
ZIP_P = (OUT_DIR.parent / "eda_cache.zip") if OUT_DIR.parent.exists() else Path("eda_cache.zip")
with zipfile.ZipFile(ZIP_P, "w", zipfile.ZIP_STORED) as _z:                 # parquet is already compressed
    for _f in sorted(OUT_DIR.iterdir()):
        if _f.is_file(): _z.write(_f, _f.name)
print(f"{ZIP_P}  {ZIP_P.stat().st_size/1e6:.0f} MB  ({len(list(OUT_DIR.iterdir()))} files)")
print("Kaggle serves no direct link to /kaggle/working. To keep this cache as an input: Save Version (Save & Run All), open the saved version, Output tab, then 'New Dataset' (no download needed). Or: right panel > Output > eda_cache.zip > three dots > Download.")